In [ ]:
#!/usr/bin/env python3
"""
V68.1 PATCHED — Practical standard protocols inside a complete-F parent domain
=======================================================================================

Seven incompressible isotropic families:
NH, MR, YEOH2, GENT, signed OGDEN1, positive-weight signed OGDEN2, and GP2.

V68.1 retains the V68 atlas definition and patches target-shape/scale search, ambient revalidation, and safe resume.  Every model
uses the same physical stiffness variable, initial shear modulus mu0 in kPa.  The unit
constitutive templates all have small-shear slope 2, so the physical response multiplier
is exactly s=mu0/2.  The universal literature core is 0.1–12000 kPa and the expanded
future-sample guard band is 0.05–20000 kPa, both sampled logarithmically.

Shape domains are model-specific.  Literature-supported cores are expanded by controlled
guard margins, and representative literature coordinates are forced into the Fisher
samplers.  Fisher / pullback geometry is recomputed over the expanded domains; no V64
cache or atlas is reused.  Shape nodes remain Fisher-adaptive, while stiffness values are
assigned by deterministic stratified logarithmic sampling to avoid a redundant Cartesian
product of every shape with every scale.  Target stiffness is optimized continuously over
the same universal domain.

Every source state is tagged as:
    INSIDE_LITERATURE_CORE
    INSIDE_BIOLOGICAL_MARGIN
    OUTSIDE_CONFIGURED_DOMAIN

Primary region labels remain complete hierarchy-closed compatibility sets at 3%, e.g.
    MR|OGDEN1|OGDEN2|GP2
Minimal adequate models remain secondary diagnostics.  Arbitrary observations outside
every model tube are labeled NONE_COMPATIBLE.

The parent protocol samples the complete practical incompressible deformation-gradient
space up to rigid rotation and principal-axis permutation.  It uses two independent
log-principal strains with every principal stretch constrained to 0.5–2.0, and stores two
pressure-free principal Kirchhoff-stress differences per deformation state.  The standard
experimental subprotocols use narrower practical ranges: UT lambda=0.70–1.50,
equibiaxial BT lambda=0.80–1.25, and simple shear gamma=-0.50–0.50.  Every standard point
is forced exactly into the parent-state set.

Compatibility labels are determined only from the complete-F parent fingerprint.  PCA is
saved for ALL (complete F), UT, BT, SH, UT+BT, UT+SH, and BT+SH while keeping those parent
labels fixed, showing whether parent-identifiable regions remain visible in subprotocols.

Exact hierarchy:
    NH ⊂ MR ⊂ OGDEN2
    NH ⊂ MR ⊂ GP2
    NH ⊂ YEOH2 ⊂ GP2
    NH ⊂ OGDEN1 ⊂ OGDEN2
    NH ⊂ GENT

Default output:
/content/drive/MyDrive/Optimal_Protocol/
    V68_practical_standard_protocols_complete_F_biological_compatibility_atlas

Colab:
    %run /content/V68_1_patched_practical_standard_protocols_complete_F_biological_compatibility_atlas.py

Environment controls:
    V68_OUTPUT_DIR=/custom/path
    V68_FAST_MODE=1
    V68_FAST_AMBIENT=1
    V68_REUSE_SOURCE_ATLAS=auto   # auto, 0, or 1
    V68_REUSE_SOURCE_ATLAS_DIR=/path/to/prior/V68/results
"""

from __future__ import annotations

import hashlib
import json
import logging
import math
import os
import platform
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import qmc


# =============================================================================
# Configuration and hierarchy
# =============================================================================


@dataclass(frozen=True)
class Config:
    project_name: str = "V68_1_patched_practical_standard_protocols_complete_F_biological_compatibility_atlas"
    drive_root: str = (
        "/content/drive/MyDrive/Optimal_Protocol/"
        "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
    )
    seed: int = 20260806

    # Standard diagnostic protocols use broad but experimentally practical soft-tissue
    # ranges.  They are diagnostic subsets of the wider complete-F parent domain.
    # Eighty-one points per path give fine one-dimensional resolution without making the
    # subprotocol fingerprints unnecessarily large.
    n_per_mode: int = 81
    ut_min: float = 0.70
    ut_max: float = 1.50
    bt_min: float = 0.80
    bt_max: float = 1.25
    sh_min: float = -0.50
    sh_max: float = 0.50

    # Complete-F parent domain for incompressible isotropic response.  Any admissible F
    # is represented, up to rigid rotation and principal-axis permutation, by ordered
    # principal logarithmic strains e1>=e2>=e3 with e1+e2+e3=0.  Unlike the previous Hencky-disk construction, this parent, the parent domain now enforces the physically transparent componentwise bound
    # 0.5 <= lambda_i <= 2.0 for every principal stretch.  The admissible set is a
    # hexagon in the trace-free log-strain plane (a triangle after permutation reduction).
    parent_f_states: int = 2601
    parent_f_candidate_points: int = 32768
    parent_f_boundary_points: int = 192
    parent_principal_stretch_min: float = 0.50
    parent_principal_stretch_max: float = 2.00

    # Universal initial-shear-modulus domain, in kPa.  The core is literature-supported;
    # the wider construction domain is a guard band for future biological samples.
    mu0_core_min_kpa: float = 0.10
    mu0_core_max_kpa: float = 12000.0
    mu0_min_kpa: float = 0.05
    mu0_max_kpa: float = 20000.0
    nh_source_scale_points: int = 25
    source_log_scale_strata: int = 31

    # Literature-supported scalar shape cores and expanded construction domains.
    mr_rho_core_min: float = 0.0
    mr_rho_core_max: float = 1.0
    mr_rho_min: float = 0.0
    mr_rho_max: float = 1.0

    yeoh_beta_core_min: float = 0.0
    yeoh_beta_core_max: float = 20.0
    yeoh_beta_min: float = 0.0
    yeoh_beta_max: float = 25.0

    # With 0.5 <= lambda_i <= 2.0 and detF=1, the largest retained I1-3 is 2.25.
    # Gent requires gamma*(I1-3)<1 over every fingerprint component.  The construction
    # cap 0.42 retains a small locking margin while covering a substantially wider
    # biologically plausible shape range than the previous over-restricted 0.07 cap.
    gent_gamma_core_min: float = 0.0
    gent_gamma_core_max: float = 0.35
    gent_gamma_min: float = 0.0
    gent_gamma_max: float = 0.42
    gent_singularity_margin: float = 1.0e-6

    ogden_alpha_core_min: float = -25.0
    ogden_alpha_core_max: float = 25.0
    ogden_alpha_min: float = -30.0
    ogden_alpha_max: float = 30.0
    ogden_alpha_series_threshold: float = 1.0e-5

    # OGDEN2 canonical interior.  Exact one-term boundaries remain represented by hierarchy closure.
    ogden2_weight_epsilon: float = 1.0e-4
    ogden2_alpha_gap_epsilon: float = 1.0e-4

    # GP2 normalized shape cores and expanded guard-band domains after profiling s=C10+C01.
    gp2_rho_core_min: float = 0.0
    gp2_rho_core_max: float = 1.0
    gp2_rho_min: float = 0.0
    gp2_rho_max: float = 1.0
    gp2_beta20_core_min: float = 0.0
    gp2_beta20_core_max: float = 20.0
    gp2_beta20_min: float = 0.0
    gp2_beta20_max: float = 25.0
    gp2_beta11_core_min: float = 0.0
    gp2_beta11_core_max: float = 12.0
    gp2_beta11_min: float = 0.0
    gp2_beta11_max: float = 15.0
    gp2_beta02_core_min: float = 0.0
    gp2_beta02_core_max: float = 20.0
    gp2_beta02_min: float = 0.0
    gp2_beta02_max: float = 25.0

    # Deterministic bounded measurement resolution.
    final_noise_fraction: float = 0.03
    noise_levels: Tuple[float, ...] = (0.0, 0.01, 0.02, 0.03, 0.05)
    mode_floor_per_unit_noise: float = 1.0 / 6.0
    critical_noise_tolerance: float = 3.0e-8

    # All models share the same absolute stiffness domain.  Internally the response multiplier
    # is s=mu0/2 because the unit templates have infinitesimal shear modulus 2.

    # One-dimensional Fisher grids.
    scalar_source_points: int = 301
    scalar_target_points: int = 401
    scalar_fisher_pilot_points: int = 8001
    fisher_uniform_mixture: float = 0.03

    # Multidimensional Fisher-volume sampling.
    ogden2_pilot_candidates: int = 12288
    ogden2_source_points: int = 601
    ogden2_target_points: int = 2200
    ogden2_candidate_pool_factor: int = 8
    ogden2_density_power: float = 0.30

    gp2_pilot_candidates: int = 16384
    gp2_source_points: int = 801
    gp2_target_points: int = 3000
    gp2_candidate_pool_factor: int = 8
    gp2_density_power: float = 0.28

    multidim_metric_eigen_floor_relative: float = 1.0e-11

    # Pairwise continuous refinement.  V68.1 screens candidates both by the original
    # coarse tube score and by scale-invariant response direction.  This prevents a true
    # shape from being discarded merely because its first scale estimate is imperfect.
    coarse_starts: int = 5
    optimizer_maxiter: int = 150
    optimizer_maxfev: int = 1800
    local_refinement_critical_noise_cap: float = 0.12
    scale_refinement_pool: int = 16
    shape_screening_pool: int = 24

    # Ambient searches use larger rescue pools because a bounded noisy observation may sit
    # exactly on the 3% tube boundary.
    observation_coarse_starts: int = 8
    observation_scale_refinement_pool: int = 64
    observation_shape_screening_pool: int = 96

    # Source-state construction audit.
    n_audit_shapes_scalar: int = 7
    n_audit_shapes_ogden2: int = 10
    n_audit_shapes_gp2: int = 10
    n_audit_reps: int = 2
    audit_mu0_min_kpa: float = 0.10
    audit_mu0_max_kpa: float = 12000.0

    # Ambient observation classifier.
    run_ambient_validation: bool = True
    observation_scalar_target_points: int = 451
    observation_ogden2_target_points: int = 2600
    observation_gp2_target_points: int = 3600
    observation_optimizer_maxiter: int = 140
    observation_optimizer_maxfev: int = 1800
    observation_refinement_score_cap: float = 2.5
    known_revalidation_scalar_target_points: int = 751
    known_revalidation_ogden2_target_points: int = 4200
    known_revalidation_gp2_target_points: int = 6000
    known_revalidation_seed_offset: int = 131
    off_manifold_candidates_per_type: int = 18
    certified_none_margin: float = 0.20
    minimum_certified_none_examples: int = 12
    maximum_none_revalidation_examples: int = 30
    revalidation_ogden2_target_points: int = 4200
    revalidation_gp2_target_points: int = 5200

    # Visualization.
    pca_components: int = 3
    generated_cloud_points_per_model: int = 600
    run_generated_cloud_visualization: bool = True


BASE_CFG = Config()
SCALAR_PARENTS: Tuple[str, ...] = ("MR", "YEOH2", "GENT", "OGDEN1")
MULTIDIM_MODELS: Tuple[str, ...] = ("OGDEN2", "GP2")
PARENT_MODELS: Tuple[str, ...] = SCALAR_PARENTS + MULTIDIM_MODELS
ALL_MODELS: Tuple[str, ...] = ("NH",) + PARENT_MODELS
MODEL_ORDER = {m: i for i, m in enumerate(ALL_MODELS)}
MODEL_SHAPE_DIMS: Dict[str, int] = {
    "NH": 0,
    "MR": 1,
    "YEOH2": 1,
    "GENT": 1,
    "OGDEN1": 1,
    "OGDEN2": 3,
    "GP2": 4,
}

# Exact model-containment relations used for hierarchy closure.
STRICT_SUPERS: Dict[str, Tuple[str, ...]] = {
    "NH": ("MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"),
    "MR": ("OGDEN2", "GP2"),
    "YEOH2": ("GP2",),
    "GENT": tuple(),
    "OGDEN1": ("OGDEN2",),
    "OGDEN2": tuple(),
    "GP2": tuple(),
}

# Literature sources used to choose conservative biological cores.  These are construction
# envelopes, not claims that parameters are statistically uniform inside the intervals.
LITERATURE_REFERENCES: Tuple[Dict[str, str], ...] = (
    {
        "key": "MIHAI_2015_BRAIN_FAT",
        "citation": "Mihai et al., J. R. Soc. Interface 12 (2015) 20150486",
        "doi": "10.1098/rsif.2015.0486",
        "use": "brain/fat NH, MR, Gent, and signed Ogden shape behavior",
    },
    {
        "key": "BENNION_2022_IN_VIVO_BRAIN",
        "citation": "Bennion et al., in-vivo human brain material properties (2022)",
        "doi": "10.1098/rsif.2022.0557",
        "use": "human-brain Ogden exponent near -19 and sub-kPa stiffness",
    },
    {
        "key": "PIERRAT_2020_MENINGES",
        "citation": "Pierrat et al., Front. Bioeng. Biotechnol. 8 (2020) 801",
        "doi": "10.3389/fbioe.2020.00801",
        "use": "positive Ogden exponents and high-kPa/MPa meningeal stiffness",
    },
    {
        "key": "DASTJERDI_2018_BREAST",
        "citation": "Dastjerdi et al., BioMed Research International (2018) 3438470",
        "doi": "10.1155/2018/3438470",
        "use": "breast MR, Yeoh, and second-order polynomial parameters",
    },
    {
        "key": "SAMANI_PLEWES_BREAST",
        "citation": "Samani and Plewes breast-tissue polynomial parameter sets",
        "doi": "10.1088/0031-9155/49/18/014",
        "use": "GP2 normalized coefficient magnitudes",
    },
)

# Representative shape anchors extracted from the literature ranges above.  They are forced
# into adaptive source/target candidate sets so biologically important narrow strata cannot
# be missed by Fisher-volume sampling.
LITERATURE_SCALAR_ANCHORS: Dict[str, Tuple[float, ...]] = {
    "MR": (0.0, 0.40, 0.90, 0.99, 1.0),
    "YEOH2": (0.0, 1.0, 6.0, 12.0, 17.0, 20.0),
    "GENT": (0.0, 0.015, 0.030, 0.045, 0.060, 0.070),
    "OGDEN1": (-25.0, -19.0, -10.0, -2.0, 0.0, 2.0, 8.0, 16.55, 25.0),
}

# One canonical GP2 anchor reconstructed from a widely reused adipose-breast coefficient set
# C10=310 Pa, C01=300 Pa, C20=3800 Pa, C11=2250 Pa, C02=4720 Pa.
GP2_LITERATURE_ANCHOR_PHYSICAL: Tuple[Tuple[float, float, float, float], ...] = (
    (300.0 / 610.0, 3800.0 / 610.0, 2250.0 / 610.0, 4720.0 / 610.0),
)

MODEL_COLORS = {
    "NH": "tab:green",
    "MR": "tab:blue",
    "YEOH2": "tab:orange",
    "GENT": "tab:purple",
    "OGDEN1": "tab:red",
    "OGDEN2": "tab:cyan",
    "GP2": "tab:brown",
    "NONE": "black",
}
MODEL_MARKERS = {
    "NH": "*",
    "MR": "o",
    "YEOH2": "^",
    "GENT": "P",
    "OGDEN1": "s",
    "OGDEN2": "D",
    "GP2": "X",
    "NONE": "x",
}


# =============================================================================
# Utilities# =============================================================================
# Utilities
# =============================================================================


def mount_google_drive() -> None:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False)
    except ImportError:
        print("[INFO] google.colab unavailable; running without Drive mount.")


def resolved_config() -> Config:
    if os.environ.get("V68_FAST_MODE", "").strip() == "1":
        return replace(
            BASE_CFG,
            project_name=BASE_CFG.project_name + "_FAST",
            n_per_mode=17,
            parent_f_states=121,
            parent_f_candidate_points=1024,
            parent_f_boundary_points=30,
            nh_source_scale_points=7,
            source_log_scale_strata=9,
            scalar_source_points=13,
            scalar_target_points=25,
            scalar_fisher_pilot_points=241,
            ogden2_pilot_candidates=256,
            ogden2_source_points=27,
            ogden2_target_points=40,
            ogden2_candidate_pool_factor=3,
            gp2_pilot_candidates=384,
            gp2_source_points=17,
            gp2_target_points=50,
            gp2_candidate_pool_factor=3,
            coarse_starts=2,
            optimizer_maxiter=30,
            optimizer_maxfev=180,
            scale_refinement_pool=6,
            shape_screening_pool=8,
            observation_coarse_starts=3,
            observation_scale_refinement_pool=10,
            observation_shape_screening_pool=12,
            observation_scalar_target_points=25,
            observation_ogden2_target_points=40,
            observation_gp2_target_points=50,
            known_revalidation_scalar_target_points=31,
            known_revalidation_ogden2_target_points=50,
            known_revalidation_gp2_target_points=60,
            n_audit_shapes_scalar=1,
            n_audit_shapes_ogden2=1,
            n_audit_shapes_gp2=1,
            n_audit_reps=1,
            off_manifold_candidates_per_type=2,
            minimum_certified_none_examples=1,
            maximum_none_revalidation_examples=3,
            revalidation_ogden2_target_points=50,
            revalidation_gp2_target_points=60,
            run_ambient_validation=os.environ.get("V68_FAST_AMBIENT", "").strip() == "1",
            generated_cloud_points_per_model=30,
            run_generated_cloud_visualization=False,
        )
    return BASE_CFG


def setup_output(cfg: Config) -> Tuple[Path, logging.Logger]:
    override = os.environ.get("V68_OUTPUT_DIR", "").strip()
    outdir = Path(override if override else cfg.drive_root)
    outdir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(cfg.project_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    fh = logging.FileHandler(outdir / "V68_1_patched_run.log", mode="w", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return outdir, logger


def save_json(path: Path, payload: Mapping) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def response_scale_from_mu0_kpa(mu0_kpa: float) -> float:
    """Convert universal initial shear modulus to the multiplier of unit templates."""
    value = float(mu0_kpa)
    if not np.isfinite(value) or value <= 0.0:
        raise ValueError("mu0_kpa must be finite and positive.")
    return 0.5 * value


def mu0_kpa_from_response_scale(scale_kpa: float) -> float:
    return 2.0 * float(scale_kpa)


def logarithmic_mu0_values(
    cfg: Config,
    n: int,
    seed_offset: int = 0,
    core_only: bool = False,
) -> np.ndarray:
    """Deterministic stratified log sampling of the universal stiffness range."""
    n = int(n)
    if n <= 0:
        return np.empty(0, dtype=np.float64)
    lo = cfg.mu0_core_min_kpa if core_only else cfg.mu0_min_kpa
    hi = cfg.mu0_core_max_kpa if core_only else cfg.mu0_max_kpa
    if n == 1:
        return np.array([math.sqrt(lo * hi)], dtype=np.float64)
    rng = np.random.default_rng(cfg.seed + 7000 + int(seed_offset))
    u = (np.arange(n, dtype=np.float64) + 0.5) / n
    rng.shuffle(u)
    values = np.exp(math.log(lo) + u * (math.log(hi) - math.log(lo)))
    # Explicitly retain both guard-band boundaries and the geometric center.
    values[0] = lo
    values[-1] = hi
    if n >= 3:
        values[n // 2] = math.sqrt(lo * hi)
    return values


def scalar_core_bounds(cfg: Config, model: str) -> Tuple[float, float]:
    if model == "MR":
        return cfg.mr_rho_core_min, cfg.mr_rho_core_max
    if model == "YEOH2":
        return cfg.yeoh_beta_core_min, cfg.yeoh_beta_core_max
    if model == "GENT":
        return cfg.gent_gamma_core_min, cfg.gent_gamma_core_max
    if model == "OGDEN1":
        return cfg.ogden_alpha_core_min, cfg.ogden_alpha_core_max
    raise KeyError(model)


def interval_zone(value: float, core: Tuple[float, float], expanded: Tuple[float, float]) -> str:
    v = float(value)
    tol = 1.0e-10 * max(1.0, abs(expanded[0]), abs(expanded[1]))
    if core[0] - tol <= v <= core[1] + tol:
        return "INSIDE_LITERATURE_CORE"
    if expanded[0] - tol <= v <= expanded[1] + tol:
        return "INSIDE_BIOLOGICAL_MARGIN"
    return "OUTSIDE_CONFIGURED_DOMAIN"


def scale_support_zone(cfg: Config, mu0_kpa: float) -> str:
    return interval_zone(
        mu0_kpa,
        (cfg.mu0_core_min_kpa, cfg.mu0_core_max_kpa),
        (cfg.mu0_min_kpa, cfg.mu0_max_kpa),
    )


def shape_support_zone(cfg: Config, model: str, physical: Mapping[str, float]) -> str:
    if model == "NH":
        return "INSIDE_LITERATURE_CORE"
    if model in SCALAR_PARENTS:
        key = {"MR": "rho", "YEOH2": "beta", "GENT": "gamma", "OGDEN1": "alpha"}[model]
        return interval_zone(float(physical[key]), scalar_core_bounds(cfg, model), scalar_bounds(cfg, model))
    if model == "OGDEN2":
        zones = [
            interval_zone(
                float(physical["alpha1"]),
                (cfg.ogden_alpha_core_min, cfg.ogden_alpha_core_max),
                (cfg.ogden_alpha_min, cfg.ogden_alpha_max),
            ),
            interval_zone(
                float(physical["alpha2"]),
                (cfg.ogden_alpha_core_min, cfg.ogden_alpha_core_max),
                (cfg.ogden_alpha_min, cfg.ogden_alpha_max),
            ),
        ]
    elif model == "GP2":
        specs = (
            ("rho", (cfg.gp2_rho_core_min, cfg.gp2_rho_core_max), (cfg.gp2_rho_min, cfg.gp2_rho_max)),
            ("beta20", (cfg.gp2_beta20_core_min, cfg.gp2_beta20_core_max), (cfg.gp2_beta20_min, cfg.gp2_beta20_max)),
            ("beta11", (cfg.gp2_beta11_core_min, cfg.gp2_beta11_core_max), (cfg.gp2_beta11_min, cfg.gp2_beta11_max)),
            ("beta02", (cfg.gp2_beta02_core_min, cfg.gp2_beta02_core_max), (cfg.gp2_beta02_min, cfg.gp2_beta02_max)),
        )
        zones = [interval_zone(float(physical[k]), core, expanded) for k, core, expanded in specs]
    else:
        raise KeyError(model)
    if "OUTSIDE_CONFIGURED_DOMAIN" in zones:
        return "OUTSIDE_CONFIGURED_DOMAIN"
    if "INSIDE_BIOLOGICAL_MARGIN" in zones:
        return "INSIDE_BIOLOGICAL_MARGIN"
    return "INSIDE_LITERATURE_CORE"


def combine_support_zones(shape_zone: str, scale_zone: str) -> str:
    zones = {shape_zone, scale_zone}
    if "OUTSIDE_CONFIGURED_DOMAIN" in zones:
        return "OUTSIDE_CONFIGURED_DOMAIN"
    if "INSIDE_BIOLOGICAL_MARGIN" in zones:
        return "INSIDE_BIOLOGICAL_MARGIN"
    return "INSIDE_LITERATURE_CORE"


def literature_domain_table(cfg: Config) -> pd.DataFrame:
    rows = [
        {"model": "ALL", "parameter": "mu0_kPa", "core_min": cfg.mu0_core_min_kpa, "core_max": cfg.mu0_core_max_kpa, "atlas_min": cfg.mu0_min_kpa, "atlas_max": cfg.mu0_max_kpa},
        {"model": "MR", "parameter": "rho", "core_min": cfg.mr_rho_core_min, "core_max": cfg.mr_rho_core_max, "atlas_min": cfg.mr_rho_min, "atlas_max": cfg.mr_rho_max},
        {"model": "YEOH2", "parameter": "beta=C20/C10", "core_min": cfg.yeoh_beta_core_min, "core_max": cfg.yeoh_beta_core_max, "atlas_min": cfg.yeoh_beta_min, "atlas_max": cfg.yeoh_beta_max},
        {"model": "GENT", "parameter": "gamma=1/Jm", "core_min": cfg.gent_gamma_core_min, "core_max": cfg.gent_gamma_core_max, "atlas_min": cfg.gent_gamma_min, "atlas_max": cfg.gent_gamma_max},
        {"model": "OGDEN1/2", "parameter": "alpha", "core_min": cfg.ogden_alpha_core_min, "core_max": cfg.ogden_alpha_core_max, "atlas_min": cfg.ogden_alpha_min, "atlas_max": cfg.ogden_alpha_max},
        {"model": "GP2", "parameter": "rho", "core_min": cfg.gp2_rho_core_min, "core_max": cfg.gp2_rho_core_max, "atlas_min": cfg.gp2_rho_min, "atlas_max": cfg.gp2_rho_max},
        {"model": "GP2", "parameter": "beta20", "core_min": cfg.gp2_beta20_core_min, "core_max": cfg.gp2_beta20_core_max, "atlas_min": cfg.gp2_beta20_min, "atlas_max": cfg.gp2_beta20_max},
        {"model": "GP2", "parameter": "beta11", "core_min": cfg.gp2_beta11_core_min, "core_max": cfg.gp2_beta11_core_max, "atlas_min": cfg.gp2_beta11_min, "atlas_max": cfg.gp2_beta11_max},
        {"model": "GP2", "parameter": "beta02", "core_min": cfg.gp2_beta02_core_min, "core_max": cfg.gp2_beta02_core_max, "atlas_min": cfg.gp2_beta02_min, "atlas_max": cfg.gp2_beta02_max},
    ]
    return pd.DataFrame(rows)


def ordered_signature(models: Iterable[str]) -> str:
    values = sorted(set(models), key=lambda m: MODEL_ORDER[m])
    return "|".join(values) if values else "NONE"


def hierarchy_closure(models: Iterable[str]) -> Tuple[str, ...]:
    closed = set(models)
    changed = True
    while changed:
        changed = False
        for model in list(closed):
            for sup in STRICT_SUPERS[model]:
                if sup not in closed:
                    closed.add(sup)
                    changed = True
    return tuple(sorted(closed, key=lambda m: MODEL_ORDER[m]))


def minimal_compatible_models(models: Iterable[str]) -> Tuple[str, ...]:
    closed = set(hierarchy_closure(models))
    minimal: List[str] = []
    for model in sorted(closed, key=lambda m: MODEL_ORDER[m]):
        is_dominated = False
        for candidate in closed:
            if model in STRICT_SUPERS.get(candidate, tuple()):
                is_dominated = True
                break
        if not is_dominated:
            minimal.append(model)
    return tuple(minimal)


def region_label_from_compatibility(models: Iterable[str]) -> Tuple[str, str, str]:
    closed = hierarchy_closure(models)
    if not closed:
        return "NONE_COMPATIBLE", "NONE", "OUTSIDE_ALL_3PCT_MODEL_TUBES"
    signature = ordered_signature(closed)
    return signature, signature, f"FULL_COMPATIBILITY_SET_CARDINALITY_{len(closed)}"


def safe_pca(x: np.ndarray, n_components: int = 3) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    mean = np.mean(x, axis=0)
    xc = x - mean
    _, s, vt = np.linalg.svd(xc, full_matrices=False)
    k = min(n_components, vt.shape[0])
    components = vt[:k]
    scores = xc @ components.T
    var = s**2
    ratio = var[:k] / max(float(np.sum(var)), 1.0e-30)
    return scores, components, ratio


# =============================================================================
# Protocol and constitutive models
# =============================================================================


@dataclass(frozen=True)
class Protocol:
    parent_log_principal: np.ndarray
    parent_principal_stretches: np.ndarray
    parent_state_origin: np.ndarray
    ut: np.ndarray
    bt: np.ndarray
    sh: np.ndarray
    mode_slices: Mapping[str, slice]
    mode_names: np.ndarray

    @property
    def dimension(self) -> int:
        return int(self.mode_names.size)

    @property
    def parent_slice(self) -> slice:
        return self.mode_slices["F_PARENT"]

    @property
    def atlas_indices(self) -> np.ndarray:
        sl = self.parent_slice
        return np.arange(sl.start, sl.stop, dtype=int)

    @property
    def n_parent_states(self) -> int:
        return int(self.parent_log_principal.shape[0])


def _standard_protocol_principal_logs(
    ut: np.ndarray,
    bt: np.ndarray,
    sh: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    rows: List[np.ndarray] = []
    origins: List[str] = []
    for lam in ut:
        logs = np.log(np.array([lam, lam**-0.5, lam**-0.5], dtype=np.float64))
        rows.append(np.sort(logs)[::-1]); origins.append("UT_ANCHOR")
    for lam in bt:
        logs = np.log(np.array([lam, lam, lam**-2.0], dtype=np.float64))
        rows.append(np.sort(logs)[::-1]); origins.append("BT_ANCHOR")
    for gam in sh:
        root = math.sqrt(1.0 + 0.25 * float(gam)**2)
        stretches = np.array([root + 0.5*float(gam), 1.0, root - 0.5*float(gam)])
        rows.append(np.sort(np.log(stretches))[::-1]); origins.append("SH_ANCHOR")
    return np.stack(rows), np.asarray(origins, dtype=object)


def _complete_f_parent_states(
    cfg: Config,
    ut: np.ndarray,
    bt: np.ndarray,
    sh: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Sample complete incompressible F-space with bounded principal stretches.

    The trace-free log-strain plane is sampled by a scrambled Sobol sequence subject to
    log(lambda_i) in [log(parent_min), log(parent_max)].  The resulting full-plane hexagon
    is reduced by sorting principal strains because isotropic response is invariant to
    principal-axis permutation.  All practical UT/BT/SH states and the exact polygon
    boundary are retained as mandatory anchors; maximin selection fills the interior.
    """
    anchors, anchor_origins = _standard_protocol_principal_logs(ut, bt, sh)
    lam_min = float(cfg.parent_principal_stretch_min)
    lam_max = float(cfg.parent_principal_stretch_max)
    if not (0.0 < lam_min < 1.0 < lam_max):
        raise ValueError("Parent principal-stretch bounds must satisfy 0 < min < 1 < max.")
    log_min = math.log(lam_min)
    log_max = math.log(lam_max)

    anchor_stretches = np.exp(anchors)
    if np.min(anchor_stretches) < lam_min - 1.0e-12 or np.max(anchor_stretches) > lam_max + 1.0e-12:
        raise ValueError(
            "At least one standard-protocol anchor lies outside the complete-F principal-stretch "
            f"bounds [{lam_min:.6g}, {lam_max:.6g}]."
        )

    n_candidates = max(int(cfg.parent_f_candidate_points), int(cfg.parent_f_states))
    # Oversample before rejection from the bounding square.  About 75% of the square is
    # admissible for reciprocal symmetric bounds such as 0.5–2.0.
    n_draw = max(2 * n_candidates, 2)
    m = int(math.ceil(math.log2(n_draw)))
    sobol = qmc.Sobol(d=2, scramble=True, seed=cfg.seed + 901)
    uv = np.asarray(sobol.random_base2(m), dtype=np.float64)
    e12 = log_min + (log_max - log_min) * uv
    e3 = -np.sum(e12, axis=1)
    candidates = np.column_stack([e12, e3])
    admissible = np.all(candidates >= log_min - 1.0e-14, axis=1) & np.all(
        candidates <= log_max + 1.0e-14, axis=1
    )
    candidates = candidates[admissible]
    if len(candidates) < n_candidates:
        raise RuntimeError(
            f"Only {len(candidates)} admissible complete-F candidates were generated; "
            f"requested at least {n_candidates}."
        )
    candidates = candidates[:n_candidates]
    candidates = np.sort(candidates, axis=1)[:, ::-1]

    # Exact boundary of the trace-free log-strain hexagon.  Its six vertices are the
    # permutations of (log_max, 0, log_min) for reciprocal bounds; the general formula
    # below remains valid for any bounds that contain identity.
    raw_vertices = np.array([
        [log_max, -log_max-log_min, log_min],
        [log_max, log_min, -log_max-log_min],
        [-log_max-log_min, log_min, log_max],
        [log_min, -log_max-log_min, log_max],
        [log_min, log_max, -log_max-log_min],
        [-log_max-log_min, log_max, log_min],
    ], dtype=np.float64)
    if not np.all((raw_vertices >= log_min - 1.0e-12) & (raw_vertices <= log_max + 1.0e-12)):
        # For non-reciprocal bounds, construct boundary numerically from admissible square
        # intersections instead of relying on the reciprocal-bound closed form.
        verts = []
        for a in (log_min, log_max):
            for b in (log_min, log_max):
                c = -a-b
                if log_min - 1.0e-12 <= c <= log_max + 1.0e-12:
                    verts.append([a,b,c])
        for fixed in (log_min, log_max):
            for other in (log_min, log_max):
                third = -fixed-other
                if log_min - 1.0e-12 <= third <= log_max + 1.0e-12:
                    verts.extend([[fixed,third,other],[third,fixed,other],[other,fixed,third]])
        raw_vertices = np.unique(np.round(np.asarray(verts,dtype=np.float64),14),axis=0)
        if len(raw_vertices) < 3:
            raise RuntimeError("Unable to construct complete-F parent boundary.")
        # Order polygon vertices in the trace-free plane.
        b1=np.array([1.0,-1.0,0.0])/math.sqrt(2.0)
        b2=np.array([1.0,1.0,-2.0])/math.sqrt(6.0)
        ang=np.arctan2(raw_vertices@b2,raw_vertices@b1)
        raw_vertices=raw_vertices[np.argsort(ang)]

    n_edges = len(raw_vertices)
    per_edge = max(2, int(math.ceil(cfg.parent_f_boundary_points / n_edges)))
    boundary_parts=[]
    for i in range(n_edges):
        a=raw_vertices[i]
        b=raw_vertices[(i+1)%n_edges]
        t=np.linspace(0.0,1.0,per_edge,endpoint=False)[:,None]
        boundary_parts.append((1.0-t)*a[None,:]+t*b[None,:])
    boundary=np.vstack(boundary_parts)[: int(cfg.parent_f_boundary_points)]
    boundary=np.sort(boundary,axis=1)[:,::-1]

    identity = np.zeros((1, 3), dtype=np.float64)
    all_rows = np.vstack([anchors, identity, boundary, candidates])
    all_origins = np.concatenate([
        anchor_origins,
        np.array(["IDENTITY"], dtype=object),
        np.array(["BOUNDARY"] * len(boundary), dtype=object),
        np.array(["SOBOL_INTERIOR"] * len(candidates), dtype=object),
    ])

    rounded = np.round(all_rows, 13)
    _, unique_idx = np.unique(rounded, axis=0, return_index=True)
    unique_idx = np.sort(unique_idx)
    rows = all_rows[unique_idx]
    origins = all_origins[unique_idx]

    mandatory = np.where(origins != "SOBOL_INTERIOR")[0].tolist()
    target_n = max(int(cfg.parent_f_states), len(mandatory))
    selected = list(mandatory)
    chosen = np.zeros(len(rows), dtype=bool)
    chosen[selected] = True
    min_d2 = np.full(len(rows), np.inf, dtype=np.float64)
    for idx in selected:
        min_d2 = np.minimum(min_d2, np.sum((rows - rows[idx])**2, axis=1))
    while len(selected) < target_n:
        score = min_d2.copy()
        score[chosen] = -np.inf
        idx = int(np.argmax(score))
        selected.append(idx)
        chosen[idx] = True
        min_d2 = np.minimum(min_d2, np.sum((rows - rows[idx])**2, axis=1))

    selected = np.asarray(selected, dtype=int)
    states = rows[selected]
    selected_origins = origins[selected]
    stretches = np.exp(states)
    if np.min(stretches) < lam_min - 1.0e-12 or np.max(stretches) > lam_max + 1.0e-12:
        raise RuntimeError("Complete-F sampler retained a state outside its principal-stretch bounds.")
    order = np.lexsort((states[:, 2], states[:, 1], states[:, 0], np.linalg.norm(states, axis=1)))
    return states[order], selected_origins[order]


def make_protocol(cfg: Config) -> Protocol:
    n = cfg.n_per_mode
    ut = np.linspace(cfg.ut_min, cfg.ut_max, n, dtype=np.float64)
    bt = np.linspace(cfg.bt_min, cfg.bt_max, n, dtype=np.float64)
    sh = np.linspace(cfg.sh_min, cfg.sh_max, n, dtype=np.float64)
    parent_logs, parent_origins = _complete_f_parent_states(cfg, ut, bt, sh)
    parent_stretches = np.exp(parent_logs)

    n_parent_components = 2 * len(parent_logs)
    slices = {
        "F_PARENT": slice(0, n_parent_components),
        "UT": slice(n_parent_components, n_parent_components + n),
        "BT": slice(n_parent_components + n, n_parent_components + 2*n),
        "SH": slice(n_parent_components + 2*n, n_parent_components + 3*n),
    }
    names = np.array(
        ["F_PARENT"] * n_parent_components + ["UT"] * n + ["BT"] * n + ["SH"] * n
    )
    return Protocol(
        parent_log_principal=parent_logs,
        parent_principal_stretches=parent_stretches,
        parent_state_origin=parent_origins,
        ut=ut, bt=bt, sh=sh, mode_slices=slices, mode_names=names,
    )


def constitutive_bases(protocol: Protocol) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    lam = protocol.parent_principal_stretches
    l1, l2, l3 = lam[:, 0], lam[:, 1], lam[:, 2]
    parent_w1 = np.column_stack([
        2.0 * (l1**2 - l3**2),
        2.0 * (l2**2 - l3**2),
    ]).reshape(-1)
    parent_w2 = np.column_stack([
        2.0 * (l3**-2.0 - l1**-2.0),
        2.0 * (l3**-2.0 - l2**-2.0),
    ]).reshape(-1)
    parent_i1m3_state = np.sum(lam**2, axis=1) - 3.0
    parent_i2m3_state = np.sum(lam**-2.0, axis=1) - 3.0
    parent_i1m3 = np.repeat(parent_i1m3_state, 2)
    parent_i2m3 = np.repeat(parent_i2m3_state, 2)

    lam_ut = protocol.ut
    ut_w1 = 2.0 * (lam_ut**2 - lam_ut**(-1.0))
    ut_w2 = 2.0 * (lam_ut - lam_ut**(-2.0))
    ut_i1m3 = lam_ut**2 + 2.0 / lam_ut - 3.0
    ut_i2m3 = lam_ut**(-2.0) + 2.0 * lam_ut - 3.0

    lam_bt = protocol.bt
    bt_w1 = 2.0 * (lam_bt**2 - lam_bt**(-4.0))
    bt_w2 = 2.0 * (lam_bt**4 - lam_bt**(-2.0))
    bt_i1m3 = 2.0 * lam_bt**2 + lam_bt**(-4.0) - 3.0
    bt_i2m3 = 2.0 * lam_bt**(-2.0) + lam_bt**4 - 3.0

    gam = protocol.sh
    sh_w1 = 2.0 * gam
    sh_w2 = 2.0 * gam
    sh_i1m3 = gam**2
    sh_i2m3 = gam**2

    return (
        np.concatenate([parent_w1, ut_w1, bt_w1, sh_w1]).astype(np.float64),
        np.concatenate([parent_w2, ut_w2, bt_w2, sh_w2]).astype(np.float64),
        np.concatenate([parent_i1m3, ut_i1m3, bt_i1m3, sh_i1m3]).astype(np.float64),
        np.concatenate([parent_i2m3, ut_i2m3, bt_i2m3, sh_i2m3]).astype(np.float64),
    )


def scalar_bounds(cfg: Config, model: str) -> Tuple[float, float]:
    if model == "MR":
        return cfg.mr_rho_min, cfg.mr_rho_max
    if model == "YEOH2":
        return cfg.yeoh_beta_min, cfg.yeoh_beta_max
    if model == "GENT":
        return cfg.gent_gamma_min, cfg.gent_gamma_max
    if model == "OGDEN1":
        return cfg.ogden_alpha_min, cfg.ogden_alpha_max
    raise KeyError(model)


def scalar_nested_coordinate(model: str) -> float:
    if model in ("MR", "YEOH2", "GENT"):
        return 0.0
    if model == "OGDEN1":
        return 2.0
    raise KeyError(model)


def _ogden_power_difference_over_alpha(
    alpha: float,
    log_a: np.ndarray,
    log_b: np.ndarray,
    series_threshold: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return 4*(a^alpha-b^alpha)/alpha and its derivative d/dalpha.

    Here ``log_a`` and ``log_b`` contain logarithms of positive stretches.
    The alpha=0 singularity is removable under incompressibility and is
    evaluated with an analytic Taylor series.
    """
    alpha = float(alpha)
    a = np.asarray(log_a, dtype=np.float64)
    b = np.asarray(log_b, dtype=np.float64)
    if abs(alpha) <= float(series_threshold):
        a2, b2 = a * a, b * b
        a3, b3 = a2 * a, b2 * b
        a4, b4 = a3 * a, b3 * b
        a5, b5 = a4 * a, b4 * b
        value = 4.0 * (
            (a - b)
            + 0.5 * alpha * (a2 - b2)
            + (alpha**2 / 6.0) * (a3 - b3)
            + (alpha**3 / 24.0) * (a4 - b4)
            + (alpha**4 / 120.0) * (a5 - b5)
        )
        derivative = 4.0 * (
            0.5 * (a2 - b2)
            + (alpha / 3.0) * (a3 - b3)
            + (alpha**2 / 8.0) * (a4 - b4)
            + (alpha**3 / 30.0) * (a5 - b5)
        )
        return value, derivative

    ea = np.exp(alpha * a)
    eb = np.exp(alpha * b)
    difference = ea - eb
    derivative_numerator = alpha * (a * ea - b * eb) - difference
    return 4.0 * difference / alpha, 4.0 * derivative_numerator / (alpha * alpha)


def ogden1_template_and_derivative(
    alpha: float,
    protocol: Protocol,
    series_threshold: float = 1.0e-5,
) -> Tuple[np.ndarray, np.ndarray]:
    """Unit-total-modulus signed one-term incompressible Ogden response."""
    alpha = float(alpha)

    logs = protocol.parent_log_principal
    f13, df13 = _ogden_power_difference_over_alpha(
        alpha, logs[:, 0], logs[:, 2], series_threshold
    )
    f23, df23 = _ogden_power_difference_over_alpha(
        alpha, logs[:, 1], logs[:, 2], series_threshold
    )
    parent = np.column_stack([f13, f23]).reshape(-1)
    dparent = np.column_stack([df13, df23]).reshape(-1)

    lam = protocol.ut
    if lam.size:
        log_lam = np.log(lam)
        ut, dut = _ogden_power_difference_over_alpha(
            alpha, log_lam, -0.5 * log_lam, series_threshold
        )
    else:
        ut = dut = np.empty(0, dtype=np.float64)

    lam = protocol.bt
    if lam.size:
        log_lam = np.log(lam)
        bt, dbt = _ogden_power_difference_over_alpha(
            alpha, log_lam, -2.0 * log_lam, series_threshold
        )
    else:
        bt = dbt = np.empty(0, dtype=np.float64)

    gam = protocol.sh
    if gam.size:
        root = np.sqrt(1.0 + 0.25 * gam**2)
        l1 = root + 0.5 * gam
        l2 = root - 0.5 * gam
        denom = np.sqrt(gam**2 + 4.0)
        sh_raw, dsh_raw = _ogden_power_difference_over_alpha(
            alpha, np.log(l1), np.log(l2), series_threshold
        )
        sh = sh_raw / denom
        dsh = dsh_raw / denom
    else:
        sh = dsh = np.empty(0, dtype=np.float64)

    return (
        np.concatenate([parent, ut, bt, sh]),
        np.concatenate([dparent, dut, dbt, dsh]),
    )


def scalar_template_and_jacobian(
    cfg: Config,
    model: str,
    coordinate: float,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    b1, b2, i1m3, _i2m3 = bases
    if model == "NH":
        return b1.copy(), np.zeros((b1.size, 0), dtype=np.float64)
    if model == "MR":
        f = (1.0 - coordinate) * b1 + coordinate * b2
        return f, (b2 - b1)[:, None]
    if model == "YEOH2":
        f = b1 * (1.0 + 2.0 * coordinate * i1m3)
        return f, (2.0 * i1m3 * b1)[:, None]
    if model == "GENT":
        denominator = 1.0 - float(coordinate) * i1m3
        if np.any(denominator <= cfg.gent_singularity_margin):
            raise ValueError("Gent coordinate enters the locking singularity over the protocol.")
        f = b1 / denominator
        df = b1 * i1m3 / (denominator**2)
        return f, df[:, None]
    if model == "OGDEN1":
        f, df = ogden1_template_and_derivative(
            coordinate, protocol, cfg.ogden_alpha_series_threshold
        )
        return f, df[:, None]
    raise KeyError(model)


def gp2_map_q_to_physical(cfg: Config, q: Sequence[float]) -> Dict[str, float]:
    q = np.clip(np.asarray(q, dtype=float), 0.0, 1.0)
    if q.size != 4:
        raise ValueError("GP2 requires four shape coordinates.")
    ranges = (
        (cfg.gp2_rho_min, cfg.gp2_rho_max),
        (cfg.gp2_beta20_min, cfg.gp2_beta20_max),
        (cfg.gp2_beta11_min, cfg.gp2_beta11_max),
        (cfg.gp2_beta02_min, cfg.gp2_beta02_max),
    )
    values = [lo + qi * (hi - lo) for qi, (lo, hi) in zip(q, ranges)]
    return {
        "q_rho": float(q[0]),
        "q_beta20": float(q[1]),
        "q_beta11": float(q[2]),
        "q_beta02": float(q[3]),
        "rho": float(values[0]),
        "beta20": float(values[1]),
        "beta11": float(values[2]),
        "beta02": float(values[3]),
    }


def gp2_map_physical_to_q(
    cfg: Config,
    rho: float,
    beta20: float,
    beta11: float,
    beta02: float,
) -> np.ndarray:
    values = (rho, beta20, beta11, beta02)
    ranges = (
        (cfg.gp2_rho_min, cfg.gp2_rho_max),
        (cfg.gp2_beta20_min, cfg.gp2_beta20_max),
        (cfg.gp2_beta11_min, cfg.gp2_beta11_max),
        (cfg.gp2_beta02_min, cfg.gp2_beta02_max),
    )
    q = [(float(v) - lo) / max(hi - lo, 1.0e-30) for v, (lo, hi) in zip(values, ranges)]
    return np.clip(np.asarray(q, dtype=np.float64), 0.0, 1.0)


def gp2_template_physical(
    rho: float,
    beta20: float,
    beta11: float,
    beta02: float,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> np.ndarray:
    b1, b2, x, y = bases
    w1 = (1.0 - float(rho)) + 2.0 * float(beta20) * x + float(beta11) * y
    w2 = float(rho) + float(beta11) * x + 2.0 * float(beta02) * y
    return w1 * b1 + w2 * b2


def gp2_template_and_jacobian(
    cfg: Config,
    q: Sequence[float],
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    p = gp2_map_q_to_physical(cfg, q)
    b1, b2, x, y = bases
    f = gp2_template_physical(p["rho"], p["beta20"], p["beta11"], p["beta02"], bases)
    drho = cfg.gp2_rho_max - cfg.gp2_rho_min
    db20 = cfg.gp2_beta20_max - cfg.gp2_beta20_min
    db11 = cfg.gp2_beta11_max - cfg.gp2_beta11_min
    db02 = cfg.gp2_beta02_max - cfg.gp2_beta02_min
    J = np.column_stack([
        drho * (b2 - b1),
        db20 * (2.0 * x * b1),
        db11 * (y * b1 + x * b2),
        db02 * (2.0 * y * b2),
    ])
    return f, J, p


def ogden2_map_q_to_physical(cfg: Config, q: Sequence[float]) -> Dict[str, float]:
    """Canonical rectangular q -> ordered signed OGDEN2 shape parameters.

    q = (center_fraction, gap_fraction, weight_fraction) in [0,1]^3 maps onto
    alpha1 < alpha2 and 0 < w < 1 over the configured signed exponent interval.
    The exact OGDEN1 boundaries are handled by hierarchy closure.  The exact MR
    stratum alpha1=-2, alpha2=+2, w=rho lies in the interior.
    """
    qc, qg, qw = np.clip(np.asarray(q, dtype=float), 0.0, 1.0)
    amin = float(cfg.ogden_alpha_min)
    amax = float(cfg.ogden_alpha_max)
    arange = amax - amin
    if not (amin <= -2.0 and amax >= 2.0 and amin < amax):
        raise ValueError("Signed OGDEN2 range must contain -2 and +2.")
    gmin = min(max(cfg.ogden2_alpha_gap_epsilon, 1.0e-10), 0.5 * arange)
    gap = gmin + qg * (arange - gmin)
    available = max(arange - gap, 0.0)
    alpha1 = amin + qc * available
    alpha2 = alpha1 + gap
    eps = min(max(cfg.ogden2_weight_epsilon, 1.0e-8), 0.1)
    w = eps + qw * (1.0 - 2.0 * eps)
    return {
        "q_center": float(qc),
        "q_gap": float(qg),
        "q_weight": float(qw),
        "alpha1": float(alpha1),
        "alpha2": float(alpha2),
        "weight1": float(w),
        "weight2": float(1.0 - w),
        "alpha_gap": float(gap),
    }


def ogden2_map_physical_to_q(
    cfg: Config,
    alpha1: float,
    alpha2: float,
    weight1: float,
) -> np.ndarray:
    """Inverse of the canonical interior map for ordered physical parameters."""
    a1, a2 = sorted((float(alpha1), float(alpha2)))
    w = float(weight1) if float(alpha1) <= float(alpha2) else 1.0 - float(weight1)
    amin = float(cfg.ogden_alpha_min)
    amax = float(cfg.ogden_alpha_max)
    arange = amax - amin
    gmin = min(max(cfg.ogden2_alpha_gap_epsilon, 1.0e-10), 0.5 * arange)
    gap = float(np.clip(a2 - a1, gmin, arange))
    qg = (gap - gmin) / max(arange - gmin, 1.0e-30)
    available = max(arange - gap, 0.0)
    qc = 0.5 if available <= 1.0e-14 else (a1 - amin) / available
    eps = min(max(cfg.ogden2_weight_epsilon, 1.0e-8), 0.1)
    qw = (w - eps) / max(1.0 - 2.0 * eps, 1.0e-30)
    return np.clip(np.array([qc, qg, qw], dtype=np.float64), 0.0, 1.0)


def ogden2_template_physical(
    alpha1: float,
    alpha2: float,
    weight1: float,
    protocol: Protocol,
    series_threshold: float = 1.0e-5,
) -> np.ndarray:
    f1 = ogden1_template_and_derivative(alpha1, protocol, series_threshold)[0]
    f2 = ogden1_template_and_derivative(alpha2, protocol, series_threshold)[0]
    w = float(weight1)
    return w * f1 + (1.0 - w) * f2


def ogden2_template_and_jacobian(
    cfg: Config,
    q: Sequence[float],
    protocol: Protocol,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    qc, qg, qw = np.clip(np.asarray(q, dtype=float), 0.0, 1.0)
    p = ogden2_map_q_to_physical(cfg, (qc, qg, qw))
    a1, a2, w = p["alpha1"], p["alpha2"], p["weight1"]
    f1, d1 = ogden1_template_and_derivative(
        a1, protocol, cfg.ogden_alpha_series_threshold
    )
    f2, d2 = ogden1_template_and_derivative(
        a2, protocol, cfg.ogden_alpha_series_threshold
    )
    f = w * f1 + (1.0 - w) * f2

    arange = float(cfg.ogden_alpha_max - cfg.ogden_alpha_min)
    gmin = min(max(cfg.ogden2_alpha_gap_epsilon, 1.0e-10), 0.5 * arange)
    dg_dqg = arange - gmin
    gap = p["alpha_gap"]
    available = arange - gap
    da1_dqc = available
    da2_dqc = available
    da1_dqg = -qc * dg_dqg
    da2_dqg = (1.0 - qc) * dg_dqg
    eps = min(max(cfg.ogden2_weight_epsilon, 1.0e-8), 0.1)
    dw_dqw = 1.0 - 2.0 * eps

    df_dqc = w * d1 * da1_dqc + (1.0 - w) * d2 * da2_dqc
    df_dqg = w * d1 * da1_dqg + (1.0 - w) * d2 * da2_dqg
    df_dqw = dw_dqw * (f1 - f2)
    J = np.column_stack([df_dqc, df_dqg, df_dqw])
    return f, J, p


def model_template_and_jacobian(
    cfg: Config,
    model: str,
    coordinate: Sequence[float],
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    if model == "OGDEN2":
        return ogden2_template_and_jacobian(cfg, coordinate, protocol)
    if model == "GP2":
        return gp2_template_and_jacobian(cfg, coordinate, bases)
    x = float(np.asarray(coordinate, dtype=float).reshape(-1)[0]) if model != "NH" else 0.0
    f, J = scalar_template_and_jacobian(cfg, model, x, protocol, bases)
    physical: Dict[str, float] = {"coordinate": x}
    if model == "OGDEN1":
        physical["alpha"] = x
    elif model == "MR":
        physical["rho"] = x
    elif model == "YEOH2":
        physical["beta"] = x
    elif model == "GENT":
        physical["gamma"] = x
        physical["Jm"] = float("inf") if x == 0.0 else 1.0 / x
    return f, J, physical


def model_template(
    cfg: Config,
    model: str,
    coordinate: Sequence[float],
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> np.ndarray:
    return model_template_and_jacobian(cfg, model, coordinate, protocol, bases)[0]


def scale_free_response(f: np.ndarray) -> np.ndarray:
    f = np.asarray(f, dtype=np.float64)
    return f / max(float(np.linalg.norm(f)), 1.0e-14)


# =============================================================================
# Bounded brush and Fisher geometry
# =============================================================================


def brush_shape_per_unit_noise(cfg: Config, protocol: Protocol, center: np.ndarray) -> np.ndarray:
    center = np.asarray(center, dtype=np.float64)
    h = np.empty_like(center)

    psl = protocol.parent_slice
    parent = center[psl].reshape(protocol.n_parent_states, 2)
    parent_peak = np.maximum(np.max(np.abs(parent), axis=1), 1.0e-14)
    h[psl] = (np.abs(parent) + cfg.mode_floor_per_unit_noise * parent_peak[:, None]).reshape(-1)

    for name in ("UT", "BT", "SH"):
        sl = protocol.mode_slices[name]
        peak = max(float(np.max(np.abs(center[sl]))), 1.0e-14)
        h[sl] = np.abs(center[sl]) + cfg.mode_floor_per_unit_noise * peak
    return np.maximum(h, 1.0e-14)


def brush_shape_per_unit_noise_batch(cfg: Config, protocol: Protocol, centers: np.ndarray) -> np.ndarray:
    centers = np.asarray(centers, dtype=np.float64)
    h = np.empty_like(centers)

    psl = protocol.parent_slice
    parent = centers[:, psl].reshape(len(centers), protocol.n_parent_states, 2)
    parent_peak = np.maximum(np.max(np.abs(parent), axis=2), 1.0e-14)
    h[:, psl] = (
        np.abs(parent) + cfg.mode_floor_per_unit_noise * parent_peak[:, :, None]
    ).reshape(len(centers), -1)

    for name in ("UT", "BT", "SH"):
        sl = protocol.mode_slices[name]
        peak = np.maximum(np.max(np.abs(centers[:, sl]), axis=1), 1.0e-14)
        h[:, sl] = np.abs(centers[:, sl]) + cfg.mode_floor_per_unit_noise * peak[:, None]
    return np.maximum(h, 1.0e-14)


def required_noise_fraction(
    cfg: Config,
    protocol: Protocol,
    source: np.ndarray,
    target: np.ndarray,
) -> float:
    idx = protocol.atlas_indices
    hs = brush_shape_per_unit_noise(cfg, protocol, source)[idx]
    ht = brush_shape_per_unit_noise(cfg, protocol, target)[idx]
    return float(np.max(np.abs(source[idx] - target[idx]) / np.maximum(hs + ht, 1.0e-30)))


def observation_model_score(
    cfg: Config,
    protocol: Protocol,
    observation: np.ndarray,
    center: np.ndarray,
) -> float:
    idx = protocol.atlas_indices
    delta = cfg.final_noise_fraction * brush_shape_per_unit_noise(cfg, protocol, center)[idx]
    return float(np.max(np.abs(observation[idx] - center[idx]) / np.maximum(delta, 1.0e-30)))


def profiled_fisher_metric(
    cfg: Config,
    protocol: Protocol,
    f: np.ndarray,
    Jshape: np.ndarray,
) -> Dict[str, object]:
    idx = protocol.atlas_indices
    f = np.asarray(f, dtype=np.float64)[idx]
    Jshape = np.asarray(Jshape, dtype=np.float64)[idx]
    parent = f.reshape(protocol.n_parent_states, 2)
    parent_peak = np.maximum(np.max(np.abs(parent), axis=1), 1.0e-14)
    h = (np.abs(parent) + cfg.mode_floor_per_unit_noise * parent_peak[:, None]).reshape(-1)
    sigma = np.maximum(cfg.final_noise_fraction * h, 1.0e-14)
    w = 1.0 / sigma**2
    js = f[:, None]
    J = np.column_stack([js, Jshape])
    G = J.T @ (w[:, None] * J)
    gss = float(G[0, 0])
    gs = G[1:, 0]
    Gshape = G[1:, 1:]
    if Gshape.size:
        Gprofiled = Gshape - np.outer(gs, gs) / max(gss, 1.0e-30)
        Gprofiled = 0.5 * (Gprofiled + Gprofiled.T)
        eig = np.linalg.eigvalsh(Gprofiled)
        eig = np.maximum(eig, 0.0)
        maxeig = max(float(np.max(eig)), 1.0e-30)
        floor = cfg.multidim_metric_eigen_floor_relative * maxeig
        positive = eig[eig > floor]
        rank = int(positive.size)
        cond = float(np.max(positive) / np.min(positive)) if positive.size >= 2 else float("inf")
        raw_volume = float(math.sqrt(max(float(np.prod(eig)), 0.0))) if eig.size else 1.0
    else:
        Gprofiled = np.empty((0, 0), dtype=float)
        eig = np.empty(0, dtype=float)
        rank = 0
        cond = float("nan")
        raw_volume = 1.0
    return {
        "G_full": G,
        "G_profiled": Gprofiled,
        "profiled_eigenvalues": eig,
        "profiled_rank": rank,
        "profiled_condition_number": cond,
        "fisher_volume_density": raw_volume,
        "g_scale_scale": gss,
    }



class ScalarFisherSampler:
    def __init__(
        self,
        cfg: Config,
        protocol: Protocol,
        bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    ) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self.pilots: Dict[str, pd.DataFrame] = {}
        for model in SCALAR_PARENTS:
            self.pilots[model] = self._build(model)

    def _build(self, model: str) -> pd.DataFrame:
        lo, hi = scalar_bounds(self.cfg, model)
        x = np.linspace(lo, hi, self.cfg.scalar_fisher_pilot_points)
        rows: List[Dict[str, object]] = []
        density = np.empty_like(x)
        for i, value in enumerate(x):
            f, J, physical = model_template_and_jacobian(
                self.cfg, model, (float(value),), self.protocol, self.bases
            )
            metric = profiled_fisher_metric(self.cfg, self.protocol, f, J)
            eig = np.asarray(metric["profiled_eigenvalues"], dtype=float)
            g = float(eig[0]) if eig.size else 0.0
            density[i] = math.sqrt(max(g, 0.0))
            row = {
                "model": model,
                "coordinate": float(value),
                "sqrt_profiled_fisher_metric": density[i],
                "profiled_fisher_metric": g,
                "condition_number": metric["profiled_condition_number"],
            }
            row.update({f"physical_{k}": v for k, v in physical.items()})
            rows.append(row)
        arc = float(np.trapezoid(density, x))
        uniform_ref = arc / max(hi - lo, 1.0e-14)
        mix = float(np.clip(self.cfg.fisher_uniform_mixture, 0.0, 1.0))
        regularized = (1.0 - mix) * density + mix * max(uniform_ref, 1.0e-12)
        cumulative = np.zeros_like(x)
        cumulative[1:] = np.cumsum(0.5 * (regularized[1:] + regularized[:-1]) * np.diff(x))
        if cumulative[-1] <= 0:
            cumulative = (x - lo) / max(hi - lo, 1.0e-14)
        else:
            cumulative /= cumulative[-1]
        table = pd.DataFrame(rows)
        table["regularized_sampling_density"] = regularized
        table["fisher_cumulative_coordinate"] = cumulative
        table["raw_fisher_arc_length"] = arc
        return table

    def grid(self, model: str, n_points: int) -> np.ndarray:
        """Fisher-arc grid with exact hierarchy and literature anchors forced in."""
        pilot = self.pilots[model]
        x = pilot["coordinate"].to_numpy(float)
        cdf = pilot["fisher_cumulative_coordinate"].to_numpy(float)
        lo, hi = scalar_bounds(self.cfg, model)
        mandatory = [lo, hi, scalar_nested_coordinate(model)]
        mandatory.extend(
            float(v) for v in LITERATURE_SCALAR_ANCHORS.get(model, tuple())
            if lo <= float(v) <= hi
        )
        mandatory = sorted(set(np.round(mandatory, 14)))
        if len(mandatory) > n_points:
            raise ValueError(f"{model}: more mandatory anchors than requested nodes.")

        candidate_targets = np.linspace(0.0, 1.0, max(4 * n_points, n_points + 1))
        candidate_nodes = np.interp(candidate_targets, cdf, x)
        candidates = np.unique(np.round(np.concatenate([candidate_nodes, mandatory]), 14))
        candidate_cdf = np.interp(candidates, x, cdf)

        selected: List[int] = []
        for value in mandatory:
            idx = int(np.argmin(np.abs(candidates - value)))
            if idx not in selected:
                selected.append(idx)
        min_distance = np.full(len(candidates), np.inf, dtype=float)
        chosen = np.zeros(len(candidates), dtype=bool)
        for idx in selected:
            chosen[idx] = True
            min_distance = np.minimum(min_distance, np.abs(candidate_cdf - candidate_cdf[idx]))
        while len(selected) < n_points:
            score = min_distance.copy()
            score[chosen] = -np.inf
            idx = int(np.argmax(score))
            selected.append(idx)
            chosen[idx] = True
            min_distance = np.minimum(min_distance, np.abs(candidate_cdf - candidate_cdf[idx]))
        nodes = np.sort(candidates[np.asarray(selected, dtype=int)])
        if len(nodes) != n_points or np.any(np.diff(nodes) <= 0):
            raise RuntimeError(f"{model}: failed to construct a strictly increasing anchored Fisher grid.")
        return nodes

    def pilot_table(self) -> pd.DataFrame:
        return pd.concat(list(self.pilots.values()), ignore_index=True)


class Ogden2FisherSampler:
    """Fisher-volume-weighted maximin sampler of the 3-D OGDEN2 shape manifold."""

    def __init__(
        self,
        cfg: Config,
        protocol: Protocol,
        bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    ) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self._pilot_q = self._sobol_candidates(cfg.ogden2_pilot_candidates)
        self._pilot_table = self._evaluate_pilot(self._pilot_q)

    def _forced_anchor_q(self) -> np.ndarray:
        anchors: List[np.ndarray] = []
        # Exact embedded MR stratum: alpha1=-2, alpha2=+2, w=rho.
        for rho in (0.01, 0.10, 0.40, 0.70, 0.90, 0.99):
            anchors.append(ogden2_map_physical_to_q(self.cfg, -2.0, 2.0, rho))
        # Near OGDEN1 closure boundaries for representative signed exponents.
        eps = min(max(self.cfg.ogden2_weight_epsilon, 1.0e-8), 0.1)
        for alpha in (-25.0, -19.0, -10.0, -2.0, 0.0, 2.0, 8.0, 16.55, 25.0):
            other = alpha + 1.0 if alpha < self.cfg.ogden_alpha_max - 1.0 else alpha - 1.0
            a1, a2 = sorted((alpha, other))
            w_alpha = 1.0 - eps if alpha == a1 else eps
            anchors.append(ogden2_map_physical_to_q(self.cfg, a1, a2, w_alpha))
        # Broad interior coverage anchors.
        anchors.extend([
            np.array([0.10, 0.15, 0.15]),
            np.array([0.90, 0.15, 0.85]),
            np.array([0.15, 0.75, 0.50]),
            np.array([0.85, 0.75, 0.50]),
            np.array([0.50, 0.35, 0.10]),
            np.array([0.50, 0.35, 0.90]),
            np.array([0.50, 0.90, 0.50]),
        ])
        return np.unique(np.round(np.clip(np.stack(anchors), 0.0, 1.0), 14), axis=0)

    def _sobol_candidates(self, n: int) -> np.ndarray:
        m = int(math.ceil(math.log2(max(n, 2))))
        engine = qmc.Sobol(d=3, scramble=True, seed=self.cfg.seed + 101)
        q = np.asarray(engine.random_base2(m)[:n], dtype=np.float64)
        return np.vstack([q, self._forced_anchor_q()])

    def _evaluate_pilot(self, q_values: np.ndarray) -> pd.DataFrame:
        rows: List[Dict[str, object]] = []
        raw = np.empty(q_values.shape[0], dtype=float)
        for i, q in enumerate(q_values):
            f, J, physical = ogden2_template_and_jacobian(self.cfg, q, self.protocol)
            metric = profiled_fisher_metric(self.cfg, self.protocol, f, J)
            raw[i] = float(metric["fisher_volume_density"])
            eig = np.asarray(metric["profiled_eigenvalues"], dtype=float)
            rows.append({
                "pilot_index": i,
                "q_center": float(q[0]),
                "q_gap": float(q[1]),
                "q_weight": float(q[2]),
                "fisher_volume_density": raw[i],
                "profiled_rank": int(metric["profiled_rank"]),
                "profiled_condition_number": float(metric["profiled_condition_number"]),
                "profiled_min_eigenvalue": float(eig[0]) if eig.size else 0.0,
                "profiled_max_eigenvalue": float(eig[-1]) if eig.size else 0.0,
                **physical,
            })
        finite = raw[np.isfinite(raw) & (raw > 0)]
        ref = float(np.median(finite)) if finite.size else 1.0
        normalized = np.nan_to_num(raw / max(ref, 1.0e-30), nan=0.0, posinf=0.0, neginf=0.0)
        mix = float(np.clip(self.cfg.fisher_uniform_mixture, 0.0, 1.0))
        regularized = (1.0 - mix) * normalized + mix
        table = pd.DataFrame(rows)
        table["normalized_fisher_volume_density"] = normalized
        table["regularized_sampling_weight"] = np.maximum(regularized, 1.0e-12)
        return table

    def _anchor_indices(self, pool_q: np.ndarray) -> List[int]:
        result: List[int] = []
        for anchor in self._forced_anchor_q():
            idx = int(np.argmin(np.sum((pool_q - anchor[None, :]) ** 2, axis=1)))
            if idx not in result:
                result.append(idx)
        return result

    def sample(self, n_points: int, seed_offset: int = 0) -> pd.DataFrame:
        pilot = self._pilot_table
        q_all = pilot[["q_center", "q_gap", "q_weight"]].to_numpy(float)
        weights = pilot["regularized_sampling_weight"].to_numpy(float)
        rng = np.random.default_rng(self.cfg.seed + 200 + seed_offset)
        pool_n = min(len(pilot), max(n_points, self.cfg.ogden2_candidate_pool_factor * n_points))
        probabilities = weights / np.sum(weights)
        forced_q = self._forced_anchor_q()
        forced_idx = np.unique([
            int(np.argmin(np.sum((q_all - anchor[None, :]) ** 2, axis=1)))
            for anchor in forced_q
        ])
        remaining = np.setdiff1d(np.arange(len(pilot), dtype=int), forced_idx, assume_unique=False)
        random_n = max(0, pool_n - len(forced_idx))
        if random_n:
            p_remaining = probabilities[remaining]
            p_remaining = p_remaining / np.sum(p_remaining)
            random_idx = rng.choice(remaining, size=random_n, replace=False, p=p_remaining)
            pool_idx = np.concatenate([forced_idx, random_idx])
        else:
            pool_idx = forced_idx[:pool_n]
        pool_q = q_all[pool_idx]
        pool_w = weights[pool_idx]
        wnorm = pool_w / max(float(np.median(pool_w)), 1.0e-30)

        selected = self._anchor_indices(pool_q)
        selected = selected[: min(len(selected), n_points)]
        min_d2 = np.full(pool_n, np.inf, dtype=float)
        chosen_mask = np.zeros(pool_n, dtype=bool)
        for idx in selected:
            chosen_mask[idx] = True
            min_d2 = np.minimum(min_d2, np.sum((pool_q - pool_q[idx]) ** 2, axis=1))
        while len(selected) < n_points:
            score = min_d2 * np.power(np.maximum(wnorm, 1.0e-12), self.cfg.ogden2_density_power)
            score[chosen_mask] = -np.inf
            idx = int(np.argmax(score))
            selected.append(idx)
            chosen_mask[idx] = True
            min_d2 = np.minimum(min_d2, np.sum((pool_q - pool_q[idx]) ** 2, axis=1))

        chosen_pilot_idx = pool_idx[np.asarray(selected, dtype=int)]
        table = pilot.iloc[chosen_pilot_idx].copy().reset_index(drop=True)
        table.insert(0, "sample_index", np.arange(len(table), dtype=int))
        # Nearest-neighbor spacing in canonical q coordinates.
        q = table[["q_center", "q_gap", "q_weight"]].to_numpy(float)
        nn = np.full(len(q), np.inf)
        chunk = 256
        for i0 in range(0, len(q), chunk):
            i1 = min(len(q), i0 + chunk)
            d2 = np.sum((q[i0:i1, None, :] - q[None, :, :]) ** 2, axis=2)
            rows = np.arange(i0, i1)
            d2[np.arange(i1 - i0), rows] = np.inf
            nn[i0:i1] = np.sqrt(np.min(d2, axis=1))
        table["nearest_neighbor_q_distance"] = nn
        return table

    def pilot_table(self) -> pd.DataFrame:
        return self._pilot_table.copy()




class Gp2FisherSampler:
    """Fisher-volume-weighted maximin sampler of the 4-D GP2 shape manifold."""

    def __init__(
        self,
        cfg: Config,
        protocol: Protocol,
        bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    ) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self._pilot_q = self._sobol_candidates(cfg.gp2_pilot_candidates)
        self._pilot_table = self._evaluate_pilot(self._pilot_q)

    def _forced_anchor_q(self) -> np.ndarray:
        anchors: List[np.ndarray] = []
        # NH and exact MR stratum.
        anchors.append(gp2_map_physical_to_q(self.cfg, 0.0, 0.0, 0.0, 0.0))
        for rho in np.linspace(self.cfg.mr_rho_min, self.cfg.mr_rho_max, 9):
            anchors.append(gp2_map_physical_to_q(self.cfg, float(rho), 0.0, 0.0, 0.0))
        # Exact YEOH2 stratum.
        for beta in np.linspace(self.cfg.yeoh_beta_min, self.cfg.yeoh_beta_max, 9):
            anchors.append(gp2_map_physical_to_q(self.cfg, 0.0, float(beta), 0.0, 0.0))
        # Literature-reconstructed GP2 breast-tissue anchor.
        for rho, beta20, beta11, beta02 in GP2_LITERATURE_ANCHOR_PHYSICAL:
            anchors.append(gp2_map_physical_to_q(self.cfg, rho, beta20, beta11, beta02))
        # Broad interior anchors.
        anchors.extend([
            np.array([0.15, 0.25, 0.25, 0.25]),
            np.array([0.85, 0.25, 0.25, 0.25]),
            np.array([0.25, 0.85, 0.25, 0.25]),
            np.array([0.25, 0.25, 0.85, 0.25]),
            np.array([0.25, 0.25, 0.25, 0.85]),
            np.array([0.75, 0.75, 0.25, 0.25]),
            np.array([0.75, 0.25, 0.75, 0.25]),
            np.array([0.75, 0.25, 0.25, 0.75]),
            np.array([0.50, 0.50, 0.50, 0.50]),
            np.array([0.90, 0.90, 0.90, 0.90]),
        ])
        return np.unique(np.round(np.clip(np.stack(anchors), 0.0, 1.0), 14), axis=0)

    def _sobol_candidates(self, n: int) -> np.ndarray:
        m = int(math.ceil(math.log2(max(n, 2))))
        engine = qmc.Sobol(d=4, scramble=True, seed=self.cfg.seed + 401)
        q = np.asarray(engine.random_base2(m)[:n], dtype=np.float64)
        return np.vstack([q, self._forced_anchor_q()])

    def _evaluate_pilot(self, q_values: np.ndarray) -> pd.DataFrame:
        rows: List[Dict[str, object]] = []
        raw = np.empty(q_values.shape[0], dtype=float)
        for i, q in enumerate(q_values):
            f, J, physical = gp2_template_and_jacobian(self.cfg, q, self.bases)
            metric = profiled_fisher_metric(self.cfg, self.protocol, f, J)
            raw[i] = float(metric["fisher_volume_density"])
            eig = np.asarray(metric["profiled_eigenvalues"], dtype=float)
            rows.append({
                "pilot_index": i,
                "q_rho": float(q[0]),
                "q_beta20": float(q[1]),
                "q_beta11": float(q[2]),
                "q_beta02": float(q[3]),
                "fisher_volume_density": raw[i],
                "profiled_rank": int(metric["profiled_rank"]),
                "profiled_condition_number": float(metric["profiled_condition_number"]),
                "profiled_min_eigenvalue": float(eig[0]) if eig.size else 0.0,
                "profiled_max_eigenvalue": float(eig[-1]) if eig.size else 0.0,
                **physical,
            })
        finite = raw[np.isfinite(raw) & (raw > 0)]
        ref = float(np.median(finite)) if finite.size else 1.0
        normalized = np.nan_to_num(raw / max(ref, 1.0e-30), nan=0.0, posinf=0.0, neginf=0.0)
        mix = float(np.clip(self.cfg.fisher_uniform_mixture, 0.0, 1.0))
        regularized = (1.0 - mix) * normalized + mix
        table = pd.DataFrame(rows)
        table["normalized_fisher_volume_density"] = normalized
        table["regularized_sampling_weight"] = np.maximum(regularized, 1.0e-12)
        return table

    def sample(self, n_points: int, seed_offset: int = 0) -> pd.DataFrame:
        pilot = self._pilot_table
        columns = ["q_rho", "q_beta20", "q_beta11", "q_beta02"]
        q_all = pilot[columns].to_numpy(float)
        weights = pilot["regularized_sampling_weight"].to_numpy(float)
        rng = np.random.default_rng(self.cfg.seed + 500 + seed_offset)
        pool_n = min(len(pilot), max(n_points, self.cfg.gp2_candidate_pool_factor * n_points))
        probabilities = weights / np.sum(weights)
        anchors = self._forced_anchor_q()
        forced_idx = np.unique([
            int(np.argmin(np.sum((q_all - anchor[None, :]) ** 2, axis=1)))
            for anchor in anchors
        ])
        remaining = np.setdiff1d(np.arange(len(pilot), dtype=int), forced_idx, assume_unique=False)
        random_n = max(0, pool_n - len(forced_idx))
        if random_n:
            p = probabilities[remaining]
            p /= np.sum(p)
            random_idx = rng.choice(remaining, size=random_n, replace=False, p=p)
            pool_idx = np.concatenate([forced_idx, random_idx])
        else:
            pool_idx = forced_idx[:pool_n]
        pool_q = q_all[pool_idx]
        pool_w = weights[pool_idx]
        wnorm = pool_w / max(float(np.median(pool_w)), 1.0e-30)

        selected: List[int] = []
        for anchor in anchors:
            idx = int(np.argmin(np.sum((pool_q - anchor[None, :]) ** 2, axis=1)))
            if idx not in selected:
                selected.append(idx)
        selected = selected[: min(len(selected), n_points)]
        min_d2 = np.full(len(pool_q), np.inf, dtype=float)
        chosen = np.zeros(len(pool_q), dtype=bool)
        for idx in selected:
            chosen[idx] = True
            min_d2 = np.minimum(min_d2, np.sum((pool_q - pool_q[idx]) ** 2, axis=1))
        while len(selected) < n_points:
            score = min_d2 * np.power(np.maximum(wnorm, 1.0e-12), self.cfg.gp2_density_power)
            score[chosen] = -np.inf
            idx = int(np.argmax(score))
            selected.append(idx)
            chosen[idx] = True
            min_d2 = np.minimum(min_d2, np.sum((pool_q - pool_q[idx]) ** 2, axis=1))

        chosen_pilot_idx = pool_idx[np.asarray(selected, dtype=int)]
        table = pilot.iloc[chosen_pilot_idx].copy().reset_index(drop=True)
        table.insert(0, "sample_index", np.arange(len(table), dtype=int))
        q = table[columns].to_numpy(float)
        nn = np.full(len(q), np.inf)
        chunk = 192
        for i0 in range(0, len(q), chunk):
            i1 = min(len(q), i0 + chunk)
            d2 = np.sum((q[i0:i1, None, :] - q[None, :, :]) ** 2, axis=2)
            rows = np.arange(i0, i1)
            d2[np.arange(i1 - i0), rows] = np.inf
            nn[i0:i1] = np.sqrt(np.min(d2, axis=1))
        table["nearest_neighbor_q_distance"] = nn
        return table

    def pilot_table(self) -> pd.DataFrame:
        return self._pilot_table.copy()


# =============================================================================
# Target assets and continuous collision searches
# =============================================================================


@dataclass
class TargetAssets:
    coordinates: np.ndarray
    templates: np.ndarray
    brush_shapes: np.ndarray
    template_norms: np.ndarray


class AtlasSamplerRegistry:
    def __init__(self, cfg: Config, protocol: Protocol, bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self.scalar = ScalarFisherSampler(cfg, protocol, bases)
        self.ogden2 = Ogden2FisherSampler(cfg, protocol, bases)
        self.gp2 = Gp2FisherSampler(cfg, protocol, bases)
        self._ogden2_cache: Dict[Tuple[int, int], pd.DataFrame] = {}
        self._gp2_cache: Dict[Tuple[int, int], pd.DataFrame] = {}

    def scalar_grid(self, model: str, n: int) -> np.ndarray:
        return self.scalar.grid(model, n)

    def ogden2_nodes(self, n: int, seed_offset: int = 0) -> pd.DataFrame:
        key = (int(n), int(seed_offset))
        if key not in self._ogden2_cache:
            self._ogden2_cache[key] = self.ogden2.sample(n, seed_offset)
        return self._ogden2_cache[key].copy()

    def gp2_nodes(self, n: int, seed_offset: int = 0) -> pd.DataFrame:
        key = (int(n), int(seed_offset))
        if key not in self._gp2_cache:
            self._gp2_cache[key] = self.gp2.sample(n, seed_offset)
        return self._gp2_cache[key].copy()


class ContinuousModelSolver:
    def __init__(
        self,
        cfg: Config,
        protocol: Protocol,
        bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
        registry: AtlasSamplerRegistry,
        scalar_target_points: Optional[int] = None,
        ogden2_target_points: Optional[int] = None,
        gp2_target_points: Optional[int] = None,
        observation_mode: bool = False,
        seed_offset: int = 0,
    ) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self.registry = registry
        self.observation_mode = observation_mode
        self.scalar_target_points = int(scalar_target_points or cfg.scalar_target_points)
        self.ogden2_target_points = int(ogden2_target_points or cfg.ogden2_target_points)
        self.gp2_target_points = int(gp2_target_points or cfg.gp2_target_points)
        self.assets: Dict[str, TargetAssets] = {}
        for model in SCALAR_PARENTS:
            grid = registry.scalar_grid(model, self.scalar_target_points)
            templates = np.stack([model_template(cfg, model, (x,), protocol, bases) for x in grid])
            self.assets[model] = TargetAssets(
                coordinates=grid[:, None], templates=templates,
                brush_shapes=brush_shape_per_unit_noise_batch(cfg, protocol, templates),
                template_norms=np.linalg.norm(templates[:, protocol.atlas_indices], axis=1),
            )
        og2 = registry.ogden2_nodes(self.ogden2_target_points, seed_offset=seed_offset)
        q2 = og2[["q_center", "q_gap", "q_weight"]].to_numpy(float)
        templates = np.stack([model_template(cfg, "OGDEN2", row, protocol, bases) for row in q2])
        self.assets["OGDEN2"] = TargetAssets(
            coordinates=q2, templates=templates,
            brush_shapes=brush_shape_per_unit_noise_batch(cfg, protocol, templates),
            template_norms=np.linalg.norm(templates[:, protocol.atlas_indices], axis=1),
        )
        gp2 = registry.gp2_nodes(self.gp2_target_points, seed_offset=seed_offset)
        qg = gp2[["q_rho", "q_beta20", "q_beta11", "q_beta02"]].to_numpy(float)
        templates = np.stack([model_template(cfg, "GP2", row, protocol, bases) for row in qg])
        self.assets["GP2"] = TargetAssets(
            coordinates=qg, templates=templates,
            brush_shapes=brush_shape_per_unit_noise_batch(cfg, protocol, templates),
            template_norms=np.linalg.norm(templates[:, protocol.atlas_indices], axis=1),
        )
        self.nh = model_template(cfg, "NH", (0.0,), protocol, bases)


    def _scale_bounds(self) -> Tuple[float, float]:
        # Absolute response multiplier corresponding to the universal mu0 range.
        return (
            response_scale_from_mu0_kpa(self.cfg.mu0_min_kpa),
            response_scale_from_mu0_kpa(self.cfg.mu0_max_kpa),
        )

    def _score(self, source_or_obs: np.ndarray, target: np.ndarray) -> float:
        if self.observation_mode:
            return observation_model_score(self.cfg, self.protocol, source_or_obs, target)
        return required_noise_fraction(self.cfg, self.protocol, source_or_obs, target)

    def _coarse_scales_and_scores(self, source: np.ndarray, assets: TargetAssets) -> Tuple[np.ndarray, np.ndarray]:
        idx = self.protocol.atlas_indices
        source_parent = np.asarray(source, dtype=np.float64)[idx]
        templates = assets.templates[:, idx]
        brushes = assets.brush_shapes[:, idx]
        lo, hi = self._scale_bounds()
        denom = np.einsum("ij,ij->i", templates, templates)
        numer = templates @ source_parent
        scales_ls = np.divide(numer, denom, out=np.ones_like(numer), where=denom > 1.0e-20)
        threshold = 1.0e-10 * np.maximum(np.max(np.abs(templates), axis=1, keepdims=True), 1.0)
        ratios = np.divide(
            source_parent[None, :], templates,
            out=np.full_like(templates, np.nan),
            where=np.abs(templates) > threshold,
        )
        ratios[(~np.isfinite(ratios)) | (ratios <= 0.0)] = np.nan
        scales_med = np.full(len(ratios), np.nan, dtype=np.float64)
        valid_ratio_rows = np.any(np.isfinite(ratios), axis=1)
        if np.any(valid_ratio_rows):
            scales_med[valid_ratio_rows] = np.nanmedian(ratios[valid_ratio_rows], axis=1)
        scales_med = np.where(np.isfinite(scales_med), scales_med, scales_ls)
        scales_ls = np.clip(scales_ls, lo, hi)
        scales_med = np.clip(scales_med, lo, hi)

        def score(scales: np.ndarray) -> np.ndarray:
            centers = scales[:, None] * templates
            if self.observation_mode:
                denom_brush = self.cfg.final_noise_fraction * scales[:, None] * brushes
            else:
                hs = brush_shape_per_unit_noise(self.cfg, self.protocol, source)[idx]
                denom_brush = hs[None, :] + scales[:, None] * brushes
            return np.max(
                np.abs(source_parent[None, :] - centers) / np.maximum(denom_brush, 1.0e-30),
                axis=1,
            )

        s_ls = score(scales_ls)
        s_med = score(scales_med)
        use_med = s_med < s_ls
        scales = np.where(use_med, scales_med, scales_ls)
        scores = np.where(use_med, s_med, s_ls)
        return scales, scores

    def _shape_screening_order(self, source: np.ndarray, assets: TargetAssets) -> np.ndarray:
        """Rank target shapes independently of positive stiffness scale.

        The cosine ranking is only a screening device; every retained candidate is still
        evaluated with the exact bounded-tube objective and a globally bounded scale search.
        """
        idx = self.protocol.atlas_indices
        y = np.asarray(source, dtype=np.float64)[idx]
        ynorm = max(float(np.linalg.norm(y)), 1.0e-30)
        denom = np.maximum(assets.template_norms * ynorm, 1.0e-30)
        cosine = (assets.templates[:, idx] @ y) / denom
        cosine = np.nan_to_num(cosine, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
        return np.argsort(-cosine, kind="mergesort")

    def _screened_candidate_indices(
        self, source: np.ndarray, assets: TargetAssets, coarse: np.ndarray
    ) -> np.ndarray:
        coarse_order = np.argsort(coarse, kind="mergesort")
        shape_order = self._shape_screening_order(source, assets)
        n_scale = (
            self.cfg.observation_scale_refinement_pool
            if self.observation_mode else self.cfg.scale_refinement_pool
        )
        n_shape = (
            self.cfg.observation_shape_screening_pool
            if self.observation_mode else self.cfg.shape_screening_pool
        )
        merged = np.concatenate([
            coarse_order[: min(int(n_scale), len(coarse_order))],
            shape_order[: min(int(n_shape), len(shape_order))],
        ])
        # Stable unique: keep the strongest coarse candidates first, then add new
        # scale-invariant candidates.
        return np.asarray(list(dict.fromkeys(int(i) for i in merged)), dtype=int)

    def _refine_scale_for_template(
        self, source: np.ndarray, template: np.ndarray, initial_scale: Optional[float] = None
    ) -> Tuple[float, float, bool]:
        slo, shi = self._scale_bounds()
        log_bounds = (math.log(slo), math.log(shi))
        result = minimize_scalar(
            lambda z: self._score(source, float(np.exp(z)) * template),
            bounds=log_bounds, method="bounded",
            options={
                "xatol": 1.0e-11 if self.observation_mode else 1.0e-9,
                "maxiter": (
                    self.cfg.observation_optimizer_maxiter
                    if self.observation_mode else self.cfg.optimizer_maxiter
                ),
            },
        )
        if np.isfinite(result.fun):
            return float(np.exp(result.x)), float(result.fun), bool(result.success)
        if initial_scale is None:
            initial_scale = math.sqrt(slo * shi)
        initial_scale = float(np.clip(initial_scale, slo, shi))
        return initial_scale, float(self._score(source, initial_scale * template)), False

    def solve_fixed_coordinate(
        self, source: np.ndarray, target_model: str, coordinate: Sequence[float]
    ) -> Dict[str, object]:
        template = model_template(
            self.cfg, target_model, coordinate, self.protocol, self.bases
        )
        scale, score, converged = self._refine_scale_for_template(source, template)
        return {
            "target_model": target_model,
            "score": score,
            "best_coordinate": np.asarray(coordinate, dtype=float).tolist(),
            "best_scale": scale,
            "best_mu0_kpa": mu0_kpa_from_response_scale(scale),
            "optimizer_success": bool(converged),
            "solution_source": "fixed_coordinate_global_scale_refinement",
        }

    def _coordinate_bounds(self, model: str) -> List[Tuple[float, float]]:
        if model in MULTIDIM_MODELS:
            return [(0.0, 1.0)] * MODEL_SHAPE_DIMS[model]
        lo, hi = scalar_bounds(self.cfg, model)
        return [(lo, hi)]

    def _objective(self, source: np.ndarray, target_model: str, x: Sequence[float]) -> float:
        d = MODEL_SHAPE_DIMS[target_model]
        coord = np.asarray(x[:d], dtype=float)
        scale = float(np.exp(x[d]))
        target = scale * model_template(self.cfg, target_model, coord, self.protocol, self.bases)
        return self._score(source, target)


    def solve(self, source: np.ndarray, target_model: str) -> Dict[str, object]:
        source = np.asarray(source, dtype=np.float64)
        assets = self.assets[target_model]
        scales, coarse = self._coarse_scales_and_scores(source, assets)
        initial_coarse_best = float(np.min(coarse))

        # Refine scale for the union of (i) best tube-score candidates and (ii) best
        # scale-invariant shape candidates.  The latter is the key V68.1 repair.
        candidates = self._screened_candidate_indices(source, assets, coarse)
        local_scale_success = True
        for idx in candidates:
            scale, score, converged = self._refine_scale_for_template(
                source, assets.templates[int(idx)], float(scales[int(idx)])
            )
            local_scale_success = local_scale_success and bool(converged)
            if score < float(coarse[int(idx)]):
                coarse[int(idx)] = score
                scales[int(idx)] = scale

        order = np.argsort(coarse, kind="mergesort")
        best_idx = int(order[0])
        best_score = float(coarse[best_idx])
        d = assets.coordinates.shape[1]
        best_x = np.concatenate([
            assets.coordinates[best_idx],
            [math.log(float(scales[best_idx]))],
        ])
        source_name = "screened_grid_global_scale_refinement"
        continuous_attempted = False
        continuous_converged = True

        if best_score <= (
            self.cfg.observation_refinement_score_cap
            if self.observation_mode
            else self.cfg.local_refinement_critical_noise_cap
        ):
            n_starts = (
                self.cfg.observation_coarse_starts
                if self.observation_mode else self.cfg.coarse_starts
            )
            starts = [
                np.concatenate([
                    assets.coordinates[int(i)],
                    [math.log(float(scales[int(i)]))],
                ])
                for i in order[: min(int(n_starts), len(order))]
            ]
            slo, shi = self._scale_bounds()
            bounds = self._coordinate_bounds(target_model) + [(math.log(slo), math.log(shi))]
            continuous_attempted = True
            continuous_converged = False
            for x0 in starts:
                result = minimize(
                    lambda z: self._objective(source, target_model, z),
                    x0=np.asarray(x0, dtype=float),
                    method="Powell",
                    bounds=bounds,
                    options={
                        "xtol": 1.0e-9 if self.observation_mode else 1.0e-8,
                        "ftol": 1.0e-9 if self.observation_mode else 1.0e-8,
                        "maxiter": (
                            self.cfg.observation_optimizer_maxiter
                            if self.observation_mode else self.cfg.optimizer_maxiter
                        ),
                        "maxfev": (
                            self.cfg.observation_optimizer_maxfev
                            if self.observation_mode else self.cfg.optimizer_maxfev
                        ),
                    },
                )
                continuous_converged = continuous_converged or bool(result.success)
                if np.isfinite(result.fun) and float(result.fun) < best_score:
                    best_score = float(result.fun)
                    best_x = np.asarray(result.x, dtype=float)
                    source_name = "continuous_local_refinement"

        coord = best_x[:d]
        _, _, physical = model_template_and_jacobian(
            self.cfg, target_model, coord, self.protocol, self.bases
        )
        best_scale = float(np.exp(best_x[d]))
        best_mu0 = mu0_kpa_from_response_scale(best_scale)
        shape_zone = shape_support_zone(self.cfg, target_model, physical)
        scale_zone = scale_support_zone(self.cfg, best_mu0)
        return {
            "target_model": target_model,
            "score": best_score,
            "best_coordinate": coord.tolist(),
            "best_scale": best_scale,
            "best_mu0_kpa": best_mu0,
            "best_shape_support_zone": shape_zone,
            "best_scale_support_zone": scale_zone,
            "best_biological_support_zone": combine_support_zones(shape_zone, scale_zone),
            "best_physical_parameters": physical,
            "optimizer_success": bool(local_scale_success and (not continuous_attempted or continuous_converged)),
            "solution_source": source_name,
            "coarse_best_score": initial_coarse_best,
            "screened_candidate_count": int(len(candidates)),
            "continuous_refinement_attempted": bool(continuous_attempted),
            "continuous_refinement_converged": bool(continuous_converged),
        }

    def solve_nh(self, source: np.ndarray) -> Dict[str, object]:
        source = np.asarray(source, dtype=np.float64)
        slo, shi = self._scale_bounds()
        log_lo, log_hi = math.log(slo), math.log(shi)
        grid = np.linspace(log_lo, log_hi, 401)
        scores = np.array([
            self._score(source, float(np.exp(z)) * self.nh) for z in grid
        ])
        idx = int(np.argmin(scores))
        best_score = float(scores[idx])
        best_log = float(grid[idx])
        left, right = grid[max(0, idx - 1)], grid[min(len(grid) - 1, idx + 1)]
        if right > left:
            result = minimize_scalar(
                lambda z: self._score(source, float(np.exp(z)) * self.nh),
                bounds=(float(left), float(right)), method="bounded",
                options={"xatol": 1.0e-11, "maxiter": self.cfg.optimizer_maxiter},
            )
            if float(result.fun) < best_score:
                best_score = float(result.fun)
                best_log = float(result.x)
        best_scale = float(np.exp(best_log))
        best_mu0 = mu0_kpa_from_response_scale(best_scale)
        scale_zone = scale_support_zone(self.cfg, best_mu0)
        return {
            "target_model": "NH",
            "score": best_score,
            "best_coordinate": [0.0],
            "best_scale": best_scale,
            "best_mu0_kpa": best_mu0,
            "best_shape_support_zone": "INSIDE_LITERATURE_CORE",
            "best_scale_support_zone": scale_zone,
            "best_biological_support_zone": scale_zone,
            "best_physical_parameters": {},
            "optimizer_success": True,
            "solution_source": "scale_only_refinement",
            "coarse_best_score": float(scores[idx]),
        }


# =============================================================================
# Source-manifold atlas
# =============================================================================


def build_source_states(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    """Build a joint biological shape-scale source atlas without a Cartesian explosion.

    Shape nodes remain Fisher-adaptive.  Each node receives a deterministic stratified
    log-stiffness value; NH uses a dedicated logarithmic stiffness grid.  Because the
    constitutive families are exactly homogeneous in mu0 and the brush is relative, this
    covers the universal scale dimension without redundantly repeating every shape at every
    scale.
    """
    rows: List[Dict[str, object]] = []
    templates: Dict[str, List[np.ndarray]] = {m: [] for m in ALL_MODELS}

    def append_state(
        model: str,
        source_index: int,
        coordinate: Sequence[float],
        unit_template: np.ndarray,
        physical: Mapping[str, float],
        mu0_kpa: float,
        extra: Optional[Mapping[str, object]] = None,
    ) -> None:
        scale = response_scale_from_mu0_kpa(mu0_kpa)
        shape_zone = shape_support_zone(cfg, model, physical)
        stiffness_zone = scale_support_zone(cfg, mu0_kpa)
        row: Dict[str, object] = {
            "source_model": model,
            "source_index": int(source_index),
            "coordinate_json": json.dumps(np.asarray(coordinate, dtype=float).tolist()),
            "source_mu0_kpa": float(mu0_kpa),
            "source_response_scale_kpa": float(scale),
            "shape_support_zone": shape_zone,
            "scale_support_zone": stiffness_zone,
            "biological_support_zone": combine_support_zones(shape_zone, stiffness_zone),
        }
        for j, value in enumerate(np.asarray(coordinate, dtype=float).reshape(-1)):
            row[f"coordinate_{j}"] = float(value)
        row.update({f"physical_{k}": v for k, v in physical.items()})
        if extra:
            row.update(dict(extra))
        rows.append(row)
        templates[model].append(scale * np.asarray(unit_template, dtype=np.float64))

    fnh = model_template(cfg, "NH", (0.0,), protocol, bases)
    nh_mu0 = np.geomspace(cfg.mu0_min_kpa, cfg.mu0_max_kpa, cfg.nh_source_scale_points)
    for i, mu0 in enumerate(nh_mu0):
        append_state("NH", i, (0.0,), fnh, {}, float(mu0))

    for model_index, model in enumerate(SCALAR_PARENTS):
        grid = registry.scalar_grid(model, cfg.scalar_source_points)
        mu0_values = logarithmic_mu0_values(cfg, len(grid), seed_offset=100 + model_index)
        for i, (x, mu0) in enumerate(zip(grid, mu0_values)):
            f, _, physical = model_template_and_jacobian(cfg, model, (x,), protocol, bases)
            append_state(model, i, (float(x),), f, physical, float(mu0))

    og2 = registry.ogden2_nodes(cfg.ogden2_source_points, seed_offset=11)
    og2_mu0 = logarithmic_mu0_values(cfg, len(og2), seed_offset=211)
    for i, ((_, r), mu0) in enumerate(zip(og2.iterrows(), og2_mu0)):
        q = np.array([r.q_center, r.q_gap, r.q_weight], dtype=float)
        f, _, physical = model_template_and_jacobian(cfg, "OGDEN2", q, protocol, bases)
        append_state(
            "OGDEN2", i, q, f, physical, float(mu0),
            {
                "fisher_volume_density": float(r.fisher_volume_density),
                "nearest_neighbor_q_distance": float(r.nearest_neighbor_q_distance),
            },
        )

    gp2 = registry.gp2_nodes(cfg.gp2_source_points, seed_offset=17)
    gp2_mu0 = logarithmic_mu0_values(cfg, len(gp2), seed_offset=317)
    for i, ((_, r), mu0) in enumerate(zip(gp2.iterrows(), gp2_mu0)):
        q = np.array([r.q_rho, r.q_beta20, r.q_beta11, r.q_beta02], dtype=float)
        f, _, physical = model_template_and_jacobian(cfg, "GP2", q, protocol, bases)
        append_state(
            "GP2", i, q, f, physical, float(mu0),
            {
                "fisher_volume_density": float(r.fisher_volume_density),
                "nearest_neighbor_q_distance": float(r.nearest_neighbor_q_distance),
            },
        )

    arrays = {m: np.stack(v) for m, v in templates.items()}
    return pd.DataFrame(rows), arrays


def source_coordinate_from_row(row: pd.Series) -> np.ndarray:
    return np.asarray(json.loads(str(row["coordinate_json"])), dtype=float)


def exact_nested_fit_from_source_row(
    source_model: str,
    target_model: str,
    row: pd.Series,
) -> Optional[Dict[str, object]]:
    physical: Dict[str, object]
    source_name: str
    coordinate: List[float]
    if source_model == "OGDEN1" and target_model == "OGDEN2":
        alpha = float(row.get("physical_alpha", row.get("coordinate_0", float("nan"))))
        physical = {"alpha1": alpha, "alpha2": alpha, "weight1": 0.5, "exact_boundary": "equal_exponents"}
        coordinate = [float("nan")] * 3
        source_name = "exact_ogden1_nested_in_ogden2"
    elif source_model == "MR" and target_model == "OGDEN2":
        rho = float(row.get("physical_rho", row.get("coordinate_0", float("nan"))))
        physical = {"alpha1": -2.0, "alpha2": 2.0, "weight1": rho, "weight2": 1.0-rho, "exact_embedded_stratum": "mooney_rivlin"}
        coordinate = [float("nan")] * 3
        source_name = "exact_mr_nested_in_ogden2"
    elif source_model == "MR" and target_model == "GP2":
        rho = float(row.get("physical_rho", row.get("coordinate_0", float("nan"))))
        physical = {"rho": rho, "beta20": 0.0, "beta11": 0.0, "beta02": 0.0, "exact_embedded_stratum": "mooney_rivlin"}
        coordinate = [float("nan")] * 4
        source_name = "exact_mr_nested_in_gp2"
    elif source_model == "YEOH2" and target_model == "GP2":
        beta = float(row.get("physical_beta", row.get("coordinate_0", float("nan"))))
        physical = {"rho": 0.0, "beta20": beta, "beta11": 0.0, "beta02": 0.0, "exact_embedded_stratum": "yeoh2"}
        coordinate = [float("nan")] * 4
        source_name = "exact_yeoh2_nested_in_gp2"
    else:
        return None
    source_scale = float(row["source_response_scale_kpa"])
    source_mu0 = float(row["source_mu0_kpa"])
    shape_zone = str(row.get("shape_support_zone", "INSIDE_LITERATURE_CORE"))
    scale_zone = str(row.get("scale_support_zone", "INSIDE_LITERATURE_CORE"))
    return {
        "target_model": target_model, "score": 0.0,
        "best_coordinate": coordinate, "best_scale": source_scale,
        "best_mu0_kpa": source_mu0,
        "best_shape_support_zone": shape_zone,
        "best_scale_support_zone": scale_zone,
        "best_biological_support_zone": combine_support_zones(shape_zone, scale_zone),
        "best_physical_parameters": physical, "optimizer_success": True,
        "solution_source": source_name, "coarse_best_score": 0.0,
    }


def propagate_exact_hierarchy_scores(
    scores: Dict[str, float],
    fits: Dict[str, Dict[str, object]],
) -> None:
    changed = True
    while changed:
        changed = False
        for subset in ALL_MODELS:
            if subset not in scores:
                continue
            for sup in STRICT_SUPERS[subset]:
                if sup not in scores or scores[subset] < scores[sup]:
                    scores[sup] = float(scores[subset])
                    if subset in fits:
                        inherited = dict(fits[subset])
                        inherited["target_model"] = sup
                        inherited["solution_source"] = f"inherited_from_exact_{subset.lower()}_subset"
                        fits[sup] = inherited
                    changed = True


def build_pairwise_source_atlas(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
    source_df: pd.DataFrame,
    source_templates: Dict[str, np.ndarray],
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    solver = ContinuousModelSolver(cfg, protocol, bases, registry)
    pair_rows: List[Dict[str, object]] = []
    atlas_rows: List[Dict[str, object]] = []
    total = len(source_df)

    for count, (_, row) in enumerate(source_df.iterrows(), start=1):
        source_model = str(row.source_model)
        source_index = int(row.source_index)
        source = source_templates[source_model][source_index]
        scores: Dict[str, float] = {source_model: 0.0}
        fits: Dict[str, Dict[str, object]] = {}

        if source_model == "NH":
            source_scale = float(row["source_response_scale_kpa"])
            source_mu0 = float(row["source_mu0_kpa"])
            scale_zone = scale_support_zone(cfg, source_mu0)
            for target in ALL_MODELS:
                scores[target] = 0.0
                fits[target] = {
                    "target_model": target, "score": 0.0, "best_coordinate": [0.0],
                    "best_scale": source_scale, "best_mu0_kpa": source_mu0,
                    "best_shape_support_zone": "INSIDE_LITERATURE_CORE",
                    "best_scale_support_zone": scale_zone,
                    "best_biological_support_zone": scale_zone,
                    "best_physical_parameters": {},
                    "optimizer_success": True, "solution_source": "exact_nested_membership",
                    "coarse_best_score": 0.0,
                }
        else:
            nhfit = solver.solve_nh(source)
            scores["NH"] = float(nhfit["score"])
            fits["NH"] = nhfit
            for target in PARENT_MODELS:
                if target == source_model:
                    continue
                fit = exact_nested_fit_from_source_row(source_model, target, row)
                if fit is None:
                    fit = solver.solve(source, target)
                scores[target] = float(fit["score"])
                fits[target] = fit
            propagate_exact_hierarchy_scores(scores, fits)

        for target, fit in fits.items():
            if target == source_model:
                continue
            pair_rows.append({
                "source_model": source_model, "source_index": source_index,
                "target_model": target,
                "critical_noise_fraction": float(scores[target]),
                "critical_noise_percent": 100.0 * float(scores[target]),
                "best_target_coordinate_json": json.dumps(fit["best_coordinate"]),
                "best_target_scale_kpa": float(fit["best_scale"]),
                "best_target_mu0_kpa": float(fit["best_mu0_kpa"]),
                "best_target_shape_support_zone": fit["best_shape_support_zone"],
                "best_target_scale_support_zone": fit["best_scale_support_zone"],
                "best_target_biological_support_zone": fit["best_biological_support_zone"],
                "best_target_physical_json": json.dumps(fit["best_physical_parameters"], sort_keys=True),
                "optimizer_success": bool(fit["optimizer_success"]),
                "solution_source": fit["solution_source"],
                "coarse_best_score": float(fit["coarse_best_score"]),
            })

        compatible = [m for m in ALL_MODELS if scores.get(m, float("inf")) <= cfg.final_noise_fraction + cfg.critical_noise_tolerance]
        compatible = list(hierarchy_closure(compatible))
        region, signature, subtype = region_label_from_compatibility(compatible)
        minimal = minimal_compatible_models(compatible)
        atlas_row = dict(row)
        atlas_row.update({
            "primary_region_at_3pct": region,
            "compatibility_set_at_3pct": signature,
            "compatibility_signature_at_3pct": signature,
            "minimal_adequate_models_at_3pct": ordered_signature(minimal),
            "collision_subtype_at_3pct": subtype,
            "source_model_included": bool(source_model in compatible),
            "atlas_point_assigned": True,
            "n_compatible_models_at_3pct": len(compatible),
        })
        for model in ALL_MODELS:
            atlas_row[f"critical_noise_to_{model.lower()}"] = float(scores.get(model, 0.0 if model == source_model else float("inf")))
        previous: set[str] = set()
        monotonic = True
        for noise in cfg.noise_levels:
            current = set(hierarchy_closure(m for m in ALL_MODELS if scores.get(m, float("inf")) <= noise + cfg.critical_noise_tolerance))
            if not previous.issubset(current):
                monotonic = False
            previous = current
            atlas_row[f"signature_at_{int(round(100*noise))}pct"] = ordered_signature(current)
        atlas_row["noise_compatibility_monotonic"] = monotonic
        atlas_rows.append(atlas_row)

        if count % max(1, total // 20) == 0 or count == total:
            logger.info("  source atlas progress: %d/%d", count, total)

    return pd.DataFrame(pair_rows), pd.DataFrame(atlas_rows)


# =============================================================================
# Ambient observation classifier

# =============================================================================
# Ambient observation classifier and NONE region
# =============================================================================


class AmbientClassifier:
    def __init__(
        self,
        cfg: Config,
        protocol: Protocol,
        bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
        registry: AtlasSamplerRegistry,
        scalar_target_points: Optional[int] = None,
        ogden2_target_points: Optional[int] = None,
        gp2_target_points: Optional[int] = None,
        seed_offset: int = 30,
    ) -> None:
        self.cfg = cfg
        self.protocol = protocol
        self.bases = bases
        self.solver = ContinuousModelSolver(
            cfg, protocol, bases, registry,
            scalar_target_points=scalar_target_points or cfg.observation_scalar_target_points,
            ogden2_target_points=ogden2_target_points or cfg.observation_ogden2_target_points,
            gp2_target_points=gp2_target_points or cfg.observation_gp2_target_points,
            observation_mode=True, seed_offset=seed_offset,
        )

    def classify(self, observation: np.ndarray) -> Dict[str, object]:
        y = np.asarray(observation, dtype=np.float64)
        if y.ndim != 1 or y.size != self.protocol.dimension:
            raise ValueError(f"Observation must have shape ({self.protocol.dimension},).")
        fits: Dict[str, Dict[str, object]] = {"NH": self.solver.solve_nh(y)}
        for model in PARENT_MODELS:
            fits[model] = self.solver.solve(y, model)
        scores = {m: float(fits[m]["score"]) for m in ALL_MODELS}
        propagate_exact_hierarchy_scores(scores, fits)
        compatible = [m for m in ALL_MODELS if scores[m] <= 1.0 + self.cfg.critical_noise_tolerance]
        compatible = list(hierarchy_closure(compatible))
        region, signature, subtype = region_label_from_compatibility(compatible)
        minimal = minimal_compatible_models(compatible)
        best_model = min(scores, key=scores.get)
        best_fit = fits[best_model]
        return {
            "primary_region": region,
            "compatibility_set": signature,
            "compatibility_signature": signature,
            "minimal_adequate_models": ordered_signature(minimal),
            "collision_subtype": subtype,
            "compatible_models": tuple(compatible),
            "minimum_model_score": float(min(scores.values())),
            "best_model_by_score": best_model,
            "best_fit_mu0_kpa": float(best_fit["best_mu0_kpa"]),
            "best_fit_biological_support_zone": best_fit["best_biological_support_zone"],
            "model_scores": scores,
            "model_fits": fits,
            "all_optimizer_success": bool(all(bool(f["optimizer_success"]) for f in fits.values())),
        }


def _ambient_result_fields(
    result: Mapping[str, object], source_model: str
) -> Dict[str, object]:
    compatible = tuple(result["compatible_models"])
    fields: Dict[str, object] = {
        "primary_region": result["primary_region"],
        "compatibility_signature": result["compatibility_signature"],
        "minimal_adequate_models": result["minimal_adequate_models"],
        "minimum_model_score": float(result["minimum_model_score"]),
        "best_model_by_score": result["best_model_by_score"],
        "best_fit_mu0_kpa": float(result["best_fit_mu0_kpa"]),
        "best_fit_biological_support_zone": result["best_fit_biological_support_zone"],
        "source_model_included": bool(source_model in compatible) if source_model in ALL_MODELS else False,
        "none_compatible": bool(result["primary_region"] == "NONE_COMPATIBLE"),
        "all_optimizer_success": bool(result["all_optimizer_success"]),
    }
    for model in ALL_MODELS:
        fields[f"score_{model.lower()}"] = float(result["model_scores"][model])
    return fields


def revalidate_failed_known_model_examples(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
    audit_df: pd.DataFrame,
    observations: np.ndarray,
    metadata: pd.DataFrame,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    failed = audit_df.index[
        (audit_df.audit_kind == "KNOWN_MODEL") & (~audit_df.source_model_included)
    ].to_numpy(dtype=int)
    if len(failed) == 0:
        return audit_df, metadata, pd.DataFrame()

    logger.warning(
        "Revalidating %d failed known-model observations with denser target assets and patched screening...",
        len(failed),
    )
    classifier = AmbientClassifier(
        cfg, protocol, bases, registry,
        scalar_target_points=max(
            cfg.known_revalidation_scalar_target_points,
            cfg.observation_scalar_target_points,
        ),
        ogden2_target_points=max(
            cfg.known_revalidation_ogden2_target_points,
            cfg.observation_ogden2_target_points,
        ),
        gp2_target_points=max(
            cfg.known_revalidation_gp2_target_points,
            cfg.observation_gp2_target_points,
        ),
        seed_offset=cfg.known_revalidation_seed_offset,
    )
    rows: List[Dict[str, object]] = []
    for idx in failed:
        old = audit_df.loc[idx].copy()
        source_model = str(old.generating_model)
        result = classifier.classify(observations[idx])
        fields = _ambient_result_fields(result, source_model)
        for key, value in fields.items():
            audit_df.loc[idx, key] = value
        metadata.loc[idx, "primary_region"] = result["primary_region"]
        metadata.loc[idx, "compatibility_signature"] = result["compatibility_signature"]
        rows.append({
            "audit_row_index": int(idx),
            "generating_model": source_model,
            "old_primary_region": old.primary_region,
            "old_minimum_model_score": float(old.minimum_model_score),
            "old_source_model_included": bool(old.source_model_included),
            "new_primary_region": result["primary_region"],
            "new_minimum_model_score": float(result["minimum_model_score"]),
            "new_source_model_included": bool(fields["source_model_included"]),
            "new_best_model_by_score": result["best_model_by_score"],
            "new_best_fit_mu0_kpa": float(result["best_fit_mu0_kpa"]),
            "all_optimizer_success": bool(result["all_optimizer_success"]),
        })
    return audit_df, metadata, pd.DataFrame(rows)


def build_ambient_audit(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
    classifier: AmbientClassifier,
    rng: np.random.Generator,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    rows: List[Dict[str, object]] = []
    observations: List[np.ndarray] = []
    metadata: List[Dict[str, object]] = []

    def record(
        kind: str, generator: str, source_model: str, coord: Sequence[float],
        mu0_kpa: float, y: np.ndarray, result: Dict[str, object],
        oracle: Optional[Mapping[str, object]] = None,
    ) -> None:
        row: Dict[str, object] = {
            "audit_kind": kind,
            "generator": generator,
            "generating_model": source_model,
            "true_coordinate_json": json.dumps(np.asarray(coord, dtype=float).tolist()),
            "true_mu0_kpa": float(mu0_kpa),
            "oracle_source_score": float(oracle["score"]) if oracle is not None else float("nan"),
            "oracle_source_mu0_kpa": float(oracle["best_mu0_kpa"]) if oracle is not None else float("nan"),
        }
        row.update(_ambient_result_fields(result, source_model))
        rows.append(row)
        observations.append(y.copy())
        metadata.append({
            "audit_kind": kind,
            "generator": generator,
            "generating_model": source_model,
            "primary_region": result["primary_region"],
            "compatibility_signature": result["compatibility_signature"],
        })

    logger.info("Running ambient known-model audit...")
    # Use exact classifier target nodes so the source center is present in the numerical atlas.
    scalar_coords = {
        m: classifier.solver.assets[m].coordinates[
            np.linspace(0, len(classifier.solver.assets[m].coordinates) - 1, cfg.n_audit_shapes_scalar).astype(int)
        ] for m in SCALAR_PARENTS
    }
    og2_coords = classifier.solver.assets["OGDEN2"].coordinates[
        np.linspace(0, len(classifier.solver.assets["OGDEN2"].coordinates) - 1, cfg.n_audit_shapes_ogden2).astype(int)
    ]
    gp2_coords = classifier.solver.assets["GP2"].coordinates[
        np.linspace(0, len(classifier.solver.assets["GP2"].coordinates) - 1, cfg.n_audit_shapes_gp2).astype(int)
    ]
    coordinates_by_model: Dict[str, np.ndarray] = {
        "NH": np.zeros((1, 1)),
        **scalar_coords,
        "OGDEN2": og2_coords,
        "GP2": gp2_coords,
    }
    for model in ALL_MODELS:
        for coord in coordinates_by_model[model]:
            for _ in range(cfg.n_audit_reps):
                mu0_kpa = float(np.exp(rng.uniform(math.log(cfg.audit_mu0_min_kpa), math.log(cfg.audit_mu0_max_kpa))))
                center = response_scale_from_mu0_kpa(mu0_kpa) * model_template(cfg, model, coord, protocol, bases)
                delta = cfg.final_noise_fraction * brush_shape_per_unit_noise(cfg, protocol, center)
                y = center + rng.uniform(-1.0, 1.0, center.size) * delta
                result = classifier.classify(y)
                oracle = (
                    classifier.solver.solve_nh(y)
                    if model == "NH"
                    else classifier.solver.solve_fixed_coordinate(y, model, coord)
                )
                record(
                    "KNOWN_MODEL", "BOUNDED_SOURCE_TUBE", model, coord,
                    mu0_kpa, y, result, oracle=oracle,
                )

    logger.info("Generating off-manifold candidates for NONE_COMPATIBLE...")
    centers: List[np.ndarray] = []
    for model in PARENT_MODELS:
        assets = classifier.solver.assets[model]
        indices = np.linspace(0, len(assets.templates) - 1, 6).astype(int)
        for template in assets.templates[indices]:
            mu0_kpa = float(np.exp(rng.uniform(math.log(cfg.mu0_core_min_kpa), math.log(cfg.mu0_core_max_kpa))))
            centers.append(response_scale_from_mu0_kpa(mu0_kpa) * template)
    n_each = cfg.off_manifold_candidates_per_type
    candidate_specs: List[Tuple[str, np.ndarray]] = []
    # Blockwise scale mismatch cannot be absorbed by one global modulus scale.
    for _ in range(n_each):
        center = centers[int(rng.integers(0, len(centers)))].copy()
        factors = rng.choice([0.55, 0.70, 1.35, 1.60], size=3, replace=True)
        while len(set(float(v) for v in factors)) < 2:
            factors = rng.choice([0.55, 0.70, 1.35, 1.60], size=3, replace=True)
        y = center.copy()
        parent = y[protocol.parent_slice].reshape(protocol.n_parent_states, 2)
        for factor, block in zip(factors, np.array_split(np.arange(protocol.n_parent_states), 3)):
            parent[block] *= factor
        y[protocol.parent_slice] = parent.reshape(-1)
        candidate_specs.append(("PARENT_F_BLOCK_SCALE_MISMATCH", y))
    # High-frequency componentwise patterns are outside smooth constitutive curves/surfaces.
    for _ in range(n_each):
        center = centers[int(rng.integers(0, len(centers)))].copy()
        h = brush_shape_per_unit_noise(cfg, protocol, center)
        pattern = np.sign(np.sin(np.arange(center.size) * rng.uniform(1.4, 2.8) + rng.uniform(0, 2 * math.pi)))
        y = center + rng.uniform(0.12, 0.25) * pattern * h
        candidate_specs.append(("OSCILLATORY_OFF_MANIFOLD", y))
    # Nonconvex mixtures of distant model states.
    for _ in range(n_each):
        a, b = rng.choice(len(centers), size=2, replace=False)
        y = 1.35 * centers[a] - 0.35 * centers[b]
        candidate_specs.append(("AFFINE_MODEL_MIX", y))

    for generator, y in candidate_specs:
        result = classifier.classify(y)
        record("OFF_MANIFOLD", generator, "NONE", (float("nan"),), float("nan"), y, result)

    return pd.DataFrame(rows), np.stack(observations), pd.DataFrame(metadata)


def revalidate_none_examples(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
    audit_df: pd.DataFrame,
    observations: np.ndarray,
    logger: logging.Logger,
) -> pd.DataFrame:
    candidates = audit_df[
        (audit_df.audit_kind == "OFF_MANIFOLD")
        & (audit_df.none_compatible)
        & (audit_df.minimum_model_score >= 1.0 + cfg.certified_none_margin)
    ]
    if len(candidates) == 0:
        return pd.DataFrame()
    chosen = candidates.head(cfg.maximum_none_revalidation_examples)
    logger.info("Revalidating %d margin-separated NONE observations on denser assets...", len(chosen))
    classifier = AmbientClassifier(
        cfg, protocol, bases, registry,
        scalar_target_points=max(cfg.observation_scalar_target_points + 150, cfg.scalar_target_points),
        ogden2_target_points=cfg.revalidation_ogden2_target_points,
        gp2_target_points=cfg.revalidation_gp2_target_points,
        seed_offset=77,
    )
    rows: List[Dict[str, object]] = []
    for idx in chosen.index:
        result = classifier.classify(observations[int(idx)])
        rows.append({
            "audit_row_index": int(idx),
            "original_minimum_score": float(audit_df.loc[idx, "minimum_model_score"]),
            "revalidated_primary_region": result["primary_region"],
            "revalidated_minimum_score": float(result["minimum_model_score"]),
            "revalidated_none": bool(result["primary_region"] == "NONE_COMPATIBLE"),
            "all_optimizer_success": bool(result["all_optimizer_success"]),
        })
    return pd.DataFrame(rows)


# =============================================================================
# Validation, summaries, and visualizations
# =============================================================================


def build_universal_mu0_scale_audit(cfg: Config) -> pd.DataFrame:
    """Verify that every unit template has infinitesimal shear slope 2.

    Therefore multiplying by s=mu0/2 gives the same physical initial shear modulus
    for all seven constitutive families.
    """
    tiny_cfg = replace(
        cfg, n_per_mode=7, ut_min=1.0, ut_max=1.0, bt_min=1.0, bt_max=1.0,
        sh_min=-1.0e-6, sh_max=1.0e-6,
        parent_f_states=31, parent_f_candidate_points=128, parent_f_boundary_points=12,
    )
    protocol = make_protocol(tiny_cfg)
    bases = constitutive_bases(protocol)
    cases: Tuple[Tuple[str, Sequence[float]], ...] = (
        ("NH", (0.0,)),
        ("MR", (0.40,)),
        ("YEOH2", (10.0,)),
        ("GENT", (0.5 * cfg.gent_gamma_core_max,)),
        ("OGDEN1", (-19.0,)),
        ("OGDEN1", (16.55,)),
        ("OGDEN2", (0.50, 0.50, 0.50)),
        ("GP2", (0.50, 0.50, 0.50, 0.50)),
    )
    rows: List[Dict[str, object]] = []
    positive = np.flatnonzero(protocol.sh > 0.0)[0]
    for model, coord in cases:
        response = model_template(tiny_cfg, model, coord, protocol, bases)
        shear = response[protocol.mode_slices["SH"]]
        slope = float(shear[positive] / protocol.sh[positive])
        rows.append({
            "model": model,
            "coordinate_json": json.dumps(list(coord)),
            "unit_template_small_shear_slope": slope,
            "expected_slope": 2.0,
            "relative_error": abs(slope - 2.0) / 2.0,
        })
    return pd.DataFrame(rows)


def build_parent_subprotocol_consistency_audit(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> pd.DataFrame:
    """Verify that retained UT/BT/SH responses are embedded in the complete-F parent set."""
    cases: Tuple[Tuple[str, Sequence[float]], ...] = (
        ("NH", (0.0,)),
        ("MR", (0.40,)),
        ("YEOH2", (10.0,)),
        ("GENT", (0.5 * cfg.gent_gamma_core_max,)),
        ("OGDEN1", (-19.0,)),
        ("OGDEN1", (8.0,)),
        ("OGDEN2", (0.35, 0.55, 0.65)),
        ("GP2", (0.45, 0.40, 0.35, 0.30)),
    )
    rows: List[Dict[str, object]] = []

    def parent_index(logs: np.ndarray) -> int:
        ordered = np.sort(np.asarray(logs, dtype=np.float64))[::-1]
        d2 = np.sum((protocol.parent_log_principal - ordered[None, :])**2, axis=1)
        idx = int(np.argmin(d2))
        if float(d2[idx]) > 1.0e-20:
            raise RuntimeError("A retained standard-protocol state is absent from the complete-F parent set.")
        return idx

    for model, coord in cases:
        response = model_template(cfg, model, coord, protocol, bases)
        parent = response[protocol.parent_slice].reshape(protocol.n_parent_states, 2)

        for j, lam in enumerate(protocol.ut):
            idx = parent_index(np.log([lam, lam**-0.5, lam**-0.5]))
            reconstructed = float(np.max(parent[idx]))
            standard = float(response[protocol.mode_slices["UT"]][j])
            rows.append({
                "model": model, "coordinate_json": json.dumps(list(coord)), "subprotocol": "UT",
                "state_index": j, "standard_response": standard,
                "parent_reconstructed_magnitude": reconstructed,
                "absolute_magnitude_error": abs(abs(standard)-abs(reconstructed)),
                "relative_magnitude_error": abs(abs(standard)-abs(reconstructed)) / max(abs(standard), abs(reconstructed), 1.0),
            })

        for j, lam in enumerate(protocol.bt):
            idx = parent_index(np.log([lam, lam, lam**-2.0]))
            reconstructed = float(np.max(parent[idx]))
            standard = float(response[protocol.mode_slices["BT"]][j])
            rows.append({
                "model": model, "coordinate_json": json.dumps(list(coord)), "subprotocol": "BT",
                "state_index": j, "standard_response": standard,
                "parent_reconstructed_magnitude": reconstructed,
                "absolute_magnitude_error": abs(abs(standard)-abs(reconstructed)),
                "relative_magnitude_error": abs(abs(standard)-abs(reconstructed)) / max(abs(standard), abs(reconstructed), 1.0),
            })

        for j, gam in enumerate(protocol.sh):
            root = math.sqrt(1.0 + 0.25*float(gam)**2)
            stretches = np.array([root + 0.5*float(gam), 1.0, root - 0.5*float(gam)])
            idx = parent_index(np.log(stretches))
            reconstructed = float(np.max(parent[idx]) / math.sqrt(float(gam)**2 + 4.0))
            standard = float(response[protocol.mode_slices["SH"]][j])
            rows.append({
                "model": model, "coordinate_json": json.dumps(list(coord)), "subprotocol": "SH",
                "state_index": j, "standard_response": standard,
                "parent_reconstructed_magnitude": reconstructed,
                "absolute_magnitude_error": abs(abs(standard)-abs(reconstructed)),
                "relative_magnitude_error": abs(abs(standard)-abs(reconstructed)) / max(abs(standard), abs(reconstructed), 1.0),
            })
    return pd.DataFrame(rows)


def build_literature_anchor_coverage_audit(
    cfg: Config,
    registry: AtlasSamplerRegistry,
    og2_source_nodes: pd.DataFrame,
    gp2_source_nodes: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for model, anchors in LITERATURE_SCALAR_ANCHORS.items():
        nodes = registry.scalar_grid(model, cfg.scalar_source_points)
        lo, hi = scalar_bounds(cfg, model)
        for anchor in anchors:
            if lo <= anchor <= hi:
                rows.append({
                    "model": model, "anchor_type": "literature_scalar",
                    "anchor_json": json.dumps([anchor]),
                    "nearest_source_coordinate_distance": float(np.min(np.abs(nodes - anchor))),
                })

    q_og2 = og2_source_nodes[["q_center", "q_gap", "q_weight"]].to_numpy(float)
    for anchor in registry.ogden2._forced_anchor_q():
        rows.append({
            "model": "OGDEN2", "anchor_type": "forced_hierarchy_or_literature",
            "anchor_json": json.dumps(anchor.tolist()),
            "nearest_source_coordinate_distance": float(np.sqrt(np.min(np.sum((q_og2 - anchor[None, :]) ** 2, axis=1)))),
        })

    q_gp2 = gp2_source_nodes[["q_rho", "q_beta20", "q_beta11", "q_beta02"]].to_numpy(float)
    for physical in GP2_LITERATURE_ANCHOR_PHYSICAL:
        anchor = gp2_map_physical_to_q(cfg, *physical)
        rows.append({
            "model": "GP2", "anchor_type": "literature_gp2",
            "anchor_json": json.dumps(anchor.tolist()),
            "nearest_source_coordinate_distance": float(np.sqrt(np.min(np.sum((q_gp2 - anchor[None, :]) ** 2, axis=1)))),
        })
    return pd.DataFrame(rows)


def build_exact_hierarchy_audit(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    nh = model_template(cfg, "NH", (0.0,), protocol, bases)
    identities = (
        ("NH_vs_MR_rho0", model_template(cfg, "MR", (0.0,), protocol, bases)),
        ("NH_vs_YEOH2_beta0", model_template(cfg, "YEOH2", (0.0,), protocol, bases)),
        ("NH_vs_GENT_gamma0", model_template(cfg, "GENT", (0.0,), protocol, bases)),
        ("NH_vs_OGDEN1_alpha2", model_template(cfg, "OGDEN1", (2.0,), protocol, bases)),
        ("NH_vs_OGDEN2_equal_alpha2", ogden2_template_physical(2.0, 2.0, 0.37, protocol, cfg.ogden_alpha_series_threshold)),
        ("NH_vs_GP2_origin", gp2_template_physical(0.0, 0.0, 0.0, 0.0, bases)),
    )
    for name, response in identities:
        rows.append({"identity": name, "parameter": float("nan"), "max_abs_error": float(np.max(np.abs(nh-response)))})

    for alpha in (-6.0, -2.0, 0.0, 2.0, 6.0):
        og1 = model_template(cfg, "OGDEN1", (alpha,), protocol, bases)
        tests = (
            ("weight1_one", ogden2_template_physical(alpha, 5.0 if alpha != 5.0 else 4.0, 1.0, protocol, cfg.ogden_alpha_series_threshold)),
            ("weight1_zero", ogden2_template_physical(-5.0 if alpha != -5.0 else -4.0, alpha, 0.0, protocol, cfg.ogden_alpha_series_threshold)),
            ("equal_exponents", ogden2_template_physical(alpha, alpha, 0.37, protocol, cfg.ogden_alpha_series_threshold)),
        )
        for side, response in tests:
            rows.append({"identity": f"OGDEN1_in_OGDEN2_{side}", "parameter": alpha, "max_abs_error": float(np.max(np.abs(og1-response)))})

    for rho in np.linspace(cfg.mr_rho_min, cfg.mr_rho_max, 21):
        mr = model_template(cfg, "MR", (float(rho),), protocol, bases)
        og2 = ogden2_template_physical(-2.0, 2.0, float(rho), protocol, cfg.ogden_alpha_series_threshold)
        gp2 = gp2_template_physical(float(rho), 0.0, 0.0, 0.0, bases)
        rows.append({"identity": "MR_in_OGDEN2_alpha_minus2_plus2", "parameter": float(rho), "max_abs_error": float(np.max(np.abs(mr-og2)))})
        rows.append({"identity": "MR_in_GP2_linear_stratum", "parameter": float(rho), "max_abs_error": float(np.max(np.abs(mr-gp2)))})

    for beta in np.linspace(cfg.yeoh_beta_min, cfg.yeoh_beta_max, 21):
        yeoh = model_template(cfg, "YEOH2", (float(beta),), protocol, bases)
        gp2 = gp2_template_physical(0.0, float(beta), 0.0, 0.0, bases)
        rows.append({"identity": "YEOH2_in_GP2_beta20_stratum", "parameter": float(beta), "max_abs_error": float(np.max(np.abs(yeoh-gp2)))})

    for a1, a2, w in ((-6.0,-1.5,0.21),(-2.0,2.0,0.31),(0.0,5.0,0.64)):
        lhs = ogden2_template_physical(a1,a2,w,protocol,cfg.ogden_alpha_series_threshold)
        rhs = ogden2_template_physical(a2,a1,1.0-w,protocol,cfg.ogden_alpha_series_threshold)
        rows.append({"identity": "OGDEN2_term_permutation", "parameter": json.dumps([a1,a2,w]), "max_abs_error": float(np.max(np.abs(lhs-rhs)))})
    return pd.DataFrame(rows)


def exact_nesting_checks(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> Dict[str, float]:
    audit = build_exact_hierarchy_audit(cfg, protocol, bases)
    groups = audit.groupby("identity", dropna=False).max(numeric_only=True)
    checks = {
        str(identity): float(row.max_abs_error)
        for identity, row in groups.iterrows()
    }
    checks["global_max_abs_error"] = float(audit.max_abs_error.max())
    return checks


def summarize_source_atlas(atlas_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for model in ALL_MODELS:
        sub = atlas_df[atlas_df.source_model == model]
        minimal_contains = sub.minimal_adequate_models_at_3pct.astype(str).str.split("|").apply(lambda x: model in x)
        rows.append({
            "source_model": model,
            "n_states": int(len(sub)),
            "source_model_inclusion": float(sub.source_model_included.mean()),
            "atlas_assignment_rate": float(sub.atlas_point_assigned.mean()),
            "noise_monotonicity_rate": float(sub.noise_compatibility_monotonic.mean()),
            "source_model_is_minimal_adequate_fraction": float(minimal_contains.mean()),
            "none_fraction": float(np.mean(sub.primary_region_at_3pct == "NONE_COMPATIBLE")),
            "mean_compatible_models": float(sub.n_compatible_models_at_3pct.mean()),
            "n_distinct_compatibility_sets": int(sub.compatibility_set_at_3pct.nunique()),
            "literature_core_fraction": float(np.mean(sub.biological_support_zone == "INSIDE_LITERATURE_CORE")),
            "guard_margin_fraction": float(np.mean(sub.biological_support_zone == "INSIDE_BIOLOGICAL_MARGIN")),
            "mu0_min_kpa": float(sub.source_mu0_kpa.min()),
            "mu0_max_kpa": float(sub.source_mu0_kpa.max()),
        })
    return pd.DataFrame(rows)


def summarize_regions(atlas_df: pd.DataFrame) -> pd.DataFrame:
    return (
        atlas_df.groupby(["source_model", "primary_region_at_3pct"], dropna=False)
        .size().reset_index(name="n_states")
        .sort_values(["source_model", "n_states"], ascending=[True, False])
    )


def summarize_ambient(audit_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for kind, sub in audit_df.groupby("audit_kind"):
        rows.append({
            "audit_kind": kind,
            "n": int(len(sub)),
            "source_model_inclusion": float(sub.source_model_included.mean()) if kind == "KNOWN_MODEL" else float("nan"),
            "none_rate": float(sub.none_compatible.mean()),
            "all_optimizer_success_rate": float(sub.all_optimizer_success.mean()),
            "minimum_score_median": float(sub.minimum_model_score.median()),
        })
    return pd.DataFrame(rows)


def plot_fisher_geometry(
    outdir: Path,
    scalar_pilot: pd.DataFrame,
    og2_pilot: pd.DataFrame,
    og2_nodes: pd.DataFrame,
    gp2_pilot: pd.DataFrame,
    gp2_nodes: pd.DataFrame,
) -> None:
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for model in SCALAR_PARENTS:
        sub = scalar_pilot[scalar_pilot.model == model]
        ax.plot(sub.coordinate, sub.sqrt_profiled_fisher_metric, linewidth=1.8, label=model)
    ax.set_xlabel("Intrinsic scalar coordinate")
    ax.set_ylabel(r"$\sqrt{G_{\mathrm{shape}}}$")
    ax.set_title("Profiled Fisher–Riemannian arc density")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / "scalar_profiled_fisher_metric.png", dpi=220)
    plt.close(fig)

    for name, pilot, dim in (("ogden2", og2_pilot, 3), ("gp2", gp2_pilot, 4)):
        fig, ax = plt.subplots(figsize=(7.8, 5.2))
        values = pilot.fisher_volume_density.to_numpy(float)
        ax.hist(np.log10(np.maximum(values, 1.0e-30)), bins=60)
        ax.set_xlabel(r"$\log_{10}\sqrt{\det G_{\mathrm{shape}}}$")
        ax.set_ylabel("Pilot count")
        ax.set_title(f"{name.upper()} Fisher-volume density over the {dim}-D shape manifold")
        fig.tight_layout()
        fig.savefig(outdir / f"{name}_fisher_volume_density_histogram.png", dpi=220)
        plt.close(fig)

    fig = plt.figure(figsize=(8.2, 6.4))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(og2_nodes.q_center, og2_nodes.q_gap, og2_nodes.q_weight,
                    c=np.log10(np.maximum(og2_nodes.fisher_volume_density, 1.0e-30)), s=12, alpha=0.8)
    ax.set_xlabel("q_center"); ax.set_ylabel("q_gap"); ax.set_zlabel("q_weight")
    ax.set_title("Fisher-adaptive OGDEN2 source nodes")
    fig.colorbar(sc, ax=ax, shrink=0.7, label="log10 Fisher volume density")
    fig.tight_layout(); fig.savefig(outdir / "ogden2_fisher_adaptive_nodes_3d.png", dpi=220); plt.close(fig)

    # GP2 4-D nodes are shown as two paired coordinate projections.
    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    sc = ax.scatter(gp2_nodes.q_rho, gp2_nodes.q_beta20,
                    c=np.log10(np.maximum(gp2_nodes.fisher_volume_density, 1.0e-30)),
                    s=12, alpha=0.75)
    ax.set_xlabel("q_rho"); ax.set_ylabel("q_beta20")
    ax.set_title("Fisher-adaptive GP2 nodes: linear/Yeoh coordinates")
    fig.colorbar(sc, ax=ax, label="log10 Fisher volume density")
    fig.tight_layout(); fig.savefig(outdir / "gp2_fisher_nodes_rho_beta20.png", dpi=220); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    sc = ax.scatter(gp2_nodes.q_beta11, gp2_nodes.q_beta02,
                    c=np.log10(np.maximum(gp2_nodes.fisher_volume_density, 1.0e-30)),
                    s=12, alpha=0.75)
    ax.set_xlabel("q_beta11"); ax.set_ylabel("q_beta02")
    ax.set_title("Fisher-adaptive GP2 nodes: mixed/I2 quadratic coordinates")
    fig.colorbar(sc, ax=ax, label="log10 Fisher volume density")
    fig.tight_layout(); fig.savefig(outdir / "gp2_fisher_nodes_beta11_beta02.png", dpi=220); plt.close(fig)


def plot_compatibility_pca(
    cfg: Config,
    outdir: Path,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    atlas_df: pd.DataFrame,
    source_templates: Dict[str, np.ndarray],
) -> pd.DataFrame:
    """Project complete-F compatibility labels into parent and standard subprotocol PCAs.

    The labels are always identified from the complete-F parent response.  Only the
    response components supplied to PCA change, so the standard-protocol plots test
    whether parent-identifiable regions remain visible in practical experiments.
    """
    del bases  # templates already contain every required protocol component.

    protocol_views: Tuple[Tuple[str, Tuple[str, ...]], ...] = (
        ("ALL", ("F_PARENT",)),
        ("UT", ("UT",)),
        ("BT", ("BT",)),
        ("SH", ("SH",)),
        ("UT_BT", ("UT", "BT")),
        ("UT_SH", ("UT", "SH")),
        ("BT_SH", ("BT", "SH")),
    )

    vectors: List[np.ndarray] = []
    labels: List[str] = []
    models: List[str] = []
    source_indices: List[int] = []
    source_mu0: List[float] = []
    support_zones: List[str] = []
    for model in ALL_MODELS:
        sub = atlas_df[atlas_df.source_model == model].sort_values("source_index")
        for i, (_, row) in enumerate(sub.iterrows()):
            vectors.append(np.asarray(source_templates[model][i], dtype=np.float64))
            labels.append(str(row.primary_region_at_3pct))
            models.append(model)
            source_indices.append(int(row.source_index))
            source_mu0.append(float(row.get("source_mu0_kpa", float("nan"))))
            support_zones.append(str(row.get("biological_support_zone", "UNKNOWN")))

    X_full = np.stack(vectors)
    unique_labels = sorted(set(labels))
    cmap = plt.get_cmap("tab20")
    label_color = {lab: cmap(i % 20) for i, lab in enumerate(unique_labels)}
    summary_rows: List[Dict[str, object]] = []
    panel_payload: List[Tuple[str, np.ndarray, np.ndarray]] = []

    for view_name, modes in protocol_views:
        indices = np.concatenate([
            np.arange(protocol.mode_slices[mode].start, protocol.mode_slices[mode].stop, dtype=int)
            for mode in modes
        ])
        # Normalize after slicing so each subprotocol PCA profiles stiffness using only
        # the measurements available in that subprotocol.
        X = np.stack([scale_free_response(row[indices]) for row in X_full])
        scores, components, ratio = safe_pca(X, cfg.pca_components)
        panel_payload.append((view_name, scores, ratio))

        fig, ax = plt.subplots(figsize=(9.2, 7.0))
        for lab in unique_labels:
            mask = np.array([value == lab for value in labels])
            ax.scatter(
                scores[mask, 0], scores[mask, 1], s=13, alpha=0.75,
                label=lab, color=label_color[lab],
            )
        ax.set_xlabel(f"PC1 ({100*ratio[0]:.2f}%)")
        ax.set_ylabel(f"PC2 ({100*ratio[1]:.2f}%)")
        ax.set_title(
            f"Complete-F compatibility regions projected into {view_name} scale-free PCA"
        )
        ax.legend(fontsize=7, ncol=2)
        fig.tight_layout()
        fig.savefig(outdir / f"compatibility_regions_pca_{view_name}_2d.png", dpi=240)
        plt.close(fig)

        coordinates = pd.DataFrame({
            "protocol_view": view_name,
            "included_modes": "|".join(modes),
            "source_model": models,
            "source_index": source_indices,
            "source_mu0_kpa": source_mu0,
            "biological_support_zone": support_zones,
            "complete_F_primary_region": labels,
            "PC1": scores[:, 0],
            "PC2": scores[:, 1],
            "PC3": scores[:, 2] if scores.shape[1] > 2 else 0.0,
        })
        coordinates.to_csv(
            outdir / f"compatibility_regions_pca_{view_name}_coordinates.csv", index=False
        )
        np.savez_compressed(
            outdir / f"compatibility_regions_pca_{view_name}_model.npz",
            protocol_view=view_name,
            included_modes=np.asarray(modes),
            selected_component_indices=indices,
            mean=np.mean(X, axis=0),
            components=components,
            explained_variance_ratio=ratio,
        )

        summary_rows.append({
            "protocol_view": view_name,
            "included_modes": "|".join(modes),
            "response_dimension": int(X.shape[1]),
            "n_states": int(X.shape[0]),
            "n_complete_F_regions": int(len(unique_labels)),
            "pc1_explained_variance": float(ratio[0]) if len(ratio) > 0 else 0.0,
            "pc2_explained_variance": float(ratio[1]) if len(ratio) > 1 else 0.0,
            "pc3_explained_variance": float(ratio[2]) if len(ratio) > 2 else 0.0,
            "pc1_pc2_cumulative_variance": float(np.sum(ratio[:2])),
            "pc1_pc2_pc3_cumulative_variance": float(np.sum(ratio[:3])),
            "region_label_source": "complete_F_parent_compatibility_atlas",
        })

        # Keep generic aliases for the complete-F parent view.
        if view_name == "ALL":
            coordinates.to_csv(outdir / "compatibility_regions_pca_coordinates.csv", index=False)
            np.savez_compressed(
                outdir / "compatibility_regions_pca_model.npz",
                protocol_view=view_name,
                included_modes=np.asarray(modes),
                selected_component_indices=indices,
                mean=np.mean(X, axis=0),
                components=components,
                explained_variance_ratio=ratio,
            )
            fig, ax = plt.subplots(figsize=(9.2, 7.0))
            for lab in unique_labels:
                mask = np.array([value == lab for value in labels])
                ax.scatter(
                    scores[mask, 0], scores[mask, 1], s=13, alpha=0.75,
                    label=lab, color=label_color[lab],
                )
            ax.set_xlabel(f"PC1 ({100*ratio[0]:.2f}%)")
            ax.set_ylabel(f"PC2 ({100*ratio[1]:.2f}%)")
            ax.set_title("Seven-model complete-F compatibility-set regions in scale-free PCA")
            ax.legend(fontsize=7, ncol=2)
            fig.tight_layout()
            fig.savefig(outdir / "compatibility_regions_pca_2d.png", dpi=240)
            plt.close(fig)

    # One compact comparison sheet uses identical region colors in every panel.
    fig, axes = plt.subplots(2, 4, figsize=(20.0, 10.5))
    for ax, (view_name, scores, ratio) in zip(axes.flat, panel_payload):
        for lab in unique_labels:
            mask = np.array([value == lab for value in labels])
            ax.scatter(scores[mask, 0], scores[mask, 1], s=5, alpha=0.55, color=label_color[lab])
        ax.set_title(f"{view_name}: PC1+PC2={100*np.sum(ratio[:2]):.1f}%")
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
    for ax in axes.flat[len(panel_payload):]:
        ax.axis("off")
    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", markersize=5,
                   markerfacecolor=label_color[lab], markeredgecolor=label_color[lab], label=lab)
        for lab in unique_labels
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=7)
    fig.suptitle(
        "Complete-F compatibility labels viewed through parent and standard loading subprotocols",
        fontsize=14,
    )
    fig.tight_layout(rect=(0.0, 0.08, 1.0, 0.96))
    fig.savefig(outdir / "compatibility_regions_pca_protocol_comparison.png", dpi=240)
    plt.close(fig)

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(outdir / "compatibility_regions_pca_protocol_summary.csv", index=False)
    return summary


def plot_ambient_pca(
    cfg: Config,
    outdir: Path,
    protocol: Protocol,
    observations: np.ndarray,
    metadata: pd.DataFrame,
) -> None:
    if len(observations) == 0:
        return
    scores, _, ratio = safe_pca(observations[:, protocol.atlas_indices], cfg.pca_components)
    labels = metadata.primary_region.astype(str).tolist()
    unique = sorted(set(labels))
    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=(9.0, 6.8))
    for i, label in enumerate(unique):
        idx = np.array([x == label for x in labels])
        ax.scatter(scores[idx, 0], scores[idx, 1], s=25, alpha=0.75, label=label, color=cmap(i % 20))
    ax.set_xlabel(f"PC1 ({100*ratio[0]:.2f}%)")
    ax.set_ylabel(f"PC2 ({100*ratio[1]:.2f}%)")
    ax.set_title("Complete-F ambient observations including NONE_COMPATIBLE")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(outdir / "ambient_observation_compatibility_pca_2d.png", dpi=240)
    plt.close(fig)


def generated_clouds(
    cfg: Config,
    protocol: Protocol,
    bases: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    registry: AtlasSamplerRegistry,
    rng: np.random.Generator,
    outdir: Path,
) -> None:
    if not cfg.run_generated_cloud_visualization:
        return
    vectors: List[np.ndarray] = []
    labels: List[str] = []
    n = cfg.generated_cloud_points_per_model
    for model in ALL_MODELS:
        if model == "NH":
            coords = np.zeros((n, 1))
        elif model in SCALAR_PARENTS:
            lo, hi = scalar_bounds(cfg, model)
            coords = rng.uniform(lo, hi, size=(n, 1))
        elif model == "OGDEN2":
            coords = registry.ogden2_nodes(n, seed_offset=88)[["q_center", "q_gap", "q_weight"]].to_numpy(float)
        else:
            coords = registry.gp2_nodes(n, seed_offset=89)[["q_rho", "q_beta20", "q_beta11", "q_beta02"]].to_numpy(float)
        mu0_values = np.exp(rng.uniform(math.log(cfg.mu0_min_kpa), math.log(cfg.mu0_max_kpa), size=n))
        for coord, mu0_kpa in zip(coords, mu0_values):
            vectors.append(response_scale_from_mu0_kpa(float(mu0_kpa)) * model_template(cfg, model, coord, protocol, bases))
            labels.append(model)
    X = np.stack(vectors)[:, protocol.atlas_indices]
    scores, _, ratio = safe_pca(X, cfg.pca_components)
    fig, ax = plt.subplots(figsize=(8.8, 6.8))
    for model in ALL_MODELS:
        idx = np.array([x == model for x in labels])
        ax.scatter(scores[idx, 0], scores[idx, 1], s=8, alpha=0.45, label=model, color=MODEL_COLORS[model])
    ax.set_xlabel(f"PC1 ({100*ratio[0]:.2f}%)"); ax.set_ylabel(f"PC2 ({100*ratio[1]:.2f}%)")
    ax.set_title("Generated complete-F biological model clouds over the universal log-stiffness range")
    ax.legend(); fig.tight_layout()
    fig.savefig(outdir / "generated_physical_model_clouds_pca_2d.png", dpi=240)
    plt.close(fig)


# =============================================================================
# Main

# =============================================================================
# Main
# =============================================================================


def main() -> None:
    mount_google_drive()
    cfg = resolved_config()
    outdir, logger = setup_output(cfg)
    rng = np.random.default_rng(cfg.seed)
    started = time.time()

    logger.info("=" * 118)
    logger.info("V68.1 PATCHED PRACTICAL STANDARD PROTOCOLS + COMPLETE-F BIOLOGICAL COMPATIBILITY ATLAS")
    logger.info("Output: %s", outdir)
    logger.info("Models: NH, MR, YEOH2, GENT, OGDEN1, OGDEN2, GP2")
    logger.info("Primary region labels are complete hierarchy-closed compatibility sets")
    logger.info("Universal mu0 range: %.4g to %.4g kPa; literature core: %.4g to %.4g kPa", cfg.mu0_min_kpa, cfg.mu0_max_kpa, cfg.mu0_core_min_kpa, cfg.mu0_core_max_kpa)
    logger.info("Standard protocols: UT lambda %.3g–%.3g, equibiaxial BT lambda %.3g–%.3g, SH gamma %.3g–%.3g", cfg.ut_min, cfg.ut_max, cfg.bt_min, cfg.bt_max, cfg.sh_min, cfg.sh_max)
    logger.info("Shape sampling: complete-F scale-profiled Fisher geometry rebuilt on literature cores plus guard margins")
    logger.info("Final atlas noise: %.2f%% bounded brush; ambient NONE_COMPATIBLE enabled", 100*cfg.final_noise_fraction)
    logger.info("=" * 118)

    domain_df = literature_domain_table(cfg)
    domain_df.to_csv(outdir / "literature_bounded_parameter_domains.csv", index=False)
    save_json(outdir / "literature_reference_registry.json", {"references": list(LITERATURE_REFERENCES)})
    pd.DataFrame({
        "mu0_kpa": np.geomspace(cfg.mu0_min_kpa, cfg.mu0_max_kpa, cfg.source_log_scale_strata),
    }).to_csv(outdir / "universal_log_stiffness_grid.csv", index=False)

    protocol = make_protocol(cfg)
    bases = constitutive_bases(protocol)
    logger.info(
        "Parent protocol: %d incompressible F states, %d pressure-free stress components, principal stretches %.3g–%.3g",
        protocol.n_parent_states, len(protocol.atlas_indices),
        cfg.parent_principal_stretch_min, cfg.parent_principal_stretch_max,
    )
    parent_table = pd.DataFrame({
        "state_index": np.arange(protocol.n_parent_states, dtype=int),
        "origin": protocol.parent_state_origin,
        "log_lambda1": protocol.parent_log_principal[:, 0],
        "log_lambda2": protocol.parent_log_principal[:, 1],
        "log_lambda3": protocol.parent_log_principal[:, 2],
        "lambda1": protocol.parent_principal_stretches[:, 0],
        "lambda2": protocol.parent_principal_stretches[:, 1],
        "lambda3": protocol.parent_principal_stretches[:, 2],
        "hencky_norm": np.linalg.norm(protocol.parent_log_principal, axis=1),
        "detF": np.prod(protocol.parent_principal_stretches, axis=1),
    })
    parent_table.to_csv(outdir / "complete_F_parent_protocol_states.csv", index=False)

    sh_root = np.sqrt(1.0 + 0.25 * protocol.sh**2)
    sh_lam_max = sh_root + 0.5 * np.abs(protocol.sh)
    sh_lam_min = 1.0 / sh_lam_max
    protocol_range_audit = pd.DataFrame([
        {"protocol": "UT", "control": "lambda", "control_min": float(protocol.ut.min()), "control_max": float(protocol.ut.max()), "n_points": len(protocol.ut), "principal_stretch_min": float(np.min(np.column_stack([protocol.ut, protocol.ut**-0.5, protocol.ut**-0.5]))), "principal_stretch_max": float(np.max(np.column_stack([protocol.ut, protocol.ut**-0.5, protocol.ut**-0.5])))},
        {"protocol": "BT_EQUIBIAXIAL", "control": "lambda", "control_min": float(protocol.bt.min()), "control_max": float(protocol.bt.max()), "n_points": len(protocol.bt), "principal_stretch_min": float(np.min(np.column_stack([protocol.bt, protocol.bt, protocol.bt**-2.0]))), "principal_stretch_max": float(np.max(np.column_stack([protocol.bt, protocol.bt, protocol.bt**-2.0])))},
        {"protocol": "SH_SIMPLE", "control": "gamma", "control_min": float(protocol.sh.min()), "control_max": float(protocol.sh.max()), "n_points": len(protocol.sh), "principal_stretch_min": float(sh_lam_min.min()), "principal_stretch_max": float(sh_lam_max.max())},
        {"protocol": "ALL_COMPLETE_F", "control": "bounded_principal_stretches", "control_min": float(cfg.parent_principal_stretch_min), "control_max": float(cfg.parent_principal_stretch_max), "n_points": protocol.n_parent_states, "principal_stretch_min": float(protocol.parent_principal_stretches.min()), "principal_stretch_max": float(protocol.parent_principal_stretches.max())},
    ])
    protocol_range_audit.to_csv(outdir / "protocol_range_and_resolution_audit.csv", index=False)

    if not (np.isclose(protocol.ut.min(), cfg.ut_min) and np.isclose(protocol.ut.max(), cfg.ut_max)
            and np.isclose(protocol.bt.min(), cfg.bt_min) and np.isclose(protocol.bt.max(), cfg.bt_max)
            and np.isclose(protocol.sh.min(), cfg.sh_min) and np.isclose(protocol.sh.max(), cfg.sh_max)):
        raise RuntimeError("Practical standard-protocol ranges were not retained exactly.")
    if (protocol.parent_principal_stretches.min() < cfg.parent_principal_stretch_min - 1.0e-12
            or protocol.parent_principal_stretches.max() > cfg.parent_principal_stretch_max + 1.0e-12):
        raise RuntimeError("Complete-F parent exceeds configured principal-stretch bounds.")

    max_i1 = float(np.max(bases[2]))
    if cfg.gent_gamma_max * max_i1 >= 1.0 - cfg.gent_singularity_margin:
        raise ValueError(
            "Configured Gent gamma range enters locking inside the bounded complete-F parent domain; "
            f"require gamma < {(1.0-cfg.gent_singularity_margin)/max_i1:.6g}."
        )

    mu0_audit = build_universal_mu0_scale_audit(cfg)
    mu0_audit.to_csv(outdir / "universal_mu0_scale_identity_audit.csv", index=False)
    if float(mu0_audit.relative_error.max()) > 1.0e-8:
        raise RuntimeError("Universal mu0 scale convention failed for at least one model.")
    logger.info("Universal mu0 convention passed for all models; max relative slope error=%.3e", float(mu0_audit.relative_error.max()))

    exact_audit = build_exact_hierarchy_audit(cfg, protocol, bases)
    exact_audit.to_csv(outdir / "exact_hierarchy_response_identity_audit.csv", index=False)
    checks = exact_nesting_checks(cfg, protocol, bases)
    max_check = max(checks.values())
    if max_check > 1.0e-9:
        raise RuntimeError(f"Exact hierarchy check failed: {checks}")
    logger.info("Exact NH/MR/YEOH2/GENT/OGDEN1/OGDEN2/GP2 identities passed; max error=%.3e", max_check)

    parent_subprotocol_audit = build_parent_subprotocol_consistency_audit(cfg, protocol, bases)
    parent_subprotocol_audit.to_csv(
        outdir / "complete_F_parent_subprotocol_consistency_audit.csv", index=False
    )
    parent_subprotocol_abs_error = float(parent_subprotocol_audit.absolute_magnitude_error.max())
    parent_subprotocol_rel_error = float(parent_subprotocol_audit.relative_magnitude_error.max())
    if parent_subprotocol_rel_error > 1.0e-11:
        raise RuntimeError(
            "Complete-F parent/subprotocol consistency failed; "
            f"max relative error={parent_subprotocol_rel_error:.3e}."
        )
    logger.info(
        "Complete-F parent contains UT/BT/SH exactly; max relative error=%.3e (max absolute %.3e)",
        parent_subprotocol_rel_error, parent_subprotocol_abs_error,
    )

    logger.info("Building scalar Fisher curves, 3-D OGDEN2 pilot, and 4-D GP2 pilot...")
    registry = AtlasSamplerRegistry(cfg, protocol, bases)
    scalar_pilot = registry.scalar.pilot_table()
    og2_pilot = registry.ogden2.pilot_table()
    gp2_pilot = registry.gp2.pilot_table()
    og2_source_nodes = registry.ogden2_nodes(cfg.ogden2_source_points, seed_offset=11)
    gp2_source_nodes = registry.gp2_nodes(cfg.gp2_source_points, seed_offset=17)
    scalar_pilot.to_csv(outdir / "scalar_fisher_geometry.csv", index=False)
    og2_pilot.to_csv(outdir / "ogden2_fisher_pilot_geometry.csv", index=False)
    gp2_pilot.to_csv(outdir / "gp2_fisher_pilot_geometry.csv", index=False)
    og2_source_nodes.to_csv(outdir / "ogden2_fisher_adaptive_source_nodes.csv", index=False)
    gp2_source_nodes.to_csv(outdir / "gp2_fisher_adaptive_source_nodes.csv", index=False)
    anchor_audit = build_literature_anchor_coverage_audit(cfg, registry, og2_source_nodes, gp2_source_nodes)
    anchor_audit.to_csv(outdir / "literature_anchor_coverage_audit.csv", index=False)
    if len(anchor_audit) and float(anchor_audit.nearest_source_coordinate_distance.max()) > 1.0e-10:
        raise RuntimeError("At least one forced literature/hierarchy anchor is absent from the source sampling.")
    logger.info("OGDEN2 pilot=%d, source=%d, median NN=%.4g", len(og2_pilot), len(og2_source_nodes), float(og2_source_nodes.nearest_neighbor_q_distance.median()))
    logger.info("GP2 pilot=%d, source=%d, median NN=%.4g", len(gp2_pilot), len(gp2_source_nodes), float(gp2_source_nodes.nearest_neighbor_q_distance.median()))
    plot_fisher_geometry(outdir, scalar_pilot, og2_pilot, og2_source_nodes, gp2_pilot, gp2_source_nodes)

    logger.info("Building source states and all directed finite-resolution compatibility searches...")
    source_df, source_templates = build_source_states(cfg, protocol, bases, registry)

    reuse_setting = os.environ.get("V68_REUSE_SOURCE_ATLAS", "auto").strip().lower()
    reuse_dir_override = os.environ.get("V68_REUSE_SOURCE_ATLAS_DIR", "").strip()
    reuse_dir = Path(reuse_dir_override) if reuse_dir_override else outdir
    reusable_paths = {
        "source": reuse_dir / "source_states.csv",
        "pairwise": reuse_dir / "pairwise_critical_noise_profiles.csv",
        "atlas": reuse_dir / "seven_model_source_atlas_states.csv",
    }
    reusable_complete = all(path.exists() for path in reusable_paths.values())
    reuse_enabled = (
        reuse_setting in {"1", "true", "yes"}
        or (reuse_setting == "auto" and reusable_complete and os.environ.get("V68_FAST_MODE", "").strip() != "1")
    )
    if reuse_setting in {"1", "true", "yes"} and not reusable_complete:
        raise FileNotFoundError(
            "V68_REUSE_SOURCE_ATLAS=1 but required source-atlas CSV files are missing from "
            f"{reuse_dir}."
        )

    if reuse_enabled:
        logger.info("Reusing completed V68 source atlas from %s; ambient stages will be recomputed.", reuse_dir)
        loaded_source = pd.read_csv(reusable_paths["source"])
        pairwise_df = pd.read_csv(reusable_paths["pairwise"])
        atlas_df = pd.read_csv(reusable_paths["atlas"])
        expected_keys = source_df[["source_model", "source_index"]].astype(str).agg("|".join, axis=1)
        loaded_keys = loaded_source[["source_model", "source_index"]].astype(str).agg("|".join, axis=1)
        if len(loaded_source) != len(source_df) or not np.array_equal(expected_keys.to_numpy(), loaded_keys.to_numpy()):
            raise RuntimeError("Reusable source atlas does not match the current deterministic source-state construction.")
        save_json(outdir / "source_atlas_reuse_audit.json", {
            "reused": True,
            "reuse_directory": str(reuse_dir),
            "source_states": int(len(loaded_source)),
            "pairwise_rows": int(len(pairwise_df)),
            "atlas_rows": int(len(atlas_df)),
        })
    else:
        pairwise_df, atlas_df = build_pairwise_source_atlas(
            cfg, protocol, bases, registry, source_df, source_templates, logger
        )
        save_json(outdir / "source_atlas_reuse_audit.json", {
            "reused": False,
            "reuse_directory": None,
            "source_states": int(len(source_df)),
            "pairwise_rows": int(len(pairwise_df)),
            "atlas_rows": int(len(atlas_df)),
        })

    source_df.to_csv(outdir / "source_states.csv", index=False)
    pairwise_df.to_csv(outdir / "pairwise_critical_noise_profiles.csv", index=False)
    atlas_df.to_csv(outdir / "seven_model_source_atlas_states.csv", index=False)

    source_summary = summarize_source_atlas(atlas_df)
    region_summary = summarize_regions(atlas_df)
    global_region_summary = atlas_df.groupby("compatibility_set_at_3pct").size().reset_index(name="n_states").sort_values("n_states", ascending=False)
    support_summary = atlas_df.groupby(["source_model", "biological_support_zone"]).size().reset_index(name="n_states")
    source_summary.to_csv(outdir / "source_atlas_summary.csv", index=False)
    region_summary.to_csv(outdir / "source_atlas_region_counts.csv", index=False)
    global_region_summary.to_csv(outdir / "global_compatibility_set_counts.csv", index=False)
    support_summary.to_csv(outdir / "biological_support_zone_counts.csv", index=False)
    logger.info("Source atlas summary:\n%s", source_summary.to_string(index=False))
    logger.info("Compatibility-set counts:\n%s", global_region_summary.to_string(index=False))

    if float(source_summary.source_model_inclusion.min()) < 1.0 - 1.0e-12:
        raise RuntimeError("Source-model inclusion fell below 100%.")
    if float(source_summary.noise_monotonicity_rate.min()) < 1.0 - 1.0e-12:
        raise RuntimeError("Noise compatibility monotonicity failed.")
    if bool(np.any(atlas_df.biological_support_zone == "OUTSIDE_CONFIGURED_DOMAIN")):
        raise RuntimeError("At least one constructed source state lies outside the configured biological domain.")
    pca_protocol_summary = plot_compatibility_pca(cfg, outdir, protocol, bases, atlas_df, source_templates)

    ambient_df = pd.DataFrame(); none_revalidation = pd.DataFrame(); known_revalidation = pd.DataFrame(); known_initial_inclusion = None
    if cfg.run_ambient_validation:
        logger.info("Building ambient observation classifier including NONE_COMPATIBLE...")
        classifier = AmbientClassifier(cfg, protocol, bases, registry)
        ambient_df, ambient_obs, ambient_meta = build_ambient_audit(
            cfg, protocol, bases, registry, classifier, rng, logger
        )
        ambient_df.to_csv(outdir / "ambient_observation_audit_initial.csv", index=False)
        np.savez_compressed(
            outdir / "ambient_observations.npz",
            observations=ambient_obs,
            audit_kind=ambient_meta.audit_kind.astype(str).to_numpy(),
            generator=ambient_meta.generator.astype(str).to_numpy(),
            generating_model=ambient_meta.generating_model.astype(str).to_numpy(),
        )

        known_initial = ambient_df[ambient_df.audit_kind == "KNOWN_MODEL"]
        known_initial_inclusion = (
            float(known_initial.source_model_included.mean()) if len(known_initial) else None
        )
        if len(known_initial) and float(known_initial.oracle_source_score.max()) > 1.0 + 1.0e-6:
            raise RuntimeError(
                "At least one bounded known-model observation lies outside its exact generating-model tube; "
                "the audit generator or brush definition is inconsistent."
            )

        ambient_df, ambient_meta, known_revalidation = revalidate_failed_known_model_examples(
            cfg, protocol, bases, registry, ambient_df, ambient_obs, ambient_meta, logger
        )
        known_revalidation.to_csv(outdir / "ambient_known_model_revalidation.csv", index=False)
        ambient_df.to_csv(outdir / "ambient_observation_audit.csv", index=False)
        ambient_summary = summarize_ambient(ambient_df)
        ambient_summary.to_csv(outdir / "ambient_observation_audit_summary.csv", index=False)
        logger.info("Ambient audit summary after patched revalidation:\n%s", ambient_summary.to_string(index=False))
        known = ambient_df[ambient_df.audit_kind == "KNOWN_MODEL"]
        if len(known) and float(known.source_model_included.mean()) < 1.0 - 1.0e-12:
            failed_rows = known.loc[~known.source_model_included, [
                "generating_model", "true_coordinate_json", "true_mu0_kpa",
                "oracle_source_score", "minimum_model_score", "best_model_by_score",
            ]]
            failed_rows.to_csv(outdir / "ambient_known_model_unresolved_failures.csv", index=False)
            raise RuntimeError(
                "Ambient known-model source inclusion remained below 100% after denser patched revalidation. "
                "See ambient_known_model_unresolved_failures.csv."
            )
        none_revalidation = revalidate_none_examples(cfg, protocol, bases, registry, ambient_df, ambient_obs, logger)
        none_revalidation.to_csv(outdir / "certified_none_revalidation.csv", index=False)
        if len(none_revalidation) and not bool(none_revalidation.revalidated_none.all()):
            raise RuntimeError("At least one margin-separated NONE example failed denser revalidation.")
        n_certified = int(np.sum((ambient_df.audit_kind == "OFF_MANIFOLD") & ambient_df.none_compatible & (ambient_df.minimum_model_score >= 1.0 + cfg.certified_none_margin)))
        if n_certified < cfg.minimum_certified_none_examples:
            logger.warning("Only %d margin-separated NONE examples found (requested minimum %d).", n_certified, cfg.minimum_certified_none_examples)
        plot_ambient_pca(cfg, outdir, protocol, ambient_obs, ambient_meta)

    generated_clouds(cfg, protocol, bases, registry, rng, outdir)

    result_summary = {
        "project": cfg.project_name,
        "patch_version": "V68.1 robust target screening + known-model revalidation + safe source-atlas resume",
        "models": list(ALL_MODELS),
        "primary_region_definition": "complete hierarchy-closed compatibility set at 3% on complete-F parent protocol",
        "parent_protocol": {
            "representation": "ordered incompressible principal log stretches; two principal Kirchhoff stress differences",
            "equivalence": "all F modulo rigid rotation and principal-axis permutation for isotropic materials",
            "n_deformation_states": protocol.n_parent_states,
            "response_dimension": len(protocol.atlas_indices),
            "principal_stretch_bounds": [cfg.parent_principal_stretch_min, cfg.parent_principal_stretch_max],
            "standard_protocol_ranges": {
                "UT_lambda": [cfg.ut_min, cfg.ut_max],
                "BT_equibiaxial_lambda": [cfg.bt_min, cfg.bt_max],
                "SH_gamma": [cfg.sh_min, cfg.sh_max],
                "SH_principal_stretch": [float(sh_lam_min.min()), float(sh_lam_max.max())],
            },
            "parent_sampled_principal_stretch_extrema": [float(protocol.parent_principal_stretches.min()), float(protocol.parent_principal_stretches.max())],
            "standard_subprotocols_retained": ["UT", "BT", "SH", "UT_BT", "UT_SH", "BT_SH"],
        },
        "physical_parameter_counts": {"NH":1,"MR":2,"YEOH2":2,"GENT":2,"OGDEN1":2,"OGDEN2":4,"GP2":5},
        "profiled_shape_dimensions": MODEL_SHAPE_DIMS,
        "atlas_type": "single joint biological shape-scale atlas",
        "universal_stiffness_variable": "initial shear modulus mu0 in kPa",
        "universal_mu0_core_kpa": [cfg.mu0_core_min_kpa, cfg.mu0_core_max_kpa],
        "universal_mu0_atlas_kpa": [cfg.mu0_min_kpa, cfg.mu0_max_kpa],
        "scale_sampling": "deterministic stratified logarithmic assignment over Fisher-adaptive shape nodes",
        "biological_support_zones": ["INSIDE_LITERATURE_CORE", "INSIDE_BIOLOGICAL_MARGIN", "OUTSIDE_CONFIGURED_DOMAIN"],
        "literature_references": list(LITERATURE_REFERENCES),
        "exact_hierarchy": {
            "NH": ["MR","YEOH2","GENT","OGDEN1","OGDEN2","GP2"],
            "MR": ["OGDEN2","GP2"],
            "YEOH2": ["GP2"],
            "OGDEN1": ["OGDEN2"],
        },
        "ogden2_parameterization": {
            "physical": ["mu1","alpha1","mu2","alpha2"],
            "universal_scale": "mu0 kPa; response multiplier s=mu0/2",
            "shape": ["w","alpha1","alpha2"],
            "signed_alpha_range": [cfg.ogden_alpha_min,cfg.ogden_alpha_max],
            "exact_mr_stratum": {"alpha1":-2.0,"alpha2":2.0,"weight1":"rho"},
        },
        "gp2_parameterization": {
            "physical": ["C10","C01","C20","C11","C02"],
            "universal_scale": "mu0=2(C10+C01) kPa",
            "shape": ["rho=C01/s","beta20=C20/s","beta11=C11/s","beta02=C02/s"],
            "exact_mr_stratum": "beta20=beta11=beta02=0",
            "exact_yeoh2_stratum": "rho=beta11=beta02=0",
        },
        "gent_parameterization": {"shape":"gamma=1/Jm", "nh_closure":"gamma=0"},
        "exact_hierarchy_checks": checks,
        "universal_mu0_scale_audit_max_relative_error": float(mu0_audit.relative_error.max()),
        "complete_F_parent_subprotocol_max_absolute_error": parent_subprotocol_abs_error,
        "complete_F_parent_subprotocol_max_relative_error": parent_subprotocol_rel_error,
        "literature_anchor_coverage_max_coordinate_distance": float(anchor_audit.nearest_source_coordinate_distance.max()) if len(anchor_audit) else None,
        "compatibility_pca_protocol_views": pca_protocol_summary[["protocol_view", "included_modes", "response_dimension"]].to_dict(orient="records"),
        "compatibility_pca_region_labels": "all PCA views retain complete-F parent compatibility labels",
        "final_noise_fraction": cfg.final_noise_fraction,
        "source_model_inclusion": float(source_summary.source_model_inclusion.min()),
        "source_noise_monotonicity": float(source_summary.noise_monotonicity_rate.min()),
        "none_region_enabled": True,
        "ambient_known_model_initial_inclusion": known_initial_inclusion,
        "ambient_known_model_inclusion": float(ambient_df[ambient_df.audit_kind=="KNOWN_MODEL"].source_model_included.mean()) if len(ambient_df) else None,
        "ambient_known_model_revalidation_count": int(len(known_revalidation)),
        "denser_none_revalidation_passed": bool(none_revalidation.revalidated_none.all()) if len(none_revalidation) else None,
        "runtime_seconds": time.time()-started,
        "config": asdict(cfg), "python": sys.version, "platform": platform.platform(),
    }
    save_json(outdir / "result_summary.json", result_summary)

    manifest_rows: List[Dict[str, object]] = []
    for path in sorted(outdir.iterdir()):
        if path.is_file():
            manifest_rows.append({"name":path.name,"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
    pd.DataFrame(manifest_rows).to_csv(outdir / "manifest.csv", index=False)

    logger.info("=" * 118)
    logger.info("DONE in %.1f s", time.time()-started)
    logger.info("Results saved to: %s", outdir)
    logger.info("Primary regions are complete-F compatibility sets over literature-bounded shapes and a universal absolute stiffness range")
    logger.info("Exact hierarchy includes NH⊂MR⊂{OGDEN2,GP2}, NH⊂YEOH2⊂GP2, NH⊂OGDEN1⊂OGDEN2, and NH⊂GENT")
    logger.info("Open result_summary.json, global_compatibility_set_counts.csv, compatibility_regions_pca_protocol_comparison.png, and compatibility_regions_pca_protocol_summary.csv first.")
    logger.info("=" * 118)


if __name__ == "__main__":
    main()


2026-08-08 23:15:04,845 | INFO | ======================================================================================================================
2026-08-08 23:15:04,846 | INFO | V68.1 PATCHED PRACTICAL STANDARD PROTOCOLS + COMPLETE-F BIOLOGICAL COMPATIBILITY ATLAS
2026-08-08 23:15:04,847 | INFO | Output: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
2026-08-08 23:15:04,848 | INFO | Models: NH, MR, YEOH2, GENT, OGDEN1, OGDEN2, GP2
2026-08-08 23:15:04,849 | INFO | Primary region labels are complete hierarchy-closed compatibility sets
2026-08-08 23:15:04,850 | INFO | Universal mu0 range: 0.05 to 2e+04 kPa; literature core: 0.1 to 1.2e+04 kPa
2026-08-08 23:15:04,851 | INFO | Standard protocols: UT lambda 0.7–1.5, equibiaxial BT lambda 0.8–1.25, SH gamma -0.5–0.5
2026-08-08 23:15:04,851 | INFO | Shape sampling: complete-F scale-profiled Fisher geometry rebuilt on literature cores plus guard margins
2026-08-08 23:15:

KeyboardInterrupt: 

In [ ]:
#!/usr/bin/env python3
"""
V68.2.4 CELL 2 — RESUMABLE HELD-OUT SEARCH REPAIR + PUBLICATION VALIDATION
================================================================

This script validates the completed V68.1 atlas without regenerating or modifying it.
It is designed around the actual V68.1 results and the remaining publication risks:

1. Frozen-output integrity and exact baseline gates.
2. Dense, multi-seed revalidation of all pairwise fits near the 3% boundary,
   plus rare-region and scalar-curve discontinuity cases.
3. Reclassification of the frozen atlas using only improved lower-distance witnesses.
4. Independent held-out model observations, absent from source and target nodes,
   under clean, bounded, clipped-Gaussian, and correlated clipped noise.
5. Complete-F sampling-resolution convergence at 1301, 2601, and 5201 states.
6. Stratified dense revalidation of every margin-separated NONE observation.
7. Quantitative visibility of complete-F labels in ALL and standard-protocol PCA spaces.
8. Scalar critical-noise smoothness and compatibility-region run diagnostics.
9. Threshold-persistence summaries at 0%, 1%, 2%, 3%, and 5%.

The script is resumable. Each expensive stage writes checkpoints into:
    <V68_ROOT>/cell2_publication_validation

Typical Colab use
------------------
%env V68_CELL2_MODE=publication
%run /content/V68_2_4_cell2_publication_validation_resumable.py

Optional stage selection:
%env V68_CELL2_STAGES=integrity,boundary,reclassify,f_convergence,heldout,none,subprotocol,threshold

Useful controls:
V68_ROOT                         frozen V68 results directory
V68_MODULE_PATH                  optional external V68.1 script; embedded fallback is included
V68_CELL2_MODE                   publication | pilot | smoke
V68_CELL2_HELDOUT_PER_MODEL      total held-out observations/model
V68_CELL2_BOUNDARY_SEEDS         comma-separated seed offsets
V68_CELL2_BOUNDARY_MAX           optional cap; 0 means all selected rows
V68_CELL2_FORCE_STAGE            comma-separated stages to recompute

No source-atlas regeneration is performed.
"""

from __future__ import annotations

import gc
import base64
import gzip
import hashlib
import importlib.util
import json
import logging
import math
import os
import platform
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import qmc
from scipy.optimize import differential_evolution

try:
    from sklearn.metrics import (
        balanced_accuracy_score,
        confusion_matrix,
        f1_score,
        silhouette_score,
    )
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
    from sklearn.preprocessing import StandardScaler
except Exception as exc:  # pragma: no cover
    raise RuntimeError(
        "Cell 2 requires scikit-learn. In Colab run: !pip -q install scikit-learn"
    ) from exc


EMBEDDED_V68_MODULE_GZIP_B64 = (
    "..."
    "p3vCnVCHAgA="
)
EMBEDDED_V68_MODULE_SHA256 = "6bad33e20705230a1d8c147260c886379b85d607e27b7c07e5a38b8a1e25c9be"

# =============================================================================
# Cell-2 configuration
# =============================================================================


@dataclass(frozen=True)
class ValidationConfig:
    project_name: str = "V68_2_4_cell2_publication_validation"
    frozen_root: str = (
        "/content/drive/MyDrive/Optimal_Protocol/"
        "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
    )
    output_subdir: str = "cell2_publication_validation"
    mode: str = "publication"
    seed: int = 20260806 + 2200

    # Dense boundary certification.
    boundary_center: float = 0.03
    boundary_half_width: float = 0.01       # certifies baseline scores in [2%, 4%]
    scalar_jump_threshold: float = 0.01
    scalar_jump_relevance_half_width: float = 0.02
    rare_region_max_count: int = 9
    boundary_seed_offsets: Tuple[int, ...] = (701, 1701)
    boundary_scalar_target_points: int = 801
    boundary_ogden2_target_points: int = 5000
    boundary_gp2_target_points: int = 7000
    boundary_coarse_starts: int = 8
    boundary_scale_pool: int = 32
    boundary_shape_pool: int = 48
    boundary_optimizer_maxiter: int = 220
    boundary_optimizer_maxfev: int = 2800
    boundary_checkpoint_every: int = 10
    boundary_max_candidates: int = 0

    # Independent held-out validation.
    heldout_per_model: int = 72
    heldout_noise_regimes: Tuple[str, ...] = (
        "CLEAN",
        "BOUNDED_INTERIOR_2P7",
        "GAUSSIAN_CLIPPED_3PCT",
        "CORRELATED_CLIPPED_3PCT",
    )
    heldout_scalar_target_points: int = 801
    heldout_ogden2_target_points: int = 5000
    heldout_gp2_target_points: int = 7000
    heldout_seed_offset: int = 2701
    heldout_checkpoint_every: int = 5
    heldout_oracle_tolerance: float = 1.0e-8

    # V68.2.4 held-out repair.  The hard publication gate deliberately uses
    # observations with numerical margin from the 3% tube boundary.  Clipped
    # 3% cases remain reported as stress tests rather than hard containment gates.
    heldout_patch_version: str = "V68.2.4_DE_RESCUE_EXACT_ORACLE"
    heldout_hard_gate_regimes: Tuple[str, ...] = ("CLEAN", "BOUNDED_INTERIOR_2P7")
    heldout_rescue_regimes: Tuple[str, ...] = ("CLEAN", "BOUNDED_INTERIOR_2P7")
    heldout_rescue_score_tolerance: float = 1.0e-6
    heldout_de_seed_offsets: Tuple[int, ...] = (4101, 5101)
    heldout_de_popsize: int = 10
    heldout_de_maxiter_first: int = 100
    heldout_de_maxiter_second: int = 180
    heldout_de_tol: float = 1.0e-7

    # Complete-F fixed-witness quadrature convergence.
    f_resolutions: Tuple[int, ...] = (1301, 2601, 5201)
    f_convergence_interior_per_stratum: int = 30
    f_convergence_nonboundary_margin: float = 0.003

    # Dense NONE revalidation.
    none_margin_score: float = 1.20
    none_checkpoint_every: int = 5

    # PCA-space label visibility.
    pca_knn_neighbors: int = 7
    pca_neighbor_purity_neighbors: int = 10
    pca_cv_splits: int = 5
    pca_cv_repeats: int = 5
    pca_rare_min_count: int = 5

    # Publication gates. Boundary-adjacent observations are reported, not forced stable.
    minimum_heldout_inclusion: float = 0.995
    minimum_none_revalidation: float = 1.0
    minimum_f_nonboundary_label_agreement: float = 0.99
    maximum_f_median_abs_score_change: float = 0.0015


ALL_STAGES = (
    "integrity",
    "boundary",
    "reclassify",
    "f_convergence",
    "heldout",
    "none",
    "subprotocol",
    "threshold",
)


def resolved_validation_config() -> ValidationConfig:
    mode = os.environ.get("V68_CELL2_MODE", "publication").strip().lower()
    root = os.environ.get("V68_ROOT", ValidationConfig.frozen_root).strip()
    cfg = ValidationConfig(frozen_root=root, mode=mode)

    seed_text = os.environ.get("V68_CELL2_BOUNDARY_SEEDS", "").strip()
    if seed_text:
        seeds = tuple(int(x.strip()) for x in seed_text.split(",") if x.strip())
        cfg = replace(cfg, boundary_seed_offsets=seeds)

    if os.environ.get("V68_CELL2_HELDOUT_PER_MODEL", "").strip():
        cfg = replace(
            cfg,
            heldout_per_model=int(os.environ["V68_CELL2_HELDOUT_PER_MODEL"]),
        )
    if os.environ.get("V68_CELL2_BOUNDARY_MAX", "").strip():
        cfg = replace(
            cfg,
            boundary_max_candidates=int(os.environ["V68_CELL2_BOUNDARY_MAX"]),
        )

    if mode == "pilot":
        cfg = replace(
            cfg,
            boundary_seed_offsets=(701,),
            boundary_scalar_target_points=601,
            boundary_ogden2_target_points=3600,
            boundary_gp2_target_points=4800,
            heldout_per_model=24,
            heldout_scalar_target_points=601,
            heldout_ogden2_target_points=3600,
            heldout_gp2_target_points=4800,
            f_resolutions=(1301, 2601),
            pca_cv_repeats=2,
        )
    elif mode == "smoke":
        cfg = replace(
            cfg,
            boundary_seed_offsets=(701,),
            boundary_scalar_target_points=121,
            boundary_ogden2_target_points=300,
            boundary_gp2_target_points=400,
            boundary_max_candidates=6,
            heldout_per_model=4,
            heldout_scalar_target_points=121,
            heldout_ogden2_target_points=300,
            heldout_gp2_target_points=400,
            f_resolutions=(1301, 2601),
            f_convergence_interior_per_stratum=2,
            pca_cv_repeats=1,
        )
    elif mode != "publication":
        raise ValueError("V68_CELL2_MODE must be publication, pilot, or smoke.")
    return cfg


# =============================================================================
# General utilities
# =============================================================================


def mount_google_drive() -> None:
    if Path("/content/drive/MyDrive").exists():
        return
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception:
        pass


def setup_logger(outdir: Path) -> logging.Logger:
    outdir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("V68_CELL2")
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(outdir / "cell2_run.log", mode="a")
    fh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.addHandler(fh)
    return logger


def save_json(path: Path, payload: Mapping) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=_json_default)


def _json_default(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def parse_stages() -> Tuple[str, ...]:
    text = os.environ.get("V68_CELL2_STAGES", "").strip()
    if not text:
        return ALL_STAGES
    stages = tuple(x.strip().lower() for x in text.split(",") if x.strip())
    unknown = sorted(set(stages) - set(ALL_STAGES))
    if unknown:
        raise ValueError(f"Unknown Cell-2 stages: {unknown}")
    return stages


def force_stages() -> set[str]:
    text = os.environ.get("V68_CELL2_FORCE_STAGE", "").strip()
    return {x.strip().lower() for x in text.split(",") if x.strip()}


def _materialize_embedded_v68_module() -> Path:
    """Write the bundled frozen V68.1 implementation to a local runtime path."""
    payload = gzip.decompress(base64.b64decode("".join(EMBEDDED_V68_MODULE_GZIP_B64)))
    digest = hashlib.sha256(payload).hexdigest()
    if digest != EMBEDDED_V68_MODULE_SHA256:
        raise RuntimeError(
            f"Embedded V68.1 module checksum mismatch: {digest} != {EMBEDDED_V68_MODULE_SHA256}"
        )
    filename = "V68_1_patched_practical_standard_protocols_complete_F_biological_compatibility_atlas.py"
    candidates = [Path("/content") / filename, Path.cwd() / filename, Path("/tmp") / filename]
    last_error = None
    for target in candidates:
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != digest:
                target.write_bytes(payload)
            print(f"[V68 Cell 2] materialized bundled V68.1 module at {target}", flush=True)
            return target.resolve()
        except OSError as exc:
            last_error = exc
    raise FileNotFoundError(f"Could not materialize the embedded V68.1 module: {last_error}")


def locate_v68_module(root: Path) -> Path:
    """Locate an external V68.1 module or use the self-contained bundled copy."""
    filename = "V68_1_patched_practical_standard_protocols_complete_F_biological_compatibility_atlas.py"
    explicit = os.environ.get("V68_MODULE_PATH", "").strip()
    script_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

    candidates = [
        Path(explicit).expanduser() if explicit else None,
        script_dir / filename,
        script_dir / "v68_cell2" / filename,
        Path("/content/v68_cell2") / filename,
        Path("/content") / filename,
        Path.cwd() / filename,
        root / filename,
        root.parent / filename,
        Path("/content/drive/MyDrive/Optimal_Protocol") / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate.resolve()

    # The script is self-contained: this path works even when its full source was pasted
    # directly into a Colab notebook cell and no sibling file was extracted.
    return _materialize_embedded_v68_module()


def import_v68(path: Path):
    spec = importlib.util.spec_from_file_location("v68_1_frozen_module", path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def read_csv_required(root: Path, name: str) -> pd.DataFrame:
    path = root / name
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def coordinate_from_json(value: object) -> np.ndarray:
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=np.float64)
    return np.asarray(value, dtype=np.float64)


def row_key(source_model: str, source_index: int, target_model: str) -> str:
    return f"{source_model}:{int(source_index)}->{target_model}"


def flush_rows(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, index=False)


def append_checkpoint(path: Path, row: Mapping[str, object]) -> None:
    frame = pd.DataFrame([dict(row)])
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def reconstruct_source_response(M, cfg, protocol, bases, row: pd.Series) -> np.ndarray:
    model = str(row.source_model)
    coord = coordinate_from_json(row.coordinate_json)
    template = M.model_template(cfg, model, coord, protocol, bases)
    scale = M.response_scale_from_mu0_kpa(float(row.source_mu0_kpa))
    return scale * template


def source_lookup(source_df: pd.DataFrame) -> Dict[Tuple[str, int], pd.Series]:
    return {
        (str(row.source_model), int(row.source_index)): row
        for _, row in source_df.iterrows()
    }


# =============================================================================
# Stage 1: frozen integrity and baseline gates
# =============================================================================


def stage_integrity(
    V, cfg: ValidationConfig, root: Path, outdir: Path, logger: logging.Logger
) -> Dict[str, object]:
    required = [
        "result_summary.json",
        "source_states.csv",
        "seven_model_source_atlas_states.csv",
        "pairwise_critical_noise_profiles.csv",
        "source_atlas_summary.csv",
        "global_compatibility_set_counts.csv",
        "ambient_observation_audit.csv",
        "ambient_observations.npz",
        "certified_none_revalidation.csv",
        "complete_F_parent_protocol_states.csv",
        "compatibility_regions_pca_protocol_summary.csv",
    ]
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Frozen V68 output is incomplete: {missing}")

    source = read_csv_required(root, "source_states.csv")
    atlas = read_csv_required(root, "seven_model_source_atlas_states.csv")
    pairwise = read_csv_required(root, "pairwise_critical_noise_profiles.csv")
    ambient = read_csv_required(root, "ambient_observation_audit.csv")
    none = read_csv_required(root, "certified_none_revalidation.csv")
    parent = read_csv_required(root, "complete_F_parent_protocol_states.csv")

    with (root / "result_summary.json").open("r", encoding="utf-8") as f:
        result_summary = json.load(f)

    checks = {
        "source_rows_equal_atlas_rows": len(source) == len(atlas),
        "expected_source_state_count_2631": len(source) == 2631,
        "expected_pairwise_count": len(pairwise) == len(source) * (len(V.ALL_MODELS) - 1),
        "source_model_inclusion_100pct": bool(atlas.source_model_included.all()),
        "source_assignment_100pct": bool(atlas.atlas_point_assigned.all()),
        "noise_monotonicity_100pct": bool(atlas.noise_compatibility_monotonic.all()),
        "known_model_inclusion_100pct": bool(
            ambient.loc[ambient.audit_kind == "KNOWN_MODEL", "source_model_included"].all()
        ),
        "none_revalidation_all_pass": bool(len(none) > 0 and none.revalidated_none.all()),
        "parent_state_count_2601": len(parent) == 2601,
        "parent_detF_error_below_1e_12": float(np.max(np.abs(parent.detF - 1.0))) < 1.0e-12,
        "parent_stretches_inside_bounds": bool(
            parent[["lambda1", "lambda2", "lambda3"]].min().min() >= 0.5 - 1.0e-12
            and parent[["lambda1", "lambda2", "lambda3"]].max().max() <= 2.0 + 1.0e-12
        ),
    }
    if not all(checks.values()):
        failed = [k for k, v in checks.items() if not v]
        raise RuntimeError(f"Frozen-atlas integrity failed: {failed}")

    file_rows = []
    for name in required:
        path = root / name
        file_rows.append({
            "name": name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
    pd.DataFrame(file_rows).to_csv(outdir / "frozen_input_hashes.csv", index=False)

    summary = {
        "checks": checks,
        "n_source_states": len(source),
        "n_pairwise_rows": len(pairwise),
        "n_ambient_rows": len(ambient),
        "n_certified_none": len(none),
        "n_parent_states": len(parent),
        "frozen_result_summary": result_summary,
    }
    save_json(outdir / "integrity_summary.json", summary)
    logger.info("Integrity passed: 2631 source states, 15786 pairwise fits, 100%% known inclusion.")
    return summary


# =============================================================================
# Stage 2: dense multi-seed boundary certification
# =============================================================================


def select_boundary_candidates(
    cfg: ValidationConfig,
    source_df: pd.DataFrame,
    atlas_df: pd.DataFrame,
    pairwise_df: pd.DataFrame,
) -> pd.DataFrame:
    candidates: Dict[str, Dict[str, object]] = {}

    def add(row: pd.Series, reason: str) -> None:
        key = row_key(row.source_model, row.source_index, row.target_model)
        payload = candidates.get(key, dict(row))
        old = str(payload.get("selection_reason", ""))
        reasons = {x for x in old.split("|") if x}
        reasons.add(reason)
        payload["selection_reason"] = "|".join(sorted(reasons))
        payload["candidate_key"] = key
        candidates[key] = payload

    lo = cfg.boundary_center - cfg.boundary_half_width
    hi = cfg.boundary_center + cfg.boundary_half_width
    for _, row in pairwise_df[pairwise_df.critical_noise_fraction.between(lo, hi)].iterrows():
        add(row, "BASELINE_2_TO_4PCT")

    counts = atlas_df.compatibility_set_at_3pct.value_counts()
    rare_labels = set(counts[counts <= cfg.rare_region_max_count].index.astype(str))
    rare_states = atlas_df[atlas_df.compatibility_set_at_3pct.astype(str).isin(rare_labels)][
        ["source_model", "source_index"]
    ]
    rare_pairs = pairwise_df.merge(rare_states, on=["source_model", "source_index"], how="inner")
    for _, row in rare_pairs.iterrows():
        add(row, "RARE_REGION_SOURCE")

    scalar = source_df[source_df.source_model.isin(["MR", "YEOH2", "GENT", "OGDEN1"])][
        ["source_model", "source_index", "coordinate_0"]
    ]
    merged = pairwise_df.merge(scalar, on=["source_model", "source_index"], how="inner")
    for (source_model, target_model), group in merged.groupby(["source_model", "target_model"]):
        group = group.sort_values("coordinate_0").reset_index(drop=True)
        scores = group.critical_noise_fraction.to_numpy(float)
        for i, jump in enumerate(np.abs(np.diff(scores))):
            if jump <= cfg.scalar_jump_threshold:
                continue
            if min(
                abs(scores[i] - cfg.boundary_center),
                abs(scores[i + 1] - cfg.boundary_center),
            ) > cfg.scalar_jump_relevance_half_width:
                continue
            add(group.iloc[i], "SCALAR_NEAR_BOUNDARY_JUMP")
            add(group.iloc[i + 1], "SCALAR_NEAR_BOUNDARY_JUMP")

    out = pd.DataFrame(candidates.values())
    if len(out) == 0:
        raise RuntimeError("No boundary candidates were selected.")
    out["distance_to_3pct"] = np.abs(out.critical_noise_fraction - cfg.boundary_center)
    out = out.sort_values(
        ["distance_to_3pct", "source_model", "source_index", "target_model"]
    ).reset_index(drop=True)

    if cfg.boundary_max_candidates > 0 and len(out) > cfg.boundary_max_candidates:
        # Preserve target-model coverage before taking closest remaining cases.
        chosen: List[int] = []
        for _, group in out.groupby("target_model", sort=True):
            chosen.extend(group.head(1).index.tolist())
        remaining = [i for i in out.index if i not in set(chosen)]
        need = max(0, cfg.boundary_max_candidates - len(chosen))
        chosen.extend(remaining[:need])
        out = out.loc[sorted(set(chosen))].reset_index(drop=True)
    return out


def boundary_validation_config(V, base_cfg, cfg: ValidationConfig):
    return replace(
        base_cfg,
        coarse_starts=cfg.boundary_coarse_starts,
        scale_refinement_pool=cfg.boundary_scale_pool,
        shape_screening_pool=cfg.boundary_shape_pool,
        optimizer_maxiter=cfg.boundary_optimizer_maxiter,
        optimizer_maxfev=cfg.boundary_optimizer_maxfev,
    )


def run_boundary_seed(
    V,
    val_cfg: ValidationConfig,
    model_cfg,
    protocol,
    bases,
    registry,
    source_df: pd.DataFrame,
    candidates: pd.DataFrame,
    seed_offset: int,
    checkpoint_path: Path,
    logger: logging.Logger,
) -> pd.DataFrame:
    existing = pd.read_csv(checkpoint_path) if checkpoint_path.exists() else pd.DataFrame()
    done = set(existing.candidate_key.astype(str)) if len(existing) else set()
    lookup = source_lookup(source_df)
    response_cache: Dict[Tuple[str, int], np.ndarray] = {}

    logger.info(
        "Boundary seed %d: building dense assets scalar=%d, OGDEN2=%d, GP2=%d...",
        seed_offset,
        val_cfg.boundary_scalar_target_points,
        val_cfg.boundary_ogden2_target_points,
        val_cfg.boundary_gp2_target_points,
    )
    solver = V.ContinuousModelSolver(
        model_cfg,
        protocol,
        bases,
        registry,
        scalar_target_points=val_cfg.boundary_scalar_target_points,
        ogden2_target_points=val_cfg.boundary_ogden2_target_points,
        gp2_target_points=val_cfg.boundary_gp2_target_points,
        observation_mode=False,
        seed_offset=seed_offset,
    )

    pending = candidates[~candidates.candidate_key.astype(str).isin(done)]
    for number, (_, candidate) in enumerate(pending.iterrows(), start=1):
        sm = str(candidate.source_model)
        si = int(candidate.source_index)
        tm = str(candidate.target_model)
        skey = (sm, si)
        if skey not in response_cache:
            response_cache[skey] = reconstruct_source_response(
                V, model_cfg, protocol, bases, lookup[skey]
            )
        source = response_cache[skey]
        fit = solver.solve_nh(source) if tm == "NH" else solver.solve(source, tm)
        row = {
            "candidate_key": str(candidate.candidate_key),
            "selection_reason": str(candidate.selection_reason),
            "source_model": sm,
            "source_index": si,
            "target_model": tm,
            "seed_offset": int(seed_offset),
            "baseline_score": float(candidate.critical_noise_fraction),
            "dense_score": float(fit["score"]),
            "dense_target_coordinate_json": json.dumps(fit["best_coordinate"]),
            "dense_target_mu0_kpa": float(fit["best_mu0_kpa"]),
            "dense_target_scale_kpa": float(fit["best_scale"]),
            "optimizer_success": bool(fit["optimizer_success"]),
            "solution_source": str(fit["solution_source"]),
            "coarse_best_score": float(fit.get("coarse_best_score", np.nan)),
            "screened_candidate_count": int(fit.get("screened_candidate_count", 0)),
            "continuous_refinement_attempted": bool(
                fit.get("continuous_refinement_attempted", False)
            ),
            "continuous_refinement_converged": bool(
                fit.get("continuous_refinement_converged", True)
            ),
        }
        append_checkpoint(checkpoint_path, row)
        if number % val_cfg.boundary_checkpoint_every == 0 or number == len(pending):
            logger.info(
                "  boundary seed %d progress: %d/%d",
                seed_offset,
                number,
                len(pending),
            )

    del solver
    gc.collect()
    return pd.read_csv(checkpoint_path)


def infer_baseline_witness_model(solution_source: str, source_model: str, target_model: str) -> str:
    """Return a model whose template is the stored exact witness.

    Hierarchy propagation sometimes stores a subset coordinate in a superset row
    (for example a YEOH2 coordinate in a GP2 row).  The subset response is still an
    exact member of the superset manifold, so it is the correct fixed witness for
    complete-F quadrature convergence.
    """
    text = str(solution_source).lower()
    prefix = "inherited_from_exact_"
    suffix = "_subset"
    if text.startswith(prefix) and text.endswith(suffix):
        name = text[len(prefix):-len(suffix)].upper()
        return name
    if text.startswith("exact_") or "nested_in" in text:
        return str(source_model)
    return str(target_model)


def valid_coordinate_json(value: object, expected_dim: int) -> bool:
    try:
        q = coordinate_from_json(value)
    except Exception:
        return False
    return bool(q.size == expected_dim and np.all(np.isfinite(q)))


def combine_boundary_results(
    cfg: ValidationConfig,
    candidates: pd.DataFrame,
    seed_results: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    candidate_lookup = candidates.set_index("candidate_key")
    for key, group in seed_results.groupby("candidate_key", sort=False):
        candidate = candidate_lookup.loc[str(key)]
        dense_best_idx = group.dense_score.astype(float).idxmin()
        dense_best = group.loc[dense_best_idx]
        baseline = float(candidate.critical_noise_fraction)
        dense_best_score = float(dense_best.dense_score)
        if baseline <= dense_best_score:
            certified_score = baseline
            best_coordinate = str(candidate.best_target_coordinate_json)
            best_mu0 = float(candidate.best_target_mu0_kpa)
            witness = "FROZEN_BASELINE_WITNESS"
            witness_model = infer_baseline_witness_model(
                str(candidate.solution_source),
                str(candidate.source_model),
                str(candidate.target_model),
            )
        else:
            certified_score = dense_best_score
            best_coordinate = str(dense_best.dense_target_coordinate_json)
            best_mu0 = float(dense_best.dense_target_mu0_kpa)
            witness = f"DENSE_SEED_{int(dense_best.seed_offset)}"
            witness_model = str(candidate.target_model)
        dense_labels = group.dense_score.astype(float) <= cfg.boundary_center
        rows.append({
            "candidate_key": str(key),
            "selection_reason": str(candidate.selection_reason),
            "source_model": str(candidate.source_model),
            "source_index": int(candidate.source_index),
            "target_model": str(candidate.target_model),
            "baseline_score": baseline,
            "dense_best_score": dense_best_score,
            "certified_score": min(baseline, dense_best_score),
            "score_reduction": baseline - min(baseline, dense_best_score),
            "best_target_coordinate_json": best_coordinate,
            "best_target_mu0_kpa": best_mu0,
            "baseline_solution_source": str(candidate.solution_source),
            "witness_template_model": witness_model,
            "witness_source": witness,
            "dense_seed_score_min": float(group.dense_score.min()),
            "dense_seed_score_max": float(group.dense_score.max()),
            "dense_seed_score_spread": float(group.dense_score.max() - group.dense_score.min()),
            "dense_seed_label_agreement": bool(dense_labels.nunique() == 1),
            "baseline_compatible": bool(baseline <= cfg.boundary_center),
            "certified_compatible": bool(min(baseline, dense_best_score) <= cfg.boundary_center),
            "all_dense_optimizer_success": bool(group.optimizer_success.all()),
        })
    return pd.DataFrame(rows).sort_values(
        ["source_model", "source_index", "target_model"]
    ).reset_index(drop=True)


def stage_boundary(
    V,
    vcfg: ValidationConfig,
    base_cfg,
    protocol,
    bases,
    registry,
    root: Path,
    outdir: Path,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    final_path = outdir / "boundary_certification_summary.csv"
    candidates_path = outdir / "boundary_certification_candidates.csv"
    if final_path.exists() and candidates_path.exists() and not force:
        logger.info("Boundary certification already exists; loading checkpointed result.")
        return pd.read_csv(candidates_path), pd.read_csv(final_path)

    source = read_csv_required(root, "source_states.csv")
    atlas = read_csv_required(root, "seven_model_source_atlas_states.csv")
    pairwise = read_csv_required(root, "pairwise_critical_noise_profiles.csv")
    candidates = select_boundary_candidates(vcfg, source, atlas, pairwise)
    candidates.to_csv(candidates_path, index=False)
    logger.info("Selected %d directed pairwise fits for dense boundary certification.", len(candidates))

    model_cfg = boundary_validation_config(V, base_cfg, vcfg)
    result_frames = []
    for seed in vcfg.boundary_seed_offsets:
        checkpoint = outdir / f"boundary_seed_{seed}_checkpoint.csv"
        result_frames.append(
            run_boundary_seed(
                V,
                vcfg,
                model_cfg,
                protocol,
                bases,
                registry,
                source,
                candidates,
                seed,
                checkpoint,
                logger,
            )
        )
    seed_results = pd.concat(result_frames, ignore_index=True)
    seed_results.to_csv(outdir / "boundary_all_seed_results.csv", index=False)
    summary = combine_boundary_results(vcfg, candidates, seed_results)
    summary.to_csv(final_path, index=False)

    metrics = {
        "n_candidates": len(summary),
        "n_scores_improved": int(np.sum(summary.score_reduction > 1.0e-10)),
        "n_new_compatible_witnesses": int(
            np.sum((~summary.baseline_compatible) & summary.certified_compatible)
        ),
        "dense_seed_label_agreement": float(summary.dense_seed_label_agreement.mean()),
        "median_dense_seed_score_spread": float(summary.dense_seed_score_spread.median()),
        "maximum_dense_seed_score_spread": float(summary.dense_seed_score_spread.max()),
        "all_optimizer_success_rate": float(summary.all_dense_optimizer_success.mean()),
    }
    save_json(outdir / "boundary_certification_metrics.json", metrics)
    logger.info(
        "Boundary certification complete: %d improved fits; %d new <=3%% witnesses.",
        metrics["n_scores_improved"],
        metrics["n_new_compatible_witnesses"],
    )
    return candidates, summary


# =============================================================================
# Stage 3: atlas reclassification from improved witnesses
# =============================================================================


def stage_reclassify(
    V,
    cfg: ValidationConfig,
    root: Path,
    outdir: Path,
    boundary_summary: pd.DataFrame,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    output_path = outdir / "certified_source_atlas_states.csv"
    metrics_path = outdir / "certified_atlas_change_summary.json"
    if output_path.exists() and metrics_path.exists() and not force:
        with metrics_path.open("r", encoding="utf-8") as f:
            return pd.read_csv(output_path), json.load(f)

    atlas = read_csv_required(root, "seven_model_source_atlas_states.csv")
    updates = {
        row_key(row.source_model, row.source_index, row.target_model): float(row.certified_score)
        for _, row in boundary_summary.iterrows()
    }
    rows: List[Dict[str, object]] = []
    for _, row in atlas.iterrows():
        new = dict(row)
        scores: Dict[str, float] = {}
        for model in V.ALL_MODELS:
            col = f"critical_noise_to_{model.lower()}"
            score = float(row[col])
            key = row_key(str(row.source_model), int(row.source_index), model)
            if key in updates:
                score = min(score, updates[key])
            scores[model] = score
            new[f"certified_critical_noise_to_{model.lower()}"] = score
        compatible = [
            model
            for model in V.ALL_MODELS
            if scores[model] <= cfg.boundary_center + 3.0e-8
        ]
        compatible = list(V.hierarchy_closure(compatible))
        region, signature, subtype = V.region_label_from_compatibility(compatible)
        minimal = V.minimal_compatible_models(compatible)
        new.update({
            "certified_primary_region_at_3pct": region,
            "certified_compatibility_set_at_3pct": signature,
            "certified_minimal_adequate_models_at_3pct": V.ordered_signature(minimal),
            "certified_collision_subtype_at_3pct": subtype,
            "certified_n_compatible_models_at_3pct": len(compatible),
            "certified_label_changed": bool(signature != str(row.compatibility_set_at_3pct)),
        })
        rows.append(new)
    certified = pd.DataFrame(rows)
    certified.to_csv(output_path, index=False)

    changes = certified[certified.certified_label_changed]
    transition = (
        changes.groupby(
            ["compatibility_set_at_3pct", "certified_compatibility_set_at_3pct"]
        )
        .size()
        .reset_index(name="n_states")
        .sort_values("n_states", ascending=False)
    )
    transition.to_csv(outdir / "certified_atlas_label_transitions.csv", index=False)
    counts = (
        certified.groupby("certified_compatibility_set_at_3pct")
        .size()
        .reset_index(name="n_states")
        .sort_values("n_states", ascending=False)
    )
    counts.to_csv(outdir / "certified_global_compatibility_set_counts.csv", index=False)
    metrics = {
        "n_states": len(certified),
        "n_label_changes": int(certified.certified_label_changed.sum()),
        "label_change_fraction": float(certified.certified_label_changed.mean()),
        "n_original_regions": int(certified.compatibility_set_at_3pct.nunique()),
        "n_certified_regions": int(certified.certified_compatibility_set_at_3pct.nunique()),
        "source_model_inclusion_after_certification": bool(
            all(
                str(row.source_model)
                in str(row.certified_compatibility_set_at_3pct).split("|")
                for _, row in certified.iterrows()
            )
        ),
    }
    save_json(metrics_path, metrics)
    logger.info(
        "Certified reclassification changed %d/%d source labels.",
        metrics["n_label_changes"],
        metrics["n_states"],
    )
    return certified, metrics


# =============================================================================
# Stage 4: complete-F sampling-resolution convergence
# =============================================================================


def select_f_convergence_cases(
    V,
    cfg: ValidationConfig,
    source_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    boundary_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for _, row in boundary_summary.iterrows():
        rows.append({
            "case_source": "BOUNDARY_CERTIFICATION",
            "source_model": row.source_model,
            "source_index": int(row.source_index),
            "target_model": row.target_model,
            "reference_score": float(row.certified_score),
            "best_target_coordinate_json": row.best_target_coordinate_json,
            "best_target_mu0_kpa": float(row.best_target_mu0_kpa),
            "witness_template_model": str(row.witness_template_model),
        })

    rng = np.random.default_rng(cfg.seed + 44)
    strata = {
        "LOW": pairwise[pairwise.critical_noise_fraction < 0.015],
        "MID": pairwise[pairwise.critical_noise_fraction.between(0.045, 0.08)],
        "HIGH": pairwise[pairwise.critical_noise_fraction > 0.10],
    }
    boundary_keys = {
        row_key(r.source_model, r.source_index, r.target_model)
        for _, r in boundary_summary.iterrows()
    }
    for label, frame in strata.items():
        frame = frame[
            ~frame.apply(
                lambda r: row_key(r.source_model, r.source_index, r.target_model)
                in boundary_keys,
                axis=1,
            )
        ]
        n = min(cfg.f_convergence_interior_per_stratum, len(frame))
        if n == 0:
            continue
        chosen = frame.iloc[rng.choice(len(frame), size=n, replace=False)]
        for _, row in chosen.iterrows():
            rows.append({
                "case_source": f"INTERIOR_{label}",
                "source_model": row.source_model,
                "source_index": int(row.source_index),
                "target_model": row.target_model,
                "reference_score": float(row.critical_noise_fraction),
                "best_target_coordinate_json": row.best_target_coordinate_json,
                "best_target_mu0_kpa": float(row.best_target_mu0_kpa),
                "witness_template_model": infer_baseline_witness_model(
                    str(row.solution_source), str(row.source_model), str(row.target_model)
                ),
            })
    out = pd.DataFrame(rows).drop_duplicates(
        ["source_model", "source_index", "target_model"]
    )
    source_map = source_lookup(source_df)
    repaired = []
    for _, row in out.iterrows():
        witness_model = str(row.witness_template_model)
        expected_dim = int(V.MODEL_SHAPE_DIMS[witness_model])
        coordinate_json = row.best_target_coordinate_json
        mu0 = float(row.best_target_mu0_kpa)
        if not valid_coordinate_json(coordinate_json, expected_dim):
            src = source_map[(str(row.source_model), int(row.source_index))]
            witness_model = str(row.source_model)
            coordinate_json = str(src.coordinate_json)
            mu0 = float(src.source_mu0_kpa)
        repaired.append((witness_model, coordinate_json, mu0))
    out["witness_template_model"] = [x[0] for x in repaired]
    out["best_target_coordinate_json"] = [x[1] for x in repaired]
    out["best_target_mu0_kpa"] = [x[2] for x in repaired]
    out["case_key"] = out.apply(
        lambda r: row_key(r.source_model, r.source_index, r.target_model), axis=1
    )
    return out.reset_index(drop=True)


def f_resolution_config(base_cfg, resolution: int):
    if resolution == int(base_cfg.parent_f_states):
        return base_cfg
    candidate_points = int(2 ** math.ceil(math.log2(max(8 * resolution, 1024))))
    boundary_points = max(96, int(round(base_cfg.parent_f_boundary_points * resolution / base_cfg.parent_f_states)))
    return replace(
        base_cfg,
        parent_f_states=int(resolution),
        parent_f_candidate_points=candidate_points,
        parent_f_boundary_points=boundary_points,
    )


def stage_f_convergence(
    V,
    cfg: ValidationConfig,
    base_cfg,
    root: Path,
    outdir: Path,
    boundary_summary: pd.DataFrame,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    final_path = outdir / "complete_F_resolution_convergence.csv"
    metrics_path = outdir / "complete_F_resolution_convergence_summary.json"
    if final_path.exists() and metrics_path.exists() and not force:
        with metrics_path.open("r", encoding="utf-8") as f:
            return pd.read_csv(final_path), json.load(f)

    source = read_csv_required(root, "source_states.csv")
    pairwise = read_csv_required(root, "pairwise_critical_noise_profiles.csv")
    lookup = source_lookup(source)
    cases = select_f_convergence_cases(V, cfg, source, pairwise, boundary_summary)
    cases.to_csv(outdir / "complete_F_resolution_cases.csv", index=False)
    logger.info("Complete-F convergence: %d fixed source-target witnesses.", len(cases))

    result_rows: List[Dict[str, object]] = []
    for resolution in cfg.f_resolutions:
        logger.info("  building complete-F protocol with %d states...", resolution)
        rcfg = f_resolution_config(base_cfg, int(resolution))
        protocol = V.make_protocol(rcfg)
        bases = V.constitutive_bases(protocol)
        for _, case in cases.iterrows():
            source_row = lookup[(str(case.source_model), int(case.source_index))]
            source_response = reconstruct_source_response(
                V, rcfg, protocol, bases, source_row
            )
            target_coord = coordinate_from_json(case.best_target_coordinate_json)
            target_template = V.model_template(
                rcfg, str(case.witness_template_model), target_coord, protocol, bases
            )
            target_response = V.response_scale_from_mu0_kpa(
                float(case.best_target_mu0_kpa)
            ) * target_template
            score = V.required_noise_fraction(
                rcfg, protocol, source_response, target_response
            )
            result_rows.append({
                "case_key": str(case.case_key),
                "case_source": str(case.case_source),
                "source_model": str(case.source_model),
                "source_index": int(case.source_index),
                "target_model": str(case.target_model),
                "witness_template_model": str(case.witness_template_model),
                "resolution": int(resolution),
                "fixed_witness_score": float(score),
                "compatible_at_3pct": bool(score <= cfg.boundary_center),
                "reference_optimized_score": float(case.reference_score),
            })
        del protocol, bases
        gc.collect()

    result = pd.DataFrame(result_rows)
    result.to_csv(final_path, index=False)
    pivot = result.pivot(index="case_key", columns="resolution", values="fixed_witness_score")
    highest = max(cfg.f_resolutions)
    baseline = 2601 if 2601 in cfg.f_resolutions else sorted(cfg.f_resolutions)[-2]
    delta = (pivot[highest] - pivot[baseline]).abs()
    ref = cases.set_index("case_key").reference_score
    nonboundary = (ref - cfg.boundary_center).abs() >= cfg.f_convergence_nonboundary_margin
    label_high = pivot[highest] <= cfg.boundary_center
    label_base = pivot[baseline] <= cfg.boundary_center
    agreement = float((label_high[nonboundary] == label_base[nonboundary]).mean()) if nonboundary.any() else float("nan")
    metrics = {
        "n_cases": len(cases),
        "resolutions": list(cfg.f_resolutions),
        "baseline_resolution": baseline,
        "highest_resolution": highest,
        "median_absolute_score_change": float(delta.median()),
        "p95_absolute_score_change": float(delta.quantile(0.95)),
        "maximum_absolute_score_change": float(delta.max()),
        "nonboundary_case_count": int(nonboundary.sum()),
        "nonboundary_label_agreement": agreement,
        "pass_median_score_change": bool(
            float(delta.median()) <= cfg.maximum_f_median_abs_score_change
        ),
        "pass_nonboundary_label_agreement": bool(
            np.isnan(agreement)
            or agreement >= cfg.minimum_f_nonboundary_label_agreement
        ),
    }
    save_json(metrics_path, metrics)
    logger.info(
        "F-resolution convergence: median |2601-5201|=%.4g; nonboundary agreement=%.4f.",
        metrics["median_absolute_score_change"],
        metrics["nonboundary_label_agreement"],
    )
    return result, metrics


# =============================================================================
# Stage 5: independent held-out observations
# =============================================================================


def normalized_nearest_distance(point: np.ndarray, nodes: np.ndarray) -> float:
    point = np.asarray(point, dtype=float)
    nodes = np.asarray(nodes, dtype=float)
    if nodes.ndim == 1:
        nodes = nodes[:, None]
    if point.ndim == 0:
        point = point[None]
    return float(np.sqrt(np.min(np.sum((nodes - point[None, :]) ** 2, axis=1))))


def generate_heldout_coordinates(
    V,
    cfg: ValidationConfig,
    model_cfg,
    registry,
    classifier,
) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed + 500)
    rows: List[Dict[str, object]] = []
    n = cfg.heldout_per_model
    regimes = cfg.heldout_noise_regimes

    def regime_for(i: int) -> str:
        return regimes[i % len(regimes)]

    for model_index, model in enumerate(V.ALL_MODELS):
        if model == "NH":
            coords = np.zeros((n, 1), dtype=float)
        elif model in V.SCALAR_PARENTS:
            lo, hi = V.scalar_bounds(model_cfg, model)
            sobol = qmc.Sobol(d=1, scramble=True, seed=cfg.seed + 600 + model_index)
            m = int(math.ceil(math.log2(max(n, 2))))
            u = sobol.random_base2(m)[:n, 0]
            # Shift away from exact nested endpoints while retaining the full guard domain.
            u = np.clip((u + 0.5 / max(n, 2)) % 1.0, 1.0e-6, 1.0 - 1.0e-6)
            coords = (lo + (hi - lo) * u)[:, None]
        else:
            d = V.MODEL_SHAPE_DIMS[model]
            sobol = qmc.Sobol(d=d, scramble=True, seed=cfg.seed + 600 + model_index)
            m = int(math.ceil(math.log2(max(n, 2))))
            coords = sobol.random_base2(m)[:n]
            coords = np.clip(coords, 1.0e-6, 1.0 - 1.0e-6)

        if model == "NH":
            source_nodes = np.zeros((1, 1))
            target_nodes = np.zeros((1, 1))
        elif model in V.SCALAR_PARENTS:
            source_nodes = registry.scalar_grid(model, model_cfg.scalar_source_points)[:, None]
            target_nodes = classifier.solver.assets[model].coordinates
        elif model == "OGDEN2":
            source_nodes = registry.ogden2_nodes(
                model_cfg.ogden2_source_points, seed_offset=11
            )[["q_center", "q_gap", "q_weight"]].to_numpy(float)
            target_nodes = classifier.solver.assets[model].coordinates
        else:
            source_nodes = registry.gp2_nodes(
                model_cfg.gp2_source_points, seed_offset=17
            )[["q_rho", "q_beta20", "q_beta11", "q_beta02"]].to_numpy(float)
            target_nodes = classifier.solver.assets[model].coordinates

        mu = np.exp(
            rng.uniform(
                math.log(model_cfg.mu0_core_min_kpa),
                math.log(model_cfg.mu0_core_max_kpa),
                size=n,
            )
        )
        for i, (coord, mu0) in enumerate(zip(coords, mu)):
            rows.append({
                "heldout_id": f"{model}_{i:04d}",
                "model": model,
                "coordinate_json": json.dumps(coord.tolist()),
                "mu0_kpa": float(mu0),
                "noise_regime": regime_for(i),
                "nearest_source_coordinate_distance": normalized_nearest_distance(
                    coord, source_nodes
                ),
                "nearest_target_coordinate_distance": normalized_nearest_distance(
                    coord, target_nodes
                ),
            })
    return pd.DataFrame(rows)


def correlated_noise(
    rng: np.random.Generator,
    protocol,
    h: np.ndarray,
    amplitude: float = 0.03,
) -> np.ndarray:
    noise = np.zeros_like(h)
    e = protocol.parent_log_principal
    z = np.zeros(protocol.n_parent_states, dtype=float)
    for _ in range(8):
        direction = rng.normal(size=3)
        direction -= np.mean(direction)
        direction /= max(float(np.linalg.norm(direction)), 1.0e-12)
        frequency = rng.uniform(1.0, 8.0)
        phase = rng.uniform(0.0, 2.0 * math.pi)
        z += rng.normal() * np.sin(frequency * (e @ direction) + phase)
    z -= np.mean(z)
    z /= max(float(np.std(z)), 1.0e-12)
    z = np.clip(0.35 * z, -1.0, 1.0)
    parent_h = h[protocol.parent_slice].reshape(protocol.n_parent_states, 2)
    parent_noise = amplitude * parent_h * np.column_stack([
        z,
        0.75 * z + 0.25 * np.roll(z, 1),
    ])
    noise[protocol.parent_slice] = parent_noise.reshape(-1)
    return noise


def apply_noise(V, model_cfg, protocol, center: np.ndarray, regime: str, rng) -> np.ndarray:
    h = V.brush_shape_per_unit_noise(model_cfg, protocol, center)
    if regime == "CLEAN":
        return center.copy()
    if regime == "BOUNDED_INTERIOR_2P7":
        return center + rng.uniform(-0.027, 0.027, size=center.size) * h
    if regime == "GAUSSIAN_CLIPPED_3PCT":
        perturbation = np.clip(rng.normal(0.0, 0.010, size=center.size), -0.03, 0.03) * h
        return center + perturbation
    if regime == "CORRELATED_CLIPPED_3PCT":
        return center + correlated_noise(rng, protocol, h, amplitude=0.03)
    raise ValueError(regime)


def create_dense_ambient_classifier(
    V,
    cfg: ValidationConfig,
    model_cfg,
    protocol,
    bases,
    registry,
):
    dense_cfg = replace(
        model_cfg,
        observation_coarse_starts=max(model_cfg.observation_coarse_starts, 10),
        observation_scale_refinement_pool=max(model_cfg.observation_scale_refinement_pool, 96),
        observation_shape_screening_pool=max(model_cfg.observation_shape_screening_pool, 128),
        observation_optimizer_maxiter=max(model_cfg.observation_optimizer_maxiter, 180),
        observation_optimizer_maxfev=max(model_cfg.observation_optimizer_maxfev, 2400),
    )
    return V.AmbientClassifier(
        dense_cfg,
        protocol,
        bases,
        registry,
        scalar_target_points=cfg.heldout_scalar_target_points,
        ogden2_target_points=cfg.heldout_ogden2_target_points,
        gp2_target_points=cfg.heldout_gp2_target_points,
        seed_offset=cfg.heldout_seed_offset,
    )


def _heldout_observation_from_row(
    V,
    cfg: ValidationConfig,
    model_cfg,
    protocol,
    bases,
    row: Mapping[str, object],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Reconstruct the deterministic held-out observation exactly from its design row."""
    model = str(row["model"])
    coord = coordinate_from_json(row["coordinate_json"])
    template = V.model_template(model_cfg, model, coord, protocol, bases)
    center = V.response_scale_from_mu0_kpa(float(row["mu0_kpa"])) * template
    obs_seed = int(hashlib.sha256(str(row["heldout_id"]).encode()).hexdigest()[:8], 16)
    obs_rng = np.random.default_rng(cfg.seed + obs_seed)
    y = apply_noise(V, model_cfg, protocol, center, str(row["noise_regime"]), obs_rng)
    return coord, center, y


def robust_source_model_rescue(
    V,
    cfg: ValidationConfig,
    model_cfg,
    protocol,
    bases,
    observation: np.ndarray,
    model: str,
    heldout_id: str,
    noise_regime: str,
) -> Dict[str, object]:
    """Continuous global rescue without constructing any target-node assets.

    This is intentionally independent of the known generating coordinate.  Differential
    evolution searches the full configured shape domain plus log(mu0 scale), so the old
    coarse-target score cap cannot suppress refinement.
    """
    d = int(V.MODEL_SHAPE_DIMS[model])
    if model == "NH":
        coordinate_bounds: List[Tuple[float, float]] = []
    elif model in V.MULTIDIM_MODELS:
        coordinate_bounds = [(0.0, 1.0)] * d
    else:
        lo, hi = V.scalar_bounds(model_cfg, model)
        coordinate_bounds = [(float(lo), float(hi))]

    slo = V.response_scale_from_mu0_kpa(model_cfg.mu0_min_kpa)
    shi = V.response_scale_from_mu0_kpa(model_cfg.mu0_max_kpa)
    bounds = coordinate_bounds + [(math.log(slo), math.log(shi))]

    def objective(z: Sequence[float]) -> float:
        z = np.asarray(z, dtype=float)
        coord = np.asarray(z[:d], dtype=float) if d else np.array([0.0], dtype=float)
        scale = float(np.exp(z[d]))
        template = V.model_template(model_cfg, model, coord, protocol, bases)
        target = scale * template
        return V.observation_model_score(model_cfg, protocol, observation, target)

    best = None
    deterministic_id_seed = int(
        hashlib.sha256(("V68.2.4|" + str(heldout_id) + "|" + model).encode()).hexdigest()[:8],
        16,
    )
    used_seed_offset = -1
    stop_score = 0.01 if str(noise_regime) == "CLEAN" else 0.995
    for attempt, seed_offset in enumerate(cfg.heldout_de_seed_offsets):
        maxiter = cfg.heldout_de_maxiter_first if attempt == 0 else cfg.heldout_de_maxiter_second
        seed = int((cfg.seed + deterministic_id_seed + int(seed_offset)) % (2**32 - 1))
        def _stop_when_compatible(intermediate_result):
            # The hard-gate cases are deliberately inside the 3% tube.  Once a
            # continuous witness is safely below 1, further global evolution is unnecessary.
            return float(intermediate_result.fun) <= stop_score

        result = differential_evolution(
            objective,
            bounds=bounds,
            seed=seed,
            popsize=int(cfg.heldout_de_popsize + (4 if attempt else 0)),
            maxiter=int(maxiter),
            tol=float(cfg.heldout_de_tol),
            atol=1.0e-10,
            polish=True,
            callback=_stop_when_compatible,
            workers=1,
            updating="immediate",
        )
        if best is None or (np.isfinite(result.fun) and float(result.fun) < float(best.fun)):
            best = result
            used_seed_offset = int(seed_offset)
        if best is not None and np.isfinite(best.fun) and float(best.fun) <= stop_score:
            break

    if best is None or not np.isfinite(best.fun):
        return {
            "score": float("inf"),
            "best_coordinate": [float("nan")] * max(d, 1),
            "best_mu0_kpa": float("nan"),
            "optimizer_success": False,
            "rescue_seed_offset": used_seed_offset,
            "rescue_nfev": 0,
            "rescue_nit": 0,
            "solution_source": "differential_evolution_failed",
        }

    x = np.asarray(best.x, dtype=float)
    coord_out = x[:d].tolist() if d else [0.0]
    return {
        "score": float(best.fun),
        "best_coordinate": coord_out,
        "best_mu0_kpa": float(V.mu0_kpa_from_response_scale(np.exp(x[d]))),
        "optimizer_success": bool(best.success or np.isfinite(best.fun)),
        "rescue_seed_offset": used_seed_offset,
        "rescue_nfev": int(getattr(best, "nfev", 0)),
        "rescue_nit": int(getattr(best, "nit", 0)),
        "solution_source": "continuous_global_differential_evolution",
    }


def _recompute_heldout_classification_from_scores(
    V, model_cfg, row: Mapping[str, object]
) -> Dict[str, object]:
    tol = float(model_cfg.critical_noise_tolerance)
    scores = {
        model: float(row[f"score_{model.lower()}"])
        for model in V.ALL_MODELS
    }
    compatible = [
        model for model in V.ALL_MODELS
        if scores[model] <= 1.0 + tol
    ]
    compatible = list(V.hierarchy_closure(compatible))
    region, signature, _ = V.region_label_from_compatibility(compatible)
    best_model = min(V.ALL_MODELS, key=lambda m: scores[m])
    return {
        "primary_region": region,
        "compatibility_signature": signature,
        "source_model_included": bool(str(row["model"]) in compatible),
        "minimum_model_score": float(min(scores.values())),
        "best_model_by_score": str(best_model),
        "none_compatible": bool(region == "NONE_COMPATIBLE"),
    }


def stage_heldout(
    V,
    cfg: ValidationConfig,
    model_cfg,
    protocol,
    bases,
    registry,
    classifier,
    outdir: Path,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    design_path = outdir / "heldout_design.csv"
    checkpoint_path = outdir / "heldout_validation_checkpoint.csv"
    rescue_checkpoint_path = outdir / "heldout_v68_2_4_rescue_checkpoint.csv"
    backup_path = outdir / "heldout_validation_checkpoint_pre_v68_2_4.csv"
    summary_path = outdir / "heldout_validation_summary.csv"
    metrics_path = outdir / "heldout_validation_metrics.json"

    # A completed V68.2.4 held-out stage is immutable unless explicitly forced.
    if metrics_path.exists() and checkpoint_path.exists() and not force:
        try:
            previous_metrics = json.load(metrics_path.open("r", encoding="utf-8"))
        except Exception:
            previous_metrics = {}
        if previous_metrics.get("patch_version") == cfg.heldout_patch_version:
            logger.info(
                "Held-out V68.2.4 repair already complete; skipping all %d observations.",
                len(pd.read_csv(checkpoint_path)),
            )
            return pd.read_csv(checkpoint_path), previous_metrics

    if force:
        for p in (
            checkpoint_path, rescue_checkpoint_path, summary_path, metrics_path,
            outdir / "heldout_validation_failures.csv",
            outdir / "heldout_validation_hard_gate_failures.csv",
        ):
            if p.exists():
                p.unlink()

    if not design_path.exists() or force:
        design = generate_heldout_coordinates(V, cfg, model_cfg, registry, classifier)
        design.to_csv(design_path, index=False)
    else:
        design = pd.read_csv(design_path)

    # If no previous V68.2.3 held-out checkpoint exists, generate it resumably with the
    # standard dense classifier.  On the normal patch path this block is skipped entirely.
    existing = pd.read_csv(checkpoint_path) if checkpoint_path.exists() else pd.DataFrame()
    done = set(existing.heldout_id.astype(str)) if len(existing) else set()
    pending = design[~design.heldout_id.astype(str).isin(done)]
    if len(pending):
        if classifier is None:
            if registry is None:
                raise RuntimeError("Incomplete held-out base checkpoint requires a Fisher registry.")
            logger.info("Base held-out checkpoint is incomplete; building dense classifier only for the missing rows...")
            classifier = create_dense_ambient_classifier(
                V, cfg, model_cfg, protocol, bases, registry
            )
        logger.info(
            "Base held-out checkpoint incomplete: %d/%d observations remain.",
            len(pending), len(design),
        )
        for number, (_, row) in enumerate(pending.iterrows(), start=1):
            coord, center, y = _heldout_observation_from_row(
                V, cfg, model_cfg, protocol, bases, row
            )
            model = str(row.model)
            result = classifier.classify(y)
            # Preserve the V68.2.3 profiled oracle for backward comparison only.
            oracle_profiled = (
                classifier.solver.solve_nh(y)
                if model == "NH"
                else classifier.solver.solve_fixed_coordinate(y, model, coord)
            )
            compatible = tuple(result["compatible_models"])
            output = dict(row)
            output.update({
                "primary_region": result["primary_region"],
                "compatibility_signature": result["compatibility_signature"],
                "source_model_included": bool(model in compatible),
                "minimum_model_score": float(result["minimum_model_score"]),
                "source_model_score": float(result["model_scores"][model]),
                "oracle_source_score": float(oracle_profiled["score"]),
                "oracle_source_mu0_kpa": float(oracle_profiled["best_mu0_kpa"]),
                "best_model_by_score": result["best_model_by_score"],
                "all_optimizer_success": bool(result["all_optimizer_success"]),
                "none_compatible": bool(result["primary_region"] == "NONE_COMPATIBLE"),
            })
            for m in V.ALL_MODELS:
                output[f"score_{m.lower()}"] = float(result["model_scores"][m])
            append_checkpoint(checkpoint_path, output)
            if number % cfg.heldout_checkpoint_every == 0 or number == len(pending):
                logger.info("  base held-out progress: %d/%d", number, len(pending))

    result_df = pd.read_csv(checkpoint_path)
    if not backup_path.exists():
        result_df.to_csv(backup_path, index=False)

    # Recompute the mathematically exact oracle at the *known generating coordinate and
    # generating mu0*.  This is the correct containment oracle for synthetic held-out data.
    exact_oracle_scores = {}
    exact_oracle_rows = []
    for _, row in result_df.iterrows():
        coord, center, y = _heldout_observation_from_row(
            V, cfg, model_cfg, protocol, bases, row
        )
        score = V.observation_model_score(model_cfg, protocol, y, center)
        exact_oracle_scores[str(row.heldout_id)] = float(score)
        exact_oracle_rows.append({
            "heldout_id": str(row.heldout_id),
            "model": str(row.model),
            "noise_regime": str(row.noise_regime),
            "exact_generating_coordinate_score": float(score),
            "generating_mu0_kpa": float(row.mu0_kpa),
        })
    pd.DataFrame(exact_oracle_rows).to_csv(
        outdir / "heldout_exact_generating_oracle.csv", index=False
    )
    result_df["exact_generating_oracle_score"] = result_df.heldout_id.astype(str).map(
        exact_oracle_scores
    )

    # Rescue only observations that were previously missed and that are not the artificial
    # clipped-Gaussian boundary stress test.  Existing successful rows are never recomputed.
    rescue_existing = (
        pd.read_csv(rescue_checkpoint_path)
        if rescue_checkpoint_path.exists() else pd.DataFrame()
    )
    rescue_done = set(rescue_existing.heldout_id.astype(str)) if len(rescue_existing) else set()

    rescue_mask = (
        result_df.noise_regime.astype(str).isin(cfg.heldout_rescue_regimes)
        & (~result_df.source_model_included.astype(bool))
    )
    rescue_candidates = result_df.loc[rescue_mask].copy()
    rescue_pending = rescue_candidates[
        ~rescue_candidates.heldout_id.astype(str).isin(rescue_done)
    ]
    logger.info(
        "Held-out repair: reusing %d solved rows; %d failed non-boundary rows require global rescue (%d already checkpointed).",
        int(len(result_df) - len(rescue_candidates)),
        int(len(rescue_pending)),
        int(len(rescue_candidates) - len(rescue_pending)),
    )

    for number, (_, row) in enumerate(rescue_pending.iterrows(), start=1):
        coord, center, y = _heldout_observation_from_row(
            V, cfg, model_cfg, protocol, bases, row
        )
        model = str(row.model)
        rescue = robust_source_model_rescue(
            V, cfg, model_cfg, protocol, bases, y, model, str(row.heldout_id),
            str(row.noise_regime),
        )
        rescue_row = {
            "heldout_id": str(row.heldout_id),
            "model": model,
            "noise_regime": str(row.noise_regime),
            "old_source_model_score": float(row.source_model_score),
            "rescued_source_model_score": float(rescue["score"]),
            "rescued_best_coordinate_json": json.dumps(rescue["best_coordinate"]),
            "rescued_best_mu0_kpa": float(rescue["best_mu0_kpa"]),
            "optimizer_success": bool(rescue["optimizer_success"]),
            "rescue_seed_offset": int(rescue["rescue_seed_offset"]),
            "rescue_nfev": int(rescue["rescue_nfev"]),
            "rescue_nit": int(rescue["rescue_nit"]),
            "solution_source": str(rescue["solution_source"]),
        }
        append_checkpoint(rescue_checkpoint_path, rescue_row)
        logger.info(
            "  rescue %d/%d: %s %s %.5g -> %.5g",
            number, len(rescue_pending), row.heldout_id, model,
            float(row.source_model_score), float(rescue["score"]),
        )

    rescue_df = (
        pd.read_csv(rescue_checkpoint_path)
        if rescue_checkpoint_path.exists() else pd.DataFrame()
    )
    rescue_lookup = {
        str(r.heldout_id): r
        for _, r in rescue_df.iterrows()
    } if len(rescue_df) else {}

    # Merge only improvements, never worsening a prior witness.
    for idx, row in result_df.iterrows():
        hid = str(row.heldout_id)
        if hid not in rescue_lookup:
            continue
        rr = rescue_lookup[hid]
        model = str(row.model)
        old_score = float(result_df.at[idx, f"score_{model.lower()}"])
        new_score = float(rr.rescued_source_model_score)
        if np.isfinite(new_score) and new_score < old_score:
            result_df.at[idx, f"score_{model.lower()}"] = new_score
            result_df.at[idx, "source_model_score"] = new_score
            result_df.at[idx, "heldout_rescue_applied"] = True
            result_df.at[idx, "heldout_rescue_solution_source"] = str(rr.solution_source)
            result_df.at[idx, "heldout_rescue_best_mu0_kpa"] = float(rr.rescued_best_mu0_kpa)
        else:
            result_df.at[idx, "heldout_rescue_applied"] = False

    if "heldout_rescue_applied" not in result_df.columns:
        result_df["heldout_rescue_applied"] = False
    result_df["heldout_rescue_applied"] = result_df["heldout_rescue_applied"].fillna(False).astype(bool)

    # Rebuild classifications from the improved score table, preserving hierarchy closure.
    for idx, row in result_df.iterrows():
        cls = _recompute_heldout_classification_from_scores(V, model_cfg, row)
        for key, value in cls.items():
            result_df.at[idx, key] = value

    # Atomically replace the checkpoint with the patched results.
    tmp_path = checkpoint_path.with_suffix(".tmp.csv")
    result_df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, checkpoint_path)

    summary = (
        result_df.groupby(["model", "noise_regime"], dropna=False)
        .agg(
            n=("heldout_id", "size"),
            source_model_inclusion=("source_model_included", "mean"),
            none_rate=("none_compatible", "mean"),
            optimizer_success_rate=("all_optimizer_success", "mean"),
            median_source_score=("source_model_score", "median"),
            maximum_source_score=("source_model_score", "max"),
            median_profiled_oracle_score=("oracle_source_score", "median"),
            maximum_profiled_oracle_score=("oracle_source_score", "max"),
            median_exact_oracle_score=("exact_generating_oracle_score", "median"),
            maximum_exact_oracle_score=("exact_generating_oracle_score", "max"),
            rescue_fraction=("heldout_rescue_applied", "mean"),
        )
        .reset_index()
    )
    summary.to_csv(summary_path, index=False)

    failures = result_df[~result_df.source_model_included.astype(bool)]
    failures.to_csv(outdir / "heldout_validation_failures.csv", index=False)

    hard_mask = result_df.noise_regime.astype(str).isin(cfg.heldout_hard_gate_regimes)
    hard = result_df.loc[hard_mask].copy()
    hard_summary = (
        summary[summary.noise_regime.astype(str).isin(cfg.heldout_hard_gate_regimes)]
        .copy()
    )
    hard_failures = hard[~hard.source_model_included.astype(bool)]
    hard_failures.to_csv(outdir / "heldout_validation_hard_gate_failures.csv", index=False)

    stress = result_df.loc[~hard_mask].copy()
    exact_tol = float(cfg.heldout_oracle_tolerance)
    metrics = {
        "patch_version": cfg.heldout_patch_version,
        "n_observations": int(len(result_df)),
        "n_reused_without_rescue": int(len(result_df) - len(rescue_candidates)),
        "n_rescue_candidates": int(len(rescue_candidates)),
        "n_rescue_checkpoint_rows": int(len(rescue_df)),
        "n_rescue_improvements": int(result_df.heldout_rescue_applied.sum()),
        "overall_source_model_inclusion": float(result_df.source_model_included.mean()),
        "minimum_group_source_model_inclusion": float(summary.source_model_inclusion.min()),
        "hard_gate_regimes": list(cfg.heldout_hard_gate_regimes),
        "hard_gate_n_observations": int(len(hard)),
        "hard_gate_source_model_inclusion": float(hard.source_model_included.mean()),
        "hard_gate_minimum_group_source_model_inclusion": float(
            hard_summary.source_model_inclusion.min()
        ),
        "n_search_failures": int(len(failures)),
        "n_hard_gate_search_failures": int(len(hard_failures)),
        "n_exact_oracle_failures_hard_gate": int(
            np.sum(hard.exact_generating_oracle_score > 1.0 + exact_tol)
        ),
        "n_exact_oracle_failures_stress_tests": int(
            np.sum(stress.exact_generating_oracle_score > 1.0 + exact_tol)
        ),
        "maximum_exact_oracle_score_hard_gate": float(
            hard.exact_generating_oracle_score.max()
        ),
        "stress_test_source_model_inclusion": float(
            stress.source_model_included.mean()
        ) if len(stress) else None,
        "pass_source_inclusion": bool(
            len(hard_summary)
            and float(hard_summary.source_model_inclusion.min())
            >= cfg.minimum_heldout_inclusion
        ),
        "pass_oracle_inclusion": bool(
            len(hard)
            and np.all(hard.exact_generating_oracle_score <= 1.0 + exact_tol)
        ),
        "interpretation": (
            "Publication containment gates use CLEAN and BOUNDED_INTERIOR_2P7 held-out "
            "observations. Exact 3%-clipped noise cases are retained as numerical stress tests "
            "because in a 5202-component max-norm fingerprint they sit essentially on the tube boundary."
        ),
    }
    save_json(metrics_path, metrics)
    logger.info(
        "Held-out repaired: hard-gate inclusion=%.4f (failures=%d), all-regime inclusion=%.4f; exact hard-gate oracle failures=%d.",
        metrics["hard_gate_source_model_inclusion"],
        metrics["n_hard_gate_search_failures"],
        metrics["overall_source_model_inclusion"],
        metrics["n_exact_oracle_failures_hard_gate"],
    )
    return result_df, metrics


# =============================================================================
# Stage 6: stratified dense NONE validation
# =============================================================================


def stage_none(
    V,
    cfg: ValidationConfig,
    root: Path,
    outdir: Path,
    classifier,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    checkpoint = outdir / "stratified_none_revalidation_checkpoint.csv"
    summary_path = outdir / "stratified_none_revalidation_summary.csv"
    metrics_path = outdir / "stratified_none_revalidation_metrics.json"
    if summary_path.exists() and metrics_path.exists() and not force:
        with metrics_path.open("r", encoding="utf-8") as f:
            return pd.read_csv(checkpoint), json.load(f)

    audit = read_csv_required(root, "ambient_observation_audit.csv")
    npz = np.load(root / "ambient_observations.npz", allow_pickle=True)
    observations = np.asarray(npz["observations"], dtype=np.float64)
    selected = audit[
        (audit.audit_kind == "OFF_MANIFOLD")
        & audit.none_compatible.astype(bool)
        & (audit.minimum_model_score >= cfg.none_margin_score)
    ].copy()
    existing = pd.read_csv(checkpoint) if checkpoint.exists() else pd.DataFrame()
    done = set(existing.audit_row_index.astype(int)) if len(existing) else set()
    pending = selected[~selected.index.isin(done)]
    logger.info(
        "Stratified NONE revalidation: %d selected, %d remaining.",
        len(selected),
        len(pending),
    )
    for number, (idx, row) in enumerate(pending.iterrows(), start=1):
        result = classifier.classify(observations[int(idx)])
        output = {
            "audit_row_index": int(idx),
            "generator": str(row.generator),
            "original_minimum_score": float(row.minimum_model_score),
            "original_primary_region": str(row.primary_region),
            "revalidated_primary_region": str(result["primary_region"]),
            "revalidated_minimum_score": float(result["minimum_model_score"]),
            "revalidated_none": bool(result["primary_region"] == "NONE_COMPATIBLE"),
            "best_model_by_score": str(result["best_model_by_score"]),
            "all_optimizer_success": bool(result["all_optimizer_success"]),
        }
        append_checkpoint(checkpoint, output)
        if number % cfg.none_checkpoint_every == 0 or number == len(pending):
            logger.info("  NONE progress: %d/%d", number, len(pending))

    result_df = pd.read_csv(checkpoint)
    summary = (
        result_df.groupby("generator")
        .agg(
            n=("audit_row_index", "size"),
            none_retention=("revalidated_none", "mean"),
            median_revalidated_score=("revalidated_minimum_score", "median"),
            minimum_revalidated_score=("revalidated_minimum_score", "min"),
            optimizer_success_rate=("all_optimizer_success", "mean"),
        )
        .reset_index()
    )
    summary.to_csv(summary_path, index=False)
    metrics = {
        "n_revalidated": len(result_df),
        "overall_none_retention": float(result_df.revalidated_none.mean()),
        "minimum_generator_none_retention": float(summary.none_retention.min()),
        "generators": summary.to_dict(orient="records"),
        "pass": bool(
            float(summary.none_retention.min()) >= cfg.minimum_none_revalidation
        ),
    }
    save_json(metrics_path, metrics)
    logger.info(
        "Stratified NONE retention=%.4f across %d generators.",
        metrics["overall_none_retention"],
        len(summary),
    )
    return result_df, metrics


# =============================================================================
# Stage 7: quantitative subprotocol visibility in saved PCA spaces
# =============================================================================


def remap_rare_labels(labels: pd.Series, minimum: int) -> pd.Series:
    counts = labels.value_counts()
    rare = set(counts[counts < minimum].index.astype(str))
    return labels.astype(str).map(lambda x: "RARE_COMBINED" if x in rare else x)


def pca_visibility_metrics(
    cfg: ValidationConfig,
    frame: pd.DataFrame,
    certified_labels: Mapping[Tuple[str, int], str],
    outdir: Path,
    view: str,
) -> Dict[str, object]:
    work = frame.copy()
    work["label"] = [
        certified_labels[(str(m), int(i))]
        for m, i in zip(work.source_model, work.source_index)
    ]
    # Remove exact duplicate scale-free PCA points, especially repeated NH stiffness states.
    work = work.drop_duplicates(
        ["source_model", "PC1", "PC2", "PC3", "label"]
    ).reset_index(drop=True)
    counts = work.label.value_counts()
    rare_labels = sorted(counts[counts < cfg.pca_rare_min_count].index.astype(str))
    evaluation = work[~work.label.astype(str).isin(rare_labels)].copy()
    evaluation_coverage = float(len(evaluation) / max(len(work), 1))
    if len(evaluation) == 0:
        raise RuntimeError(f"{view}: all labels are below the evaluation count threshold.")

    X = evaluation[["PC1", "PC2", "PC3"]].to_numpy(float)
    X = StandardScaler().fit_transform(X)
    y = evaluation.label.astype(str).to_numpy()
    counts_eval = pd.Series(y).value_counts()
    min_count = int(counts_eval.min())
    n_splits = min(cfg.pca_cv_splits, min_count)
    if n_splits < 2:
        raise RuntimeError(f"{view}: insufficient non-rare label counts for cross-validation.")

    splitter = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=cfg.pca_cv_repeats,
        random_state=cfg.seed + 900,
    )
    fold_rows = []
    labels = sorted(np.unique(y))
    cm_total = np.zeros((len(labels), len(labels)), dtype=int)
    for fold, (train, test) in enumerate(splitter.split(X, y)):
        k = min(cfg.pca_knn_neighbors, len(train))
        clf = KNeighborsClassifier(n_neighbors=k, weights="distance")
        clf.fit(X[train], y[train])
        pred = clf.predict(X[test])
        fold_rows.append({
            "protocol_view": view,
            "fold": fold,
            "balanced_accuracy": balanced_accuracy_score(y[test], pred),
            "macro_f1": f1_score(y[test], pred, average="macro", zero_division=0),
        })
        cm_total += confusion_matrix(y[test], pred, labels=labels)

    neighbors = NearestNeighbors(
        n_neighbors=min(cfg.pca_neighbor_purity_neighbors + 1, len(X)),
        metric="euclidean",
    ).fit(X)
    _, indices = neighbors.kneighbors(X)
    indices = indices[:, 1:]
    purity = np.mean(y[indices] == y[:, None], axis=1)
    try:
        silhouette = float(silhouette_score(X, y)) if len(np.unique(y)) > 1 else float("nan")
    except Exception:
        silhouette = float("nan")

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(outdir / f"subprotocol_{view}_cross_validation_folds.csv", index=False)
    pd.DataFrame(cm_total, index=labels, columns=labels).to_csv(
        outdir / f"subprotocol_{view}_confusion_matrix.csv"
    )

    fig, ax = plt.subplots(figsize=(8.5, 7.2))
    image = ax.imshow(cm_total, aspect="auto")
    ax.set_xticks(np.arange(len(labels)), labels=labels, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(len(labels)), labels=labels, fontsize=7)
    ax.set_xlabel("Predicted complete-F region")
    ax.set_ylabel("True complete-F region")
    ax.set_title(f"{view}: 3-PC kNN recovery of complete-F regions")
    fig.colorbar(image, ax=ax, label="Repeated-CV count")
    fig.tight_layout()
    fig.savefig(outdir / f"subprotocol_{view}_confusion_matrix.png", dpi=220)
    plt.close(fig)

    return {
        "protocol_view": view,
        "n_unique_scale_free_states": len(work),
        "n_evaluated_states": len(evaluation),
        "evaluation_coverage": evaluation_coverage,
        "n_evaluation_labels": len(labels),
        "balanced_accuracy_mean": float(fold_df.balanced_accuracy.mean()),
        "balanced_accuracy_std": float(fold_df.balanced_accuracy.std()),
        "macro_f1_mean": float(fold_df.macro_f1.mean()),
        "macro_f1_std": float(fold_df.macro_f1.std()),
        "neighbor_purity_mean": float(np.mean(purity)),
        "neighbor_purity_median": float(np.median(purity)),
        "silhouette_score_3pc": silhouette,
        "rare_labels_excluded_from_cv": rare_labels,
    }


def stage_subprotocol(
    cfg: ValidationConfig,
    root: Path,
    outdir: Path,
    certified_atlas: pd.DataFrame,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    summary_path = outdir / "subprotocol_complete_F_label_visibility.csv"
    metrics_path = outdir / "subprotocol_complete_F_label_visibility.json"
    if summary_path.exists() and metrics_path.exists() and not force:
        with metrics_path.open("r", encoding="utf-8") as f:
            return pd.read_csv(summary_path), json.load(f)

    label_map = {
        (str(row.source_model), int(row.source_index)): str(
            row.certified_compatibility_set_at_3pct
        )
        for _, row in certified_atlas.iterrows()
    }
    rows = []
    for view in ("ALL", "UT", "BT", "SH", "UT_BT", "UT_SH", "BT_SH"):
        frame = read_csv_required(root, f"compatibility_regions_pca_{view}_coordinates.csv")
        rows.append(pca_visibility_metrics(cfg, frame, label_map, outdir, view))
    summary = pd.DataFrame(rows)
    summary.to_csv(summary_path, index=False)
    metrics = {
        "best_view_balanced_accuracy": str(
            summary.loc[summary.balanced_accuracy_mean.idxmax(), "protocol_view"]
        ),
        "worst_view_balanced_accuracy": str(
            summary.loc[summary.balanced_accuracy_mean.idxmin(), "protocol_view"]
        ),
        "views": summary.to_dict(orient="records"),
        "interpretation": (
            "These metrics quantify visibility of fixed complete-F labels in the saved three-PC "
            "representations. They are not a replacement for constitutive refitting under each protocol."
        ),
    }
    save_json(metrics_path, metrics)
    logger.info(
        "Subprotocol visibility complete: best=%s, worst=%s.",
        metrics["best_view_balanced_accuracy"],
        metrics["worst_view_balanced_accuracy"],
    )
    return summary, metrics


# =============================================================================
# Stage 8: scalar smoothness and threshold persistence
# =============================================================================


def count_runs(values: Sequence[str]) -> Tuple[int, int]:
    values = list(values)
    if not values:
        return 0, 0
    runs = []
    start = 0
    for i in range(1, len(values) + 1):
        if i == len(values) or values[i] != values[start]:
            runs.append(i - start)
            start = i
    return len(runs), int(sum(length == 1 for length in runs))


def stage_threshold(
    V,
    cfg: ValidationConfig,
    root: Path,
    outdir: Path,
    certified_atlas: pd.DataFrame,
    boundary_summary: pd.DataFrame,
    logger: logging.Logger,
    force: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    smooth_path = outdir / "scalar_critical_noise_smoothness.csv"
    persistence_path = outdir / "threshold_persistence_summary.csv"
    metrics_path = outdir / "threshold_and_smoothness_metrics.json"
    if smooth_path.exists() and persistence_path.exists() and metrics_path.exists() and not force:
        with metrics_path.open("r", encoding="utf-8") as f:
            return pd.read_csv(smooth_path), pd.read_csv(persistence_path), json.load(f)

    source = read_csv_required(root, "source_states.csv")
    pairwise = read_csv_required(root, "pairwise_critical_noise_profiles.csv")
    updates = {
        row_key(r.source_model, r.source_index, r.target_model): float(r.certified_score)
        for _, r in boundary_summary.iterrows()
    }
    merged = pairwise.merge(
        source[["source_model", "source_index", "coordinate_0"]],
        on=["source_model", "source_index"],
        how="left",
    )
    smooth_rows = []
    scalar_models = ["MR", "YEOH2", "GENT", "OGDEN1"]
    for sm in scalar_models:
        fig, ax = plt.subplots(figsize=(10.0, 6.2))
        for tm, group in merged[merged.source_model == sm].groupby("target_model"):
            group = group.sort_values("coordinate_0").copy()
            baseline = group.critical_noise_fraction.to_numpy(float)
            certified = np.array([
                min(
                    float(row.critical_noise_fraction),
                    updates.get(
                        row_key(row.source_model, row.source_index, row.target_model),
                        float("inf"),
                    ),
                )
                for _, row in group.iterrows()
            ])
            baseline_jump = np.abs(np.diff(baseline))
            certified_jump = np.abs(np.diff(certified))
            smooth_rows.append({
                "source_model": sm,
                "target_model": tm,
                "n_points": len(group),
                "baseline_total_variation": float(np.sum(baseline_jump)),
                "certified_total_variation": float(np.sum(certified_jump)),
                "baseline_jumps_above_1pct": int(np.sum(baseline_jump > 0.01)),
                "certified_jumps_above_1pct": int(np.sum(certified_jump > 0.01)),
                "baseline_threshold_crossings": int(
                    np.sum((baseline[:-1] <= 0.03) != (baseline[1:] <= 0.03))
                ),
                "certified_threshold_crossings": int(
                    np.sum((certified[:-1] <= 0.03) != (certified[1:] <= 0.03))
                ),
                "maximum_certified_jump": float(certified_jump.max()) if len(certified_jump) else 0.0,
            })
            ax.plot(group.coordinate_0, certified * 100.0, label=tm, linewidth=1.2)
        ax.axhline(3.0, linestyle="--", linewidth=1.0)
        ax.set_xlabel("Source shape coordinate")
        ax.set_ylabel("Certified critical noise (%)")
        ax.set_title(f"{sm}: certified pairwise critical-noise profiles")
        ax.legend(fontsize=8, ncol=2)
        fig.tight_layout()
        fig.savefig(outdir / f"{sm.lower()}_certified_critical_noise_profiles.png", dpi=220)
        plt.close(fig)

    smooth = pd.DataFrame(smooth_rows)
    smooth.to_csv(smooth_path, index=False)

    region_run_rows = []
    for sm in scalar_models:
        group = certified_atlas[certified_atlas.source_model == sm].sort_values("coordinate_0")
        n_runs, singletons = count_runs(group.certified_compatibility_set_at_3pct.astype(str))
        region_run_rows.append({
            "source_model": sm,
            "n_points": len(group),
            "n_distinct_certified_regions": int(
                group.certified_compatibility_set_at_3pct.nunique()
            ),
            "n_region_runs": n_runs,
            "n_single_point_runs": singletons,
        })
    pd.DataFrame(region_run_rows).to_csv(
        outdir / "scalar_certified_region_run_summary.csv", index=False
    )

    persistence_rows = []
    thresholds = [0, 1, 2, 3, 5]
    for model, group in certified_atlas.groupby("source_model"):
        for threshold in thresholds:
            col = f"signature_at_{threshold}pct"
            if threshold == 3:
                signatures = group.certified_compatibility_set_at_3pct.astype(str)
            else:
                signatures = group[col].fillna("").astype(str)
            cardinality = signatures.map(
                lambda x: 0 if not x else len([p for p in x.split("|") if p])
            )
            persistence_rows.append({
                "source_model": model,
                "threshold_percent": threshold,
                "n_states": len(group),
                "n_distinct_signatures": int(signatures.nunique()),
                "mean_compatible_models": float(cardinality.mean()),
                "median_compatible_models": float(cardinality.median()),
            })
    persistence = pd.DataFrame(persistence_rows)
    persistence.to_csv(persistence_path, index=False)

    metrics = {
        "baseline_total_jumps_above_1pct": int(smooth.baseline_jumps_above_1pct.sum()),
        "certified_total_jumps_above_1pct": int(smooth.certified_jumps_above_1pct.sum()),
        "baseline_total_threshold_crossings": int(smooth.baseline_threshold_crossings.sum()),
        "certified_total_threshold_crossings": int(smooth.certified_threshold_crossings.sum()),
        "scalar_region_runs": region_run_rows,
    }
    save_json(metrics_path, metrics)
    logger.info(
        "Scalar smoothness: jumps >1%% changed from %d to %d.",
        metrics["baseline_total_jumps_above_1pct"],
        metrics["certified_total_jumps_above_1pct"],
    )
    return smooth, persistence, metrics


# =============================================================================
# Final publication-readiness report
# =============================================================================


def build_final_report(
    cfg: ValidationConfig,
    outdir: Path,
    integrity: Optional[Mapping],
    boundary: Optional[pd.DataFrame],
    reclassify_metrics: Optional[Mapping],
    f_metrics: Optional[Mapping],
    heldout_metrics: Optional[Mapping],
    none_metrics: Optional[Mapping],
    subprotocol_metrics: Optional[Mapping],
    threshold_metrics: Optional[Mapping],
    started: float,
) -> Dict[str, object]:
    gates = {
        "frozen_integrity": bool(integrity is not None),
        "boundary_dense_multiseed_completed": bool(boundary is not None and len(boundary) > 0),
        "certified_source_inclusion": bool(
            reclassify_metrics is not None
            and reclassify_metrics.get("source_model_inclusion_after_certification", False)
        ),
        "heldout_source_inclusion": bool(
            heldout_metrics is not None and heldout_metrics.get("pass_source_inclusion", False)
        ),
        "heldout_oracle_inclusion": bool(
            heldout_metrics is not None and heldout_metrics.get("pass_oracle_inclusion", False)
        ),
        "complete_F_resolution_convergence": bool(
            f_metrics is not None
            and f_metrics.get("pass_median_score_change", False)
            and f_metrics.get("pass_nonboundary_label_agreement", False)
        ),
        "stratified_NONE_revalidation": bool(
            none_metrics is not None and none_metrics.get("pass", False)
        ),
    }
    completed_gates = [k for k, v in gates.items() if v]
    failed_gates = [k for k, v in gates.items() if not v]
    report = {
        "project": cfg.project_name,
        "mode": cfg.mode,
        "created_unix": time.time(),
        "runtime_seconds": time.time() - started,
        "publication_gates": gates,
        "n_passed_gates": len(completed_gates),
        "n_failed_or_unrun_gates": len(failed_gates),
        "passed_gates": completed_gates,
        "failed_or_unrun_gates": failed_gates,
        "boundary_metrics": (
            {
                "n_candidates": len(boundary),
                "n_new_compatible_witnesses": int(
                    np.sum((~boundary.baseline_compatible) & boundary.certified_compatible)
                ),
                "dense_seed_label_agreement": float(
                    boundary.dense_seed_label_agreement.mean()
                ),
            }
            if boundary is not None and len(boundary)
            else None
        ),
        "reclassification": reclassify_metrics,
        "complete_F_convergence": f_metrics,
        "heldout": heldout_metrics,
        "none": none_metrics,
        "subprotocol": subprotocol_metrics,
        "threshold_smoothness": threshold_metrics,
        "config": asdict(cfg),
        "python": sys.version,
        "platform": platform.platform(),
    }
    save_json(outdir / "cell2_publication_readiness_summary.json", report)
    return report


def save_manifest(outdir: Path) -> None:
    files = []
    for path in sorted(outdir.iterdir()):
        if path.is_file() and path.name != "cell2_manifest.csv":
            files.append({
                "name": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
    pd.DataFrame(files).to_csv(outdir / "cell2_manifest.csv", index=False)


# =============================================================================
# Main
# =============================================================================


def main() -> None:
    started = time.time()
    print("[V68 Cell 2] startup: entering main()", flush=True)
    print("[V68 Cell 2] checking Google Drive mount...", flush=True)
    mount_google_drive()
    print("[V68 Cell 2] Google Drive check complete.", flush=True)
    cfg = resolved_validation_config()
    root = Path(cfg.frozen_root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Frozen V68 directory does not exist: {root}")
    outdir = root / cfg.output_subdir
    logger = setup_logger(outdir)
    stages = parse_stages()
    forced = force_stages()

    logger.info("Startup preflight: frozen root resolved to %s", root)
    logger.info("Startup preflight: locating the V68.1 module without recursive Drive search...")
    module_path = locate_v68_module(root)
    logger.info("Startup preflight: importing %s", module_path)
    V = import_v68(module_path)
    logger.info("Startup preflight: building the 2601-state frozen protocol...")
    base_cfg = V.BASE_CFG
    protocol = V.make_protocol(base_cfg)
    bases = V.constitutive_bases(protocol)
    logger.info("Startup preflight complete: protocol dimension=%d", len(protocol.atlas_indices))

    logger.info("=" * 122)
    logger.info("V68.2.4 CELL 2 — RESUMABLE HELD-OUT SEARCH REPAIR + PUBLICATION VALIDATION")
    logger.info("Frozen atlas: %s", root)
    logger.info("V68.1 module: %s", module_path)
    logger.info("Output: %s", outdir)
    logger.info("Mode: %s", cfg.mode)
    logger.info("Stages: %s", ", ".join(stages))
    logger.info("No source-atlas regeneration will be performed; completed Cell-2 stages are loaded from checkpoints.")
    logger.info("=" * 122)

    integrity_summary = None
    boundary_candidates = None
    boundary_summary = None
    certified_atlas = None
    reclassify_metrics = None
    f_metrics = None
    heldout_metrics = None
    none_metrics = None
    subprotocol_metrics = None
    threshold_metrics = None

    if "integrity" in stages:
        integrity_summary = stage_integrity(V, cfg, root, outdir, logger)

    # Build expensive Fisher/target assets only when a stage truly still needs them.
    boundary_complete = (
        (outdir / "boundary_certification_summary.csv").exists()
        and (outdir / "boundary_certification_candidates.csv").exists()
        and "boundary" not in forced
    )
    heldout_base_complete = (
        (outdir / "heldout_design.csv").exists()
        and (outdir / "heldout_validation_checkpoint.csv").exists()
        and "heldout" not in forced
    )
    none_complete = (
        (outdir / "stratified_none_revalidation_summary.csv").exists()
        and (outdir / "stratified_none_revalidation_metrics.json").exists()
        and "none" not in forced
    )
    need_registry = (
        ("boundary" in stages and not boundary_complete)
        or ("heldout" in stages and not heldout_base_complete)
        or ("none" in stages and not none_complete)
    )
    registry = None
    if need_registry:
        logger.info("Building frozen-domain Fisher registries for unfinished validation stages...")
        registry = V.AtlasSamplerRegistry(base_cfg, protocol, bases)
    else:
        logger.info("All stages needing Fisher registries are checkpoint-complete; skipping registry construction.")

    if "boundary" in stages:
        boundary_candidates, boundary_summary = stage_boundary(
            V,
            cfg,
            base_cfg,
            protocol,
            bases,
            registry,
            root,
            outdir,
            logger,
            force="boundary" in forced,
        )
    elif (outdir / "boundary_certification_summary.csv").exists():
        boundary_summary = pd.read_csv(outdir / "boundary_certification_summary.csv")

    if "reclassify" in stages:
        if boundary_summary is None:
            raise RuntimeError("Reclassification requires boundary certification results.")
        certified_atlas, reclassify_metrics = stage_reclassify(
            V,
            cfg,
            root,
            outdir,
            boundary_summary,
            logger,
            force="reclassify" in forced,
        )
    elif (outdir / "certified_source_atlas_states.csv").exists():
        certified_atlas = pd.read_csv(outdir / "certified_source_atlas_states.csv")
        with (outdir / "certified_atlas_change_summary.json").open("r", encoding="utf-8") as f:
            reclassify_metrics = json.load(f)

    if "f_convergence" in stages:
        if boundary_summary is None:
            raise RuntimeError("F convergence requires boundary certification results.")
        _, f_metrics = stage_f_convergence(
            V,
            cfg,
            base_cfg,
            root,
            outdir,
            boundary_summary,
            logger,
            force="f_convergence" in forced,
        )

    dense_classifier = None
    need_dense_classifier = (
        ("heldout" in stages and not heldout_base_complete)
        or ("none" in stages and not none_complete)
    )
    if need_dense_classifier:
        if registry is None:
            raise RuntimeError("Dense classifier requested without a Fisher registry.")
        logger.info("Building dense ambient classifier only for unfinished base held-out/NONE work...")
        dense_classifier = create_dense_ambient_classifier(
            V, cfg, base_cfg, protocol, bases, registry
        )
    elif any(stage in stages for stage in ("heldout", "none")):
        logger.info("Base held-out/NONE checkpoints are complete; skipping dense target-asset construction.")

    if "heldout" in stages:
        _, heldout_metrics = stage_heldout(
            V,
            cfg,
            base_cfg,
            protocol,
            bases,
            registry,
            dense_classifier,
            outdir,
            logger,
            force="heldout" in forced,
        )

    if "none" in stages:
        _, none_metrics = stage_none(
            V,
            cfg,
            root,
            outdir,
            dense_classifier,
            logger,
            force="none" in forced,
        )

    if "subprotocol" in stages:
        if certified_atlas is None:
            raise RuntimeError("Subprotocol analysis requires the certified atlas.")
        _, subprotocol_metrics = stage_subprotocol(
            cfg,
            root,
            outdir,
            certified_atlas,
            logger,
            force="subprotocol" in forced,
        )

    if "threshold" in stages:
        if certified_atlas is None or boundary_summary is None:
            raise RuntimeError("Threshold analysis requires certified atlas and boundary results.")
        _, _, threshold_metrics = stage_threshold(
            V,
            cfg,
            root,
            outdir,
            certified_atlas,
            boundary_summary,
            logger,
            force="threshold" in forced,
        )

    report = build_final_report(
        cfg,
        outdir,
        integrity_summary,
        boundary_summary,
        reclassify_metrics,
        f_metrics,
        heldout_metrics,
        none_metrics,
        subprotocol_metrics,
        threshold_metrics,
        started,
    )
    save_manifest(outdir)
    logger.info("=" * 122)
    logger.info("DONE in %.1f s", time.time() - started)
    logger.info("Passed publication gates: %d", report["n_passed_gates"])
    logger.info("Failed or unrun gates: %s", report["failed_or_unrun_gates"])
    logger.info("Open cell2_publication_readiness_summary.json first.")
    logger.info("Results: %s", outdir)
    logger.info("=" * 122)


if __name__ == "__main__":
    main()


[V68 Cell 2] startup: entering main()
[V68 Cell 2] checking Google Drive mount...
[V68 Cell 2] Google Drive check complete.
2026-08-07 12:23:43,224 | INFO | Startup preflight: frozen root resolved to /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas


INFO:V68_CELL2:Startup preflight: frozen root resolved to /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas


2026-08-07 12:23:43,227 | INFO | Startup preflight: locating the V68.1 module without recursive Drive search...


INFO:V68_CELL2:Startup preflight: locating the V68.1 module without recursive Drive search...


2026-08-07 12:23:43,229 | INFO | Startup preflight: importing /content/V68_1_patched_practical_standard_protocols_complete_F_biological_compatibility_atlas.py


...

INFO:V68_CELL2:Open cell2_publication_readiness_summary.json first.


2026-08-07 12:25:00,816 | INFO | Results: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/cell2_publication_validation


INFO:V68_CELL2:Results: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/cell2_publication_validation


2026-08-07 12:25:00,818 | INFO | ==========================================================================================================================


INFO:V68_CELL2:==========================================================================================================================


In [ ]:
#!/usr/bin/env python3
"""
V68.3.9 CELL 3 — STANDALONE EXAMPLE 1

Certified model-set preservation exposes out-of-atlas extrapolation risk.

This script is intentionally STANDALONE:
    - it does NOT import V68, V68.1, Cell 1, or Cell 2 Python code;
    - it only reads the already-produced frozen CSV results:
          <ROOT>/cell2_publication_validation/certified_source_atlas_states.csv
          <ROOT>/pairwise_critical_noise_profiles.csv
    - all UT constitutive equations needed for the figure are implemented here.

Fixed manuscript example
------------------------
Hidden source:
    YEOH2 #30

Certified complete-F compatibility set:
    YEOH2 | GENT | OGDEN2 | GP2

Certified atlas principal-stretch limit:
    lambda <= 2.00

Predeclared out-of-atlas validation stretch:
    lambda = 2.50

The outlier stretch is fixed a priori in this script. It is not optimized after
looking at model separation.

Scientific interpretation
-------------------------
Inside the certified complete-F atlas, all members of the compatibility set
explain the sample within the declared 3% resolution.

Outside the certified atlas, those same certified witness parameterizations are
propagated forward. Their predictions are NOT certified there. The purpose is
only to demonstrate extrapolation risk: choosing one compatible model without
additional evidence can discard the correct outlier behavior.

Outputs
-------
<ROOT>/cell3_example1_certified_set_outlier_v39_standalone/
    example1_selected_source.json
    example1_certified_witnesses.csv
    example1_outlier_predictions.csv
    example1_summary.json
    example1_certified_set_outlier_figure.png
    example1_certified_set_outlier_figure.pdf
"""

from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# Configuration
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/Optimal_Protocol/"
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)

CELL2_DIR = ROOT / "cell2_publication_validation"
OUTDIR = ROOT / "cell3_example1_certified_set_outlier_v39_standalone"
OUTDIR.mkdir(parents=True, exist_ok=True)

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]

FIXED_SOURCE_MODEL = "YEOH2"
FIXED_SOURCE_INDEX = 30

ATLAS_LAMBDA_MIN = 0.50
ATLAS_LAMBDA_MAX = 2.00
FIXED_OUTLIER_LAMBDA = 2.50
PLOT_LAMBDA_MAX = 2.80
N_PLOT_POINTS = 301

OUTLIER_MISS_TOLERANCE_PERCENT = 20.0
GENT_SINGULARITY_MARGIN = 1.0e-6
OGDEN_ALPHA_SERIES_THRESHOLD = 1.0e-5


# =============================================================================
# Drive
# =============================================================================

def ensure_drive() -> None:
    if Path("/content/drive/MyDrive").exists():
        print("[Cell 3] Google Drive already mounted.")
        return

    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "Google Drive is not mounted and google.colab is unavailable."
        ) from exc

    drive.mount("/content/drive", force_remount=False)


# =============================================================================
# Small utilities
# =============================================================================

def parse_json_dict(value) -> dict:
    if value is None:
        return {}
    s = str(value)
    if not s or s.lower() == "nan":
        return {}
    try:
        obj = json.loads(s)
    except Exception:
        return {}
    return obj if isinstance(obj, dict) else {}


def parse_json_array(value) -> np.ndarray:
    if value is None:
        return np.empty(0, dtype=float)
    s = str(value)
    if not s or s.lower() == "nan":
        return np.empty(0, dtype=float)
    try:
        return np.asarray(json.loads(s), dtype=float).reshape(-1)
    except Exception:
        return np.empty(0, dtype=float)


def as_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes"}


def certified_models_from_row(row: pd.Series) -> list[str]:
    return [
        x
        for x in str(row["certified_compatibility_set_at_3pct"]).split("|")
        if x
    ]


def certified_noise_fraction(row: pd.Series, model: str) -> float:
    return float(row[f"certified_critical_noise_to_{model.lower()}"])


# =============================================================================
# Standalone incompressible UT constitutive equations
# =============================================================================

def ut_bases(lam: np.ndarray):
    """
    Exact UT basis convention used in V68.

    b1 = 2(lambda^2 - lambda^-1)
    b2 = 2(lambda - lambda^-2)
    x  = I1 - 3 = lambda^2 + 2/lambda - 3
    y  = I2 - 3 = lambda^-2 + 2 lambda - 3
    """
    lam = np.asarray(lam, dtype=np.float64)

    b1 = 2.0 * (lam**2 - lam**-1.0)
    b2 = 2.0 * (lam - lam**-2.0)
    x = lam**2 + 2.0 / lam - 3.0
    y = lam**-2.0 + 2.0 * lam - 3.0

    return b1, b2, x, y


def ogden_power_difference_over_alpha(
    alpha: float,
    log_a: np.ndarray,
    log_b: np.ndarray,
) -> np.ndarray:
    """
    V68 signed Ogden convention:
        4 * (a^alpha - b^alpha) / alpha

    The alpha=0 removable singularity is evaluated by the same Taylor expansion
    convention used in V68.
    """
    alpha = float(alpha)
    a = np.asarray(log_a, dtype=np.float64)
    b = np.asarray(log_b, dtype=np.float64)

    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        a2, b2 = a * a, b * b
        a3, b3 = a2 * a, b2 * b
        a4, b4 = a3 * a, b3 * b
        a5, b5 = a4 * a, b4 * b

        return 4.0 * (
            (a - b)
            + 0.5 * alpha * (a2 - b2)
            + (alpha**2 / 6.0) * (a3 - b3)
            + (alpha**3 / 24.0) * (a4 - b4)
            + (alpha**4 / 120.0) * (a5 - b5)
        )

    ea = np.exp(alpha * a)
    eb = np.exp(alpha * b)
    return 4.0 * (ea - eb) / alpha


def ogden1_ut_template(alpha: float, lam: np.ndarray) -> np.ndarray:
    lam = np.asarray(lam, dtype=np.float64)
    log_lam = np.log(lam)

    return ogden_power_difference_over_alpha(
        alpha,
        log_lam,
        -0.5 * log_lam,
    )


def ut_template_from_physical(
    model: str,
    physical: dict,
    coordinate: np.ndarray,
    lam: np.ndarray,
) -> np.ndarray:
    """
    Return the V68 unit-response-scale UT template for one model.

    response_kPa = response_scale_kPa * unit_template
    """
    model = str(model)
    lam = np.asarray(lam, dtype=np.float64)
    coordinate = np.asarray(coordinate, dtype=float).reshape(-1)

    b1, b2, x, y = ut_bases(lam)

    if model == "NH":
        return b1

    if model == "MR":
        rho = float(
            physical.get(
                "rho",
                coordinate[0] if coordinate.size else 0.0,
            )
        )
        return (1.0 - rho) * b1 + rho * b2

    if model == "YEOH2":
        beta = float(
            physical.get(
                "beta",
                physical.get(
                    "coordinate",
                    coordinate[0] if coordinate.size else 0.0,
                ),
            )
        )
        return b1 * (1.0 + 2.0 * beta * x)

    if model == "GENT":
        gamma = float(
            physical.get(
                "gamma",
                physical.get(
                    "coordinate",
                    coordinate[0] if coordinate.size else 0.0,
                ),
            )
        )

        denominator = 1.0 - gamma * x
        out = np.full_like(b1, np.nan, dtype=np.float64)

        valid = denominator > GENT_SINGULARITY_MARGIN
        out[valid] = b1[valid] / denominator[valid]

        return out

    if model == "OGDEN1":
        alpha = float(
            physical.get(
                "alpha",
                coordinate[0] if coordinate.size else 2.0,
            )
        )
        return ogden1_ut_template(alpha, lam)

    if model == "OGDEN2":
        if "alpha1" not in physical or "alpha2" not in physical:
            raise ValueError(
                "OGDEN2 witness is missing alpha1/alpha2 in physical JSON."
            )

        alpha1 = float(physical["alpha1"])
        alpha2 = float(physical["alpha2"])

        if "weight1" in physical:
            weight1 = float(physical["weight1"])
        elif "weight2" in physical:
            weight1 = 1.0 - float(physical["weight2"])
        else:
            raise ValueError(
                "OGDEN2 witness is missing weight1/weight2 in physical JSON."
            )

        f1 = ogden1_ut_template(alpha1, lam)
        f2 = ogden1_ut_template(alpha2, lam)

        return weight1 * f1 + (1.0 - weight1) * f2

    if model == "GP2":
        required = ("rho", "beta20", "beta11", "beta02")
        missing = [k for k in required if k not in physical]
        if missing:
            raise ValueError(
                f"GP2 witness is missing physical parameters: {missing}"
            )

        rho = float(physical["rho"])
        beta20 = float(physical["beta20"])
        beta11 = float(physical["beta11"])
        beta02 = float(physical["beta02"])

        w1 = (
            (1.0 - rho)
            + 2.0 * beta20 * x
            + beta11 * y
        )
        w2 = (
            rho
            + beta11 * x
            + 2.0 * beta02 * y
        )

        return w1 * b1 + w2 * b2

    raise KeyError(f"Unknown model: {model}")


# =============================================================================
# Source / target physical-parameter reconstruction
# =============================================================================

def source_physical_parameters(row: pd.Series) -> dict:
    """
    Recover physical parameters directly from the certified source-atlas row.
    """
    model = str(row["source_model"])
    q = parse_json_array(row.get("coordinate_json", ""))

    if model == "NH":
        return {}

    if model == "MR":
        return {"rho": float(row["physical_rho"])}

    if model == "YEOH2":
        if "physical_beta" in row and pd.notna(row["physical_beta"]):
            return {"beta": float(row["physical_beta"])}
        return {"beta": float(q[0])}

    if model == "GENT":
        if "physical_gamma" in row and pd.notna(row["physical_gamma"]):
            return {"gamma": float(row["physical_gamma"])}
        return {"gamma": float(q[0])}

    if model == "OGDEN1":
        if "physical_alpha" in row and pd.notna(row["physical_alpha"]):
            return {"alpha": float(row["physical_alpha"])}
        return {"alpha": float(q[0])}

    if model == "OGDEN2":
        return {
            "alpha1": float(row["physical_alpha1"]),
            "alpha2": float(row["physical_alpha2"]),
            "weight1": float(row["physical_weight1"]),
        }

    if model == "GP2":
        return {
            "rho": float(row["physical_rho"]),
            "beta20": float(row["physical_beta20"]),
            "beta11": float(row["physical_beta11"]),
            "beta02": float(row["physical_beta02"]),
        }

    raise KeyError(model)


def pairwise_lookup_table(pairwise_df: pd.DataFrame):
    return pairwise_df.set_index(
        ["source_model", "source_index", "target_model"],
        drop=False,
    )


def reconstruct_certified_witness(
    source_row: pd.Series,
    target_model: str,
    pair_index,
    lam: np.ndarray,
):
    """
    Reconstruct the actual frozen witness used for the certified relation.

    For the source family itself, use the source atlas coordinates and scale.
    For another compatible family, use the frozen pairwise best physical
    parameters and scale.

    This also handles exact nested witnesses such as YEOH2 -> GP2, whose
    coordinate_json can contain NaNs, because the physical JSON contains the
    exact embedded-stratum parameters.
    """
    source_model = str(source_row["source_model"])
    source_index = int(source_row["source_index"])

    if target_model == source_model:
        coordinate = parse_json_array(source_row["coordinate_json"])
        physical = source_physical_parameters(source_row)
        response_scale = float(source_row["source_response_scale_kpa"])
        mu0 = float(source_row["source_mu0_kpa"])
        solution_source = "exact_source_state"

    else:
        key = (source_model, source_index, target_model)

        try:
            fit = pair_index.loc[key]
        except KeyError as exc:
            raise KeyError(
                f"Missing frozen pairwise witness for "
                f"{source_model}:{source_index}->{target_model}"
            ) from exc

        if isinstance(fit, pd.DataFrame):
            fit = fit.iloc[0]

        coordinate = parse_json_array(
            fit.get("best_target_coordinate_json", "")
        )
        physical = parse_json_dict(
            fit.get("best_target_physical_json", "")
        )
        response_scale = float(fit["best_target_scale_kpa"])
        mu0 = float(fit["best_target_mu0_kpa"])
        solution_source = str(fit["solution_source"])

    template = ut_template_from_physical(
        target_model,
        physical,
        coordinate,
        lam,
    )

    prediction = response_scale * template

    return {
        "target_model": target_model,
        "certified_required_noise_fraction":
            certified_noise_fraction(source_row, target_model),
        "certified_required_noise_percent":
            100.0 * certified_noise_fraction(source_row, target_model),
        "coordinate_json": json.dumps(
            coordinate.tolist(),
            allow_nan=True,
        ),
        "physical_json": json.dumps(
            physical,
            sort_keys=True,
            allow_nan=True,
        ),
        "response_scale_kpa": response_scale,
        "mu0_kpa": mu0,
        "solution_source": solution_source,
        "prediction": prediction,
    }


# =============================================================================
# Fixed manuscript example
# =============================================================================

def evaluate_fixed_example(
    source_row: pd.Series,
    pair_index,
    lam: np.ndarray,
):
    compatible_models = certified_models_from_row(source_row)

    if FIXED_SOURCE_MODEL not in compatible_models:
        raise RuntimeError(
            "The fixed source family is absent from its own certified set."
        )

    witnesses = []
    predictions = []

    for model in compatible_models:
        w = reconstruct_certified_witness(
            source_row,
            model,
            pair_index,
            lam,
        )

        # Hard internal consistency check.
        if (
            w["certified_required_noise_fraction"]
            > 0.03 + 1.0e-10
        ):
            raise RuntimeError(
                f"{model} is listed as compatible but its certified "
                f"required noise is "
                f"{100*w['certified_required_noise_fraction']:.4f}%."
            )

        predictions.append(w.pop("prediction"))
        witnesses.append(w)

    P = np.stack(predictions)

    source_pos = compatible_models.index(FIXED_SOURCE_MODEL)
    truth = P[source_pos]

    outlier_matches = np.flatnonzero(
        np.isclose(
            lam,
            FIXED_OUTLIER_LAMBDA,
            rtol=0.0,
            atol=1.0e-12,
        )
    )

    if outlier_matches.size != 1:
        raise RuntimeError(
            f"Expected one exact lambda={FIXED_OUTLIER_LAMBDA} grid point; "
            f"found {outlier_matches.size}."
        )

    out_idx = int(outlier_matches[0])

    if not np.isfinite(truth[out_idx]):
        raise RuntimeError(
            "Hidden source has no finite response at the fixed outlier."
        )

    errors = np.full(len(compatible_models), np.nan, dtype=float)

    for i in range(len(compatible_models)):
        if np.isfinite(P[i, out_idx]):
            errors[i] = (
                100.0
                * abs(P[i, out_idx] - truth[out_idx])
                / max(abs(float(truth[out_idx])), 1.0e-12)
            )

    finite_at_outlier = np.isfinite(P[:, out_idx])
    finite_values = P[finite_at_outlier, out_idx]

    if finite_values.size < 2:
        raise RuntimeError(
            "Fewer than two compatible models have finite continuation "
            "at the fixed outlier."
        )

    spread_percent = (
        100.0
        * (float(np.max(finite_values)) - float(np.min(finite_values)))
        / max(abs(float(truth[out_idx])), 1.0e-12)
    )

    n_finite = int(np.sum(np.isfinite(errors)))
    n_miss = int(
        np.sum(
            errors[np.isfinite(errors)]
            > OUTLIER_MISS_TOLERANCE_PERCENT
        )
    )
    miss_fraction = n_miss / max(n_finite, 1)

    # Pointwise spread curve for visualization only.
    finite_count_curve = np.sum(np.isfinite(P), axis=0)

    with np.errstate(invalid="ignore"):
        low_curve = np.nanmin(P, axis=0)
        high_curve = np.nanmax(P, axis=0)

    truth_floor = max(
        0.02 * float(np.nanmax(np.abs(truth))),
        1.0e-12,
    )
    denom_curve = np.maximum(np.abs(truth), truth_floor)

    spread_curve = (
        100.0 * (high_curve - low_curve) / denom_curve
    )
    spread_curve[finite_count_curve < 2] = np.nan

    witness_df = pd.DataFrame(witnesses)
    witness_df["prediction_at_fixed_outlier_kpa"] = P[:, out_idx]
    witness_df["error_at_fixed_outlier_percent"] = errors
    witness_df["finite_at_fixed_outlier"] = finite_at_outlier
    witness_df["is_hidden_source_family"] = (
        witness_df["target_model"].astype(str)
        == FIXED_SOURCE_MODEL
    )

    return {
        "source_model": FIXED_SOURCE_MODEL,
        "source_index": FIXED_SOURCE_INDEX,
        "source_mu0_kpa": float(source_row["source_mu0_kpa"]),
        "coordinate_json": str(source_row["coordinate_json"]),
        "biological_support_zone": str(
            source_row["biological_support_zone"]
        ),
        "certified_compatibility_set":
            "|".join(compatible_models),
        "n_certified_models": int(len(compatible_models)),
        "fixed_outlier_lambda":
            float(FIXED_OUTLIER_LAMBDA),
        "spread_percent_at_fixed_outlier":
            float(spread_percent),
        "n_finite_models_at_fixed_outlier":
            int(n_finite),
        "n_models_miss_tolerance":
            int(n_miss),
        "miss_tolerance_percent":
            float(OUTLIER_MISS_TOLERANCE_PERCENT),
        "single_family_miss_fraction":
            float(miss_fraction),
        "_models": compatible_models,
        "_truth": truth,
        "_predictions": P,
        "_spread": spread_curve,
        "_out_idx": out_idx,
        "_witness_df": witness_df,
    }


# =============================================================================
# Figure
# =============================================================================

def make_figure(
    result: dict,
    source_row: pd.Series,
    lam: np.ndarray,
):
    models = result["_models"]
    P = result["_predictions"]
    truth = result["_truth"]
    spread = result["_spread"]
    witness_df = result["_witness_df"]
    out_idx = result["_out_idx"]

    fig = plt.figure(figsize=(15.2, 10.8))

    # ------------------------------------------------------------------
    # A — UT projection of the complete-F certified set
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 1)

    atlas_mask = lam <= ATLAS_LAMBDA_MAX + 1.0e-12

    for i, model in enumerate(models):
        ax.plot(
            lam[atlas_mask],
            P[i, atlas_mask],
            linewidth=2.0,
            label=model,
        )

    ax.plot(
        lam[atlas_mask],
        truth[atlas_mask],
        linewidth=3.0,
        linestyle="--",
        label=f"Hidden source ({FIXED_SOURCE_MODEL})",
    )

    ax.set_xlim(ATLAS_LAMBDA_MIN, ATLAS_LAMBDA_MAX)
    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel("Nominal stress (kPa)")
    ax.set_title(
        "A. UT projection of complete-F-compatible models\n"
        f"Certified set: {result['certified_compatibility_set']}"
    )
    ax.legend(fontsize=7.5, loc="best", ncol=2)

    # ------------------------------------------------------------------
    # B — Complete-F certification scores
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 2)

    scores = np.array(
        [
            100.0
            * certified_noise_fraction(source_row, m)
            for m in MODELS
        ],
        dtype=float,
    )

    compatible_set = set(models)
    compatible = np.array(
        [m in compatible_set for m in MODELS],
        dtype=bool,
    )

    x = np.arange(len(MODELS))

    ax.bar(
        x[~compatible],
        scores[~compatible],
        alpha=0.35,
        label="Not compatible",
    )
    ax.bar(
        x[compatible],
        scores[compatible],
        alpha=0.90,
        label="Certified compatible",
    )

    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.8,
        label="3% threshold",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(MODELS, rotation=35)
    ax.set_ylabel("Certified required noise (%)")
    ax.set_title(
        "B. Compatibility was established over the complete-F atlas\n"
        "No unique model is justified within the declared 3% resolution"
    )
    ax.legend(fontsize=8, loc="best")

    # ------------------------------------------------------------------
    # C — Fixed out-of-atlas extrapolation
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 3)

    ax.axvspan(
        ATLAS_LAMBDA_MIN,
        ATLAS_LAMBDA_MAX,
        alpha=0.10,
        label="Certified atlas domain",
    )
    ax.axvspan(
        ATLAS_LAMBDA_MAX,
        PLOT_LAMBDA_MAX,
        alpha=0.06,
        label="Out-of-atlas extrapolation",
    )

    for i, model in enumerate(models):
        ax.plot(
            lam,
            P[i],
            linewidth=2.0,
            label=model,
        )

    ax.plot(
        lam,
        truth,
        linewidth=3.0,
        linestyle="--",
        label=f"Hidden truth ({FIXED_SOURCE_MODEL})",
    )

    ax.axvline(
        FIXED_OUTLIER_LAMBDA,
        linestyle=":",
        linewidth=2.0,
        label=f"Predeclared λ={FIXED_OUTLIER_LAMBDA:.2f}",
    )

    ax.scatter(
        [FIXED_OUTLIER_LAMBDA],
        [truth[out_idx]],
        marker="*",
        s=170,
        edgecolor="black",
        linewidth=0.8,
        zorder=7,
    )

    ax.set_xlim(ATLAS_LAMBDA_MIN, PLOT_LAMBDA_MAX)
    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel("Nominal stress (kPa)")
    ax.set_title(
        "C. The same certified-compatible witnesses diverge outside the atlas\n"
        f"Spread at fixed λ={FIXED_OUTLIER_LAMBDA:.2f}: "
        f"{result['spread_percent_at_fixed_outlier']:.1f}%"
    )
    ax.legend(fontsize=7.1, loc="best", ncol=2)

    # ------------------------------------------------------------------
    # D — Single-family miss risk at fixed outlier
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 4)

    finite = (
        witness_df["finite_at_fixed_outlier"]
        .astype(bool)
        .to_numpy()
    )
    errors = witness_df[
        "error_at_fixed_outlier_percent"
    ].to_numpy(float)

    xx = np.arange(len(witness_df))

    ax.bar(
        xx[finite],
        errors[finite],
    )

    for i in xx[~finite]:
        ax.text(
            i,
            0.03,
            "no finite\ncontinuation",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=7.5,
            rotation=90,
        )

    ax.axhline(
        OUTLIER_MISS_TOLERANCE_PERCENT,
        linestyle="--",
        linewidth=1.8,
        label=(
            f"{OUTLIER_MISS_TOLERANCE_PERCENT:.0f}% "
            "outlier-error threshold"
        ),
    )

    ax.set_xticks(xx)
    ax.set_xticklabels(
        witness_df["target_model"].astype(str),
        rotation=35,
    )
    ax.set_ylabel(
        f"Relative error at λ={FIXED_OUTLIER_LAMBDA:.2f} (%)"
    )
    ax.set_title(
        "D. Arbitrarily selecting one compatible family can lose the correct behavior\n"
        f"{result['n_models_miss_tolerance']}/"
        f"{result['n_finite_models_at_fixed_outlier']} "
        f"finite compatible families exceed "
        f"{OUTLIER_MISS_TOLERANCE_PERCENT:.0f}% error"
    )
    ax.legend(fontsize=8, loc="best")

    fig.suptitle(
        "Example 1 — Certified model-set preservation exposes out-of-atlas extrapolation risk",
        fontsize=15.3,
    )

    fig.tight_layout(rect=[0, 0.01, 1, 0.965])

    png = OUTDIR / "example1_certified_set_outlier_figure.png"
    pdf = OUTDIR / "example1_certified_set_outlier_figure.pdf"

    fig.savefig(
        png,
        dpi=320,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    return png, pdf


# =============================================================================
# Main
# =============================================================================

def main():
    ensure_drive()

    atlas_path = (
        CELL2_DIR / "certified_source_atlas_states.csv"
    )
    pairwise_path = (
        ROOT / "pairwise_critical_noise_profiles.csv"
    )

    if not atlas_path.is_file():
        raise FileNotFoundError(
            f"Missing certified atlas CSV:\n{atlas_path}"
        )

    if not pairwise_path.is_file():
        raise FileNotFoundError(
            f"Missing frozen pairwise witness CSV:\n{pairwise_path}"
        )

    atlas_df = pd.read_csv(atlas_path)
    pairwise_df = pd.read_csv(pairwise_path)

    print(
        f"[Cell 3] Loaded {len(atlas_df)} certified source states."
    )
    print(
        f"[Cell 3] Loaded {len(pairwise_df)} frozen pairwise witnesses."
    )
    print("[Cell 3] Standalone mode: no V68 Python file imported.")

    source_matches = atlas_df[
        (
            atlas_df["source_model"].astype(str)
            == FIXED_SOURCE_MODEL
        )
        & (
            atlas_df["source_index"].astype(int)
            == FIXED_SOURCE_INDEX
        )
    ].copy()

    if len(source_matches) != 1:
        raise RuntimeError(
            f"Expected exactly one "
            f"{FIXED_SOURCE_MODEL} #{FIXED_SOURCE_INDEX}; "
            f"found {len(source_matches)}."
        )

    source_row = source_matches.iloc[0]

    if as_bool(source_row["certified_label_changed"]):
        raise RuntimeError(
            "The fixed source changed label during Cell-2 certification."
        )

    print(
        f"[Cell 3] Fixed hidden source: "
        f"{FIXED_SOURCE_MODEL} #{FIXED_SOURCE_INDEX}"
    )
    print(
        f"[Cell 3] Certified compatibility set: "
        f"{source_row['certified_compatibility_set_at_3pct']}"
    )
    print(
        f"[Cell 3] Fixed outlier stretch: "
        f"lambda={FIXED_OUTLIER_LAMBDA:.2f} "
        f"(atlas maximum={ATLAS_LAMBDA_MAX:.2f})"
    )

    # Ensure 2.50 is represented exactly.
    base_lam = np.linspace(
        ATLAS_LAMBDA_MIN,
        PLOT_LAMBDA_MAX,
        N_PLOT_POINTS,
        dtype=np.float64,
    )
    lam = np.unique(
        np.concatenate(
            [
                base_lam,
                np.array(
                    [FIXED_OUTLIER_LAMBDA],
                    dtype=np.float64,
                ),
            ]
        )
    )

    pair_index = pairwise_lookup_table(pairwise_df)

    result = evaluate_fixed_example(
        source_row,
        pair_index,
        lam,
    )

    # ------------------------------------------------------------------
    # Save tables
    # ------------------------------------------------------------------
    public_result = {
        k: v
        for k, v in result.items()
        if not k.startswith("_")
    }

    with (
        OUTDIR / "example1_selected_source.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            public_result,
            f,
            indent=2,
        )

    witness_df = result["_witness_df"].copy()
    witness_df.to_csv(
        OUTDIR / "example1_certified_witnesses.csv",
        index=False,
    )

    curve_df = pd.DataFrame(
        {"lambda": lam}
    )
    curve_df["hidden_truth"] = result["_truth"]

    for i, model in enumerate(result["_models"]):
        curve_df[
            f"{model}_certified_witness"
        ] = result["_predictions"][i]

    curve_df["is_inside_certified_atlas"] = (
        lam <= ATLAS_LAMBDA_MAX + 1.0e-12
    ).astype(int)

    curve_df["is_fixed_outlier"] = np.isclose(
        lam,
        FIXED_OUTLIER_LAMBDA,
        rtol=0.0,
        atol=1.0e-12,
    ).astype(int)

    curve_df["prediction_spread_percent"] = (
        result["_spread"]
    )

    curve_df.to_csv(
        OUTDIR / "example1_outlier_predictions.csv",
        index=False,
    )

    summary = {
        "scientific_message": (
            "The complete-F atlas identifies every constitutive family "
            "that explains the sample within 3% over the certified domain. "
            "Those same certified witnesses are propagated to a predeclared "
            "out-of-atlas stretch lambda=2.50. Their separation shows that "
            "arbitrarily selecting one compatible family can discard the "
            "correct outlier behavior, whereas retaining the complete "
            "compatibility set preserves the constitutive uncertainty."
        ),
        "scope_statement": (
            "No prediction at lambda>2.0 is certified by the atlas. "
            "The out-of-atlas calculation is used only to demonstrate "
            "extrapolation risk."
        ),
        "source_selection_statement": (
            "YEOH2 #30 was fixed from the previously completed screening. "
            "This standalone script performs no source search and no "
            "outlier-stretch optimization."
        ),
        **public_result,
    }

    with (
        OUTDIR / "example1_summary.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            summary,
            f,
            indent=2,
        )

    png, pdf = make_figure(
        result,
        source_row,
        lam,
    )

    print("\n" + "=" * 112)
    print("[Cell 3] EXAMPLE 1 COMPLETE — STANDALONE")
    print(
        f"[Cell 3] Hidden source: "
        f"{result['source_model']} #{result['source_index']}"
    )
    print(
        f"[Cell 3] Certified compatibility set: "
        f"{result['certified_compatibility_set']}"
    )
    print(
        f"[Cell 3] Fixed outlier stretch: "
        f"lambda={result['fixed_outlier_lambda']:.2f}"
    )
    print(
        f"[Cell 3] Compatible-model spread at fixed outlier: "
        f"{result['spread_percent_at_fixed_outlier']:.2f}%"
    )
    print(
        f"[Cell 3] Single-family miss fraction (> "
        f"{OUTLIER_MISS_TOLERANCE_PERCENT:.0f}% error): "
        f"{100*result['single_family_miss_fraction']:.1f}% "
        f"({result['n_models_miss_tolerance']}/"
        f"{result['n_finite_models_at_fixed_outlier']})"
    )

    # Useful explicit model-by-model values.
    print("[Cell 3] Model predictions at lambda=2.50:")
    for _, row in witness_df.iterrows():
        pred = row["prediction_at_fixed_outlier_kpa"]
        err = row["error_at_fixed_outlier_percent"]

        pred_text = (
            f"{float(pred):.6g} kPa"
            if np.isfinite(pred)
            else "no finite continuation"
        )
        err_text = (
            f"{float(err):.2f}%"
            if np.isfinite(err)
            else "n/a"
        )

        print(
            f"           {row['target_model']:<8s} "
            f"{pred_text:<24s} error={err_text}"
        )

    print(f"[Cell 3] Figure PNG: {png}")
    print(f"[Cell 3] Figure PDF: {pdf}")
    print(f"[Cell 3] Results: {OUTDIR}")
    print("=" * 112)


if __name__ == "__main__":
    main()


[Cell 3] Google Drive already mounted.
[Cell 3] Loaded 2631 certified source states.
[Cell 3] Loaded 15786 frozen pairwise witnesses.
[Cell 3] Standalone mode: no V68 Python file imported.
[Cell 3] Fixed hidden source: YEOH2 #30
[Cell 3] Certified compatibility set: YEOH2|GENT|OGDEN2|GP2
[Cell 3] Fixed outlier stretch: lambda=2.50 (atlas maximum=2.00)

[Cell 3] EXAMPLE 1 COMPLETE — STANDALONE
[Cell 3] Hidden source: YEOH2 #30
[Cell 3] Certified compatibility set: YEOH2|GENT|OGDEN2|GP2
[Cell 3] Fixed outlier stretch: lambda=2.50
[Cell 3] Compatible-model spread at fixed outlier: 36.37%
[Cell 3] Single-family miss fraction (> 20% error): 50.0% (2/4)
[Cell 3] Model predictions at lambda=2.50:
           YEOH2    1893.57 kPa              error=0.00%
           GENT     2582.18 kPa              error=36.37%
           OGDEN2   2351.14 kPa              error=24.16%
           GP2      1893.57 kPa              error=0.00%
[Cell 3] Figure PNG: /content/drive/MyDrive/Optimal_Protocol/V68_practi

In [ ]:
#!/usr/bin/env python3
"""
V68.4.5 CELL 3 — STANDALONE EXAMPLE 2
When constitutive complexity is actually necessary.

Scientific question
-------------------
Can conventional loading protocols make a simpler constitutive model look
adequate even though the complete-F atlas proves that the simpler model is not
sufficient?

This standalone example searches biologically supported source states whose
CERTIFIED complete-F compatibility set is unique:
    OGDEN2 only
or
    GP2 only.

For each candidate source:
    1) generate conventional UT + equibiaxial BT + simple-shear responses;
    2) fit the simpler one-shape-parameter families
         NH, MR, YEOH2, GENT, OGDEN1
       to those standard protocols only;
    3) keep examples where a simpler model fits the standard protocols very well;
    4) use the frozen complete-F certification scores to show that the same
       simpler model nevertheless requires >3% noise over the full atlas;
    5) locate the in-atlas deformation state F* where the fitted simpler model
       deviates most strongly from the hidden complex source;
    6) visualize exactly what deformation exposes the missing complexity.

The code does NOT import V68/V68.1 or any earlier Python cell.
It only reads:
    <ROOT>/cell2_publication_validation/certified_source_atlas_states.csv

All constitutive equations used here are implemented directly below.

Figure
------
A. Conventional UT/BT/SH: complex truth vs best simpler fit
B. Complete-F certified required-noise scores for all seven families
C. Dense complete-F map of simpler-model prediction error
D. Stress components at the maximally discriminating in-atlas state F*

Outputs
-------
<ROOT>/cell3_example2_complexity_required_v45_standalone/
    example2_search_results.csv
    example2_selected_source.json
    example2_standard_protocol_fit_summary.csv
    example2_complete_F_error_map.csv
    example2_Fstar.json
    example2_summary.json
    example2_complexity_required_figure.png
    example2_complexity_required_figure.pdf
"""

from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar


# =============================================================================
# Configuration
# =============================================================================

# Google Drive may be mounted at the usual /content/drive, or at a clean
# fallback mountpoint if /content/drive contains stale local files.
MYDRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_BASENAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)

def preferred_root() -> Path:
    return MYDRIVE_ROOT / "Optimal_Protocol" / PROJECT_BASENAME

# These are resolved in main() after Drive discovery.
ROOT = preferred_root()
CELL2_DIR = ROOT / "cell2_publication_validation"
OUTDIR = ROOT / "cell3_example2_complexity_required_v45_standalone"

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
SIMPLE_MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1"]
COMPLEX_SOURCE_MODELS = ["OGDEN2", "GP2"]

# Standard experimental protocols used only for the conventional fit.
# Keep all three modes at the manuscript-declared resolution: 81 states/mode.
N_STANDARD_PER_MODE = 81
UT = np.linspace(0.70, 1.50, N_STANDARD_PER_MODE)
BT = np.linspace(0.80, 1.25, N_STANDARD_PER_MODE)
SH = np.linspace(-0.50, 0.50, N_STANDARD_PER_MODE)

if not (len(UT) == len(BT) == len(SH) == N_STANDARD_PER_MODE):
    raise RuntimeError(
        "Standard-protocol discretization mismatch: "
        "UT, BT, and SH must each contain 81 states."
    )

# Complete-F visualization domain.
LAMBDA_MIN = 0.50
LAMBDA_MAX = 2.00
N_LOG_GRID = 121

# Source search.
MAX_SOURCE_CASES_PER_COMPLEX_MODEL = 120
MAX_STANDARD_FIT_NRMSE_PERCENT = 3.0

# Require the complete-F certification to reject the simpler fit clearly.
MIN_COMPLETE_F_REQUIRED_NOISE_PERCENT = 4.0

# Prefer a clear but not pathological complete-F failure.
TARGET_COMPLETE_F_NOISE_PERCENT = 8.0

# Numerical settings.
GENT_SINGULARITY_MARGIN = 1.0e-6
OGDEN_ALPHA_SERIES_THRESHOLD = 1.0e-5
EPS = 1.0e-12


# =============================================================================
# Drive
# =============================================================================

def _expected_certified_csv_under(mydrive_root: Path) -> Path:
    return (
        mydrive_root
        / "Optimal_Protocol"
        / PROJECT_BASENAME
        / "cell2_publication_validation"
        / "certified_source_atlas_states.csv"
    )


def ensure_drive() -> None:
    """
    Ensure that this notebook can see the same Google Drive that contains V68.

    Important subtlety:
    merely having /content/drive/MyDrive as a directory does NOT prove that
    Google Drive is mounted. A previous failed script can create local folders
    there, causing Colab's drive.mount('/content/drive') to fail with
    "Mountpoint must not already contain files".

    Strategy:
      1) If the expected certified atlas is already visible, use it.
      2) Otherwise mount Google Drive at a clean fallback mountpoint
         /content/v68_drive instead of touching/deleting /content/drive.
    """
    global MYDRIVE_ROOT, ROOT, CELL2_DIR, OUTDIR

    usual = Path("/content/drive/MyDrive")
    expected_usual = _expected_certified_csv_under(usual)

    if expected_usual.is_file():
        MYDRIVE_ROOT = usual
        ROOT = preferred_root()
        CELL2_DIR = ROOT / "cell2_publication_validation"
        OUTDIR = ROOT / "cell3_example2_complexity_required_v45_standalone"
        print("[Example 2] Google Drive data visible at /content/drive/MyDrive.")
        return

    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "Google Drive data are not visible and google.colab is unavailable."
        ) from exc

    fallback_mount = Path("/content/v68_drive")

    # Make sure the fallback mountpoint is clean. This is a dedicated path owned
    # only by this script; unlike /content/drive we do not delete user content.
    if fallback_mount.exists():
        try:
            entries = list(fallback_mount.iterdir())
        except Exception:
            entries = []
        if entries:
            # If this is already a useful Drive mount, reuse it.
            fallback_mydrive = fallback_mount / "MyDrive"
            if fallback_mydrive.is_dir():
                MYDRIVE_ROOT = fallback_mydrive
                ROOT = preferred_root()
                CELL2_DIR = ROOT / "cell2_publication_validation"
                OUTDIR = ROOT / "cell3_example2_complexity_required_v45_standalone"
                print("[Example 2] Reusing Drive at /content/v68_drive/MyDrive.")
                return

            raise RuntimeError(
                "Dedicated fallback mountpoint /content/v68_drive is non-empty "
                "but does not look like a Google Drive mount. Restart the runtime "
                "or remove that temporary directory before retrying."
            )

    print(
        "[Example 2] /content/drive is not exposing the real MyDrive. "
        "Mounting Google Drive at the clean fallback /content/v68_drive ..."
    )
    print(
        "[Example 2] When authorization appears, choose the Google account "
        "that contains Optimal_Protocol/V68_practical_standard_protocols_..."
    )

    drive.mount(str(fallback_mount), force_remount=False)

    fallback_mydrive = fallback_mount / "MyDrive"
    if not fallback_mydrive.is_dir():
        raise RuntimeError(
            "Google Drive mounted, but /content/v68_drive/MyDrive was not created."
        )

    MYDRIVE_ROOT = fallback_mydrive
    ROOT = preferred_root()
    CELL2_DIR = ROOT / "cell2_publication_validation"
    OUTDIR = ROOT / "cell3_example2_complexity_required_v45_standalone"

    print(f"[Example 2] Active MyDrive root: {MYDRIVE_ROOT}")



def _looks_like_certified_atlas(path: Path) -> tuple[bool, str]:
    required_columns = {
        "source_model",
        "source_index",
        "source_mu0_kpa",
        "source_response_scale_kpa",
        "certified_compatibility_set_at_3pct",
        "certified_label_changed",
    }
    try:
        header = pd.read_csv(path, nrows=3)
    except Exception as exc:
        return False, f"unreadable: {exc}"

    missing = sorted(required_columns.difference(header.columns))
    if missing:
        return False, f"missing columns: {missing}"
    return True, "ok"


def resolve_certified_atlas_path() -> tuple[Path, Path]:
    """
    Locate the authoritative frozen Cell-2 certified atlas even if the V68
    directory was moved/renamed or exists as a Drive duplicate such as "(1)".

    Returns
    -------
    (resolved_root, certified_atlas_csv)

    Safety:
      * exact preferred path is tried first;
      * then Optimal_Protocol is searched;
      * only if needed, all MyDrive is searched;
      * candidate CSVs must contain the certified-atlas columns;
      * if more than one plausible atlas remains, the script stops rather than
        silently choosing a potentially stale version.
    """
    preferred = preferred_root()
    expected = (
        preferred
        / "cell2_publication_validation"
        / "certified_source_atlas_states.csv"
    )
    if expected.is_file():
        ok, reason = _looks_like_certified_atlas(expected)
        if ok:
            return preferred, expected
        raise RuntimeError(
            f"The expected certified atlas exists but failed validation:\n"
            f"{expected}\nReason: {reason}"
        )

    mydrive = MYDRIVE_ROOT
    optimal = mydrive / "Optimal_Protocol"

    search_roots = []
    if optimal.is_dir():
        search_roots.append(optimal)
    if mydrive.is_dir():
        search_roots.append(mydrive)

    all_candidates: list[Path] = []
    searched: list[Path] = []

    for base in search_roots:
        searched.append(base)
        found = []
        for pattern in (
            "certified_source_atlas_states.csv",
            "certified_source_atlas_states*.csv",
        ):
            try:
                found.extend(base.rglob(pattern))
            except Exception:
                pass

        for p in found:
            if p.is_file() and p not in all_candidates:
                all_candidates.append(p)

        # If Optimal_Protocol already yielded a valid candidate, do not scan
        # all MyDrive unnecessarily.
        if base == optimal:
            valid_here = [
                p for p in all_candidates
                if _looks_like_certified_atlas(p)[0]
            ]
            if valid_here:
                break

    valid: list[Path] = []
    rejected: list[tuple[Path, str]] = []
    for p in all_candidates:
        ok, reason = _looks_like_certified_atlas(p)
        if ok:
            valid.append(p)
        else:
            rejected.append((p, reason))

    def inferred_root(p: Path) -> Path:
        # Canonical layout:
        # <ROOT>/cell2_publication_validation/certified_source_atlas_states.csv
        parent = p.parent
        if parent.name.startswith("cell2_publication_validation"):
            return parent.parent
        # Conservative fallback if the file was moved into a nested Cell-2 dir.
        for anc in p.parents:
            if anc.name.startswith(
                "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
            ):
                return anc
        return p.parent

    if len(valid) == 1:
        root = inferred_root(valid[0])
        print(
            "[Example 2] Auto-discovered certified atlas:\n"
            f"  {valid[0]}"
        )
        print(
            "[Example 2] Resolved frozen V68 root:\n"
            f"  {root}"
        )
        return root, valid[0]

    if len(valid) > 1:
        # Strong preference for the exact V68 project-name family and canonical
        # Cell-2 directory.
        preferred = [
            p for p in valid
            if (
                "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
                in str(p)
                and p.parent.name.startswith("cell2_publication_validation")
            )
        ]
        if len(preferred) == 1:
            root = inferred_root(preferred[0])
            print(
                "[Example 2] Multiple certified-atlas files exist; selected the "
                "unique canonical V68 Cell-2 candidate:\n"
                f"  {preferred[0]}"
            )
            return root, preferred[0]

        listing = "\n".join(f"  - {p}" for p in valid[:30])
        raise FileNotFoundError(
            "Multiple valid certified_source_atlas_states CSVs were found, so "
            "the script will not guess which frozen atlas is authoritative.\n"
            f"Candidates:\n{listing}\n\n"
            "Set PREFERRED_ROOT at the top of the script to the intended V68 folder."
        )

    # If no certified CSV exists, look for evidence of where the V68 atlas is.
    evidence: list[Path] = []
    for base in search_roots:
        for pattern in (
            "seven_model_source_atlas_states.csv",
            "pairwise_critical_noise_profiles.csv",
            "cell2_publication_readiness_summary.json",
            "boundary_certification_summary.csv",
            "example2_search_results.csv",
        ):
            try:
                for p in base.rglob(pattern):
                    if p.is_file() and p not in evidence:
                        evidence.append(p)
            except Exception:
                pass
        if evidence and base == optimal:
            break

    searched_text = "\n".join(f"  - {p}" for p in searched)
    evidence_text = (
        "\n".join(f"  - {p}" for p in evidence[:40])
        if evidence
        else "  (none found)"
    )

    # Give a small mounted-Drive directory diagnostic after the refresh attempt.
    top_level = []
    optimal_children = []
    try:
        top_level = sorted(p.name for p in MYDRIVE_ROOT.iterdir())[:40]
    except Exception:
        pass
    try:
        optimal_children = sorted(
            p.name for p in (MYDRIVE_ROOT / "Optimal_Protocol").iterdir()
        )[:60]
    except Exception:
        pass

    top_text = (
        "\n".join(f"  - {name}" for name in top_level)
        if top_level else "  (unable to list MyDrive)"
    )
    optimal_text = (
        "\n".join(f"  - {name}" for name in optimal_children)
        if optimal_children else "  (Optimal_Protocol is absent or unreadable)"
    )

    raise FileNotFoundError(
        "No valid certified_source_atlas_states.csv was found after a fresh Drive mount.\n\n"
        "Searched:\n"
        f"{searched_text}\n\n"
        "Related V68/Cell-2 evidence found:\n"
        f"{evidence_text}\n\n"
        "Visible MyDrive top-level entries:\n"
        f"{top_text}\n\n"
        "Visible Optimal_Protocol entries:\n"
        f"{optimal_text}\n\n"
        "Because the file is visible in the Google Drive web UI but not through "
        "the active MyDrive mount, this indicates that Colab is authorized to a "
        "different Google account or the mount is not exposing the same My Drive. "
        "Re-run the cell and authorize the exact Google account that owns the V68 folder."
    )


# =============================================================================
# General utilities
# =============================================================================

def parse_json_array(value) -> np.ndarray:
    try:
        return np.asarray(json.loads(str(value)), dtype=float).reshape(-1)
    except Exception:
        return np.empty(0, dtype=float)


def as_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes"}


def certified_models(row: pd.Series) -> list[str]:
    return [
        m
        for m in str(row["certified_compatibility_set_at_3pct"]).split("|")
        if m
    ]


def certified_required_noise_percent(row: pd.Series, model: str) -> float:
    return 100.0 * float(
        row[f"certified_critical_noise_to_{model.lower()}"]
    )


def safe_float(row: pd.Series, key: str, default=np.nan) -> float:
    if key in row and pd.notna(row[key]):
        return float(row[key])
    return float(default)


# =============================================================================
# Physical parameter extraction from frozen atlas states
# =============================================================================

def source_physical_parameters(row: pd.Series) -> dict:
    model = str(row["source_model"])
    q = parse_json_array(row.get("coordinate_json", ""))

    if model == "NH":
        return {}

    if model == "MR":
        value = safe_float(row, "physical_rho")
        if not np.isfinite(value):
            value = float(q[0])
        return {"rho": value}

    if model == "YEOH2":
        value = safe_float(row, "physical_beta")
        if not np.isfinite(value):
            value = float(q[0])
        return {"beta": value}

    if model == "GENT":
        value = safe_float(row, "physical_gamma")
        if not np.isfinite(value):
            value = float(q[0])
        return {"gamma": value}

    if model == "OGDEN1":
        value = safe_float(row, "physical_alpha")
        if not np.isfinite(value):
            value = float(q[0])
        return {"alpha": value}

    if model == "OGDEN2":
        return {
            "alpha1": float(row["physical_alpha1"]),
            "alpha2": float(row["physical_alpha2"]),
            "weight1": float(row["physical_weight1"]),
        }

    if model == "GP2":
        return {
            "rho": float(row["physical_rho"]),
            "beta20": float(row["physical_beta20"]),
            "beta11": float(row["physical_beta11"]),
            "beta02": float(row["physical_beta02"]),
        }

    raise KeyError(model)


# =============================================================================
# Constitutive implementation
# =============================================================================

def invariants_from_principal(lam: np.ndarray):
    """
    lam: (..., 3) principal stretches, det(lam)=1.
    """
    lam = np.asarray(lam, dtype=np.float64)
    l2 = lam**2

    I1 = np.sum(l2, axis=-1)
    I2 = (
        l2[..., 0] * l2[..., 1]
        + l2[..., 1] * l2[..., 2]
        + l2[..., 2] * l2[..., 0]
    )

    return I1, I2


def ogden_shifted_term(alpha: float, log_lam: np.ndarray) -> np.ndarray:
    """
    Return 4*(lambda^alpha - 1)/alpha.

    The '-1' is a common principal-stress shift and therefore has no effect on
    pressure-independent stress differences or shear stress after rotation.
    It makes the alpha->0 limit numerically stable.
    """
    alpha = float(alpha)
    L = np.asarray(log_lam, dtype=np.float64)

    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            L
            + 0.5 * alpha * L**2
            + (alpha**2 / 6.0) * L**3
            + (alpha**3 / 24.0) * L**4
            + (alpha**4 / 120.0) * L**5
        )

    return 4.0 * np.expm1(alpha * L) / alpha


def principal_tau_shape(
    model: str,
    physical: dict,
    lam: np.ndarray,
) -> np.ndarray:
    """
    Unit-response-scale principal Kirchhoff stress contribution, modulo pressure.

    Output shape = lam.shape, i.e. (..., 3).
    Any additive isotropic term is irrelevant for the pressure-independent
    quantities used below.
    """
    model = str(model)
    lam = np.asarray(lam, dtype=np.float64)

    I1, I2 = invariants_from_principal(lam)
    x = I1 - 3.0
    y = I2 - 3.0

    if model == "NH":
        return 2.0 * lam**2

    if model == "MR":
        rho = float(physical["rho"])
        return 2.0 * (
            (1.0 - rho) * lam**2
            - rho * lam**-2.0
        )

    if model == "YEOH2":
        beta = float(physical["beta"])
        multiplier = 1.0 + 2.0 * beta * x
        return 2.0 * multiplier[..., None] * lam**2

    if model == "GENT":
        gamma = float(physical["gamma"])
        denom = 1.0 - gamma * x

        out = np.full_like(lam, np.nan, dtype=np.float64)
        valid = denom > GENT_SINGULARITY_MARGIN
        out[valid] = (
            2.0
            * lam[valid] ** 2
            / denom[valid, None]
        )
        return out

    if model == "OGDEN1":
        alpha = float(physical["alpha"])
        return ogden_shifted_term(alpha, np.log(lam))

    if model == "OGDEN2":
        a1 = float(physical["alpha1"])
        a2 = float(physical["alpha2"])
        w1 = float(physical["weight1"])

        f1 = ogden_shifted_term(a1, np.log(lam))
        f2 = ogden_shifted_term(a2, np.log(lam))
        return w1 * f1 + (1.0 - w1) * f2

    if model == "GP2":
        rho = float(physical["rho"])
        beta20 = float(physical["beta20"])
        beta11 = float(physical["beta11"])
        beta02 = float(physical["beta02"])

        W1 = (
            (1.0 - rho)
            + 2.0 * beta20 * x
            + beta11 * y
        )
        W2 = (
            rho
            + beta11 * x
            + 2.0 * beta02 * y
        )

        return 2.0 * (
            W1[..., None] * lam**2
            - W2[..., None] * lam**-2.0
        )

    raise KeyError(model)


def principal_differences(
    model: str,
    physical: dict,
    response_scale_kpa: float,
    lam: np.ndarray,
) -> np.ndarray:
    """
    Return two pressure-independent principal Kirchhoff stress differences:
        d13 = tau1 - tau3
        d23 = tau2 - tau3
    """
    q = principal_tau_shape(model, physical, lam)
    d13 = q[..., 0] - q[..., 2]
    d23 = q[..., 1] - q[..., 2]

    return float(response_scale_kpa) * np.stack(
        [d13, d23],
        axis=-1,
    )


# =============================================================================
# Standard protocol responses
# =============================================================================

def ut_principal_stretches(lam: np.ndarray) -> np.ndarray:
    lam = np.asarray(lam, dtype=np.float64)
    return np.stack(
        [
            lam,
            lam**-0.5,
            lam**-0.5,
        ],
        axis=-1,
    )


def bt_principal_stretches(lam: np.ndarray) -> np.ndarray:
    """
    Equibiaxial in-plane stretch:
        lambda1 = lambda2 = lambda
        lambda3 = lambda^-2
    """
    lam = np.asarray(lam, dtype=np.float64)
    return np.stack(
        [
            lam,
            lam,
            lam**-2.0,
        ],
        axis=-1,
    )


def simple_shear_response(
    model: str,
    physical: dict,
    response_scale_kpa: float,
    gamma: np.ndarray,
) -> np.ndarray:
    """
    Kirchhoff shear stress tau_12 for simple shear
        F = [[1, gamma, 0],
             [0, 1,     0],
             [0, 0,     1]]

    Pressure does not affect tau_12.

    The principal stretches and eigenvectors of B are evaluated directly at
    each gamma so the implementation works for invariant and Ogden families.
    """
    gamma = np.asarray(gamma, dtype=np.float64)
    out = np.zeros_like(gamma)

    for k, g in enumerate(gamma):
        F = np.array(
            [
                [1.0, g, 0.0],
                [0.0, 1.0, 0.0],
                [0.0, 0.0, 1.0],
            ],
            dtype=float,
        )

        B = F @ F.T
        eigvals, eigvecs = np.linalg.eigh(B)

        # Sort descending for a consistent principal order.
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        V = eigvecs[:, order]

        lam = np.sqrt(np.maximum(eigvals, 0.0))[None, :]
        q = principal_tau_shape(
            model,
            physical,
            lam,
        )[0]

        if not np.all(np.isfinite(q)):
            out[k] = np.nan
            continue

        tau = V @ np.diag(
            float(response_scale_kpa) * q
        ) @ V.T

        out[k] = tau[0, 1]

    return out


def standard_protocol_curves(
    model: str,
    physical: dict,
    response_scale_kpa: float,
):
    ut_lam = ut_principal_stretches(UT)
    bt_lam = bt_principal_stretches(BT)

    ut_diff = principal_differences(
        model,
        physical,
        response_scale_kpa,
        ut_lam,
    )[:, 0]  # axial vs lateral

    bt_diff = principal_differences(
        model,
        physical,
        response_scale_kpa,
        bt_lam,
    )[:, 0]  # in-plane vs out-of-plane

    sh_tau12 = simple_shear_response(
        model,
        physical,
        response_scale_kpa,
        SH,
    )

    return {
        "UT": ut_diff,
        "BT": bt_diff,
        "SH": sh_tau12,
    }


# =============================================================================
# Conventional-fit metric and scale profiling
# =============================================================================

def protocol_normalizers(truth_curves: dict) -> dict:
    norms = {}
    for name, y in truth_curves.items():
        peak = max(float(np.nanmax(np.abs(y))), EPS)
        norms[name] = peak
    return norms


def standard_protocol_nrmse_percent(
    truth_curves: dict,
    pred_curves: dict,
) -> float:
    """
    Equal-weight protocol-normalized RMS error.
    """
    norms = protocol_normalizers(truth_curves)

    residual_blocks = []
    for name in ["UT", "BT", "SH"]:
        t = np.asarray(truth_curves[name], dtype=float)
        p = np.asarray(pred_curves[name], dtype=float)

        finite = np.isfinite(t) & np.isfinite(p)
        if not np.all(finite):
            return np.inf

        residual_blocks.append(
            (p - t) / norms[name]
        )

    r = np.concatenate(residual_blocks)
    return 100.0 * float(np.sqrt(np.mean(r**2)))


def fit_scale_for_shape(
    truth_curves: dict,
    unit_curves: dict,
) -> float:
    """
    Weighted least-squares profile of one positive response scale.
    Each protocol is normalized by its truth peak so UT/BT/SH contribute
    comparably.
    """
    norms = protocol_normalizers(truth_curves)

    yy = []
    tt = []

    for name in ["UT", "BT", "SH"]:
        y = np.asarray(truth_curves[name], dtype=float)
        t = np.asarray(unit_curves[name], dtype=float)

        if not np.all(np.isfinite(t)):
            return np.nan

        yy.append(y / norms[name])
        tt.append(t / norms[name])

    y = np.concatenate(yy)
    t = np.concatenate(tt)

    denom = float(np.dot(t, t))
    if denom <= EPS:
        return np.nan

    scale = float(np.dot(y, t) / denom)
    return max(scale, EPS)


# =============================================================================
# Shape bounds from the frozen biological atlas
# =============================================================================

def scalar_shape_bounds(
    atlas_df: pd.DataFrame,
    model: str,
) -> tuple[float, float]:
    sub = atlas_df[
        atlas_df["source_model"].astype(str) == model
    ].copy()

    if not len(sub):
        raise RuntimeError(f"No atlas states for {model}")

    if model == "MR":
        values = sub["physical_rho"].to_numpy(float)
    elif model == "YEOH2":
        if "physical_beta" in sub.columns:
            values = sub["physical_beta"].to_numpy(float)
        else:
            values = np.array(
                [
                    parse_json_array(v)[0]
                    for v in sub["coordinate_json"]
                ],
                dtype=float,
            )
    elif model == "GENT":
        if "physical_gamma" in sub.columns:
            values = sub["physical_gamma"].to_numpy(float)
        else:
            values = np.array(
                [
                    parse_json_array(v)[0]
                    for v in sub["coordinate_json"]
                ],
                dtype=float,
            )
    elif model == "OGDEN1":
        if "physical_alpha" in sub.columns:
            values = sub["physical_alpha"].to_numpy(float)
        else:
            values = np.array(
                [
                    parse_json_array(v)[0]
                    for v in sub["coordinate_json"]
                ],
                dtype=float,
            )
    else:
        raise KeyError(model)

    values = values[np.isfinite(values)]
    if not len(values):
        raise RuntimeError(f"No finite shape bounds for {model}")

    lo = float(np.min(values))
    hi = float(np.max(values))

    if not hi > lo:
        hi = lo + 1.0e-8

    return lo, hi


def scalar_physical(model: str, x: float) -> dict:
    if model == "MR":
        return {"rho": float(x)}
    if model == "YEOH2":
        return {"beta": float(x)}
    if model == "GENT":
        return {"gamma": float(x)}
    if model == "OGDEN1":
        return {"alpha": float(x)}
    raise KeyError(model)


# =============================================================================
# Fit simpler model to UT + BT + SH only
# =============================================================================

def fit_simple_model_to_standard_protocols(
    model: str,
    truth_curves: dict,
    atlas_df: pd.DataFrame,
):
    if model == "NH":
        physical = {}
        unit = standard_protocol_curves(
            model,
            physical,
            1.0,
        )
        scale = fit_scale_for_shape(
            truth_curves,
            unit,
        )
        pred = standard_protocol_curves(
            model,
            physical,
            scale,
        )
        error = standard_protocol_nrmse_percent(
            truth_curves,
            pred,
        )

        return {
            "model": model,
            "shape_parameter": np.nan,
            "shape_parameter_name": "",
            "response_scale_kpa": scale,
            "standard_protocol_nrmse_percent": error,
            "physical": physical,
            "curves": pred,
        }

    lo, hi = scalar_shape_bounds(
        atlas_df,
        model,
    )

    def objective(x):
        physical = scalar_physical(
            model,
            float(x),
        )
        unit = standard_protocol_curves(
            model,
            physical,
            1.0,
        )
        scale = fit_scale_for_shape(
            truth_curves,
            unit,
        )

        if not np.isfinite(scale):
            return 1.0e9

        pred = standard_protocol_curves(
            model,
            physical,
            scale,
        )

        err = standard_protocol_nrmse_percent(
            truth_curves,
            pred,
        )

        return (
            err
            if np.isfinite(err)
            else 1.0e9
        )

    result = minimize_scalar(
        objective,
        bounds=(lo, hi),
        method="bounded",
        options={
            "xatol": 1.0e-10,
            "maxiter": 320,
        },
    )

    x = float(result.x)
    physical = scalar_physical(model, x)

    unit = standard_protocol_curves(
        model,
        physical,
        1.0,
    )
    scale = fit_scale_for_shape(
        truth_curves,
        unit,
    )
    pred = standard_protocol_curves(
        model,
        physical,
        scale,
    )
    error = standard_protocol_nrmse_percent(
        truth_curves,
        pred,
    )

    name = {
        "MR": "rho",
        "YEOH2": "beta",
        "GENT": "gamma",
        "OGDEN1": "alpha",
    }[model]

    return {
        "model": model,
        "shape_parameter": x,
        "shape_parameter_name": name,
        "response_scale_kpa": scale,
        "standard_protocol_nrmse_percent": error,
        "physical": physical,
        "curves": pred,
    }


def fit_all_simple_models(
    source_row: pd.Series,
    atlas_df: pd.DataFrame,
):
    source_model = str(source_row["source_model"])
    source_physical = source_physical_parameters(
        source_row,
    )
    source_scale = float(
        source_row["source_response_scale_kpa"]
    )

    truth = standard_protocol_curves(
        source_model,
        source_physical,
        source_scale,
    )

    fits = []

    for model in SIMPLE_MODELS:
        fit = fit_simple_model_to_standard_protocols(
            model,
            truth,
            atlas_df,
        )
        fit["complete_F_required_noise_percent"] = (
            certified_required_noise_percent(
                source_row,
                model,
            )
        )
        fits.append(fit)

    fits.sort(
        key=lambda f: (
            f["standard_protocol_nrmse_percent"],
            f["complete_F_required_noise_percent"],
        )
    )

    return truth, fits


# =============================================================================
# Source search
# =============================================================================

def candidate_sources(
    atlas_df: pd.DataFrame,
) -> pd.DataFrame:
    d = atlas_df.copy()

    # Unique complete-F regions only.
    keep = []
    for _, row in d.iterrows():
        model = str(row["source_model"])

        if model not in COMPLEX_SOURCE_MODELS:
            keep.append(False)
            continue

        if as_bool(row["certified_label_changed"]):
            keep.append(False)
            continue

        if certified_models(row) != [model]:
            keep.append(False)
            continue

        if str(row["biological_support_zone"]) not in {
            "INSIDE_LITERATURE_CORE",
            "INSIDE_BIOLOGICAL_MARGIN",
        }:
            keep.append(False)
            continue

        keep.append(True)

    d = d[np.asarray(keep, dtype=bool)].copy()

    # Prefer full literature-core states when available.
    core = d[
        (d["shape_support_zone"].astype(str) == "INSIDE_LITERATURE_CORE")
        & (d["scale_support_zone"].astype(str) == "INSIDE_LITERATURE_CORE")
    ].copy()

    if len(core) >= 20:
        d = core

    groups = []

    for model in COMPLEX_SOURCE_MODELS:
        sub = d[
            d["source_model"].astype(str) == model
        ].copy()

        if not len(sub):
            continue

        n = min(
            MAX_SOURCE_CASES_PER_COMPLEX_MODEL,
            len(sub),
        )

        idx = np.linspace(
            0,
            len(sub) - 1,
            n,
        ).round().astype(int)

        groups.append(
            sub.iloc[np.unique(idx)]
        )

    if not groups:
        raise RuntimeError(
            "No biologically supported unique OGDEN2/GP2 states were found."
        )

    return pd.concat(
        groups,
        axis=0,
    ).reset_index(drop=True)


def evaluate_source_for_example(
    source_row: pd.Series,
    atlas_df: pd.DataFrame,
):
    truth_curves, fits = fit_all_simple_models(
        source_row,
        atlas_df,
    )

    # Best conventional fit among strictly simpler model families.
    best = fits[0]

    conv_err = float(
        best["standard_protocol_nrmse_percent"]
    )
    complete_noise = float(
        best["complete_F_required_noise_percent"]
    )

    if conv_err > MAX_STANDARD_FIT_NRMSE_PERCENT:
        return None

    if complete_noise < MIN_COMPLETE_F_REQUIRED_NOISE_PERCENT:
        return None

    # Favor:
    #   - very good conventional fit,
    #   - clearly >3% complete-F rejection,
    #   - moderate/clear rejection rather than the most pathological endpoint.
    noise_moderation = math.exp(
        -0.5
        * (
            math.log(
                max(complete_noise, EPS)
                / TARGET_COMPLETE_F_NOISE_PERCENT
            )
            / 0.9
        )
        ** 2
    )

    score = (
        (complete_noise - 3.0)
        * noise_moderation
        / (0.25 + conv_err)
    )

    return {
        "source_model": str(source_row["source_model"]),
        "source_index": int(source_row["source_index"]),
        "source_mu0_kpa": float(source_row["source_mu0_kpa"]),
        "best_simple_model": str(best["model"]),
        "standard_protocol_nrmse_percent": conv_err,
        "best_simple_complete_F_required_noise_percent":
            complete_noise,
        "selection_score": float(score),
        "_truth_curves": truth_curves,
        "_fits": fits,
        "_best_fit": best,
    }


# =============================================================================
# Dense complete-F domain
# =============================================================================

def complete_F_grid():
    """
    Unique intrinsic incompressible principal-stretch domain modulo permutation.

    e_i = log(lambda_i)
    e1 + e2 + e3 = 0
    lambda_i in [0.5, 2]
    ordering: e1 >= e2 >= e3

    Returns a dense triangular-like 2-D domain in (e1,e2).
    """
    emin = math.log(LAMBDA_MIN)
    emax = math.log(LAMBDA_MAX)

    e1_values = np.linspace(
        emin,
        emax,
        N_LOG_GRID,
    )
    e2_values = np.linspace(
        emin,
        emax,
        N_LOG_GRID,
    )

    rows = []

    for e1 in e1_values:
        for e2 in e2_values:
            e3 = -e1 - e2

            if not (
                emin - 1.0e-12
                <= e3
                <= emax + 1.0e-12
            ):
                continue

            # Unique permutation wedge.
            if not (
                e1 >= e2 - 1.0e-12
                and e2 >= e3 - 1.0e-12
            ):
                continue

            lam = np.exp(
                np.array([e1, e2, e3])
            )

            rows.append(
                (
                    e1,
                    e2,
                    e3,
                    lam[0],
                    lam[1],
                    lam[2],
                )
            )

    grid = pd.DataFrame(
        rows,
        columns=[
            "e1",
            "e2",
            "e3",
            "lambda1",
            "lambda2",
            "lambda3",
        ],
    )

    if not len(grid):
        raise RuntimeError(
            "Complete-F grid is empty."
        )

    return grid


def complete_F_error_analysis(
    source_row: pd.Series,
    best_fit: dict,
):
    grid = complete_F_grid()

    lam = grid[
        ["lambda1", "lambda2", "lambda3"]
    ].to_numpy(float)

    source_model = str(source_row["source_model"])
    source_phys = source_physical_parameters(
        source_row,
    )
    source_scale = float(
        source_row["source_response_scale_kpa"]
    )

    truth = principal_differences(
        source_model,
        source_phys,
        source_scale,
        lam,
    )

    simple = principal_differences(
        str(best_fit["model"]),
        best_fit["physical"],
        float(best_fit["response_scale_kpa"]),
        lam,
    )

    finite = (
        np.all(np.isfinite(truth), axis=1)
        & np.all(np.isfinite(simple), axis=1)
    )

    # Relative 2-component error, with a small global floor near undeformed states.
    truth_norm = np.linalg.norm(
        truth,
        axis=1,
    )
    diff_norm = np.linalg.norm(
        simple - truth,
        axis=1,
    )

    floor = max(
        0.02 * float(np.max(truth_norm[finite])),
        EPS,
    )

    relative_error_percent = (
        100.0
        * diff_norm
        / np.maximum(truth_norm, floor)
    )

    relative_error_percent[~finite] = np.nan

    # A second diagnostic normalized directly by a 3% reference scale.
    three_percent_units = (
        relative_error_percent / 3.0
    )

    valid_idx = np.flatnonzero(
        np.isfinite(relative_error_percent)
    )

    if not len(valid_idx):
        raise RuntimeError(
            "No finite complete-F comparison states."
        )

    fstar_pos = int(
        valid_idx[
            np.argmax(
                relative_error_percent[valid_idx]
            )
        ]
    )

    grid["truth_d13_kpa"] = truth[:, 0]
    grid["truth_d23_kpa"] = truth[:, 1]
    grid["simple_d13_kpa"] = simple[:, 0]
    grid["simple_d23_kpa"] = simple[:, 1]
    grid["relative_error_percent"] = (
        relative_error_percent
    )
    grid["three_percent_reference_units"] = (
        three_percent_units
    )
    grid["is_Fstar"] = 0
    grid.loc[fstar_pos, "is_Fstar"] = 1

    Fstar = {
        "grid_row": int(fstar_pos),
        "e1": float(grid.iloc[fstar_pos]["e1"]),
        "e2": float(grid.iloc[fstar_pos]["e2"]),
        "e3": float(grid.iloc[fstar_pos]["e3"]),
        "lambda1": float(
            grid.iloc[fstar_pos]["lambda1"]
        ),
        "lambda2": float(
            grid.iloc[fstar_pos]["lambda2"]
        ),
        "lambda3": float(
            grid.iloc[fstar_pos]["lambda3"]
        ),
        "relative_error_percent": float(
            grid.iloc[fstar_pos][
                "relative_error_percent"
            ]
        ),
        "three_percent_reference_units": float(
            grid.iloc[fstar_pos][
                "three_percent_reference_units"
            ]
        ),
        "truth_d13_kpa": float(
            grid.iloc[fstar_pos]["truth_d13_kpa"]
        ),
        "truth_d23_kpa": float(
            grid.iloc[fstar_pos]["truth_d23_kpa"]
        ),
        "simple_d13_kpa": float(
            grid.iloc[fstar_pos]["simple_d13_kpa"]
        ),
        "simple_d23_kpa": float(
            grid.iloc[fstar_pos]["simple_d23_kpa"]
        ),
    }

    return grid, Fstar


# =============================================================================
# Figure
# =============================================================================

def make_figure(
    source_row: pd.Series,
    result: dict,
    error_grid: pd.DataFrame,
    Fstar: dict,
):
    source_model = result["source_model"]
    simple_model = result["best_simple_model"]

    truth = result["_truth_curves"]
    fit = result["_best_fit"]
    pred = fit["curves"]

    fig = plt.figure(
        figsize=(15.4, 11.0)
    )

    # ------------------------------------------------------------------
    # A — Conventional protocols
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 1)

    ax.plot(
        UT,
        truth["UT"],
        linewidth=2.6,
        label=f"{source_model} truth — UT",
    )
    ax.plot(
        UT,
        pred["UT"],
        linestyle="--",
        linewidth=2.2,
        label=f"{simple_model} fit — UT",
    )

    # Normalize BT and SH x axes visually into a separated offset representation
    # while keeping their physical values on the legend/labels below.
    # For clarity, use three mini curve groups on one panel with normalized x.
    # UT occupies 0..1, BT 1.25..2.25, SH 2.5..3.5.
    xu = np.linspace(0.0, 1.0, len(UT))
    xb = np.linspace(1.25, 2.25, len(BT))
    xs = np.linspace(2.50, 3.50, len(SH))

    ax.clear()

    ax.plot(
        xu,
        truth["UT"],
        linewidth=2.6,
        label=f"{source_model} truth",
    )
    ax.plot(
        xu,
        pred["UT"],
        linestyle="--",
        linewidth=2.2,
        label=f"{simple_model} simpler fit",
    )

    ax.plot(
        xb,
        truth["BT"],
        linewidth=2.6,
    )
    ax.plot(
        xb,
        pred["BT"],
        linestyle="--",
        linewidth=2.2,
    )

    ax.plot(
        xs,
        truth["SH"],
        linewidth=2.6,
    )
    ax.plot(
        xs,
        pred["SH"],
        linestyle="--",
        linewidth=2.2,
    )

    ax.axvline(1.125, linewidth=0.8, alpha=0.35)
    ax.axvline(2.375, linewidth=0.8, alpha=0.35)

    ax.set_xticks(
        [0.5, 1.75, 3.0]
    )
    ax.set_xticklabels(
        [
            "UT\nλ=0.70–1.50",
            "BT\nλ=0.80–1.25",
            "SH\nγ=−0.50–0.50",
        ]
    )

    ax.set_ylabel(
        "Pressure-independent stress response (kPa)"
    )
    ax.set_title(
        "A. Standard protocols make the simpler model look adequate\n"
        f"{simple_model} pooled protocol NRMSE = "
        f"{result['standard_protocol_nrmse_percent']:.2f}%"
    )
    ax.legend(fontsize=8, loc="best")

    # ------------------------------------------------------------------
    # B — Certified complete-F model adequacy
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 2)

    required = np.array(
        [
            certified_required_noise_percent(
                source_row,
                m,
            )
            for m in MODELS
        ],
        dtype=float,
    )

    x = np.arange(len(MODELS))
    ax.bar(x, required)
    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.8,
        label="3% atlas threshold",
    )

    source_idx = MODELS.index(
        source_model
    )
    simple_idx = MODELS.index(
        simple_model
    )

    ax.scatter(
        [source_idx],
        [required[source_idx]],
        s=120,
        marker="*",
        edgecolor="black",
        linewidth=0.7,
        zorder=5,
        label="Hidden source family",
    )

    ax.scatter(
        [simple_idx],
        [required[simple_idx]],
        s=90,
        marker="D",
        edgecolor="black",
        linewidth=0.7,
        zorder=5,
        label="Best simpler conventional fit",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        MODELS,
        rotation=35,
    )
    ax.set_ylabel(
        "Certified required noise (%)"
    )
    ax.set_title(
        "B. Complete-F certification rejects the simpler explanation\n"
        f"{simple_model} requires "
        f"{result['best_simple_complete_F_required_noise_percent']:.2f}% "
        "noise over the full atlas"
    )
    ax.legend(fontsize=8, loc="best")

    # ------------------------------------------------------------------
    # C — Complete-F error map
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 3)

    sc = ax.scatter(
        error_grid["e1"],
        error_grid["e2"],
        c=error_grid["relative_error_percent"],
        s=13,
    )

    cb = fig.colorbar(
        sc,
        ax=ax,
        fraction=0.047,
        pad=0.03,
    )
    cb.set_label(
        f"{simple_model} vs {source_model} relative error (%)"
    )

    ax.scatter(
        [Fstar["e1"]],
        [Fstar["e2"]],
        s=150,
        marker="*",
        edgecolor="black",
        linewidth=0.9,
        label="Maximum-discrimination state F*",
        zorder=7,
    )

    ax.set_xlabel(
        "Principal log strain e₁ = log λ₁"
    )
    ax.set_ylabel(
        "Principal log strain e₂ = log λ₂"
    )
    ax.set_title(
        "C. Complete-F domain reveals where the simpler model fails\n"
        f"max error = {Fstar['relative_error_percent']:.1f}% "
        f"at λ=({Fstar['lambda1']:.3f}, "
        f"{Fstar['lambda2']:.3f}, "
        f"{Fstar['lambda3']:.3f})"
    )
    ax.legend(fontsize=8, loc="best")

    # ------------------------------------------------------------------
    # D — F* component comparison
    # ------------------------------------------------------------------
    ax = fig.add_subplot(2, 2, 4)

    labels = [
        r"$\tau_1-\tau_3$",
        r"$\tau_2-\tau_3$",
    ]

    truth_vals = np.array(
        [
            Fstar["truth_d13_kpa"],
            Fstar["truth_d23_kpa"],
        ]
    )

    simple_vals = np.array(
        [
            Fstar["simple_d13_kpa"],
            Fstar["simple_d23_kpa"],
        ]
    )

    xx = np.arange(2)
    width = 0.36

    ax.bar(
        xx - width / 2,
        truth_vals,
        width,
        label=f"{source_model} hidden truth",
    )
    ax.bar(
        xx + width / 2,
        simple_vals,
        width,
        label=f"{simple_model} simpler fit",
    )

    ax.set_xticks(xx)
    ax.set_xticklabels(labels)
    ax.set_ylabel(
        "Principal Kirchhoff stress difference (kPa)"
    )
    ax.set_title(
        "D. A specific in-atlas multiaxial state exposes the missing complexity\n"
        f"F*: λ₁={Fstar['lambda1']:.3f}, "
        f"λ₂={Fstar['lambda2']:.3f}, "
        f"λ₃={Fstar['lambda3']:.3f}"
    )
    ax.legend(fontsize=8, loc="best")

    fig.suptitle(
        "Example 2 — Complete-F characterization shows when extra constitutive complexity is necessary",
        fontsize=15.2,
    )

    fig.tight_layout(
        rect=[0, 0.01, 1, 0.965]
    )

    png = (
        OUTDIR
        / "example2_complexity_required_figure.png"
    )
    pdf = (
        OUTDIR
        / "example2_complexity_required_figure.pdf"
    )

    fig.savefig(
        png,
        dpi=320,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    return png, pdf


# =============================================================================
# Main
# =============================================================================

def main():
    global ROOT, CELL2_DIR, OUTDIR

    ensure_drive()

    ROOT, atlas_path = resolve_certified_atlas_path()
    CELL2_DIR = atlas_path.parent
    OUTDIR = ROOT / "cell3_example2_complexity_required_v45_standalone"
    OUTDIR.mkdir(parents=True, exist_ok=True)

    print(f"[Example 2] Certified atlas path: {atlas_path}")
    print(f"[Example 2] Output directory: {OUTDIR}")

    atlas_df = pd.read_csv(
        atlas_path
    )

    print(
        f"[Example 2] Loaded "
        f"{len(atlas_df)} certified source states."
    )
    print(
        "[Example 2] Standalone mode: "
        "no V68 Python file imported."
    )
    print(
        f"[Example 2] Standard protocols: "
        f"UT={len(UT)}, BT={len(BT)}, SH={len(SH)} states."
    )

    pool = candidate_sources(
        atlas_df
    )

    print(
        f"[Example 2] Searching "
        f"{len(pool)} unique OGDEN2/GP2 source states..."
    )

    public_rows = []
    private_results = []

    for i, (_, row) in enumerate(
        pool.iterrows(),
        start=1,
    ):
        if (
            i == 1
            or i % 20 == 0
            or i == len(pool)
        ):
            print(
                f"[Example 2] "
                f"Evaluating {i}/{len(pool)}..."
            )

        result = evaluate_source_for_example(
            row,
            atlas_df,
        )

        if result is None:
            continue

        private_results.append(
            (
                row,
                result,
            )
        )

        public = {
            k: v
            for k, v in result.items()
            if not k.startswith("_")
        }
        public_rows.append(public)

        pd.DataFrame(
            public_rows
        ).sort_values(
            "selection_score",
            ascending=False,
        ).to_csv(
            OUTDIR
            / "example2_search_results.csv",
            index=False,
        )

        print(
            f"[Example 2]   candidate: "
            f"{result['source_model']} "
            f"#{result['source_index']} | "
            f"simple={result['best_simple_model']} | "
            f"standard={result['standard_protocol_nrmse_percent']:.2f}% | "
            f"complete-F required noise="
            f"{result['best_simple_complete_F_required_noise_percent']:.2f}%"
        )

    if not private_results:
        raise RuntimeError(
            "No Example-2 case satisfied the current criteria. "
            "The script did not fabricate a result. "
            "Consider increasing MAX_STANDARD_FIT_NRMSE_PERCENT "
            "or lowering MIN_COMPLETE_F_REQUIRED_NOISE_PERCENT."
        )

    source_row, best = max(
        private_results,
        key=lambda pair: pair[1][
            "selection_score"
        ],
    )

    print(
        f"[Example 2] Selected "
        f"{best['source_model']} "
        f"#{best['source_index']} "
        f"with simpler {best['best_simple_model']}."
    )

    # ------------------------------------------------------------------
    # Dense complete-F map
    # ------------------------------------------------------------------
    error_grid, Fstar = (
        complete_F_error_analysis(
            source_row,
            best["_best_fit"],
        )
    )

    # ------------------------------------------------------------------
    # Exports
    # ------------------------------------------------------------------
    public_best = {
        k: v
        for k, v in best.items()
        if not k.startswith("_")
    }

    with (
        OUTDIR
        / "example2_selected_source.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            public_best,
            f,
            indent=2,
        )

    fit_rows = []

    for fit in best["_fits"]:
        fit_rows.append(
            {
                "model": fit["model"],
                "shape_parameter_name":
                    fit["shape_parameter_name"],
                "shape_parameter":
                    fit["shape_parameter"],
                "response_scale_kpa":
                    fit["response_scale_kpa"],
                "standard_protocol_nrmse_percent":
                    fit["standard_protocol_nrmse_percent"],
                "complete_F_required_noise_percent":
                    fit["complete_F_required_noise_percent"],
                "is_best_simple_fit":
                    fit["model"]
                    == best["best_simple_model"],
            }
        )

    pd.DataFrame(
        fit_rows
    ).to_csv(
        OUTDIR
        / "example2_standard_protocol_fit_summary.csv",
        index=False,
    )

    error_grid.to_csv(
        OUTDIR
        / "example2_complete_F_error_map.csv",
        index=False,
    )

    with (
        OUTDIR
        / "example2_Fstar.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            Fstar,
            f,
            indent=2,
        )

    summary = {
        "scientific_message": (
            "A simpler constitutive family can fit the conventional UT, BT, "
            "and SH protocols very well while still being rejected by the "
            "complete-F atlas. The atlas therefore identifies cases where "
            "additional constitutive complexity is genuinely required and "
            "locates the multiaxial deformation state that exposes the "
            "difference."
        ),
        "source_model": best["source_model"],
        "source_index": best["source_index"],
        "certified_complete_F_set":
            str(
                source_row[
                    "certified_compatibility_set_at_3pct"
                ]
            ),
        "best_simple_model":
            best["best_simple_model"],
        "standard_protocol_nrmse_percent":
            best[
                "standard_protocol_nrmse_percent"
            ],
        "best_simple_complete_F_required_noise_percent":
            best[
                "best_simple_complete_F_required_noise_percent"
            ],
        "standard_protocol_definition": {
            "states_per_mode": N_STANDARD_PER_MODE,
            "UT_lambda_range": [float(UT[0]), float(UT[-1])],
            "BT_lambda_range": [float(BT[0]), float(BT[-1])],
            "SH_gamma_range": [float(SH[0]), float(SH[-1])],
        },
        "Fstar": Fstar,
    }

    with (
        OUTDIR
        / "example2_summary.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
        )

    png, pdf = make_figure(
        source_row,
        best,
        error_grid,
        Fstar,
    )

    print(
        "\n"
        + "=" * 116
    )
    print(
        "[Example 2] COMPLETE — "
        "COMPLEXITY REQUIRED"
    )
    print(
        f"[Example 2] Hidden source: "
        f"{best['source_model']} "
        f"#{best['source_index']}"
    )
    print(
        f"[Example 2] Certified complete-F set: "
        f"{source_row['certified_compatibility_set_at_3pct']}"
    )
    print(
        f"[Example 2] Best simpler standard-protocol fit: "
        f"{best['best_simple_model']}"
    )
    print(
        f"[Example 2] UT+BT+SH pooled NRMSE: "
        f"{best['standard_protocol_nrmse_percent']:.3f}%"
    )
    print(
        f"[Example 2] Same simpler model complete-F "
        f"required noise: "
        f"{best['best_simple_complete_F_required_noise_percent']:.3f}%"
    )
    print(
        f"[Example 2] F*: "
        f"lambda=("
        f"{Fstar['lambda1']:.4f}, "
        f"{Fstar['lambda2']:.4f}, "
        f"{Fstar['lambda3']:.4f})"
    )
    print(
        f"[Example 2] F* relative error: "
        f"{Fstar['relative_error_percent']:.2f}%"
    )
    print(
        f"[Example 2] Figure PNG: {png}"
    )
    print(
        f"[Example 2] Figure PDF: {pdf}"
    )
    print(
        f"[Example 2] Results: {OUTDIR}"
    )
    print(
        "=" * 116
    )


if __name__ == "__main__":
    main()


[Example 2] /content/drive is not exposing the real MyDrive. Mounting Google Drive at the clean fallback /content/v68_drive ...
[Example 2] When authorization appears, choose the Google account that contains Optimal_Protocol/V68_practical_standard_protocols_...
Mounted at /content/v68_drive
[Example 2] Active MyDrive root: /content/v68_drive/MyDrive
[Example 2] Certified atlas path: /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/cell2_publication_validation/certified_source_atlas_states.csv
[Example 2] Output directory: /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/cell3_example2_complexity_required_v45_standalone
[Example 2] Loaded 2631 certified source states.
[Example 2] Standalone mode: no V68 Python file imported.
[Example 2] Standard protocols: UT=81, BT=81, SH=81 states.
[Example 2] Searching 240 unique OGDEN2/GP2 source states...
[Exam

In [ ]:
#!/usr/bin/env python3
"""
V68.5.3 CELL 3 — STANDALONE EXAMPLE 3
ROBUST NONE-COMPATIBLE / MODEL-LIBRARY INADEQUACY EXAMPLE

Core scientific message
-----------------------
Optimization always returns a "best" model.  A compatibility atlas must also be
able to say that the best model is still not adequate.

This example uses a withheld, incompressible isotropic exponential
(Demiray-type) constitutive response that is NOT one of the seven atlas families.
The hidden response is deliberately moderate and smooth:

    W ~ exp[b (I1 - 3)],       b = 1.0

The response scale is arbitrary within the atlas biological scale range because
all candidate models profile/optimize their own positive scale.

The script then performs two different analyses:

1) Conventional characterization
   Fit all seven candidate families using only standard:
       - uniaxial tension/compression (UT)
       - equibiaxial loading (BT)
       - simple shear (SH)

   A candidate model can fit these conventional protocols extremely well.

2) Complete-F absolute adequacy
   Continuously optimize EVERY candidate family over the exact frozen 2601-state
   V68 complete-F parent domain using the same V68-style symmetric
   required-noise metric:

       max |source - target| / (brush(source) + brush(target))

   with the same per-state floor factor 1/6.

   The calculation is repeated independently on a denser intrinsic validation
   grid.  A hard NONE_COMPATIBLE verdict is issued only when the best continuously
   optimized family remains safely above 5% mismatch on BOTH domains.

No V68 / V68.1 / Cell-1 / Cell-2 Python file is imported.

Required frozen data
--------------------
<ROOT>/complete_F_parent_protocol_states.csv

Outputs
-------
<ROOT>/cell3_example3_none_compatible_v53_continuous_standalone/
    example3_standard_fit_summary.csv
    example3_completeF_continuous_parent.csv
    example3_completeF_continuous_validation.csv
    example3_validation_error_map.csv
    example3_Fstar.json
    example3_summary.json
    example3_none_compatible_figure.png
    example3_none_compatible_figure.pdf
"""

from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import differential_evolution, minimize_scalar


# =============================================================================
# Configuration
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/Optimal_Protocol/"
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)

OUTDIR = ROOT / "cell3_example3_none_compatible_v53_continuous_standalone"
OUTDIR.mkdir(parents=True, exist_ok=True)

PARENT_STATES_CSV = ROOT / "complete_F_parent_protocol_states.csv"

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]

# Exact standard-protocol ranges/resolution used by V68.
UT = np.linspace(0.70, 1.50, 81)
BT = np.linspace(0.80, 1.25, 81)
SH = np.linspace(-0.50, 0.50, 81)

# Withheld constitutive truth.
WITHHELD_MODEL_NAME = "EXPONENTIAL_I1"
WITHHELD_B = 1.0
WITHHELD_RESPONSE_SCALE_KPA = 50.0  # corresponds to a moderate biological scale

# Atlas adequacy thresholds.
ATLAS_NOISE_THRESHOLD_FRACTION = 0.03
ROBUST_NONE_MARGIN_FRACTION = 0.05

# Exact V68 parent brush-floor convention.
MODE_FLOOR_PER_UNIT_NOISE = 1.0 / 6.0

# Candidate parameter domains used by the V68 biological atlas.
# These are physical coordinates; scale is optimized separately.
PHYSICAL_BOUNDS = {
    "NH": [],
    "MR": [(0.0, 1.0)],                                  # rho
    "YEOH2": [(0.0, 25.0)],                              # beta
    "GENT": [(0.0, 0.42)],                               # gamma
    "OGDEN1": [(-30.0, 30.0)],                           # alpha
    "OGDEN2": [(-30.0, 30.0), (-30.0, 30.0), (1.0e-4, 0.9999)],
    "GP2": [(0.0, 1.0), (0.0, 25.0), (0.0, 15.0), (0.0, 25.0)],
}

# V68 universal scale domain:
# mu0 = 0.05 .. 20000 kPa and response_scale = 0.5*mu0.
RESPONSE_SCALE_BOUNDS_KPA = (0.025, 10000.0)

# Independent dense intrinsic grid for convergence / robustness.
VALIDATION_GRID_N = 81

# Interior robustness region for F*.
INTERIOR_LAMBDA_MIN = 0.55
INTERIOR_LAMBDA_MAX = 1.90

# Exclude near-identity states from the manuscript-highlighted interior F*.
# A state must satisfy max_i |log(lambda_i)| >= this value.
MIN_INTERIOR_LOG_STRAIN_MAGNITUDE = 0.20

# Optimizer settings.
DE_SEEDS = (17, 41)
DE_MAXITER_SCALAR = 100
DE_MAXITER_MULTI = 180
DE_POPSIZE_SCALAR = 10
DE_POPSIZE_MULTI = 14
DE_TOL = 1.0e-9

EPS = 1.0e-14
GENT_SINGULARITY_MARGIN = 1.0e-6
OGDEN_ALPHA_SERIES_THRESHOLD = 1.0e-5


# =============================================================================
# Drive
# =============================================================================

def ensure_drive() -> None:
    if Path("/content/drive/MyDrive").exists():
        print("[Example 3] Google Drive already mounted.")
        return

    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "Google Drive is not mounted and google.colab is unavailable."
        ) from exc

    drive.mount("/content/drive", force_remount=False)


# =============================================================================
# Constitutive implementation
# =============================================================================

def invariants_from_principal(lam: np.ndarray):
    lam = np.asarray(lam, dtype=np.float64)
    l2 = lam**2

    I1 = np.sum(l2, axis=-1)
    I2 = (
        l2[..., 0] * l2[..., 1]
        + l2[..., 1] * l2[..., 2]
        + l2[..., 2] * l2[..., 0]
    )

    return I1, I2


def ogden_shifted_term(alpha: float, log_lam: np.ndarray) -> np.ndarray:
    """
    Pressure-independent Ogden principal contribution.

    4*(lambda^alpha - 1)/alpha

    The additive '-1' is an isotropic principal-stress shift and therefore
    cancels in the pressure-independent principal stress differences.
    """
    alpha = float(alpha)
    L = np.asarray(log_lam, dtype=np.float64)

    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            L
            + 0.5 * alpha * L**2
            + (alpha**2 / 6.0) * L**3
            + (alpha**3 / 24.0) * L**4
            + (alpha**4 / 120.0) * L**5
        )

    return 4.0 * np.expm1(alpha * L) / alpha


def physical_from_x(model: str, x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float).reshape(-1)

    if model == "NH":
        return {}
    if model == "MR":
        return {"rho": float(x[0])}
    if model == "YEOH2":
        return {"beta": float(x[0])}
    if model == "GENT":
        return {"gamma": float(x[0])}
    if model == "OGDEN1":
        return {"alpha": float(x[0])}
    if model == "OGDEN2":
        return {
            "alpha1": float(x[0]),
            "alpha2": float(x[1]),
            "weight1": float(x[2]),
        }
    if model == "GP2":
        return {
            "rho": float(x[0]),
            "beta20": float(x[1]),
            "beta11": float(x[2]),
            "beta02": float(x[3]),
        }

    raise KeyError(model)


def principal_tau_shape(
    model: str,
    physical: dict,
    lam: np.ndarray,
) -> np.ndarray:
    """
    Unit-response-scale principal Kirchhoff stress contribution, modulo pressure.

    Output shape: (..., 3)
    """
    model = str(model)
    lam = np.asarray(lam, dtype=np.float64)

    I1, I2 = invariants_from_principal(lam)
    x = I1 - 3.0
    y = I2 - 3.0

    if model == "NH":
        return 2.0 * lam**2

    if model == "MR":
        rho = float(physical["rho"])
        return 2.0 * (
            (1.0 - rho) * lam**2
            - rho * lam**-2.0
        )

    if model == "YEOH2":
        beta = float(physical["beta"])
        multiplier = 1.0 + 2.0 * beta * x
        return 2.0 * multiplier[..., None] * lam**2

    if model == "GENT":
        gamma = float(physical["gamma"])
        denom = 1.0 - gamma * x

        out = np.full_like(lam, np.nan, dtype=np.float64)
        valid = denom > GENT_SINGULARITY_MARGIN

        out[valid] = (
            2.0
            * lam[valid]**2
            / denom[valid, None]
        )
        return out

    if model == "OGDEN1":
        alpha = float(physical["alpha"])
        return ogden_shifted_term(
            alpha,
            np.log(lam),
        )

    if model == "OGDEN2":
        a1 = float(physical["alpha1"])
        a2 = float(physical["alpha2"])
        w1 = float(physical["weight1"])

        f1 = ogden_shifted_term(a1, np.log(lam))
        f2 = ogden_shifted_term(a2, np.log(lam))

        return w1 * f1 + (1.0 - w1) * f2

    if model == "GP2":
        rho = float(physical["rho"])
        beta20 = float(physical["beta20"])
        beta11 = float(physical["beta11"])
        beta02 = float(physical["beta02"])

        W1 = (
            (1.0 - rho)
            + 2.0 * beta20 * x
            + beta11 * y
        )

        W2 = (
            rho
            + beta11 * x
            + 2.0 * beta02 * y
        )

        return 2.0 * (
            W1[..., None] * lam**2
            - W2[..., None] * lam**-2.0
        )

    raise KeyError(model)


def withheld_principal_tau_shape(
    lam: np.ndarray,
    b: float = WITHHELD_B,
) -> np.ndarray:
    """
    Withheld incompressible exponential I1 law.

    The unit-scale principal contribution is

        tau_i ~ 2 exp[b (I1 - 3)] lambda_i^2

    so b -> 0 continuously approaches the NH unit template.
    """
    lam = np.asarray(lam, dtype=np.float64)
    I1, _ = invariants_from_principal(lam)
    x = I1 - 3.0

    return (
        2.0
        * np.exp(float(b) * x)[..., None]
        * lam**2
    )


def principal_differences_from_tau(
    tau: np.ndarray,
) -> np.ndarray:
    tau = np.asarray(tau, dtype=np.float64)

    return np.stack(
        [
            tau[..., 0] - tau[..., 2],
            tau[..., 1] - tau[..., 2],
        ],
        axis=-1,
    )


def model_parent_response(
    model: str,
    physical: dict,
    scale_kpa: float,
    lam: np.ndarray,
) -> np.ndarray:
    q = principal_tau_shape(
        model,
        physical,
        lam,
    )

    return (
        float(scale_kpa)
        * principal_differences_from_tau(q)
    )


def withheld_parent_response(
    lam: np.ndarray,
) -> np.ndarray:
    q = withheld_principal_tau_shape(
        lam,
        WITHHELD_B,
    )

    return (
        WITHHELD_RESPONSE_SCALE_KPA
        * principal_differences_from_tau(q)
    )


# =============================================================================
# Exact V68-style complete-F required-noise metric
# =============================================================================

def parent_brush_shape(center: np.ndarray) -> np.ndarray:
    """
    Exact parent-state part of the V68 brush:
        h = |response| + (1/6)*state_peak
    """
    center = np.asarray(
        center,
        dtype=np.float64,
    ).reshape(-1, 2)

    peak = np.maximum(
        np.max(np.abs(center), axis=1),
        EPS,
    )

    return (
        np.abs(center)
        + MODE_FLOOR_PER_UNIT_NOISE
        * peak[:, None]
    )


def required_noise_fraction(
    source: np.ndarray,
    target: np.ndarray,
) -> float:
    """
    Symmetric V68 required-noise fraction over the complete-F parent.

        max |source-target| / (h_source + h_target)
    """
    source = np.asarray(
        source,
        dtype=np.float64,
    ).reshape(-1, 2)

    target = np.asarray(
        target,
        dtype=np.float64,
    ).reshape(-1, 2)

    if source.shape != target.shape:
        raise ValueError(
            "source and target response shapes differ."
        )

    if not np.all(np.isfinite(target)):
        return np.inf

    hs = parent_brush_shape(source)
    ht = parent_brush_shape(target)

    return float(
        np.max(
            np.abs(source - target)
            / np.maximum(hs + ht, 1.0e-30)
        )
    )


# =============================================================================
# Standard-protocol geometry
# =============================================================================

def ut_principal_stretches(
    lam: np.ndarray,
) -> np.ndarray:
    lam = np.asarray(lam, dtype=np.float64)

    return np.stack(
        [
            lam,
            lam**-0.5,
            lam**-0.5,
        ],
        axis=-1,
    )


def bt_principal_stretches(
    lam: np.ndarray,
) -> np.ndarray:
    lam = np.asarray(lam, dtype=np.float64)

    return np.stack(
        [
            lam,
            lam,
            lam**-2.0,
        ],
        axis=-1,
    )


def simple_shear_geometry(
    gamma: np.ndarray,
):
    """
    Precompute principal stretches and the linear map from principal stresses to
    tau_12 for every simple-shear point.
    """
    gamma = np.asarray(
        gamma,
        dtype=np.float64,
    )

    principal_lam = []
    shear_coeff = []

    for g in gamma:
        F = np.array(
            [
                [1.0, g, 0.0],
                [0.0, 1.0, 0.0],
                [0.0, 0.0, 1.0],
            ],
            dtype=float,
        )

        B = F @ F.T

        eigvals, eigvecs = np.linalg.eigh(B)

        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]

        principal_lam.append(
            np.sqrt(
                np.maximum(eigvals, 0.0)
            )
        )

        # tau_12 = sum_i V_1i V_2i tau_i
        shear_coeff.append(
            eigvecs[0, :]
            * eigvecs[1, :]
        )

    return (
        np.asarray(
            principal_lam,
            dtype=np.float64,
        ),
        np.asarray(
            shear_coeff,
            dtype=np.float64,
        ),
    )


UT_LAM = ut_principal_stretches(UT)
BT_LAM = bt_principal_stretches(BT)
SH_LAM, SH_COEFF = simple_shear_geometry(SH)


def model_standard_unit_curves(
    model: str,
    physical: dict,
) -> dict:
    ut_tau = principal_tau_shape(
        model,
        physical,
        UT_LAM,
    )
    bt_tau = principal_tau_shape(
        model,
        physical,
        BT_LAM,
    )
    sh_tau = principal_tau_shape(
        model,
        physical,
        SH_LAM,
    )

    return {
        "UT": (
            ut_tau[:, 0]
            - ut_tau[:, 2]
        ),
        "BT": (
            bt_tau[:, 0]
            - bt_tau[:, 2]
        ),
        "SH": np.sum(
            SH_COEFF * sh_tau,
            axis=1,
        ),
    }


def withheld_standard_curves() -> dict:
    ut_tau = withheld_principal_tau_shape(
        UT_LAM,
        WITHHELD_B,
    )
    bt_tau = withheld_principal_tau_shape(
        BT_LAM,
        WITHHELD_B,
    )
    sh_tau = withheld_principal_tau_shape(
        SH_LAM,
        WITHHELD_B,
    )

    return {
        "UT": (
            WITHHELD_RESPONSE_SCALE_KPA
            * (ut_tau[:, 0] - ut_tau[:, 2])
        ),
        "BT": (
            WITHHELD_RESPONSE_SCALE_KPA
            * (bt_tau[:, 0] - bt_tau[:, 2])
        ),
        "SH": (
            WITHHELD_RESPONSE_SCALE_KPA
            * np.sum(
                SH_COEFF * sh_tau,
                axis=1,
            )
        ),
    }


# =============================================================================
# Conventional fit metric
# =============================================================================

def standard_protocol_normalizers(
    truth: dict,
) -> dict:
    return {
        name: max(
            float(
                np.max(
                    np.abs(
                        np.asarray(
                            truth[name],
                            dtype=float,
                        )
                    )
                )
            ),
            EPS,
        )
        for name in ("UT", "BT", "SH")
    }


def profiled_standard_scale(
    truth: dict,
    unit: dict,
) -> float:
    norms = standard_protocol_normalizers(
        truth
    )

    y_blocks = []
    t_blocks = []

    for name in ("UT", "BT", "SH"):
        y = (
            np.asarray(
                truth[name],
                dtype=float,
            )
            / norms[name]
        )

        t = (
            np.asarray(
                unit[name],
                dtype=float,
            )
            / norms[name]
        )

        if not np.all(
            np.isfinite(t)
        ):
            return np.nan

        y_blocks.append(y)
        t_blocks.append(t)

    y = np.concatenate(y_blocks)
    t = np.concatenate(t_blocks)

    denom = float(
        np.dot(t, t)
    )

    if denom <= EPS:
        return np.nan

    scale = float(
        np.dot(y, t)
        / denom
    )

    return float(
        np.clip(
            scale,
            RESPONSE_SCALE_BOUNDS_KPA[0],
            RESPONSE_SCALE_BOUNDS_KPA[1],
        )
    )


def standard_nrmse_percent(
    truth: dict,
    prediction: dict,
) -> float:
    norms = standard_protocol_normalizers(
        truth
    )

    blocks = []

    for name in ("UT", "BT", "SH"):
        y = np.asarray(
            truth[name],
            dtype=float,
        )

        p = np.asarray(
            prediction[name],
            dtype=float,
        )

        if not np.all(
            np.isfinite(p)
        ):
            return np.inf

        blocks.append(
            (p - y)
            / norms[name]
        )

    residual = np.concatenate(
        blocks
    )

    return (
        100.0
        * float(
            np.sqrt(
                np.mean(
                    residual**2
                )
            )
        )
    )


# =============================================================================
# Continuous standard-protocol fitting
# =============================================================================

def fit_standard_family(
    model: str,
    truth: dict,
) -> dict:
    if model == "NH":
        physical = {}
        unit = model_standard_unit_curves(
            model,
            physical,
        )

        scale = profiled_standard_scale(
            truth,
            unit,
        )

        prediction = {
            name: scale * unit[name]
            for name in unit
        }

        return {
            "model": model,
            "physical": physical,
            "scale_kpa": scale,
            "standard_nrmse_percent":
                standard_nrmse_percent(
                    truth,
                    prediction,
                ),
            "prediction": prediction,
        }

    def objective(x):
        physical = physical_from_x(
            model,
            np.asarray(x),
        )

        unit = model_standard_unit_curves(
            model,
            physical,
        )

        if any(
            not np.all(np.isfinite(v))
            for v in unit.values()
        ):
            return 1.0e9

        scale = profiled_standard_scale(
            truth,
            unit,
        )

        if not np.isfinite(scale):
            return 1.0e9

        prediction = {
            name: scale * unit[name]
            for name in unit
        }

        return standard_nrmse_percent(
            truth,
            prediction,
        )

    bounds = PHYSICAL_BOUNDS[model]

    if len(bounds) == 1:
        lo, hi = bounds[0]

        result = minimize_scalar(
            lambda z: objective([z]),
            bounds=(lo, hi),
            method="bounded",
            options={
                "xatol": 1.0e-10,
                "maxiter": 500,
            },
        )

        x_best = np.array(
            [result.x],
            dtype=float,
        )

    else:
        best_de = None

        for seed in DE_SEEDS:
            result = differential_evolution(
                objective,
                bounds,
                seed=seed,
                maxiter=120,
                popsize=12,
                tol=1.0e-9,
                polish=True,
                updating="immediate",
                workers=1,
            )

            if (
                best_de is None
                or result.fun < best_de.fun
            ):
                best_de = result

        x_best = np.asarray(
            best_de.x,
            dtype=float,
        )

    physical = physical_from_x(
        model,
        x_best,
    )

    unit = model_standard_unit_curves(
        model,
        physical,
    )

    scale = profiled_standard_scale(
        truth,
        unit,
    )

    prediction = {
        name: scale * unit[name]
        for name in unit
    }

    return {
        "model": model,
        "physical": physical,
        "scale_kpa": float(scale),
        "standard_nrmse_percent":
            float(
                standard_nrmse_percent(
                    truth,
                    prediction,
                )
            ),
        "prediction": prediction,
    }


# =============================================================================
# Continuous complete-F fitting
# =============================================================================

def fit_completeF_family(
    model: str,
    source: np.ndarray,
    lam: np.ndarray,
    stage_name: str,
) -> dict:
    """
    Jointly optimize physical shape and positive response scale against the
    exact V68-style required-noise objective.
    """
    shape_bounds = PHYSICAL_BOUNDS[model]

    log_scale_bounds = (
        math.log(
            RESPONSE_SCALE_BOUNDS_KPA[0]
        ),
        math.log(
            RESPONSE_SCALE_BOUNDS_KPA[1]
        ),
    )

    full_bounds = (
        shape_bounds
        + [log_scale_bounds]
    )

    def objective(z):
        z = np.asarray(
            z,
            dtype=float,
        )

        if model == "NH":
            physical = {}
            log_scale = float(z[0])
        else:
            physical = physical_from_x(
                model,
                z[:-1],
            )
            log_scale = float(z[-1])

        scale = float(
            np.exp(log_scale)
        )

        try:
            target = model_parent_response(
                model,
                physical,
                scale,
                lam,
            )
        except Exception:
            return 1.0e6

        if not np.all(
            np.isfinite(target)
        ):
            return 1.0e6

        return required_noise_fraction(
            source,
            target,
        )

    # NH is a 1-D scale-only problem.
    if model == "NH":
        result = minimize_scalar(
            lambda z: objective([z]),
            bounds=log_scale_bounds,
            method="bounded",
            options={
                "xatol": 1.0e-10,
                "maxiter": 600,
            },
        )

        x_best = np.array(
            [result.x],
            dtype=float,
        )
        best_fun = float(
            result.fun
        )
        best_seed = -1

    else:
        dimension = len(
            full_bounds
        )

        maxiter = (
            DE_MAXITER_MULTI
            if dimension >= 4
            else DE_MAXITER_SCALAR
        )

        popsize = (
            DE_POPSIZE_MULTI
            if dimension >= 4
            else DE_POPSIZE_SCALAR
        )

        best_result = None
        best_seed = None

        for seed in DE_SEEDS:
            result = differential_evolution(
                objective,
                full_bounds,
                seed=seed,
                maxiter=maxiter,
                popsize=popsize,
                tol=DE_TOL,
                polish=True,
                updating="immediate",
                workers=1,
            )

            if (
                best_result is None
                or result.fun
                < best_result.fun
            ):
                best_result = result
                best_seed = seed

        x_best = np.asarray(
            best_result.x,
            dtype=float,
        )

        best_fun = float(
            best_result.fun
        )

    if model == "NH":
        physical = {}
        scale = float(
            np.exp(
                x_best[0]
            )
        )
    else:
        physical = physical_from_x(
            model,
            x_best[:-1],
        )
        scale = float(
            np.exp(
                x_best[-1]
            )
        )

    prediction = model_parent_response(
        model,
        physical,
        scale,
        lam,
    )

    score = required_noise_fraction(
        source,
        prediction,
    )

    print(
        f"[Example 3]   {stage_name:<10s} "
        f"{model:<6s} continuous mismatch = "
        f"{100.0*score:.4f}%"
    )

    return {
        "model": model,
        "physical": physical,
        "scale_kpa": scale,
        "required_noise_fraction": float(score),
        "required_noise_percent": float(
            100.0 * score
        ),
        "optimizer_seed": int(best_seed),
        "prediction": prediction,
    }


# =============================================================================
# Independent dense complete-F validation domain
# =============================================================================

def dense_intrinsic_grid(
    n: int,
) -> pd.DataFrame:
    """
    Unique incompressible principal-stretch wedge:
        e1 + e2 + e3 = 0
        lambda_i in [0.5, 2]
        e1 >= e2 >= e3
    """
    emin = math.log(0.5)
    emax = math.log(2.0)

    e1_values = np.linspace(
        emin,
        emax,
        int(n),
    )

    e2_values = np.linspace(
        emin,
        emax,
        int(n),
    )

    rows = []

    for e1 in e1_values:
        for e2 in e2_values:
            e3 = -e1 - e2

            if not (
                emin - 1.0e-12
                <= e3
                <= emax + 1.0e-12
            ):
                continue

            if not (
                e1 >= e2 - 1.0e-12
                and e2 >= e3 - 1.0e-12
            ):
                continue

            lam = np.exp(
                np.array(
                    [e1, e2, e3],
                    dtype=float,
                )
            )

            rows.append(
                (
                    e1,
                    e2,
                    e3,
                    lam[0],
                    lam[1],
                    lam[2],
                )
            )

    grid = pd.DataFrame(
        rows,
        columns=[
            "e1",
            "e2",
            "e3",
            "lambda1",
            "lambda2",
            "lambda3",
        ],
    )

    if not len(grid):
        raise RuntimeError(
            "Dense validation grid is empty."
        )

    return grid


# =============================================================================
# F* localization
# =============================================================================

def pointwise_required_noise_fraction(
    source: np.ndarray,
    target: np.ndarray,
) -> np.ndarray:
    source = np.asarray(
        source,
        dtype=float,
    ).reshape(-1, 2)

    target = np.asarray(
        target,
        dtype=float,
    ).reshape(-1, 2)

    hs = parent_brush_shape(
        source
    )

    ht = parent_brush_shape(
        target
    )

    return np.max(
        np.abs(source - target)
        / np.maximum(
            hs + ht,
            1.0e-30,
        ),
        axis=1,
    )


def localize_Fstar(
    grid: pd.DataFrame,
    truth: np.ndarray,
    prediction: np.ndarray,
):
    point_score = (
        100.0
        * pointwise_required_noise_fraction(
            truth,
            prediction,
        )
    )

    out = grid.copy()
    out["pointwise_required_noise_percent"] = (
        point_score
    )

    out["truth_d13_kpa"] = (
        truth[:, 0]
    )
    out["truth_d23_kpa"] = (
        truth[:, 1]
    )
    out["winner_d13_kpa"] = (
        prediction[:, 0]
    )
    out["winner_d23_kpa"] = (
        prediction[:, 1]
    )

    global_idx = int(
        np.nanargmax(
            point_score
        )
    )

    lam = out[
        [
            "lambda1",
            "lambda2",
            "lambda3",
        ]
    ].to_numpy(float)

    # Manuscript-highlighted interior state:
    #   1) all principal stretches remain away from the atlas boundaries;
    #   2) the state is meaningfully deformed rather than arbitrarily close to F=I.
    log_lam = np.log(lam)
    deformation_magnitude = np.max(
        np.abs(log_lam),
        axis=1,
    )

    interior_mask = (
        np.all(
            (
                lam
                >= INTERIOR_LAMBDA_MIN
            )
            & (
                lam
                <= INTERIOR_LAMBDA_MAX
            ),
            axis=1,
        )
        & (
            deformation_magnitude
            >= MIN_INTERIOR_LOG_STRAIN_MAGNITUDE
        )
    )

    if np.any(interior_mask):
        interior_positions = np.flatnonzero(
            interior_mask
        )

        interior_idx = int(
            interior_positions[
                np.nanargmax(
                    point_score[
                        interior_mask
                    ]
                )
            ]
        )
    else:
        interior_idx = global_idx

    out["is_global_Fstar"] = 0
    out["is_interior_Fstar"] = 0

    out.loc[
        global_idx,
        "is_global_Fstar",
    ] = 1

    out.loc[
        interior_idx,
        "is_interior_Fstar",
    ] = 1

    def package(idx):
        row = out.iloc[idx]

        return {
            "row": int(idx),
            "e1": float(row["e1"]),
            "e2": float(row["e2"]),
            "e3": float(row["e3"]),
            "lambda1": float(row["lambda1"]),
            "lambda2": float(row["lambda2"]),
            "lambda3": float(row["lambda3"]),
            "max_abs_log_lambda": float(
                max(
                    abs(float(row["e1"])),
                    abs(float(row["e2"])),
                    abs(float(row["e3"])),
                )
            ),
            "pointwise_required_noise_percent":
                float(
                    row[
                        "pointwise_required_noise_percent"
                    ]
                ),
            "truth_d13_kpa":
                float(
                    row["truth_d13_kpa"]
                ),
            "truth_d23_kpa":
                float(
                    row["truth_d23_kpa"]
                ),
            "winner_d13_kpa":
                float(
                    row["winner_d13_kpa"]
                ),
            "winner_d23_kpa":
                float(
                    row["winner_d23_kpa"]
                ),
        }

    return (
        out,
        package(global_idx),
        package(interior_idx),
    )


# =============================================================================
# Figure
# =============================================================================

def make_figure(
    truth_standard: dict,
    standard_fits: dict,
    parent_fits: dict,
    validation_fits: dict,
    validation_map: pd.DataFrame,
    global_Fstar: dict,
    interior_Fstar: dict,
):
    winner = min(
        standard_fits.values(),
        key=lambda r:
            r["standard_nrmse_percent"],
    )

    winner_model = str(
        winner["model"]
    )

    fig = plt.figure(
        figsize=(15.5, 11.0)
    )

    # ------------------------------------------------------------------
    # A — conventional protocols
    # ------------------------------------------------------------------
    ax = fig.add_subplot(
        2,
        2,
        1,
    )

    xu = np.linspace(
        0.0,
        1.0,
        len(UT),
    )
    xb = np.linspace(
        1.25,
        2.25,
        len(BT),
    )
    xs = np.linspace(
        2.50,
        3.50,
        len(SH),
    )

    pred = winner["prediction"]

    ax.plot(
        xu,
        truth_standard["UT"],
        linewidth=2.6,
        label="Withheld truth",
    )
    ax.plot(
        xu,
        pred["UT"],
        linestyle="--",
        linewidth=2.2,
        label=f"Best conventional fit: {winner_model}",
    )

    ax.plot(
        xb,
        truth_standard["BT"],
        linewidth=2.6,
    )
    ax.plot(
        xb,
        pred["BT"],
        linestyle="--",
        linewidth=2.2,
    )

    ax.plot(
        xs,
        truth_standard["SH"],
        linewidth=2.6,
    )
    ax.plot(
        xs,
        pred["SH"],
        linestyle="--",
        linewidth=2.2,
    )

    ax.axvline(
        1.125,
        linewidth=0.8,
        alpha=0.35,
    )
    ax.axvline(
        2.375,
        linewidth=0.8,
        alpha=0.35,
    )

    ax.set_xticks(
        [0.5, 1.75, 3.0]
    )

    ax.set_xticklabels(
        [
            "UT\nλ=0.70–1.50",
            "BT\nλ=0.80–1.25",
            "SH\nγ=−0.50–0.50",
        ]
    )

    ax.set_ylabel(
        "Pressure-independent stress response (kPa)"
    )

    ax.set_title(
        "A. Standard protocols give an apparently excellent model fit\n"
        f"{winner_model} pooled UT+BT+SH NRMSE = "
        f"{winner['standard_nrmse_percent']:.3f}%"
    )

    ax.legend(
        fontsize=8,
        loc="best",
    )

    # ------------------------------------------------------------------
    # B — conventional relative ranking
    # ------------------------------------------------------------------
    ax = fig.add_subplot(
        2,
        2,
        2,
    )

    order = sorted(
        MODELS,
        key=lambda m:
            standard_fits[m][
                "standard_nrmse_percent"
            ],
    )

    vals = np.array(
        [
            standard_fits[m][
                "standard_nrmse_percent"
            ]
            for m in order
        ],
        dtype=float,
    )

    x = np.arange(
        len(order)
    )

    ax.bar(
        x,
        vals,
    )

    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.8,
        label="3% reference",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        order,
        rotation=35,
    )

    ax.set_ylabel(
        "Pooled UT+BT+SH NRMSE (%)"
    )

    ax.set_title(
        "B. Relative model selection always returns a winner\n"
        "A low ranking error does not establish absolute adequacy"
    )

    ax.legend(
        fontsize=8,
        loc="best",
    )

    # ------------------------------------------------------------------
    # C — continuous complete-F NONE verdict
    # ------------------------------------------------------------------
    ax = fig.add_subplot(
        2,
        2,
        3,
    )

    parent_vals = np.array(
        [
            parent_fits[m][
                "required_noise_percent"
            ]
            for m in MODELS
        ],
        dtype=float,
    )

    validation_vals = np.array(
        [
            validation_fits[m][
                "required_noise_percent"
            ]
            for m in MODELS
        ],
        dtype=float,
    )

    xx = np.arange(
        len(MODELS)
    )

    width = 0.36

    ax.bar(
        xx - width / 2,
        parent_vals,
        width,
        label="Frozen V68 parent (2601 states)",
    )

    ax.bar(
        xx + width / 2,
        validation_vals,
        width,
        label="Independent dense validation",
    )

    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.8,
        label="3% compatibility threshold",
    )

    ax.axhline(
        5.0,
        linestyle=":",
        linewidth=1.6,
        label="5% robust-NONE margin",
    )

    ax.set_xticks(xx)
    ax.set_xticklabels(
        MODELS,
        rotation=35,
    )

    ax.set_ylabel(
        "Continuously optimized required noise (%)"
    )

    parent_min = float(
        np.min(parent_vals)
    )

    validation_min = float(
        np.min(validation_vals)
    )

    ax.set_title(
        "C. Absolute complete-F adequacy rejects the entire seven-model library\n"
        f"best family: {parent_min:.2f}% parent, "
        f"{validation_min:.2f}% independent validation"
    )

    ax.legend(
        fontsize=7.4,
        loc="best",
    )

    # ------------------------------------------------------------------
    # D — failure localization
    # ------------------------------------------------------------------
    ax = fig.add_subplot(
        2,
        2,
        4,
    )

    sc = ax.scatter(
        validation_map["e1"],
        validation_map["e2"],
        c=validation_map[
            "pointwise_required_noise_percent"
        ],
        s=13,
    )

    cb = fig.colorbar(
        sc,
        ax=ax,
        fraction=0.047,
        pad=0.03,
    )

    cb.set_label(
        f"{winner_model} pointwise required noise (%)"
    )

    ax.scatter(
        [global_Fstar["e1"]],
        [global_Fstar["e2"]],
        marker="*",
        s=160,
        edgecolor="black",
        linewidth=0.9,
        label="Global F*",
        zorder=7,
    )

    ax.scatter(
        [interior_Fstar["e1"]],
        [interior_Fstar["e2"]],
        marker="D",
        s=75,
        edgecolor="black",
        linewidth=0.8,
        label=(
            f"Deformed interior F* "
            f"({INTERIOR_LAMBDA_MIN:.2f}≤λᵢ≤{INTERIOR_LAMBDA_MAX:.2f}, "
            f"max|log λᵢ|≥{MIN_INTERIOR_LOG_STRAIN_MAGNITUDE:.2f})"
        ),
        zorder=7,
    )

    ax.set_xlabel(
        "Principal log strain e₁ = log λ₁"
    )

    ax.set_ylabel(
        "Principal log strain e₂ = log λ₂"
    )

    ax.set_title(
        "D. The forced conventional winner fails at identifiable in-atlas states\n"
        f"global max = "
        f"{global_Fstar['pointwise_required_noise_percent']:.1f}%; "
        f"deformed-interior max = "
        f"{interior_Fstar['pointwise_required_noise_percent']:.1f}%"
    )

    ax.legend(
        fontsize=7.5,
        loc="best",
    )

    fig.suptitle(
        "Example 3 — A best conventional fit can still be NONE-compatible",
        fontsize=15.5,
    )

    fig.tight_layout(
        rect=[
            0,
            0.01,
            1,
            0.965,
        ]
    )

    png = (
        OUTDIR
        / "example3_none_compatible_figure.png"
    )

    pdf = (
        OUTDIR
        / "example3_none_compatible_figure.pdf"
    )

    fig.savefig(
        png,
        dpi=320,
        bbox_inches="tight",
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    return png, pdf


# =============================================================================
# Main
# =============================================================================

def main():
    ensure_drive()

    if not PARENT_STATES_CSV.is_file():
        raise FileNotFoundError(
            "Missing frozen V68 complete-F parent-state CSV:\n"
            f"{PARENT_STATES_CSV}"
        )

    parent_df = pd.read_csv(
        PARENT_STATES_CSV
    )

    required_cols = {
        "log_lambda1",
        "log_lambda2",
        "log_lambda3",
        "lambda1",
        "lambda2",
        "lambda3",
        "detF",
    }

    missing = sorted(
        required_cols
        - set(parent_df.columns)
    )

    if missing:
        raise RuntimeError(
            f"Parent-state CSV is missing columns: {missing}"
        )

    parent_lam = parent_df[
        [
            "lambda1",
            "lambda2",
            "lambda3",
        ]
    ].to_numpy(float)

    print(
        f"[Example 3] Loaded frozen complete-F parent: "
        f"{len(parent_df)} states."
    )

    print(
        "[Example 3] Standalone mode: "
        "no V68 Python file imported."
    )

    print(
        f"[Example 3] Withheld truth: "
        f"{WITHHELD_MODEL_NAME}, b={WITHHELD_B:.2f}, "
        f"response scale={WITHHELD_RESPONSE_SCALE_KPA:.1f} kPa."
    )

    # ------------------------------------------------------------------
    # 1. Conventional UT + BT + SH fits
    # ------------------------------------------------------------------
    truth_standard = (
        withheld_standard_curves()
    )

    standard_fits = {}

    print(
        "[Example 3] Continuously fitting all seven families "
        "to conventional UT+BT+SH..."
    )

    for model in MODELS:
        fit = fit_standard_family(
            model,
            truth_standard,
        )

        standard_fits[model] = fit

        print(
            f"[Example 3]   {model:<6s} "
            f"standard NRMSE = "
            f"{fit['standard_nrmse_percent']:.4f}%"
        )

    conventional_winner = min(
        standard_fits.values(),
        key=lambda r:
            r["standard_nrmse_percent"],
    )

    # ------------------------------------------------------------------
    # 2. Frozen exact V68 parent continuous adequacy
    # ------------------------------------------------------------------
    parent_truth = (
        withheld_parent_response(
            parent_lam
        )
    )

    parent_fits = {}

    print(
        "[Example 3] Continuous complete-F optimization "
        "on the frozen 2601-state V68 parent..."
    )

    for model in MODELS:
        parent_fits[model] = (
            fit_completeF_family(
                model,
                parent_truth,
                parent_lam,
                "parent",
            )
        )

    # ------------------------------------------------------------------
    # 3. Independent dense intrinsic convergence validation
    # ------------------------------------------------------------------
    validation_df = dense_intrinsic_grid(
        VALIDATION_GRID_N
    )

    validation_lam = (
        validation_df[
            [
                "lambda1",
                "lambda2",
                "lambda3",
            ]
        ].to_numpy(float)
    )

    validation_truth = (
        withheld_parent_response(
            validation_lam
        )
    )

    validation_fits = {}

    print(
        f"[Example 3] Re-optimizing all seven families on "
        f"an independent {len(validation_df)}-state dense intrinsic grid..."
    )

    for model in MODELS:
        validation_fits[model] = (
            fit_completeF_family(
                model,
                validation_truth,
                validation_lam,
                "validate",
            )
        )

    parent_best_model = min(
        MODELS,
        key=lambda m:
            parent_fits[m][
                "required_noise_fraction"
            ],
    )

    validation_best_model = min(
        MODELS,
        key=lambda m:
            validation_fits[m][
                "required_noise_fraction"
            ],
    )

    parent_min = float(
        parent_fits[
            parent_best_model
        ][
            "required_noise_fraction"
        ]
    )

    validation_min = float(
        validation_fits[
            validation_best_model
        ][
            "required_noise_fraction"
        ]
    )

    # ------------------------------------------------------------------
    # Hard publication gates
    # ------------------------------------------------------------------
    if (
        conventional_winner[
            "standard_nrmse_percent"
        ]
        > 3.0
    ):
        raise RuntimeError(
            "The withheld example is not deceptive enough under "
            "standard UT+BT+SH: best conventional fit exceeds 3%."
        )

    if (
        parent_min
        <= ROBUST_NONE_MARGIN_FRACTION
    ):
        raise RuntimeError(
            "Frozen-parent NONE margin is not robust enough: "
            f"best continuous family = {100*parent_min:.3f}% "
            f"(required > {100*ROBUST_NONE_MARGIN_FRACTION:.1f}%)."
        )

    if (
        validation_min
        <= ROBUST_NONE_MARGIN_FRACTION
    ):
        raise RuntimeError(
            "Independent-grid NONE margin is not robust enough: "
            f"best continuous family = {100*validation_min:.3f}% "
            f"(required > {100*ROBUST_NONE_MARGIN_FRACTION:.1f}%)."
        )

    # ------------------------------------------------------------------
    # 4. Localize failure of the conventional winner
    # ------------------------------------------------------------------
    winner_model = str(
        conventional_winner["model"]
    )

    winner_validation_fit = (
        validation_fits[
            winner_model
        ]
    )

    validation_map, global_Fstar, interior_Fstar = (
        localize_Fstar(
            validation_df,
            validation_truth,
            winner_validation_fit[
                "prediction"
            ],
        )
    )

    # ------------------------------------------------------------------
    # Exports
    # ------------------------------------------------------------------
    standard_rows = []

    for model in MODELS:
        fit = standard_fits[model]

        standard_rows.append(
            {
                "model": model,
                "standard_nrmse_percent":
                    fit[
                        "standard_nrmse_percent"
                    ],
                "scale_kpa":
                    fit["scale_kpa"],
                "physical_json":
                    json.dumps(
                        fit["physical"],
                        sort_keys=True,
                    ),
                "is_conventional_winner":
                    model
                    == winner_model,
            }
        )

    pd.DataFrame(
        standard_rows
    ).sort_values(
        "standard_nrmse_percent"
    ).to_csv(
        OUTDIR
        / "example3_standard_fit_summary.csv",
        index=False,
    )

    parent_rows = []

    for model in MODELS:
        fit = parent_fits[model]

        parent_rows.append(
            {
                "model": model,
                "required_noise_fraction":
                    fit[
                        "required_noise_fraction"
                    ],
                "required_noise_percent":
                    fit[
                        "required_noise_percent"
                    ],
                "scale_kpa":
                    fit["scale_kpa"],
                "physical_json":
                    json.dumps(
                        fit["physical"],
                        sort_keys=True,
                    ),
                "optimizer_seed":
                    fit["optimizer_seed"],
                "is_best_family":
                    model
                    == parent_best_model,
            }
        )

    pd.DataFrame(
        parent_rows
    ).sort_values(
        "required_noise_fraction"
    ).to_csv(
        OUTDIR
        / "example3_completeF_continuous_parent.csv",
        index=False,
    )

    validation_rows = []

    for model in MODELS:
        fit = validation_fits[model]

        validation_rows.append(
            {
                "model": model,
                "required_noise_fraction":
                    fit[
                        "required_noise_fraction"
                    ],
                "required_noise_percent":
                    fit[
                        "required_noise_percent"
                    ],
                "scale_kpa":
                    fit["scale_kpa"],
                "physical_json":
                    json.dumps(
                        fit["physical"],
                        sort_keys=True,
                    ),
                "optimizer_seed":
                    fit["optimizer_seed"],
                "is_best_family":
                    model
                    == validation_best_model,
            }
        )

    pd.DataFrame(
        validation_rows
    ).sort_values(
        "required_noise_fraction"
    ).to_csv(
        OUTDIR
        / "example3_completeF_continuous_validation.csv",
        index=False,
    )

    validation_map.to_csv(
        OUTDIR
        / "example3_validation_error_map.csv",
        index=False,
    )

    fstar_package = {
        "global_Fstar":
            global_Fstar,
        "interior_Fstar":
            interior_Fstar,
        "interior_domain":
            {
                "lambda_min":
                    INTERIOR_LAMBDA_MIN,
                "lambda_max":
                    INTERIOR_LAMBDA_MAX,
                "minimum_max_abs_log_lambda":
                    MIN_INTERIOR_LOG_STRAIN_MAGNITUDE,
            },
    }

    with (
        OUTDIR
        / "example3_Fstar.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            fstar_package,
            f,
            indent=2,
        )

    summary = {
        "scientific_message": (
            "The withheld exponential isotropic response is fitted extremely "
            "well by a candidate family under standard UT, BT, and SH, yet "
            "continuous optimization over the complete-F domain shows that "
            "every atlas family lies outside the declared 3% compatibility "
            "tube. The correct model-library verdict is therefore "
            "NONE_COMPATIBLE rather than a forced best-model label."
        ),
        "withheld_truth": {
            "model": WITHHELD_MODEL_NAME,
            "b": WITHHELD_B,
            "response_scale_kpa":
                WITHHELD_RESPONSE_SCALE_KPA,
        },
        "metric": {
            "type":
                "V68 symmetric required-noise fraction",
            "mode_floor_per_unit_noise":
                MODE_FLOOR_PER_UNIT_NOISE,
            "compatibility_threshold_percent":
                100
                * ATLAS_NOISE_THRESHOLD_FRACTION,
            "robust_none_margin_percent":
                100
                * ROBUST_NONE_MARGIN_FRACTION,
            "highlighted_interior_Fstar_min_max_abs_log_lambda":
                MIN_INTERIOR_LOG_STRAIN_MAGNITUDE,
        },
        "conventional_winner": {
            "model": winner_model,
            "pooled_UT_BT_SH_NRMSE_percent":
                conventional_winner[
                    "standard_nrmse_percent"
                ],
        },
        "frozen_parent": {
            "n_states":
                int(len(parent_df)),
            "best_family":
                parent_best_model,
            "minimum_required_noise_percent":
                100.0
                * parent_min,
        },
        "independent_validation": {
            "n_states":
                int(len(validation_df)),
            "best_family":
                validation_best_model,
            "minimum_required_noise_percent":
                100.0
                * validation_min,
        },
        "verdict":
            "NONE_COMPATIBLE",
        "global_Fstar":
            global_Fstar,
        "interior_Fstar":
            interior_Fstar,
    }

    with (
        OUTDIR
        / "example3_summary.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
        )

    png, pdf = make_figure(
        truth_standard,
        standard_fits,
        parent_fits,
        validation_fits,
        validation_map,
        global_Fstar,
        interior_Fstar,
    )

    print(
        "\n"
        + "=" * 122
    )

    print(
        "[Example 3] COMPLETE — "
        "ROBUST NONE-COMPATIBLE CASE"
    )

    print(
        f"[Example 3] Withheld truth: "
        f"{WITHHELD_MODEL_NAME}, "
        f"b={WITHHELD_B:.2f}"
    )

    print(
        f"[Example 3] Conventional winner: "
        f"{winner_model} | "
        f"UT+BT+SH NRMSE="
        f"{conventional_winner['standard_nrmse_percent']:.4f}%"
    )

    print(
        f"[Example 3] Frozen-parent best family: "
        f"{parent_best_model} | "
        f"required noise="
        f"{100*parent_min:.4f}%"
    )

    print(
        f"[Example 3] Independent-grid best family: "
        f"{validation_best_model} | "
        f"required noise="
        f"{100*validation_min:.4f}%"
    )

    print(
        "[Example 3] Verdict: NONE_COMPATIBLE "
        "(all continuously optimized families remain >5%)."
    )

    print(
        f"[Example 3] Global F*: "
        f"lambda=("
        f"{global_Fstar['lambda1']:.4f}, "
        f"{global_Fstar['lambda2']:.4f}, "
        f"{global_Fstar['lambda3']:.4f}) | "
        f"pointwise required noise="
        f"{global_Fstar['pointwise_required_noise_percent']:.2f}%"
    )

    print(
        f"[Example 3] Deformed interior F*: "
        f"lambda=("
        f"{interior_Fstar['lambda1']:.4f}, "
        f"{interior_Fstar['lambda2']:.4f}, "
        f"{interior_Fstar['lambda3']:.4f}) | "
        f"pointwise required noise="
        f"{interior_Fstar['pointwise_required_noise_percent']:.2f}%"
    )

    print(
        f"[Example 3] Figure PNG: "
        f"{png}"
    )

    print(
        f"[Example 3] Figure PDF: "
        f"{pdf}"
    )

    print(
        f"[Example 3] Results: "
        f"{OUTDIR}"
    )

    print(
        "=" * 122
    )


if __name__ == "__main__":
    main()


[Example 3] Google Drive already mounted.
[Example 3] Loaded frozen complete-F parent: 2601 states.
[Example 3] Standalone mode: no V68 Python file imported.
[Example 3] Withheld truth: EXPONENTIAL_I1, b=1.00, response scale=50.0 kPa.
[Example 3] Continuously fitting all seven families to conventional UT+BT+SH...
[Example 3]   NH     standard NRMSE = 7.2522%
[Example 3]   MR     standard NRMSE = 7.2522%
[Example 3]   YEOH2  standard NRMSE = 0.6689%
[Example 3]   GENT   standard NRMSE = 3.8357%
[Example 3]   OGDEN1 standard NRMSE = 4.2117%
[Example 3]   OGDEN2 standard NRMSE = 0.2219%
[Example 3]   GP2    standard NRMSE = 0.6689%
[Example 3] Continuous complete-F optimization on the frozen 2601-state V68 parent...
[Example 3]   parent     NH     continuous mismatch = 43.6985%
[Example 3]   parent     MR     continuous mismatch = 43.6985%
[Example 3]   parent     YEOH2  continuous mismatch = 12.6095%
[Example 3]   parent     GENT   continuous mismatch = 12.6095%
[Example 3]   parent     

In [ ]:
#!/usr/bin/env python3
"""
V68 PUBLICATION FIGURES 3–5 — standalone final-cell script
==========================================================

Purpose
-------
Generate the manuscript's final data-driven Figures 3, 4, and 5 from the
already-completed V68/V68.2 frozen outputs.  This script does NOT import any
earlier notebook cell or V68 Python module.

It is intended to be pasted/run as the LAST notebook cell after the V68 atlas
and Cell-2 publication validation have completed.

Generated figures
-----------------
Figure 3 — Final certified constitutive-response atlas
    A. Scale-free complete-F PCA colored by FINAL CERTIFIED compatibility set
    B. Certified compatibility-set frequency distribution

Figure 4 — Numerical certification and robustness
    A. Independent held-out source-family recovery by noise regime
    B. Dense boundary certification (baseline vs certified scores)
    C. Threshold persistence of atlas-wide compatibility-set cardinality
    D. Complete-F parent-resolution convergence
    E. Exact hierarchy and parent/subprotocol identity errors
    F. Stratified off-manifold NONE retention

Figure 5 — Information loss under standard loading protocols
    A–D. Independently constructed scale-free PCA views: ALL, UT, BT, SH
    E. Response-space dimensionality
    F. Repeated-CV three-PC visibility (balanced accuracy and macro-F1)

Outputs
-------
<ROOT>/publication_figures_3_5/
    Figure_3_Final_Certified_Atlas.png/.pdf/.svg
    Figure_4_Numerical_Certification.png/.pdf/.svg
    Figure_5_Protocol_Information_Loss.png/.pdf/.svg
    figure3_region_summary.csv
    figure4_publication_metrics.json
    figure5_visibility_recomputed.csv
    region_color_key.csv
    publication_figures_manifest.json

Notes
-----
* The expensive constitutive optimization is NOT rerun here.  Figures 3–5 are
  publication summaries of the frozen, certified numerical solution already
  produced by Cells 1–2.
* Figure-5 visibility is recomputed independently from the saved PCA coordinates
  and FINAL CERTIFIED labels using the exact Cell-2 7-NN, 5-fold x 5-repeat
  diagnostic, including the same cross-validation random-state seed.
* The script verifies the source/certified/PCA key alignment and fails loudly if
  required frozen outputs are missing or inconsistent.
"""

from __future__ import annotations

import json
import math
import os
import hashlib
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import balanced_accuracy_score, f1_score
except Exception as exc:
    raise RuntimeError(
        "scikit-learn is required for the independent Figure-5 visibility "
        "recomputation. In Colab it is normally preinstalled."
    ) from exc


# =============================================================================
# Configuration
# =============================================================================

PROJECT_BASENAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)
CELL2_SUBDIR = "cell2_publication_validation"
FIG_SUBDIR = "publication_figures_3_5"

ALL_MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
PCA_VIEWS = ["ALL", "UT", "BT", "SH"]
THRESHOLDS_PERCENT = [0, 1, 2, 3, 5]

# Exact Cell-2 settings used for the published 3-PC visibility diagnostic.
PCA_K = 7
PCA_CV_SPLITS = 5
PCA_CV_REPEATS = 5
PCA_RARE_MIN_COUNT = 5
CELL2_VALIDATION_SEED = 20260806 + 2200
PCA_RANDOM_SEED = CELL2_VALIDATION_SEED + 900  # exact V68.2.4 Cell-2 PCA CV seed

HARD_HELDOUT_REGIMES = ("CLEAN", "BOUNDED_INTERIOR_2P7")
HELDOUT_GATE = 0.995
NONE_GATE = 1.0
RESOLUTION_MEDIAN_GATE = 0.0015
RESOLUTION_LABEL_GATE = 0.99
HIERARCHY_ERROR_GATE = 1.0e-9
PARENT_SUBPROTOCOL_ERROR_GATE = 1.0e-11

# Journal-scale export.
PNG_DPI = 600
FIG_WIDE = (13.2, 6.3)
FIG_GRID = (13.2, 8.5)

# Use a deterministic, colorblind-conscious set.  The mapping is assigned to
# regions by descending certified frequency and shared by Figures 3 and 5.
REGION_PALETTE = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
    "#56B4E9", "#000000", "#7A5195", "#EF5675", "#FFA600",
    "#4C78A8", "#F58518", "#54A24B", "#B279A2", "#9D755D",
    "#BAB0AC", "#72B7B2", "#FF9DA6", "#A0CBE8", "#8CD17D",
]

# Consistent protocol colors in Figure 5.
PROTOCOL_COLORS = {
    "ALL": "#222222",
    "UT": "#0072B2",
    "BT": "#D55E00",
    "SH": "#009E73",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.0,
    "axes.titlesize": 10.0,
    "axes.labelsize": 9.0,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})


# =============================================================================
# Drive / root discovery
# =============================================================================

def _required_anchor(root: Path) -> Path:
    return root / CELL2_SUBDIR / "certified_source_atlas_states.csv"


def _candidate_roots() -> List[Path]:
    candidates: List[Path] = []

    explicit = os.environ.get("V68_ROOT", "").strip()
    if explicit:
        candidates.append(Path(explicit).expanduser())

    for mydrive in (
        Path("/content/v68_drive/MyDrive"),
        Path("/content/drive/MyDrive"),
        Path("/content/v68_figures_drive/MyDrive"),
    ):
        candidates.append(mydrive / "Optimal_Protocol" / PROJECT_BASENAME)

    # Preserve order, remove duplicates.
    out: List[Path] = []
    seen = set()
    for p in candidates:
        s = str(p)
        if s not in seen:
            out.append(p)
            seen.add(s)
    return out


def _find_existing_root() -> Path | None:
    for root in _candidate_roots():
        if _required_anchor(root).is_file():
            return root.resolve()

    # Limited discovery inside known mounted MyDrive roots only.
    for mydrive in (
        Path("/content/v68_drive/MyDrive"),
        Path("/content/drive/MyDrive"),
        Path("/content/v68_figures_drive/MyDrive"),
    ):
        optimal = mydrive / "Optimal_Protocol"
        if not optimal.is_dir():
            continue
        try:
            matches = list(optimal.glob(PROJECT_BASENAME + "*"))
        except Exception:
            matches = []
        valid = [p for p in matches if _required_anchor(p).is_file()]
        if len(valid) == 1:
            return valid[0].resolve()
        if len(valid) > 1:
            listing = "\n".join(f"  - {p}" for p in valid)
            raise RuntimeError(
                "Multiple V68 roots containing certified Cell-2 outputs were found. "
                "Set V68_ROOT explicitly before running this cell.\n" + listing
            )
    return None


def _mount_fallback_drive() -> None:
    """Mount Drive only if no valid V68 root is currently visible."""
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        return

    mountpoint = Path("/content/v68_figures_drive")
    if (mountpoint / "MyDrive").is_dir():
        return

    if mountpoint.exists():
        try:
            entries = list(mountpoint.iterdir())
        except Exception:
            entries = []
        if entries:
            raise RuntimeError(
                f"{mountpoint} is non-empty and is not a usable Drive mount. "
                "Restart the runtime or set V68_ROOT explicitly."
            )

    print("[Figures 3–5] Mounting Google Drive at /content/v68_figures_drive ...")
    drive.mount(str(mountpoint), force_remount=False)


def resolve_root() -> Path:
    root = _find_existing_root()
    if root is not None:
        return root

    _mount_fallback_drive()
    root = _find_existing_root()
    if root is not None:
        return root

    searched = "\n".join(f"  - {p}" for p in _candidate_roots())
    raise FileNotFoundError(
        "Could not locate the frozen V68 atlas with certified Cell-2 outputs.\n"
        "Searched:\n" + searched + "\n"
        "Set os.environ['V68_ROOT'] to the exact atlas folder and rerun."
    )


# =============================================================================
# Utilities
# =============================================================================

def require(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"Required frozen output is missing:\n{path}")
    return path


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(require(path))


def read_json(path: Path) -> Mapping:
    with require(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: Path, payload: Mapping) -> None:
    def default(x):
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, Path):
            return str(x)
        raise TypeError(type(x).__name__)

    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=default)


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def save_figure(fig: plt.Figure, outdir: Path, stem: str) -> Dict[str, str]:
    paths = {}
    for ext in ("png", "pdf", "svg"):
        path = outdir / f"{stem}.{ext}"
        if ext == "png":
            fig.savefig(path, dpi=PNG_DPI)
        else:
            fig.savefig(path)
        paths[ext] = str(path)
    return paths


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.12, 1.06, label,
        transform=ax.transAxes,
        fontsize=13, fontweight="bold",
        va="top", ha="left",
    )


def clean_axes(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def compact_signature(signature: str) -> str:
    """Short but unambiguous labels for crowded publication legends."""
    parts = str(signature).split("|")
    repl = {
        "YEOH2": "Y2",
        "GENT": "G",
        "OGDEN1": "O1",
        "OGDEN2": "O2",
        "GP2": "GP2",
        "MR": "MR",
        "NH": "NH",
    }
    return "·".join(repl.get(p, p) for p in parts if p)


def full_signature_from_short_note() -> str:
    return "Y2=YEOH2; G=GENT; O1=OGDEN1; O2=OGDEN2"


def merge_certified_labels(
    coords: pd.DataFrame,
    certified: pd.DataFrame,
) -> pd.DataFrame:
    required_coords = {"source_model", "source_index", "PC1", "PC2", "PC3"}
    missing = required_coords.difference(coords.columns)
    if missing:
        raise RuntimeError(f"PCA coordinate file missing columns: {sorted(missing)}")

    label_col = "certified_compatibility_set_at_3pct"
    if label_col not in certified.columns:
        raise RuntimeError(f"Certified atlas missing {label_col}")

    labels = certified[
        ["source_model", "source_index", label_col]
    ].copy()
    work = coords.merge(
        labels,
        on=["source_model", "source_index"],
        how="left",
        validate="one_to_one",
    )
    if work[label_col].isna().any():
        bad = work[work[label_col].isna()][["source_model", "source_index"]].head()
        raise RuntimeError(
            "Could not align all PCA states to final certified labels. "
            f"Examples:\n{bad}"
        )
    work["certified_label"] = work[label_col].astype(str)
    return work


def pca_metadata(root: Path, view: str) -> Tuple[np.ndarray, int]:
    npz_path = root / f"compatibility_regions_pca_{view}_model.npz"
    if npz_path.is_file():
        with np.load(npz_path, allow_pickle=True) as z:
            ratio = np.asarray(z["explained_variance_ratio"], dtype=float)
            if "selected_component_indices" in z:
                dimension = int(len(z["selected_component_indices"]))
            else:
                dimension = -1
        return ratio, dimension

    summary_path = root / "compatibility_regions_pca_protocol_summary.csv"
    summary = read_csv(summary_path)
    row = summary[summary.protocol_view.astype(str) == view]
    if len(row) != 1:
        raise RuntimeError(f"Cannot resolve PCA metadata for {view}")
    r = row.iloc[0]
    ratio = np.asarray([
        float(r.pc1_explained_variance),
        float(r.pc2_explained_variance),
        float(r.pc3_explained_variance),
    ])
    return ratio, int(r.response_dimension)


def region_order_and_colors(
    certified: pd.DataFrame,
) -> Tuple[List[str], Dict[str, str], pd.DataFrame]:
    label_col = "certified_compatibility_set_at_3pct"
    counts = (
        certified.groupby(label_col)
        .size()
        .reset_index(name="n_states")
        .sort_values(["n_states", label_col], ascending=[False, True])
        .reset_index(drop=True)
    )
    total = int(counts.n_states.sum())
    counts["fraction"] = counts.n_states / total
    order = counts[label_col].astype(str).tolist()
    colors = {
        lab: REGION_PALETTE[i % len(REGION_PALETTE)]
        for i, lab in enumerate(order)
    }
    counts["short_label"] = [compact_signature(x) for x in order]
    counts["color_hex"] = [colors[x] for x in order]
    return order, colors, counts


def get_pca_view(
    root: Path,
    certified: pd.DataFrame,
    view: str,
) -> Tuple[pd.DataFrame, np.ndarray, int]:
    coords = read_csv(root / f"compatibility_regions_pca_{view}_coordinates.csv")
    work = merge_certified_labels(coords, certified)
    ratio, dim = pca_metadata(root, view)
    return work, ratio, dim


def weighted_threshold_persistence(
    certified: pd.DataFrame,
    cell2: Path,
) -> pd.DataFrame:
    rows = []
    can_compute = True
    for threshold in THRESHOLDS_PERCENT:
        if threshold != 3 and f"signature_at_{threshold}pct" not in certified.columns:
            can_compute = False
            break

    if can_compute:
        for threshold in THRESHOLDS_PERCENT:
            if threshold == 3:
                sig = certified["certified_compatibility_set_at_3pct"].fillna("").astype(str)
            else:
                sig = certified[f"signature_at_{threshold}pct"].fillna("").astype(str)
            card = sig.map(
                lambda x: 0 if not x else len([p for p in x.split("|") if p])
            )
            rows.append({
                "threshold_percent": threshold,
                "mean_compatible_models": float(card.mean()),
                "median_compatible_models": float(card.median()),
                "n_states": int(len(card)),
            })
        return pd.DataFrame(rows)

    # Exact fallback to Cell-2 saved per-family persistence, weighted by n_states.
    persistence = read_csv(cell2 / "threshold_persistence_summary.csv")
    for threshold, g in persistence.groupby("threshold_percent"):
        w = g.n_states.to_numpy(float)
        rows.append({
            "threshold_percent": float(threshold),
            "mean_compatible_models": float(
                np.average(g.mean_compatible_models.to_numpy(float), weights=w)
            ),
            "median_compatible_models": float("nan"),
            "n_states": int(w.sum()),
        })
    return pd.DataFrame(rows).sort_values("threshold_percent").reset_index(drop=True)


def recompute_visibility(
    work: pd.DataFrame,
    view: str,
) -> Dict[str, object]:
    """
    Reproduce Cell-2's fixed-label three-PC visibility diagnostic.
    """
    work = work.copy()
    work["label"] = work["certified_label"].astype(str)

    # Match Cell-2: remove exact duplicate scale-free PCA points, especially NH
    # stiffness replicas.
    work = work.drop_duplicates(
        ["source_model", "PC1", "PC2", "PC3", "label"]
    ).reset_index(drop=True)

    counts = work.label.value_counts()
    rare_labels = sorted(counts[counts < PCA_RARE_MIN_COUNT].index.astype(str))
    evaluation = work[~work.label.isin(rare_labels)].copy()
    coverage = float(len(evaluation) / max(len(work), 1))
    if len(evaluation) == 0:
        raise RuntimeError(f"{view}: no states remain for visibility evaluation.")

    X = evaluation[["PC1", "PC2", "PC3"]].to_numpy(float)
    X = StandardScaler().fit_transform(X)
    y = evaluation.label.to_numpy(str)
    min_count = int(pd.Series(y).value_counts().min())
    n_splits = min(PCA_CV_SPLITS, min_count)
    if n_splits < 2:
        raise RuntimeError(f"{view}: insufficient label counts for repeated CV.")

    splitter = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=PCA_CV_REPEATS,
        random_state=PCA_RANDOM_SEED,
    )
    ba = []
    f1 = []
    for train, test in splitter.split(X, y):
        k = min(PCA_K, len(train))
        clf = KNeighborsClassifier(n_neighbors=k, weights="distance")
        clf.fit(X[train], y[train])
        pred = clf.predict(X[test])
        ba.append(balanced_accuracy_score(y[test], pred))
        f1.append(f1_score(y[test], pred, average="macro", zero_division=0))

    return {
        "protocol_view": view,
        "n_unique_scale_free_states": int(len(work)),
        "n_evaluated_states": int(len(evaluation)),
        "evaluation_coverage": coverage,
        "n_evaluation_labels": int(len(np.unique(y))),
        "balanced_accuracy_mean": float(np.mean(ba)),
        "balanced_accuracy_std": float(np.std(ba, ddof=1)),
        "macro_f1_mean": float(np.mean(f1)),
        "macro_f1_std": float(np.std(f1, ddof=1)),
        "rare_labels_excluded_from_cv": "|".join(rare_labels),
    }


# =============================================================================
# Figure 3
# =============================================================================

def make_figure3(
    root: Path,
    cell2: Path,
    outdir: Path,
    certified: pd.DataFrame,
    region_order: Sequence[str],
    region_colors: Mapping[str, str],
    region_counts: pd.DataFrame,
) -> Tuple[Dict[str, str], Mapping]:
    work, ratio, _ = get_pca_view(root, certified, "ALL")

    fig, axes = plt.subplots(
        1, 2,
        figsize=FIG_WIDE,
        gridspec_kw={"width_ratios": [1.28, 1.0]},
    )
    ax = axes[0]
    for label in region_order:
        g = work[work.certified_label == label]
        ax.scatter(
            g.PC1, g.PC2,
            s=11, alpha=0.64,
            color=region_colors[label],
            edgecolors="none",
            rasterized=True,
        )
    ax.set_xlabel(f"PC1 ({100*ratio[0]:.2f}%)")
    ax.set_ylabel(f"PC2 ({100*ratio[1]:.2f}%)")
    ax.set_title("Scale-free complete-deformation PCA")
    ax.text(
        0.02, 0.02,
        f"PC1–PC2: {100*np.sum(ratio[:2]):.2f}% variance\n"
        "Compatibility evaluated in 5,202-D",
        transform=ax.transAxes,
        ha="left", va="bottom", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.8", alpha=0.9),
    )
    clean_axes(ax)
    panel_label(ax, "A")

    ax = axes[1]
    counts = region_counts.copy().iloc[::-1]
    y = np.arange(len(counts))
    labels = counts.short_label.astype(str).tolist()
    colors = counts.color_hex.tolist()
    bars = ax.barh(y, counts.n_states, color=colors, alpha=0.90)
    ax.set_yticks(y, labels)
    ax.set_xlabel("Certified source states")
    ax.set_title("Compatibility-set frequency")
    xmax = float(counts.n_states.max())
    ax.set_xlim(0, xmax * 1.22)
    for bar, n, frac in zip(bars, counts.n_states, counts.fraction):
        ax.text(
            bar.get_width() + 0.015*xmax,
            bar.get_y() + bar.get_height()/2,
            f"{int(n)}  ({100*float(frac):.1f}%)",
            va="center", ha="left", fontsize=7.2,
        )
    clean_axes(ax)
    panel_label(ax, "B")
    ax.text(
        0.99, -0.11, full_signature_from_short_note(),
        transform=ax.transAxes, ha="right", va="top", fontsize=7.0,
    )

    fig.suptitle(
        "Final certified constitutive-response atlas",
        fontsize=12.0, fontweight="bold", y=0.995,
    )
    fig.tight_layout(rect=(0, 0.02, 1, 0.965))

    paths = save_figure(fig, outdir, "Figure_3_Final_Certified_Atlas")
    plt.close(fig)

    summary = {
        "n_source_states": int(len(certified)),
        "n_certified_regions": int(len(region_order)),
        "pc1_explained_variance": float(ratio[0]),
        "pc2_explained_variance": float(ratio[1]),
        "pc3_explained_variance": float(ratio[2]) if len(ratio) > 2 else None,
        "pc1_pc2_cumulative": float(np.sum(ratio[:2])),
        "pc1_pc2_pc3_cumulative": float(np.sum(ratio[:3])),
    }
    return paths, summary


# =============================================================================
# Figure 4
# =============================================================================

def heldout_by_regime(cell2: Path) -> pd.DataFrame:
    checkpoint = cell2 / "heldout_validation_checkpoint.csv"
    if checkpoint.is_file():
        df = pd.read_csv(checkpoint)
        required = {"noise_regime", "source_model_included"}
        if required.issubset(df.columns):
            return (
                df.groupby("noise_regime")
                .agg(
                    n=("source_model_included", "size"),
                    inclusion=("source_model_included", "mean"),
                )
                .reset_index()
            )

    summary = read_csv(cell2 / "heldout_validation_summary.csv")
    # Exact weighted aggregation from model x regime groups.
    rows = []
    for regime, g in summary.groupby("noise_regime"):
        n = g.n.to_numpy(float)
        rows.append({
            "noise_regime": regime,
            "n": int(n.sum()),
            "inclusion": float(np.average(g.source_model_inclusion, weights=n)),
        })
    return pd.DataFrame(rows)


def convergence_metrics(cell2: Path) -> Tuple[pd.DataFrame, Mapping]:
    conv = read_csv(cell2 / "complete_F_resolution_convergence.csv")
    pivot = conv.pivot(
        index="case_key",
        columns="resolution",
        values="fixed_witness_score",
    )
    resolutions = sorted(int(x) for x in pivot.columns)
    highest = max(resolutions)
    baseline = 2601 if 2601 in resolutions else resolutions[-2]
    delta = (pivot[highest] - pivot[baseline]).abs().dropna()

    cases = conv.drop_duplicates("case_key").set_index("case_key")
    if "reference_optimized_score" in cases.columns:
        ref = cases["reference_optimized_score"].astype(float)
        common = delta.index.intersection(ref.index)
        nonboundary = (ref.loc[common] - 0.03).abs() >= 0.003
        high_label = pivot.loc[common, highest] <= 0.03
        base_label = pivot.loc[common, baseline] <= 0.03
        if nonboundary.any():
            agreement = float(
                (high_label[nonboundary] == base_label[nonboundary]).mean()
            )
        else:
            agreement = float("nan")
    else:
        agreement = float("nan")

    metrics = {
        "n_cases": int(len(delta)),
        "baseline_resolution": int(baseline),
        "highest_resolution": int(highest),
        "median_absolute_score_change": float(delta.median()),
        "p95_absolute_score_change": float(delta.quantile(0.95)),
        "maximum_absolute_score_change": float(delta.max()),
        "nonboundary_label_agreement": agreement,
    }
    return delta.reset_index(name="absolute_score_change"), metrics


def identity_metrics(root: Path) -> Mapping[str, float]:
    exact = read_csv(root / "exact_hierarchy_response_identity_audit.csv")
    parent = read_csv(root / "complete_F_parent_subprotocol_consistency_audit.csv")
    return {
        "hierarchy_max_abs_error": float(exact.max_abs_error.astype(float).max()),
        "parent_subprotocol_max_relative_error": float(
            parent.relative_magnitude_error.astype(float).max()
        ),
    }


def none_by_generator(cell2: Path) -> pd.DataFrame:
    summary_path = cell2 / "stratified_none_revalidation_summary.csv"
    if summary_path.is_file():
        return pd.read_csv(summary_path)
    check = read_csv(cell2 / "stratified_none_revalidation_checkpoint.csv")
    return (
        check.groupby("generator")
        .agg(
            n=("audit_row_index", "size"),
            none_retention=("revalidated_none", "mean"),
        )
        .reset_index()
    )


def make_figure4(
    root: Path,
    cell2: Path,
    outdir: Path,
    certified: pd.DataFrame,
) -> Tuple[Dict[str, str], Mapping]:
    held = heldout_by_regime(cell2)
    boundary = read_csv(cell2 / "boundary_certification_summary.csv")
    persistence = weighted_threshold_persistence(certified, cell2)
    delta_df, conv_metrics = convergence_metrics(cell2)
    ident = identity_metrics(root)
    none = none_by_generator(cell2)

    fig, axes = plt.subplots(2, 3, figsize=FIG_GRID)
    axes = axes.ravel()

    # A — held-out recovery
    ax = axes[0]
    regime_order = [
        "CLEAN",
        "BOUNDED_INTERIOR_2P7",
        "GAUSSIAN_CLIPPED_3PCT",
        "CORRELATED_CLIPPED_3PCT",
    ]
    held = held.set_index("noise_regime").reindex(
        [r for r in regime_order if r in held.noise_regime.values]
    ).reset_index()
    short = {
        "CLEAN": "Clean",
        "BOUNDED_INTERIOR_2P7": "2.7% bounded",
        "GAUSSIAN_CLIPPED_3PCT": "3% clipped\nGaussian",
        "CORRELATED_CLIPPED_3PCT": "3% clipped\ncorrelated",
    }
    hard_colors = []
    for r in held.noise_regime:
        hard_colors.append("#0072B2" if r in HARD_HELDOUT_REGIMES else "#BDBDBD")
    bars = ax.bar(
        np.arange(len(held)),
        100*held.inclusion.to_numpy(float),
        color=hard_colors,
    )
    ax.axhline(100*HELDOUT_GATE, color="0.25", linestyle="--", linewidth=1.0)
    ax.set_ylim(0, 104)
    ax.set_ylabel("Source-family recovery (%)")
    ax.set_xticks(
        np.arange(len(held)),
        [short.get(x, x) for x in held.noise_regime],
    )
    for bar, value, n in zip(bars, held.inclusion, held.n):
        ax.text(
            bar.get_x()+bar.get_width()/2,
            min(102.3, 100*float(value)+1.2),
            f"{100*float(value):.1f}%\n(n={int(n)})",
            ha="center", va="bottom", fontsize=7,
        )
    ax.set_title("Independent held-out recovery")
    clean_axes(ax)
    panel_label(ax, "A")

    # B — dense boundary certification
    ax = axes[1]
    x = 100*boundary.baseline_score.to_numpy(float)
    y = 100*boundary.certified_score.to_numpy(float)
    new_compat = (
        (~boundary.baseline_compatible.astype(bool))
        & boundary.certified_compatible.astype(bool)
    )
    ax.scatter(
        x[~new_compat], y[~new_compat],
        s=12, alpha=0.45, color="#7F7F7F", edgecolors="none",
        rasterized=True,
    )
    if new_compat.any():
        ax.scatter(
            x[new_compat], y[new_compat],
            s=20, alpha=0.90, color="#009E73", edgecolors="none",
            label="New ≤3% witness",
            rasterized=True,
        )
    lo = min(float(np.nanmin(x)), float(np.nanmin(y)), 1.8)
    hi = max(float(np.nanmax(x)), float(np.nanmax(y)), 4.2)
    ax.plot([lo, hi], [lo, hi], color="0.25", linestyle=":", linewidth=1.0)
    ax.axvline(3.0, color="#D55E00", linestyle="--", linewidth=1.0)
    ax.axhline(3.0, color="#D55E00", linestyle="--", linewidth=1.0)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Frozen score (%)")
    ax.set_ylabel("Certified score (%)")
    n_improved = int(np.sum(boundary.score_reduction.to_numpy(float) > 1e-10))
    n_new = int(new_compat.sum())
    seed_agree = float(boundary.dense_seed_label_agreement.astype(float).mean())
    ax.text(
        0.03, 0.97,
        f"n={len(boundary)}\nimproved={n_improved}\nnew compatible={n_new}\nseed agreement={100*seed_agree:.1f}%",
        transform=ax.transAxes, ha="left", va="top", fontsize=7.2,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="0.82", alpha=0.9),
    )
    ax.set_title("Dense boundary certification")
    clean_axes(ax)
    panel_label(ax, "B")

    # C — threshold persistence
    ax = axes[2]
    p = persistence.sort_values("threshold_percent")
    ax.plot(
        p.threshold_percent,
        p.mean_compatible_models,
        marker="o", color="#0072B2",
    )
    for x0, y0 in zip(p.threshold_percent, p.mean_compatible_models):
        ax.text(x0, y0+0.035, f"{y0:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(THRESHOLDS_PERCENT)
    ax.set_xlabel("Declared tolerance (%)")
    ax.set_ylabel("Mean compatible families")
    ax.set_title("Threshold persistence")
    ax.set_ylim(bottom=max(1.0, float(p.mean_compatible_models.min())-0.18))
    clean_axes(ax)
    panel_label(ax, "C")

    # D — complete-F resolution convergence
    ax = axes[3]
    d = delta_df.absolute_score_change.to_numpy(float)
    # A compact boxplot plus the individual distribution.  The y-scale stays
    # linear because the publication gate is an absolute score difference.
    rng = np.random.default_rng(20260808)
    jitter = rng.normal(1.0, 0.035, size=len(d))
    ax.scatter(
        jitter, d,
        s=7, alpha=0.18, color="#0072B2", edgecolors="none", rasterized=True,
    )
    ax.boxplot(
        [d], positions=[1], widths=0.20, showfliers=False,
        boxprops=dict(color="0.25"),
        whiskerprops=dict(color="0.25"),
        capprops=dict(color="0.25"),
        medianprops=dict(color="#D55E00", linewidth=1.8),
    )
    ax.axhline(
        RESOLUTION_MEDIAN_GATE,
        color="0.25", linestyle="--", linewidth=1.0,
        label="Median-change gate",
    )
    ax.set_xlim(0.65, 1.35)
    ax.set_xticks([1], [f"{conv_metrics['baseline_resolution']}→{conv_metrics['highest_resolution']}"])
    ax.set_ylabel("|score change|")
    agreement = conv_metrics["nonboundary_label_agreement"]
    agreement_text = (
        "n/a" if not np.isfinite(agreement) else f"{100*agreement:.1f}%"
    )
    ax.text(
        0.03, 0.97,
        f"median={conv_metrics['median_absolute_score_change']:.3g}\n"
        f"p95={conv_metrics['p95_absolute_score_change']:.3g}\n"
        f"max={conv_metrics['maximum_absolute_score_change']:.3g}\n"
        f"non-boundary labels={agreement_text}",
        transform=ax.transAxes, ha="left", va="top", fontsize=7.2,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="0.82", alpha=0.9),
    )
    ax.set_title("Parent-resolution convergence")
    clean_axes(ax)
    panel_label(ax, "D")

    # E — exact hierarchy / parent embedding
    ax = axes[4]
    cats = ["Exact hierarchy", "Parent→UT/BT/SH"]
    actual = np.array([
        ident["hierarchy_max_abs_error"],
        ident["parent_subprotocol_max_relative_error"],
    ], dtype=float)
    gates = np.array([
        HIERARCHY_ERROR_GATE,
        PARENT_SUBPROTOCOL_ERROR_GATE,
    ], dtype=float)
    plot_actual = np.maximum(actual, 1e-17)
    ycat = np.arange(2)
    for yi, a, g in zip(ycat, plot_actual, gates):
        ax.plot([a, g], [yi, yi], color="0.75", linewidth=2.0)
        ax.scatter(a, yi, s=38, color="#0072B2", zorder=3, label="Measured max error" if yi == 0 else None)
        ax.scatter(g, yi, s=45, marker="x", color="#D55E00", zorder=3, label="Acceptance limit" if yi == 0 else None)
    ax.set_xscale("log")
    ax.set_yticks(ycat, cats)
    ax.invert_yaxis()
    ax.set_xlabel("Maximum identity error")
    ax.set_title("Exact identities and embedding")
    ax.legend(frameon=False, loc="lower right")
    for yi, val in zip(ycat, actual):
        ax.text(
            max(float(val), 1e-17)*1.35, yi-0.12,
            f"{val:.2e}", fontsize=7, ha="left", va="center",
        )
    clean_axes(ax)
    panel_label(ax, "E")

    # F — off-manifold NONE retention
    ax = axes[5]
    nframe = none.copy()
    if "none_retention" not in nframe.columns:
        raise RuntimeError("NONE summary does not contain none_retention.")
    bars = ax.bar(
        np.arange(len(nframe)),
        100*nframe.none_retention.to_numpy(float),
        color="#009E73",
    )
    ax.axhline(100*NONE_GATE, color="0.25", linestyle="--", linewidth=1.0)
    ax.set_ylim(0, 104)
    ax.set_ylabel("NONE retained (%)")
    ax.set_xticks(
        np.arange(len(nframe)),
        [str(x).replace("_", "\n") for x in nframe.generator],
    )
    for bar, value, n in zip(bars, nframe.none_retention, nframe.n):
        ax.text(
            bar.get_x()+bar.get_width()/2,
            min(102.3, 100*float(value)+1.2),
            f"{100*float(value):.0f}%\n(n={int(n)})",
            ha="center", va="bottom", fontsize=7,
        )
    ax.set_title("Stratified off-manifold rejection")
    clean_axes(ax)
    panel_label(ax, "F")

    fig.suptitle(
        "Numerical certification and robustness",
        fontsize=12.0, fontweight="bold", y=0.995,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    paths = save_figure(fig, outdir, "Figure_4_Numerical_Certification")
    plt.close(fig)

    # Compact machine-readable summary used to update/check manuscript text.
    metrics = {
        "heldout_by_regime": held.to_dict(orient="records"),
        "hard_validation_n": int(
            held[held.noise_regime.isin(HARD_HELDOUT_REGIMES)].n.sum()
        ),
        "hard_validation_min_regime_inclusion": float(
            held[held.noise_regime.isin(HARD_HELDOUT_REGIMES)].inclusion.min()
        ),
        "boundary_n_candidates": int(len(boundary)),
        "boundary_n_improved": n_improved,
        "boundary_n_new_compatible": n_new,
        "boundary_dense_seed_label_agreement": seed_agree,
        "threshold_persistence": persistence.to_dict(orient="records"),
        "resolution_convergence": conv_metrics,
        "identity": ident,
        "none_by_generator": nframe.to_dict(orient="records"),
        "overall_none_retention": float(
            np.average(nframe.none_retention, weights=nframe.n)
        ),
    }
    return paths, metrics


# =============================================================================
# Figure 5
# =============================================================================

def make_figure5(
    root: Path,
    cell2: Path,
    outdir: Path,
    certified: pd.DataFrame,
    region_order: Sequence[str],
    region_colors: Mapping[str, str],
) -> Tuple[Dict[str, str], pd.DataFrame, Mapping]:
    views: Dict[str, pd.DataFrame] = {}
    ratios: Dict[str, np.ndarray] = {}
    dims: Dict[str, int] = {}
    visibility_rows = []

    for view in PCA_VIEWS:
        work, ratio, dim = get_pca_view(root, certified, view)
        views[view] = work
        ratios[view] = ratio
        dims[view] = dim
        visibility_rows.append(recompute_visibility(work, view))

    visibility = pd.DataFrame(visibility_rows)
    visibility.to_csv(outdir / "figure5_visibility_recomputed.csv", index=False)

    # Optional consistency check against Cell-2's saved diagnostic.
    saved_path = cell2 / "subprotocol_complete_F_label_visibility.csv"
    visibility_check = {}
    if saved_path.is_file():
        saved = pd.read_csv(saved_path)
        saved = saved[saved.protocol_view.astype(str).isin(PCA_VIEWS)].copy()
        merged = visibility.merge(
            saved[[
                "protocol_view",
                "balanced_accuracy_mean",
                "macro_f1_mean",
            ]],
            on="protocol_view",
            suffixes=("_recomputed", "_saved"),
            how="inner",
        )
        if len(merged):
            ba_diff = np.max(np.abs(
                merged.balanced_accuracy_mean_recomputed
                - merged.balanced_accuracy_mean_saved
            ))
            f1_diff = np.max(np.abs(
                merged.macro_f1_mean_recomputed
                - merged.macro_f1_mean_saved
            ))
            visibility_check = {
                "max_abs_balanced_accuracy_difference_vs_cell2": float(ba_diff),
                "max_abs_macro_f1_difference_vs_cell2": float(f1_diff),
            }
            if ba_diff > 5e-10 or f1_diff > 5e-10:
                print(
                    "[Figures 3–5] WARNING: independently recomputed Figure-5 "
                    f"visibility differs from saved Cell-2 values: BA={ba_diff:.3e}, "
                    f"F1={f1_diff:.3e}"
                )

    fig, axes = plt.subplots(2, 3, figsize=FIG_GRID)
    flat = axes.ravel()

    # A–D: PCA views.
    for panel_i, view in enumerate(PCA_VIEWS):
        ax = flat[panel_i]
        work = views[view]
        ratio = ratios[view]
        for label in region_order:
            g = work[work.certified_label == label]
            if len(g) == 0:
                continue
            ax.scatter(
                g.PC1, g.PC2,
                s=7.5, alpha=0.52,
                color=region_colors[label],
                edgecolors="none",
                rasterized=True,
            )
        ax.set_xlabel(f"PC1 ({100*ratio[0]:.1f}%)")
        ax.set_ylabel(f"PC2 ({100*ratio[1]:.1f}%)")
        dim_text = f"d={dims[view]:,}" if dims[view] > 0 else "d=?"
        ax.set_title(
            f"{view}: {dim_text}; PC1+PC2={100*np.sum(ratio[:2]):.1f}%"
        )
        clean_axes(ax)
        panel_label(ax, chr(ord("A")+panel_i))

    # E: dimensions
    ax = flat[4]
    dim_values = np.asarray([dims[v] for v in PCA_VIEWS], dtype=float)
    bars = ax.bar(
        np.arange(4),
        dim_values,
        color=[PROTOCOL_COLORS[v] for v in PCA_VIEWS],
    )
    ax.set_yscale("log")
    ax.set_xticks(np.arange(4), PCA_VIEWS)
    ax.set_ylabel("Response dimension (log scale)")
    ax.set_title("Available response dimension")
    for bar, val in zip(bars, dim_values):
        ax.text(
            bar.get_x()+bar.get_width()/2,
            val*1.12,
            f"{int(val):,}",
            ha="center", va="bottom", fontsize=7.4,
        )
    clean_axes(ax)
    panel_label(ax, "E")

    # F: 3-PC label visibility
    ax = flat[5]
    v = visibility.set_index("protocol_view").loc[PCA_VIEWS].reset_index()
    x = np.arange(4)
    width = 0.36
    ax.bar(
        x-width/2,
        v.balanced_accuracy_mean,
        width,
        yerr=v.balanced_accuracy_std,
        capsize=2.0,
        label="Balanced accuracy",
        color="#0072B2",
        alpha=0.88,
    )
    ax.bar(
        x+width/2,
        v.macro_f1_mean,
        width,
        yerr=v.macro_f1_std,
        capsize=2.0,
        label="Macro-F1",
        color="#D55E00",
        alpha=0.88,
    )
    ax.set_xticks(x, PCA_VIEWS)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Repeated-CV score")
    ax.set_title("Visibility of fixed complete-F labels")
    ax.legend(frameon=False, loc="lower left")
    for xi, ba, mf in zip(x, v.balanced_accuracy_mean, v.macro_f1_mean):
        ax.text(xi-width/2, ba+0.025, f"{ba:.3f}", ha="center", va="bottom", fontsize=6.8)
        ax.text(xi+width/2, mf+0.025, f"{mf:.3f}", ha="center", va="bottom", fontsize=6.8)
    clean_axes(ax)
    panel_label(ax, "F")

    # Single compact region legend shared by A–D.
    handles = [
        Line2D(
            [0], [0], marker="o", linestyle="",
            markersize=5.0,
            markerfacecolor=region_colors[label],
            markeredgecolor="none",
            label=compact_signature(label),
        )
        for label in region_order
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=7,
        frameon=False,
        fontsize=6.4,
        bbox_to_anchor=(0.5, -0.005),
        columnspacing=1.1,
        handletextpad=0.25,
    )
    fig.text(
        0.995, 0.012, full_signature_from_short_note(),
        ha="right", va="bottom", fontsize=6.4,
    )
    fig.suptitle(
        "Information loss under standard loading protocols",
        fontsize=12.0, fontweight="bold", y=0.995,
    )
    fig.tight_layout(rect=(0, 0.075, 1, 0.965))
    paths = save_figure(fig, outdir, "Figure_5_Protocol_Information_Loss")
    plt.close(fig)

    metrics = {
        "pca_dimensions": {k: int(v) for k, v in dims.items()},
        "pca_explained_variance": {
            k: {
                "pc1": float(ratios[k][0]),
                "pc2": float(ratios[k][1]),
                "pc3": float(ratios[k][2]) if len(ratios[k]) > 2 else None,
                "pc1_pc2": float(np.sum(ratios[k][:2])),
                "pc1_pc2_pc3": float(np.sum(ratios[k][:3])),
            }
            for k in PCA_VIEWS
        },
        "visibility": visibility.to_dict(orient="records"),
        "visibility_consistency_vs_saved_cell2": visibility_check,
    }
    return paths, visibility, metrics


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    root = resolve_root()
    cell2 = root / CELL2_SUBDIR
    outdir = root / FIG_SUBDIR
    outdir.mkdir(parents=True, exist_ok=True)

    print("=" * 108)
    print("V68 PUBLICATION FIGURES 3–5 — STANDALONE FINAL-CELL GENERATOR")
    print("=" * 108)
    print(f"Frozen V68 root: {root}")
    print(f"Cell-2 results:   {cell2}")
    print(f"Output:           {outdir}")

    certified = read_csv(cell2 / "certified_source_atlas_states.csv")

    required_cert_cols = {
        "source_model",
        "source_index",
        "certified_compatibility_set_at_3pct",
    }
    missing = required_cert_cols.difference(certified.columns)
    if missing:
        raise RuntimeError(
            "Certified atlas is missing required columns: " + ", ".join(sorted(missing))
        )

    # Basic frozen-result sanity checks.
    keys = certified[["source_model", "source_index"]].astype(str).agg("|".join, axis=1)
    if keys.duplicated().any():
        raise RuntimeError("Certified atlas contains duplicate source_model/source_index keys.")

    region_order, region_colors, region_counts = region_order_and_colors(certified)
    region_counts.to_csv(outdir / "figure3_region_summary.csv", index=False)
    pd.DataFrame({
        "certified_compatibility_set": region_order,
        "short_label": [compact_signature(x) for x in region_order],
        "color_hex": [region_colors[x] for x in region_order],
    }).to_csv(outdir / "region_color_key.csv", index=False)

    print(
        f"[Figure 3] Solving final label/PCA merge for {len(certified)} source states "
        f"and {len(region_order)} certified compatibility sets..."
    )
    fig3_paths, fig3_metrics = make_figure3(
        root, cell2, outdir, certified,
        region_order, region_colors, region_counts,
    )
    print("[Figure 3] complete.")

    print("[Figure 4] Recomputing publication certification summaries from frozen raw outputs...")
    fig4_paths, fig4_metrics = make_figure4(
        root, cell2, outdir, certified,
    )
    save_json(outdir / "figure4_publication_metrics.json", fig4_metrics)
    print("[Figure 4] complete.")

    print("[Figure 5] Recomputing fixed-label 3-PC visibility and generating protocol views...")
    fig5_paths, visibility, fig5_metrics = make_figure5(
        root, cell2, outdir, certified,
        region_order, region_colors,
    )
    print("[Figure 5] complete.")

    manifest = {
        "root": str(root),
        "output_directory": str(outdir),
        "figure3": fig3_paths,
        "figure4": fig4_paths,
        "figure5": fig5_paths,
        "figure3_metrics": fig3_metrics,
        "figure4_metrics": fig4_metrics,
        "figure5_metrics": fig5_metrics,
    }
    save_json(outdir / "publication_figures_manifest.json", manifest)

    # File integrity manifest.
    file_rows = []
    for p in sorted(outdir.iterdir()):
        if p.is_file():
            file_rows.append({
                "name": p.name,
                "bytes": int(p.stat().st_size),
                "sha256": sha256(p),
            })
    pd.DataFrame(file_rows).to_csv(
        outdir / "publication_figures_file_manifest.csv", index=False
    )

    print("=" * 108)
    print("DONE — publication Figures 3, 4, and 5 generated.")
    print(f"Figure 3 PNG: {fig3_paths['png']}")
    print(f"Figure 4 PNG: {fig4_paths['png']}")
    print(f"Figure 5 PNG: {fig5_paths['png']}")
    print(f"Metrics:      {outdir / 'publication_figures_manifest.json'}")
    print("=" * 108)


if __name__ == "__main__":
    main()


V68 PUBLICATION FIGURES 3–5 — STANDALONE FINAL-CELL GENERATOR
Frozen V68 root: /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
Cell-2 results:   /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/cell2_publication_validation
Output:           /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/publication_figures_3_5
[Figure 3] Solving final label/PCA merge for 2631 source states and 14 certified compatibility sets...
[Figure 3] complete.
[Figure 4] Recomputing publication certification summaries from frozen raw outputs...
[Figure 4] complete.
[Figure 5] Recomputing fixed-label 3-PC visibility and generating protocol views...
[Figure 5] complete.
DONE — publication Figures 3, 4, and 5 generated.
Figure 3 PNG: /content/v68_drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocol

In [ ]:
#!/usr/bin/env python3
"""
V68.5.5 — TRELOAR PUBLIC EXPERIMENTAL VALIDATION — LEAKAGE-FREE FROZEN-ATLAS LOOKUP
================================================================

Purpose
-------
Validate the practical discrimination principle of the frozen V68
complete-deformation constitutive atlas using the classical Treloar vulcanized-
rubber dataset.

Experimental design
-------------------
The validation is deliberately asymmetric and blinded:

    TRAINING / CALIBRATION:
        Treloar uniaxial tension only.

    COMPLETELY WITHHELD:
        Treloar pure shear / planar tension,
        Treloar equibiaxial tension.

Only data whose principal stretches lie inside the frozen V68 complete-F parent
domain are retained:

        0.5 <= lambda_i <= 2.0.

For the public Steinmann reproduction this retains:
    - uniaxial:      lambda <= 2.0,
    - planar/pure:   lambda <= 2.0 and 1/lambda >= 0.5,
    - equibiaxial:   lambda <= 2.0 and 1/lambda^2 >= 0.5.

Workflow
--------
1. Download the three public machine-readable Treloar CSV files.
2. Convert their first-Piola stress P11 to the pressure-free Kirchhoff/Cauchy
   stress difference used by V68:

       tau11 - tau33 = lambda * P11,

   because J=1 and the thickness/free transverse direction has zero traction.

3. Fit all seven V68 model families using UNAXIAL DATA ONLY.
4. Determine experimentally plausible fitted families from training data only:
       primary gate: Delta BIC <= 10,
       sensitivity:  Delta BIC <= 2 and <= 6.
5. Among pairs for which BOTH models are plausible, find hidden-divergence pairs:

       fixed fitted UT pair distance <= 3%
       fixed fitted complete-F distance > 3%.

6. Freeze the COMPLETE SET of training-plausible hidden-divergence pairs
   before looking at either withheld experiment. A primary pair is retained
   only as a compact headline example, but every qualifying pair is audited
   and reported to eliminate pair-selection/cherry-picking concerns.
7. For every qualifying pair, identify from predictions only:
       (a) the global complete-F deformation witness;
       (b) the most discriminating actually measured withheld point across
           planar and equibiaxial experiments.
8. Only then reveal the withheld experimental stresses.
9. Quantify for EVERY qualifying pair whether:
       - the preselected witness favors the same model as the complete withheld
         dataset;
       - predicted pair separation correlates with empirical discrimination;
       - the preselected measured witness lies near the top of the empirical
         discrimination ranking.
10. Report sensitivity of the hidden-divergence conclusion to Delta-BIC
    plausibility cutoffs 2, 6, and 10, and report training-vs-holdout rankings
    for all seven families.

Important scope
---------------
The Treloar CSVs in thermalCANN are a machine-readable reproduction of the
Treloar data as published/reproduced by Steinmann, Hossain & Possart. They are
not a modern raw-data deposition.

The repository CSV column 'pk1' is interpreted as first Piola-Kirchhoff stress
in MPa. It is converted to kPa before V68 fitting (1 MPa = 1000 kPa).

Treloar "pure shear" is the classical planar-tension / pure-shear deformation
with principal stretches:

       (lambda, 1, 1/lambda),

and is NOT V68's simple-shear protocol.

Dependency
----------
The exact frozen V68.1 constitutive implementation is bundled inside this file.
No Cell-2 CSV and no sampled atlas file are required.

Google Drive is used only for saving outputs.

Frozen-atlas validation contract
--------------------------------
This final version DOES use the precomputed frozen V68 atlas.

The experiment-specific constitutive parameters are still fitted once from the
Treloar uniaxial training data, because the material parameters are unknown.
After that calibration:

    1. The primary MR and OGDEN1 fitted states are projected onto SAME-FAMILY
       source nodes of the frozen certified V68 atlas.
    2. The atlas node's certified 3% compatibility label and certified directed
       critical-noise distances are READ from the frozen Cell-2 atlas.
    3. The MR-source -> OGDEN1 target witness is reconstructed from the FROZEN
       pairwise atlas witness stored by V68/Cell-2, not re-optimized.
    4. That frozen atlas witness selects the most discriminating deformation
       available in the completely withheld planar/equibiaxial measurements.
    5. Only then are the held-out Treloar stresses used for validation.

Thus the final primary claim is an actual FROZEN-ATLAS LOOKUP validation.
The direct continuous MR--OGDEN1 fitted-state comparison is retained only as a
diagnostic cross-check and is not presented as the atlas result.

Standalone / persistence contract
---------------------------------
This file is fully standalone for Google Colab:

    - Google Drive availability is established as the first runtime action. An existing valid mount is reused; otherwise Drive is mounted safely.
    - No external V68 source file is required. The exact frozen V68.1 source is
      embedded in this file and checksum-locked.
    - A fixed TRELOAR_VALIDATION_OUTPUT directory is created directly under the exact existing V68 project root before the V68
      dependency is materialized and before any public data are downloaded.
    - A write/read-back persistence test must pass before computation starts.
    - Every CSV/JSON/PNG/PDF output is immediately verified on Drive.
    - A SHA-256 artifact manifest is written at completion.
    - If a run fails after Drive setup, the traceback is saved as _RUN_ERROR.txt.
    - _RUN_COMPLETE.txt is created only after a successful run.

"""

from __future__ import annotations

import hashlib
import base64
import gzip
import importlib.util
import json
import math
import os
import sys
import urllib.request
import traceback
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize_scalar
from scipy.stats import spearmanr

# =============================================================================
# Configuration
# =============================================================================

PROJECT_FOLDER_NAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)

# Paths are intentionally unresolved until Google Drive is force-mounted and
# a real persistence test has passed.
DRIVE_ROOT = None
ROOT = None
OUTPUT_BASE = None
OUTDIR = None

VISIBLE_OUTPUT_FOLDER_NAME = "TRELOAR_VALIDATION_OUTPUT"
VISIBLE_ROOT_MARKER_NAME = "TRELOAR_VALIDATION_OUTPUT_LOCATION.txt"

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
ATLAS_TOLERANCE_FRACTION = 0.03
PAIR_TOLERANCE_FRACTION = ATLAS_TOLERANCE_FRACTION
RNG_SEED = 20260809

TRELOAR_URLS = {
    "uniaxial": (
        "https://raw.githubusercontent.com/llamm-de/thermalCANN/main/"
        "data/Treloar/uniaxial_steinmann.csv"
    ),
    "pure_shear_planar": (
        "https://raw.githubusercontent.com/llamm-de/thermalCANN/main/"
        "data/Treloar/pure_shear_steinmann.csv"
    ),
    "equibiaxial": (
        "https://raw.githubusercontent.com/llamm-de/thermalCANN/main/"
        "data/Treloar/equibiaxial_steinmann.csv"
    ),
}

# The Steinmann-reproduction CSV values are first Piola stress in MPa.
SOURCE_STRESS_TO_KPA = 1000.0

# Model fitting settings.
DE_POPSIZE = 18
DE_MAXITER = 220
DE_TOL = 5.0e-10

# Numerical floor used only in normalized discrimination metrics.
NORMALIZATION_FLOOR_KPA = 1.0e-12

# Training-data plausibility gate.
PRIMARY_DELTA_BIC_MAX = 10.0
DELTA_BIC_SENSITIVITY = (2.0, 6.0, 10.0)

# Manuscript-primary support rule: training data only.
# A model must satisfy BOTH thresholds. With the frozen Treloar UT fit this
# yields exactly MR and OGDEN1.
PRIMARY_PUBLICATION_DELTA_BIC_MAX = 2.0
PRIMARY_PUBLICATION_DELTA_AICC_MAX = 2.0

# Frozen atlas files. The source atlas lives in ROOT; the certified
# reclassification lives in ROOT/cell2_publication_validation.
FROZEN_ATLAS_EXPECTED_SOURCE_ROWS = 2631
FROZEN_ATLAS_EXPECTED_REGION_COUNT = 14
FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES = 47
FROZEN_ATLAS_NUMERICAL_TOL = 3.0e-8

# The primary frozen-atlas source and target are derived at runtime from the
# training-only information-criterion screen. No expected model names and no
# complete-deformation separation criterion are allowed to enter selection.

# Publication provenance metadata. The CSV files themselves contain only
# "stretch" and "pk1"; the MPa interpretation follows the Steinmann et al.
# reproduction/publication convention for Treloar nominal stress P.
TRELOAR_ORIGINAL_DOI = "10.1039/TF9444000059"
STEINMANN_REPRODUCTION_DOI = "10.1007/s00419-012-0610-z"
THERMALCANN_REPOSITORY = "https://github.com/llamm-de/thermalCANN"
STRESS_UNIT_PROVENANCE_NOTE = (
    "thermalCANN CSV header is 'pk1' without an embedded unit. "
    "The analysis interprets pk1 as nominal/first-Piola stress P11 in MPa, "
    "consistent with the Steinmann-Hossain-Possart Treloar reproduction, "
    "whose stress plots/reporting use P in MPa."
)


# =============================================================================
# Drive + V68 import
# =============================================================================

def mount_google_drive_first() -> Path:
    """
    Make Google Drive available as the FIRST runtime operation.

    Startup logic is intentionally safe for both fresh and already-mounted
    Colab sessions:

      1. If a valid MyDrive directory already exists, use it directly.
      2. Otherwise mount at /content/drive if that mountpoint is empty/usable.
      3. If /content/drive is occupied but is not a usable Drive mount, mount
         at a fresh empty dedicated mountpoint.

    Persistent write/read-back verification is performed immediately afterward
    by configure_exact_visible_drive_output().
    """
    print("[V68.5.5] STEP 1/4 — ENSURING GOOGLE DRIVE IS AVAILABLE FIRST...")

    existing_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
    ]
    existing = next(
        (p for p in existing_candidates if p.is_dir()),
        None,
    )

    if existing is not None:
        print(
            "[V68.5.5] Existing Google Drive mount detected; "
            "reusing it instead of remounting a non-empty mountpoint."
        )
        print(f"[V68.5.5] Google Drive available at: {existing}")
        return existing.resolve()

    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "V68.5.1 is intended to run as a standalone Google Colab script. "
            "google.colab could not be imported, so persistent Drive saving "
            "cannot be guaranteed."
        ) from exc

    primary_mountpoint = Path("/content/drive")

    # Colab refuses to mount onto a non-empty ordinary directory. Use the
    # standard mountpoint only when it is absent or empty.
    primary_usable = (
        not primary_mountpoint.exists()
        or (
            primary_mountpoint.is_dir()
            and not any(primary_mountpoint.iterdir())
        )
    )

    if primary_usable:
        mountpoint = primary_mountpoint
    else:
        # The standard mountpoint contains files but no usable MyDrive.
        # Never delete or overwrite it. Mount to a fresh, script-owned path.
        base = Path("/content/v68_google_drive")
        mountpoint = base
        suffix = 0

        while mountpoint.exists() and (
            not mountpoint.is_dir()
            or any(mountpoint.iterdir())
        ):
            suffix += 1
            mountpoint = Path(f"{base}_{suffix:02d}")

        mountpoint.mkdir(parents=True, exist_ok=True)

    print(f"[V68.5.5] Mounting Google Drive at: {mountpoint}")
    drive.mount(
        str(mountpoint),
        force_remount=False,
    )

    candidates = [
        mountpoint / "MyDrive",
        mountpoint / "My Drive",
    ]
    drive_root = next(
        (p for p in candidates if p.is_dir()),
        None,
    )

    if drive_root is None:
        raise RuntimeError(
            "Google Drive mount returned successfully, but no MyDrive "
            f"directory was found beneath {mountpoint}."
        )

    print(f"[V68.5.5] Google Drive mounted at: {drive_root}")
    return drive_root.resolve()


def configure_exact_visible_drive_output(
    drive_root: Path,
) -> Path:
    """
    Save directly into the exact V68 project root visible in Google Drive.

    Expected existing root:
      My Drive/
        Optimal_Protocol/
          V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/

    Output:
      TRELOAR_VALIDATION_OUTPUT/

    A root-level marker file is also written alongside the existing Cell-2 and
    publication folders so it can be seen immediately in the Drive web UI.
    """
    global DRIVE_ROOT, ROOT, OUTPUT_BASE, OUTDIR

    DRIVE_ROOT = Path(drive_root).resolve()
    ROOT = (
        DRIVE_ROOT
        / "Optimal_Protocol"
        / PROJECT_FOLDER_NAME
    )

    if not ROOT.is_dir():
        raise RuntimeError(
            "Expected existing V68 project root was not found on the mounted "
            "Google Drive:\n"
            f"{ROOT}\n\n"
            "The script will not create a different substitute root."
        )

    observed_names = {
        p.name for p in ROOT.iterdir()
    }

    expected_anchors = {
        "cell2_publication_validation",
        "publication_figures_3_5",
    }
    found_anchors = sorted(
        expected_anchors.intersection(
            observed_names
        )
    )

    print("[V68.5.5] Exact existing V68 root:")
    print(f"           {ROOT}")
    print(
        "[V68.5.5] Existing anchor folders detected: "
        + (
            ", ".join(found_anchors)
            if found_anchors
            else "(none)"
        )
    )

    if not found_anchors:
        preview = sorted(observed_names)[:25]
        raise RuntimeError(
            "The mounted path exists but does not look like the V68 directory "
            "shown in Google Drive. Expected at least one anchor folder "
            "('cell2_publication_validation' or 'publication_figures_3_5').\n"
            f"Mounted path: {ROOT}\n"
            f"Observed entries: {preview}\n\n"
            "Refusing to save to the wrong Drive location."
        )

    OUTPUT_BASE = ROOT
    OUTDIR = (
        ROOT
        / VISIBLE_OUTPUT_FOLDER_NAME
    )
    OUTDIR.mkdir(
        parents=False,
        exist_ok=True,
    )

    # Root-level marker visible directly in the exact folder shown in Drive UI.
    root_marker = (
        ROOT
        / VISIBLE_ROOT_MARKER_NAME
    )
    marker_payload = (
        "Treloar validation output location\n"
        f"ROOT={ROOT}\n"
        f"OUTDIR={OUTDIR}\n"
        f"created_at={datetime.now().isoformat()}\n"
    )

    with root_marker.open(
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(marker_payload)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    if not root_marker.is_file():
        raise RuntimeError(
            f"Root visibility marker was not created: {root_marker}"
        )
    if (
        root_marker.read_text(
            encoding="utf-8"
        )
        != marker_payload
    ):
        raise RuntimeError(
            f"Root marker read-back failed: {root_marker}"
        )

    # Output-folder persistence sentinel.
    sentinel = (
        OUTDIR
        / "_DRIVE_PERSISTENCE_TEST.txt"
    )
    sentinel_payload = (
        "V68.5.4 frozen-atlas publication exact-root persistence test\n"
        f"ROOT={ROOT}\n"
        f"OUTDIR={OUTDIR}\n"
    )

    with sentinel.open(
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            sentinel_payload
        )
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    if not sentinel.is_file():
        raise RuntimeError(
            f"Output sentinel was not created: {sentinel}"
        )
    if (
        sentinel.read_text(
            encoding="utf-8"
        )
        != sentinel_payload
    ):
        raise RuntimeError(
            f"Output sentinel read-back failed: {sentinel}"
        )

    # Parent enumeration check.
    visible_names = {
        p.name for p in ROOT.iterdir()
    }
    if (
        VISIBLE_OUTPUT_FOLDER_NAME
        not in visible_names
    ):
        raise RuntimeError(
            "TRELOAR_VALIDATION_OUTPUT was created by path but is not visible "
            "when the exact V68 parent directory is enumerated."
        )
    if (
        VISIBLE_ROOT_MARKER_NAME
        not in visible_names
    ):
        raise RuntimeError(
            "Root marker was written but is not visible when the exact V68 "
            "parent directory is enumerated."
        )

    OUTDIR = OUTDIR.resolve()

    print(
        "[V68.5.5] EXACT V68-ROOT DRIVE OUTPUT VERIFIED."
    )
    print(
        "[V68.5.5] Output folder created DIRECTLY in the visible V68 root:"
    )
    print(f"           {OUTDIR}")
    print(
        "[V68.5.5] Root-level marker created:"
    )
    print(f"           {root_marker}")
    print(
        "[V68.5.5] Parent-directory enumeration check: PASS"
    )

    return OUTDIR


def require_verified_drive_output() -> Path:
    if OUTDIR is None:
        raise RuntimeError(
            "Drive output has not been configured yet."
        )

    outdir = Path(OUTDIR).resolve()
    if not outdir.is_dir():
        raise RuntimeError(
            f"Verified Drive output directory does not exist: {outdir}"
        )
    if not str(outdir).startswith("/content/drive/"):
        raise RuntimeError(
            f"Refusing non-Drive output directory: {outdir}"
        )
    return outdir


def _verify_saved_file(
    path: Path,
    minimum_bytes: int = 1,
) -> Path:
    outdir = require_verified_drive_output()
    path = Path(path).resolve()

    try:
        path.relative_to(outdir)
    except ValueError as exc:
        raise RuntimeError(
            "Refusing to treat a file outside the verified run folder as an "
            f"analysis output.\nOUTDIR={outdir}\npath={path}"
        ) from exc

    if not path.is_file():
        raise RuntimeError(
            f"Expected saved file does not exist: {path}"
        )

    size = int(path.stat().st_size)
    if size < minimum_bytes:
        raise RuntimeError(
            f"Saved file is unexpectedly small ({size} bytes): {path}"
        )

    with path.open("rb") as handle:
        first = handle.read(1)
    if size > 0 and not first:
        raise RuntimeError(
            f"Could not read saved file back from Drive: {path}"
        )

    print(
        f"[V68.5.5][DRIVE SAVED] {path} "
        f"({size / 1024.0:.2f} KiB)"
    )
    return path


def save_dataframe_checked(
    df: pd.DataFrame,
    path: Path,
    *,
    index: bool = False,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return _verify_saved_file(path)


def save_json_checked(
    payload: Mapping[str, object],
    path: Path,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            allow_nan=True,
        )
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    return _verify_saved_file(path)


def save_figure_checked(
    fig,
    path: Path,
    **kwargs,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, **kwargs)
    return _verify_saved_file(path, minimum_bytes=100)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_verified_artifact_manifest(
    outdir: Path,
) -> Path:
    outdir = Path(outdir).resolve()
    rows = []

    for path in sorted(outdir.iterdir(), key=lambda p: p.name.lower()):
        if not path.is_file():
            continue
        if path.name == "_ARTIFACT_MANIFEST_SHA256.csv":
            continue

        rows.append(
            {
                "filename": path.name,
                "absolute_drive_path": str(path.resolve()),
                "size_bytes": int(path.stat().st_size),
                "sha256": sha256_file(path),
            }
        )

    return save_dataframe_checked(
        pd.DataFrame(rows),
        outdir / "_ARTIFACT_MANIFEST_SHA256.csv",
        index=False,
    )


def write_run_complete_marker(
    outdir: Path,
) -> Path:
    outdir = Path(outdir).resolve()
    marker = outdir / "_RUN_COMPLETE.txt"
    payload = (
        "V68.5.4 completed successfully.\n"
        f"OUTDIR={outdir}\n"
        f"completed_at={datetime.now().isoformat()}\n"
    )

    with marker.open("w", encoding="utf-8") as handle:
        handle.write(payload)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    return _verify_saved_file(marker)


def write_run_error_marker(
    exc: BaseException,
) -> None:
    if OUTDIR is None:
        return

    try:
        outdir = require_verified_drive_output()
        path = outdir / "_RUN_ERROR.txt"
        payload = (
            "V68.5.4 failed after Drive setup.\n\n"
            + "".join(
                traceback.format_exception(
                    type(exc),
                    exc,
                    exc.__traceback__,
                )
            )
        )

        with path.open("w", encoding="utf-8") as handle:
            handle.write(payload)
            handle.flush()
            try:
                os.fsync(handle.fileno())
            except OSError:
                pass

        _verify_saved_file(path)
    except Exception as marker_error:
        print(
            "[V68.5.5] WARNING: could not save _RUN_ERROR.txt: "
            f"{marker_error}"
        )


def print_saved_artifact_manifest(
    outdir: Path,
) -> None:
    outdir = Path(outdir).resolve()
    files = sorted(
        [p for p in outdir.iterdir() if p.is_file()],
        key=lambda p: p.name.lower(),
    )

    print("\n" + "=" * 100)
    print("[V68.5.5] VERIFIED GOOGLE DRIVE ARTIFACT MANIFEST")
    print("=" * 100)
    print(f"Directory: {outdir}")

    for path in files:
        print(
            f"  {path.name:<60} "
            f"{path.stat().st_size / 1024.0:10.2f} KiB"
        )




# =============================================================================
# Bundled frozen V68.1 dependency
# =============================================================================

EMBEDDED_V68_MODULE_SHA256 = "6bad33e20705230a1d8c147260c886379b85d607e27b7c07e5a38b8a1e25c9be"
EMBEDDED_V68_MODULE_GZIP_B64 = (
    "..."
)


def _materialize_embedded_v68_module() -> Path:
    """Materialize the exact bundled V68.1 source and verify its checksum."""
    raw = gzip.decompress(
        base64.b64decode("".join(EMBEDDED_V68_MODULE_GZIP_B64))
    )
    observed = hashlib.sha256(raw).hexdigest()
    if observed != EMBEDDED_V68_MODULE_SHA256:
        raise RuntimeError(
            "Embedded V68.1 checksum mismatch: "
            f"{observed} != {EMBEDDED_V68_MODULE_SHA256}"
        )

    filename = (
        "V68_1_patched_practical_standard_protocols_complete_F_"
        "biological_compatibility_atlas.py"
    )
    candidates = [
        Path("/content") / filename,
        Path("/tmp") / filename,
        Path.cwd() / filename,
    ]

    last_error = None
    for target in candidates:
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            if (
                (not target.is_file())
                or hashlib.sha256(target.read_bytes()).hexdigest()
                != EMBEDDED_V68_MODULE_SHA256
            ):
                target.write_bytes(raw)

            final_digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if final_digest != EMBEDDED_V68_MODULE_SHA256:
                raise RuntimeError(
                    f"Materialized V68.1 checksum mismatch at {target}"
                )

            print(
                f"[V68.5.5] Materialized bundled V68.1 temporary dependency at {target} "
                f"(sha256={final_digest[:12]}...)"
            )
            return target.resolve()
        except OSError as exc:
            last_error = exc

    raise FileNotFoundError(
        "Could not materialize bundled V68.1 dependency. "
        f"Last error: {last_error}"
    )

def import_v68_module():
    """
    Import ONLY the checksum-locked V68.1 source bundled in this script.
    No external V68 source file is searched for or required.
    """
    require_verified_drive_output()

    module_path = _materialize_embedded_v68_module()

    imported_digest = hashlib.sha256(
        Path(module_path).read_bytes()
    ).hexdigest()
    if imported_digest != EMBEDDED_V68_MODULE_SHA256:
        raise RuntimeError(
            "Bundled V68.1 checksum changed unexpectedly after materialization."
        )

    module_name = "v68_5_0_bundled_frozen_dependency"
    spec = importlib.util.spec_from_file_location(
        module_name,
        str(module_path),
    )
    if spec is None or spec.loader is None:
        raise ImportError(
            f"Could not create import spec for bundled V68.1 at {module_path}"
        )

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(module_name, None)
        raise

    print(
        "[V68.5.5] Imported checksum-locked BUNDLED V68.1 "
        f"(sha256={imported_digest[:12]}...)"
    )
    print("[V68.5.5] External V68 source dependency: NONE")
    return module





# =============================================================================
# Frozen certified V68 atlas loading and lookup
# =============================================================================

def _atlas_row_key(
    source_model: str,
    source_index: int,
    target_model: str,
) -> str:
    return (
        f"{str(source_model)}::{int(source_index)}::{str(target_model)}"
    )


def _coordinate_json(value: object) -> np.ndarray:
    if isinstance(value, str):
        return np.asarray(
            json.loads(value),
            dtype=np.float64,
        )
    return np.asarray(
        value,
        dtype=np.float64,
    )


def reconstruct_certified_atlas_from_frozen_outputs(
    V,
    baseline_atlas: pd.DataFrame,
    boundary_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Recreate Cell-2 Stage-3 labels exactly from already-frozen baseline atlas
    scores plus already-frozen boundary-certification witnesses.

    This performs NO constitutive optimization and NO atlas regeneration.
    """
    updates = {
        _atlas_row_key(
            row.source_model,
            row.source_index,
            row.target_model,
        ): float(row.certified_score)
        for _, row in boundary_summary.iterrows()
    }

    rows = []
    for _, row in baseline_atlas.iterrows():
        new = dict(row)
        scores = {}

        for model in MODELS:
            original_col = (
                f"critical_noise_to_{model.lower()}"
            )
            if original_col not in row.index:
                raise RuntimeError(
                    f"Frozen baseline atlas lacks {original_col}"
                )

            score = float(row[original_col])
            key = _atlas_row_key(
                str(row.source_model),
                int(row.source_index),
                model,
            )
            if key in updates:
                score = min(
                    score,
                    float(updates[key]),
                )

            scores[model] = score
            new[
                f"certified_critical_noise_to_{model.lower()}"
            ] = score

        compatible = [
            model
            for model in MODELS
            if (
                scores[model]
                <= ATLAS_TOLERANCE_FRACTION
                + FROZEN_ATLAS_NUMERICAL_TOL
            )
        ]
        compatible = list(
            V.hierarchy_closure(compatible)
        )
        region, signature, subtype = (
            V.region_label_from_compatibility(
                compatible
            )
        )
        minimal = V.minimal_compatible_models(
            compatible
        )

        new.update(
            {
                "certified_primary_region_at_3pct":
                    region,
                "certified_compatibility_set_at_3pct":
                    signature,
                "certified_minimal_adequate_models_at_3pct":
                    V.ordered_signature(minimal),
                "certified_collision_subtype_at_3pct":
                    subtype,
                "certified_n_compatible_models_at_3pct":
                    int(len(compatible)),
                "certified_label_changed":
                    bool(
                        signature
                        != str(
                            row[
                                "compatibility_set_at_3pct"
                            ]
                        )
                    ),
            }
        )
        rows.append(new)

    return pd.DataFrame(rows)


def load_and_verify_frozen_atlas_assets(
    V,
) -> Dict[str, object]:
    """
    Load the PRECOMPUTED source atlas and Cell-2 certified atlas from the exact
    V68 Drive root.

    If the canonical certified CSV is absent but the frozen baseline atlas and
    frozen boundary summary exist, reconstruct Cell-2 Stage-3 labels in memory.
    No model is fitted and no atlas search is run during reconstruction.
    """
    require_verified_drive_output()

    if ROOT is None:
        raise RuntimeError(
            "Exact V68 ROOT is not configured."
        )

    root = Path(ROOT)
    cell2 = root / "cell2_publication_validation"

    baseline_path = (
        root
        / "seven_model_source_atlas_states.csv"
    )
    pairwise_path = (
        root
        / "pairwise_critical_noise_profiles.csv"
    )
    source_path = (
        root
        / "source_states.csv"
    )
    certified_path = (
        cell2
        / "certified_source_atlas_states.csv"
    )
    boundary_path = (
        cell2
        / "boundary_certification_summary.csv"
    )
    change_summary_path = (
        cell2
        / "certified_atlas_change_summary.json"
    )

    required_frozen = [
        baseline_path,
        pairwise_path,
        source_path,
    ]
    missing = [
        str(p)
        for p in required_frozen
        if not p.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Frozen V68 atlas lookup requires existing precomputed "
            "source-atlas assets. Missing:\n  "
            + "\n  ".join(missing)
        )

    baseline = pd.read_csv(
        baseline_path
    )
    pairwise = pd.read_csv(
        pairwise_path
    )
    source = pd.read_csv(
        source_path
    )

    source_mode = "canonical_cell2_certified_csv"
    if certified_path.is_file():
        certified = pd.read_csv(
            certified_path
        )
    else:
        if not boundary_path.is_file():
            raise FileNotFoundError(
                "Neither the canonical certified atlas nor the frozen "
                "Cell-2 boundary summary is available.\n"
                f"Expected certified: {certified_path}\n"
                f"Expected boundary summary: {boundary_path}"
            )
        boundary = pd.read_csv(
            boundary_path
        )
        certified = (
            reconstruct_certified_atlas_from_frozen_outputs(
                V,
                baseline,
                boundary,
            )
        )
        source_mode = (
            "reconstructed_in_memory_from_frozen_baseline_plus_"
            "frozen_cell2_boundary_summary"
        )

        # Save only a COPY in the Treloar output folder for reproducibility.
        # Do not modify the frozen Cell-2 directory.
        save_dataframe_checked(
            certified,
            OUTDIR
            / "FROZEN_ATLAS_reconstructed_certified_copy.csv",
            index=False,
        )

    boundary = (
        pd.read_csv(boundary_path)
        if boundary_path.is_file()
        else pd.DataFrame()
    )

    # --------------------------------------------------------------
    # Integrity checks.
    # --------------------------------------------------------------
    if len(certified) != FROZEN_ATLAS_EXPECTED_SOURCE_ROWS:
        raise RuntimeError(
            "Frozen certified atlas has unexpected row count: "
            f"{len(certified)} != "
            f"{FROZEN_ATLAS_EXPECTED_SOURCE_ROWS}"
        )

    if len(source) != FROZEN_ATLAS_EXPECTED_SOURCE_ROWS:
        raise RuntimeError(
            "Frozen source_states.csv has unexpected row count: "
            f"{len(source)}"
        )

    key_cols = [
        "source_model",
        "source_index",
    ]
    source_keys = (
        source[key_cols]
        .astype(str)
        .agg("|".join, axis=1)
        .to_numpy()
    )
    certified_keys = (
        certified[key_cols]
        .astype(str)
        .agg("|".join, axis=1)
        .to_numpy()
    )
    if not np.array_equal(
        source_keys,
        certified_keys,
    ):
        raise RuntimeError(
            "Frozen certified atlas keys do not match source_states.csv."
        )

    label_col = (
        "certified_compatibility_set_at_3pct"
    )
    if label_col not in certified.columns:
        raise RuntimeError(
            f"Frozen certified atlas lacks {label_col}"
        )

    n_regions = int(
        certified[label_col]
        .astype(str)
        .nunique()
    )
    if (
        n_regions
        != FROZEN_ATLAS_EXPECTED_REGION_COUNT
    ):
        raise RuntimeError(
            "Frozen certified atlas region count mismatch: "
            f"{n_regions} != "
            f"{FROZEN_ATLAS_EXPECTED_REGION_COUNT}"
        )

    source_inclusion = bool(
        all(
            str(row.source_model)
            in str(
                row[
                    label_col
                ]
            ).split("|")
            for _, row
            in certified.iterrows()
        )
    )
    if not source_inclusion:
        raise RuntimeError(
            "Frozen certified atlas failed source-model inclusion."
        )

    n_changes = None
    if (
        "certified_label_changed"
        in certified.columns
    ):
        n_changes = int(
            certified[
                "certified_label_changed"
            ].astype(bool).sum()
        )
        if (
            n_changes
            != FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES
        ):
            raise RuntimeError(
                "Frozen certified atlas label-change count mismatch: "
                f"{n_changes} != "
                f"{FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES}"
            )

    change_summary = {}
    if change_summary_path.is_file():
        with change_summary_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            change_summary = json.load(handle)

    audit = {
        "atlas_source_mode":
            source_mode,
        "baseline_atlas_path":
            str(baseline_path),
        "pairwise_profile_path":
            str(pairwise_path),
        "source_states_path":
            str(source_path),
        "certified_atlas_path":
            (
                str(certified_path)
                if certified_path.is_file()
                else None
            ),
        "boundary_summary_path":
            (
                str(boundary_path)
                if boundary_path.is_file()
                else None
            ),
        "n_source_states":
            int(len(source)),
        "n_certified_atlas_states":
            int(len(certified)),
        "n_pairwise_rows":
            int(len(pairwise)),
        "n_certified_regions":
            n_regions,
        "n_certified_label_changes":
            n_changes,
        "source_model_inclusion":
            source_inclusion,
        "cell2_change_summary":
            change_summary,
        "important_method_note":
            (
                "The experimental models are calibrated once from Treloar UT. "
                "All subsequent atlas labels, directed critical-noise scores, "
                "and pairwise witnesses used for the primary atlas claim are "
                "read from or reconstructed from frozen V68/Cell-2 outputs; "
                "no atlas optimization is rerun."
            ),
    }
    save_json_checked(
        audit,
        OUTDIR
        / "FROZEN_ATLAS_integrity_and_source_audit.json",
    )

    print(
        "[V68.5.5] Frozen certified atlas loaded and verified:"
    )
    print(
        f"           states={len(certified)}, "
        f"regions={n_regions}, "
        f"label changes={n_changes}, "
        f"mode={source_mode}"
    )

    return {
        "source": source,
        "baseline": baseline,
        "certified": certified,
        "pairwise": pairwise,
        "boundary": boundary,
        "audit": audit,
    }


def reconstruct_atlas_source_response(
    V,
    cfg,
    row: pd.Series,
    *,
    override_response_scale: float | None = None,
) -> np.ndarray:
    coordinate = _coordinate_json(
        row["coordinate_json"]
    )
    template = V.model_template(
        cfg,
        str(row["source_model"]),
        coordinate,
        V68_PROTOCOL,
        V68_BASES,
    )
    if override_response_scale is None:
        scale = float(
            row["source_response_scale_kpa"]
        )
    else:
        scale = float(
            override_response_scale
        )

    return (
        scale
        * np.asarray(
            template,
            dtype=np.float64,
        )
    )


def project_experimental_fit_to_frozen_atlas(
    V,
    cfg,
    fit_row: pd.Series,
    certified_atlas: pd.DataFrame,
) -> Dict[str, object]:
    """
    Nearest same-family FROZEN atlas-node lookup.

    Atlas shape coordinates are frozen. Because V68's response geometry is
    homogeneous in stiffness and the compatibility brush is relative, each
    source node is evaluated at the EXPERIMENTALLY FITTED response scale during
    projection. This prevents the atlas's deterministic one-scale-per-shape
    sampling from creating an artificial nearest-node error.
    """
    model = str(
        fit_row["model"]
    )
    experimental_scale = float(
        fit_row["best_response_scale_kpa"]
    )
    experimental_response = np.asarray(
        fit_row["_full_response"],
        dtype=np.float64,
    )

    bank = (
        certified_atlas[
            certified_atlas[
                "source_model"
            ].astype(str)
            == model
        ]
        .copy()
        .sort_values("source_index")
        .reset_index(drop=True)
    )

    if bank.empty:
        raise RuntimeError(
            f"Frozen atlas has no source nodes for {model}."
        )

    best_score = float("inf")
    best_row = None
    best_response = None

    for _, row in bank.iterrows():
        candidate = (
            reconstruct_atlas_source_response(
                V,
                cfg,
                row,
                override_response_scale=
                    experimental_scale,
            )
        )
        score = float(
            V.required_noise_fraction(
                cfg,
                V68_PROTOCOL,
                experimental_response,
                candidate,
            )
        )
        if score < best_score:
            best_score = score
            best_row = row
            best_response = candidate

    assert best_row is not None
    assert best_response is not None

    label = str(
        best_row[
            "certified_compatibility_set_at_3pct"
        ]
    )
    members = [
        item
        for item in label.split("|")
        if item
    ]

    certified_scores_percent = {}
    for target in MODELS:
        col = (
            f"certified_critical_noise_to_{target.lower()}"
        )
        if col in best_row.index:
            certified_scores_percent[target] = (
                100.0
                * float(best_row[col])
            )

    return {
        "experimental_model":
            model,
        "experimental_coordinate_json":
            str(
                fit_row[
                    "best_coordinate_json"
                ]
            ),
        "experimental_mu0_kpa":
            float(
                fit_row["best_mu0_kpa"]
            ),
        "frozen_source_model":
            str(best_row["source_model"]),
        "frozen_source_index":
            int(best_row["source_index"]),
        "frozen_coordinate_json":
            str(best_row["coordinate_json"]),
        "frozen_source_mu0_kpa":
            float(best_row["source_mu0_kpa"]),
        "projection_required_noise_fraction":
            float(best_score),
        "projection_required_noise_percent":
            float(100.0 * best_score),
        "certified_primary_region":
            str(
                best_row[
                    "certified_primary_region_at_3pct"
                ]
            ),
        "certified_compatibility_set":
            label,
        "certified_compatibility_members":
            members,
        "certified_minimal_adequate_models":
            str(
                best_row[
                    "certified_minimal_adequate_models_at_3pct"
                ]
            ),
        "certified_scores_percent":
            certified_scores_percent,
        "_frozen_row":
            best_row,
        "_scale_matched_response":
            best_response,
    }


def select_publication_primary_models_from_fits(
    fits: pd.DataFrame,
) -> Dict[str, object]:
    """Freeze the primary experimental candidates using UT training statistics only."""
    supported = (
        fits.loc[
            fits["publication_primary_supported"].astype(bool)
        ]
        .copy()
        .sort_values(["BIC", "AICc", "model"])
        .reset_index(drop=True)
    )
    if len(supported) != 2:
        raise RuntimeError(
            "Publication-primary training rule must retain exactly two models; "
            f"found {len(supported)}: "
            f"{supported[['model','training_delta_BIC','training_delta_AICc']].to_dict(orient='records')}"
        )

    source_model = str(supported.loc[0, "model"])
    target_model = str(supported.loc[1, "model"])
    return {
        "supported_models": supported["model"].astype(str).tolist(),
        "source_model": source_model,
        "target_model": target_model,
        "selection_basis": (
            "training only: Delta-BIC<=2 AND Delta-AICc<=2; "
            "direction ordered by BIC, then AICc, then model name"
        ),
        "completeF_used_for_selection": False,
        "heldout_stress_used_for_selection": False,
        "source_delta_BIC": float(supported.loc[0, "training_delta_BIC"]),
        "source_delta_AICc": float(supported.loc[0, "training_delta_AICc"]),
        "target_delta_BIC": float(supported.loc[1, "training_delta_BIC"]),
        "target_delta_AICc": float(supported.loc[1, "training_delta_AICc"]),
    }


def audit_local_frozen_atlas_transfer(
    V,
    cfg,
    fit_row: pd.Series,
    certified_atlas: pd.DataFrame,
    target_model: str,
    *,
    nearest_k: int = 10,
    localization_threshold_fraction: float = 0.03,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    """
    Post-selection stability audit around an experimental same-family atlas lookup.

    Every frozen source node from the experimental family is evaluated at the
    experimentally fitted stiffness. Nodes are ranked only by experimental-to-node
    projection discrepancy. Their already-certified source->target distances are
    then read from the frozen atlas. No target optimization and no pair selection
    is performed here.
    """
    source_model = str(fit_row["model"])
    experimental_scale = float(fit_row["best_response_scale_kpa"])
    experimental_response = np.asarray(fit_row["_full_response"], dtype=np.float64)

    bank = (
        certified_atlas[
            certified_atlas["source_model"].astype(str) == source_model
        ]
        .copy()
        .sort_values("source_index")
        .reset_index(drop=True)
    )
    if bank.empty:
        raise RuntimeError(f"Frozen atlas has no source nodes for {source_model}.")

    target_col = f"certified_critical_noise_to_{str(target_model).lower()}"
    if target_col not in bank.columns:
        raise RuntimeError(
            f"Frozen certified atlas does not contain {target_col}."
        )

    rows = []
    for _, row in bank.iterrows():
        candidate = reconstruct_atlas_source_response(
            V, cfg, row, override_response_scale=experimental_scale
        )
        projection = float(
            V.required_noise_fraction(
                cfg, V68_PROTOCOL, experimental_response, candidate
            )
        )
        target_noise = float(row[target_col])
        rows.append({
            "source_model": source_model,
            "source_index": int(row["source_index"]),
            "source_coordinate_json": str(row["coordinate_json"]),
            "projection_required_noise_fraction": projection,
            "projection_required_noise_percent": 100.0 * projection,
            "certified_compatibility_set": str(
                row["certified_compatibility_set_at_3pct"]
            ),
            "target_model": str(target_model),
            "certified_source_to_target_noise_fraction": target_noise,
            "certified_source_to_target_noise_percent": 100.0 * target_noise,
            "target_excluded_at_3pct": bool(
                target_noise > ATLAS_TOLERANCE_FRACTION
            ),
        })

    all_nodes = (
        pd.DataFrame(rows)
        .sort_values(["projection_required_noise_fraction", "source_index"])
        .reset_index(drop=True)
    )
    nearest = all_nodes.head(max(1, int(nearest_k))).copy()
    local = all_nodes[
        all_nodes["projection_required_noise_fraction"].astype(float)
        <= float(localization_threshold_fraction) + 1.0e-12
    ].copy()

    if len(local):
        local_scores = local[
            "certified_source_to_target_noise_percent"
        ].to_numpy(float)
        local_exclusion_fraction = float(
            local["target_excluded_at_3pct"].astype(bool).mean()
        )
        all_local_exclude = bool(
            local["target_excluded_at_3pct"].astype(bool).all()
        )
        local_min = float(np.min(local_scores))
        local_med = float(np.median(local_scores))
        local_max = float(np.max(local_scores))
    else:
        local_exclusion_fraction = float("nan")
        all_local_exclude = False
        local_min = local_med = local_max = float("nan")

    summary = {
        "source_model": source_model,
        "target_model": str(target_model),
        "nearest_k": int(min(max(1, int(nearest_k)), len(all_nodes))),
        "nearest_source_index": int(all_nodes.loc[0, "source_index"]),
        "nearest_projection_percent": float(
            all_nodes.loc[0, "projection_required_noise_percent"]
        ),
        "nearest_source_to_target_noise_percent": float(
            all_nodes.loc[0, "certified_source_to_target_noise_percent"]
        ),
        "localization_threshold_percent": 100.0 * float(localization_threshold_fraction),
        "n_same_family_frozen_nodes": int(len(all_nodes)),
        "n_nodes_within_localization_threshold": int(len(local)),
        "fraction_local_nodes_excluding_target_at_3pct": local_exclusion_fraction,
        "all_local_nodes_exclude_target_at_3pct": all_local_exclude,
        "local_source_to_target_noise_percent_min": local_min,
        "local_source_to_target_noise_percent_median": local_med,
        "local_source_to_target_noise_percent_max": local_max,
        "selection_role": "POST_SELECTION_ROBUSTNESS_AUDIT_ONLY",
    }
    return nearest, local, summary

def frozen_pairwise_witness_for_source_node(
    V,
    cfg,
    assets: Mapping[str, object],
    source_lookup: Mapping[str, object],
    target_model: str,
) -> Dict[str, object]:
    """
    Recover the already-frozen V68 pairwise target witness for one source node.

    If Cell-2 improved this directed fit, use the Cell-2 certified target witness.
    Otherwise use the original frozen pairwise witness. No new optimization.
    """
    source_row = source_lookup[
        "_frozen_row"
    ]
    source_model = str(
        source_row["source_model"]
    )
    source_index = int(
        source_row["source_index"]
    )

    pairwise = assets["pairwise"]
    match = pairwise[
        (pairwise["source_model"].astype(str) == source_model)
        & (
            pairwise["source_index"].astype(int)
            == source_index
        )
        & (
            pairwise["target_model"].astype(str)
            == str(target_model)
        )
    ]

    if len(match) != 1:
        raise RuntimeError(
            "Expected exactly one frozen directed pairwise row for "
            f"{source_model}[{source_index}] -> {target_model}; "
            f"found {len(match)}."
        )

    baseline = match.iloc[0]
    baseline_score = float(
        baseline["critical_noise_fraction"]
    )
    coordinate_json = str(
        baseline[
            "best_target_coordinate_json"
        ]
    )
    target_mu0_kpa = float(
        baseline["best_target_mu0_kpa"]
    )
    witness_source = "frozen_V68_baseline_pairwise_profile"
    certified_score = baseline_score

    boundary = assets["boundary"]
    if len(boundary):
        bmatch = boundary[
            (boundary["source_model"].astype(str) == source_model)
            & (
                boundary["source_index"].astype(int)
                == source_index
            )
            & (
                boundary["target_model"].astype(str)
                == str(target_model)
            )
        ]
        if len(bmatch) > 1:
            raise RuntimeError(
                "Frozen Cell-2 boundary summary contains duplicate directed rows."
            )
        if len(bmatch) == 1:
            br = bmatch.iloc[0]
            boundary_score = float(
                br["certified_score"]
            )
            if boundary_score <= baseline_score + 1.0e-15:
                certified_score = boundary_score
                coordinate_json = str(
                    br[
                        "best_target_coordinate_json"
                    ]
                )
                target_mu0_kpa = float(
                    br[
                        "best_target_mu0_kpa"
                    ]
                )
                witness_source = (
                    "frozen_Cell2_certified_pairwise_witness"
                )

    certified_col = (
        f"certified_critical_noise_to_{str(target_model).lower()}"
    )
    if certified_col in source_row.index:
        atlas_certified_score = float(
            source_row[certified_col]
        )
        if (
            abs(
                atlas_certified_score
                - certified_score
            )
            > 2.0e-8
        ):
            # The row may have inherited hierarchy closure; for MR->OGDEN1 this
            # should not occur. Treat any mismatch as a hard publication error.
            raise RuntimeError(
                "Frozen pairwise witness score does not match certified atlas "
                f"score for {source_model}[{source_index}] -> {target_model}: "
                f"{certified_score} vs {atlas_certified_score}"
            )

    coordinate = _coordinate_json(
        coordinate_json
    )

    # Match the frozen source-node relation to the experimental source stiffness.
    frozen_source_scale = float(
        source_row[
            "source_response_scale_kpa"
        ]
    )
    experimental_source_scale = float(
        source_lookup[
            "experimental_mu0_kpa"
        ]
    )
    experimental_source_scale = float(
        V.response_scale_from_mu0_kpa(
            experimental_source_scale
        )
    )

    frozen_target_scale = float(
        V.response_scale_from_mu0_kpa(
            target_mu0_kpa
        )
    )
    scale_ratio = (
        frozen_target_scale
        / max(
            frozen_source_scale,
            1.0e-300,
        )
    )
    matched_target_scale = (
        scale_ratio
        * experimental_source_scale
    )

    target_template = V.model_template(
        cfg,
        str(target_model),
        coordinate,
        V68_PROTOCOL,
        V68_BASES,
    )
    target_response = (
        matched_target_scale
        * np.asarray(
            target_template,
            dtype=np.float64,
        )
    )
    source_response = np.asarray(
        source_lookup[
            "_scale_matched_response"
        ],
        dtype=np.float64,
    )

    state_score, component_score = (
        completeF_statewise_separation(
            V,
            cfg,
            source_response,
            target_response,
        )
    )
    witness_state_index = int(
        np.argmax(state_score)
    )
    witness_component_index = int(
        np.argmax(
            component_score[
                witness_state_index
            ]
        )
    )
    witness_stretches = np.asarray(
        V68_PROTOCOL.parent_principal_stretches[
            witness_state_index
        ],
        dtype=np.float64,
    )

    return {
        "source_model":
            source_model,
        "source_index":
            source_index,
        "target_model":
            str(target_model),
        "frozen_certified_critical_noise_fraction":
            float(certified_score),
        "frozen_certified_critical_noise_percent":
            float(100.0 * certified_score),
        "frozen_target_coordinate_json":
            coordinate_json,
        "frozen_target_mu0_kpa":
            float(target_mu0_kpa),
        "frozen_target_witness_source":
            witness_source,
        "stiffness_scale_ratio_target_to_source":
            float(scale_ratio),
        "global_completeF_witness_state_index":
            witness_state_index,
        "global_completeF_witness_component_index":
            witness_component_index,
        "global_completeF_witness_lambda1":
            float(witness_stretches[0]),
        "global_completeF_witness_lambda2":
            float(witness_stretches[1]),
        "global_completeF_witness_lambda3":
            float(witness_stretches[2]),
        "global_completeF_witness_separation_fraction":
            float(
                state_score[
                    witness_state_index
                ]
            ),
        "global_completeF_witness_separation_percent":
            float(
                100.0
                * state_score[
                    witness_state_index
                ]
            ),
        "_source_response":
            source_response,
        "_target_response":
            target_response,
        "_target_coordinate":
            coordinate,
        "_matched_target_scale":
            float(matched_target_scale),
    }


def select_measured_witness_from_frozen_atlas_pair(
    V,
    cfg,
    dataset: Mapping[str, object],
    atlas_pair: Mapping[str, object],
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Select the most discriminating ACTUALLY MEASURED holdout point from the
    FROZEN atlas source->target witness, before experimental held-out stresses
    are consulted.

    The returned table then attaches the experimental stresses only after the
    selection index has been frozen.
    """
    source_model = str(
        atlas_pair["source_model"]
    )
    target_model = str(
        atlas_pair["target_model"]
    )

    source_response = np.asarray(
        atlas_pair["_source_response"],
        dtype=np.float64,
    )
    source_row = None  # reconstructed below from scale-matched full response is enough

    target_coordinate = np.asarray(
        atlas_pair["_target_coordinate"],
        dtype=np.float64,
    )
    target_scale = float(
        atlas_pair["_matched_target_scale"]
    )

    # Recover source coordinate from the primary source atlas lookup stored later.
    # It is injected into atlas_pair before this function is called.
    source_coordinate = np.asarray(
        atlas_pair["_source_coordinate"],
        dtype=np.float64,
    )
    source_scale = float(
        atlas_pair["_source_scale"]
    )

    records = []

    for mode in (
        "pure_shear_planar",
        "equibiaxial",
    ):
        frame = dataset["modes"][mode]
        states = frame[
            [
                "lambda1",
                "lambda2",
                "lambda3",
            ]
        ].to_numpy(float)

        source_unit = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                source_model,
                source_coordinate,
                states,
            )
        )
        target_unit = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                target_model,
                target_coordinate,
                states,
            )
        )

        source_prediction = (
            source_scale
            * source_unit
        )
        target_prediction = (
            target_scale
            * target_unit
        )
        predicted_separation = (
            pointwise_pair_separation(
                cfg,
                source_prediction,
                target_prediction,
            )
        )

        for local_index in range(len(frame)):
            records.append(
                {
                    "mode":
                        mode,
                    "local_index":
                        int(local_index),
                    "stretch":
                        float(
                            frame.loc[
                                local_index,
                                "stretch",
                            ]
                        ),
                    "lambda1":
                        float(
                            frame.loc[
                                local_index,
                                "lambda1",
                            ]
                        ),
                    "lambda2":
                        float(
                            frame.loc[
                                local_index,
                                "lambda2",
                            ]
                        ),
                    "lambda3":
                        float(
                            frame.loc[
                                local_index,
                                "lambda3",
                            ]
                        ),
                    "frozen_atlas_source_prediction_kPa":
                        float(
                            source_prediction[
                                local_index
                            ]
                        ),
                    "frozen_atlas_target_prediction_kPa":
                        float(
                            target_prediction[
                                local_index
                            ]
                        ),
                    "frozen_atlas_predicted_separation_fraction":
                        float(
                            predicted_separation[
                                local_index
                            ]
                        ),
                }
            )

    table = pd.DataFrame(
        records
    )
    selected_index = int(
        table[
            "frozen_atlas_predicted_separation_fraction"
        ]
        .to_numpy(float)
        .argmax()
    )

    table[
        "atlas_preselected_measured_witness"
    ] = False
    table.loc[
        selected_index,
        "atlas_preselected_measured_witness",
    ] = True

    # Only now attach held-out experimental stresses.
    observed = []
    for _, row in table.iterrows():
        frame = dataset["modes"][
            str(row["mode"])
        ]
        observed.append(
            float(
                frame.loc[
                    int(row["local_index"]),
                    "v68_tau13_kPa",
                ]
            )
        )
    table[
        "experimental_tau13_kPa"
    ] = observed
    table[
        "frozen_atlas_predicted_separation_percent"
    ] = (
        100.0
        * table[
            "frozen_atlas_predicted_separation_fraction"
        ]
    )

    selected = table.loc[
        selected_index
    ]

    summary = {
        "atlas_selected_measured_witness_mode":
            str(selected["mode"]),
        "atlas_selected_measured_witness_local_index":
            int(selected["local_index"]),
        "atlas_selected_measured_witness_stretch":
            float(selected["stretch"]),
        "atlas_selected_measured_witness_lambda1":
            float(selected["lambda1"]),
        "atlas_selected_measured_witness_lambda2":
            float(selected["lambda2"]),
        "atlas_selected_measured_witness_lambda3":
            float(selected["lambda3"]),
        "atlas_selected_measured_witness_predicted_separation_percent":
            float(
                selected[
                    "frozen_atlas_predicted_separation_percent"
                ]
            ),
        "atlas_selected_measured_witness_experimental_tau13_kPa":
            float(
                selected[
                    "experimental_tau13_kPa"
                ]
            ),
    }

    return table, summary


def evaluate_experimental_primary_models_at_atlas_selected_witness(
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    atlas_witness_summary: Mapping[str, object],
    primary_models: Sequence[str],
) -> Dict[str, object]:
    """Compare the training-retained candidates at the atlas-selected holdout state."""
    primary_models = tuple(str(m) for m in primary_models)
    if len(primary_models) != 2:
        raise ValueError("Exactly two primary models are required.")
    lookup = {str(row["model"]): row for _, row in fits.iterrows()}

    mode = str(atlas_witness_summary["atlas_selected_measured_witness_mode"])
    local_index = int(atlas_witness_summary["atlas_selected_measured_witness_local_index"])
    if mode == "pure_shear_planar":
        pred_key = "_planar_prediction"
    elif mode == "equibiaxial":
        pred_key = "_equibiaxial_prediction"
    else:
        raise KeyError(mode)

    frame = dataset["modes"][mode]
    truth = float(frame.loc[local_index, "v68_tau13_kPa"])
    predictions = {}
    errors = {}
    for model in primary_models:
        prediction = float(np.asarray(lookup[model][pred_key], dtype=np.float64)[local_index])
        predictions[model] = prediction
        errors[model] = abs(truth - prediction)

    preferred = min(errors, key=errors.get)
    combined_preferred = min(
        primary_models,
        key=lambda model: float(lookup[model]["heldout_combined_nrmse_percent"]),
    )
    m0, m1 = primary_models
    return {
        "atlas_selected_witness_experimental_stress_kPa": truth,
        "primary_model_1": m0,
        "primary_model_2": m1,
        "experimental_fitted_prediction_model_1_kPa": float(predictions[m0]),
        "experimental_fitted_prediction_model_2_kPa": float(predictions[m1]),
        "experimental_fitted_abs_error_model_1_kPa": float(errors[m0]),
        "experimental_fitted_abs_error_model_2_kPa": float(errors[m1]),
        "atlas_selected_witness_preferred_experimental_fit": preferred,
        "combined_holdout_preferred_primary_model": combined_preferred,
        "atlas_witness_preference_matches_combined_holdout": bool(
            preferred == combined_preferred
        ),
    }



# =============================================================================
# Public Treloar dataset import
# =============================================================================

def download_csv(url: str, destination: Path) -> pd.DataFrame:
    # Do not contact the network until persistent Drive saving is verified.
    require_verified_drive_output()
    destination = Path(destination).resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "V68-Treloar-public-experimental-validation/1.0"
        },
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()

    destination.write_bytes(payload)
    _verify_saved_file(
        destination,
        minimum_bytes=10,
    )
    df = pd.read_csv(destination)

    # Public benchmark CSVs may contain incidental whitespace in headers
    # (for example " pk1"). Normalize headers before schema validation.
    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    if len(set(df.columns)) != len(df.columns):
        raise ValueError(
            f"{destination.name} has duplicate column names after "
            f"whitespace normalization: {list(df.columns)}"
        )

    required = {"stretch", "pk1"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"{destination.name} must contain columns {sorted(required)}; "
            f"found {list(df.columns)}"
        )

    df = df[["stretch", "pk1"]].copy()
    df["stretch"] = pd.to_numeric(
        df["stretch"], errors="raise"
    ).astype(float)
    df["pk1"] = pd.to_numeric(
        df["pk1"], errors="raise"
    ).astype(float)

    if np.any(df["stretch"].to_numpy() <= 0.0):
        raise ValueError(
            f"{destination.name} contains non-positive stretch."
        )

    return df


def principal_stretches_for_mode(
    mode: str,
    stretch: np.ndarray,
) -> np.ndarray:
    lam = np.asarray(stretch, dtype=np.float64)

    if mode == "uniaxial":
        return np.column_stack(
            [lam, lam ** (-0.5), lam ** (-0.5)]
        )

    if mode == "pure_shear_planar":
        # Classical pure shear / planar tension, not simple shear.
        return np.column_stack(
            [lam, np.ones_like(lam), lam ** (-1.0)]
        )

    if mode == "equibiaxial":
        return np.column_stack(
            [lam, lam, lam ** (-2.0)]
        )

    raise KeyError(mode)


def filter_to_v68_parent_domain(
    cfg,
    mode: str,
    df: pd.DataFrame,
) -> pd.DataFrame:
    states = principal_stretches_for_mode(
        mode,
        df["stretch"].to_numpy(float),
    )
    lo = float(cfg.parent_principal_stretch_min)
    hi = float(cfg.parent_principal_stretch_max)

    keep = (
        np.all(states >= lo - 1.0e-12, axis=1)
        & np.all(states <= hi + 1.0e-12, axis=1)
    )

    out = df.loc[keep].copy().reset_index(drop=True)
    if out.empty:
        raise ValueError(
            f"No {mode} data remain inside V68 parent domain "
            f"[{lo}, {hi}]."
        )

    out_states = principal_stretches_for_mode(
        mode,
        out["stretch"].to_numpy(float),
    )
    out["lambda1"] = out_states[:, 0]
    out["lambda2"] = out_states[:, 1]
    out["lambda3"] = out_states[:, 2]

    # Source CSV is P11 in MPa. Under J=1 with the free reference direction
    # tractionless, tau11 - tau33 = lambda1 * P11.
    out["pk1_MPa"] = out["pk1"].astype(float)
    out["pk1_kPa"] = (
        SOURCE_STRESS_TO_KPA * out["pk1_MPa"]
    )
    out["v68_tau13_kPa"] = (
        out["lambda1"] * out["pk1_kPa"]
    )

    return out


def load_treloar_dataset(
    cfg,
    outdir: Path,
) -> Dict[str, object]:
    raw = {}
    filtered = {}

    for mode, url in TRELOAR_URLS.items():
        raw_df = download_csv(
            url,
            outdir / f"downloaded_{mode}.csv",
        )
        raw[mode] = raw_df
        filtered[mode] = filter_to_v68_parent_domain(
            cfg,
            mode,
            raw_df,
        )
        save_dataframe_checked(
            filtered[mode],
            outdir / f"v68_domain_{mode}.csv",
            index=False,
        )

    return {
        "material": "Treloar vulcanized natural rubber",
        "publication_title":
            "Stress-strain data for vulcanised rubber under various types of deformation",
        "original_author": "L. R. G. Treloar",
        "original_year": 1944,
        "repository":
            "llamm-de/thermalCANN — Steinmann reproduction",
        "training_mode": "uniaxial",
        "heldout_modes": [
            "pure_shear_planar",
            "equibiaxial",
        ],
        "raw": raw,
        "modes": filtered,
    }



def build_treloar_provenance_audit(
    dataset: Mapping[str, object],
) -> Dict[str, object]:
    """
    Record exactly what is source-native and what is an explicit interpretation.

    This avoids implying that the GitHub CSV itself contains a unit declaration.
    """
    raw_columns = {
        mode: [str(c) for c in dataset["raw"][mode].columns]
        for mode in TRELOAR_URLS
    }
    raw_ranges = {}
    for mode in TRELOAR_URLS:
        df = dataset["raw"][mode]
        raw_ranges[mode] = {
            "n_raw_rows": int(len(df)),
            "stretch_min": float(df["stretch"].min()),
            "stretch_max": float(df["stretch"].max()),
            "pk1_min_source_units": float(df["pk1"].min()),
            "pk1_max_source_units": float(df["pk1"].max()),
        }

    retained_ranges = {}
    for mode in TRELOAR_URLS:
        df = dataset["modes"][mode]
        retained_ranges[mode] = {
            "n_retained_rows": int(len(df)),
            "stretch_min": float(df["stretch"].min()),
            "stretch_max": float(df["stretch"].max()),
            "lambda1_min": float(df["lambda1"].min()),
            "lambda1_max": float(df["lambda1"].max()),
            "lambda2_min": float(df["lambda2"].min()),
            "lambda2_max": float(df["lambda2"].max()),
            "lambda3_min": float(df["lambda3"].min()),
            "lambda3_max": float(df["lambda3"].max()),
        }

    return {
        "original_dataset": "Treloar vulcanized rubber",
        "original_doi": TRELOAR_ORIGINAL_DOI,
        "machine_readable_source_repository": THERMALCANN_REPOSITORY,
        "machine_readable_variant": "Steinmann reproduction",
        "steinmann_reproduction_doi": STEINMANN_REPRODUCTION_DOI,
        "csv_urls": dict(TRELOAR_URLS),
        "source_native_columns_after_whitespace_normalization": raw_columns,
        "source_native_numeric_ranges": raw_ranges,
        "v68_domain_retained_ranges": retained_ranges,
        "source_csv_embeds_stress_unit": False,
        "analysis_interpreted_stress_measure": "P11 first Piola-Kirchhoff / nominal stress",
        "analysis_interpreted_source_stress_unit": "MPa",
        "conversion_to_analysis_unit": "1 MPa = 1000 kPa",
        "conversion_to_v68_pressure_free_response":
            "tau11 - tau33 = lambda1 * P11 because J=1 and P33=0 in the free thickness direction",
        "stress_unit_provenance_note": STRESS_UNIT_PROVENANCE_NOTE,
        "important_reproducibility_note":
            "The public CSV is a machine-readable reproduction, not a modern raw experimental deposition.",
    }


# =============================================================================
# V68-standard response formulas evaluated at arbitrary experimental points
# =============================================================================

def _ogden_power_difference_over_alpha(
    alpha: float,
    log_a: np.ndarray,
    log_b: np.ndarray,
    series_threshold: float,
) -> np.ndarray:
    """Same removable-limit convention used by V68.1."""
    alpha = float(alpha)
    a = np.asarray(log_a, dtype=np.float64)
    b = np.asarray(log_b, dtype=np.float64)
    if abs(alpha) <= float(series_threshold):
        a2, b2 = a * a, b * b
        a3, b3 = a2 * a, b2 * b
        a4, b4 = a3 * a, b3 * b
        a5, b5 = a4 * a, b4 * b
        return 4.0 * (
            (a - b)
            + 0.5 * alpha * (a2 - b2)
            + (alpha**2 / 6.0) * (a3 - b3)
            + (alpha**3 / 24.0) * (a4 - b4)
            + (alpha**4 / 120.0) * (a5 - b5)
        )
    return 4.0 * (np.exp(alpha * a) - np.exp(alpha * b)) / alpha


def diagonal_first_stress_difference_unit_template(
    V,
    cfg,
    model: str,
    coordinate: Sequence[float],
    principal_stretches: np.ndarray,
) -> np.ndarray:
    """
    Evaluate the exact V68 unit-scale constitutive response tau11-tau33 on
    arbitrary incompressible diagonal principal-stretch states.

    This reproduces the first response component of V68's complete-F parent.
    """
    lam = np.asarray(
        principal_stretches,
        dtype=np.float64,
    )
    if lam.ndim != 2 or lam.shape[1] != 3:
        raise ValueError(
            "principal_stretches must have shape (n,3)."
        )
    if np.any(lam <= 0.0):
        raise ValueError(
            "Principal stretches must be positive."
        )

    l1 = lam[:, 0]
    l2 = lam[:, 1]
    l3 = lam[:, 2]

    coord = np.asarray(
        coordinate,
        dtype=np.float64,
    ).reshape(-1)

    # Exact first complete-F component used in V68 constitutive_bases().
    b1 = 2.0 * (l1**2 - l3**2)
    b2 = 2.0 * (l3**-2.0 - l1**-2.0)
    i1m3 = l1**2 + l2**2 + l3**2 - 3.0
    i2m3 = l1**-2.0 + l2**-2.0 + l3**-2.0 - 3.0

    def ogden1(alpha: float) -> np.ndarray:
        return _ogden_power_difference_over_alpha(
            alpha,
            np.log(l1),
            np.log(l3),
            cfg.ogden_alpha_series_threshold,
        )

    if model == "NH":
        return b1

    if model == "MR":
        rho = float(coord[0])
        return (1.0 - rho) * b1 + rho * b2

    if model == "YEOH2":
        beta = float(coord[0])
        return b1 * (
            1.0 + 2.0 * beta * i1m3
        )

    if model == "GENT":
        gent_gamma = float(coord[0])
        denom = 1.0 - gent_gamma * i1m3
        if np.any(
            denom <= cfg.gent_singularity_margin
        ):
            return np.full_like(b1, np.nan)
        return b1 / denom

    if model == "OGDEN1":
        return ogden1(float(coord[0]))

    if model == "OGDEN2":
        p = V.ogden2_map_q_to_physical(
            cfg, coord
        )
        return (
            float(p["weight1"])
            * ogden1(float(p["alpha1"]))
            + float(p["weight2"])
            * ogden1(float(p["alpha2"]))
        )

    if model == "GP2":
        p = V.gp2_map_q_to_physical(
            cfg, coord
        )
        rho = float(p["rho"])
        beta20 = float(p["beta20"])
        beta11 = float(p["beta11"])
        beta02 = float(p["beta02"])

        w1 = (
            (1.0 - rho)
            + 2.0 * beta20 * i1m3
            + beta11 * i2m3
        )
        w2 = (
            rho
            + beta11 * i1m3
            + 2.0 * beta02 * i2m3
        )
        return w1 * b1 + w2 * b2

    raise KeyError(model)



def self_test_diagonal_evaluator_against_v68(
    V,
    cfg,
) -> None:
    """
    Verify the arbitrary diagonal evaluator against V68's exact retained
    uniaxial and equibiaxial protocol slices before touching experimental data.
    """
    representatives = {
        "NH": [0.0],
        "MR": [0.30],
        "YEOH2": [0.20],
        "GENT": [0.10],
        "OGDEN1": [1.30],
        "OGDEN2": [0.20, 0.70, 0.40],
        "GP2": [0.30, 0.20, 0.50, 0.70],
    }

    max_error = 0.0

    for model, coordinate in representatives.items():
        coordinate = np.asarray(
            coordinate,
            dtype=np.float64,
        )
        exact_full = V.model_template(
            cfg,
            model,
            coordinate,
            V68_PROTOCOL,
            V68_BASES,
        )

        ut_lam = V68_PROTOCOL.ut
        ut_states = np.column_stack(
            [
                ut_lam,
                ut_lam ** (-0.5),
                ut_lam ** (-0.5),
            ]
        )
        predicted_ut = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                model,
                coordinate,
                ut_states,
            )
        )
        exact_ut = exact_full[
            V68_PROTOCOL.mode_slices["UT"]
        ]

        bt_lam = V68_PROTOCOL.bt
        bt_states = np.column_stack(
            [
                bt_lam,
                bt_lam,
                bt_lam ** (-2.0),
            ]
        )
        predicted_bt = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                model,
                coordinate,
                bt_states,
            )
        )
        exact_bt = exact_full[
            V68_PROTOCOL.mode_slices["BT"]
        ]

        error = max(
            float(
                np.max(
                    np.abs(
                        predicted_ut
                        - exact_ut
                    )
                )
            ),
            float(
                np.max(
                    np.abs(
                        predicted_bt
                        - exact_bt
                    )
                )
            ),
        )
        max_error = max(
            max_error,
            error,
        )

    if max_error > 5.0e-9:
        raise RuntimeError(
            "Arbitrary diagonal-response evaluator does not "
            f"match frozen V68 conventions; max abs error={max_error:.3e}"
        )

    print(
        "[V68.5.5] Diagonal-response evaluator self-test PASS; "
        f"max abs difference={max_error:.3e}"
    )


# =============================================================================
# Axial-only conventional fitting
# =============================================================================

def nrmse_fraction(truth: np.ndarray, pred: np.ndarray) -> float:
    truth = np.asarray(truth, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    denom = max(float(np.sqrt(np.mean(truth**2))), 1.0e-30)
    return float(np.sqrt(np.mean((truth - pred) ** 2)) / denom)


def best_positive_scale_least_squares(
    y: np.ndarray,
    template: np.ndarray,
    scale_lo: float,
    scale_hi: float,
) -> Tuple[float, np.ndarray, float, float]:
    y = np.asarray(y, dtype=np.float64)
    f = np.asarray(template, dtype=np.float64)
    if not np.all(np.isfinite(f)):
        return math.sqrt(scale_lo * scale_hi), np.full_like(y, np.nan), float("inf"), float("inf")
    denom = float(np.dot(f, f))
    if denom <= 1.0e-30:
        scale = math.sqrt(scale_lo * scale_hi)
    else:
        scale = float(np.dot(y, f) / denom)
    scale = float(np.clip(scale, scale_lo, scale_hi))
    pred = scale * f
    residual = y - pred
    rss = float(np.dot(residual, residual))
    nrmse = nrmse_fraction(y, pred)
    return scale, pred, rss, nrmse


def pooled_nrmse_fraction(
    truths: Sequence[np.ndarray],
    predictions: Sequence[np.ndarray],
) -> float:
    y = np.concatenate(
        [np.asarray(v, dtype=np.float64) for v in truths]
    )
    p = np.concatenate(
        [np.asarray(v, dtype=np.float64) for v in predictions]
    )
    denom = max(
        float(np.sqrt(np.mean(y**2))),
        1.0e-30,
    )
    return float(
        np.sqrt(np.mean((y - p) ** 2)) / denom
    )


def fit_all_models_to_uniaxial(
    V,
    cfg,
    dataset: Mapping[str, object],
) -> pd.DataFrame:
    train = dataset["modes"]["uniaxial"]
    planar = dataset["modes"]["pure_shear_planar"]
    biaxial = dataset["modes"]["equibiaxial"]

    train_states = train[
        ["lambda1", "lambda2", "lambda3"]
    ].to_numpy(float)
    planar_states = planar[
        ["lambda1", "lambda2", "lambda3"]
    ].to_numpy(float)
    biaxial_states = biaxial[
        ["lambda1", "lambda2", "lambda3"]
    ].to_numpy(float)

    y_train = train["v68_tau13_kPa"].to_numpy(float)
    y_planar = planar["v68_tau13_kPa"].to_numpy(float)
    y_biaxial = biaxial["v68_tau13_kPa"].to_numpy(float)

    scale_lo = float(
        V.response_scale_from_mu0_kpa(
            cfg.mu0_min_kpa
        )
    )
    scale_hi = float(
        V.response_scale_from_mu0_kpa(
            cfg.mu0_max_kpa
        )
    )

    rows = []

    for model_index, model in enumerate(MODELS):
        print(
            f"[V68.5.5] Uniaxial-only fit: {model}"
        )
        d = int(V.MODEL_SHAPE_DIMS[model])

        def evaluate(coord: Sequence[float]):
            template = (
                diagonal_first_stress_difference_unit_template(
                    V,
                    cfg,
                    model,
                    coord,
                    train_states,
                )
            )
            scale, pred, rss, nrmse = (
                best_positive_scale_least_squares(
                    y_train,
                    template,
                    scale_lo,
                    scale_hi,
                )
            )
            return {
                "scale": scale,
                "pred": pred,
                "rss": rss,
                "nrmse": nrmse,
            }

        if model == "NH":
            coord = np.array(
                [0.0], dtype=np.float64
            )
            best = evaluate(coord)

        elif model in V.SCALAR_PARENTS:
            lo, hi = V.scalar_bounds(
                cfg, model
            )
            opt = minimize_scalar(
                lambda x: evaluate(
                    [float(x)]
                )["nrmse"],
                bounds=(float(lo), float(hi)),
                method="bounded",
                options={
                    "xatol": 1.0e-11,
                    "maxiter": 500,
                },
            )
            coord = np.array(
                [float(opt.x)],
                dtype=np.float64,
            )
            best = evaluate(coord)

        else:
            bounds = [(0.0, 1.0)] * d
            opt = differential_evolution(
                lambda q: evaluate(
                    np.asarray(
                        q,
                        dtype=np.float64,
                    )
                )["nrmse"],
                bounds=bounds,
                seed=RNG_SEED
                + 1000 * model_index,
                popsize=DE_POPSIZE,
                maxiter=DE_MAXITER,
                tol=DE_TOL,
                atol=1.0e-12,
                polish=True,
                workers=1,
                updating="immediate",
            )
            coord = np.asarray(
                opt.x,
                dtype=np.float64,
            )
            best = evaluate(coord)

        scale = float(best["scale"])

        full_template = V.model_template(
            cfg,
            model,
            coord,
            V68_PROTOCOL,
            V68_BASES,
        )
        full_response = (
            scale
            * np.asarray(
                full_template,
                dtype=np.float64,
            )
        )

        planar_template = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                model,
                coord,
                planar_states,
            )
        )
        biaxial_template = (
            diagonal_first_stress_difference_unit_template(
                V,
                cfg,
                model,
                coord,
                biaxial_states,
            )
        )

        planar_prediction = (
            scale * planar_template
        )
        biaxial_prediction = (
            scale * biaxial_template
        )

        planar_nrmse = nrmse_fraction(
            y_planar,
            planar_prediction,
        )
        biaxial_nrmse = nrmse_fraction(
            y_biaxial,
            biaxial_prediction,
        )
        combined_nrmse = pooled_nrmse_fraction(
            [y_planar, y_biaxial],
            [
                planar_prediction,
                biaxial_prediction,
            ],
        )

        n = int(len(y_train))
        k = int(d + 1)
        rss_floor = max(
            float(best["rss"]),
            np.finfo(float).eps
            * max(
                float(
                    np.dot(
                        y_train,
                        y_train,
                    )
                ),
                1.0,
            ),
        )

        aic = float(
            n * np.log(rss_floor / n)
            + 2.0 * k
        )
        bic = float(
            n * np.log(rss_floor / n)
            + k * np.log(n)
        )

        # AICc is saved as a diagnostic because n=7 is small.
        if n > k + 1:
            aicc = float(
                aic
                + (
                    2.0 * k * (k + 1)
                    / (n - k - 1)
                )
            )
        else:
            aicc = float("inf")

        _, _, physical = (
            V.model_template_and_jacobian(
                cfg,
                model,
                coord,
                V68_PROTOCOL,
                V68_BASES,
            )
        )
        mu0 = float(
            V.mu0_kpa_from_response_scale(
                scale
            )
        )

        try:
            shape_zone = V.shape_support_zone(
                cfg,
                model,
                physical,
            )
            scale_zone = V.scale_support_zone(
                cfg,
                mu0,
            )
            biological_zone = (
                V.combine_support_zones(
                    shape_zone,
                    scale_zone,
                )
            )
        except Exception:
            shape_zone = "UNKNOWN"
            scale_zone = "UNKNOWN"
            biological_zone = "UNKNOWN"

        rows.append(
            {
                "model": model,
                "n_shape_parameters": d,
                "n_fitted_parameters": k,
                "best_coordinate_json":
                    json.dumps(
                        [
                            float(x)
                            for x in coord
                        ]
                    ),
                "best_physical_parameters_json":
                    json.dumps(
                        physical,
                        sort_keys=True,
                    ),
                "best_response_scale_kpa":
                    scale,
                "best_mu0_kpa":
                    mu0,
                "shape_support_zone":
                    shape_zone,
                "scale_support_zone":
                    scale_zone,
                "biological_support_zone":
                    biological_zone,
                "uniaxial_rss":
                    float(best["rss"]),
                "uniaxial_nrmse":
                    float(best["nrmse"]),
                "uniaxial_nrmse_percent":
                    float(
                        100.0
                        * best["nrmse"]
                    ),
                "AIC": aic,
                "AICc": aicc,
                "BIC": bic,
                "heldout_planar_nrmse":
                    float(planar_nrmse),
                "heldout_planar_nrmse_percent":
                    float(
                        100.0
                        * planar_nrmse
                    ),
                "heldout_equibiaxial_nrmse":
                    float(biaxial_nrmse),
                "heldout_equibiaxial_nrmse_percent":
                    float(
                        100.0
                        * biaxial_nrmse
                    ),
                "heldout_combined_nrmse":
                    float(combined_nrmse),
                "heldout_combined_nrmse_percent":
                    float(
                        100.0
                        * combined_nrmse
                    ),
                "_uniaxial_prediction":
                    np.asarray(
                        best["pred"],
                        dtype=np.float64,
                    ),
                "_planar_prediction":
                    np.asarray(
                        planar_prediction,
                        dtype=np.float64,
                    ),
                "_equibiaxial_prediction":
                    np.asarray(
                        biaxial_prediction,
                        dtype=np.float64,
                    ),
                "_full_response":
                    full_response,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "uniaxial_nrmse",
                "BIC",
                "n_fitted_parameters",
            ]
        )
        .reset_index(drop=True)
    )


# =============================================================================
# Corrected atlas-guided protocol-discrimination analysis
# =============================================================================

def mode_brush_shape(cfg, curve: np.ndarray) -> np.ndarray:
    """
    V68 standard-mode brush per unit relative noise.

    For UT/BT/SH, V68 uses |response| plus a small protocol-level peak floor.
    """
    curve = np.asarray(curve, dtype=np.float64)
    peak = max(float(np.max(np.abs(curve))), NORMALIZATION_FLOOR_KPA)
    return np.maximum(
        np.abs(curve) + float(cfg.mode_floor_per_unit_noise) * peak,
        NORMALIZATION_FLOOR_KPA,
    )


def fixed_pair_required_noise_fraction(
    cfg,
    response_a: np.ndarray,
    response_b: np.ndarray,
) -> float:
    """
    Symmetric bounded-tube distance between two fixed response curves.

    This is the standard-protocol analogue of V68.required_noise_fraction:
        max |a-b| / (h_a + h_b)
    """
    a = np.asarray(response_a, dtype=np.float64)
    b = np.asarray(response_b, dtype=np.float64)
    ha = mode_brush_shape(cfg, a)
    hb = mode_brush_shape(cfg, b)
    return float(
        np.max(
            np.abs(a - b)
            / np.maximum(ha + hb, NORMALIZATION_FLOOR_KPA)
        )
    )


def completeF_statewise_separation(
    V,
    cfg,
    response_a: np.ndarray,
    response_b: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return normalized complete-F separation per parent deformation state.

    Each parent state has two pressure-free stress-difference components. The
    state score is the maximum normalized disagreement over those two components.
    """
    a = np.asarray(response_a, dtype=np.float64)
    b = np.asarray(response_b, dtype=np.float64)

    idx = V68_PROTOCOL.atlas_indices
    ha = V.brush_shape_per_unit_noise(
        cfg, V68_PROTOCOL, a
    )[idx]
    hb = V.brush_shape_per_unit_noise(
        cfg, V68_PROTOCOL, b
    )[idx]

    component_score = (
        np.abs(a[idx] - b[idx])
        / np.maximum(ha + hb, NORMALIZATION_FLOOR_KPA)
    )
    component_score = component_score.reshape(
        V68_PROTOCOL.n_parent_states, 2
    )
    state_score = np.max(component_score, axis=1)
    return state_score, component_score


def shear_pointwise_predicted_separation(
    cfg,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
) -> np.ndarray:
    """
    Pointwise pair separation on the withheld shear protocol, using predictions
    only. No experimental shear stress enters this calculation.
    """
    pa = np.asarray(prediction_a, dtype=np.float64)
    pb = np.asarray(prediction_b, dtype=np.float64)
    ha = mode_brush_shape(cfg, pa)
    hb = mode_brush_shape(cfg, pb)
    return (
        np.abs(pa - pb)
        / np.maximum(ha + hb, NORMALIZATION_FLOOR_KPA)
    )


def experimental_discrimination_contrast(
    cfg,
    truth: np.ndarray,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
) -> np.ndarray:
    """
    How strongly each revealed experimental point favors one candidate over the
    other, independent of direction of preference.

    contrast = ||y-pa| - |y-pb|| / h_y
    """
    y = np.asarray(truth, dtype=np.float64)
    pa = np.asarray(prediction_a, dtype=np.float64)
    pb = np.asarray(prediction_b, dtype=np.float64)
    hy = mode_brush_shape(cfg, y)
    return (
        np.abs(np.abs(y - pa) - np.abs(y - pb))
        / np.maximum(hy, NORMALIZATION_FLOOR_KPA)
    )



def add_training_model_plausibility(
    fits: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add BIC and AICc support using UNAXIAL TRAINING DATA ONLY.

    Manuscript-primary support:
        Delta-BIC <= 2 AND Delta-AICc <= 2.

    Broader Delta-BIC <= 10 remains a supplementary robustness criterion.
    """
    out = fits.copy()

    best_bic = float(out["BIC"].min())
    out["training_delta_BIC"] = (
        out["BIC"].astype(float) - best_bic
    )
    out["training_relative_BIC_support"] = np.exp(
        -0.5 * out["training_delta_BIC"].astype(float)
    )

    finite_aicc = (
        out["AICc"]
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    if not finite_aicc.notna().any():
        raise RuntimeError("No finite AICc values are available.")
    best_aicc = float(finite_aicc.min())
    out["training_delta_AICc"] = (
        out["AICc"].astype(float) - best_aicc
    )

    for threshold in DELTA_BIC_SENSITIVITY:
        label = int(round(threshold))
        out[f"training_plausible_deltaBIC_{label}"] = (
            out["training_delta_BIC"].astype(float)
            <= float(threshold) + 1.0e-12
        )

    out["training_plausible_primary"] = (
        out["training_delta_BIC"].astype(float)
        <= PRIMARY_DELTA_BIC_MAX + 1.0e-12
    )

    out["publication_primary_supported"] = (
        (
            out["training_delta_BIC"].astype(float)
            <= PRIMARY_PUBLICATION_DELTA_BIC_MAX + 1.0e-12
        )
        & (
            out["training_delta_AICc"].astype(float)
            <= PRIMARY_PUBLICATION_DELTA_AICC_MAX + 1.0e-12
        )
    )

    return out



def build_pairwise_discrimination_table(
    V,
    cfg,
    fits: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Analyze all 21 fitted-family pairs using training fits and predictions only.

    No withheld experimental stress values are used in this function.
    """
    lookup = {
        str(row["model"]): row
        for _, row in fits.iterrows()
    }

    complete_matrix = pd.DataFrame(
        np.zeros(
            (len(MODELS), len(MODELS)),
            dtype=np.float64,
        ),
        index=MODELS,
        columns=MODELS,
    )

    rows = []

    for i, model_a in enumerate(MODELS):
        row_a = lookup[model_a]
        ut_a = np.asarray(
            row_a["_uniaxial_prediction"],
            dtype=np.float64,
        )
        full_a = np.asarray(
            row_a["_full_response"],
            dtype=np.float64,
        )
        planar_a = np.asarray(
            row_a["_planar_prediction"],
            dtype=np.float64,
        )
        biaxial_a = np.asarray(
            row_a["_equibiaxial_prediction"],
            dtype=np.float64,
        )

        for j in range(i + 1, len(MODELS)):
            model_b = MODELS[j]
            row_b = lookup[model_b]

            ut_b = np.asarray(
                row_b["_uniaxial_prediction"],
                dtype=np.float64,
            )
            full_b = np.asarray(
                row_b["_full_response"],
                dtype=np.float64,
            )
            planar_b = np.asarray(
                row_b["_planar_prediction"],
                dtype=np.float64,
            )
            biaxial_b = np.asarray(
                row_b["_equibiaxial_prediction"],
                dtype=np.float64,
            )

            ut_distance = (
                fixed_pair_required_noise_fraction(
                    cfg,
                    ut_a,
                    ut_b,
                )
            )
            complete_distance = float(
                V.required_noise_fraction(
                    cfg,
                    V68_PROTOCOL,
                    full_a,
                    full_b,
                )
            )

            complete_matrix.loc[
                model_a, model_b
            ] = (
                complete_matrix.loc[
                    model_b, model_a
                ]
            ) = 100.0 * complete_distance

            planar_sep = (
                pointwise_pair_separation(
                    cfg,
                    planar_a,
                    planar_b,
                )
            )
            biaxial_sep = (
                pointwise_pair_separation(
                    cfg,
                    biaxial_a,
                    biaxial_b,
                )
            )

            nrmse_a = float(
                row_a[
                    "uniaxial_nrmse_percent"
                ]
            )
            nrmse_b = float(
                row_b[
                    "uniaxial_nrmse_percent"
                ]
            )
            dbic_a = float(
                row_a["training_delta_BIC"]
            )
            dbic_b = float(
                row_b["training_delta_BIC"]
            )
            daicc_a = float(
                row_a["training_delta_AICc"]
            )
            daicc_b = float(
                row_b["training_delta_AICc"]
            )

            both_plausible = bool(
                row_a[
                    "training_plausible_primary"
                ]
                and row_b[
                    "training_plausible_primary"
                ]
            )

            ut_indistinguishable = bool(
                ut_distance
                <= PAIR_TOLERANCE_FRACTION
                + 1.0e-12
            )
            complete_distinguishable = bool(
                complete_distance
                > PAIR_TOLERANCE_FRACTION
                + 1.0e-12
            )

            rows.append(
                {
                    "model_a": model_a,
                    "model_b": model_b,
                    "uniaxial_pair_required_noise_fraction":
                        float(ut_distance),
                    "uniaxial_pair_required_noise_percent":
                        float(
                            100.0 * ut_distance
                        ),
                    "uniaxially_indistinguishable_at_3pct":
                        ut_indistinguishable,
                    "completeF_required_noise_fraction":
                        float(complete_distance),
                    "completeF_required_noise_percent":
                        float(
                            100.0
                            * complete_distance
                        ),
                    "completeF_distinguishable_at_3pct":
                        complete_distinguishable,
                    "hidden_completeF_divergence":
                        bool(
                            ut_indistinguishable
                            and complete_distinguishable
                        ),
                    "training_delta_BIC_a":
                        dbic_a,
                    "training_delta_BIC_b":
                        dbic_b,
                    "training_delta_AICc_a":
                        daicc_a,
                    "training_delta_AICc_b":
                        daicc_b,
                    "pair_max_delta_BIC":
                        float(
                            max(dbic_a, dbic_b)
                        ),
                    "pair_max_delta_AICc":
                        float(
                            max(daicc_a, daicc_b)
                        ),
                    "both_models_training_plausible":
                        both_plausible,
                    "both_models_publication_primary_supported":
                        bool(
                            row_a["publication_primary_supported"]
                            and row_b["publication_primary_supported"]
                        ),
                    "plausible_hidden_completeF_divergence":
                        bool(
                            both_plausible
                            and ut_indistinguishable
                            and complete_distinguishable
                        ),
                    "uniaxial_nrmse_a_percent":
                        nrmse_a,
                    "uniaxial_nrmse_b_percent":
                        nrmse_b,
                    "pair_worst_uniaxial_nrmse_percent":
                        float(
                            max(nrmse_a, nrmse_b)
                        ),
                    "pair_mean_uniaxial_nrmse_percent":
                        float(
                            0.5
                            * (nrmse_a + nrmse_b)
                        ),
                    "max_planar_prediction_separation_percent":
                        float(
                            100.0
                            * np.max(planar_sep)
                        ),
                    "max_equibiaxial_prediction_separation_percent":
                        float(
                            100.0
                            * np.max(biaxial_sep)
                        ),
                    "max_available_holdout_prediction_separation_percent":
                        float(
                            100.0
                            * max(
                                np.max(planar_sep),
                                np.max(biaxial_sep),
                            )
                        ),
                }
            )

    return complete_matrix, pd.DataFrame(rows)


def select_publication_primary_pair(
    pair_table: pd.DataFrame,
) -> pd.Series:
    """
    Recover the manuscript-primary pair AFTER the training-only model screen.

    Selection criterion:
      - both models satisfy Delta-BIC <= 2 AND Delta-AICc <= 2.

    Deliberately NOT used for selection:
      - fitted UT pair distance,
      - fitted complete-F pair distance,
      - planar/equibiaxial stresses,
      - frozen-atlas compatibility scores.

    The pair-table row is returned only so direct fitted-state quantities can be
    reported later as supplementary diagnostics.
    """
    candidates = pair_table[
        pair_table[
            "both_models_publication_primary_supported"
        ].astype(bool)
    ].copy()

    if len(candidates) != 1:
        cols = [
            "model_a",
            "model_b",
            "training_delta_BIC_a",
            "training_delta_BIC_b",
            "training_delta_AICc_a",
            "training_delta_AICc_b",
        ]
        raise RuntimeError(
            "Training-only BIC/AICc rule must yield exactly one model pair; "
            f"found {len(candidates)}: "
            f"{candidates[cols].to_dict(orient='records')}"
        )

    return candidates.iloc[0].copy()



def select_discriminating_pair(
    pair_table: pd.DataFrame,
) -> Tuple[pd.Series, str, bool]:
    """
    Pre-specified pair selection using training data + complete-F predictions.

    Primary validation pair must satisfy all three:
      1. both models: Delta-BIC <= 10 on uniaxial training;
      2. fixed fitted UT distance <= 3%;
      3. fixed fitted complete-F distance > 3%.

    Ranking does not use planar or equibiaxial experimental stresses.
    """
    plausible_pairs = pair_table[
        pair_table[
            "both_models_training_plausible"
        ].astype(bool)
    ].copy()

    hidden = plausible_pairs[
        plausible_pairs[
            "plausible_hidden_completeF_divergence"
        ].astype(bool)
    ].copy()

    if len(hidden):
        pool = hidden
        selection_class = (
            "PLAUSIBLE_UT_HIDDEN_COMPLETEF_DIVERGENCE"
        )
        eligible = True

    elif len(plausible_pairs):
        pool = plausible_pairs
        selection_class = (
            "NO_PLAUSIBLE_HIDDEN_DIVERGENCE__"
            "BEST_PLAUSIBLE_PAIR_DIAGNOSTIC_ONLY"
        )
        eligible = False

    else:
        pool = pair_table.copy()
        selection_class = (
            "FEWER_THAN_TWO_PLAUSIBLE_MODELS__"
            "TOP_SUPPORTED_PAIR_DIAGNOSTIC_ONLY"
        )
        eligible = False

    # Strongest statistical support first, then quality of the weaker training
    # fit, then complete-F separation. All criteria are training/prediction only.
    chosen = (
        pool.sort_values(
            [
                "pair_max_delta_BIC",
                "pair_worst_uniaxial_nrmse_percent",
                "pair_mean_uniaxial_nrmse_percent",
                "completeF_required_noise_percent",
                "model_a",
                "model_b",
            ],
            ascending=[
                True,
                True,
                True,
                False,
                True,
                True,
            ],
        )
        .iloc[0]
        .copy()
    )

    return chosen, selection_class, eligible


def pointwise_pair_separation(
    cfg,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
) -> np.ndarray:
    pa = np.asarray(
        prediction_a,
        dtype=np.float64,
    )
    pb = np.asarray(
        prediction_b,
        dtype=np.float64,
    )
    ha = mode_brush_shape(cfg, pa)
    hb = mode_brush_shape(cfg, pb)

    return (
        np.abs(pa - pb)
        / np.maximum(
            ha + hb,
            NORMALIZATION_FLOOR_KPA,
        )
    )


def empirical_discrimination_contrast(
    cfg,
    truth: np.ndarray,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
) -> np.ndarray:
    y = np.asarray(
        truth,
        dtype=np.float64,
    )
    pa = np.asarray(
        prediction_a,
        dtype=np.float64,
    )
    pb = np.asarray(
        prediction_b,
        dtype=np.float64,
    )
    hy = mode_brush_shape(cfg, y)

    return (
        np.abs(
            np.abs(y - pa)
            - np.abs(y - pb)
        )
        / np.maximum(
            hy,
            NORMALIZATION_FLOOR_KPA,
        )
    )


def analyze_selected_pair(
    V,
    cfg,
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    selected_pair: pd.Series,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Freeze a plausible hidden-divergence pair, choose witnesses without using
    withheld truth, then reveal planar + equibiaxial experimental stresses.
    """
    lookup = {
        str(row["model"]): row
        for _, row in fits.iterrows()
    }

    model_a = str(
        selected_pair["model_a"]
    )
    model_b = str(
        selected_pair["model_b"]
    )
    row_a = lookup[model_a]
    row_b = lookup[model_b]

    full_a = np.asarray(
        row_a["_full_response"],
        dtype=np.float64,
    )
    full_b = np.asarray(
        row_b["_full_response"],
        dtype=np.float64,
    )

    # ------------------------------------------------------------------
    # Global complete-F witness.
    # ------------------------------------------------------------------
    state_score, component_score = (
        completeF_statewise_separation(
            V,
            cfg,
            full_a,
            full_b,
        )
    )

    global_index = int(
        np.argmax(state_score)
    )
    component_index = int(
        np.argmax(
            component_score[global_index]
        )
    )
    global_stretches = np.asarray(
        V68_PROTOCOL.parent_principal_stretches[
            global_index
        ],
        dtype=np.float64,
    )

    # ------------------------------------------------------------------
    # Candidate measured points across BOTH withheld protocols.
    # Selection is prediction-only.
    # ------------------------------------------------------------------
    mode_records = []

    for mode, pred_key in [
        (
            "pure_shear_planar",
            "_planar_prediction",
        ),
        (
            "equibiaxial",
            "_equibiaxial_prediction",
        ),
    ]:
        df = dataset["modes"][mode]
        pred_a = np.asarray(
            row_a[pred_key],
            dtype=np.float64,
        )
        pred_b = np.asarray(
            row_b[pred_key],
            dtype=np.float64,
        )

        predicted_sep = (
            pointwise_pair_separation(
                cfg,
                pred_a,
                pred_b,
            )
        )

        for local_index in range(len(df)):
            mode_records.append(
                {
                    "mode": mode,
                    "local_index":
                        int(local_index),
                    "stretch":
                        float(
                            df.loc[
                                local_index,
                                "stretch",
                            ]
                        ),
                    "lambda1":
                        float(
                            df.loc[
                                local_index,
                                "lambda1",
                            ]
                        ),
                    "lambda2":
                        float(
                            df.loc[
                                local_index,
                                "lambda2",
                            ]
                        ),
                    "lambda3":
                        float(
                            df.loc[
                                local_index,
                                "lambda3",
                            ]
                        ),
                    "prediction_model_a_kPa":
                        float(
                            pred_a[
                                local_index
                            ]
                        ),
                    "prediction_model_b_kPa":
                        float(
                            pred_b[
                                local_index
                            ]
                        ),
                    "predicted_pair_separation_fraction":
                        float(
                            predicted_sep[
                                local_index
                            ]
                        ),
                }
            )

    witness_table = pd.DataFrame(
        mode_records
    )

    witness_row_index = int(
        witness_table[
            "predicted_pair_separation_fraction"
        ].to_numpy(float).argmax()
    )
    witness_mode = str(
        witness_table.loc[
            witness_row_index,
            "mode",
        ]
    )
    witness_local_index = int(
        witness_table.loc[
            witness_row_index,
            "local_index",
        ]
    )

    # ------------------------------------------------------------------
    # NOW reveal withheld experimental stresses.
    # ------------------------------------------------------------------
    all_predicted_sep = []
    all_empirical_contrast = []

    for mode, pred_key in [
        (
            "pure_shear_planar",
            "_planar_prediction",
        ),
        (
            "equibiaxial",
            "_equibiaxial_prediction",
        ),
    ]:
        df = dataset["modes"][mode]
        truth = df[
            "v68_tau13_kPa"
        ].to_numpy(float)
        pred_a = np.asarray(
            row_a[pred_key],
            dtype=np.float64,
        )
        pred_b = np.asarray(
            row_b[pred_key],
            dtype=np.float64,
        )

        sep = pointwise_pair_separation(
            cfg,
            pred_a,
            pred_b,
        )
        contrast = (
            empirical_discrimination_contrast(
                cfg,
                truth,
                pred_a,
                pred_b,
            )
        )

        all_predicted_sep.extend(
            sep.tolist()
        )
        all_empirical_contrast.extend(
            contrast.tolist()
        )

        mode_mask = (
            witness_table["mode"]
            == mode
        )
        idxs = witness_table.index[
            mode_mask
        ].to_numpy()

        witness_table.loc[
            idxs,
            "experimental_tau13_kPa",
        ] = truth
        witness_table.loc[
            idxs,
            "absolute_error_model_a_kPa",
        ] = np.abs(
            truth - pred_a
        )
        witness_table.loc[
            idxs,
            "absolute_error_model_b_kPa",
        ] = np.abs(
            truth - pred_b
        )
        witness_table.loc[
            idxs,
            "empirical_discrimination_contrast_fraction",
        ] = contrast

    witness_table[
        "predicted_pair_separation_percent"
    ] = (
        100.0
        * witness_table[
            "predicted_pair_separation_fraction"
        ]
    )
    witness_table[
        "empirical_discrimination_contrast_percent"
    ] = (
        100.0
        * witness_table[
            "empirical_discrimination_contrast_fraction"
        ]
    )
    witness_table[
        "preselected_measured_witness"
    ] = False
    witness_table.loc[
        witness_row_index,
        "preselected_measured_witness",
    ] = True

    wr = witness_table.loc[
        witness_row_index
    ]
    error_a = float(
        wr[
            "absolute_error_model_a_kPa"
        ]
    )
    error_b = float(
        wr[
            "absolute_error_model_b_kPa"
        ]
    )

    if error_a < error_b:
        witness_preferred = model_a
    elif error_b < error_a:
        witness_preferred = model_b
    else:
        witness_preferred = "TIE"

    combined_a = float(
        row_a[
            "heldout_combined_nrmse_percent"
        ]
    )
    combined_b = float(
        row_b[
            "heldout_combined_nrmse_percent"
        ]
    )

    if combined_a < combined_b:
        combined_preferred = model_a
    elif combined_b < combined_a:
        combined_preferred = model_b
    else:
        combined_preferred = "TIE"

    predicted_array = np.asarray(
        all_predicted_sep,
        dtype=np.float64,
    )
    empirical_array = np.asarray(
        all_empirical_contrast,
        dtype=np.float64,
    )

    if (
        len(predicted_array) >= 3
        and np.std(predicted_array) > 0.0
        and np.std(empirical_array) > 0.0
    ):
        rho, pvalue = spearmanr(
            predicted_array,
            empirical_array,
        )
        rho = float(rho)
        pvalue = float(pvalue)
    else:
        rho = float("nan")
        pvalue = float("nan")

    witness_contrast = float(
        wr[
            "empirical_discrimination_contrast_fraction"
        ]
    )
    witness_percentile = float(
        100.0
        * np.mean(
            empirical_array
            <= witness_contrast
        )
    )

    summary = {
        "model_a": model_a,
        "model_b": model_b,
        "training_delta_BIC_model_a":
            float(
                selected_pair[
                    "training_delta_BIC_a"
                ]
            ),
        "training_delta_BIC_model_b":
            float(
                selected_pair[
                    "training_delta_BIC_b"
                ]
            ),
        "both_models_training_plausible":
            bool(
                selected_pair[
                    "both_models_training_plausible"
                ]
            ),
        "uniaxial_pair_required_noise_percent":
            float(
                selected_pair[
                    "uniaxial_pair_required_noise_percent"
                ]
            ),
        "uniaxially_indistinguishable_at_3pct":
            bool(
                selected_pair[
                    "uniaxially_indistinguishable_at_3pct"
                ]
            ),
        "completeF_required_noise_percent":
            float(
                selected_pair[
                    "completeF_required_noise_percent"
                ]
            ),
        "hidden_completeF_divergence":
            bool(
                selected_pair[
                    "hidden_completeF_divergence"
                ]
            ),
        "global_completeF_witness_state_index":
            global_index,
        "global_completeF_witness_component_index":
            component_index,
        "global_completeF_witness_lambda1":
            float(global_stretches[0]),
        "global_completeF_witness_lambda2":
            float(global_stretches[1]),
        "global_completeF_witness_lambda3":
            float(global_stretches[2]),
        "global_completeF_witness_separation_percent":
            float(
                100.0
                * state_score[
                    global_index
                ]
            ),
        "preselected_measured_witness_mode":
            witness_mode,
        "preselected_measured_witness_local_index":
            witness_local_index,
        "preselected_measured_witness_stretch":
            float(wr["stretch"]),
        "preselected_measured_witness_lambda1":
            float(wr["lambda1"]),
        "preselected_measured_witness_lambda2":
            float(wr["lambda2"]),
        "preselected_measured_witness_lambda3":
            float(wr["lambda3"]),
        "preselected_measured_witness_predicted_separation_percent":
            float(
                wr[
                    "predicted_pair_separation_percent"
                ]
            ),
        "preselected_measured_witness_experimental_tau13_kPa":
            float(
                wr[
                    "experimental_tau13_kPa"
                ]
            ),
        "preselected_measured_witness_prediction_model_a_kPa":
            float(
                wr[
                    "prediction_model_a_kPa"
                ]
            ),
        "preselected_measured_witness_prediction_model_b_kPa":
            float(
                wr[
                    "prediction_model_b_kPa"
                ]
            ),
        "preselected_measured_witness_abs_error_model_a_kPa":
            error_a,
        "preselected_measured_witness_abs_error_model_b_kPa":
            error_b,
        "preselected_measured_witness_preferred_model":
            witness_preferred,
        "heldout_planar_nrmse_model_a_percent":
            float(
                row_a[
                    "heldout_planar_nrmse_percent"
                ]
            ),
        "heldout_planar_nrmse_model_b_percent":
            float(
                row_b[
                    "heldout_planar_nrmse_percent"
                ]
            ),
        "heldout_equibiaxial_nrmse_model_a_percent":
            float(
                row_a[
                    "heldout_equibiaxial_nrmse_percent"
                ]
            ),
        "heldout_equibiaxial_nrmse_model_b_percent":
            float(
                row_b[
                    "heldout_equibiaxial_nrmse_percent"
                ]
            ),
        "heldout_combined_nrmse_model_a_percent":
            combined_a,
        "heldout_combined_nrmse_model_b_percent":
            combined_b,
        "heldout_combined_preferred_model":
            combined_preferred,
        "witness_preference_matches_combined_holdout_preference":
            bool(
                witness_preferred
                == combined_preferred
                and witness_preferred
                != "TIE"
            ),
        "predicted_separation_vs_empirical_contrast_spearman_rho":
            rho,
        "predicted_separation_vs_empirical_contrast_pvalue":
            pvalue,
        "preselected_witness_empirical_contrast_percent":
            float(
                100.0
                * witness_contrast
            ),
        "preselected_witness_empirical_contrast_percentile":
            witness_percentile,
    }

    return witness_table, summary



# =============================================================================
# Publication-final all-pair and sensitivity audits
# =============================================================================

def build_plausibility_sensitivity_table(
    fits: pd.DataFrame,
    pair_table: pd.DataFrame,
) -> pd.DataFrame:
    """
    Sensitivity of the hidden-divergence conclusion to training-data support.

    Includes BIC-only thresholds 2, 6, and 10 plus the manuscript-primary
    joint Delta-BIC<=2 AND Delta-AICc<=2 rule.
    """
    rows = []

    for threshold in DELTA_BIC_SENSITIVITY:
        models = (
            fits.loc[
                fits["training_delta_BIC"].astype(float)
                <= float(threshold) + 1.0e-12,
                "model",
            ]
            .astype(str)
            .tolist()
        )
        model_set = set(models)
        subset = pair_table[
            pair_table.apply(
                lambda r: (
                    str(r["model_a"]) in model_set
                    and str(r["model_b"]) in model_set
                ),
                axis=1,
            )
        ].copy()
        hidden = subset[
            subset["hidden_completeF_divergence"].astype(bool)
        ]

        rows.append(
            {
                "criterion": f"BIC_only_delta<={threshold:g}",
                "delta_BIC_threshold": float(threshold),
                "delta_AICc_threshold": np.nan,
                "plausible_models": "|".join(models),
                "n_plausible_models": int(len(models)),
                "n_possible_pairs_within_plausible_set": int(len(subset)),
                "n_hidden_completeF_divergence_pairs": int(len(hidden)),
                "fraction_plausible_pairs_hidden_divergence":
                    float(len(hidden) / max(len(subset), 1)),
                "all_plausible_pairs_are_hidden_divergence":
                    bool(len(subset) > 0 and len(hidden) == len(subset)),
            }
        )

    strict_models = (
        fits.loc[
            fits["publication_primary_supported"].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )
    strict_set = set(strict_models)
    strict_subset = pair_table[
        pair_table.apply(
            lambda r: (
                str(r["model_a"]) in strict_set
                and str(r["model_b"]) in strict_set
            ),
            axis=1,
        )
    ].copy()
    strict_hidden = strict_subset[
        strict_subset["hidden_completeF_divergence"].astype(bool)
    ]

    rows.append(
        {
            "criterion": "JOINT_BIC_AICc_delta<=2",
            "delta_BIC_threshold": PRIMARY_PUBLICATION_DELTA_BIC_MAX,
            "delta_AICc_threshold": PRIMARY_PUBLICATION_DELTA_AICC_MAX,
            "plausible_models": "|".join(strict_models),
            "n_plausible_models": int(len(strict_models)),
            "n_possible_pairs_within_plausible_set": int(len(strict_subset)),
            "n_hidden_completeF_divergence_pairs": int(len(strict_hidden)),
            "fraction_plausible_pairs_hidden_divergence":
                float(len(strict_hidden) / max(len(strict_subset), 1)),
            "all_plausible_pairs_are_hidden_divergence":
                bool(
                    len(strict_subset) > 0
                    and len(strict_hidden) == len(strict_subset)
                ),
        }
    )

    return pd.DataFrame(rows)



def build_model_ranking_audit(
    fits: pd.DataFrame,
) -> pd.DataFrame:
    """
    Report training and withheld rankings for every family without using the
    withheld ranking for model inclusion.
    """
    out = fits[
        [
            "model",
            "uniaxial_nrmse_percent",
            "BIC",
            "AICc",
            "training_delta_BIC",
            "training_delta_AICc",
            "training_relative_BIC_support",
            "publication_primary_supported",
            "training_plausible_deltaBIC_2",
            "training_plausible_deltaBIC_6",
            "training_plausible_deltaBIC_10",
            "heldout_planar_nrmse_percent",
            "heldout_equibiaxial_nrmse_percent",
            "heldout_combined_nrmse_percent",
        ]
    ].copy()

    out["training_BIC_rank"] = (
        out["BIC"]
        .rank(method="min", ascending=True)
        .astype(int)
    )
    out["training_NRMSE_rank"] = (
        out["uniaxial_nrmse_percent"]
        .rank(method="min", ascending=True)
        .astype(int)
    )
    out["heldout_planar_NRMSE_rank"] = (
        out["heldout_planar_nrmse_percent"]
        .rank(method="min", ascending=True)
        .astype(int)
    )
    out["heldout_equibiaxial_NRMSE_rank"] = (
        out["heldout_equibiaxial_nrmse_percent"]
        .rank(method="min", ascending=True)
        .astype(int)
    )
    out["heldout_combined_NRMSE_rank"] = (
        out["heldout_combined_nrmse_percent"]
        .rank(method="min", ascending=True)
        .astype(int)
    )

    return out.sort_values(
        ["training_BIC_rank", "model"]
    ).reset_index(drop=True)


def audit_all_plausible_hidden_pairs(
    V,
    cfg,
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    pair_table: pd.DataFrame,
    outdir: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    """
    Audit EVERY pair that was qualified using training data and complete-F
    predictions before either withheld experimental stress dataset is consulted.

    No post-hoc pair selection is used in the aggregate conclusions.
    """
    qualified = (
        pair_table.loc[
            pair_table[
                "plausible_hidden_completeF_divergence"
            ].astype(bool)
        ]
        .copy()
        .sort_values(
            [
                "pair_max_delta_BIC",
                "pair_worst_uniaxial_nrmse_percent",
                "completeF_required_noise_percent",
                "model_a",
                "model_b",
            ],
            ascending=[True, True, False, True, True],
        )
        .reset_index(drop=True)
    )

    audit_rows = []
    witness_frames = []

    for pair_index, pair in qualified.iterrows():
        model_a = str(pair["model_a"])
        model_b = str(pair["model_b"])

        print(
            f"[V68.5.5] Publication audit pair "
            f"{pair_index + 1}/{len(qualified)}: "
            f"{model_a} vs {model_b}"
        )

        witness_table, summary = analyze_selected_pair(
            V,
            cfg,
            dataset,
            fits,
            pair,
        )

        pair_id = f"{model_a}__{model_b}"
        witness_table = witness_table.copy()
        witness_table.insert(0, "pair_id", pair_id)
        witness_table.insert(1, "model_a", model_a)
        witness_table.insert(2, "model_b", model_b)
        witness_frames.append(witness_table)

        audit_rows.append(
            {
                "pair_id": pair_id,
                "model_a": model_a,
                "model_b": model_b,
                "training_delta_BIC_model_a":
                    summary["training_delta_BIC_model_a"],
                "training_delta_BIC_model_b":
                    summary["training_delta_BIC_model_b"],
                "uniaxial_pair_required_noise_percent":
                    summary["uniaxial_pair_required_noise_percent"],
                "completeF_required_noise_percent":
                    summary["completeF_required_noise_percent"],
                "global_completeF_witness_lambda1":
                    summary["global_completeF_witness_lambda1"],
                "global_completeF_witness_lambda2":
                    summary["global_completeF_witness_lambda2"],
                "global_completeF_witness_lambda3":
                    summary["global_completeF_witness_lambda3"],
                "global_completeF_witness_separation_percent":
                    summary["global_completeF_witness_separation_percent"],
                "preselected_measured_witness_mode":
                    summary["preselected_measured_witness_mode"],
                "preselected_measured_witness_stretch":
                    summary["preselected_measured_witness_stretch"],
                "preselected_measured_witness_predicted_separation_percent":
                    summary["preselected_measured_witness_predicted_separation_percent"],
                "preselected_measured_witness_preferred_model":
                    summary["preselected_measured_witness_preferred_model"],
                "heldout_planar_nrmse_model_a_percent":
                    summary["heldout_planar_nrmse_model_a_percent"],
                "heldout_planar_nrmse_model_b_percent":
                    summary["heldout_planar_nrmse_model_b_percent"],
                "heldout_equibiaxial_nrmse_model_a_percent":
                    summary["heldout_equibiaxial_nrmse_model_a_percent"],
                "heldout_equibiaxial_nrmse_model_b_percent":
                    summary["heldout_equibiaxial_nrmse_model_b_percent"],
                "heldout_combined_nrmse_model_a_percent":
                    summary["heldout_combined_nrmse_model_a_percent"],
                "heldout_combined_nrmse_model_b_percent":
                    summary["heldout_combined_nrmse_model_b_percent"],
                "heldout_combined_preferred_model":
                    summary["heldout_combined_preferred_model"],
                "witness_preference_matches_combined_holdout_preference":
                    summary[
                        "witness_preference_matches_combined_holdout_preference"
                    ],
                "predicted_separation_vs_empirical_contrast_spearman_rho":
                    summary[
                        "predicted_separation_vs_empirical_contrast_spearman_rho"
                    ],
                "predicted_separation_vs_empirical_contrast_pvalue":
                    summary[
                        "predicted_separation_vs_empirical_contrast_pvalue"
                    ],
                "preselected_witness_empirical_contrast_percentile":
                    summary[
                        "preselected_witness_empirical_contrast_percentile"
                    ],
            }
        )

    audit = pd.DataFrame(audit_rows)
    witnesses = (
        pd.concat(witness_frames, ignore_index=True)
        if witness_frames
        else pd.DataFrame()
    )

    if len(audit):
        rho = audit[
            "predicted_separation_vs_empirical_contrast_spearman_rho"
        ].astype(float)

        aggregate = {
            "n_qualified_plausible_hidden_divergence_pairs":
                int(len(audit)),
            "n_witness_preferences_matching_combined_holdout":
                int(
                    audit[
                        "witness_preference_matches_combined_holdout_preference"
                    ].astype(bool).sum()
                ),
            "fraction_witness_preferences_matching_combined_holdout":
                float(
                    audit[
                        "witness_preference_matches_combined_holdout_preference"
                    ].astype(bool).mean()
                ),
            "n_preselected_witnesses_top_quartile_empirical_contrast":
                int(
                    (
                        audit[
                            "preselected_witness_empirical_contrast_percentile"
                        ].astype(float)
                        >= 75.0
                    ).sum()
                ),
            "fraction_preselected_witnesses_top_quartile_empirical_contrast":
                float(
                    (
                        audit[
                            "preselected_witness_empirical_contrast_percentile"
                        ].astype(float)
                        >= 75.0
                    ).mean()
                ),
            "n_preselected_witnesses_top_decile_empirical_contrast":
                int(
                    (
                        audit[
                            "preselected_witness_empirical_contrast_percentile"
                        ].astype(float)
                        >= 90.0
                    ).sum()
                ),
            "fraction_preselected_witnesses_top_decile_empirical_contrast":
                float(
                    (
                        audit[
                            "preselected_witness_empirical_contrast_percentile"
                        ].astype(float)
                        >= 90.0
                    ).mean()
                ),
            "mean_spearman_rho_prediction_separation_vs_empirical_contrast":
                float(np.nanmean(rho)),
            "minimum_spearman_rho_prediction_separation_vs_empirical_contrast":
                float(np.nanmin(rho)),
            "median_spearman_rho_prediction_separation_vs_empirical_contrast":
                float(np.nanmedian(rho)),
            "minimum_completeF_pair_distance_percent":
                float(
                    audit[
                        "completeF_required_noise_percent"
                    ].astype(float).min()
                ),
            "maximum_completeF_pair_distance_percent":
                float(
                    audit[
                        "completeF_required_noise_percent"
                    ].astype(float).max()
                ),
            "maximum_uniaxial_pair_distance_percent":
                float(
                    audit[
                        "uniaxial_pair_required_noise_percent"
                    ].astype(float).max()
                ),
            "all_qualified_pairs_remain_within_3pct_uniaxial":
                bool(
                    (
                        audit[
                            "uniaxial_pair_required_noise_percent"
                        ].astype(float)
                        <= 3.0 + 1.0e-10
                    ).all()
                ),
            "all_qualified_pairs_exceed_3pct_completeF":
                bool(
                    (
                        audit[
                            "completeF_required_noise_percent"
                        ].astype(float)
                        > 3.0
                    ).all()
                ),
        }
    else:
        aggregate = {
            "n_qualified_plausible_hidden_divergence_pairs": 0,
            "publication_all_pair_audit_available": False,
        }

    save_dataframe_checked(
        audit,
        outdir / "ALL_plausible_hidden_pair_holdout_audit.csv",
        index=False,
    )
    if len(witnesses):
        save_dataframe_checked(
            witnesses,
            outdir / "ALL_plausible_hidden_pair_witness_points.csv",
            index=False,
        )
    save_json_checked(
        aggregate,
        outdir / "ALL_plausible_hidden_pair_audit_summary.json",
    )

    return audit, witnesses, aggregate


def make_all_pair_audit_figure(
    audit: pd.DataFrame,
    sensitivity: pd.DataFrame,
    outdir: Path,
) -> None:
    if audit.empty:
        return

    labels = [
        f"{a}–{b}"
        for a, b in zip(
            audit["model_a"].astype(str),
            audit["model_b"].astype(str),
        )
    ]
    positions = np.arange(len(audit))

    fig = plt.figure(figsize=(14.5, 10.0))

    ax = fig.add_subplot(2, 2, 1)
    ax.bar(
        positions - 0.18,
        audit["uniaxial_pair_required_noise_percent"].to_numpy(float),
        width=0.36,
        label="UT distance",
    )
    ax.bar(
        positions + 0.18,
        audit["completeF_required_noise_percent"].to_numpy(float),
        width=0.36,
        label="Complete-F distance",
    )
    ax.axhline(3.0, linestyle="--", linewidth=1.3)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Required noise (%)")
    ax.set_title("A. Every plausible pair: UT ambiguity vs complete-F separation")
    ax.legend(fontsize=8)

    ax = fig.add_subplot(2, 2, 2)
    ax.bar(
        positions,
        audit[
            "predicted_separation_vs_empirical_contrast_spearman_rho"
        ].to_numpy(float),
    )
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylim(-1.05, 1.05)
    ax.set_ylabel("Spearman ρ")
    ax.set_title("B. Prediction-only separation vs experimental discrimination")

    ax = fig.add_subplot(2, 2, 3)
    ax.bar(
        positions,
        audit[
            "preselected_witness_empirical_contrast_percentile"
        ].to_numpy(float),
    )
    ax.axhline(75.0, linestyle="--", linewidth=1.3)
    ax.axhline(90.0, linestyle=":", linewidth=1.3)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylim(0.0, 105.0)
    ax.set_ylabel("Empirical discrimination percentile")
    ax.set_title("C. Preselected measured witness quality")

    ax = fig.add_subplot(2, 2, 4)
    sensitivity_plot = sensitivity[
        sensitivity["criterion"].astype(str).str.startswith("BIC_only")
    ].copy()
    x = sensitivity_plot["delta_BIC_threshold"].to_numpy(float)
    y = sensitivity_plot[
        "n_hidden_completeF_divergence_pairs"
    ].to_numpy(float)
    n_pairs = sensitivity_plot[
        "n_possible_pairs_within_plausible_set"
    ].to_numpy(float)
    ax.plot(x, y, marker="o", label="Hidden-divergence pairs")
    ax.plot(x, n_pairs, marker="s", label="All plausible pairs")
    ax.set_xlabel("ΔBIC plausibility threshold")
    ax.set_ylabel("Number of model pairs")
    ax.set_xticks(x)
    ax.set_title("D. Plausibility-threshold sensitivity")
    ax.legend(fontsize=8)

    fig.suptitle(
        "Treloar publication audit — all training-qualified model pairs",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0.01, 1, 0.96])

    save_figure_checked(
        fig,
        outdir / "SUPPLEMENTARY_Treloar_ALL_PAIR_audit.png",
        dpi=320,
        bbox_inches="tight",
    )
    save_figure_checked(
        fig,
        outdir / "SUPPLEMENTARY_Treloar_ALL_PAIR_audit.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


def build_publication_primary_summary_table(
    fits: pd.DataFrame,
    primary_summary: Mapping[str, object],
    atlas_source_lookup: Mapping[str, object],
    frozen_pair: Mapping[str, object],
    atlas_witness_summary: Mapping[str, object],
    atlas_witness_experimental_test: Mapping[str, object],
) -> pd.DataFrame:
    """Compact manuscript-facing table separating atlas and direct diagnostics."""
    lookup = {str(row["model"]): row for _, row in fits.iterrows()}
    model_a = str(primary_summary["model_a"])
    model_b = str(primary_summary["model_b"])
    source_model = str(frozen_pair["source_model"])
    target_model = str(frozen_pair["target_model"])

    records = []
    for model in (model_a, model_b):
        row = lookup[model]
        records.append({
            "row_type": "EXPERIMENT_FIT",
            "model": model,
            "UT_NRMSE_percent": float(row["uniaxial_nrmse_percent"]),
            "Delta_BIC": float(row["training_delta_BIC"]),
            "Delta_AICc": float(row["training_delta_AICc"]),
            "planar_holdout_NRMSE_percent": float(row["heldout_planar_nrmse_percent"]),
            "equibiaxial_holdout_NRMSE_percent": float(row["heldout_equibiaxial_nrmse_percent"]),
            "combined_holdout_NRMSE_percent": float(row["heldout_combined_nrmse_percent"]),
        })

    records.append({
        "row_type": "FROZEN_ATLAS_PRIMARY",
        "model": f"{source_model}_ATLAS_NODE_TO_{target_model}",
        "atlas_source_index": int(atlas_source_lookup["frozen_source_index"]),
        "atlas_projection_required_noise_percent": float(
            atlas_source_lookup["projection_required_noise_percent"]
        ),
        "atlas_certified_compatibility_set": str(
            atlas_source_lookup["certified_compatibility_set"]
        ),
        "atlas_certified_source_to_target_noise_percent": float(
            atlas_source_lookup["certified_scores_percent"][target_model]
        ),
        "atlas_global_witness_lambda1": float(frozen_pair["global_completeF_witness_lambda1"]),
        "atlas_global_witness_lambda2": float(frozen_pair["global_completeF_witness_lambda2"]),
        "atlas_global_witness_lambda3": float(frozen_pair["global_completeF_witness_lambda3"]),
        "atlas_selected_measured_witness_mode": str(
            atlas_witness_summary["atlas_selected_measured_witness_mode"]
        ),
        "atlas_selected_measured_witness_stretch": float(
            atlas_witness_summary["atlas_selected_measured_witness_stretch"]
        ),
        "atlas_selected_witness_preferred_experimental_fit": str(
            atlas_witness_experimental_test["atlas_selected_witness_preferred_experimental_fit"]
        ),
        "combined_holdout_preferred_primary_model": str(
            atlas_witness_experimental_test["combined_holdout_preferred_primary_model"]
        ),
        "atlas_witness_preference_matches_combined_holdout": bool(
            atlas_witness_experimental_test["atlas_witness_preference_matches_combined_holdout"]
        ),
        "direct_fitted_pair_completeF_distance_percent_DIAGNOSTIC_ONLY": float(
            primary_summary["completeF_required_noise_percent"]
        ),
    })
    return pd.DataFrame(records)




def make_publication_primary_figure(
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    primary_summary: Mapping[str, object],
    atlas_source_lookup: Mapping[str, object],
    frozen_pair: Mapping[str, object],
    atlas_witness_summary: Mapping[str, object],
    outdir: Path,
) -> None:
    """Main figure centered on training-only candidate retention and frozen-atlas lookup."""
    lookup = {str(row["model"]): row for _, row in fits.iterrows()}
    model_a = str(primary_summary["model_a"])
    model_b = str(primary_summary["model_b"])
    source_model = str(frozen_pair["source_model"])
    target_model = str(frozen_pair["target_model"])

    train = dataset["modes"]["uniaxial"]
    planar = dataset["modes"]["pure_shear_planar"]
    biaxial = dataset["modes"]["equibiaxial"]
    fig = plt.figure(figsize=(15.5, 5.5))

    ax = fig.add_subplot(1, 3, 1)
    ax.scatter(train["stretch"], train["v68_tau13_kPa"], s=44, label="Treloar UT data", zorder=8)
    for model in (model_a, model_b):
        row = lookup[model]
        ax.plot(
            train["stretch"], row["_uniaxial_prediction"], linewidth=2.6,
            label=(f"{model}: {float(row['uniaxial_nrmse_percent']):.2f}% NRMSE, "
                   f"ΔBIC={float(row['training_delta_BIC']):.2f}, "
                   f"ΔAICc={float(row['training_delta_AICc']):.2f}")
        )
    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title("A. Uniaxial calibration only")
    ax.legend(fontsize=7.5)

    ax = fig.add_subplot(1, 3, 2)
    atlas_score = float(atlas_source_lookup["certified_scores_percent"][target_model])
    projection = float(atlas_source_lookup["projection_required_noise_percent"])
    reverse_text = f"{source_model}→{target_model}"
    values = [projection, atlas_score]
    labels = ["Projection\nfit→atlas", f"Certified\n{reverse_text}"]
    ax.bar(np.arange(2), values)
    ax.axhline(3.0, linestyle="--", linewidth=1.3, label="3% compatibility threshold")
    ax.set_xticks(np.arange(2)); ax.set_xticklabels(labels)
    ax.set_ylabel("Normalized required noise (%)")
    ax.set_title(
        "B. Frozen V68 atlas lookup\n"
        f"{source_model} node #{atlas_source_lookup['frozen_source_index']}: "
        f"{atlas_source_lookup['certified_compatibility_set']}\n"
        f"stored witness = ({frozen_pair['global_completeF_witness_lambda1']:.2f}, "
        f"{frozen_pair['global_completeF_witness_lambda2']:.2f}, "
        f"{frozen_pair['global_completeF_witness_lambda3']:.2f})"
    )
    ax.legend(fontsize=8)

    ax = fig.add_subplot(1, 3, 3)
    mode = str(atlas_witness_summary["atlas_selected_measured_witness_mode"])
    witness_stretch = float(atlas_witness_summary["atlas_selected_measured_witness_stretch"])
    if mode == "equibiaxial":
        frame=biaxial; pred_key="_equibiaxial_prediction"; x_label="Equibiaxial stretch, λ"; title_mode="equibiaxial"
    elif mode == "pure_shear_planar":
        frame=planar; pred_key="_planar_prediction"; x_label="Planar/pure-shear stretch, λ"; title_mode="planar"
    else:
        raise KeyError(mode)
    ax.scatter(frame["stretch"], frame["v68_tau13_kPa"], s=44, label=f"Withheld {title_mode} data", zorder=8)
    for model in (model_a, model_b):
        ax.plot(frame["stretch"], lookup[model][pred_key], linewidth=2.6, label=f"{model} UT-fit prediction")
    selected_row = frame[np.isclose(frame["stretch"].to_numpy(float), witness_stretch, rtol=0.0, atol=1.0e-12)]
    if len(selected_row) != 1:
        raise RuntimeError("Atlas-selected measured witness stretch was not found uniquely.")
    ax.scatter([witness_stretch], [float(selected_row["v68_tau13_kPa"].iloc[0])], s=175, marker="*", zorder=10, label="frozen-atlas selected witness")
    ax.set_xlabel(x_label); ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title(
        "C. Independent holdout at atlas-guided witness\n"
        f"atlas-predicted separation = {atlas_witness_summary['atlas_selected_measured_witness_predicted_separation_percent']:.1f}%"
    )
    ax.legend(fontsize=7.5)
    fig.suptitle("Treloar external validation of the frozen V68 compatibility atlas", fontsize=14)
    fig.tight_layout(rect=[0,0.01,1,0.93])
    save_figure_checked(fig, outdir/"Treloar_PUBLICATION_PRIMARY_FROZEN_ATLAS.png", dpi=350, bbox_inches="tight")
    save_figure_checked(fig, outdir/"Treloar_PUBLICATION_PRIMARY_FROZEN_ATLAS.pdf", bbox_inches="tight")
    plt.show(); plt.close(fig)





# =============================================================================
# Plotting
# =============================================================================

def make_validation_figure(
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    pair_table: pd.DataFrame,
    selected_pair: pd.Series,
    witness_table: pd.DataFrame,
    summary: Mapping[str, object],
    outdir: Path,
) -> None:
    lookup = {
        str(row["model"]): row
        for _, row in fits.iterrows()
    }

    model_a = str(
        selected_pair["model_a"]
    )
    model_b = str(
        selected_pair["model_b"]
    )

    train = dataset["modes"]["uniaxial"]
    planar = dataset["modes"]["pure_shear_planar"]
    biaxial = dataset["modes"]["equibiaxial"]

    fig = plt.figure(
        figsize=(15.2, 10.8)
    )

    # A — training
    ax = fig.add_subplot(2, 2, 1)
    ax.scatter(
        train["stretch"],
        train["v68_tau13_kPa"],
        s=42,
        label="Treloar UT training",
        zorder=8,
    )

    for _, row in fits.iterrows():
        model = str(row["model"])
        plausible = bool(
            row["training_plausible_primary"]
        )

        if model in {model_a, model_b}:
            ax.plot(
                train["stretch"],
                row["_uniaxial_prediction"],
                linewidth=2.8,
                label=(
                    f"{model}: "
                    f"{float(row['uniaxial_nrmse_percent']):.2f}%"
                    f", ΔBIC={float(row['training_delta_BIC']):.2f}"
                ),
            )
        elif plausible:
            ax.plot(
                train["stretch"],
                row["_uniaxial_prediction"],
                linewidth=1.8,
                alpha=0.75,
                label=(
                    f"{model} plausible"
                ),
            )
        else:
            ax.plot(
                train["stretch"],
                row["_uniaxial_prediction"],
                linewidth=0.9,
                alpha=0.22,
            )

    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel(
        r"$\tau_{11}-\tau_{33}$ (kPa)"
    )
    ax.set_title(
        "A. Calibration only: uniaxial tension"
    )
    ax.legend(fontsize=8)

    # B — withheld planar/pure shear
    ax = fig.add_subplot(2, 2, 2)
    ax.scatter(
        planar["stretch"],
        planar["v68_tau13_kPa"],
        s=42,
        label="Withheld planar experiment",
        zorder=8,
    )
    ax.plot(
        planar["stretch"],
        lookup[model_a][
            "_planar_prediction"
        ],
        linewidth=2.6,
        label=(
            f"{model_a}: "
            f"{float(lookup[model_a]['heldout_planar_nrmse_percent']):.2f}%"
        ),
    )
    ax.plot(
        planar["stretch"],
        lookup[model_b][
            "_planar_prediction"
        ],
        linewidth=2.6,
        label=(
            f"{model_b}: "
            f"{float(lookup[model_b]['heldout_planar_nrmse_percent']):.2f}%"
        ),
    )

    witness = witness_table[
        witness_table[
            "preselected_measured_witness"
        ].astype(bool)
    ].iloc[0]
    if (
        str(witness["mode"])
        == "pure_shear_planar"
    ):
        ax.scatter(
            [float(witness["stretch"])],
            [
                float(
                    witness[
                        "experimental_tau13_kPa"
                    ]
                )
            ],
            s=150,
            marker="*",
            zorder=10,
            label="preselected witness",
        )

    ax.set_xlabel(
        "Planar/pure-shear stretch, λ"
    )
    ax.set_ylabel(
        r"$\tau_{11}-\tau_{33}$ (kPa)"
    )
    ax.set_title(
        "B. Withheld planar/pure-shear experiment"
    )
    ax.legend(fontsize=8)

    # C — withheld equibiaxial
    ax = fig.add_subplot(2, 2, 3)
    ax.scatter(
        biaxial["stretch"],
        biaxial["v68_tau13_kPa"],
        s=42,
        label="Withheld equibiaxial experiment",
        zorder=8,
    )
    ax.plot(
        biaxial["stretch"],
        lookup[model_a][
            "_equibiaxial_prediction"
        ],
        linewidth=2.6,
        label=(
            f"{model_a}: "
            f"{float(lookup[model_a]['heldout_equibiaxial_nrmse_percent']):.2f}%"
        ),
    )
    ax.plot(
        biaxial["stretch"],
        lookup[model_b][
            "_equibiaxial_prediction"
        ],
        linewidth=2.6,
        label=(
            f"{model_b}: "
            f"{float(lookup[model_b]['heldout_equibiaxial_nrmse_percent']):.2f}%"
        ),
    )

    if (
        str(witness["mode"])
        == "equibiaxial"
    ):
        ax.scatter(
            [float(witness["stretch"])],
            [
                float(
                    witness[
                        "experimental_tau13_kPa"
                    ]
                )
            ],
            s=150,
            marker="*",
            zorder=10,
            label="preselected witness",
        )

    ax.set_xlabel(
        "Equibiaxial stretch, λ"
    )
    ax.set_ylabel(
        r"$\tau_{11}-\tau_{33}$ (kPa)"
    )
    ax.set_title(
        "C. Withheld equibiaxial experiment"
    )
    ax.legend(fontsize=8)

    # D — pair geometry
    ax = fig.add_subplot(2, 2, 4)

    x = pair_table[
        "uniaxial_pair_required_noise_percent"
    ].to_numpy(float)
    y = pair_table[
        "completeF_required_noise_percent"
    ].to_numpy(float)

    plausible_hidden = pair_table[
        "plausible_hidden_completeF_divergence"
    ].astype(bool).to_numpy()

    plausible_pair = pair_table[
        "both_models_training_plausible"
    ].astype(bool).to_numpy()

    ax.scatter(
        x[~plausible_pair],
        y[~plausible_pair],
        s=38,
        alpha=0.22,
        label="Pair contains implausible model",
    )
    ax.scatter(
        x[
            plausible_pair
            & ~plausible_hidden
        ],
        y[
            plausible_pair
            & ~plausible_hidden
        ],
        s=70,
        alpha=0.8,
        label="Plausible pair",
    )
    if np.any(plausible_hidden):
        ax.scatter(
            x[plausible_hidden],
            y[plausible_hidden],
            s=92,
            marker="D",
            label="Plausible hidden-divergence pair",
        )

    selected_mask = (
        (
            pair_table["model_a"]
            == model_a
        )
        & (
            pair_table["model_b"]
            == model_b
        )
    ) | (
        (
            pair_table["model_a"]
            == model_b
        )
        & (
            pair_table["model_b"]
            == model_a
        )
    )
    sr = pair_table[
        selected_mask
    ].iloc[0]

    ax.scatter(
        [
            float(
                sr[
                    "uniaxial_pair_required_noise_percent"
                ]
            )
        ],
        [
            float(
                sr[
                    "completeF_required_noise_percent"
                ]
            )
        ],
        s=210,
        marker="*",
        edgecolor="black",
        linewidth=0.9,
        zorder=10,
        label=(
            f"Selected: {model_a}/{model_b}"
        ),
    )

    ax.axvline(
        100.0
        * PAIR_TOLERANCE_FRACTION,
        linestyle="--",
        linewidth=1.2,
    )
    ax.axhline(
        100.0
        * PAIR_TOLERANCE_FRACTION,
        linestyle="--",
        linewidth=1.2,
    )
    ax.set_xlabel(
        "Fixed fitted uniaxial pair distance (%)"
    )
    ax.set_ylabel(
        "Fixed fitted complete-F distance (%)"
    )
    ax.set_title(
        "D. Hidden constitutive divergence"
    )
    ax.legend(fontsize=8)

    fig.suptitle(
        "Treloar external validation: "
        "uniaxial calibration → planar + biaxial holdout\n"
        f"{model_a} vs {model_b}: UT "
        f"{summary['uniaxial_pair_required_noise_percent']:.2f}% "
        "→ complete-F "
        f"{summary['completeF_required_noise_percent']:.2f}%",
        fontsize=14,
    )

    fig.tight_layout(
        rect=[0, 0.01, 1, 0.94]
    )

    save_figure_checked(
        fig,
        outdir
        / "SUPPLEMENTARY_Treloar_diagnostic_validation.png",
        dpi=320,
        bbox_inches="tight",
    )
    save_figure_checked(
        fig,
        outdir
        / "SUPPLEMENTARY_Treloar_diagnostic_validation.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


# =============================================================================
# Validation
# =============================================================================

def validate_treloar(
    V,
    cfg,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    require_verified_drive_output()
    print("\n" + "=" * 100)
    print(
        "[V68.5.5] TRELOAR PUBLIC EXPERIMENTAL VALIDATION"
    )
    print("=" * 100)

    # Load and verify the PRECOMPUTED frozen atlas before touching the
    # experimental dataset. This is an actual atlas-lookup validation.
    frozen_assets = load_and_verify_frozen_atlas_assets(
        V
    )

    dataset = load_treloar_dataset(
        cfg,
        OUTDIR,
    )

    provenance_audit = build_treloar_provenance_audit(
        dataset
    )
    save_json_checked(
        provenance_audit,
        OUTDIR / "Treloar_data_provenance_and_unit_audit.json",
    )

    train = dataset["modes"]["uniaxial"]
    planar = dataset["modes"]["pure_shear_planar"]
    biaxial = dataset["modes"]["equibiaxial"]

    print(
        "[V68.5.5] V68-domain points retained: "
        f"UT={len(train)}, "
        f"planar={len(planar)}, "
        f"equibiaxial={len(biaxial)}"
    )
    print(
        "[V68.5.5] Training stresses: UNAXIAL ONLY. "
        "Planar and equibiaxial stresses are withheld."
    )

    # ------------------------------------------------------------------
    # 1. Fit every V68 family using uniaxial data only.
    # ------------------------------------------------------------------
    fits = fit_all_models_to_uniaxial(
        V,
        cfg,
        dataset,
    )
    fits = add_training_model_plausibility(
        fits
    )

    plausible_2 = (
        fits.loc[
            fits[
                "training_plausible_deltaBIC_2"
            ].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )
    plausible_6 = (
        fits.loc[
            fits[
                "training_plausible_deltaBIC_6"
            ].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )
    plausible_10 = (
        fits.loc[
            fits[
                "training_plausible_deltaBIC_10"
            ].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )
    publication_primary_models = (
        fits.loc[
            fits["publication_primary_supported"].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )

    # Freeze the primary candidates and direction from UT training statistics ONLY.
    primary_training_selection = select_publication_primary_models_from_fits(fits)
    primary_source_model = str(primary_training_selection["source_model"])
    primary_target_model = str(primary_training_selection["target_model"])
    if publication_primary_models != primary_training_selection["supported_models"]:
        # Ordering can differ; set equality is what matters.
        if set(publication_primary_models) != set(primary_training_selection["supported_models"]):
            raise RuntimeError("Training-only primary model screens disagree.")
    save_json_checked(
        primary_training_selection,
        OUTDIR / "PUBLICATION_PRIMARY_training_only_selection.json",
    )
    print(
        "[V68.5.5] Training-only primary candidates frozen BEFORE any "
        f"complete-F pair diagnostic: {primary_source_model} -> {primary_target_model}"
    )

    # ------------------------------------------------------------------
    # 2. Pair geometry — still blind to withheld truth.
    # ------------------------------------------------------------------
    complete_matrix, pair_table = (
        build_pairwise_discrimination_table(
            V,
            cfg,
            fits,
        )
    )
    save_dataframe_checked(
        complete_matrix,
        OUTDIR
        / "pairwise_completeF_distance_percent.csv",
        index=True,
    )
    save_dataframe_checked(
        pair_table,
        OUTDIR
        / "pairwise_protocol_discrimination.csv",
        index=False,
    )

    sensitivity_table = build_plausibility_sensitivity_table(
        fits,
        pair_table,
    )
    save_dataframe_checked(
        sensitivity_table,
        OUTDIR / "plausibility_deltaBIC_sensitivity.csv",
        index=False,
    )

    model_ranking_audit = build_model_ranking_audit(
        fits
    )
    save_dataframe_checked(
        model_ranking_audit,
        OUTDIR / "model_training_vs_holdout_ranking_audit.csv",
        index=False,
    )

    manuscript_model_support = (
        fits[
            [
                "model",
                "uniaxial_nrmse_percent",
                "BIC",
                "training_delta_BIC",
                "AICc",
                "training_delta_AICc",
                "publication_primary_supported",
                "heldout_planar_nrmse_percent",
                "heldout_equibiaxial_nrmse_percent",
                "heldout_combined_nrmse_percent",
            ]
        ]
        .copy()
        .sort_values(["training_delta_BIC", "model"])
    )
    save_dataframe_checked(
        manuscript_model_support,
        OUTDIR / "PUBLICATION_model_support_and_holdout_table.csv",
        index=False,
    )

    primary_pair = select_publication_primary_pair(
        pair_table
    )

    # ------------------------------------------------------------------
    # 3. ACTUAL FROZEN-ATLAS LOOKUP.
    # ------------------------------------------------------------------
    fit_lookup = {
        str(row["model"]): row
        for _, row in fits.iterrows()
    }

    atlas_lookup_source = project_experimental_fit_to_frozen_atlas(
        V, cfg, fit_lookup[primary_source_model], frozen_assets["certified"]
    )
    atlas_lookup_target = project_experimental_fit_to_frozen_atlas(
        V, cfg, fit_lookup[primary_target_model], frozen_assets["certified"]
    )
    atlas_lookup_by_model = {
        primary_source_model: atlas_lookup_source,
        primary_target_model: atlas_lookup_target,
    }
    atlas_lookup_public = {
        model: {k: v for k, v in lookup.items() if not str(k).startswith("_")}
        for model, lookup in atlas_lookup_by_model.items()
    }
    save_json_checked(
        atlas_lookup_public,
        OUTDIR / "FROZEN_ATLAS_primary_same_family_lookup.json",
    )
    save_dataframe_checked(
        pd.DataFrame([
            {
                "experimental_model": item["experimental_model"],
                "experimental_mu0_kpa": item["experimental_mu0_kpa"],
                "frozen_source_model": item["frozen_source_model"],
                "frozen_source_index": item["frozen_source_index"],
                "frozen_coordinate_json": item["frozen_coordinate_json"],
                "projection_required_noise_percent": item["projection_required_noise_percent"],
                "certified_primary_region": item["certified_primary_region"],
                "certified_compatibility_set": item["certified_compatibility_set"],
                "certified_minimal_adequate_models": item["certified_minimal_adequate_models"],
                "certified_noise_to_primary_source_percent": item["certified_scores_percent"].get(primary_source_model, np.nan),
                "certified_noise_to_primary_target_percent": item["certified_scores_percent"].get(primary_target_model, np.nan),
            }
            for item in atlas_lookup_by_model.values()
        ]),
        OUTDIR / "FROZEN_ATLAS_primary_lookup_table.csv",
        index=False,
    )

    # Post-selection local transfer stability audit. This NEVER changes the pair.
    local_nearest_source, local_within_source, local_summary_source = audit_local_frozen_atlas_transfer(
        V, cfg, fit_lookup[primary_source_model], frozen_assets["certified"],
        primary_target_model, nearest_k=10,
    )
    local_nearest_target, local_within_target, local_summary_target = audit_local_frozen_atlas_transfer(
        V, cfg, fit_lookup[primary_target_model], frozen_assets["certified"],
        primary_source_model, nearest_k=10,
    )
    save_dataframe_checked(
        local_nearest_source,
        OUTDIR / f"FROZEN_ATLAS_local_transfer_nearest10_{primary_source_model}_to_{primary_target_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_within_source,
        OUTDIR / f"FROZEN_ATLAS_local_transfer_within3pct_{primary_source_model}_to_{primary_target_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_nearest_target,
        OUTDIR / f"FROZEN_ATLAS_local_transfer_nearest10_{primary_target_model}_to_{primary_source_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_within_target,
        OUTDIR / f"FROZEN_ATLAS_local_transfer_within3pct_{primary_target_model}_to_{primary_source_model}.csv",
        index=False,
    )
    local_transfer_summary = {
        f"{primary_source_model}_to_{primary_target_model}": local_summary_source,
        f"{primary_target_model}_to_{primary_source_model}": local_summary_target,
    }
    save_json_checked(
        local_transfer_summary,
        OUTDIR / "FROZEN_ATLAS_local_transfer_stability.json",
    )

    # Direction is already frozen from training-only BIC/AICc support.
    primary_atlas_source_lookup = atlas_lookup_source
    frozen_pair = frozen_pairwise_witness_for_source_node(
        V, cfg, frozen_assets, primary_atlas_source_lookup, primary_target_model
    )

    # Inject frozen source shape/scale for measured-protocol witness evaluation.
    frozen_source_row = primary_atlas_source_lookup[
        "_frozen_row"
    ]
    frozen_pair[
        "_source_coordinate"
    ] = _coordinate_json(
        frozen_source_row[
            "coordinate_json"
        ]
    )
    frozen_pair[
        "_source_scale"
    ] = float(
        fit_lookup[primary_source_model][
            "best_response_scale_kpa"
        ]
    )

    atlas_witness_table, atlas_witness_summary = (
        select_measured_witness_from_frozen_atlas_pair(
            V,
            cfg,
            dataset,
            frozen_pair,
        )
    )
    save_dataframe_checked(
        atlas_witness_table,
        OUTDIR
        / "FROZEN_ATLAS_primary_measured_witness_selection.csv",
        index=False,
    )

    atlas_witness_experimental_test = (
        evaluate_experimental_primary_models_at_atlas_selected_witness(
            dataset,
            fits,
            atlas_witness_summary,
            primary_training_selection["supported_models"],
        )
    )

    frozen_pair_public = {
        key: value
        for key, value
        in frozen_pair.items()
        if not str(key).startswith("_")
    }
    save_json_checked(
        {
            "atlas_source_lookup":
                {
                    key: value
                    for key, value
                    in primary_atlas_source_lookup.items()
                    if not str(key).startswith("_")
                },
            "frozen_directed_pairwise_witness":
                frozen_pair_public,
            "atlas_selected_measured_witness":
                atlas_witness_summary,
            "experimental_test_at_atlas_selected_witness":
                atlas_witness_experimental_test,
        },
        OUTDIR
        / "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json",
    )

    # Publication gate checks the frozen-atlas prediction AFTER training-only pair freezing.
    atlas_source_to_target = float(
        primary_atlas_source_lookup["certified_scores_percent"][primary_target_model]
    )
    atlas_lookup_primary_pass = bool(
        atlas_source_to_target > 100.0 * ATLAS_TOLERANCE_FRACTION
    )
    if not atlas_lookup_primary_pass:
        raise RuntimeError(
            f"Frozen atlas does not exclude {primary_target_model} from the nearest "
            f"{primary_source_model} source node at 3%; the intended discrimination "
            "claim is therefore not supported."
        )

    print("[V68.5.5] PRIMARY FROZEN-ATLAS LOOKUP:")
    print(
        f"           experimental {primary_source_model} -> frozen {primary_source_model} node "
        f"#{atlas_lookup_source['frozen_source_index']} "
        f"(projection {atlas_lookup_source['projection_required_noise_percent']:.4f}%)"
    )
    print(
        f"           certified source-node compatibility set: "
        f"{atlas_lookup_source['certified_compatibility_set']}"
    )
    print(
        f"           certified {primary_source_model} -> {primary_target_model} critical noise: "
        f"{atlas_source_to_target:.3f}%"
    )
    print(
        f"           local transfer audit within 3%: n={local_summary_source['n_nodes_within_localization_threshold']}, "
        f"all exclude target={local_summary_source['all_local_nodes_exclude_target_at_3pct']}"
    )
    print(
        f"           frozen atlas measured witness: "
        f"{atlas_witness_summary['atlas_selected_measured_witness_mode']} "
        f"stretch="
        f"{atlas_witness_summary['atlas_selected_measured_witness_stretch']:.3f}"
    )

    # Broad Delta-BIC<=10 pair remains supplementary only.
    selected_pair, selection_class, eligible = (
        select_discriminating_pair(
            pair_table
        )
    )

    # ------------------------------------------------------------------
    # 3. Choose witnesses from predictions; then reveal withheld truth.
    # ------------------------------------------------------------------
    witness_table, pair_summary = (
        analyze_selected_pair(
            V,
            cfg,
            dataset,
            fits,
            primary_pair,
        )
    )
    save_dataframe_checked(
        witness_table,
        OUTDIR
        / "withheld_measured_witness_validation.csv",
        index=False,
    )

    # Publication-final audit: ALL pairs were qualified before revealing
    # either withheld experimental dataset.
    all_pair_audit, all_pair_witnesses, all_pair_aggregate = (
        audit_all_plausible_hidden_pairs(
            V,
            cfg,
            dataset,
            fits,
            pair_table,
            OUTDIR,
        )
    )

    make_all_pair_audit_figure(
        all_pair_audit,
        sensitivity_table,
        OUTDIR,
    )

    publication_primary_table = (
        build_publication_primary_summary_table(
            fits,
            pair_summary,
            primary_atlas_source_lookup,
            frozen_pair,
            atlas_witness_summary,
            atlas_witness_experimental_test,
        )
    )
    save_dataframe_checked(
        publication_primary_table,
        OUTDIR / "PUBLICATION_PRIMARY_FROZEN_ATLAS_summary.csv",
        index=False,
    )
    make_publication_primary_figure(
        dataset,
        fits,
        pair_summary,
        primary_atlas_source_lookup,
        frozen_pair,
        atlas_witness_summary,
        OUTDIR,
    )

    n_plausible_pairs = int(
        pair_table[
            "both_models_training_plausible"
        ].astype(bool).sum()
    )
    n_plausible_hidden = int(
        pair_table[
            "plausible_hidden_completeF_divergence"
        ].astype(bool).sum()
    )

    best_bic_model = str(
        fits.sort_values(
            [
                "BIC",
                "uniaxial_nrmse",
                "n_fitted_parameters",
            ]
        ).iloc[0]["model"]
    )
    best_nrmse_model = str(
        fits.sort_values(
            [
                "uniaxial_nrmse",
                "BIC",
                "n_fitted_parameters",
            ]
        ).iloc[0]["model"]
    )
    best_holdout_model = str(
        fits.sort_values(
            [
                "heldout_combined_nrmse",
                "uniaxial_nrmse",
            ]
        ).iloc[0]["model"]
    )

    plausible_holdout_pool = fits.loc[
        fits["training_plausible_primary"].astype(bool)
    ].copy()
    best_plausible_holdout_model = str(
        plausible_holdout_pool.sort_values(
            [
                "heldout_combined_nrmse",
                "uniaxial_nrmse",
            ]
        ).iloc[0]["model"]
    )

    if (
        len(fits) >= 3
        and np.std(fits["BIC"].to_numpy(float)) > 0
        and np.std(
            fits["heldout_combined_nrmse_percent"].to_numpy(float)
        ) > 0
    ):
        ranking_rho, ranking_p = spearmanr(
            -fits["BIC"].to_numpy(float),
            -fits["heldout_combined_nrmse_percent"].to_numpy(float),
        )
        ranking_rho = float(ranking_rho)
        ranking_p = float(ranking_p)
    else:
        ranking_rho = float("nan")
        ranking_p = float("nan")

    summary = {
        "dataset":
            "Treloar vulcanized rubber",
        "source_repository":
            "llamm-de/thermalCANN",
        "source_variant":
            "Steinmann reproduction of Treloar data",
        "Treloar_original_DOI":
            TRELOAR_ORIGINAL_DOI,
        "Steinmann_reproduction_DOI":
            STEINMANN_REPRODUCTION_DOI,
        "stress_unit_provenance":
            STRESS_UNIT_PROVENANCE_NOTE,
        "stress_input_measure":
            "first Piola-Kirchhoff P11",
        "source_stress_unit":
            "MPa",
        "analysis_stress_unit":
            "kPa",
        "training_protocol":
            "uniaxial tension only",
        "heldout_protocols": [
            "pure shear / planar tension",
            "equibiaxial tension",
        ],
        "v68_parent_principal_stretch_bounds":
            [
                float(
                    cfg.parent_principal_stretch_min
                ),
                float(
                    cfg.parent_principal_stretch_max
                ),
            ],
        "n_uniaxial_training_points":
            int(len(train)),
        "n_planar_heldout_points":
            int(len(planar)),
        "n_equibiaxial_heldout_points":
            int(len(biaxial)),
        "primary_training_plausibility_deltaBIC_max":
            PRIMARY_DELTA_BIC_MAX,
        "publication_primary_deltaBIC_max":
            PRIMARY_PUBLICATION_DELTA_BIC_MAX,
        "publication_primary_deltaAICc_max":
            PRIMARY_PUBLICATION_DELTA_AICC_MAX,
        "publication_primary_supported_models":
            publication_primary_models,
        "publication_primary_pair":
            list(primary_training_selection["supported_models"]),
        "publication_primary_training_only_selection":
            primary_training_selection,
        "frozen_atlas_local_transfer_stability":
            local_transfer_summary,
        "primary_result_type":
            "FROZEN_CERTIFIED_ATLAS_LOOKUP",
        "frozen_atlas_integrity":
            frozen_assets["audit"],
        "frozen_atlas_primary_source_lookup":
            {
                key: value
                for key, value
                in primary_atlas_source_lookup.items()
                if not str(key).startswith("_")
            },
        "frozen_atlas_primary_directed_pairwise_witness":
            frozen_pair_public,
        "frozen_atlas_selected_measured_witness":
            atlas_witness_summary,
        "experimental_test_at_frozen_atlas_selected_witness":
            atlas_witness_experimental_test,
        "training_plausible_models_deltaBIC_2":
            plausible_2,
        "training_plausible_models_deltaBIC_6":
            plausible_6,
        "training_plausible_models_deltaBIC_10":
            plausible_10,
        "n_training_plausible_models_deltaBIC_10":
            int(len(plausible_10)),
        "uniaxial_BIC_winner":
            best_bic_model,
        "uniaxial_NRMSE_winner":
            best_nrmse_model,
        "heldout_combined_NRMSE_winner_all_seven":
            best_holdout_model,
        "heldout_combined_NRMSE_winner_within_training_plausible_models":
            best_plausible_holdout_model,
        "training_BIC_vs_holdout_combined_NRMSE_rank_spearman_rho":
            ranking_rho,
        "training_BIC_vs_holdout_combined_NRMSE_rank_spearman_pvalue":
            ranking_p,
        "n_total_family_pairs":
            int(len(pair_table)),
        "n_pairs_with_both_models_training_plausible":
            n_plausible_pairs,
        "n_plausible_hidden_completeF_divergence_pairs":
            n_plausible_hidden,
        "plausibility_sensitivity": sensitivity_table.to_dict(
            orient="records"
        ),
        "all_pair_publication_audit":
            all_pair_aggregate,
        "supplementary_BIC10_pair_selection_class":
            selection_class,
        "supplementary_BIC10_external_discrimination_eligible":
            bool(eligible),
        "publication_primary_external_validation_eligible":
            bool(
                atlas_lookup_primary_pass
                and atlas_witness_experimental_test[
                    "atlas_witness_preference_matches_combined_holdout"
                ]
            ),
        **pair_summary,
        "validation_interpretation": (
            "PUBLICATION PRIMARY — FROZEN ATLAS LOOKUP: MR and OGDEN1 are "
            "jointly supported by the uniaxial training data. The experimentally "
            "fitted MR state is projected to its nearest same-family node in the "
            "precomputed certified V68 atlas. The atlas node's frozen certified "
            "compatibility set and frozen MR->OGDEN1 critical-noise score are "
            "queried directly, and the stored frozen pairwise witness selects "
            "the measured holdout deformation. The withheld Treloar experiment "
            "is then used only to test that atlas-guided discrimination. Direct "
            "continuous fitted-state complete-F distances are supplementary "
            "diagnostics and are not the atlas result."
        ),
    }

    public_columns = [
        c
        for c in fits.columns
        if not str(c).startswith("_")
    ]
    save_dataframe_checked(
        fits[public_columns],
        OUTDIR
        / "Treloar_uniaxial_fit_results.csv",
        index=False,
    )

    save_json_checked(
        summary,
        OUTDIR
        / "Treloar_external_validation_summary.json",
    )

    publication_text = (
        "TRELOAR PUBLICATION-PRIMARY FROZEN-ATLAS RESULT\n"
        "================================================\n"
        f"Training: uniaxial only ({len(train)} points)\n"
        f"Held out: planar ({len(planar)}) + equibiaxial ({len(biaxial)})\n"
        "Primary model support: Delta-BIC<=2 AND Delta-AICc<=2\n"
        f"Supported primary models: {', '.join(publication_primary_models)}\n"
        "\nFROZEN ATLAS LOOKUP\n"
        "-------------------\n"
        f"Experimental MR fit -> frozen MR source node "
        f"#{primary_atlas_source_lookup['frozen_source_index']}\n"
        f"Projection distance: "
        f"{primary_atlas_source_lookup['projection_required_noise_percent']:.4f}%\n"
        f"Certified compatibility set: "
        f"{primary_atlas_source_lookup['certified_compatibility_set']}\n"
        f"Certified MR->OGDEN1 critical noise: "
        f"{primary_atlas_source_lookup['certified_scores_percent']['OGDEN1']:.3f}%\n"
        f"Frozen global witness: "
        f"({frozen_pair['global_completeF_witness_lambda1']:.3f}, "
        f"{frozen_pair['global_completeF_witness_lambda2']:.3f}, "
        f"{frozen_pair['global_completeF_witness_lambda3']:.3f})\n"
        f"Atlas-selected measured witness: "
        f"{atlas_witness_summary['atlas_selected_measured_witness_mode']} "
        f"at stretch="
        f"{atlas_witness_summary['atlas_selected_measured_witness_stretch']:.3f}\n"
        f"Atlas-selected witness preferred experimental fit: "
        f"{atlas_witness_experimental_test['atlas_selected_witness_preferred_experimental_fit']}\n"
        f"Combined holdout preferred primary model: "
        f"{atlas_witness_experimental_test['combined_holdout_preferred_primary_model']}\n"
        f"Preference agreement: "
        f"{atlas_witness_experimental_test['atlas_witness_preference_matches_combined_holdout']}\n"
        "\nDIAGNOSTIC ONLY\n"
        "---------------\n"
        f"Direct fitted MR--OGDEN1 UT pair distance: "
        f"{pair_summary['uniaxial_pair_required_noise_percent']:.3f}%\n"
        f"Direct fitted MR--OGDEN1 complete-F distance: "
        f"{pair_summary['completeF_required_noise_percent']:.3f}%\n"
        "\nPrimary interpretation:\n"
        "The experimental parameters are calibrated once from uniaxial data. "
        "The complete-deformation discrimination result is then obtained by "
        "lookup in the precomputed certified V68 atlas, including its stored "
        "pairwise witness. Independent planar/equibiaxial data are used only "
        "after the atlas lookup to test the predicted discrimination.\n"
    )
    publication_text_path = (
        OUTDIR / "PUBLICATION_PRIMARY_FROZEN_ATLAS_RESULT.txt"
    )
    with publication_text_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(publication_text)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    _verify_saved_file(publication_text_path)

    make_validation_figure(
        dataset,
        fits,
        pair_table,
        selected_pair,
        witness_table,
        summary,
        OUTDIR,
    )

    print("[V68.5.5] Summary:")
    print(
        json.dumps(
            summary,
            indent=2,
            allow_nan=True,
        )
    )

    return fits, summary


def main() -> None:
    global V68_PROTOCOL, V68_BASES

    # FIRST runtime action: safely establish Google Drive.
    drive_root = mount_google_drive_first()

    # Verify the exact visible V68 root and create its output folder before
    # dependency loading, data download, or scientific computation.
    run_dir = configure_exact_visible_drive_output(
        drive_root
    )

    print("[V68.5.5] STEP 3/4 — loading bundled frozen V68.1...")
    V = import_v68_module()
    cfg = V.BASE_CFG

    V68_PROTOCOL = V.make_protocol(cfg)
    V68_BASES = V.constitutive_bases(
        V68_PROTOCOL
    )

    print("[V68.5.5] STEP 4/4 — starting Treloar validation.")
    print("           TRAIN = uniaxial only")
    print("           HOLDOUT = planar/pure shear + equibiaxial")
    print("           pair qualification never uses held-out stresses")
    print("           PRIMARY MODELS = joint Delta-BIC<=2 and Delta-AICc<=2")
    print("           PRIMARY RESULT = lookup in PRECOMPUTED CERTIFIED V68 ATLAS")
    print("           SUPPLEMENT = every BIC<=10 qualified pair")
    print(
        f"[V68.5.5] Complete-F parent: "
        f"{V68_PROTOCOL.n_parent_states} states, "
        f"{len(V68_PROTOCOL.atlas_indices)} response components."
    )

    self_test_diagonal_evaluator_against_v68(
        V,
        cfg,
    )

    validate_treloar(
        V,
        cfg,
    )

    # Persist explicit proof of completion.
    write_run_complete_marker(
        run_dir
    )
    write_verified_artifact_manifest(
        run_dir
    )

    final_parent_names = {
        p.name for p in ROOT.iterdir()
    }
    if (
        VISIBLE_OUTPUT_FOLDER_NAME
        not in final_parent_names
    ):
        raise RuntimeError(
            "FINAL CHECK FAILED: TRELOAR_VALIDATION_OUTPUT is not visible "
            "in the exact V68 root."
        )
    if (
        VISIBLE_ROOT_MARKER_NAME
        not in final_parent_names
    ):
        raise RuntimeError(
            "FINAL CHECK FAILED: root-level Treloar location marker is missing."
        )

    print(
        "[V68.5.5] FINAL EXACT-ROOT VISIBILITY CHECK: PASS"
    )

    print_saved_artifact_manifest(
        run_dir
    )

    print("\n" + "=" * 100)
    print("[V68.5.5] RUN COMPLETED AND VERIFIED ON GOOGLE DRIVE")
    print("=" * 100)
    print(f"Persistent run directory:\n{run_dir}")
    print(
        f"Root-level visibility marker:\n{ROOT / VISIBLE_ROOT_MARKER_NAME}"
    )


if __name__ == "__main__":
    try:
        main()
    except BaseException as exc:
        write_run_error_marker(exc)
        raise


[V68.5.5] STEP 1/4 — ENSURING GOOGLE DRIVE IS AVAILABLE FIRST...
[V68.5.5] Existing Google Drive mount detected; reusing it instead of remounting a non-empty mountpoint.
[V68.5.5] Google Drive available at: /content/drive/MyDrive
[V68.5.5] Exact existing V68 root:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
[V68.5.5] Existing anchor folders detected: cell2_publication_validation, publication_figures_3_5
[V68.5.5] EXACT V68-ROOT DRIVE OUTPUT VERIFIED.
[V68.5.5] Output folder created DIRECTLY in the visible V68 root:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/TRELOAR_VALIDATION_OUTPUT
[V68.5.5] Root-level marker created:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/TRELOAR_VALIDATION_OUTPUT_LOCATION.txt
[V68.5.5] Parent-directory enumeration chec

In [ ]:
#!/usr/bin/env python3
"""
V68.6.1 — HUMAN BRAIN SOFT-TISSUE EXTERNAL VALIDATION + LOCAL TRANSFER AUDIT
=====================================================
FULLY STANDALONE + PRECOMPUTED FROZEN-ATLAS LOOKUP

Purpose
-------
Provide an independent soft-biological-tissue validation of the frozen V68
complete-deformation constitutive compatibility atlas using public human-brain
mechanical data from Budday et al. (2017), distributed in machine-readable form
through the HyperSmart material repository.

Regional cases
--------------
The script analyzes four anatomical regions separately:

    - cortex
    - basal ganglia
    - corona radiata
    - corpus callosum

These are four regional cases from ONE experimental study, not four independent
studies.

Blinded experimental design
---------------------------
For each region:

    TRAINING / CALIBRATION:
        uniaxial COMPRESSION only (lambda < 1).

    COMPLETELY WITHHELD:
        uniaxial TENSION (lambda > 1),
        SIMPLE SHEAR.

The lambda=1 zero-stress point is excluded from model calibration and holdout
scoring.

The public axial data are nominal/first-Piola stress P11 in kPa. For
incompressible uniaxial deformation with zero transverse traction,

    tau11 - tau22 = lambda * P11,

so the script converts the axial data into the pressure-free Kirchhoff/Cauchy
stress difference used by V68. The simple-shear stress is used directly because
P12 = tau12 for the incompressible simple-shear map.

Frozen-atlas validation
-----------------------
The experiment-specific parameters are fitted ONCE from compression data because
the tissue parameters are unknown. After that:

    1. Models supported by compression data are determined without using tension
       or shear truth.
    2. Each supported experimental fit is projected onto a SAME-FAMILY source
       node of the PRECOMPUTED certified V68 atlas.
    3. The source node's certified compatibility set and certified directed
       critical-noise distances are read from the frozen Cell-2 atlas.
    4. A primary source->target pair is selected using training support +
       frozen-atlas incompatibility only.
    5. The stored frozen V68/Cell-2 pairwise witness is recovered; no atlas
       optimization is rerun.
    6. That frozen atlas relation selects the most discriminating deformation
       among ACTUALLY MEASURED withheld tension/shear states.
    7. Only after the deformation is frozen are withheld experimental stresses
       revealed and used for validation.

Primary pre-holdout eligibility
-------------------------------
A regional case supports the primary validation only when:

    - at least two families satisfy BOTH Delta-BIC <= 2 and Delta-AICc <= 2
      on compression-only training;
    - the two fitted compression responses are within the declared 3% training
      ambiguity tube;
    - both experimental fits project to same-family frozen atlas nodes within 3%;
    - the frozen source atlas node excludes the target family at the certified
      3% tolerance.

If no pair satisfies these rules, the region is reported as NOT ELIGIBLE.
No tension or shear stress is used to rescue or choose a pair.

Public data
-----------
HyperSmart machine-readable representations of:

    Budday S. et al.
    "Mechanical characterization of human brain tissue."
    Acta Biomaterialia 48 (2017) 319-340.
    DOI: 10.1016/j.actbio.2016.10.036

HyperSmart marks these files as Soft Biological Tissues / Human, nominal axial
stress in kPa, simple shear in kPa, with data_source=1.

Standalone / persistence contract
---------------------------------
This file is fully standalone for Google Colab:

    - Google Drive availability is established as the FIRST runtime action.
    - The exact existing V68 project root is required and verified using existing
      project anchor folders.
    - Output is written directly to:
          SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT/
      under that exact V68 root.
    - A root-level visibility marker is written alongside the existing V68 files.
    - The exact frozen V68.1 source is embedded and checksum-locked.
    - No external V68 Python source file is required.
    - Frozen atlas/Cell-2 CSV outputs are READ from the existing V68 Drive root.
    - Every CSV/JSON/PNG/PDF write is verified immediately on Drive.
    - A SHA-256 artifact manifest is written at completion.
    - _RUN_COMPLETE.txt is written only after successful completion.
    - If execution fails after Drive setup, the traceback is persisted as
      _RUN_ERROR.txt.
"""

from __future__ import annotations

import hashlib
import base64
import gzip
import importlib.util
import json
import math
import os
import sys
import urllib.request
import traceback
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize_scalar
from scipy.stats import spearmanr

try:
    import yaml
except ImportError as exc:
    raise ImportError(
        "PyYAML is required. In Colab run: !pip -q install pyyaml"
    ) from exc

# =============================================================================
# Configuration
# =============================================================================

PROJECT_FOLDER_NAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)

# Paths are intentionally unresolved until Google Drive is force-mounted and
# a real persistence test has passed.
DRIVE_ROOT = None
ROOT = None
OUTPUT_BASE = None
OUTDIR = None

VISIBLE_OUTPUT_FOLDER_NAME = "SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT"
VISIBLE_ROOT_MARKER_NAME = "SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT_LOCATION.txt"

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
ATLAS_TOLERANCE_FRACTION = 0.03
PAIR_TOLERANCE_FRACTION = ATLAS_TOLERANCE_FRACTION
RNG_SEED = 20260809

BRAIN_DATASET_URLS = {
    "cortex": (
        "https://raw.githubusercontent.com/dhnp90/HyperSmart/main/"
        "src/material_repository/cortex.yaml"
    ),
    "basal_ganglia": (
        "https://raw.githubusercontent.com/dhnp90/HyperSmart/main/"
        "src/material_repository/basalGanglia.yaml"
    ),
    "corona_radiata": (
        "https://raw.githubusercontent.com/dhnp90/HyperSmart/main/"
        "src/material_repository/coronaRadiata.yaml"
    ),
    "corpus_callosum": (
        "https://raw.githubusercontent.com/dhnp90/HyperSmart/main/"
        "src/material_repository/corpusCallosum.yaml"
    ),
}

BRAIN_REGION_ORDER = (
    "cortex",
    "basal_ganglia",
    "corona_radiata",
    "corpus_callosum",
)

BUDDAY_DOI = "10.1016/j.actbio.2016.10.036"
HYPERSMART_REPOSITORY = "https://github.com/dhnp90/HyperSmart"

# Model fitting settings.
DE_POPSIZE = 18
DE_MAXITER = 220
DE_TOL = 5.0e-10

# Numerical floor used only in normalized discrimination metrics.
NORMALIZATION_FLOOR_KPA = 1.0e-12

# Primary manuscript support rule: COMPRESSION TRAINING DATA ONLY.
PRIMARY_PUBLICATION_DELTA_BIC_MAX = 2.0
PRIMARY_PUBLICATION_DELTA_AICC_MAX = 2.0

# Broader training-support sensitivity is reported but is not used to rescue
# an ineligible primary case.
DELTA_BIC_SENSITIVITY = (2.0, 6.0, 10.0)

# Exact-neutral stretch is excluded from both fitting and holdout metrics.
UNITY_STRETCH_TOL = 1.0e-10

# Frozen atlas files. The source atlas lives in ROOT; the certified
# reclassification lives in ROOT/cell2_publication_validation.
FROZEN_ATLAS_EXPECTED_SOURCE_ROWS = 2631
FROZEN_ATLAS_EXPECTED_REGION_COUNT = 14
FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES = 47
FROZEN_ATLAS_NUMERICAL_TOL = 3.0e-8

# =============================================================================
# Drive + V68 import
# =============================================================================

def mount_google_drive_first() -> Path:
    """
    Make Google Drive available as the FIRST runtime operation.

    Startup logic is intentionally safe for both fresh and already-mounted
    Colab sessions:

      1. If a valid MyDrive directory already exists, use it directly.
      2. Otherwise mount at /content/drive if that mountpoint is empty/usable.
      3. If /content/drive is occupied but is not a usable Drive mount, mount
         at a fresh empty dedicated mountpoint.

    Persistent write/read-back verification is performed immediately afterward
    by configure_exact_visible_drive_output().
    """
    print("[V68.6.1] STEP 1/4 — ENSURING GOOGLE DRIVE IS AVAILABLE FIRST...")

    existing_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
    ]
    existing = next(
        (p for p in existing_candidates if p.is_dir()),
        None,
    )

    if existing is not None:
        print(
            "[V68.6.1] Existing Google Drive mount detected; "
            "reusing it instead of remounting a non-empty mountpoint."
        )
        print(f"[V68.6.1] Google Drive available at: {existing}")
        return existing.resolve()

    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "V68.5.1 is intended to run as a standalone Google Colab script. "
            "google.colab could not be imported, so persistent Drive saving "
            "cannot be guaranteed."
        ) from exc

    primary_mountpoint = Path("/content/drive")

    # Colab refuses to mount onto a non-empty ordinary directory. Use the
    # standard mountpoint only when it is absent or empty.
    primary_usable = (
        not primary_mountpoint.exists()
        or (
            primary_mountpoint.is_dir()
            and not any(primary_mountpoint.iterdir())
        )
    )

    if primary_usable:
        mountpoint = primary_mountpoint
    else:
        # The standard mountpoint contains files but no usable MyDrive.
        # Never delete or overwrite it. Mount to a fresh, script-owned path.
        base = Path("/content/v68_google_drive")
        mountpoint = base
        suffix = 0

        while mountpoint.exists() and (
            not mountpoint.is_dir()
            or any(mountpoint.iterdir())
        ):
            suffix += 1
            mountpoint = Path(f"{base}_{suffix:02d}")

        mountpoint.mkdir(parents=True, exist_ok=True)

    print(f"[V68.6.1] Mounting Google Drive at: {mountpoint}")
    drive.mount(
        str(mountpoint),
        force_remount=False,
    )

    candidates = [
        mountpoint / "MyDrive",
        mountpoint / "My Drive",
    ]
    drive_root = next(
        (p for p in candidates if p.is_dir()),
        None,
    )

    if drive_root is None:
        raise RuntimeError(
            "Google Drive mount returned successfully, but no MyDrive "
            f"directory was found beneath {mountpoint}."
        )

    print(f"[V68.6.1] Google Drive mounted at: {drive_root}")
    return drive_root.resolve()


def configure_exact_visible_drive_output(
    drive_root: Path,
) -> Path:
    """
    Save directly into the exact V68 project root visible in Google Drive.

    Expected existing root:
      My Drive/
        Optimal_Protocol/
          V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/

    Output:
      SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT/

    A root-level marker file is also written alongside the existing V68 project
    folders so it can be seen immediately in the Drive web UI.
    """
    global DRIVE_ROOT, ROOT, OUTPUT_BASE, OUTDIR

    DRIVE_ROOT = Path(drive_root).resolve()
    ROOT = (
        DRIVE_ROOT
        / "Optimal_Protocol"
        / PROJECT_FOLDER_NAME
    )

    if not ROOT.is_dir():
        raise RuntimeError(
            "Expected existing V68 project root was not found on the mounted "
            "Google Drive:\n"
            f"{ROOT}\n\n"
            "The script will not create a different substitute root."
        )

    observed_names = {
        p.name for p in ROOT.iterdir()
    }

    expected_anchors = {
        "cell2_publication_validation",
        "publication_figures_3_5",
    }
    found_anchors = sorted(
        expected_anchors.intersection(
            observed_names
        )
    )

    print("[V68.6.1] Exact existing V68 root:")
    print(f"           {ROOT}")
    print(
        "[V68.6.1] Existing anchor folders detected: "
        + (
            ", ".join(found_anchors)
            if found_anchors
            else "(none)"
        )
    )

    if not found_anchors:
        preview = sorted(observed_names)[:25]
        raise RuntimeError(
            "The mounted path exists but does not look like the V68 directory "
            "shown in Google Drive. Expected at least one anchor folder "
            "('cell2_publication_validation' or 'publication_figures_3_5').\n"
            f"Mounted path: {ROOT}\n"
            f"Observed entries: {preview}\n\n"
            "Refusing to save to the wrong Drive location."
        )

    OUTPUT_BASE = ROOT
    OUTDIR = (
        ROOT
        / VISIBLE_OUTPUT_FOLDER_NAME
    )
    OUTDIR.mkdir(
        parents=False,
        exist_ok=True,
    )

    # Root-level marker visible directly in the exact folder shown in Drive UI.
    root_marker = (
        ROOT
        / VISIBLE_ROOT_MARKER_NAME
    )
    marker_payload = (
        "Human-brain validation output location\n"
        f"ROOT={ROOT}\n"
        f"OUTDIR={OUTDIR}\n"
        f"created_at={datetime.now().isoformat()}\n"
    )

    with root_marker.open(
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(marker_payload)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    if not root_marker.is_file():
        raise RuntimeError(
            f"Root visibility marker was not created: {root_marker}"
        )
    if (
        root_marker.read_text(
            encoding="utf-8"
        )
        != marker_payload
    ):
        raise RuntimeError(
            f"Root marker read-back failed: {root_marker}"
        )

    # Output-folder persistence sentinel.
    sentinel = (
        OUTDIR
        / "_DRIVE_PERSISTENCE_TEST.txt"
    )
    sentinel_payload = (
        "V68.6.0 human-brain frozen-atlas persistence test\n"
        f"ROOT={ROOT}\n"
        f"OUTDIR={OUTDIR}\n"
    )

    with sentinel.open(
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            sentinel_payload
        )
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    if not sentinel.is_file():
        raise RuntimeError(
            f"Output sentinel was not created: {sentinel}"
        )
    if (
        sentinel.read_text(
            encoding="utf-8"
        )
        != sentinel_payload
    ):
        raise RuntimeError(
            f"Output sentinel read-back failed: {sentinel}"
        )

    # Parent enumeration check.
    visible_names = {
        p.name for p in ROOT.iterdir()
    }
    if (
        VISIBLE_OUTPUT_FOLDER_NAME
        not in visible_names
    ):
        raise RuntimeError(
            "SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT was created by path but is not visible "
            "when the exact V68 parent directory is enumerated."
        )
    if (
        VISIBLE_ROOT_MARKER_NAME
        not in visible_names
    ):
        raise RuntimeError(
            "Root marker was written but is not visible when the exact V68 "
            "parent directory is enumerated."
        )

    OUTDIR = OUTDIR.resolve()

    print(
        "[V68.6.1] EXACT V68-ROOT DRIVE OUTPUT VERIFIED."
    )
    print(
        "[V68.6.1] Output folder created DIRECTLY in the visible V68 root:"
    )
    print(f"           {OUTDIR}")
    print(
        "[V68.6.1] Root-level marker created:"
    )
    print(f"           {root_marker}")
    print(
        "[V68.6.1] Parent-directory enumeration check: PASS"
    )

    return OUTDIR


def require_verified_drive_output() -> Path:
    if OUTDIR is None:
        raise RuntimeError(
            "Drive output has not been configured yet."
        )

    outdir = Path(OUTDIR).resolve()
    if not outdir.is_dir():
        raise RuntimeError(
            f"Verified Drive output directory does not exist: {outdir}"
        )
    if not str(outdir).startswith("/content/drive/"):
        raise RuntimeError(
            f"Refusing non-Drive output directory: {outdir}"
        )
    return outdir


def _verify_saved_file(
    path: Path,
    minimum_bytes: int = 1,
) -> Path:
    outdir = require_verified_drive_output()
    path = Path(path).resolve()

    try:
        path.relative_to(outdir)
    except ValueError as exc:
        raise RuntimeError(
            "Refusing to treat a file outside the verified run folder as an "
            f"analysis output.\nOUTDIR={outdir}\npath={path}"
        ) from exc

    if not path.is_file():
        raise RuntimeError(
            f"Expected saved file does not exist: {path}"
        )

    size = int(path.stat().st_size)
    if size < minimum_bytes:
        raise RuntimeError(
            f"Saved file is unexpectedly small ({size} bytes): {path}"
        )

    with path.open("rb") as handle:
        first = handle.read(1)
    if size > 0 and not first:
        raise RuntimeError(
            f"Could not read saved file back from Drive: {path}"
        )

    print(
        f"[V68.6.1][DRIVE SAVED] {path} "
        f"({size / 1024.0:.2f} KiB)"
    )
    return path


def save_dataframe_checked(
    df: pd.DataFrame,
    path: Path,
    *,
    index: bool = False,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return _verify_saved_file(path)


def save_json_checked(
    payload: Mapping[str, object],
    path: Path,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            allow_nan=True,
        )
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    return _verify_saved_file(path)


def save_figure_checked(
    fig,
    path: Path,
    **kwargs,
) -> Path:
    require_verified_drive_output()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, **kwargs)
    return _verify_saved_file(path, minimum_bytes=100)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_verified_artifact_manifest(
    outdir: Path,
) -> Path:
    outdir = Path(outdir).resolve()
    rows = []

    for path in sorted(outdir.iterdir(), key=lambda p: p.name.lower()):
        if not path.is_file():
            continue
        if path.name == "_ARTIFACT_MANIFEST_SHA256.csv":
            continue

        rows.append(
            {
                "filename": path.name,
                "absolute_drive_path": str(path.resolve()),
                "size_bytes": int(path.stat().st_size),
                "sha256": sha256_file(path),
            }
        )

    return save_dataframe_checked(
        pd.DataFrame(rows),
        outdir / "_ARTIFACT_MANIFEST_SHA256.csv",
        index=False,
    )


def write_run_complete_marker(
    outdir: Path,
) -> Path:
    outdir = Path(outdir).resolve()
    marker = outdir / "_RUN_COMPLETE.txt"
    payload = (
        "V68.6.0 completed successfully.\n"
        f"OUTDIR={outdir}\n"
        f"completed_at={datetime.now().isoformat()}\n"
    )

    with marker.open("w", encoding="utf-8") as handle:
        handle.write(payload)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

    return _verify_saved_file(marker)


def write_run_error_marker(
    exc: BaseException,
) -> None:
    if OUTDIR is None:
        return

    try:
        outdir = require_verified_drive_output()
        path = outdir / "_RUN_ERROR.txt"
        payload = (
            "V68.6.0 failed after Drive setup.\n\n"
            + "".join(
                traceback.format_exception(
                    type(exc),
                    exc,
                    exc.__traceback__,
                )
            )
        )

        with path.open("w", encoding="utf-8") as handle:
            handle.write(payload)
            handle.flush()
            try:
                os.fsync(handle.fileno())
            except OSError:
                pass

        _verify_saved_file(path)
    except Exception as marker_error:
        print(
            "[V68.6.1] WARNING: could not save _RUN_ERROR.txt: "
            f"{marker_error}"
        )


def print_saved_artifact_manifest(
    outdir: Path,
) -> None:
    outdir = Path(outdir).resolve()
    files = sorted(
        [p for p in outdir.iterdir() if p.is_file()],
        key=lambda p: p.name.lower(),
    )

    print("\n" + "=" * 100)
    print("[V68.6.1] VERIFIED GOOGLE DRIVE ARTIFACT MANIFEST")
    print("=" * 100)
    print(f"Directory: {outdir}")

    for path in files:
        print(
            f"  {path.name:<60} "
            f"{path.stat().st_size / 1024.0:10.2f} KiB"
        )




# =============================================================================
# Bundled frozen V68.1 dependency
# =============================================================================

EMBEDDED_V68_MODULE_SHA256 = "6bad33e20705230a1d8c147260c886379b85d607e27b7c07e5a38b8a1e25c9be"
EMBEDDED_V68_MODULE_GZIP_B64 = (
    "..."
)


def _materialize_embedded_v68_module() -> Path:
    """Materialize the exact bundled V68.1 source and verify its checksum."""
    raw = gzip.decompress(
        base64.b64decode("".join(EMBEDDED_V68_MODULE_GZIP_B64))
    )
    observed = hashlib.sha256(raw).hexdigest()
    if observed != EMBEDDED_V68_MODULE_SHA256:
        raise RuntimeError(
            "Embedded V68.1 checksum mismatch: "
            f"{observed} != {EMBEDDED_V68_MODULE_SHA256}"
        )

    filename = (
        "V68_1_patched_practical_standard_protocols_complete_F_"
        "biological_compatibility_atlas.py"
    )
    candidates = [
        Path("/content") / filename,
        Path("/tmp") / filename,
        Path.cwd() / filename,
    ]

    last_error = None
    for target in candidates:
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            if (
                (not target.is_file())
                or hashlib.sha256(target.read_bytes()).hexdigest()
                != EMBEDDED_V68_MODULE_SHA256
            ):
                target.write_bytes(raw)

            final_digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if final_digest != EMBEDDED_V68_MODULE_SHA256:
                raise RuntimeError(
                    f"Materialized V68.1 checksum mismatch at {target}"
                )

            print(
                f"[V68.6.1] Materialized bundled V68.1 temporary dependency at {target} "
                f"(sha256={final_digest[:12]}...)"
            )
            return target.resolve()
        except OSError as exc:
            last_error = exc

    raise FileNotFoundError(
        "Could not materialize bundled V68.1 dependency. "
        f"Last error: {last_error}"
    )

def import_v68_module():
    """
    Import ONLY the checksum-locked V68.1 source bundled in this script.
    No external V68 source file is searched for or required.
    """
    require_verified_drive_output()

    module_path = _materialize_embedded_v68_module()

    imported_digest = hashlib.sha256(
        Path(module_path).read_bytes()
    ).hexdigest()
    if imported_digest != EMBEDDED_V68_MODULE_SHA256:
        raise RuntimeError(
            "Bundled V68.1 checksum changed unexpectedly after materialization."
        )

    module_name = "v68_5_0_bundled_frozen_dependency"
    spec = importlib.util.spec_from_file_location(
        module_name,
        str(module_path),
    )
    if spec is None or spec.loader is None:
        raise ImportError(
            f"Could not create import spec for bundled V68.1 at {module_path}"
        )

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(module_name, None)
        raise

    print(
        "[V68.6.1] Imported checksum-locked BUNDLED V68.1 "
        f"(sha256={imported_digest[:12]}...)"
    )
    print("[V68.6.1] External V68 source dependency: NONE")
    return module





# =============================================================================
# Frozen certified V68 atlas loading and lookup
# =============================================================================

def _atlas_row_key(
    source_model: str,
    source_index: int,
    target_model: str,
) -> str:
    return (
        f"{str(source_model)}::{int(source_index)}::{str(target_model)}"
    )


def _coordinate_json(value: object) -> np.ndarray:
    if isinstance(value, str):
        return np.asarray(
            json.loads(value),
            dtype=np.float64,
        )
    return np.asarray(
        value,
        dtype=np.float64,
    )


def reconstruct_certified_atlas_from_frozen_outputs(
    V,
    baseline_atlas: pd.DataFrame,
    boundary_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Recreate Cell-2 Stage-3 labels exactly from already-frozen baseline atlas
    scores plus already-frozen boundary-certification witnesses.

    This performs NO constitutive optimization and NO atlas regeneration.
    """
    updates = {
        _atlas_row_key(
            row.source_model,
            row.source_index,
            row.target_model,
        ): float(row.certified_score)
        for _, row in boundary_summary.iterrows()
    }

    rows = []
    for _, row in baseline_atlas.iterrows():
        new = dict(row)
        scores = {}

        for model in MODELS:
            original_col = (
                f"critical_noise_to_{model.lower()}"
            )
            if original_col not in row.index:
                raise RuntimeError(
                    f"Frozen baseline atlas lacks {original_col}"
                )

            score = float(row[original_col])
            key = _atlas_row_key(
                str(row.source_model),
                int(row.source_index),
                model,
            )
            if key in updates:
                score = min(
                    score,
                    float(updates[key]),
                )

            scores[model] = score
            new[
                f"certified_critical_noise_to_{model.lower()}"
            ] = score

        compatible = [
            model
            for model in MODELS
            if (
                scores[model]
                <= ATLAS_TOLERANCE_FRACTION
                + FROZEN_ATLAS_NUMERICAL_TOL
            )
        ]
        compatible = list(
            V.hierarchy_closure(compatible)
        )
        region, signature, subtype = (
            V.region_label_from_compatibility(
                compatible
            )
        )
        minimal = V.minimal_compatible_models(
            compatible
        )

        new.update(
            {
                "certified_primary_region_at_3pct":
                    region,
                "certified_compatibility_set_at_3pct":
                    signature,
                "certified_minimal_adequate_models_at_3pct":
                    V.ordered_signature(minimal),
                "certified_collision_subtype_at_3pct":
                    subtype,
                "certified_n_compatible_models_at_3pct":
                    int(len(compatible)),
                "certified_label_changed":
                    bool(
                        signature
                        != str(
                            row[
                                "compatibility_set_at_3pct"
                            ]
                        )
                    ),
            }
        )
        rows.append(new)

    return pd.DataFrame(rows)


def load_and_verify_frozen_atlas_assets(
    V,
) -> Dict[str, object]:
    """
    Load the PRECOMPUTED source atlas and Cell-2 certified atlas from the exact
    V68 Drive root.

    If the canonical certified CSV is absent but the frozen baseline atlas and
    frozen boundary summary exist, reconstruct Cell-2 Stage-3 labels in memory.
    No model is fitted and no atlas search is run during reconstruction.
    """
    require_verified_drive_output()

    if ROOT is None:
        raise RuntimeError(
            "Exact V68 ROOT is not configured."
        )

    root = Path(ROOT)
    cell2 = root / "cell2_publication_validation"

    baseline_path = (
        root
        / "seven_model_source_atlas_states.csv"
    )
    pairwise_path = (
        root
        / "pairwise_critical_noise_profiles.csv"
    )
    source_path = (
        root
        / "source_states.csv"
    )
    certified_path = (
        cell2
        / "certified_source_atlas_states.csv"
    )
    boundary_path = (
        cell2
        / "boundary_certification_summary.csv"
    )
    change_summary_path = (
        cell2
        / "certified_atlas_change_summary.json"
    )

    required_frozen = [
        baseline_path,
        pairwise_path,
        source_path,
    ]
    missing = [
        str(p)
        for p in required_frozen
        if not p.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Frozen V68 atlas lookup requires existing precomputed "
            "source-atlas assets. Missing:\n  "
            + "\n  ".join(missing)
        )

    baseline = pd.read_csv(
        baseline_path
    )
    pairwise = pd.read_csv(
        pairwise_path
    )
    source = pd.read_csv(
        source_path
    )

    source_mode = "canonical_cell2_certified_csv"
    if certified_path.is_file():
        certified = pd.read_csv(
            certified_path
        )
    else:
        if not boundary_path.is_file():
            raise FileNotFoundError(
                "Neither the canonical certified atlas nor the frozen "
                "Cell-2 boundary summary is available.\n"
                f"Expected certified: {certified_path}\n"
                f"Expected boundary summary: {boundary_path}"
            )
        boundary = pd.read_csv(
            boundary_path
        )
        certified = (
            reconstruct_certified_atlas_from_frozen_outputs(
                V,
                baseline,
                boundary,
            )
        )
        source_mode = (
            "reconstructed_in_memory_from_frozen_baseline_plus_"
            "frozen_cell2_boundary_summary"
        )

        # Save only a COPY in the human-brain output folder for reproducibility.
        # Do not modify the frozen Cell-2 directory.
        save_dataframe_checked(
            certified,
            OUTDIR
            / "FROZEN_ATLAS_reconstructed_certified_copy.csv",
            index=False,
        )

    boundary = (
        pd.read_csv(boundary_path)
        if boundary_path.is_file()
        else pd.DataFrame()
    )

    # --------------------------------------------------------------
    # Integrity checks.
    # --------------------------------------------------------------
    if len(certified) != FROZEN_ATLAS_EXPECTED_SOURCE_ROWS:
        raise RuntimeError(
            "Frozen certified atlas has unexpected row count: "
            f"{len(certified)} != "
            f"{FROZEN_ATLAS_EXPECTED_SOURCE_ROWS}"
        )

    if len(source) != FROZEN_ATLAS_EXPECTED_SOURCE_ROWS:
        raise RuntimeError(
            "Frozen source_states.csv has unexpected row count: "
            f"{len(source)}"
        )

    key_cols = [
        "source_model",
        "source_index",
    ]
    source_keys = (
        source[key_cols]
        .astype(str)
        .agg("|".join, axis=1)
        .to_numpy()
    )
    certified_keys = (
        certified[key_cols]
        .astype(str)
        .agg("|".join, axis=1)
        .to_numpy()
    )
    if not np.array_equal(
        source_keys,
        certified_keys,
    ):
        raise RuntimeError(
            "Frozen certified atlas keys do not match source_states.csv."
        )

    label_col = (
        "certified_compatibility_set_at_3pct"
    )
    if label_col not in certified.columns:
        raise RuntimeError(
            f"Frozen certified atlas lacks {label_col}"
        )

    n_regions = int(
        certified[label_col]
        .astype(str)
        .nunique()
    )
    if (
        n_regions
        != FROZEN_ATLAS_EXPECTED_REGION_COUNT
    ):
        raise RuntimeError(
            "Frozen certified atlas region count mismatch: "
            f"{n_regions} != "
            f"{FROZEN_ATLAS_EXPECTED_REGION_COUNT}"
        )

    source_inclusion = bool(
        all(
            str(row.source_model)
            in str(
                row[
                    label_col
                ]
            ).split("|")
            for _, row
            in certified.iterrows()
        )
    )
    if not source_inclusion:
        raise RuntimeError(
            "Frozen certified atlas failed source-model inclusion."
        )

    n_changes = None
    if (
        "certified_label_changed"
        in certified.columns
    ):
        n_changes = int(
            certified[
                "certified_label_changed"
            ].astype(bool).sum()
        )
        if (
            n_changes
            != FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES
        ):
            raise RuntimeError(
                "Frozen certified atlas label-change count mismatch: "
                f"{n_changes} != "
                f"{FROZEN_ATLAS_EXPECTED_CERTIFIED_LABEL_CHANGES}"
            )

    change_summary = {}
    if change_summary_path.is_file():
        with change_summary_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            change_summary = json.load(handle)

    audit = {
        "atlas_source_mode":
            source_mode,
        "baseline_atlas_path":
            str(baseline_path),
        "pairwise_profile_path":
            str(pairwise_path),
        "source_states_path":
            str(source_path),
        "certified_atlas_path":
            (
                str(certified_path)
                if certified_path.is_file()
                else None
            ),
        "boundary_summary_path":
            (
                str(boundary_path)
                if boundary_path.is_file()
                else None
            ),
        "n_source_states":
            int(len(source)),
        "n_certified_atlas_states":
            int(len(certified)),
        "n_pairwise_rows":
            int(len(pairwise)),
        "n_certified_regions":
            n_regions,
        "n_certified_label_changes":
            n_changes,
        "source_model_inclusion":
            source_inclusion,
        "cell2_change_summary":
            change_summary,
        "important_method_note":
            (
                "The experimental models are calibrated once from Human-brain UT. "
                "All subsequent atlas labels, directed critical-noise scores, "
                "and pairwise witnesses used for the primary atlas claim are "
                "read from or reconstructed from frozen V68/Cell-2 outputs; "
                "no atlas optimization is rerun."
            ),
    }
    save_json_checked(
        audit,
        OUTDIR
        / "FROZEN_ATLAS_integrity_and_source_audit.json",
    )

    print(
        "[V68.6.1] Frozen certified atlas loaded and verified:"
    )
    print(
        f"           states={len(certified)}, "
        f"regions={n_regions}, "
        f"label changes={n_changes}, "
        f"mode={source_mode}"
    )

    return {
        "source": source,
        "baseline": baseline,
        "certified": certified,
        "pairwise": pairwise,
        "boundary": boundary,
        "audit": audit,
    }


def reconstruct_atlas_source_response(
    V,
    cfg,
    row: pd.Series,
    *,
    override_response_scale: float | None = None,
) -> np.ndarray:
    coordinate = _coordinate_json(
        row["coordinate_json"]
    )
    template = V.model_template(
        cfg,
        str(row["source_model"]),
        coordinate,
        V68_PROTOCOL,
        V68_BASES,
    )
    if override_response_scale is None:
        scale = float(
            row["source_response_scale_kpa"]
        )
    else:
        scale = float(
            override_response_scale
        )

    return (
        scale
        * np.asarray(
            template,
            dtype=np.float64,
        )
    )


def project_experimental_fit_to_frozen_atlas(
    V,
    cfg,
    fit_row: pd.Series,
    certified_atlas: pd.DataFrame,
) -> Dict[str, object]:
    """
    Nearest same-family FROZEN atlas-node lookup.

    Atlas shape coordinates are frozen. Because V68's response geometry is
    homogeneous in stiffness and the compatibility brush is relative, each
    source node is evaluated at the EXPERIMENTALLY FITTED response scale during
    projection. This prevents the atlas's deterministic one-scale-per-shape
    sampling from creating an artificial nearest-node error.
    """
    model = str(
        fit_row["model"]
    )
    experimental_scale = float(
        fit_row["best_response_scale_kpa"]
    )
    experimental_response = np.asarray(
        fit_row["_full_response"],
        dtype=np.float64,
    )

    bank = (
        certified_atlas[
            certified_atlas[
                "source_model"
            ].astype(str)
            == model
        ]
        .copy()
        .sort_values("source_index")
        .reset_index(drop=True)
    )

    if bank.empty:
        raise RuntimeError(
            f"Frozen atlas has no source nodes for {model}."
        )

    best_score = float("inf")
    best_row = None
    best_response = None

    for _, row in bank.iterrows():
        candidate = (
            reconstruct_atlas_source_response(
                V,
                cfg,
                row,
                override_response_scale=
                    experimental_scale,
            )
        )
        score = float(
            V.required_noise_fraction(
                cfg,
                V68_PROTOCOL,
                experimental_response,
                candidate,
            )
        )
        if score < best_score:
            best_score = score
            best_row = row
            best_response = candidate

    assert best_row is not None
    assert best_response is not None

    label = str(
        best_row[
            "certified_compatibility_set_at_3pct"
        ]
    )
    members = [
        item
        for item in label.split("|")
        if item
    ]

    certified_scores_percent = {}
    for target in MODELS:
        col = (
            f"certified_critical_noise_to_{target.lower()}"
        )
        if col in best_row.index:
            certified_scores_percent[target] = (
                100.0
                * float(best_row[col])
            )

    return {
        "experimental_model":
            model,
        "experimental_coordinate_json":
            str(
                fit_row[
                    "best_coordinate_json"
                ]
            ),
        "experimental_mu0_kpa":
            float(
                fit_row["best_mu0_kpa"]
            ),
        "frozen_source_model":
            str(best_row["source_model"]),
        "frozen_source_index":
            int(best_row["source_index"]),
        "frozen_coordinate_json":
            str(best_row["coordinate_json"]),
        "frozen_source_mu0_kpa":
            float(best_row["source_mu0_kpa"]),
        "projection_required_noise_fraction":
            float(best_score),
        "projection_required_noise_percent":
            float(100.0 * best_score),
        "certified_primary_region":
            str(
                best_row[
                    "certified_primary_region_at_3pct"
                ]
            ),
        "certified_compatibility_set":
            label,
        "certified_compatibility_members":
            members,
        "certified_minimal_adequate_models":
            str(
                best_row[
                    "certified_minimal_adequate_models_at_3pct"
                ]
            ),
        "certified_scores_percent":
            certified_scores_percent,
        "_frozen_row":
            best_row,
        "_scale_matched_response":
            best_response,
    }


def select_publication_primary_models_from_fits(
    fits: pd.DataFrame,
) -> Dict[str, object]:
    """Freeze the primary experimental candidates using UT training statistics only."""
    supported = (
        fits.loc[
            fits["publication_primary_supported"].astype(bool)
        ]
        .copy()
        .sort_values(["BIC", "AICc", "model"])
        .reset_index(drop=True)
    )
    if len(supported) != 2:
        raise RuntimeError(
            "Publication-primary training rule must retain exactly two models; "
            f"found {len(supported)}: "
            f"{supported[['model','training_delta_BIC','training_delta_AICc']].to_dict(orient='records')}"
        )

    source_model = str(supported.loc[0, "model"])
    target_model = str(supported.loc[1, "model"])
    return {
        "supported_models": supported["model"].astype(str).tolist(),
        "source_model": source_model,
        "target_model": target_model,
        "selection_basis": (
            "training only: Delta-BIC<=2 AND Delta-AICc<=2; "
            "direction ordered by BIC, then AICc, then model name"
        ),
        "completeF_used_for_selection": False,
        "heldout_stress_used_for_selection": False,
        "source_delta_BIC": float(supported.loc[0, "training_delta_BIC"]),
        "source_delta_AICc": float(supported.loc[0, "training_delta_AICc"]),
        "target_delta_BIC": float(supported.loc[1, "training_delta_BIC"]),
        "target_delta_AICc": float(supported.loc[1, "training_delta_AICc"]),
    }


def audit_local_frozen_atlas_transfer(
    V,
    cfg,
    fit_row: pd.Series,
    certified_atlas: pd.DataFrame,
    target_model: str,
    *,
    nearest_k: int = 10,
    localization_threshold_fraction: float = 0.03,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    """
    Post-selection stability audit around an experimental same-family atlas lookup.

    Every frozen source node from the experimental family is evaluated at the
    experimentally fitted stiffness. Nodes are ranked only by experimental-to-node
    projection discrepancy. Their already-certified source->target distances are
    then read from the frozen atlas. No target optimization and no pair selection
    is performed here.
    """
    source_model = str(fit_row["model"])
    experimental_scale = float(fit_row["best_response_scale_kpa"])
    experimental_response = np.asarray(fit_row["_full_response"], dtype=np.float64)

    bank = (
        certified_atlas[
            certified_atlas["source_model"].astype(str) == source_model
        ]
        .copy()
        .sort_values("source_index")
        .reset_index(drop=True)
    )
    if bank.empty:
        raise RuntimeError(f"Frozen atlas has no source nodes for {source_model}.")

    target_col = f"certified_critical_noise_to_{str(target_model).lower()}"
    if target_col not in bank.columns:
        raise RuntimeError(
            f"Frozen certified atlas does not contain {target_col}."
        )

    rows = []
    for _, row in bank.iterrows():
        candidate = reconstruct_atlas_source_response(
            V, cfg, row, override_response_scale=experimental_scale
        )
        projection = float(
            V.required_noise_fraction(
                cfg, V68_PROTOCOL, experimental_response, candidate
            )
        )
        target_noise = float(row[target_col])
        rows.append({
            "source_model": source_model,
            "source_index": int(row["source_index"]),
            "source_coordinate_json": str(row["coordinate_json"]),
            "projection_required_noise_fraction": projection,
            "projection_required_noise_percent": 100.0 * projection,
            "certified_compatibility_set": str(
                row["certified_compatibility_set_at_3pct"]
            ),
            "target_model": str(target_model),
            "certified_source_to_target_noise_fraction": target_noise,
            "certified_source_to_target_noise_percent": 100.0 * target_noise,
            "target_excluded_at_3pct": bool(
                target_noise > ATLAS_TOLERANCE_FRACTION
            ),
        })

    all_nodes = (
        pd.DataFrame(rows)
        .sort_values(["projection_required_noise_fraction", "source_index"])
        .reset_index(drop=True)
    )
    nearest = all_nodes.head(max(1, int(nearest_k))).copy()
    local = all_nodes[
        all_nodes["projection_required_noise_fraction"].astype(float)
        <= float(localization_threshold_fraction) + 1.0e-12
    ].copy()

    if len(local):
        local_scores = local[
            "certified_source_to_target_noise_percent"
        ].to_numpy(float)
        local_exclusion_fraction = float(
            local["target_excluded_at_3pct"].astype(bool).mean()
        )
        all_local_exclude = bool(
            local["target_excluded_at_3pct"].astype(bool).all()
        )
        local_min = float(np.min(local_scores))
        local_med = float(np.median(local_scores))
        local_max = float(np.max(local_scores))
    else:
        local_exclusion_fraction = float("nan")
        all_local_exclude = False
        local_min = local_med = local_max = float("nan")

    summary = {
        "source_model": source_model,
        "target_model": str(target_model),
        "nearest_k": int(min(max(1, int(nearest_k)), len(all_nodes))),
        "nearest_source_index": int(all_nodes.loc[0, "source_index"]),
        "nearest_projection_percent": float(
            all_nodes.loc[0, "projection_required_noise_percent"]
        ),
        "nearest_source_to_target_noise_percent": float(
            all_nodes.loc[0, "certified_source_to_target_noise_percent"]
        ),
        "localization_threshold_percent": 100.0 * float(localization_threshold_fraction),
        "n_same_family_frozen_nodes": int(len(all_nodes)),
        "n_nodes_within_localization_threshold": int(len(local)),
        "fraction_local_nodes_excluding_target_at_3pct": local_exclusion_fraction,
        "all_local_nodes_exclude_target_at_3pct": all_local_exclude,
        "local_source_to_target_noise_percent_min": local_min,
        "local_source_to_target_noise_percent_median": local_med,
        "local_source_to_target_noise_percent_max": local_max,
        "selection_role": "POST_SELECTION_ROBUSTNESS_AUDIT_ONLY",
    }
    return nearest, local, summary

def frozen_pairwise_witness_for_source_node(
    V,
    cfg,
    assets: Mapping[str, object],
    source_lookup: Mapping[str, object],
    target_model: str,
) -> Dict[str, object]:
    """
    Recover the already-frozen V68 pairwise target witness for one source node.

    If Cell-2 improved this directed fit, use the Cell-2 certified target witness.
    Otherwise use the original frozen pairwise witness. No new optimization.
    """
    source_row = source_lookup[
        "_frozen_row"
    ]
    source_model = str(
        source_row["source_model"]
    )
    source_index = int(
        source_row["source_index"]
    )

    pairwise = assets["pairwise"]
    match = pairwise[
        (pairwise["source_model"].astype(str) == source_model)
        & (
            pairwise["source_index"].astype(int)
            == source_index
        )
        & (
            pairwise["target_model"].astype(str)
            == str(target_model)
        )
    ]

    if len(match) != 1:
        raise RuntimeError(
            "Expected exactly one frozen directed pairwise row for "
            f"{source_model}[{source_index}] -> {target_model}; "
            f"found {len(match)}."
        )

    baseline = match.iloc[0]
    baseline_score = float(
        baseline["critical_noise_fraction"]
    )
    coordinate_json = str(
        baseline[
            "best_target_coordinate_json"
        ]
    )
    target_mu0_kpa = float(
        baseline["best_target_mu0_kpa"]
    )
    witness_source = "frozen_V68_baseline_pairwise_profile"
    certified_score = baseline_score

    boundary = assets["boundary"]
    if len(boundary):
        bmatch = boundary[
            (boundary["source_model"].astype(str) == source_model)
            & (
                boundary["source_index"].astype(int)
                == source_index
            )
            & (
                boundary["target_model"].astype(str)
                == str(target_model)
            )
        ]
        if len(bmatch) > 1:
            raise RuntimeError(
                "Frozen Cell-2 boundary summary contains duplicate directed rows."
            )
        if len(bmatch) == 1:
            br = bmatch.iloc[0]
            boundary_score = float(
                br["certified_score"]
            )
            if boundary_score <= baseline_score + 1.0e-15:
                certified_score = boundary_score
                coordinate_json = str(
                    br[
                        "best_target_coordinate_json"
                    ]
                )
                target_mu0_kpa = float(
                    br[
                        "best_target_mu0_kpa"
                    ]
                )
                witness_source = (
                    "frozen_Cell2_certified_pairwise_witness"
                )

    certified_col = (
        f"certified_critical_noise_to_{str(target_model).lower()}"
    )
    if certified_col in source_row.index:
        atlas_certified_score = float(
            source_row[certified_col]
        )
        if (
            abs(
                atlas_certified_score
                - certified_score
            )
            > 2.0e-8
        ):
            # The row may have inherited hierarchy closure; for MR->OGDEN1 this
            # should not occur. Treat any mismatch as a hard publication error.
            raise RuntimeError(
                "Frozen pairwise witness score does not match certified atlas "
                f"score for {source_model}[{source_index}] -> {target_model}: "
                f"{certified_score} vs {atlas_certified_score}"
            )

    coordinate = _coordinate_json(
        coordinate_json
    )

    # Match the frozen source-node relation to the experimental source stiffness.
    frozen_source_scale = float(
        source_row[
            "source_response_scale_kpa"
        ]
    )
    experimental_source_scale = float(
        source_lookup[
            "experimental_mu0_kpa"
        ]
    )
    experimental_source_scale = float(
        V.response_scale_from_mu0_kpa(
            experimental_source_scale
        )
    )

    frozen_target_scale = float(
        V.response_scale_from_mu0_kpa(
            target_mu0_kpa
        )
    )
    scale_ratio = (
        frozen_target_scale
        / max(
            frozen_source_scale,
            1.0e-300,
        )
    )
    matched_target_scale = (
        scale_ratio
        * experimental_source_scale
    )

    target_template = V.model_template(
        cfg,
        str(target_model),
        coordinate,
        V68_PROTOCOL,
        V68_BASES,
    )
    target_response = (
        matched_target_scale
        * np.asarray(
            target_template,
            dtype=np.float64,
        )
    )
    source_response = np.asarray(
        source_lookup[
            "_scale_matched_response"
        ],
        dtype=np.float64,
    )

    state_score, component_score = (
        completeF_statewise_separation(
            V,
            cfg,
            source_response,
            target_response,
        )
    )
    witness_state_index = int(
        np.argmax(state_score)
    )
    witness_component_index = int(
        np.argmax(
            component_score[
                witness_state_index
            ]
        )
    )
    witness_stretches = np.asarray(
        V68_PROTOCOL.parent_principal_stretches[
            witness_state_index
        ],
        dtype=np.float64,
    )

    return {
        "source_model":
            source_model,
        "source_index":
            source_index,
        "target_model":
            str(target_model),
        "frozen_certified_critical_noise_fraction":
            float(certified_score),
        "frozen_certified_critical_noise_percent":
            float(100.0 * certified_score),
        "frozen_target_coordinate_json":
            coordinate_json,
        "frozen_target_mu0_kpa":
            float(target_mu0_kpa),
        "frozen_target_witness_source":
            witness_source,
        "stiffness_scale_ratio_target_to_source":
            float(scale_ratio),
        "global_completeF_witness_state_index":
            witness_state_index,
        "global_completeF_witness_component_index":
            witness_component_index,
        "global_completeF_witness_lambda1":
            float(witness_stretches[0]),
        "global_completeF_witness_lambda2":
            float(witness_stretches[1]),
        "global_completeF_witness_lambda3":
            float(witness_stretches[2]),
        "global_completeF_witness_separation_fraction":
            float(
                state_score[
                    witness_state_index
                ]
            ),
        "global_completeF_witness_separation_percent":
            float(
                100.0
                * state_score[
                    witness_state_index
                ]
            ),
        "_source_response":
            source_response,
        "_target_response":
            target_response,
        "_target_coordinate":
            coordinate,
        "_matched_target_scale":
            float(matched_target_scale),
    }




# =============================================================================
# Public human-brain dataset import
# =============================================================================

def download_yaml_checked(
    url: str,
    destination: Path,
) -> Mapping[str, object]:
    require_verified_drive_output()
    destination = Path(destination).resolve()
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None
    for attempt in range(1, 4):
        try:
            request = urllib.request.Request(
                url,
                headers={
                    "User-Agent":
                        "V68.6.0-human-brain-frozen-atlas-validation/1.0"
                },
            )
            with urllib.request.urlopen(
                request,
                timeout=60,
            ) as response:
                payload = response.read()

            destination.write_bytes(payload)
            _verify_saved_file(
                destination,
                minimum_bytes=20,
            )

            data = yaml.safe_load(
                payload.decode("utf-8")
            )
            if not isinstance(data, Mapping):
                raise ValueError(
                    f"YAML at {url} did not contain a mapping."
                )
            return data

        except Exception as exc:
            last_error = exc
            print(
                f"[V68.6.1] YAML download attempt "
                f"{attempt}/3 failed for {url}: {exc}"
            )

    raise RuntimeError(
        f"Could not download public brain dataset after 3 attempts: {url}"
    ) from last_error


def parse_brain_dataset(
    raw: Mapping[str, object],
) -> Dict[str, object]:
    """
    Parse the HyperSmart Budday axial + simple-shear schema and convert axial
    nominal stress into the V68 pressure-free Kirchhoff response.
    """
    if (
        str(
            raw.get(
                "material_class",
                "",
            )
        ).strip()
        != "Soft Biological Tissues"
    ):
        raise ValueError(
            "Expected HyperSmart material_class='Soft Biological Tissues'."
        )

    if (
        str(
            raw.get(
                "material_subclass",
                "",
            )
        ).strip()
        != "Human"
    ):
        raise ValueError(
            "Expected HyperSmart material_subclass='Human'."
        )

    if (
        str(
            raw.get(
                "stress_measure",
                "",
            )
        )
        .strip()
        .lower()
        != "nominal"
    ):
        raise ValueError(
            "This script requires nominal axial stress."
        )

    units = raw.get(
        "unit_of_measure",
        {},
    )
    if not isinstance(
        units,
        Mapping,
    ):
        raise ValueError(
            "Missing unit_of_measure mapping."
        )
    if (
        str(
            units.get(
                "axial",
                "",
            )
        ).lower()
        != "kpa"
    ):
        raise ValueError(
            "Expected axial stress units kPa."
        )
    if (
        str(
            units.get(
                "simple_shear",
                "",
            )
        ).lower()
        != "kpa"
    ):
        raise ValueError(
            "Expected simple-shear stress units kPa."
        )

    data = raw.get(
        "data",
        {},
    )
    if not isinstance(
        data,
        Mapping,
    ):
        raise ValueError(
            "Missing data mapping."
        )

    axial = data.get(
        "axial",
        {},
    )
    shear = data.get(
        "simple_shear",
        {},
    )
    if not isinstance(
        axial,
        Mapping,
    ) or not isinstance(
        shear,
        Mapping,
    ):
        raise ValueError(
            "Dataset must contain axial and simple_shear mappings."
        )

    lam = np.asarray(
        axial.get(
            "stretch",
            [],
        ),
        dtype=np.float64,
    )
    p11_nominal = np.asarray(
        axial.get(
            "stress",
            [],
        ),
        dtype=np.float64,
    )
    gamma = np.asarray(
        shear.get(
            "shear_parameter",
            [],
        ),
        dtype=np.float64,
    )
    tau12 = np.asarray(
        shear.get(
            "stress",
            [],
        ),
        dtype=np.float64,
    )

    if (
        lam.ndim != 1
        or p11_nominal.shape != lam.shape
        or len(lam) < 8
    ):
        raise ValueError(
            "Invalid axial arrays."
        )
    if (
        gamma.ndim != 1
        or tau12.shape != gamma.shape
        or len(gamma) < 5
    ):
        raise ValueError(
            "Invalid simple-shear arrays."
        )
    if np.any(
        lam <= 0.0
    ):
        raise ValueError(
            "Axial stretch must be positive."
        )

    # Incompressible uniaxial:
    # tau11 - tau22 = sigma11 = lambda * P11, with transverse traction zero.
    tau_axial = (
        lam
        * p11_nominal
    )

    compression_mask = (
        lam
        < 1.0
        - UNITY_STRETCH_TOL
    )
    tension_mask = (
        lam
        > 1.0
        + UNITY_STRETCH_TOL
    )

    if (
        int(
            compression_mask.sum()
        )
        < 4
    ):
        raise ValueError(
            "Fewer than four compression points are available."
        )
    if (
        int(
            tension_mask.sum()
        )
        < 4
    ):
        raise ValueError(
            "Fewer than four tension points are available."
        )

    return {
        "material":
            str(
                raw.get(
                    "material",
                    "unknown",
                )
            ),
        "publication_title":
            str(
                raw.get(
                    "publication_title",
                    "",
                )
            ),
        "author":
            str(
                raw.get(
                    "author",
                    "",
                )
            ),
        "year":
            int(
                raw.get(
                    "year",
                    0,
                )
            ),
        "citation":
            str(
                raw.get(
                    "citation",
                    "",
                )
            ),
        "doi":
            str(
                raw.get(
                    "doi",
                    "",
                )
            ),
        "description":
            str(
                raw.get(
                    "description",
                    "",
                )
            ),
        "data_source":
            raw.get(
                "data_source",
                None,
            ),
        "lambda_all":
            lam,
        "axial_nominal_stress_kpa_all":
            p11_nominal,
        "axial_kirchhoff_stress_kpa_all":
            tau_axial,
        "compression_lambda":
            lam[
                compression_mask
            ],
        "compression_tau_kpa":
            tau_axial[
                compression_mask
            ],
        "tension_lambda":
            lam[
                tension_mask
            ],
        "tension_tau_kpa":
            tau_axial[
                tension_mask
            ],
        "gamma":
            gamma,
        "shear_tau_kpa":
            tau12,
    }


# =============================================================================
# Exact V68 response convention on arbitrary UT / simple-shear points
# =============================================================================

def _brain_ogden_power_difference_over_alpha(
    alpha: float,
    log_a: np.ndarray,
    log_b: np.ndarray,
    series_threshold: float,
) -> np.ndarray:
    alpha = float(
        alpha
    )
    a = np.asarray(
        log_a,
        dtype=np.float64,
    )
    b = np.asarray(
        log_b,
        dtype=np.float64,
    )

    if (
        abs(alpha)
        <= float(
            series_threshold
        )
    ):
        a2 = a * a
        b2 = b * b
        a3 = a2 * a
        b3 = b2 * b
        a4 = a3 * a
        b4 = b3 * b
        a5 = a4 * a
        b5 = b4 * b

        return 4.0 * (
            (a - b)
            + 0.5
            * alpha
            * (a2 - b2)
            + (
                alpha**2
                / 6.0
            )
            * (a3 - b3)
            + (
                alpha**3
                / 24.0
            )
            * (a4 - b4)
            + (
                alpha**4
                / 120.0
            )
            * (a5 - b5)
        )

    return (
        4.0
        * (
            np.exp(
                alpha * a
            )
            - np.exp(
                alpha * b
            )
        )
        / alpha
    )


def brain_unit_template(
    V,
    cfg,
    model: str,
    coordinate: Sequence[float],
    *,
    lam: np.ndarray | None = None,
    gamma: np.ndarray | None = None,
) -> np.ndarray:
    """
    Exact frozen-V68 unit-scale constitutive response at arbitrary uniaxial
    tension/compression OR simple-shear points.
    """
    if (
        (lam is None)
        == (gamma is None)
    ):
        raise ValueError(
            "Provide exactly one of lam or gamma."
        )

    coord = np.asarray(
        coordinate,
        dtype=np.float64,
    ).reshape(-1)

    if lam is not None:
        x = np.asarray(
            lam,
            dtype=np.float64,
        )

        b1 = 2.0 * (
            x**2
            - x**-1.0
        )
        b2 = 2.0 * (
            x
            - x**-2.0
        )
        i1m3 = (
            x**2
            + 2.0 / x
            - 3.0
        )
        i2m3 = (
            x**-2.0
            + 2.0 * x
            - 3.0
        )

        def ogden1(
            alpha: float,
        ) -> np.ndarray:
            logx = np.log(
                x
            )
            return _brain_ogden_power_difference_over_alpha(
                alpha,
                logx,
                -0.5 * logx,
                cfg.ogden_alpha_series_threshold,
            )

    else:
        g = np.asarray(
            gamma,
            dtype=np.float64,
        )

        b1 = 2.0 * g
        b2 = 2.0 * g
        i1m3 = g**2
        i2m3 = g**2

        def ogden1(
            alpha: float,
        ) -> np.ndarray:
            root = np.sqrt(
                1.0
                + 0.25
                * g**2
            )
            l1 = (
                root
                + 0.5 * g
            )
            l2 = (
                root
                - 0.5 * g
            )
            raw = _brain_ogden_power_difference_over_alpha(
                alpha,
                np.log(l1),
                np.log(l2),
                cfg.ogden_alpha_series_threshold,
            )
            return (
                raw
                / np.sqrt(
                    g**2
                    + 4.0
                )
            )

    if model == "NH":
        return b1

    if model == "MR":
        rho = float(
            coord[0]
        )
        return (
            (1.0 - rho)
            * b1
            + rho * b2
        )

    if model == "YEOH2":
        beta = float(
            coord[0]
        )
        return (
            b1
            * (
                1.0
                + 2.0
                * beta
                * i1m3
            )
        )

    if model == "GENT":
        gent_gamma = float(
            coord[0]
        )
        denom = (
            1.0
            - gent_gamma
            * i1m3
        )
        if np.any(
            denom
            <= cfg.gent_singularity_margin
        ):
            return np.full_like(
                b1,
                np.nan,
            )
        return (
            b1
            / denom
        )

    if model == "OGDEN1":
        return ogden1(
            float(
                coord[0]
            )
        )

    if model == "OGDEN2":
        p = V.ogden2_map_q_to_physical(
            cfg,
            coord,
        )
        return (
            float(
                p[
                    "weight1"
                ]
            )
            * ogden1(
                float(
                    p[
                        "alpha1"
                    ]
                )
            )
            + float(
                p[
                    "weight2"
                ]
            )
            * ogden1(
                float(
                    p[
                        "alpha2"
                    ]
                )
            )
        )

    if model == "GP2":
        p = V.gp2_map_q_to_physical(
            cfg,
            coord,
        )
        rho = float(
            p["rho"]
        )
        beta20 = float(
            p["beta20"]
        )
        beta11 = float(
            p["beta11"]
        )
        beta02 = float(
            p["beta02"]
        )

        w1 = (
            (1.0 - rho)
            + 2.0
            * beta20
            * i1m3
            + beta11
            * i2m3
        )
        w2 = (
            rho
            + beta11
            * i1m3
            + 2.0
            * beta02
            * i2m3
        )
        return (
            w1 * b1
            + w2 * b2
        )

    raise KeyError(
        model
    )


def self_test_brain_response_evaluator(
    V,
    cfg,
) -> None:
    representatives = {
        "NH": [0.0],
        "MR": [0.30],
        "YEOH2": [0.20],
        "GENT": [0.10],
        "OGDEN1": [1.30],
        "OGDEN2": [
            0.20,
            0.70,
            0.40,
        ],
        "GP2": [
            0.30,
            0.20,
            0.50,
            0.70,
        ],
    }

    max_error = 0.0

    for model, coordinate in representatives.items():
        coordinate = np.asarray(
            coordinate,
            dtype=np.float64,
        )

        exact = V.model_template(
            cfg,
            model,
            coordinate,
            V68_PROTOCOL,
            V68_BASES,
        )

        ut = brain_unit_template(
            V,
            cfg,
            model,
            coordinate,
            lam=V68_PROTOCOL.ut,
        )
        sh = brain_unit_template(
            V,
            cfg,
            model,
            coordinate,
            gamma=V68_PROTOCOL.sh,
        )

        exact_ut = exact[
            V68_PROTOCOL.mode_slices[
                "UT"
            ]
        ]
        exact_sh = exact[
            V68_PROTOCOL.mode_slices[
                "SH"
            ]
        ]

        err = max(
            float(
                np.max(
                    np.abs(
                        ut
                        - exact_ut
                    )
                )
            ),
            float(
                np.max(
                    np.abs(
                        sh
                        - exact_sh
                    )
                )
            ),
        )
        max_error = max(
            max_error,
            err,
        )

    if (
        max_error
        > 5.0e-9
    ):
        raise RuntimeError(
            "Brain arbitrary-point evaluator does not match frozen V68 "
            f"conventions; max abs error={max_error:.3e}"
        )

    print(
        "[V68.6.1] Brain UT/SH evaluator self-test PASS; "
        f"max abs difference={max_error:.3e}"
    )


# =============================================================================
# Compression-only model fitting
# =============================================================================

def nrmse_fraction(
    truth: np.ndarray,
    pred: np.ndarray,
) -> float:
    y = np.asarray(
        truth,
        dtype=np.float64,
    )
    p = np.asarray(
        pred,
        dtype=np.float64,
    )
    denom = max(
        float(
            np.sqrt(
                np.mean(
                    y**2
                )
            )
        ),
        1.0e-30,
    )
    return float(
        np.sqrt(
            np.mean(
                (y - p) ** 2
            )
        )
        / denom
    )


def best_positive_scale_least_squares(
    truth: np.ndarray,
    template: np.ndarray,
    scale_lo: float,
    scale_hi: float,
) -> Tuple[
    float,
    np.ndarray,
    float,
    float,
]:
    y = np.asarray(
        truth,
        dtype=np.float64,
    )
    t = np.asarray(
        template,
        dtype=np.float64,
    )

    if (
        not np.all(
            np.isfinite(t)
        )
        or float(
            np.dot(
                t,
                t,
            )
        )
        <= 1.0e-30
    ):
        return (
            float(scale_lo),
            np.full_like(
                y,
                np.nan,
            ),
            float("inf"),
            float("inf"),
        )

    scale = float(
        np.dot(
            y,
            t,
        )
        / np.dot(
            t,
            t,
        )
    )
    scale = float(
        np.clip(
            scale,
            scale_lo,
            scale_hi,
        )
    )
    pred = (
        scale
        * t
    )
    residual = (
        y - pred
    )
    rss = float(
        np.dot(
            residual,
            residual,
        )
    )
    nrmse = nrmse_fraction(
        y,
        pred,
    )
    return (
        scale,
        pred,
        rss,
        nrmse,
    )


def pooled_nrmse_fraction(
    truths: Sequence[np.ndarray],
    predictions: Sequence[np.ndarray],
) -> float:
    y = np.concatenate(
        [
            np.asarray(
                v,
                dtype=np.float64,
            )
            for v in truths
        ]
    )
    p = np.concatenate(
        [
            np.asarray(
                v,
                dtype=np.float64,
            )
            for v in predictions
        ]
    )
    return nrmse_fraction(
        y,
        p,
    )


def fit_all_models_to_compression(
    V,
    cfg,
    dataset: Mapping[str, object],
    region_seed: int,
) -> pd.DataFrame:
    lam_train = np.asarray(
        dataset[
            "compression_lambda"
        ],
        dtype=np.float64,
    )
    y_train = np.asarray(
        dataset[
            "compression_tau_kpa"
        ],
        dtype=np.float64,
    )
    lam_tension = np.asarray(
        dataset[
            "tension_lambda"
        ],
        dtype=np.float64,
    )
    y_tension = np.asarray(
        dataset[
            "tension_tau_kpa"
        ],
        dtype=np.float64,
    )
    gamma = np.asarray(
        dataset["gamma"],
        dtype=np.float64,
    )
    y_shear = np.asarray(
        dataset[
            "shear_tau_kpa"
        ],
        dtype=np.float64,
    )

    scale_lo = float(
        V.response_scale_from_mu0_kpa(
            cfg.mu0_min_kpa
        )
    )
    scale_hi = float(
        V.response_scale_from_mu0_kpa(
            cfg.mu0_max_kpa
        )
    )

    rows = []

    for model_index, model in enumerate(
        MODELS
    ):
        print(
            f"[V68.6.1] Compression-only fit: {model}"
        )

        d = int(
            V.MODEL_SHAPE_DIMS[
                model
            ]
        )

        def evaluate(
            coordinate: Sequence[float],
        ):
            template = brain_unit_template(
                V,
                cfg,
                model,
                coordinate,
                lam=lam_train,
            )
            scale, pred, rss, nrmse = (
                best_positive_scale_least_squares(
                    y_train,
                    template,
                    scale_lo,
                    scale_hi,
                )
            )
            return {
                "scale":
                    scale,
                "pred":
                    pred,
                "rss":
                    rss,
                "nrmse":
                    nrmse,
            }

        if model == "NH":
            coordinate = np.array(
                [0.0],
                dtype=np.float64,
            )
            best = evaluate(
                coordinate
            )

        elif model in V.SCALAR_PARENTS:
            lo, hi = V.scalar_bounds(
                cfg,
                model,
            )
            opt = minimize_scalar(
                lambda value: evaluate(
                    [
                        float(
                            value
                        )
                    ]
                )["nrmse"],
                bounds=(
                    float(lo),
                    float(hi),
                ),
                method="bounded",
                options={
                    "xatol":
                        1.0e-11,
                    "maxiter":
                        500,
                },
            )
            coordinate = np.array(
                [
                    float(
                        opt.x
                    )
                ],
                dtype=np.float64,
            )
            best = evaluate(
                coordinate
            )

        else:
            bounds = [
                (
                    0.0,
                    1.0,
                )
            ] * d

            opt = differential_evolution(
                lambda q: evaluate(
                    np.asarray(
                        q,
                        dtype=np.float64,
                    )
                )["nrmse"],
                bounds=bounds,
                seed=(
                    RNG_SEED
                    + region_seed
                    + 1000
                    * model_index
                ),
                popsize=DE_POPSIZE,
                maxiter=DE_MAXITER,
                tol=DE_TOL,
                atol=1.0e-12,
                polish=True,
                workers=1,
                updating="immediate",
            )

            coordinate = np.asarray(
                opt.x,
                dtype=np.float64,
            )
            best = evaluate(
                coordinate
            )

        scale = float(
            best["scale"]
        )

        tension_unit = brain_unit_template(
            V,
            cfg,
            model,
            coordinate,
            lam=lam_tension,
        )
        shear_unit = brain_unit_template(
            V,
            cfg,
            model,
            coordinate,
            gamma=gamma,
        )

        tension_prediction = (
            scale
            * tension_unit
        )
        shear_prediction = (
            scale
            * shear_unit
        )

        full_template = V.model_template(
            cfg,
            model,
            coordinate,
            V68_PROTOCOL,
            V68_BASES,
        )
        full_response = (
            scale
            * np.asarray(
                full_template,
                dtype=np.float64,
            )
        )

        tension_nrmse = nrmse_fraction(
            y_tension,
            tension_prediction,
        )
        shear_nrmse = nrmse_fraction(
            y_shear,
            shear_prediction,
        )
        combined_nrmse = pooled_nrmse_fraction(
            [
                y_tension,
                y_shear,
            ],
            [
                tension_prediction,
                shear_prediction,
            ],
        )

        n = int(
            len(
                y_train
            )
        )
        k = int(
            d + 1
        )

        rss_floor = max(
            float(
                best["rss"]
            ),
            np.finfo(
                float
            ).eps
            * max(
                float(
                    np.dot(
                        y_train,
                        y_train,
                    )
                ),
                1.0,
            ),
        )

        aic = float(
            n
            * np.log(
                rss_floor / n
            )
            + 2.0 * k
        )
        bic = float(
            n
            * np.log(
                rss_floor / n
            )
            + k
            * np.log(
                n
            )
        )

        if (
            n
            > k + 1
        ):
            aicc = float(
                aic
                + (
                    2.0
                    * k
                    * (
                        k + 1
                    )
                    / (
                        n
                        - k
                        - 1
                    )
                )
            )
        else:
            aicc = float(
                "inf"
            )

        _, _, physical = (
            V.model_template_and_jacobian(
                cfg,
                model,
                coordinate,
                V68_PROTOCOL,
                V68_BASES,
            )
        )
        mu0 = float(
            V.mu0_kpa_from_response_scale(
                scale
            )
        )

        rows.append(
            {
                "model":
                    model,
                "n_shape_parameters":
                    d,
                "n_fitted_parameters":
                    k,
                "best_coordinate_json":
                    json.dumps(
                        [
                            float(x)
                            for x
                            in coordinate
                        ]
                    ),
                "best_physical_parameters_json":
                    json.dumps(
                        physical,
                        sort_keys=True,
                    ),
                "best_response_scale_kpa":
                    scale,
                "best_mu0_kpa":
                    mu0,
                "compression_rss":
                    float(
                        best[
                            "rss"
                        ]
                    ),
                "compression_nrmse":
                    float(
                        best[
                            "nrmse"
                        ]
                    ),
                "compression_nrmse_percent":
                    float(
                        100.0
                        * best[
                            "nrmse"
                        ]
                    ),
                "AIC":
                    aic,
                "AICc":
                    aicc,
                "BIC":
                    bic,
                "heldout_tension_nrmse":
                    float(
                        tension_nrmse
                    ),
                "heldout_tension_nrmse_percent":
                    float(
                        100.0
                        * tension_nrmse
                    ),
                "heldout_shear_nrmse":
                    float(
                        shear_nrmse
                    ),
                "heldout_shear_nrmse_percent":
                    float(
                        100.0
                        * shear_nrmse
                    ),
                "heldout_combined_nrmse":
                    float(
                        combined_nrmse
                    ),
                "heldout_combined_nrmse_percent":
                    float(
                        100.0
                        * combined_nrmse
                    ),
                "_compression_prediction":
                    np.asarray(
                        best[
                            "pred"
                        ],
                        dtype=np.float64,
                    ),
                "_tension_prediction":
                    np.asarray(
                        tension_prediction,
                        dtype=np.float64,
                    ),
                "_shear_prediction":
                    np.asarray(
                        shear_prediction,
                        dtype=np.float64,
                    ),
                "_full_response":
                    np.asarray(
                        full_response,
                        dtype=np.float64,
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "compression_nrmse",
                "BIC",
                "n_fitted_parameters",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# Training support, pair ambiguity, and atlas-directed selection
# =============================================================================

def add_brain_training_support(
    fits: pd.DataFrame,
) -> pd.DataFrame:
    out = fits.copy()

    best_bic = float(
        out[
            "BIC"
        ].min()
    )
    out[
        "training_delta_BIC"
    ] = (
        out[
            "BIC"
        ].astype(float)
        - best_bic
    )

    finite_aicc = (
        out[
            "AICc"
        ]
        .astype(float)
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )
    if not finite_aicc.notna().any():
        raise RuntimeError(
            "No finite AICc values."
        )

    best_aicc = float(
        finite_aicc.min()
    )
    out[
        "training_delta_AICc"
    ] = (
        out[
            "AICc"
        ].astype(float)
        - best_aicc
    )

    out[
        "publication_primary_supported"
    ] = (
        (
            out[
                "training_delta_BIC"
            ].astype(float)
            <= PRIMARY_PUBLICATION_DELTA_BIC_MAX
            + 1.0e-12
        )
        & (
            out[
                "training_delta_AICc"
            ].astype(float)
            <= PRIMARY_PUBLICATION_DELTA_AICC_MAX
            + 1.0e-12
        )
    )

    for threshold in DELTA_BIC_SENSITIVITY:
        label = int(
            round(
                threshold
            )
        )
        out[
            f"training_plausible_deltaBIC_{label}"
        ] = (
            out[
                "training_delta_BIC"
            ].astype(float)
            <= float(
                threshold
            )
            + 1.0e-12
        )

    return out


def mode_brush_shape(
    cfg,
    curve: np.ndarray,
) -> np.ndarray:
    curve = np.asarray(
        curve,
        dtype=np.float64,
    )
    peak = max(
        float(
            np.max(
                np.abs(
                    curve
                )
            )
        ),
        NORMALIZATION_FLOOR_KPA,
    )
    return np.maximum(
        np.abs(
            curve
        )
        + float(
            cfg.mode_floor_per_unit_noise
        )
        * peak,
        NORMALIZATION_FLOOR_KPA,
    )


def fixed_pair_required_noise_fraction(
    cfg,
    response_a: np.ndarray,
    response_b: np.ndarray,
) -> float:
    a = np.asarray(
        response_a,
        dtype=np.float64,
    )
    b = np.asarray(
        response_b,
        dtype=np.float64,
    )
    ha = mode_brush_shape(
        cfg,
        a,
    )
    hb = mode_brush_shape(
        cfg,
        b,
    )
    return float(
        np.max(
            np.abs(
                a - b
            )
            / np.maximum(
                ha + hb,
                NORMALIZATION_FLOOR_KPA,
            )
        )
    )


def completeF_statewise_separation(
    V,
    cfg,
    response_a: np.ndarray,
    response_b: np.ndarray,
) -> Tuple[
    np.ndarray,
    np.ndarray,
]:
    a = np.asarray(
        response_a,
        dtype=np.float64,
    )
    b = np.asarray(
        response_b,
        dtype=np.float64,
    )

    idx = (
        V68_PROTOCOL.atlas_indices
    )
    ha = V.brush_shape_per_unit_noise(
        cfg,
        V68_PROTOCOL,
        a,
    )[idx]
    hb = V.brush_shape_per_unit_noise(
        cfg,
        V68_PROTOCOL,
        b,
    )[idx]

    component_score = (
        np.abs(
            a[idx]
            - b[idx]
        )
        / np.maximum(
            ha + hb,
            NORMALIZATION_FLOOR_KPA,
        )
    )
    component_score = component_score.reshape(
        V68_PROTOCOL.n_parent_states,
        2,
    )
    state_score = np.max(
        component_score,
        axis=1,
    )

    return (
        state_score,
        component_score,
    )


def pointwise_pair_separation(
    cfg,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
) -> np.ndarray:
    pa = np.asarray(
        prediction_a,
        dtype=np.float64,
    )
    pb = np.asarray(
        prediction_b,
        dtype=np.float64,
    )
    ha = mode_brush_shape(
        cfg,
        pa,
    )
    hb = mode_brush_shape(
        cfg,
        pb,
    )
    return (
        np.abs(
            pa - pb
        )
        / np.maximum(
            ha + hb,
            NORMALIZATION_FLOOR_KPA,
        )
    )


def build_training_pair_table(
    cfg,
    fits: pd.DataFrame,
) -> pd.DataFrame:
    lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }

    rows = []
    for i, model_a in enumerate(
        MODELS
    ):
        for model_b in MODELS[
            i + 1:
        ]:
            row_a = lookup[
                model_a
            ]
            row_b = lookup[
                model_b
            ]

            distance = fixed_pair_required_noise_fraction(
                cfg,
                np.asarray(
                    row_a[
                        "_compression_prediction"
                    ],
                    dtype=np.float64,
                ),
                np.asarray(
                    row_b[
                        "_compression_prediction"
                    ],
                    dtype=np.float64,
                ),
            )

            rows.append(
                {
                    "model_a":
                        model_a,
                    "model_b":
                        model_b,
                    "compression_pair_required_noise_fraction":
                        float(
                            distance
                        ),
                    "compression_pair_required_noise_percent":
                        float(
                            100.0
                            * distance
                        ),
                    "compression_indistinguishable_at_3pct":
                        bool(
                            distance
                            <= ATLAS_TOLERANCE_FRACTION
                            + 1.0e-12
                        ),
                }
            )

    return pd.DataFrame(
        rows
    )


def build_support_sensitivity_table(
    fits: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for threshold in DELTA_BIC_SENSITIVITY:
        models = (
            fits.loc[
                fits[
                    "training_delta_BIC"
                ].astype(float)
                <= float(
                    threshold
                )
                + 1.0e-12,
                "model",
            ]
            .astype(str)
            .tolist()
        )
        rows.append(
            {
                "criterion":
                    f"BIC_only_delta<={threshold:g}",
                "delta_BIC_threshold":
                    float(
                        threshold
                    ),
                "delta_AICc_threshold":
                    np.nan,
                "supported_models":
                    "|".join(
                        models
                    ),
                "n_supported_models":
                    int(
                        len(
                            models
                        )
                    ),
            }
        )

    strict = (
        fits.loc[
            fits[
                "publication_primary_supported"
            ].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )
    rows.append(
        {
            "criterion":
                "JOINT_BIC_AICc_delta<=2",
            "delta_BIC_threshold":
                PRIMARY_PUBLICATION_DELTA_BIC_MAX,
            "delta_AICc_threshold":
                PRIMARY_PUBLICATION_DELTA_AICC_MAX,
            "supported_models":
                "|".join(
                    strict
                ),
            "n_supported_models":
                int(
                    len(
                        strict
                    )
                ),
        }
    )

    return pd.DataFrame(
        rows
    )


def atlas_lookup_for_supported_models(
    V,
    cfg,
    fits: pd.DataFrame,
    certified_atlas: pd.DataFrame,
    supported_models: Sequence[str],
) -> Dict[
    str,
    Dict[str, object],
]:
    fit_lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }

    results = {}
    for model in supported_models:
        lookup = project_experimental_fit_to_frozen_atlas(
            V,
            cfg,
            fit_lookup[
                model
            ],
            certified_atlas,
        )
        results[
            model
        ] = lookup

    return results


def select_primary_atlas_pair_training_only(
    fits: pd.DataFrame,
    training_pairs: pd.DataFrame,
    atlas_lookups: Mapping[
        str,
        Mapping[str, object],
    ],
) -> Tuple[
    Dict[str, object] | None,
    pd.DataFrame,
]:
    """
    Select a directed source->target pair with NO held-out truth.

    Ordering:
      1. source model with best compression BIC support;
      2. target model with best compression BIC support;
      3. larger frozen-atlas certified exclusion as tie-breaker.

    Required:
      - both models jointly BIC/AICc supported;
      - fitted compression curves within 3%;
      - both same-family atlas projections within 3%;
      - frozen source atlas node excludes target at >3%.
    """
    fit_lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }

    supported = [
        str(model)
        for model in atlas_lookups
    ]

    pair_distance = {}
    for _, row in training_pairs.iterrows():
        key = frozenset(
            [
                str(
                    row[
                        "model_a"
                    ]
                ),
                str(
                    row[
                        "model_b"
                    ]
                ),
            ]
        )
        pair_distance[
            key
        ] = float(
            row[
                "compression_pair_required_noise_percent"
            ]
        )

    candidates = []

    for source_model in supported:
        source_lookup = atlas_lookups[
            source_model
        ]

        source_projection = float(
            source_lookup[
                "projection_required_noise_percent"
            ]
        )

        for target_model in supported:
            if (
                target_model
                == source_model
            ):
                continue

            target_lookup = atlas_lookups[
                target_model
            ]
            target_projection = float(
                target_lookup[
                    "projection_required_noise_percent"
                ]
            )

            comp_distance = float(
                pair_distance[
                    frozenset(
                        [
                            source_model,
                            target_model,
                        ]
                    )
                ]
            )

            atlas_score = float(
                source_lookup[
                    "certified_scores_percent"
                ].get(
                    target_model,
                    np.nan,
                )
            )
            reverse_score = float(
                target_lookup[
                    "certified_scores_percent"
                ].get(
                    source_model,
                    np.nan,
                )
            )

            source_row = fit_lookup[
                source_model
            ]
            target_row = fit_lookup[
                target_model
            ]

            eligible = bool(
                np.isfinite(
                    atlas_score
                )
                and comp_distance
                <= 100.0
                * ATLAS_TOLERANCE_FRACTION
                + 1.0e-10
                and source_projection
                <= 100.0
                * ATLAS_TOLERANCE_FRACTION
                + 1.0e-10
                and target_projection
                <= 100.0
                * ATLAS_TOLERANCE_FRACTION
                + 1.0e-10
                and atlas_score
                > 100.0
                * ATLAS_TOLERANCE_FRACTION
            )

            candidates.append(
                {
                    "source_model":
                        source_model,
                    "target_model":
                        target_model,
                    "source_delta_BIC":
                        float(
                            source_row[
                                "training_delta_BIC"
                            ]
                        ),
                    "source_delta_AICc":
                        float(
                            source_row[
                                "training_delta_AICc"
                            ]
                        ),
                    "target_delta_BIC":
                        float(
                            target_row[
                                "training_delta_BIC"
                            ]
                        ),
                    "target_delta_AICc":
                        float(
                            target_row[
                                "training_delta_AICc"
                            ]
                        ),
                    "compression_pair_distance_percent":
                        comp_distance,
                    "source_atlas_projection_percent":
                        source_projection,
                    "target_atlas_projection_percent":
                        target_projection,
                    "frozen_source_to_target_critical_noise_percent":
                        atlas_score,
                    "frozen_target_to_source_critical_noise_percent":
                        reverse_score,
                    "eligible_before_holdout":
                        eligible,
                }
            )

    table = pd.DataFrame(
        candidates
    )

    eligible_table = table[
        table[
            "eligible_before_holdout"
        ].astype(bool)
    ].copy()

    if eligible_table.empty:
        return (
            None,
            table,
        )

    chosen = (
        eligible_table.sort_values(
            [
                "source_delta_BIC",
                "source_delta_AICc",
                "target_delta_BIC",
                "target_delta_AICc",
                "frozen_source_to_target_critical_noise_percent",
                "source_model",
                "target_model",
            ],
            ascending=[
                True,
                True,
                True,
                True,
                False,
                True,
                True,
            ],
        )
        .iloc[0]
        .to_dict()
    )

    return (
        chosen,
        table,
    )


# =============================================================================
# Frozen-atlas measured witness on withheld tension + simple shear
# =============================================================================

def select_brain_measured_witness_from_frozen_atlas_pair(
    V,
    cfg,
    dataset: Mapping[str, object],
    frozen_pair: Mapping[str, object],
) -> Tuple[
    pd.DataFrame,
    Dict[str, object],
]:
    source_model = str(
        frozen_pair[
            "source_model"
        ]
    )
    target_model = str(
        frozen_pair[
            "target_model"
        ]
    )

    source_coordinate = np.asarray(
        frozen_pair[
            "_source_coordinate"
        ],
        dtype=np.float64,
    )
    target_coordinate = np.asarray(
        frozen_pair[
            "_target_coordinate"
        ],
        dtype=np.float64,
    )
    source_scale = float(
        frozen_pair[
            "_source_scale"
        ]
    )
    target_scale = float(
        frozen_pair[
            "_matched_target_scale"
        ]
    )

    records = []

    # Prediction-only selection on withheld TENSION states.
    tension_lam = np.asarray(
        dataset[
            "tension_lambda"
        ],
        dtype=np.float64,
    )
    source_tension = (
        source_scale
        * brain_unit_template(
            V,
            cfg,
            source_model,
            source_coordinate,
            lam=tension_lam,
        )
    )
    target_tension = (
        target_scale
        * brain_unit_template(
            V,
            cfg,
            target_model,
            target_coordinate,
            lam=tension_lam,
        )
    )
    sep_tension = pointwise_pair_separation(
        cfg,
        source_tension,
        target_tension,
    )

    for i, lam in enumerate(
        tension_lam
    ):
        records.append(
            {
                "mode":
                    "tension",
                "local_index":
                    int(i),
                "deformation_parameter":
                    float(
                        lam
                    ),
                "frozen_atlas_source_prediction_kPa":
                    float(
                        source_tension[
                            i
                        ]
                    ),
                "frozen_atlas_target_prediction_kPa":
                    float(
                        target_tension[
                            i
                        ]
                    ),
                "frozen_atlas_predicted_separation_fraction":
                    float(
                        sep_tension[
                            i
                        ]
                    ),
            }
        )

    # Prediction-only selection on withheld SIMPLE-SHEAR states.
    gamma = np.asarray(
        dataset[
            "gamma"
        ],
        dtype=np.float64,
    )
    source_shear = (
        source_scale
        * brain_unit_template(
            V,
            cfg,
            source_model,
            source_coordinate,
            gamma=gamma,
        )
    )
    target_shear = (
        target_scale
        * brain_unit_template(
            V,
            cfg,
            target_model,
            target_coordinate,
            gamma=gamma,
        )
    )
    sep_shear = pointwise_pair_separation(
        cfg,
        source_shear,
        target_shear,
    )

    for i, value in enumerate(
        gamma
    ):
        records.append(
            {
                "mode":
                    "simple_shear",
                "local_index":
                    int(i),
                "deformation_parameter":
                    float(
                        value
                    ),
                "frozen_atlas_source_prediction_kPa":
                    float(
                        source_shear[
                            i
                        ]
                    ),
                "frozen_atlas_target_prediction_kPa":
                    float(
                        target_shear[
                            i
                        ]
                    ),
                "frozen_atlas_predicted_separation_fraction":
                    float(
                        sep_shear[
                            i
                        ]
                    ),
            }
        )

    table = pd.DataFrame(
        records
    )
    selected_index = int(
        table[
            "frozen_atlas_predicted_separation_fraction"
        ]
        .to_numpy(float)
        .argmax()
    )

    table[
        "atlas_preselected_measured_witness"
    ] = False
    table.loc[
        selected_index,
        "atlas_preselected_measured_witness",
    ] = True

    # Only after the witness index is frozen do we attach experimental truth.
    truth = []
    for _, row in table.iterrows():
        mode = str(
            row[
                "mode"
            ]
        )
        local_index = int(
            row[
                "local_index"
            ]
        )

        if mode == "tension":
            truth.append(
                float(
                    dataset[
                        "tension_tau_kpa"
                    ][
                        local_index
                    ]
                )
            )
        elif mode == "simple_shear":
            truth.append(
                float(
                    dataset[
                        "shear_tau_kpa"
                    ][
                        local_index
                    ]
                )
            )
        else:
            raise KeyError(
                mode
            )

    table[
        "experimental_stress_kPa"
    ] = truth
    table[
        "frozen_atlas_predicted_separation_percent"
    ] = (
        100.0
        * table[
            "frozen_atlas_predicted_separation_fraction"
        ]
    )

    selected = table.loc[
        selected_index
    ]

    return (
        table,
        {
            "atlas_selected_measured_witness_mode":
                str(
                    selected[
                        "mode"
                    ]
                ),
            "atlas_selected_measured_witness_local_index":
                int(
                    selected[
                        "local_index"
                    ]
                ),
            "atlas_selected_measured_witness_deformation_parameter":
                float(
                    selected[
                        "deformation_parameter"
                    ]
                ),
            "atlas_selected_measured_witness_predicted_separation_percent":
                float(
                    selected[
                        "frozen_atlas_predicted_separation_percent"
                    ]
                ),
            "atlas_selected_measured_witness_experimental_stress_kPa":
                float(
                    selected[
                        "experimental_stress_kPa"
                    ]
                ),
        },
    )


def evaluate_experimental_pair_at_brain_atlas_witness(
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    source_model: str,
    target_model: str,
    witness_summary: Mapping[str, object],
) -> Dict[str, object]:
    fit_lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }

    mode = str(
        witness_summary[
            "atlas_selected_measured_witness_mode"
        ]
    )
    index = int(
        witness_summary[
            "atlas_selected_measured_witness_local_index"
        ]
    )

    if mode == "tension":
        pred_key = (
            "_tension_prediction"
        )
        truth = float(
            dataset[
                "tension_tau_kpa"
            ][
                index
            ]
        )
    elif mode == "simple_shear":
        pred_key = (
            "_shear_prediction"
        )
        truth = float(
            dataset[
                "shear_tau_kpa"
            ][
                index
            ]
        )
    else:
        raise KeyError(
            mode
        )

    predictions = {}
    errors = {}

    for model in (
        source_model,
        target_model,
    ):
        prediction = float(
            np.asarray(
                fit_lookup[
                    model
                ][
                    pred_key
                ],
                dtype=np.float64,
            )[
                index
            ]
        )
        predictions[
            model
        ] = prediction
        errors[
            model
        ] = abs(
            truth
            - prediction
        )

    preferred_witness = min(
        errors,
        key=errors.get,
    )

    preferred_combined = min(
        (
            source_model,
            target_model,
        ),
        key=lambda model: float(
            fit_lookup[
                model
            ][
                "heldout_combined_nrmse_percent"
            ]
        ),
    )

    return {
        "experimental_truth_kPa":
            truth,
        "source_model":
            source_model,
        "target_model":
            target_model,
        "source_prediction_kPa":
            float(
                predictions[
                    source_model
                ]
            ),
        "target_prediction_kPa":
            float(
                predictions[
                    target_model
                ]
            ),
        "source_abs_error_kPa":
            float(
                errors[
                    source_model
                ]
            ),
        "target_abs_error_kPa":
            float(
                errors[
                    target_model
                ]
            ),
        "atlas_selected_witness_preferred_model":
            preferred_witness,
        "combined_holdout_preferred_model":
            preferred_combined,
        "witness_preference_matches_combined_holdout":
            bool(
                preferred_witness
                == preferred_combined
            ),
    }


# =============================================================================
# Publication figure
# =============================================================================

def make_brain_region_figure(
    dataset: Mapping[str, object],
    fits: pd.DataFrame,
    region_summary: Mapping[str, object],
    outdir: Path,
) -> None:
    fit_lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }

    source_model = str(
        region_summary[
            "primary_source_model"
        ]
    )
    target_model = str(
        region_summary[
            "primary_target_model"
        ]
    )

    source = fit_lookup[
        source_model
    ]
    target = fit_lookup[
        target_model
    ]

    fig = plt.figure(
        figsize=(
            14.8,
            10.2,
        )
    )

    # A. Compression training.
    ax = fig.add_subplot(
        2,
        2,
        1,
    )
    ax.scatter(
        dataset[
            "compression_lambda"
        ],
        dataset[
            "compression_tau_kpa"
        ],
        s=38,
        label="compression training",
        zorder=8,
    )
    ax.plot(
        dataset[
            "compression_lambda"
        ],
        source[
            "_compression_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{source_model}: "
            f"{float(source['compression_nrmse_percent']):.2f}%"
        ),
    )
    ax.plot(
        dataset[
            "compression_lambda"
        ],
        target[
            "_compression_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{target_model}: "
            f"{float(target['compression_nrmse_percent']):.2f}%"
        ),
    )
    ax.set_xlabel(
        "Stretch, λ"
    )
    ax.set_ylabel(
        "Kirchhoff stress (kPa)"
    )
    ax.set_title(
        "A. Compression-only calibration"
    )
    ax.legend(
        fontsize=8,
    )

    # B. Frozen atlas.
    ax = fig.add_subplot(
        2,
        2,
        2,
    )
    values = [
        float(
            region_summary[
                "source_atlas_projection_percent"
            ]
        ),
        float(
            region_summary[
                "target_atlas_projection_percent"
            ]
        ),
        float(
            region_summary[
                "frozen_source_to_target_critical_noise_percent"
            ]
        ),
        float(
            region_summary[
                "frozen_target_to_source_critical_noise_percent"
            ]
        ),
    ]
    labels = [
        f"{source_model}\nprojection",
        f"{target_model}\nprojection",
        f"{source_model}→{target_model}",
        f"{target_model}→{source_model}",
    ]
    ax.bar(
        np.arange(
            len(
                values
            )
        ),
        values,
    )
    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.3,
        label="3% threshold",
    )
    ax.set_xticks(
        np.arange(
            len(
                labels
            )
        )
    )
    ax.set_xticklabels(
        labels
    )
    ax.set_ylabel(
        "Required noise (%)"
    )
    ax.set_title(
        "B. Frozen certified atlas lookup"
    )
    ax.legend(
        fontsize=8,
    )

    witness_mode = str(
        region_summary[
            "atlas_selected_measured_witness_mode"
        ]
    )
    witness_parameter = float(
        region_summary[
            "atlas_selected_measured_witness_deformation_parameter"
        ]
    )

    # C. Tension holdout.
    ax = fig.add_subplot(
        2,
        2,
        3,
    )
    ax.scatter(
        dataset[
            "tension_lambda"
        ],
        dataset[
            "tension_tau_kpa"
        ],
        s=38,
        label="withheld tension",
        zorder=8,
    )
    ax.plot(
        dataset[
            "tension_lambda"
        ],
        source[
            "_tension_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{source_model}: "
            f"{float(source['heldout_tension_nrmse_percent']):.1f}%"
        ),
    )
    ax.plot(
        dataset[
            "tension_lambda"
        ],
        target[
            "_tension_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{target_model}: "
            f"{float(target['heldout_tension_nrmse_percent']):.1f}%"
        ),
    )
    if (
        witness_mode
        == "tension"
    ):
        idx = int(
            region_summary[
                "atlas_selected_measured_witness_local_index"
            ]
        )
        ax.scatter(
            [
                dataset[
                    "tension_lambda"
                ][
                    idx
                ]
            ],
            [
                dataset[
                    "tension_tau_kpa"
                ][
                    idx
                ]
            ],
            s=145,
            marker="*",
            zorder=10,
            label="atlas-selected witness",
        )
    ax.set_xlabel(
        "Stretch, λ"
    )
    ax.set_ylabel(
        "Kirchhoff stress (kPa)"
    )
    ax.set_title(
        "C. Completely withheld tension"
    )
    ax.legend(
        fontsize=8,
    )

    # D. Simple shear holdout.
    ax = fig.add_subplot(
        2,
        2,
        4,
    )
    ax.scatter(
        dataset[
            "gamma"
        ],
        dataset[
            "shear_tau_kpa"
        ],
        s=38,
        label="withheld simple shear",
        zorder=8,
    )
    ax.plot(
        dataset[
            "gamma"
        ],
        source[
            "_shear_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{source_model}: "
            f"{float(source['heldout_shear_nrmse_percent']):.1f}%"
        ),
    )
    ax.plot(
        dataset[
            "gamma"
        ],
        target[
            "_shear_prediction"
        ],
        linewidth=2.5,
        label=(
            f"{target_model}: "
            f"{float(target['heldout_shear_nrmse_percent']):.1f}%"
        ),
    )
    if (
        witness_mode
        == "simple_shear"
    ):
        idx = int(
            region_summary[
                "atlas_selected_measured_witness_local_index"
            ]
        )
        ax.scatter(
            [
                dataset[
                    "gamma"
                ][
                    idx
                ]
            ],
            [
                dataset[
                    "shear_tau_kpa"
                ][
                    idx
                ]
            ],
            s=145,
            marker="*",
            zorder=10,
            label="atlas-selected witness",
        )
    ax.set_xlabel(
        "Simple-shear parameter, γ"
    )
    ax.set_ylabel(
        "Shear stress (kPa)"
    )
    ax.set_title(
        "D. Completely withheld simple shear"
    )
    ax.legend(
        fontsize=8,
    )

    fig.suptitle(
        "Frozen-atlas external validation — "
        f"{dataset['material']}\n"
        f"{source_model}→{target_model}, "
        f"atlas witness: {witness_mode} "
        f"{witness_parameter:.4g}",
        fontsize=14,
    )
    fig.tight_layout(
        rect=[
            0,
            0.01,
            1,
            0.94,
        ]
    )

    save_figure_checked(
        fig,
        outdir
        / "BRAIN_PUBLICATION_FROZEN_ATLAS.png",
        dpi=350,
        bbox_inches="tight",
    )
    save_figure_checked(
        fig,
        outdir
        / "BRAIN_PUBLICATION_FROZEN_ATLAS.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(
        fig
    )


# =============================================================================
# One regional case
# =============================================================================

def validate_brain_region(
    V,
    cfg,
    frozen_assets: Mapping[str, object],
    region_key: str,
    region_index: int,
) -> Dict[str, object]:
    print(
        "\n"
        + "=" * 100
    )
    print(
        f"[V68.6.1] HUMAN BRAIN REGION: {region_key}"
    )
    print(
        "=" * 100
    )

    region_dir = (
        OUTDIR
        / region_key
    )
    region_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw = download_yaml_checked(
        BRAIN_DATASET_URLS[
            region_key
        ],
        region_dir
        / "downloaded_HyperSmart_dataset.yaml",
    )
    dataset = parse_brain_dataset(
        raw
    )

    # Verify all points lie within frozen standard UT/SH protocol ranges.
    all_lam = np.concatenate(
        [
            dataset[
                "compression_lambda"
            ],
            dataset[
                "tension_lambda"
            ],
        ]
    )
    if (
        float(
            np.min(
                all_lam
            )
        )
        < float(
            cfg.ut_min
        )
        - 1.0e-12
        or float(
            np.max(
                all_lam
            )
        )
        > float(
            cfg.ut_max
        )
        + 1.0e-12
    ):
        raise ValueError(
            f"{region_key}: axial data lie outside frozen V68 UT bounds."
        )

    gamma = np.asarray(
        dataset[
            "gamma"
        ],
        dtype=np.float64,
    )
    if (
        float(
            np.min(
                gamma
            )
        )
        < float(
            cfg.sh_min
        )
        - 1.0e-12
        or float(
            np.max(
                gamma
            )
        )
        > float(
            cfg.sh_max
        )
        + 1.0e-12
    ):
        raise ValueError(
            f"{region_key}: shear data lie outside frozen V68 SH bounds."
        )

    save_dataframe_checked(
        pd.DataFrame(
            {
                "stretch":
                    dataset[
                        "compression_lambda"
                    ],
                "kirchhoff_stress_kPa":
                    dataset[
                        "compression_tau_kpa"
                    ],
            }
        ),
        region_dir
        / "TRAIN_compression_only.csv",
        index=False,
    )

    save_dataframe_checked(
        pd.DataFrame(
            {
                "stretch":
                    dataset[
                        "tension_lambda"
                    ],
                "kirchhoff_stress_kPa":
                    dataset[
                        "tension_tau_kpa"
                    ],
            }
        ),
        region_dir
        / "HOLDOUT_tension.csv",
        index=False,
    )

    save_dataframe_checked(
        pd.DataFrame(
            {
                "gamma":
                    dataset[
                        "gamma"
                    ],
                "shear_stress_kPa":
                    dataset[
                        "shear_tau_kpa"
                    ],
            }
        ),
        region_dir
        / "HOLDOUT_simple_shear.csv",
        index=False,
    )

    fits = fit_all_models_to_compression(
        V,
        cfg,
        dataset,
        region_seed=(
            10000
            * region_index
        ),
    )
    fits = add_brain_training_support(
        fits
    )

    public_columns = [
        column
        for column in fits.columns
        if not str(
            column
        ).startswith(
            "_"
        )
    ]
    save_dataframe_checked(
        fits[
            public_columns
        ],
        region_dir
        / "compression_only_model_fits.csv",
        index=False,
    )

    sensitivity = build_support_sensitivity_table(
        fits
    )
    save_dataframe_checked(
        sensitivity,
        region_dir
        / "training_support_sensitivity.csv",
        index=False,
    )

    training_pairs = build_training_pair_table(
        cfg,
        fits,
    )
    save_dataframe_checked(
        training_pairs,
        region_dir
        / "compression_pair_ambiguity.csv",
        index=False,
    )

    strict_supported = (
        fits.loc[
            fits[
                "publication_primary_supported"
            ].astype(bool),
            "model",
        ]
        .astype(str)
        .tolist()
    )

    base_summary = {
        "region_key":
            region_key,
        "material":
            dataset[
                "material"
            ],
        "publication_title":
            dataset[
                "publication_title"
            ],
        "citation":
            dataset[
                "citation"
            ],
        "doi":
            dataset[
                "doi"
            ],
        "data_source":
            dataset[
                "data_source"
            ],
        "n_compression_training_points":
            int(
                len(
                    dataset[
                        "compression_lambda"
                    ]
                )
            ),
        "n_tension_holdout_points":
            int(
                len(
                    dataset[
                        "tension_lambda"
                    ]
                )
            ),
        "n_shear_holdout_points":
            int(
                len(
                    dataset[
                        "gamma"
                    ]
                )
            ),
        "publication_primary_supported_models":
            strict_supported,
        "n_publication_primary_supported_models":
            int(
                len(
                    strict_supported
                )
            ),
        "compression_BIC_winner":
            str(
                fits.sort_values(
                    [
                        "BIC",
                        "compression_nrmse",
                    ]
                ).iloc[0][
                    "model"
                ]
            ),
        "compression_NRMSE_winner":
            str(
                fits.sort_values(
                    [
                        "compression_nrmse",
                        "BIC",
                    ]
                ).iloc[0][
                    "model"
                ]
            ),
        "heldout_combined_NRMSE_winner_all_seven":
            str(
                fits.sort_values(
                    [
                        "heldout_combined_nrmse",
                        "compression_nrmse",
                    ]
                ).iloc[0][
                    "model"
                ]
            ),
        "same_study_note":
            (
                "This is one anatomical-region case from the Budday et al. "
                "human-brain study, not an independent study-level replication."
            ),
    }

    if (
        len(
            strict_supported
        )
        < 2
    ):
        summary = {
            **base_summary,
            "external_validation_eligible":
                False,
            "pre_holdout_failure_reason":
                (
                    "Fewer than two model families satisfy joint "
                    "Delta-BIC<=2 and Delta-AICc<=2 on compression-only training."
                ),
        }
        save_json_checked(
            summary,
            region_dir
            / "regional_validation_summary.json",
        )
        print(
            f"[V68.6.1] {region_key}: NOT ELIGIBLE — "
            f"strict supported models={strict_supported}"
        )
        return summary

    atlas_lookups = atlas_lookup_for_supported_models(
        V,
        cfg,
        fits,
        frozen_assets[
            "certified"
        ],
        strict_supported,
    )

    lookup_public = {
        model: {
            key: value
            for key, value
            in lookup.items()
            if not str(
                key
            ).startswith(
                "_"
            )
        }
        for model, lookup
        in atlas_lookups.items()
    }
    save_json_checked(
        lookup_public,
        region_dir
        / "FROZEN_ATLAS_same_family_lookups.json",
    )

    selected, candidate_table = (
        select_primary_atlas_pair_training_only(
            fits,
            training_pairs,
            atlas_lookups,
        )
    )
    save_dataframe_checked(
        candidate_table,
        region_dir
        / "FROZEN_ATLAS_pre_holdout_pair_candidates.csv",
        index=False,
    )

    if selected is None:
        summary = {
            **base_summary,
            "external_validation_eligible":
                False,
            "pre_holdout_failure_reason":
                (
                    "No joint-BIC/AICc-supported compression-ambiguous pair "
                    "both localized within 3% of the frozen atlas and showed "
                    "frozen-atlas incompatibility above 3%."
                ),
            "frozen_atlas_same_family_lookups":
                lookup_public,
        }
        save_json_checked(
            summary,
            region_dir
            / "regional_validation_summary.json",
        )
        print(
            f"[V68.6.1] {region_key}: NOT ELIGIBLE after frozen-atlas gate."
        )
        return summary

    source_model = str(
        selected[
            "source_model"
        ]
    )
    target_model = str(
        selected[
            "target_model"
        ]
    )
    source_lookup = atlas_lookups[
        source_model
    ]
    target_lookup = atlas_lookups[
        target_model
    ]

    # Post-selection local transfer audit. Pair eligibility/selection above is already
    # frozen and this audit is not allowed to change it.
    local_nearest_source, local_within_source, local_summary_source = audit_local_frozen_atlas_transfer(
        V, cfg, fits.loc[fits['model'].astype(str) == source_model].iloc[0],
        frozen_assets["certified"], target_model, nearest_k=10,
    )
    local_nearest_target, local_within_target, local_summary_target = audit_local_frozen_atlas_transfer(
        V, cfg, fits.loc[fits['model'].astype(str) == target_model].iloc[0],
        frozen_assets["certified"], source_model, nearest_k=10,
    )
    save_dataframe_checked(
        local_nearest_source,
        region_dir / f"FROZEN_ATLAS_local_transfer_nearest10_{source_model}_to_{target_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_within_source,
        region_dir / f"FROZEN_ATLAS_local_transfer_within3pct_{source_model}_to_{target_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_nearest_target,
        region_dir / f"FROZEN_ATLAS_local_transfer_nearest10_{target_model}_to_{source_model}.csv",
        index=False,
    )
    save_dataframe_checked(
        local_within_target,
        region_dir / f"FROZEN_ATLAS_local_transfer_within3pct_{target_model}_to_{source_model}.csv",
        index=False,
    )
    local_transfer_summary = {
        f"{source_model}_to_{target_model}": local_summary_source,
        f"{target_model}_to_{source_model}": local_summary_target,
    }
    save_json_checked(
        local_transfer_summary,
        region_dir / "FROZEN_ATLAS_local_transfer_stability.json",
    )

    frozen_pair = frozen_pairwise_witness_for_source_node(
        V,
        cfg,
        frozen_assets,
        source_lookup,
        target_model,
    )

    frozen_source_row = source_lookup[
        "_frozen_row"
    ]
    frozen_pair[
        "_source_coordinate"
    ] = _coordinate_json(
        frozen_source_row[
            "coordinate_json"
        ]
    )

    fit_lookup = {
        str(row["model"]):
            row
        for _, row
        in fits.iterrows()
    }
    frozen_pair[
        "_source_scale"
    ] = float(
        fit_lookup[
            source_model
        ][
            "best_response_scale_kpa"
        ]
    )

    witness_table, witness_summary = (
        select_brain_measured_witness_from_frozen_atlas_pair(
            V,
            cfg,
            dataset,
            frozen_pair,
        )
    )
    save_dataframe_checked(
        witness_table,
        region_dir
        / "FROZEN_ATLAS_measured_witness_selection.csv",
        index=False,
    )

    experimental_test = (
        evaluate_experimental_pair_at_brain_atlas_witness(
            dataset,
            fits,
            source_model,
            target_model,
            witness_summary,
        )
    )

    # Direct experimental fitted-state geometry is a diagnostic only.
    direct_completeF = float(
        V.required_noise_fraction(
            cfg,
            V68_PROTOCOL,
            np.asarray(
                fit_lookup[
                    source_model
                ][
                    "_full_response"
                ],
                dtype=np.float64,
            ),
            np.asarray(
                fit_lookup[
                    target_model
                ][
                    "_full_response"
                ],
                dtype=np.float64,
            ),
        )
    )

    direct_compression = fixed_pair_required_noise_fraction(
        cfg,
        np.asarray(
            fit_lookup[
                source_model
            ][
                "_compression_prediction"
            ],
            dtype=np.float64,
        ),
        np.asarray(
            fit_lookup[
                target_model
            ][
                "_compression_prediction"
            ],
            dtype=np.float64,
        ),
    )

    reverse_lookup = atlas_lookups[
        target_model
    ]

    summary = {
        **base_summary,
        "external_validation_eligible":
            True,
        "primary_source_model":
            source_model,
        "primary_target_model":
            target_model,
        "source_delta_BIC":
            float(
                selected[
                    "source_delta_BIC"
                ]
            ),
        "source_delta_AICc":
            float(
                selected[
                    "source_delta_AICc"
                ]
            ),
        "target_delta_BIC":
            float(
                selected[
                    "target_delta_BIC"
                ]
            ),
        "target_delta_AICc":
            float(
                selected[
                    "target_delta_AICc"
                ]
            ),
        "compression_pair_distance_percent":
            float(
                selected[
                    "compression_pair_distance_percent"
                ]
            ),
        "source_atlas_projection_percent":
            float(
                selected[
                    "source_atlas_projection_percent"
                ]
            ),
        "target_atlas_projection_percent":
            float(
                selected[
                    "target_atlas_projection_percent"
                ]
            ),
        "source_atlas_certified_compatibility_set":
            str(
                source_lookup[
                    "certified_compatibility_set"
                ]
            ),
        "target_atlas_certified_compatibility_set":
            str(
                reverse_lookup[
                    "certified_compatibility_set"
                ]
            ),
        "frozen_source_to_target_critical_noise_percent":
            float(
                selected[
                    "frozen_source_to_target_critical_noise_percent"
                ]
            ),
        "frozen_target_to_source_critical_noise_percent":
            float(
                selected[
                    "frozen_target_to_source_critical_noise_percent"
                ]
            ),
        "frozen_pairwise_witness_source":
            str(
                frozen_pair[
                    "frozen_target_witness_source"
                ]
            ),
        "frozen_atlas_local_transfer_stability":
            local_transfer_summary,
        "frozen_global_witness_lambda1":
            float(
                frozen_pair[
                    "global_completeF_witness_lambda1"
                ]
            ),
        "frozen_global_witness_lambda2":
            float(
                frozen_pair[
                    "global_completeF_witness_lambda2"
                ]
            ),
        "frozen_global_witness_lambda3":
            float(
                frozen_pair[
                    "global_completeF_witness_lambda3"
                ]
            ),
        **witness_summary,
        **experimental_test,
        "source_tension_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    source_model
                ][
                    "heldout_tension_nrmse_percent"
                ]
            ),
        "target_tension_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    target_model
                ][
                    "heldout_tension_nrmse_percent"
                ]
            ),
        "source_shear_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    source_model
                ][
                    "heldout_shear_nrmse_percent"
                ]
            ),
        "target_shear_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    target_model
                ][
                    "heldout_shear_nrmse_percent"
                ]
            ),
        "source_combined_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    source_model
                ][
                    "heldout_combined_nrmse_percent"
                ]
            ),
        "target_combined_holdout_NRMSE_percent":
            float(
                fit_lookup[
                    target_model
                ][
                    "heldout_combined_nrmse_percent"
                ]
            ),
        "direct_fitted_compression_pair_distance_percent_DIAGNOSTIC":
            float(
                100.0
                * direct_compression
            ),
        "direct_fitted_completeF_pair_distance_percent_DIAGNOSTIC":
            float(
                100.0
                * direct_completeF
            ),
        "validation_success_witness_agrees_with_complete_holdout":
            bool(
                experimental_test[
                    "witness_preference_matches_combined_holdout"
                ]
            ),
        "primary_interpretation":
            (
                "Both families were supported using compression only and were "
                "practically indistinguishable on that training protocol. "
                "After same-family localization in the precomputed certified "
                "V68 atlas, the frozen source node excluded the target family "
                "above the declared 3% tolerance. The atlas relation then "
                "selected a measured tension/shear deformation before withheld "
                "experimental stress was consulted."
            ),
    }

    save_json_checked(
        summary,
        region_dir
        / "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json",
    )

    make_brain_region_figure(
        dataset,
        fits,
        summary,
        region_dir,
    )

    print(
        f"[V68.6.1] {region_key}: ELIGIBLE | "
        f"{source_model}->{target_model} | "
        f"atlas critical noise="
        f"{summary['frozen_source_to_target_critical_noise_percent']:.3f}% | "
        f"witness={summary['atlas_selected_measured_witness_mode']} "
        f"{summary['atlas_selected_measured_witness_deformation_parameter']:.4g} | "
        f"agreement="
        f"{summary['validation_success_witness_agrees_with_complete_holdout']}"
    )

    return summary


# =============================================================================
# Aggregate soft-tissue validation
# =============================================================================

def make_brain_aggregate_figure(
    summary_df: pd.DataFrame,
) -> None:
    eligible = summary_df[
        summary_df[
            "external_validation_eligible"
        ].astype(bool)
    ].copy()

    if eligible.empty:
        return

    fig = plt.figure(
        figsize=(
            12.5,
            5.3,
        )
    )

    ax = fig.add_subplot(
        1,
        2,
        1,
    )
    x = np.arange(
        len(
            eligible
        )
    )
    ax.bar(
        x,
        eligible[
            "frozen_source_to_target_critical_noise_percent"
        ].to_numpy(float),
    )
    ax.axhline(
        3.0,
        linestyle="--",
        linewidth=1.3,
    )
    ax.set_xticks(
        x
    )
    ax.set_xticklabels(
        eligible[
            "region_key"
        ].astype(str),
        rotation=25,
        ha="right",
    )
    ax.set_ylabel(
        "Frozen atlas critical noise (%)"
    )
    ax.set_title(
        "A. Atlas-predicted incompatibility"
    )

    ax = fig.add_subplot(
        1,
        2,
        2,
    )
    source_err = eligible[
        "source_combined_holdout_NRMSE_percent"
    ].to_numpy(float)
    target_err = eligible[
        "target_combined_holdout_NRMSE_percent"
    ].to_numpy(float)

    ax.scatter(
        source_err,
        target_err,
        s=75,
    )
    maximum = max(
        float(
            np.max(
                source_err
            )
        ),
        float(
            np.max(
                target_err
            )
        ),
        1.0,
    )
    ax.plot(
        [
            0.0,
            maximum,
        ],
        [
            0.0,
            maximum,
        ],
        linestyle="--",
        linewidth=1.2,
    )

    for _, row in eligible.iterrows():
        ax.annotate(
            str(
                row[
                    "region_key"
                ]
            ),
            (
                float(
                    row[
                        "source_combined_holdout_NRMSE_percent"
                    ]
                ),
                float(
                    row[
                        "target_combined_holdout_NRMSE_percent"
                    ]
                ),
            ),
            fontsize=8,
        )

    ax.set_xlabel(
        "Source-model holdout NRMSE (%)"
    )
    ax.set_ylabel(
        "Target-model holdout NRMSE (%)"
    )
    ax.set_title(
        "B. Independent tension+shear holdout"
    )

    fig.suptitle(
        "Human-brain soft-tissue frozen-atlas validation",
        fontsize=14,
    )
    fig.tight_layout(
        rect=[
            0,
            0.01,
            1,
            0.94,
        ]
    )

    save_figure_checked(
        fig,
        OUTDIR
        / "BRAIN_AGGREGATE_FROZEN_ATLAS_VALIDATION.png",
        dpi=350,
        bbox_inches="tight",
    )
    save_figure_checked(
        fig,
        OUTDIR
        / "BRAIN_AGGREGATE_FROZEN_ATLAS_VALIDATION.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(
        fig
    )


def validate_all_brain_regions(
    V,
    cfg,
) -> Dict[str, object]:
    frozen_assets = load_and_verify_frozen_atlas_assets(
        V
    )

    summaries = []

    for region_index, region_key in enumerate(
        BRAIN_REGION_ORDER
    ):
        try:
            summary = validate_brain_region(
                V,
                cfg,
                frozen_assets,
                region_key,
                region_index,
            )
        except Exception as exc:
            region_dir = (
                OUTDIR
                / region_key
            )
            region_dir.mkdir(
                parents=True,
                exist_ok=True,
            )
            error_summary = {
                "region_key":
                    region_key,
                "external_validation_eligible":
                    False,
                "runtime_error":
                    str(
                        exc
                    ),
                "traceback":
                    "".join(
                        traceback.format_exception(
                            type(
                                exc
                            ),
                            exc,
                            exc.__traceback__,
                        )
                    ),
            }
            save_json_checked(
                error_summary,
                region_dir
                / "regional_runtime_error.json",
            )
            print(
                f"[V68.6.1] {region_key}: ERROR — {exc}"
            )
            summary = error_summary

        summaries.append(
            summary
        )

    summary_df = pd.DataFrame(
        summaries
    )
    save_dataframe_checked(
        summary_df,
        OUTDIR
        / "ALL_BRAIN_REGIONS_summary.csv",
        index=False,
    )

    eligible_mask = (
        summary_df[
            "external_validation_eligible"
        ]
        .fillna(False)
        .astype(bool)
    )
    eligible = summary_df[
        eligible_mask
    ].copy()

    if len(
        eligible
    ):
        success_mask = (
            eligible[
                "validation_success_witness_agrees_with_complete_holdout"
            ]
            .fillna(False)
            .astype(bool)
        )
        n_success = int(
            success_mask.sum()
        )
    else:
        n_success = 0

    aggregate = {
        "dataset":
            "Budday et al. 2017 human brain — HyperSmart machine-readable data",
        "doi":
            BUDDAY_DOI,
        "n_regional_cases":
            int(
                len(
                    summary_df
                )
            ),
        "n_independent_studies":
            1,
        "regional_cases":
            list(
                BRAIN_REGION_ORDER
            ),
        "training_protocol":
            "uniaxial compression only",
        "heldout_protocols":
            [
                "uniaxial tension",
                "simple shear",
            ],
        "primary_support_rule":
            "Delta-BIC<=2 AND Delta-AICc<=2 on compression only",
        "n_regions_eligible_before_holdout":
            int(
                eligible_mask.sum()
            ),
        "fraction_regions_eligible_before_holdout":
            float(
                eligible_mask.mean()
            ),
        "n_eligible_regions_where_atlas_witness_agrees_with_combined_holdout":
            n_success,
        "fraction_eligible_regions_where_atlas_witness_agrees_with_combined_holdout":
            float(
                n_success
                / max(
                    len(
                        eligible
                    ),
                    1,
                )
            ),
        "important_scope_note":
            (
                "The four anatomical regions are regional cases from one "
                "human-brain study. Eligibility and atlas-pair selection use "
                "compression data plus frozen-atlas outputs only. Withheld "
                "tension/shear stresses are consulted only after the atlas "
                "witness is selected."
            ),
    }

    save_json_checked(
        aggregate,
        OUTDIR
        / "BRAIN_AGGREGATE_VALIDATION_SUMMARY.json",
    )

    if len(
        eligible
    ):
        make_brain_aggregate_figure(
            summary_df
        )

    print(
        "\n"
        + "=" * 100
    )
    print(
        "[V68.6.1] HUMAN-BRAIN SOFT-TISSUE AGGREGATE"
    )
    print(
        "=" * 100
    )
    print(
        json.dumps(
            aggregate,
            indent=2,
            allow_nan=True,
        )
    )

    return aggregate


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    global V68_PROTOCOL, V68_BASES

    # FIRST runtime action: establish persistent Google Drive.
    drive_root = mount_google_drive_first()

    # Bind to the exact existing V68 root before dependency import/download.
    run_dir = configure_exact_visible_drive_output(
        drive_root
    )

    print(
        "[V68.6.1] Loading bundled checksum-locked frozen V68.1..."
    )
    V = import_v68_module()
    cfg = V.BASE_CFG

    V68_PROTOCOL = V.make_protocol(
        cfg
    )
    V68_BASES = V.constitutive_bases(
        V68_PROTOCOL
    )

    print(
        "[V68.6.1] Human-brain soft-tissue validation design:"
    )
    print(
        "           TRAIN = uniaxial compression only"
    )
    print(
        "           HOLDOUT = uniaxial tension + simple shear"
    )
    print(
        "           PRIMARY RESULT = precomputed certified V68 atlas lookup"
    )
    print(
        "           held-out stresses are not used for pair/witness selection"
    )
    print(
        f"[V68.6.1] Complete-F parent: "
        f"{V68_PROTOCOL.n_parent_states} states, "
        f"{len(V68_PROTOCOL.atlas_indices)} response components."
    )

    self_test_brain_response_evaluator(
        V,
        cfg,
    )

    validate_all_brain_regions(
        V,
        cfg,
    )

    write_run_complete_marker(
        run_dir
    )
    write_verified_artifact_manifest(
        run_dir
    )

    print_saved_artifact_manifest(
        run_dir
    )

    print(
        "\n"
        + "=" * 100
    )
    print(
        "[V68.6.1] RUN COMPLETED AND VERIFIED ON GOOGLE DRIVE"
    )
    print(
        "=" * 100
    )
    print(
        f"Persistent output directory:\n{run_dir}"
    )
    print(
        f"Root-level visibility marker:\n"
        f"{ROOT / VISIBLE_ROOT_MARKER_NAME}"
    )


if __name__ == "__main__":
    try:
        main()
    except BaseException as exc:
        write_run_error_marker(
            exc
        )
        raise


[V68.6.1] STEP 1/4 — ENSURING GOOGLE DRIVE IS AVAILABLE FIRST...
[V68.6.1] Existing Google Drive mount detected; reusing it instead of remounting a non-empty mountpoint.
[V68.6.1] Google Drive available at: /content/drive/MyDrive
[V68.6.1] Exact existing V68 root:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
[V68.6.1] Existing anchor folders detected: cell2_publication_validation, publication_figures_3_5
[V68.6.1] EXACT V68-ROOT DRIVE OUTPUT VERIFIED.
[V68.6.1] Output folder created DIRECTLY in the visible V68 root:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT
[V68.6.1] Root-level marker created:
           /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT_LOCATION.txt
[V68.6.1] Parent-direct

/tmp/ipykernel_2405/631029627.py:6255: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


[V68.6.1][DRIVE SAVED] /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT/BRAIN_AGGREGATE_FROZEN_ATLAS_VALIDATION.png (241.26 KiB)
[V68.6.1][DRIVE SAVED] /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT/BRAIN_AGGREGATE_FROZEN_ATLAS_VALIDATION.pdf (17.87 KiB)

[V68.6.1] HUMAN-BRAIN SOFT-TISSUE AGGREGATE
{
  "dataset": "Budday et al. 2017 human brain \u2014 HyperSmart machine-readable data",
  "doi": "10.1016/j.actbio.2016.10.036",
  "n_regional_cases": 4,
  "n_independent_studies": 1,
  "regional_cases": [
    "cortex",
    "basal_ganglia",
    "corona_radiata",
    "corpus_callosum"
  ],
  "training_protocol": "uniaxial compression only",
  "heldout_protocols": [
    "uniaxial tension",
    "simple shear"
  ],
  "primary_support_rule": "Delta-BIC<=2 AND Delta-AICc<=2 on compression only"

In [1]:
"""Standalone Colab cell: regenerate corrected manuscript Figures 3--9.

Run this cell by itself. It mounts Google Drive if necessary, discovers the
completed atlas project, loads the frozen CSV/JSON/NPZ results created by the
earlier notebook cells, and writes presentation-only updates. No fitting or
optimization is rerun.

Optional override before running:
    os.environ["ATLAS_ROOT"] = "/content/drive/MyDrive/.../project_folder"
"""

from __future__ import annotations

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from PIL import Image


# -----------------------------------------------------------------------------
# Configuration and publication style
# -----------------------------------------------------------------------------

PROJECT_BASENAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)
OUTPUT_FOLDER = "UPDATED_MANUSCRIPT_FIGURES"
MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
PCA_VIEWS = ["ALL", "UT", "BT", "SH"]
PNG_DPI = int(os.environ.get("UPDATED_FIGURE_DPI", "600"))
OGDEN_ALPHA_SERIES_THRESHOLD = 1.0e-6
GENT_SINGULARITY_MARGIN = 1.0e-8

MODEL_SHORT = {
    "NH": "NH",
    "MR": "MR",
    "YEOH2": "Y2",
    "GENT": "GE",
    "OGDEN1": "O1",
    "OGDEN2": "O2",
    "GP2": "GP2",
}

REGION_PALETTE = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
    "#56B4E9", "#000000", "#7A5195", "#EF5675", "#FFA600",
    "#4C78A8", "#F58518", "#54A24B", "#B279A2", "#9D755D",
    "#BAB0AC", "#72B7B2", "#FF9DA6", "#A0CBE8", "#8CD17D",
]

PROTOCOL_COLORS = {
    "ALL": "#333333",
    "UT": "#0072B2",
    "BT": "#D55E00",
    "SH": "#009E73",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 10.2,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.7,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})


# -----------------------------------------------------------------------------
# Drive discovery, input checks, exports, and metadata stripping
# -----------------------------------------------------------------------------

def _atlas_anchor(root: Path) -> Path:
    return root / "cell2_publication_validation" / "certified_source_atlas_states.csv"


def mount_drive_if_needed() -> None:
    if Path("/content/drive/MyDrive").is_dir():
        return
    try:
        from google.colab import drive  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "Google Drive is not mounted and this is not a Colab runtime. "
            "Mount Drive or set ATLAS_ROOT to a locally visible project folder."
        ) from exc
    drive.mount("/content/drive", force_remount=False)


def resolve_root() -> Path:
    explicit = os.environ.get("ATLAS_ROOT", "").strip() or os.environ.get(
        "V68_ROOT", ""
    ).strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    for mydrive in (
        Path("/content/drive/MyDrive"),
        Path("/content/v68_drive/MyDrive"),
        Path("/content/v68_figures_drive/MyDrive"),
    ):
        candidates.append(mydrive / "Optimal_Protocol" / PROJECT_BASENAME)
    for root in candidates:
        if _atlas_anchor(root).is_file():
            return root.resolve()

    mount_drive_if_needed()
    for root in candidates:
        if _atlas_anchor(root).is_file():
            return root.resolve()

    search_base = Path("/content/drive/MyDrive/Optimal_Protocol")
    if search_base.is_dir():
        matches = sorted(
            p.parent.parent
            for p in search_base.glob(
                "*/cell2_publication_validation/certified_source_atlas_states.csv"
            )
        )
        matches = [p for p in matches if _atlas_anchor(p).is_file()]
        if len(matches) == 1:
            return matches[0].resolve()
        if len(matches) > 1:
            listing = "\n".join(f"  - {p}" for p in matches)
            raise RuntimeError(
                "Multiple completed atlas projects were found. Set ATLAS_ROOT "
                "to the intended one:\n" + listing
            )
    raise FileNotFoundError(
        "Could not find the completed atlas project. Set ATLAS_ROOT to the exact "
        "Drive folder containing cell2_publication_validation/."
    )


def require(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"Required saved result is missing:\n{path}")
    return path


def read_json(path: Path) -> Mapping:
    with require(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def find_result_dir(root: Path, patterns: Sequence[str], required_name: str) -> Path:
    for pattern in patterns:
        for candidate in sorted(root.glob(pattern)):
            if candidate.is_dir() and (candidate / required_name).is_file():
                return candidate
    raise FileNotFoundError(
        f"Could not find a saved result directory containing {required_name} "
        f"under {root}. Run the corresponding original generator cell once."
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_figure(fig: plt.Figure, outdir: Path, stem: str) -> dict[str, str]:
    """Save journal-ready outputs while removing identifying image metadata."""
    outdir.mkdir(parents=True, exist_ok=True)
    png = outdir / f"{stem}.png"
    pdf = outdir / f"{stem}.pdf"
    tif = outdir / f"{stem}.tiff"

    fig.savefig(png, dpi=PNG_DPI, bbox_inches="tight", metadata={})
    # Re-encode raster files without textual metadata; preserve only resolution.
    with Image.open(png) as image:
        clean = image.copy()
    clean.save(png, format="PNG", dpi=(PNG_DPI, PNG_DPI), optimize=True)
    clean.save(
        tif,
        format="TIFF",
        dpi=(PNG_DPI, PNG_DPI),
        compression="tiff_lzw",
        tiffinfo={},
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
        metadata={
            "Title": "",
            "Author": "",
            "Subject": "",
            "Keywords": "",
            "Creator": "",
            "Producer": "",
            "CreationDate": None,
            "ModDate": None,
        },
    )
    return {"png": str(png), "pdf": str(pdf), "tiff": str(tif)}


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.11, 1.06, label,
        transform=ax.transAxes,
        fontsize=13, fontweight="bold", ha="left", va="top",
    )


def clean_axes(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def compact_signature(signature: str) -> str:
    return "·".join(
        MODEL_SHORT.get(piece, piece)
        for piece in str(signature).split("|")
        if piece
    )


def as_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes"}


def annotate_bars(
    ax: plt.Axes,
    bars,
    values: Sequence[float],
    *,
    fmt: str = ".2f",
    fontsize: float = 7.5,
) -> None:
    values = np.asarray(values, dtype=float)
    ymax = max(float(np.nanmax(np.abs(values))) if len(values) else 0.0, 1.0)
    for bar, value in zip(bars, values):
        if not np.isfinite(value):
            continue
        y = value + 0.018 * ymax if value >= 0 else value - 0.018 * ymax
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y,
            format(float(value), fmt),
            ha="center",
            va="bottom" if value >= 0 else "top",
            fontsize=fontsize,
            clip_on=False,
        )
        if abs(value) < 1.0e-12:
            ax.scatter(
                [bar.get_x() + bar.get_width() / 2], [0.0],
                marker="D", s=18, color=bar.get_facecolor(), zorder=5,
            )


def source_row(certified: pd.DataFrame, model: str, index: int) -> pd.Series:
    hit = certified[
        (certified["source_model"].astype(str) == str(model))
        & (certified["source_index"].astype(int) == int(index))
    ]
    if len(hit) != 1:
        raise RuntimeError(
            f"Expected one frozen source row for {model} #{index}; found {len(hit)}."
        )
    return hit.iloc[0]


# -----------------------------------------------------------------------------
# Frozen PCA helpers for manuscript Figures 3 and 4
# -----------------------------------------------------------------------------

def pca_metadata(root: Path, view: str) -> tuple[np.ndarray, int]:
    npz = root / f"compatibility_regions_pca_{view}_model.npz"
    if npz.is_file():
        with np.load(npz, allow_pickle=True) as saved:
            ratio = np.asarray(saved["explained_variance_ratio"], dtype=float)
            dimension = (
                int(len(saved["selected_component_indices"]))
                if "selected_component_indices" in saved
                else -1
            )
        return ratio, dimension
    summary = pd.read_csv(require(root / "compatibility_regions_pca_protocol_summary.csv"))
    row = summary[summary["protocol_view"].astype(str) == view]
    if len(row) != 1:
        raise RuntimeError(f"Cannot resolve PCA metadata for {view}.")
    r = row.iloc[0]
    return np.asarray([
        r["pc1_explained_variance"],
        r["pc2_explained_variance"],
        r["pc3_explained_variance"],
    ], dtype=float), int(r["response_dimension"])


def pca_view(root: Path, certified: pd.DataFrame, view: str) -> tuple[pd.DataFrame, np.ndarray, int]:
    coords = pd.read_csv(require(root / f"compatibility_regions_pca_{view}_coordinates.csv"))
    label_col = "certified_compatibility_set_at_3pct"
    labels = certified[["source_model", "source_index", label_col]].copy()
    work = coords.merge(
        labels,
        on=["source_model", "source_index"],
        how="left",
        validate="one_to_one",
    )
    if work[label_col].isna().any():
        raise RuntimeError(f"{view} PCA coordinates do not align to all certified labels.")
    work["certified_label"] = work[label_col].astype(str)
    ratio, dimension = pca_metadata(root, view)
    return work, ratio, dimension


def region_key(certified: pd.DataFrame) -> tuple[list[str], dict[str, str], pd.DataFrame]:
    label_col = "certified_compatibility_set_at_3pct"
    counts = (
        certified.groupby(label_col).size().reset_index(name="n_states")
        .sort_values(["n_states", label_col], ascending=[False, True])
        .reset_index(drop=True)
    )
    counts["fraction"] = counts["n_states"] / counts["n_states"].sum()
    order = counts[label_col].astype(str).tolist()
    colors = {label: REGION_PALETTE[i % len(REGION_PALETTE)] for i, label in enumerate(order)}
    counts["short_label"] = [compact_signature(x) for x in order]
    counts["color_hex"] = [colors[x] for x in order]
    return order, colors, counts


def make_figure3(root: Path, certified: pd.DataFrame, outdir: Path) -> dict[str, str]:
    order, colors, counts = region_key(certified)
    work, ratio, _ = pca_view(root, certified, "ALL")
    fig, axes = plt.subplots(
        1, 2, figsize=(13.4, 6.5), layout="constrained",
        gridspec_kw={"width_ratios": [1.28, 1.0]},
    )
    ax = axes[0]
    for label in order:
        group = work[work["certified_label"] == label]
        ax.scatter(
            group["PC1"], group["PC2"], s=11, alpha=0.64,
            color=colors[label], edgecolors="none", rasterized=True,
        )
    ax.set_xlabel(f"PC1 ({100 * ratio[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({100 * ratio[1]:.1f}%)")
    ax.set_title("Scale-free parent-domain PCA")
    ax.text(
        0.02, 0.02,
        f"PC1–PC2: {100 * np.sum(ratio[:2]):.1f}% variance\n"
        "Compatibility evaluated in the original 5,202-D space",
        transform=ax.transAxes, ha="left", va="bottom", fontsize=8.0,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.8", alpha=0.9),
    )
    clean_axes(ax); panel_label(ax, "A")

    ax = axes[1]
    shown = counts.iloc[::-1].copy()
    y = np.arange(len(shown))
    bars = ax.barh(y, shown["n_states"], color=shown["color_hex"], alpha=0.90)
    ax.set_yticks(y, shown["short_label"])
    ax.set_xlabel("Certified source states")
    ax.set_title("Compatibility-set frequency")
    xmax = float(shown["n_states"].max())
    ax.set_xlim(0, 1.24 * xmax)
    for bar, n, frac in zip(bars, shown["n_states"], shown["fraction"]):
        percentage = 100.0 * float(frac)
        percentage_text = (
            "<0.1%"
            if 0.0 < percentage < 0.1
            else f"{percentage:.1f}%"
        )
        ax.text(
            bar.get_width() + 0.015 * xmax,
            bar.get_y() + bar.get_height() / 2,
            f"{int(n)}  ({percentage_text})",
            ha="left", va="center", fontsize=7.5,
        )
    clean_axes(ax); panel_label(ax, "B")
    fig.suptitle("Geometry of the frozen constitutive-response atlas", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_3_Updated_Atlas_Geometry")


def make_figure4(root: Path, certified: pd.DataFrame, outdir: Path) -> dict[str, str]:
    order, colors, _ = region_key(certified)
    views, ratios, dims = {}, {}, {}
    for view in PCA_VIEWS:
        views[view], ratios[view], dims[view] = pca_view(root, certified, view)

    visibility_path = root / "cell2_publication_validation" / "subprotocol_complete_F_label_visibility.csv"
    visibility = pd.read_csv(require(visibility_path))
    visibility = (
        visibility[visibility["protocol_view"].astype(str).isin(PCA_VIEWS)]
        .drop_duplicates("protocol_view")
        .set_index("protocol_view").loc[PCA_VIEWS].reset_index()
    )

    fig = plt.figure(figsize=(13.6, 10.0), layout="constrained")
    grid = fig.add_gridspec(
        3, 3,
        height_ratios=[1.0, 1.0, 0.32],
    )
    flat = np.asarray([
        fig.add_subplot(grid[row, column])
        for row in range(2)
        for column in range(3)
    ])
    legend_ax = fig.add_subplot(grid[2, :])
    legend_ax.axis("off")
    for panel_index, view in enumerate(PCA_VIEWS):
        ax = flat[panel_index]
        work = views[view]
        for label in order:
            group = work[work["certified_label"] == label]
            if len(group):
                ax.scatter(
                    group["PC1"], group["PC2"], s=7.5, alpha=0.52,
                    color=colors[label], edgecolors="none", rasterized=True,
                )
        ratio = ratios[view]
        ax.set_xlabel(f"PC1 ({100 * ratio[0]:.1f}%)")
        ax.set_ylabel(f"PC2 ({100 * ratio[1]:.1f}%)")
        ax.set_title(f"{view}: d={dims[view]:,}; PC1+PC2={100 * np.sum(ratio[:2]):.1f}%")
        clean_axes(ax); panel_label(ax, chr(ord("A") + panel_index))

    ax = flat[4]
    dim_values = np.asarray([dims[v] for v in PCA_VIEWS], dtype=float)
    bars = ax.bar(np.arange(4), dim_values, color=[PROTOCOL_COLORS[v] for v in PCA_VIEWS])
    ax.set_yscale("log")
    ax.set_xticks(np.arange(4), PCA_VIEWS)
    ax.set_ylabel("Response dimension (log scale)")
    ax.set_title("Available response dimension")
    for bar, value in zip(bars, dim_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, value * 1.12, f"{int(value):,}",
            ha="center", va="bottom", fontsize=7.5,
        )
    clean_axes(ax); panel_label(ax, "E")

    ax = flat[5]
    y = np.arange(len(PCA_VIEWS), dtype=float)
    offset = 0.10
    ba_mean = visibility["balanced_accuracy_mean"].to_numpy(dtype=float)
    mf_mean = visibility["macro_f1_mean"].to_numpy(dtype=float)
    ba_std = visibility.get(
        "balanced_accuracy_std", pd.Series(np.zeros(len(visibility)))
    ).to_numpy(dtype=float)
    mf_std = visibility.get(
        "macro_f1_std", pd.Series(np.zeros(len(visibility)))
    ).to_numpy(dtype=float)

    # Pair the two scores on one row per protocol. This keeps the uncertainty
    # visible without the bars and value labels that previously crowded Panel F.
    for row, ba_value, mf_value in zip(y, ba_mean, mf_mean):
        ax.plot(
            [ba_value, mf_value], [row, row],
            color="0.78", linewidth=1.1, zorder=1,
        )
    ax.errorbar(
        ba_mean, y - offset, xerr=ba_std,
        fmt="o", markersize=6.0, color="#0072B2", ecolor="#0072B2",
        elinewidth=1.2, capsize=3, label="Balanced accuracy", zorder=3,
    )
    ax.errorbar(
        mf_mean, y + offset, xerr=mf_std,
        fmt="s", markersize=5.5, color="#D55E00", ecolor="#D55E00",
        elinewidth=1.2, capsize=3, label="Macro-F1", zorder=3,
    )
    score_min = float(np.nanmin(np.r_[ba_mean - ba_std, mf_mean - mf_std]))
    x_left = max(0.0, np.floor((score_min - 0.08) * 10.0) / 10.0)
    ax.set_xlim(x_left, 1.01)
    ax.set_yticks(y, PCA_VIEWS)
    ax.invert_yaxis()
    ax.set_xlabel("Repeated-CV score")
    ax.set_ylabel("Protocol view")
    ax.set_title("Visibility of fixed parent-domain labels")
    ax.xaxis.grid(True, linestyle=":", linewidth=0.8, color="0.84")
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right", fontsize=7.8)
    clean_axes(ax); panel_label(ax, "F")

    handles = [
        Line2D(
            [0], [0], marker="o", linestyle="", markersize=5.2,
            markerfacecolor=colors[label], markeredgecolor="none",
            label=compact_signature(label),
        )
        for label in order
    ]
    legend_ax.legend(
        handles=handles,
        title="Compatibility sets",
        loc="center",
        ncol=5,
        frameon=False,
        fontsize=8.0,
        title_fontsize=8.6,
        columnspacing=1.35,
        handletextpad=0.35,
    )
    fig.suptitle("Loss of label visibility under restricted protocols", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_4_Updated_Protocol_Visibility")


# -----------------------------------------------------------------------------
# Compact constitutive-response helpers copied from the notebook generators
# -----------------------------------------------------------------------------

def json_array(value) -> np.ndarray:
    return np.asarray(json.loads(str(value)), dtype=float).reshape(-1)


def ogden_shifted_term(alpha: float, log_lam: np.ndarray) -> np.ndarray:
    alpha = float(alpha)
    values = np.asarray(log_lam, dtype=float)
    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            values + 0.5 * alpha * values**2 + alpha**2 * values**3 / 6.0
            + alpha**3 * values**4 / 24.0 + alpha**4 * values**5 / 120.0
        )
    return 4.0 * np.expm1(alpha * values) / alpha


def invariants(lam: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    lam2 = np.asarray(lam, dtype=float) ** 2
    return np.sum(lam2, axis=-1), (
        lam2[..., 0] * lam2[..., 1]
        + lam2[..., 1] * lam2[..., 2]
        + lam2[..., 2] * lam2[..., 0]
    )


def principal_tau_shape(model: str, physical: Mapping, lam: np.ndarray) -> np.ndarray:
    lam = np.asarray(lam, dtype=float)
    i1, i2 = invariants(lam)
    x, y = i1 - 3.0, i2 - 3.0
    if model == "NH":
        return 2.0 * lam**2
    if model == "MR":
        rho = float(physical["rho"])
        return 2.0 * ((1.0 - rho) * lam**2 - rho * lam**-2.0)
    if model == "YEOH2":
        beta = float(physical["beta"])
        return 2.0 * (1.0 + 2.0 * beta * x)[..., None] * lam**2
    if model == "GENT":
        gamma = float(physical["gamma"])
        denominator = 1.0 - gamma * x
        out = np.full_like(lam, np.nan)
        valid = denominator > GENT_SINGULARITY_MARGIN
        out[valid] = 2.0 * lam[valid] ** 2 / denominator[valid, None]
        return out
    if model == "OGDEN1":
        return ogden_shifted_term(float(physical["alpha"]), np.log(lam))
    if model == "OGDEN2":
        a1, a2 = float(physical["alpha1"]), float(physical["alpha2"])
        w1 = float(physical["weight1"])
        return w1 * ogden_shifted_term(a1, np.log(lam)) + (1.0 - w1) * ogden_shifted_term(a2, np.log(lam))
    if model == "GP2":
        rho = float(physical["rho"])
        b20, b11, b02 = (float(physical[k]) for k in ("beta20", "beta11", "beta02"))
        w1 = (1.0 - rho) + 2.0 * b20 * x + b11 * y
        w2 = rho + b11 * x + 2.0 * b02 * y
        return 2.0 * (w1[..., None] * lam**2 - w2[..., None] * lam**-2.0)
    raise KeyError(model)


def source_physical(row: pd.Series) -> dict:
    model = str(row["source_model"])
    coordinate = json_array(row["coordinate_json"])
    if model == "NH":
        return {}
    if model == "MR":
        return {"rho": float(row.get("physical_rho", coordinate[0]))}
    if model == "YEOH2":
        return {"beta": float(row.get("physical_beta", coordinate[0]))}
    if model == "GENT":
        return {"gamma": float(row.get("physical_gamma", coordinate[0]))}
    if model == "OGDEN1":
        return {"alpha": float(row.get("physical_alpha", coordinate[0]))}
    if model == "OGDEN2":
        return {
            "alpha1": float(row["physical_alpha1"]),
            "alpha2": float(row["physical_alpha2"]),
            "weight1": float(row["physical_weight1"]),
        }
    if model == "GP2":
        return {key: float(row[f"physical_{key}"]) for key in ("rho", "beta20", "beta11", "beta02")}
    raise KeyError(model)


def simple_shear_curve(model: str, physical: Mapping, scale: float, gamma: np.ndarray) -> np.ndarray:
    result = np.zeros_like(np.asarray(gamma, dtype=float))
    for i, value in enumerate(np.asarray(gamma, dtype=float)):
        deformation = np.array([[1.0, value, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
        left_cauchy = deformation @ deformation.T
        eigenvalues, eigenvectors = np.linalg.eigh(left_cauchy)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
        lam = np.sqrt(np.maximum(eigenvalues, 0.0))[None, :]
        q = principal_tau_shape(model, physical, lam)[0]
        tau = eigenvectors @ np.diag(float(scale) * q) @ eigenvectors.T
        result[i] = tau[0, 1]
    return result


def standard_curves(model: str, physical: Mapping, scale: float) -> dict[str, np.ndarray]:
    ut = np.linspace(0.70, 1.50, 81)
    bt = np.linspace(0.80, 1.25, 81)
    sh = np.linspace(-0.50, 0.50, 81)
    ut_lam = np.stack([ut, ut**-0.5, ut**-0.5], axis=-1)
    bt_lam = np.stack([bt, bt, bt**-2.0], axis=-1)
    ut_tau = principal_tau_shape(model, physical, ut_lam)
    bt_tau = principal_tau_shape(model, physical, bt_lam)
    return {
        "UT": float(scale) * (ut_tau[:, 0] - ut_tau[:, 2]),
        "BT": float(scale) * (bt_tau[:, 0] - bt_tau[:, 2]),
        "SH": simple_shear_curve(model, physical, scale, sh),
    }


def plot_protocol_triplet(ax: plt.Axes, truth: Mapping, prediction: Mapping, truth_label: str, pred_label: str) -> None:
    axes = {
        "UT": np.linspace(0.0, 1.0, len(truth["UT"])),
        "BT": np.linspace(1.25, 2.25, len(truth["BT"])),
        "SH": np.linspace(2.50, 3.50, len(truth["SH"])),
    }
    for mode in ("UT", "BT", "SH"):
        ax.plot(axes[mode], truth[mode], linewidth=2.6, label=truth_label if mode == "UT" else None)
        ax.plot(
            axes[mode], prediction[mode], linestyle="--", linewidth=2.2,
            label=pred_label if mode == "UT" else None,
        )
    ax.axvline(1.125, linewidth=0.8, alpha=0.35, color="0.35")
    ax.axvline(2.375, linewidth=0.8, alpha=0.35, color="0.35")
    ax.set_xticks([0.5, 1.75, 3.0], ["UT\nλ=0.70–1.50", "BT\nλ=0.80–1.25", "SH\nγ=−0.50–0.50"])
    ax.set_ylabel("Pressure-independent stress response (kPa)")


# -----------------------------------------------------------------------------
# Manuscript Figure 5: certified set and extrapolation
# -----------------------------------------------------------------------------

def make_figure5(root: Path, certified: pd.DataFrame, outdir: Path) -> dict[str, str]:
    directory = find_result_dir(
        root, ["cell3_example1_certified_set_outlier*", "*example1*"],
        "example1_outlier_predictions.csv",
    )
    result = read_json(directory / "example1_selected_source.json")
    curves = pd.read_csv(require(directory / "example1_outlier_predictions.csv"))
    witnesses = pd.read_csv(require(directory / "example1_certified_witnesses.csv"))
    row = source_row(certified, str(result["source_model"]), int(result["source_index"]))

    lam = curves["lambda"].to_numpy(float)
    truth = curves["hidden_truth"].to_numpy(float)
    prediction_columns = [c for c in curves.columns if c.endswith("_certified_witness")]
    model_columns = {c.removesuffix("_certified_witness"): c for c in prediction_columns}
    compatible_models = [x for x in str(result["certified_compatibility_set"]).split("|") if x]
    outlier = float(result["fixed_outlier_lambda"])
    out_index = int(np.argmin(np.abs(lam - outlier)))
    inside = lam <= 2.0 + 1.0e-12

    fig, axes = plt.subplots(2, 2, figsize=(14.3, 10.0), layout="constrained")
    ax = axes[0, 0]
    for model in compatible_models:
        ax.plot(lam[inside], curves.loc[inside, model_columns[model]], label=MODEL_SHORT[model])
    ax.plot(lam[inside], truth[inside], "k--", linewidth=2.7, label=f"Hidden source ({MODEL_SHORT[str(result['source_model'])]})")
    ax.set_xlim(float(lam.min()), 2.0)
    ax.set_xlabel("Uniaxial stretch, λ"); ax.set_ylabel("Nominal stress (kPa)")
    ax.set_title("Compatible-family projections inside the certified domain\nSet: " + compact_signature(str(result["certified_compatibility_set"])))
    ax.legend(ncol=2); clean_axes(ax); panel_label(ax, "A")

    ax = axes[0, 1]
    scores = np.asarray([100.0 * float(row[f"certified_critical_noise_to_{m.lower()}"]) for m in MODELS])
    compatible = np.asarray([m in compatible_models for m in MODELS])
    colors = np.where(compatible, "#0072B2", "#BDBDBD")
    bars = ax.bar(np.arange(len(MODELS)), scores, color=colors, alpha=0.92)
    ax.axhline(3.0, color="#D55E00", linestyle="--", linewidth=1.5, label="3% declared tolerance")
    ax.set_ylim(bottom=0, top=max(3.75, 1.20 * float(np.nanmax(scores))))
    ax.set_xticks(np.arange(len(MODELS)), [MODEL_SHORT[m] for m in MODELS])
    ax.set_ylabel("Certified required noise (%)")
    ax.set_title("Parent-domain compatibility scores\nAll values, including zeros, are shown")
    annotate_bars(ax, bars, scores, fmt=".2f", fontsize=7.2)
    ax.legend(); clean_axes(ax); panel_label(ax, "B")

    ax = axes[1, 0]
    ax.axvspan(float(lam.min()), 2.0, color="#0072B2", alpha=0.08, label="Certified domain")
    ax.axvspan(2.0, float(lam.max()), color="#D55E00", alpha=0.06, label="Out-of-domain extrapolation")
    for model in compatible_models:
        ax.plot(lam, curves[model_columns[model]], label=MODEL_SHORT[model])
    ax.plot(lam, truth, "k--", linewidth=2.7, label="Hidden truth")
    ax.axvline(outlier, color="0.25", linestyle=":", linewidth=1.8, label=f"Prespecified λ={outlier:.2f}")
    ax.scatter([outlier], [truth[out_index]], marker="*", s=160, color="#E69F00", edgecolor="black", zorder=8)
    ax.set_xlabel("Uniaxial stretch, λ"); ax.set_ylabel("Nominal stress (kPa)")
    ax.set_title(f"Compatible witnesses diverge outside the atlas\nSpread at λ={outlier:.2f}: {float(result['spread_percent_at_fixed_outlier']):.1f}%")
    ax.legend(ncol=2, fontsize=7.5); clean_axes(ax); panel_label(ax, "C")

    ax = axes[1, 1]
    finite = witnesses["finite_at_fixed_outlier"].map(as_bool).to_numpy()
    errors = witnesses["error_at_fixed_outlier_percent"].to_numpy(float)
    x = np.arange(len(witnesses))
    bars = ax.bar(x[finite], errors[finite], color="#0072B2")
    ax.axhline(
        float(result["miss_tolerance_percent"]), color="#D55E00", linestyle="--",
        linewidth=1.5, label=f"{float(result['miss_tolerance_percent']):.0f}% error threshold",
    )
    for index in x[~finite]:
        ax.text(index, 0.02, "no finite\ncontinuation", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_xticks(x, [MODEL_SHORT.get(str(m), str(m)) for m in witnesses["target_model"]])
    ax.set_ylabel(f"Relative error at λ={outlier:.2f} (%)")
    ax.set_title(
        "Risk from choosing a single compatible family\n"
        f"{int(result['n_models_miss_tolerance'])}/{int(result['n_finite_models_at_fixed_outlier'])} finite families exceed the error threshold"
    )
    if len(bars):
        annotate_bars(ax, bars, errors[finite], fmt=".1f", fontsize=7.2)
    ax.legend(); clean_axes(ax); panel_label(ax, "D")
    fig.suptitle("Certified model-set preservation exposes extrapolation risk", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_5_Updated_Certified_Set_Extrapolation")


# -----------------------------------------------------------------------------
# Manuscript Figure 6: complexity required in the parent domain
# -----------------------------------------------------------------------------

def make_figure6(root: Path, certified: pd.DataFrame, outdir: Path) -> dict[str, str]:
    directory = find_result_dir(
        root, ["cell3_example2_complexity_required*", "*example2*"],
        "example2_complete_F_error_map.csv",
    )
    result = read_json(directory / "example2_selected_source.json")
    fit_summary = pd.read_csv(require(directory / "example2_standard_protocol_fit_summary.csv"))
    error_map = pd.read_csv(require(directory / "example2_complete_F_error_map.csv"))
    fstar = read_json(directory / "example2_Fstar.json")
    row = source_row(certified, str(result["source_model"]), int(result["source_index"]))
    source_model = str(result["source_model"])
    simple_model = str(result["best_simple_model"])

    source_truth = standard_curves(
        source_model, source_physical(row), float(row["source_response_scale_kpa"])
    )
    simple_row = fit_summary[fit_summary["model"].astype(str) == simple_model].iloc[0]
    shape_name = str(simple_row.get("shape_parameter_name", ""))
    simple_physical = {} if simple_model == "NH" else {shape_name: float(simple_row["shape_parameter"])}
    simple_fit = standard_curves(simple_model, simple_physical, float(simple_row["response_scale_kpa"]))

    fig, axes = plt.subplots(2, 2, figsize=(14.4, 10.2), layout="constrained")
    ax = axes[0, 0]
    plot_protocol_triplet(
        ax, source_truth, simple_fit,
        f"{MODEL_SHORT[source_model]} hidden source",
        f"{MODEL_SHORT[simple_model]} conventional fit",
    )
    ax.set_title(
        "Standard protocols make the simpler model look adequate\n"
        f"Pooled NRMSE = {float(result['standard_protocol_nrmse_percent']):.2f}%"
    )
    ax.legend(); clean_axes(ax); panel_label(ax, "A")

    ax = axes[0, 1]
    required = np.asarray([100.0 * float(row[f"certified_critical_noise_to_{m.lower()}"]) for m in MODELS])
    bars = ax.bar(np.arange(len(MODELS)), required, color="#0072B2")
    ax.axhline(3.0, color="#D55E00", linestyle="--", linewidth=1.5, label="3% declared tolerance")
    ax.set_ylim(0, max(3.8, 1.23 * float(np.nanmax(required))))
    ax.set_xticks(np.arange(len(MODELS)), [MODEL_SHORT[m] for m in MODELS])
    ax.set_ylabel("Certified required noise (%)")
    ax.set_title(
        "Parent-domain certification rejects the simpler explanation\n"
        f"{MODEL_SHORT[simple_model]} requires {float(result['best_simple_complete_F_required_noise_percent']):.2f}%"
    )
    annotate_bars(ax, bars, required, fmt=".2f", fontsize=7.1)
    ax.legend(); clean_axes(ax); panel_label(ax, "B")

    ax = axes[1, 0]
    scatter = ax.scatter(
        error_map["e1"], error_map["e2"], c=error_map["relative_error_percent"],
        s=14, cmap="viridis", rasterized=True,
    )
    colorbar = fig.colorbar(scatter, ax=ax, shrink=0.84, pad=0.025)
    colorbar.set_label(f"{MODEL_SHORT[simple_model]} vs {MODEL_SHORT[source_model]} relative error (%)", labelpad=9)
    ax.scatter([fstar["e1"]], [fstar["e2"]], s=150, marker="*", color="#E69F00", edgecolor="black", linewidth=0.9, label="Maximum-discrimination state", zorder=7)
    ax.set_xlabel("Principal log strain e₁ = log λ₁")
    ax.set_ylabel("Principal log strain e₂ = log λ₂")
    ax.set_title(
        "Parent domain localizes the missing complexity\n"
        f"Maximum error = {float(fstar['relative_error_percent']):.1f}%"
    )
    ax.legend(); clean_axes(ax); panel_label(ax, "C")

    ax = axes[1, 1]
    truth_values = np.asarray([fstar["truth_d13_kpa"], fstar["truth_d23_kpa"]], dtype=float)
    simple_values = np.asarray([fstar["simple_d13_kpa"], fstar["simple_d23_kpa"]], dtype=float)
    x = np.arange(2); width = 0.36
    truth_bars = ax.bar(x - width / 2, truth_values, width, label=f"{MODEL_SHORT[source_model]} hidden source", color="#0072B2")
    simple_bars = ax.bar(x + width / 2, simple_values, width, label=f"{MODEL_SHORT[simple_model]} fit", color="#D55E00")
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_xticks(x, [r"$\tau_1-\tau_3$", r"$\tau_2-\tau_3$"])
    ax.set_ylabel("Principal Kirchhoff stress difference (kPa)")
    ax.set_title(
        "A specific in-domain state exposes the difference\n"
        f"λ=({float(fstar['lambda1']):.3f}, {float(fstar['lambda2']):.3f}, {float(fstar['lambda3']):.3f})"
    )
    annotate_bars(ax, truth_bars, truth_values, fmt=".2f", fontsize=7.4)
    annotate_bars(ax, simple_bars, simple_values, fmt=".2f", fontsize=7.4)
    ax.legend(); clean_axes(ax); panel_label(ax, "D")
    fig.suptitle("Parent-domain characterization identifies necessary constitutive complexity", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_6_Updated_Complexity_Required")


# -----------------------------------------------------------------------------
# Manuscript Figure 7: no candidate family is compatible
# -----------------------------------------------------------------------------

def withheld_curves(b: float, scale: float) -> dict[str, np.ndarray]:
    ut = np.linspace(0.70, 1.50, 81)
    bt = np.linspace(0.80, 1.25, 81)
    sh = np.linspace(-0.50, 0.50, 81)
    ut_lam = np.stack([ut, ut**-0.5, ut**-0.5], axis=-1)
    bt_lam = np.stack([bt, bt, bt**-2.0], axis=-1)

    def q(lam: np.ndarray) -> np.ndarray:
        i1, _ = invariants(lam)
        return 2.0 * np.exp(float(b) * (i1 - 3.0))[..., None] * lam**2

    result = {
        "UT": float(scale) * np.diff(q(ut_lam)[:, [2, 0]], axis=1).reshape(-1),
        "BT": float(scale) * np.diff(q(bt_lam)[:, [2, 0]], axis=1).reshape(-1),
    }
    shear = np.zeros_like(sh)
    for i, value in enumerate(sh):
        deformation = np.array([[1.0, value, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
        left_cauchy = deformation @ deformation.T
        eigenvalues, eigenvectors = np.linalg.eigh(left_cauchy)
        order = np.argsort(eigenvalues)[::-1]
        lam = np.sqrt(np.maximum(eigenvalues[order], 0.0))[None, :]
        vectors = eigenvectors[:, order]
        tau = vectors @ np.diag(float(scale) * q(lam)[0]) @ vectors.T
        shear[i] = tau[0, 1]
    result["SH"] = shear
    return result


def make_figure7(root: Path, outdir: Path) -> dict[str, str]:
    directory = find_result_dir(
        root, ["cell3_example3_none_compatible*", "*example3*"],
        "example3_standard_fit_summary.csv",
    )
    standard = pd.read_csv(require(directory / "example3_standard_fit_summary.csv"))
    parent = pd.read_csv(require(directory / "example3_completeF_continuous_parent.csv"))
    validation = pd.read_csv(require(directory / "example3_completeF_continuous_validation.csv"))
    error_map = pd.read_csv(require(directory / "example3_validation_error_map.csv"))
    fstars = read_json(directory / "example3_Fstar.json")
    summary = read_json(directory / "example3_summary.json")
    winner = standard[standard["is_conventional_winner"].map(as_bool)].iloc[0]
    winner_model = str(winner["model"])
    physical = json.loads(str(winner["physical_json"]))
    truth = withheld_curves(
        float(summary["withheld_truth"]["b"]),
        float(summary["withheld_truth"]["response_scale_kpa"]),
    )
    prediction = standard_curves(winner_model, physical, float(winner["scale_kpa"]))
    global_fstar = fstars["global_Fstar"]
    interior_fstar = fstars["interior_Fstar"]

    fig, axes = plt.subplots(2, 2, figsize=(14.4, 10.2), layout="constrained")
    ax = axes[0, 0]
    plot_protocol_triplet(ax, truth, prediction, "Withheld exponential-I₁ response", f"{MODEL_SHORT[winner_model]} conventional fit")
    ax.set_title(
        "Standard protocols give an apparently excellent fit\n"
        f"Pooled NRMSE = {float(winner['standard_nrmse_percent']):.4f}%"
    )
    ax.legend(); clean_axes(ax); panel_label(ax, "A")

    ax = axes[0, 1]
    ranked = standard.sort_values("standard_nrmse_percent")
    values = ranked["standard_nrmse_percent"].to_numpy(float)
    bars = ax.bar(np.arange(len(ranked)), values, color="#0072B2")
    positive = values[values > 0]
    if len(positive):
        lower = max(float(positive.min()) / 1.8, 1.0e-3)
        upper = max(float(positive.max()) * 1.8, lower * 10)
        ax.set_yscale("log"); ax.set_ylim(lower, upper)
    ax.set_xticks(np.arange(len(ranked)), [MODEL_SHORT[str(x)] for x in ranked["model"]])
    ax.set_ylabel("Pooled UT+BT+SH NRMSE (%) — log scale")
    ax.set_title("Relative ranking returns a winner\nThe 3% compatibility tolerance is not an NRMSE cutoff")
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value * 1.10, f"{value:.3g}", ha="center", va="bottom", fontsize=7.1)
    clean_axes(ax); panel_label(ax, "B")

    ax = axes[1, 0]
    merged = pd.DataFrame({"model": MODELS}).merge(
        parent[["model", "required_noise_percent"]].rename(columns={"required_noise_percent": "parent"}),
        on="model",
    ).merge(
        validation[["model", "required_noise_percent"]].rename(columns={"required_noise_percent": "validation"}),
        on="model",
    )
    x = np.arange(len(MODELS)); width = 0.36
    pbar = ax.bar(x - width / 2, merged["parent"], width, label="Frozen parent grid", color="#0072B2")
    vbar = ax.bar(x + width / 2, merged["validation"], width, label="Independent dense validation", color="#D55E00")
    ax.axhline(3.0, color="0.25", linestyle="--", linewidth=1.4, label="3% declared tolerance")
    ax.axhline(5.0, color="#009E73", linestyle=":", linewidth=1.4, label="5% robust-rejection margin")
    ax.set_ylim(0, 1.22 * max(float(merged["parent"].max()), float(merged["validation"].max()), 5.0))
    ax.set_xticks(x, [MODEL_SHORT[m] for m in MODELS])
    ax.set_ylabel("Continuously optimized required noise (%)")
    ax.set_title("No candidate family is compatible with the parent domain")
    annotate_bars(ax, pbar, merged["parent"], fmt=".2f", fontsize=6.8)
    annotate_bars(ax, vbar, merged["validation"], fmt=".2f", fontsize=6.8)
    ax.legend(fontsize=7.2); clean_axes(ax); panel_label(ax, "C")

    ax = axes[1, 1]
    scatter = ax.scatter(
        error_map["e1"], error_map["e2"], c=error_map["pointwise_required_noise_percent"],
        s=14, cmap="viridis", rasterized=True,
    )
    colorbar = fig.colorbar(scatter, ax=ax, shrink=0.84, pad=0.025)
    colorbar.set_label(f"{MODEL_SHORT[winner_model]} pointwise required noise (%)", labelpad=9)
    ax.scatter([global_fstar["e1"]], [global_fstar["e2"]], marker="*", s=150, color="#E69F00", edgecolor="black", label="Global maximum", zorder=7)
    ax.scatter([interior_fstar["e1"]], [interior_fstar["e2"]], marker="D", s=65, color="#CC79A7", edgecolor="black", label="Deformed-interior maximum", zorder=7)
    ax.set_xlabel("Principal log strain e₁ = log λ₁")
    ax.set_ylabel("Principal log strain e₂ = log λ₂")
    ax.set_title(
        "Failure is localized to identifiable in-domain states\n"
        f"Global {float(global_fstar['pointwise_required_noise_percent']):.1f}%; interior {float(interior_fstar['pointwise_required_noise_percent']):.1f}%"
    )
    ax.legend(); clean_axes(ax); panel_label(ax, "D")
    fig.suptitle("A strong conventional fit can still leave no compatible family", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_7_Updated_No_Compatible_Family")


# -----------------------------------------------------------------------------
# External-validation response helpers and manuscript Figures 8 and 9
# -----------------------------------------------------------------------------

def ogden_difference(alpha: float, log_a: np.ndarray, log_b: np.ndarray) -> np.ndarray:
    alpha = float(alpha)
    a, b = np.asarray(log_a, dtype=float), np.asarray(log_b, dtype=float)
    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            (a - b) + 0.5 * alpha * (a**2 - b**2)
            + alpha**2 * (a**3 - b**3) / 6.0
            + alpha**3 * (a**4 - b**4) / 24.0
            + alpha**4 * (a**5 - b**5) / 120.0
        )
    return 4.0 * (np.exp(alpha * a) - np.exp(alpha * b)) / alpha


def diagonal_first_template(model: str, coordinate: np.ndarray, states: np.ndarray) -> np.ndarray:
    lam = np.asarray(states, dtype=float)
    l1, l2, l3 = lam[:, 0], lam[:, 1], lam[:, 2]
    b1 = 2.0 * (l1**2 - l3**2)
    b2 = 2.0 * (l3**-2.0 - l1**-2.0)
    i1m3 = l1**2 + l2**2 + l3**2 - 3.0
    i2m3 = l1**-2.0 + l2**-2.0 + l3**-2.0 - 3.0
    coordinate = np.asarray(coordinate, dtype=float).reshape(-1)
    if model == "NH":
        return b1
    if model == "MR":
        return (1.0 - coordinate[0]) * b1 + coordinate[0] * b2
    if model == "YEOH2":
        return b1 * (1.0 + 2.0 * coordinate[0] * i1m3)
    if model == "GENT":
        return b1 / (1.0 - coordinate[0] * i1m3)
    if model == "OGDEN1":
        return ogden_difference(coordinate[0], np.log(l1), np.log(l3))
    raise KeyError(f"Compact external-validation renderer does not need {model}.")


def brain_template(model: str, coordinate: np.ndarray, *, lam=None, gamma=None) -> np.ndarray:
    coordinate = np.asarray(coordinate, dtype=float).reshape(-1)
    if (lam is None) == (gamma is None):
        raise ValueError("Provide exactly one of lam or gamma.")
    if lam is not None:
        x = np.asarray(lam, dtype=float)
        b1 = 2.0 * (x**2 - x**-1.0)
        b2 = 2.0 * (x - x**-2.0)
        i1m3 = x**2 + 2.0 / x - 3.0
        def o1(alpha):
            return ogden_difference(alpha, np.log(x), -0.5 * np.log(x))
    else:
        g = np.asarray(gamma, dtype=float)
        b1 = b2 = 2.0 * g
        i1m3 = g**2
        def o1(alpha):
            root = np.sqrt(1.0 + 0.25 * g**2)
            l1, l2 = root + 0.5 * g, root - 0.5 * g
            return ogden_difference(alpha, np.log(l1), np.log(l2)) / np.sqrt(g**2 + 4.0)
    if model == "NH":
        return b1
    if model == "MR":
        return (1.0 - coordinate[0]) * b1 + coordinate[0] * b2
    if model == "YEOH2":
        return b1 * (1.0 + 2.0 * coordinate[0] * i1m3)
    if model == "GENT":
        return b1 / (1.0 - coordinate[0] * i1m3)
    if model == "OGDEN1":
        return o1(coordinate[0])
    raise KeyError(f"Compact external-validation renderer does not need {model}.")


def make_figure8(root: Path, outdir: Path) -> dict[str, str]:
    directory = find_result_dir(
        root, ["TRELOAR_VALIDATION_OUTPUT", "*TRELOAR*VALIDATION*"],
        "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json",
    )
    result = read_json(directory / "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json")
    fits = pd.read_csv(require(directory / "Treloar_uniaxial_fit_results.csv"))
    train = pd.read_csv(require(directory / "v68_domain_uniaxial.csv"))
    planar = pd.read_csv(require(directory / "v68_domain_pure_shear_planar.csv"))
    biaxial = pd.read_csv(require(directory / "v68_domain_equibiaxial.csv"))
    pair = result["frozen_directed_pairwise_witness"]
    atlas = result["atlas_source_lookup"]
    witness = result["atlas_selected_measured_witness"]
    source_model, target_model = str(pair["source_model"]), str(pair["target_model"])
    selected_models = [source_model, target_model]
    fit_lookup = {str(row["model"]): row for _, row in fits.iterrows()}

    def predict(model: str, frame: pd.DataFrame) -> np.ndarray:
        row = fit_lookup[model]
        coordinate = json_array(row["best_coordinate_json"])
        states = frame[["lambda1", "lambda2", "lambda3"]].to_numpy(float)
        return float(row["best_response_scale_kpa"]) * diagonal_first_template(model, coordinate, states)

    fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.6), layout="constrained")
    ax = axes[0]
    ax.scatter(train["stretch"], train["v68_tau13_kPa"], s=44, color="black", label="Treloar uniaxial data", zorder=8)
    for model in selected_models:
        row = fit_lookup[model]
        ax.plot(train["stretch"], predict(model, train), linewidth=2.6, label=f"{MODEL_SHORT[model]}: {float(row['uniaxial_nrmse_percent']):.2f}% NRMSE")
    ax.set_xlabel("Uniaxial stretch, λ"); ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title("Uniaxial calibration only")
    ax.legend(); clean_axes(ax); panel_label(ax, "A")

    ax = axes[1]
    projection = float(atlas["projection_required_noise_percent"])
    atlas_score = float(atlas["certified_scores_percent"][target_model])
    values = np.asarray([projection, atlas_score])
    bars = ax.bar(np.arange(2), values, color=["#0072B2", "#D55E00"])
    ax.axhline(3.0, color="0.25", linestyle="--", linewidth=1.4, label="3% declared tolerance")
    ax.set_ylim(0, max(3.8, 1.25 * float(values.max())))
    ax.set_xticks(np.arange(2), [f"{MODEL_SHORT[source_model]} fit\n→ atlas", f"Certified\n{MODEL_SHORT[source_model]}→{MODEL_SHORT[target_model]}"])
    ax.set_ylabel("Required noise (%)")
    ax.set_title("Frozen atlas lookup")
    annotate_bars(ax, bars, values, fmt=".3f")
    ax.legend(); clean_axes(ax); panel_label(ax, "B")

    mode = str(witness["atlas_selected_measured_witness_mode"])
    witness_stretch = float(witness["atlas_selected_measured_witness_stretch"])
    if mode == "equibiaxial":
        frame, x_label, title_mode = biaxial, "Equibiaxial stretch, λ", "equibiaxial"
    elif mode == "pure_shear_planar":
        frame, x_label, title_mode = planar, "Planar/pure-shear stretch, λ", "planar/pure-shear"
    else:
        raise KeyError(f"Unexpected Treloar witness mode: {mode}")
    ax = axes[2]
    ax.scatter(frame["stretch"], frame["v68_tau13_kPa"], s=44, color="black", label=f"Held-out {title_mode} data", zorder=8)
    for model in selected_models:
        ax.plot(frame["stretch"], predict(model, frame), linewidth=2.6, label=f"{MODEL_SHORT[model]} calibration prediction")
    selected = frame[np.isclose(frame["stretch"].to_numpy(float), witness_stretch, rtol=0.0, atol=1.0e-12)]
    if len(selected) != 1:
        raise RuntimeError("The Treloar atlas-selected witness was not found uniquely.")
    ax.scatter([witness_stretch], [float(selected["v68_tau13_kPa"].iloc[0])], s=170, marker="*", color="#E69F00", edgecolor="black", label="Atlas-selected witness", zorder=10)
    ax.set_xlabel(x_label); ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title(f"Held-out test at atlas-selected witness\n{title_mode}, λ={witness_stretch:.3f}")
    ax.legend(fontsize=7.5); clean_axes(ax); panel_label(ax, "C")
    fig.suptitle("Held-out validation with Treloar rubber", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_8_Updated_Treloar_Heldout_Validation")


def make_figure9(root: Path, outdir: Path) -> dict[str, str]:
    base = find_result_dir(
        root, ["SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT", "*BRAIN*VALIDATION*"],
        "ALL_BRAIN_REGIONS_summary.csv",
    )
    candidates = []
    for result_path in base.glob("*/FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json"):
        result = read_json(result_path)
        if bool(result.get("external_validation_eligible", False)):
            candidates.append((result_path.parent, result))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one eligible brain region; found {len(candidates)} under {base}.")
    directory, result = candidates[0]
    fits = pd.read_csv(require(directory / "compression_only_model_fits.csv"))
    compression = pd.read_csv(require(directory / "TRAIN_compression_only.csv"))
    tension = pd.read_csv(require(directory / "HOLDOUT_tension.csv"))
    shear = pd.read_csv(require(directory / "HOLDOUT_simple_shear.csv"))
    source_model = str(result["primary_source_model"])
    target_model = str(result["primary_target_model"])
    fit_lookup = {str(row["model"]): row for _, row in fits.iterrows()}

    def predict(model: str, *, lam=None, gamma=None) -> np.ndarray:
        row = fit_lookup[model]
        coordinate = json_array(row["best_coordinate_json"])
        return float(row["best_response_scale_kpa"]) * brain_template(model, coordinate, lam=lam, gamma=gamma)

    fig, axes = plt.subplots(2, 2, figsize=(14.4, 10.2), layout="constrained")
    ax = axes[0, 0]
    xcomp = compression["stretch"].to_numpy(float)
    ax.scatter(xcomp, compression["kirchhoff_stress_kPa"], s=40, color="black", label="Compression calibration data", zorder=8)
    for model in (source_model, target_model):
        row = fit_lookup[model]
        ax.plot(xcomp, predict(model, lam=xcomp), linewidth=2.6, label=f"{MODEL_SHORT[model]}: {float(row['compression_nrmse_percent']):.2f}% NRMSE")
    ax.set_xlabel("Stretch, λ"); ax.set_ylabel("Kirchhoff stress (kPa)")
    ax.set_title("Compression-only calibration")
    ax.legend(); clean_axes(ax); panel_label(ax, "A")

    ax = axes[0, 1]
    values = np.asarray([
        float(result["source_atlas_projection_percent"]),
        float(result["target_atlas_projection_percent"]),
        float(result["frozen_source_to_target_critical_noise_percent"]),
        float(result["frozen_target_to_source_critical_noise_percent"]),
    ])
    labels = [
        f"{MODEL_SHORT[source_model]}\nprojection",
        f"{MODEL_SHORT[target_model]}\nprojection",
        f"{MODEL_SHORT[source_model]}→{MODEL_SHORT[target_model]}",
        f"{MODEL_SHORT[target_model]}→{MODEL_SHORT[source_model]}",
    ]
    bars = ax.bar(np.arange(4), values, color=["#0072B2", "#56B4E9", "#D55E00", "#E69F00"])
    positive = values[values > 0]
    if len(positive):
        ax.set_yscale("log")
        ax.set_ylim(max(float(positive.min()) / 2.1, 1.0e-2), max(float(values.max()) * 2.0, 4.2))
    ax.axhline(3.0, color="0.25", linestyle="--", linewidth=1.4, label="3% declared tolerance")
    ax.set_xticks(np.arange(4), labels)
    ax.set_ylabel("Required noise (%) — log scale")
    ax.set_title("Frozen certified-atlas lookup\nProjection and directed scores shown on one visible scale")
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value * 1.13, f"{value:.3f}%", ha="center", va="bottom", fontsize=7.4)
    ax.legend(); clean_axes(ax); panel_label(ax, "B")

    witness_mode = str(result["atlas_selected_measured_witness_mode"])
    witness_parameter = float(result["atlas_selected_measured_witness_deformation_parameter"])
    witness_index = int(result["atlas_selected_measured_witness_local_index"])
    ax = axes[1, 0]
    xtension = tension["stretch"].to_numpy(float)
    ax.scatter(xtension, tension["kirchhoff_stress_kPa"], s=40, color="black", label="Held-out tension", zorder=8)
    for model in (source_model, target_model):
        row = fit_lookup[model]
        ax.plot(xtension, predict(model, lam=xtension), linewidth=2.6, label=f"{MODEL_SHORT[model]}: {float(row['heldout_tension_nrmse_percent']):.1f}% NRMSE")
    if witness_mode == "tension":
        ax.scatter([xtension[witness_index]], [float(tension["kirchhoff_stress_kPa"].iloc[witness_index])], s=150, marker="*", color="#E69F00", edgecolor="black", label=f"Atlas-selected λ={witness_parameter:.2f}", zorder=10)
    ax.set_xlabel("Stretch, λ"); ax.set_ylabel("Kirchhoff stress (kPa)")
    title_suffix = f"; atlas-selected λ={witness_parameter:.2f}" if witness_mode == "tension" else ""
    ax.set_title("Held-out tension" + title_suffix)
    ax.legend(); clean_axes(ax); panel_label(ax, "C")

    ax = axes[1, 1]
    gamma = shear["gamma"].to_numpy(float)
    ax.scatter(gamma, shear["shear_stress_kPa"], s=40, color="black", label="Held-out simple shear", zorder=8)
    for model in (source_model, target_model):
        row = fit_lookup[model]
        ax.plot(gamma, predict(model, gamma=gamma), linewidth=2.6, label=f"{MODEL_SHORT[model]}: {float(row['heldout_shear_nrmse_percent']):.1f}% NRMSE")
    if witness_mode == "simple_shear":
        ax.scatter([gamma[witness_index]], [float(shear["shear_stress_kPa"].iloc[witness_index])], s=150, marker="*", color="#E69F00", edgecolor="black", label=f"Atlas-selected γ={witness_parameter:.2f}", zorder=10)
    ax.set_xlabel("Simple-shear parameter, γ"); ax.set_ylabel("Shear stress (kPa)")
    ax.set_title("Held-out simple shear")
    ax.legend(); clean_axes(ax); panel_label(ax, "D")
    fig.suptitle("Held-out validation with human cortex data", fontsize=13, fontweight="bold")
    return save_figure(fig, outdir, "Figure_9_Updated_Human_Cortex_Heldout_Validation")


# -----------------------------------------------------------------------------
# One-shot execution
# -----------------------------------------------------------------------------

ROOT = resolve_root()
CELL2 = ROOT / "cell2_publication_validation"
OUTDIR = ROOT / OUTPUT_FOLDER
OUTDIR.mkdir(parents=True, exist_ok=True)
CERTIFIED = pd.read_csv(require(CELL2 / "certified_source_atlas_states.csv"))

print("=" * 92)
print("STANDALONE REGENERATION OF UPDATED MANUSCRIPT FIGURES 3–9")
print("=" * 92)
print(f"Frozen project: {ROOT}")
print(f"Output folder:  {OUTDIR}")
print("Scientific fits are loaded from saved outputs; no optimization is rerun.\n")

generators = [
    (3, lambda: make_figure3(ROOT, CERTIFIED, OUTDIR)),
    (4, lambda: make_figure4(ROOT, CERTIFIED, OUTDIR)),
    (5, lambda: make_figure5(ROOT, CERTIFIED, OUTDIR)),
    (6, lambda: make_figure6(ROOT, CERTIFIED, OUTDIR)),
    (7, lambda: make_figure7(ROOT, OUTDIR)),
    (8, lambda: make_figure8(ROOT, OUTDIR)),
    (9, lambda: make_figure9(ROOT, OUTDIR)),
]

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "frozen_project_root": str(ROOT),
    "output_folder": str(OUTDIR),
    "scientific_recomputation": False,
    "source_certified_atlas": str(CELL2 / "certified_source_atlas_states.csv"),
    "figures": {},
    "presentation_corrections": [
        "GENT abbreviation standardized to GE",
        "compatibility-set separator standardized to a middle dot",
        "Figure 3 nonzero frequencies below 0.1% shown as <0.1% rather than 0.0%",
        "redundant abbreviation footers removed",
        "Figure 4 compatibility-set legend moved to a dedicated readable strip",
        "Figure 4 panel F simplified to a horizontal point-and-error-bar comparison",
        "3% terminology standardized to declared tolerance",
        "zero and small bars explicitly shown and annotated",
        "NRMSE panel no longer uses the compatibility-tolerance line",
        "colorbars allocated with constrained layout",
        "internal version labels removed",
        "prospective wording replaced by held-out validation wording",
        "brain atlas scores shown on a log scale with exact annotations",
        "raster and PDF identifying metadata cleared",
    ],
}

for number, generator in generators:
    print(f"Generating Figure {number} ...")
    paths = generator()
    manifest["figures"][str(number)] = {
        "paths": paths,
        "sha256": {kind: sha256(Path(path)) for kind, path in paths.items()},
    }
    plt.close("all")
    print(f"  saved: {paths['png']}")

manifest_path = OUTDIR / "updated_figures_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, sort_keys=True)

print("\nDONE")
print(f"Manifest: {manifest_path}")
print("Generated PNG, PDF, and TIFF versions of Figures 3–9.")


Mounted at /content/drive
STANDALONE REGENERATION OF UPDATED MANUSCRIPT FIGURES 3–9
Frozen project: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
Output folder:  /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES
Scientific fits are loaded from saved outputs; no optimization is rerun.

Generating Figure 3 ...
  saved: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_3_Updated_Atlas_Geometry.png
Generating Figure 4 ...
  saved: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_4_Updated_Protocol_Visibility.png
Generating Figure 5 ...
  saved: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biologica

In [ ]:
"""Standalone Colab cell: generate manuscript Figures 1 and 2 from code.

Run this cell by itself. The figures are conceptual and deterministic; they do
not load or modify any research data. In Colab, Google Drive is mounted and the
outputs are written beside the other updated manuscript figures.

Optional overrides before running:
    os.environ["ATLAS_ROOT"] = "/content/drive/MyDrive/.../project_folder"
    os.environ["CONCEPT_FIGURE_OUTPUT_DIR"] = "/content/drive/MyDrive/.../folder"
    os.environ["CONCEPT_FIGURE_DPI"] = "600"
"""

from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import (
    FancyArrowPatch,
    FancyBboxPatch,
    PathPatch,
    Rectangle,
)
from matplotlib.path import Path as MplPath
import numpy as np
from PIL import Image


# -----------------------------------------------------------------------------
# Configuration and publication style
# -----------------------------------------------------------------------------

PROJECT_BASENAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)
OUTPUT_FOLDER = "UPDATED_MANUSCRIPT_FIGURES"
PNG_DPI = int(os.environ.get("CONCEPT_FIGURE_DPI", "600"))

BLUE = "#0072B2"
ORANGE = "#D55E00"
TEAL = "#009E73"
SKY = "#56B4E9"
BLACK = "#111111"
MID_GRAY = "#A8A8A8"
LIGHT_GRAY = "#D1D1D1"
PALE_GRAY = "#F6F6F6"

MODEL_COLORS = {
    "NH": BLUE,
    "MR": ORANGE,
    "Y2": TEAL,
    "GE": "#333333",
    "O1": ORANGE,
    "O2": TEAL,
    "GP2": "#005CCB",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "legend.fontsize": 12.5,
    "lines.linewidth": 2.2,
    "axes.linewidth": 1.0,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "svg.fonttype": "none",
})


# -----------------------------------------------------------------------------
# Colab/Drive output handling and metadata-free export
# -----------------------------------------------------------------------------

def mount_drive_if_needed() -> None:
    if Path("/content/drive/MyDrive").is_dir():
        return
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        return
    drive.mount("/content/drive", force_remount=False)


def resolve_output_dir() -> Path:
    explicit = os.environ.get("CONCEPT_FIGURE_OUTPUT_DIR", "").strip()
    if explicit:
        return Path(explicit).expanduser().resolve()

    atlas_root = os.environ.get("ATLAS_ROOT", "").strip() or os.environ.get(
        "V68_ROOT", ""
    ).strip()
    if atlas_root:
        return (Path(atlas_root).expanduser() / OUTPUT_FOLDER).resolve()

    mount_drive_if_needed()
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.is_dir():
        project = mydrive / "Optimal_Protocol" / PROJECT_BASENAME
        return (project / OUTPUT_FOLDER).resolve()

    return (Path.cwd() / OUTPUT_FOLDER).resolve()


def save_figure(fig: plt.Figure, outdir: Path, stem: str) -> dict[str, str]:
    """Save PNG/PDF/TIFF/SVG outputs without identifying metadata."""
    outdir.mkdir(parents=True, exist_ok=True)
    paths = {
        "png": outdir / f"{stem}.png",
        "pdf": outdir / f"{stem}.pdf",
        "tiff": outdir / f"{stem}.tiff",
        "svg": outdir / f"{stem}.svg",
    }

    fig.savefig(
        paths["png"], dpi=PNG_DPI, bbox_inches="tight", pad_inches=0.05,
        metadata={},
    )
    with Image.open(paths["png"]) as image:
        clean = image.convert("RGB")
    clean.save(
        paths["png"], format="PNG", dpi=(PNG_DPI, PNG_DPI), optimize=True,
    )
    clean.save(
        paths["tiff"], format="TIFF", dpi=(PNG_DPI, PNG_DPI),
        compression="tiff_lzw", tiffinfo={},
    )
    fig.savefig(
        paths["pdf"], bbox_inches="tight", pad_inches=0.05,
        metadata={
            "Title": "", "Author": "", "Subject": "", "Keywords": "",
            "Creator": "", "Producer": "", "CreationDate": None,
            "ModDate": None,
        },
    )
    fig.savefig(
        paths["svg"], bbox_inches="tight", pad_inches=0.05,
        metadata={"Title": "", "Description": ""},
    )
    plt.close(fig)
    return {kind: str(path) for kind, path in paths.items()}


# -----------------------------------------------------------------------------
# Reusable drawing primitives
# -----------------------------------------------------------------------------

def panel(fig: plt.Figure, bounds, label: str, title: str) -> plt.Axes:
    ax = fig.add_axes(bounds)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(Rectangle(
        (0, 0), 1, 1, transform=ax.transAxes, fill=False,
        edgecolor=LIGHT_GRAY, linewidth=1.1, clip_on=False,
    ))
    ax.text(0.035, 0.975, label, fontsize=27, fontweight="bold",
            ha="left", va="top")
    ax.text(0.54, 0.955, title, fontsize=16, fontweight="bold",
            ha="center", va="top")
    return ax


def arrow(
    ax: plt.Axes,
    start,
    end,
    *,
    color=BLACK,
    linewidth=1.6,
    mutation_scale=13,
    connectionstyle="arc3",
    zorder=5,
) -> FancyArrowPatch:
    patch = FancyArrowPatch(
        start, end, arrowstyle="-|>", color=color, linewidth=linewidth,
        mutation_scale=mutation_scale, connectionstyle=connectionstyle,
        shrinkA=0, shrinkB=0, zorder=zorder,
    )
    ax.add_patch(patch)
    return patch


def rounded_box(
    ax: plt.Axes,
    xy,
    width,
    height,
    *,
    edgecolor="#555555",
    facecolor="white",
    linewidth=1.2,
    radius=0.025,
    zorder=2,
) -> FancyBboxPatch:
    patch = FancyBboxPatch(
        xy, width, height,
        boxstyle=f"round,pad=0.012,rounding_size={radius}",
        linewidth=linewidth, edgecolor=edgecolor, facecolor=facecolor,
        zorder=zorder,
    )
    ax.add_patch(patch)
    return patch


def response_axes(
    ax: plt.Axes,
    x0: float,
    y0: float,
    x1: float,
    y1: float,
    *,
    xlabel: str | None = None,
    ylabel: str | None = None,
    label_size: float = 12,
) -> None:
    arrow(ax, (x0, y0), (x1, y0), mutation_scale=12, linewidth=1.3)
    arrow(ax, (x0, y0), (x0, y1), mutation_scale=12, linewidth=1.3)
    if xlabel:
        ax.text((x0 + x1) / 2, y0 - 0.045, xlabel, fontsize=label_size,
                ha="center", va="top")
    if ylabel:
        ax.text(x0 - 0.055, (y0 + y1) / 2, ylabel, fontsize=label_size,
                ha="center", va="center", rotation=90)


def saturating_curve(t: np.ndarray, start: float, gain: float, rate: float) -> np.ndarray:
    return start + gain * (1.0 - np.exp(-rate * t)) / (1.0 - np.exp(-rate))


def plot_family_band(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    color: str,
    *,
    band: float,
    alpha: float = 0.14,
    linewidth: float = 2.0,
    zorder: int = 2,
) -> None:
    ax.fill_between(x, y - band, y + band, color=color, alpha=alpha,
                    linewidth=0, zorder=zorder)
    ax.plot(x, y, color=color, linewidth=linewidth, zorder=zorder + 1)


def fingerprint_card(
    ax: plt.Axes,
    xy,
    width,
    height,
    *,
    color=BLACK,
    band=False,
    point=False,
    linewidth=1.25,
) -> None:
    x0, y0 = xy
    rounded_box(
        ax, (x0, y0), width, height, edgecolor="#555555",
        facecolor="white", linewidth=0.95, radius=0.018,
    )
    x = np.linspace(x0 + width * 0.08, x0 + width * 0.92, 80)
    t = (x - x.min()) / (x.max() - x.min())
    mid = y0 + height * (0.28 + 0.43 * t + 0.045 * np.sin(2.3 * np.pi * t))
    if band:
        ax.fill_between(
            x, mid - height * 0.15, mid + height * 0.15,
            color=color, alpha=0.12, linewidth=0,
        )
        ax.plot(x, mid - height * 0.16, color=color, linestyle="--",
                linewidth=0.85)
        ax.plot(x, mid + height * 0.16, color=color, linestyle="--",
                linewidth=0.85)
        ax.plot(x, mid, color=color, linewidth=linewidth + 0.7)
    else:
        for shift in (-0.12, 0.0, 0.12):
            curve = mid + height * shift * (0.35 + 0.65 * t)
            ax.plot(x, curve, color=color, linewidth=linewidth)
    if point:
        idx = int(0.47 * len(x))
        ax.scatter([x[idx]], [mid[idx]], s=38, color=BLACK, zorder=8)


def mini_outcome_card(
    ax: plt.Axes,
    xy,
    width,
    height,
    *,
    outcome: str,
    framed: bool = False,
) -> None:
    x0, y0 = xy
    if framed:
        rounded_box(
            ax, (x0, y0), width, height, edgecolor="#777777",
            linewidth=0.9, radius=0.02,
        )
    else:
        response_axes(ax, x0, y0, x0 + width, y0 + height,
                      xlabel=None, ylabel=None)

    x = np.linspace(x0 + width * 0.10, x0 + width * 0.90, 80)
    t = (x - x.min()) / (x.max() - x.min())
    y_a = y0 + height * (0.43 + 0.32 * t + 0.035 * np.sin(1.8 * np.pi * t))
    y_b = y0 + height * (0.27 + 0.28 * t)
    y_c = y0 + height * (0.10 + 0.17 * t)
    band = height * 0.065
    for y, color in ((y_a, BLUE), (y_b, ORANGE), (y_c, TEAL)):
        plot_family_band(ax, x, y, color, band=band, alpha=0.13,
                         linewidth=1.35)

    if outcome == "multiple":
        tx = x0 + width * 0.42
        ty = np.interp(tx, x, (y_a + y_b) / 2)
    elif outcome == "one":
        tx = x0 + width * 0.42
        ty = np.interp(tx, x, y_a)
    elif outcome == "none":
        tx = x0 + width * 0.60
        ty = y0 + height * 0.86
    else:
        raise ValueError(outcome)
    ax.scatter([tx], [ty], s=52, color=BLACK, zorder=10)


# -----------------------------------------------------------------------------
# Figure 1: ranking versus compatibility
# -----------------------------------------------------------------------------

def make_figure_1(outdir: Path) -> dict[str, str]:
    fig = plt.figure(figsize=(15.7, 10.0))
    ax_a = panel(fig, [0.015, 0.12, 0.235, 0.845], "A", "Traditional best fit")
    ax_b = panel(fig, [0.250, 0.12, 0.390, 0.845], "B", "Compatibility within tolerance")
    ax_c = panel(fig, [0.640, 0.12, 0.345, 0.845], "C", "Three possible outcomes")

    # Panel A: conventional ranking always returns one winner.
    x0, x1, y0, y1 = 0.13, 0.86, 0.49, 0.80
    response_axes(ax_a, x0, y0, x1, y1, xlabel="Loading", ylabel="Response")
    x = np.linspace(x0, x1 - 0.02, 180)
    t = (x - x0) / (x1 - x0)
    y_a = saturating_curve(t, y0 + 0.01, 0.285, 2.6)
    # All families share the plotted origin and then separate smoothly. This
    # avoids lines beginning below and crossing the loading axis.
    y_b = y_a - (0.010 + 0.025 * t)
    y_c = y_a - (0.010 + 0.050 * t)
    ax_a.plot(x, y_a, color=BLUE, linewidth=2.3)
    ax_a.plot(x, y_b, color=ORANGE, linewidth=2.3)
    ax_a.plot(x, y_c, color=TEAL, linewidth=2.3)

    obs_t = np.array([0.05, 0.10, 0.15, 0.20, 0.27, 0.35, 0.44, 0.54, 0.65, 0.77, 0.90])
    obs_x = x0 + obs_t * (x1 - x0)
    obs_y = np.interp(obs_x, x, y_a) + np.array(
        [0.000, 0.006, 0.010, 0.008, 0.006, 0.002, 0.004, 0.001, -0.001, 0.002, 0.006]
    )
    ax_a.scatter(obs_x, obs_y, s=48, color=BLACK, zorder=10)

    legend_y = [0.655, 0.615, 0.575]
    for yy, color, label in zip(legend_y, (BLUE, ORANGE, TEAL), ("Family A", "Family B", "Family C")):
        ax_a.plot([0.60, 0.67], [yy, yy], color=color, linewidth=2.5)
        ax_a.text(0.70, yy, label, fontsize=12.5, ha="left", va="center")
    ax_a.scatter([0.635], [0.535], s=48, color=BLACK)
    ax_a.text(0.70, 0.535, "Observed data", fontsize=12.5, ha="left", va="center")

    ax_a.text(0.13, 0.395, "Fitting error (lower is better)", fontsize=13,
              ha="left", va="center")
    ax_a.plot([0.12, 0.86], [0.375, 0.375], color="#444444", linewidth=1.0)
    for yy, family, value, color in (
        (0.340, "A", "0.82", BLUE),
        (0.300, "B", "1.35", ORANGE),
        (0.260, "C", "2.10", TEAL),
    ):
        ax_a.text(0.27, yy, family, color=color, fontweight="bold",
                  fontsize=13, ha="center", va="center")
        ax_a.text(0.70, yy, value, fontsize=13, ha="center", va="center")
    rounded_box(
        ax_a, (0.115, 0.170), 0.75, 0.055,
        edgecolor="#236BFE", linewidth=1.4, radius=0.012,
    )
    ax_a.text(0.18, 0.198, r"1$^{\mathrm{st}}$", color="#236BFE",
              fontweight="bold", fontsize=13, ha="center", va="center")
    ax_a.text(0.31, 0.198, "A", color="#236BFE", fontweight="bold",
              fontsize=13, ha="center", va="center")
    ax_a.text(0.70, 0.198, "0.82", color="#236BFE", fontweight="bold",
              fontsize=13, ha="center", va="center")
    ax_a.text(0.50, 0.075, "Always picks one winner", fontsize=14,
              fontweight="bold", ha="center", va="center")

    # Panel B: tolerance neighborhoods return a set of compatible families.
    bx0, bx1, by0, by1 = 0.08, 0.66, 0.19, 0.79
    response_axes(
        ax_b, bx0, by0, bx1, by1,
        xlabel="Response coordinate 1", ylabel=None, label_size=12.5,
    )
    ax_b.text(0.115, 0.805, "Response\ncoordinate 2", fontsize=12.5,
              ha="left", va="bottom")
    bx = np.linspace(bx0 + 0.025, bx1 - 0.02, 220)
    bt = (bx - bx.min()) / (bx.max() - bx.min())
    ba = saturating_curve(bt, 0.34, 0.36, 1.55)
    bb = saturating_curve(bt, 0.29, 0.25, 1.75)
    bc = saturating_curve(bt, 0.232, 0.15, 1.55)
    plot_family_band(ax_b, bx, ba, BLUE, band=0.052, alpha=0.15, linewidth=2.2)
    plot_family_band(ax_b, bx, bb, ORANGE, band=0.052, alpha=0.15, linewidth=2.2)
    plot_family_band(ax_b, bx, bc, TEAL, band=0.035, alpha=0.14, linewidth=2.2)
    obs_x = 0.42
    obs_y = 0.5 * (np.interp(obs_x, bx, ba) + np.interp(obs_x, bx, bb))
    ax_b.scatter([obs_x], [obs_y], s=190, color=BLACK, zorder=12)

    ax_b.plot([0.71, 0.71], [0.20, 0.57], color=LIGHT_GRAY, linewidth=1.0)
    for yy, color, text_value in (
        (0.515, BLUE, "Family A\n± tolerance"),
        (0.425, ORANGE, "Family B\n± tolerance"),
        (0.335, TEAL, "Family C\n± tolerance"),
    ):
        ax_b.plot([0.73, 0.755], [yy, yy], color=color, linewidth=2.8)
        ax_b.text(0.78, yy, text_value, fontsize=12.2, ha="left", va="center")
    ax_b.scatter([0.742], [0.255], s=70, color=BLACK)
    ax_b.text(0.78, 0.255, "Observed data", fontsize=12.2,
              ha="left", va="center")
    ax_b.text(0.50, 0.075, "Compatible families: A, B", fontsize=13.5,
              ha="center", va="center")

    # Panel C: the three allowed set-valued outcomes.
    starts = [0.055, 0.375, 0.695]
    headings = ["Multiple\ncompatible", "One\ncompatible", "None\ncompatible"]
    outcomes = ["multiple", "one", "none"]
    footers = ["Compatible\nfamilies: A, B", "Compatible\nfamily: A", "Compatible\nfamilies: none"]
    for sx, heading, outcome, footer in zip(starts, headings, outcomes, footers):
        ax_c.text(sx + 0.12, 0.70, heading, fontsize=13, ha="center", va="center")
        mini_outcome_card(ax_c, (sx, 0.33), 0.24, 0.29, outcome=outcome, framed=False)
        ax_c.text(sx + 0.12, 0.275, footer, fontsize=12.2, ha="center", va="top")

    # Shared legend.
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", color=BLACK,
               markersize=8, label="Observed data"),
        Line2D([0], [0], color=BLUE, linewidth=2.7,
               label="Family A ± tolerance"),
        Line2D([0], [0], color=ORANGE, linewidth=2.7,
               label="Family B ± tolerance"),
        Line2D([0], [0], color=TEAL, linewidth=2.7,
               label="Family C ± tolerance"),
    ]
    fig.legend(
        handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.02),
        ncol=4, frameon=False, columnspacing=3.0, handlelength=2.3,
        fontsize=13,
    )
    return save_figure(fig, outdir, "Figure_1_Code_Generated_Compatibility_Concept")


# -----------------------------------------------------------------------------
# Figure 2: atlas construction, certification, and inference
# -----------------------------------------------------------------------------

def make_figure_2(outdir: Path) -> dict[str, str]:
    fig = plt.figure(figsize=(17.8, 10.0))
    ax_a = panel(fig, [0.010, 0.115, 0.220, 0.850], "A", "Deformation space")
    ax_b = panel(fig, [0.230, 0.115, 0.240, 0.850], "B", "Constitutive fingerprints")
    ax_c = panel(fig, [0.470, 0.115, 0.270, 0.850], "C", "Compatibility construction")
    ax_d = panel(fig, [0.740, 0.115, 0.250, 0.850], "D", "Certification and inference")

    # Panel A: bounded deformation domain with embedded protocol paths.
    blob_vertices = [
        (0.30, 0.77),
        (0.48, 0.87), (0.72, 0.83), (0.76, 0.76),
        (0.86, 0.63), (0.80, 0.54), (0.73, 0.49),
        (0.86, 0.37), (0.80, 0.25), (0.72, 0.20),
        (0.50, 0.17), (0.30, 0.19), (0.22, 0.22),
        (0.12, 0.36), (0.22, 0.43), (0.20, 0.50),
        (0.06, 0.62), (0.10, 0.71), (0.30, 0.77),
        (0.30, 0.77),
    ]
    blob_codes = (
        [MplPath.MOVETO]
        + [MplPath.CURVE4] * 18
        + [MplPath.CLOSEPOLY]
    )
    blob_path = MplPath(blob_vertices, blob_codes)
    ax_a.add_patch(PathPatch(blob_path, facecolor="white", edgecolor=BLACK,
                             linewidth=1.7, zorder=1))

    gx, gy = np.meshgrid(np.linspace(0.16, 0.82, 9), np.linspace(0.24, 0.78, 9))
    points = np.column_stack([gx.ravel(), gy.ravel()])
    inside = blob_path.contains_points(points)
    ax_a.scatter(points[inside, 0], points[inside, 1], s=23,
                 color="#BDBDBD", zorder=2)

    ut = np.array([[0.48, 0.72], [0.57, 0.67], [0.64, 0.59], [0.67, 0.52], [0.67, 0.45], [0.62, 0.38], [0.55, 0.33]])
    bt = np.array([[0.34, 0.64], [0.41, 0.59], [0.45, 0.52], [0.45, 0.45], [0.41, 0.38], [0.36, 0.31]])
    sh = np.array([[0.38, 0.26], [0.48, 0.30], [0.58, 0.34], [0.68, 0.37], [0.77, 0.38]])
    for path_values, color in ((ut, BLUE), (bt, ORANGE), (sh, TEAL)):
        ax_a.plot(path_values[:, 0], path_values[:, 1], color=color, linewidth=2.0,
                  marker="o", markersize=5.2, zorder=4)
    ax_a.text(0.69, 0.61, "UT", color=BLUE, fontsize=14, fontweight="bold")
    ax_a.text(0.27, 0.47, "BT", color=ORANGE, fontsize=14, fontweight="bold")
    ax_a.text(0.58, 0.25, "SH", color=TEAL, fontsize=14, fontweight="bold")
    ax_a.text(0.50, 0.095, "Complete-deformation\nparent domain\n2,601 states",
              fontsize=13, ha="center", va="center")

    # Transition arrow between the domain and family-manifold construction.
    overlay = fig.add_axes([0, 0, 1, 1])
    overlay.set_xlim(0, 1); overlay.set_ylim(0, 1); overlay.axis("off")
    arrow(overlay, (0.220, 0.545), (0.232, 0.545), mutation_scale=15, linewidth=1.8)

    # Panel B: family-specific adaptive sampling and fingerprints.
    ax_b.text(0.24, 0.86, "Constitutive\nfamilies", fontsize=13,
              ha="center", va="center")
    ax_b.text(0.56, 0.86, "Adaptive\nsampling", fontsize=13,
              ha="center", va="center")
    ax_b.text(0.84, 0.86, "Scale-shape\nresponse\nfingerprints", fontsize=13,
              ha="center", va="center")
    families = ["NH", "MR", "Y2", "GE", "O1", "O2", "GP2"]
    row_y = np.linspace(0.76, 0.25, len(families))
    rng = np.random.default_rng(120319)
    for index, (family, yy) in enumerate(zip(families, row_y)):
        color = MODEL_COLORS[family]
        rounded_box(ax_b, (0.07, yy - 0.032), 0.25, 0.064,
                    edgecolor="#555555", linewidth=0.9, radius=0.015)
        ax_b.text(0.195, yy, family, color=color, fontsize=14,
                  ha="center", va="center")
        arrow(ax_b, (0.36, yy), (0.45, yy), mutation_scale=11, linewidth=1.3)

        sample_x = 0.56 + rng.normal(0, 0.045, 8)
        sample_y = yy + rng.normal(0, 0.027, 8)
        ax_b.scatter(sample_x, sample_y, s=18, color="#BDBDBD", zorder=3)
        if index in (1, 4, 5):
            ax_b.scatter([sample_x[index % 8]], [sample_y[index % 8]],
                         s=28, color=color, zorder=5)
        arrow(ax_b, (0.67, yy), (0.75, yy), mutation_scale=11, linewidth=1.3)
        fingerprint_card(ax_b, (0.78, yy - 0.035), 0.18, 0.070,
                         color=color, linewidth=1.05)
    ax_b.text(0.56, 0.135, "⋮", fontsize=22, ha="center", va="center")
    ax_b.text(0.87, 0.135, "⋮", fontsize=22, ha="center", va="center")

    # Panel C: one source is compared with every family.
    ax_c.text(0.55, 0.86, "Compare against\nall families", fontsize=13,
              ha="center", va="center")
    ax_c.text(0.15, 0.75, "Source\nfingerprint", fontsize=12.5,
              ha="center", va="center")
    fingerprint_card(ax_c, (0.055, 0.54), 0.22, 0.16, color=BLACK,
                     band=True, linewidth=1.3)
    arrow(ax_c, (0.28, 0.62), (0.36, 0.62), mutation_scale=12, linewidth=1.4)

    results = {
        "NH": False, "MR": True, "Y2": True, "GE": False,
        "O1": False, "O2": True, "GP2": True,
    }
    cy = np.linspace(0.74, 0.28, len(families))
    for family, yy in zip(families, cy):
        ax_c.text(0.45, yy, family, color=MODEL_COLORS[family], fontsize=13,
                  ha="left", va="center")
        accepted = results[family]
        ax_c.text(0.63, yy, "✓" if accepted else "×",
                  color=TEAL if accepted else "#E31A1C", fontsize=18,
                  fontweight="bold", ha="center", va="center")
        ax_c.plot([0.72, 0.92], [yy, yy], color="#BBBBBB", linewidth=5.2,
                  solid_capstyle="butt")
        if accepted:
            ax_c.plot([0.82, 0.82], [yy - 0.018, yy + 0.018], color=BLACK,
                      linewidth=1.8)
    ax_c.plot([0.70, 0.70], [0.25, 0.78], color="#777777", linewidth=1.0)
    arrow(ax_c, (0.55, 0.24), (0.55, 0.17), mutation_scale=12, linewidth=1.4)
    rounded_box(ax_c, (0.12, 0.055), 0.75, 0.105,
                edgecolor="#333333", linewidth=1.2, radius=0.015)
    ax_c.text(0.495, 0.125, "Compatibility set", fontsize=13,
              fontweight="bold", ha="center", va="center")
    ax_c.text(0.495, 0.085, "{MR, Y2, O2, GP2}", fontsize=13.5,
              ha="center", va="center")

    # Transition arrow into certification/inference.
    arrow(overlay, (0.731, 0.545), (0.744, 0.545), mutation_scale=15, linewidth=1.8)

    # Panel D: offline certification above, online inference below.
    ax_d.text(0.50, 0.86, "Certification (offline)", fontsize=13,
              ha="center", va="center")
    rounded_box(ax_d, (0.04, 0.70), 0.20, 0.12,
                edgecolor="#236BFE", linewidth=1.2, radius=0.02)
    ax_d.text(0.14, 0.76, "Frozen\natlas", color="#005CCB", fontsize=13,
              fontweight="bold", ha="center", va="center")
    arrow(ax_d, (0.25, 0.76), (0.33, 0.76), mutation_scale=11, linewidth=1.3)
    rounded_box(ax_d, (0.35, 0.66), 0.32, 0.18,
                edgecolor="#555555", linewidth=1.0, radius=0.02)
    checks = ["Boundary", "Held-out", "Resolution", "Off-manifold"]
    for yy, text_value in zip(np.linspace(0.80, 0.69, 4), checks):
        ax_d.text(0.39, yy, "✓", color=TEAL, fontsize=13,
                  fontweight="bold", ha="center", va="center",
                  bbox=dict(boxstyle="circle,pad=0.12", facecolor="white",
                            edgecolor="#666666", linewidth=0.6))
        ax_d.text(0.44, yy, text_value, fontsize=11.5, ha="left", va="center")
    arrow(ax_d, (0.68, 0.76), (0.75, 0.76), mutation_scale=11, linewidth=1.3)
    rounded_box(ax_d, (0.77, 0.70), 0.19, 0.12,
                edgecolor="#236BFE", linewidth=1.2, radius=0.02)
    ax_d.text(0.865, 0.76, "Certified\natlas", color="#005CCB", fontsize=13,
              fontweight="bold", ha="center", va="center")

    ax_d.plot([0.04, 0.96], [0.61, 0.61], color=LIGHT_GRAY, linewidth=1.0)
    ax_d.text(0.50, 0.565, "Inference (online)", fontsize=13,
              ha="center", va="center")
    ax_d.text(0.16, 0.515, "Unknown\nresponse", fontsize=12.5,
              ha="center", va="center")
    fingerprint_card(ax_d, (0.05, 0.34), 0.22, 0.13, color=BLACK,
                     band=True, point=True, linewidth=1.15)
    ax_d.text(0.62, 0.49, "Query certified atlas", fontsize=11.5,
              ha="center", va="center")
    ax_d.plot([0.61, 0.61], [0.46, 0.31], color=BLACK, linewidth=1.2)
    ax_d.plot([0.36, 0.86], [0.31, 0.31], color=BLACK, linewidth=1.2)
    branch_x = [0.36, 0.61, 0.86]
    branch_titles = ["Multiple", "One", "None"]
    branch_outcomes = ["multiple", "one", "none"]
    for xx, heading, outcome in zip(branch_x, branch_titles, branch_outcomes):
        arrow(ax_d, (xx, 0.31), (xx, 0.255), mutation_scale=10, linewidth=1.1)
        ax_d.text(xx, 0.225, heading, fontsize=11.5, ha="center", va="bottom")
        mini_outcome_card(
            ax_d, (xx - 0.105, 0.045), 0.21, 0.16,
            outcome=outcome, framed=True,
        )

    # Shared visual vocabulary across the bottom.
    legend_ax = fig.add_axes([0.02, 0.015, 0.96, 0.075])
    legend_ax.set_xlim(0, 1); legend_ax.set_ylim(0, 1); legend_ax.axis("off")
    legend_ax.scatter([0.03], [0.52], s=70, color=BLACK)
    legend_ax.text(0.045, 0.52, "Observed data\n(source or query)", fontsize=11.5,
                   ha="left", va="center")
    legend_ax.scatter([0.18], [0.57], s=28, color="#BDBDBD")
    legend_ax.text(0.195, 0.57, "Sampled states", fontsize=11.5,
                   ha="left", va="center")
    for yy, color, label in ((0.78, BLUE, "UT protocol path"),
                             (0.49, ORANGE, "BT protocol path"),
                             (0.20, TEAL, "SH protocol path")):
        legend_ax.plot([0.31, 0.34], [yy, yy], color=color, linewidth=2.5)
        legend_ax.text(0.35, yy, label, fontsize=11.5, ha="left", va="center")
    rounded_box(legend_ax, (0.48, 0.31), 0.035, 0.42,
                edgecolor="#555555", linewidth=0.9, radius=0.01)
    legend_ax.text(0.535, 0.52, "Constitutive family", fontsize=11.5,
                   ha="left", va="center")
    fingerprint_card(legend_ax, (0.64, 0.23), 0.035, 0.55,
                     color=BLACK, band=True, linewidth=0.8)
    legend_ax.text(0.695, 0.52, "Scale-shape fingerprint\n(tolerance band)",
                   fontsize=11.0, ha="left", va="center")
    legend_ax.plot([0.84, 0.89], [0.68, 0.68], color="#BBBBBB", linewidth=5.0)
    legend_ax.plot([0.865, 0.865], [0.55, 0.81], color=BLACK, linewidth=1.5)
    legend_ax.text(0.90, 0.68, "Within tolerance", fontsize=10.8,
                   ha="left", va="center")
    legend_ax.plot([0.84, 0.89], [0.30, 0.30], color="#BBBBBB", linewidth=5.0)
    legend_ax.text(0.90, 0.30, "Outside tolerance", fontsize=10.8,
                   ha="left", va="center")

    return save_figure(fig, outdir, "Figure_2_Code_Generated_Atlas_Workflow")


# -----------------------------------------------------------------------------
# Execute as one standalone cell
# -----------------------------------------------------------------------------

OUTDIR = resolve_output_dir()
print("=" * 88)
print("STANDALONE CODE GENERATION OF CONCEPTUAL MANUSCRIPT FIGURES 1 AND 2")
print("=" * 88)
print(f"Output folder: {OUTDIR}")
print("No research data are loaded; both figures are deterministic diagrams.")

outputs = {
    "Figure 1": make_figure_1(OUTDIR),
    "Figure 2": make_figure_2(OUTDIR),
}

for figure_name, formats in outputs.items():
    print(f"\n{figure_name}")
    for format_name, path in formats.items():
        print(f"  {format_name.upper():5s}: {path}")

print("\nDONE")


STANDALONE CODE GENERATION OF CONCEPTUAL MANUSCRIPT FIGURES 1 AND 2
Output folder: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES
No research data are loaded; both figures are deterministic diagrams.

Figure 1
  PNG  : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_1_Code_Generated_Compatibility_Concept.png
  PDF  : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_1_Code_Generated_Compatibility_Concept.pdf
  TIFF : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_1_Code_Generated_Compatibility_Concept.tiff
  SVG  : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_

In [2]:
"""Standalone Colab cell: generate one-page combined manuscript Figure 5.

Run this cell by itself at the end of the atlas notebook (or in a fresh Colab
runtime). It mounts Google Drive if needed, discovers the completed V68 atlas
project, loads only previously saved CSV/JSON results, and creates a compact
5-row x 2-column summary of the three controlled tests and two held-out
validations. No fitting, optimization, atlas construction, or certification is
rerun. Displayed percentages use reader-level precision: three decimal places
below 1%, two from 1% to 10%, and one above 10%.

Optional override before execution:
    os.environ["ATLAS_ROOT"] = "/content/drive/MyDrive/.../project_folder"

Optional raster resolution override:
    os.environ["COMBINED_FIGURE_DPI"] = "600"
"""

from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from PIL import Image


# =============================================================================
# Configuration and publication style
# =============================================================================

PROJECT_BASENAME = (
    "V68_practical_standard_protocols_complete_F_biological_compatibility_atlas"
)
OUTPUT_FOLDER = "UPDATED_MANUSCRIPT_FIGURES"
OUTPUT_STEM = "Figure_5_Combined_Atlas_Outcomes"
PNG_DPI = int(os.environ.get("COMBINED_FIGURE_DPI", "600"))

MODELS = ["NH", "MR", "YEOH2", "GENT", "OGDEN1", "OGDEN2", "GP2"]
MODEL_SHORT = {
    "NH": "NH",
    "MR": "MR",
    "YEOH2": "Y2",
    "GENT": "GE",
    "OGDEN1": "O1",
    "OGDEN2": "O2",
    "GP2": "GP2",
}

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
GOLD = "#E69F00"
LIGHT_BLUE = "#56B4E9"
GRAY = "#777777"
LIGHT_GRAY = "#D5D5D5"
BLACK = "#111111"

OGDEN_ALPHA_SERIES_THRESHOLD = 1.0e-6
GENT_SINGULARITY_MARGIN = 1.0e-8

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 6.5,
        "axes.titlesize": 7.0,
        "axes.labelsize": 6.2,
        "xtick.labelsize": 5.5,
        "ytick.labelsize": 5.5,
        "legend.fontsize": 5.3,
        "axes.linewidth": 0.60,
        "lines.linewidth": 1.25,
        "savefig.facecolor": "white",
        "figure.facecolor": "white",
    }
)


def format_percent(value: float) -> str:
    """Format percentages consistently with the main manuscript."""
    value = float(value)
    magnitude = abs(value)
    decimals = 3 if magnitude < 1.0 else (2 if magnitude <= 10.0 else 1)
    return f"{value:.{decimals}f}"


# =============================================================================
# Drive discovery and file handling
# =============================================================================

def _atlas_anchor(root: Path) -> Path:
    return root / "cell2_publication_validation" / "certified_source_atlas_states.csv"


def mount_drive_if_needed() -> None:
    if Path("/content/drive/MyDrive").is_dir():
        return
    try:
        from google.colab import drive  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "Google Drive is not mounted and this is not a Colab runtime. "
            "Mount Drive or set ATLAS_ROOT to a locally visible project folder."
        ) from exc
    drive.mount("/content/drive", force_remount=False)


def resolve_root() -> Path:
    explicit = (
        os.environ.get("ATLAS_ROOT", "").strip()
        or os.environ.get("V68_ROOT", "").strip()
    )
    candidates: list[Path] = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    for mydrive in (
        Path("/content/drive/MyDrive"),
        Path("/content/v68_drive/MyDrive"),
        Path("/content/v68_figures_drive/MyDrive"),
    ):
        candidates.append(mydrive / "Optimal_Protocol" / PROJECT_BASENAME)

    for root in candidates:
        if _atlas_anchor(root).is_file():
            return root.resolve()

    mount_drive_if_needed()
    for root in candidates:
        if _atlas_anchor(root).is_file():
            return root.resolve()

    search_base = Path("/content/drive/MyDrive/Optimal_Protocol")
    if search_base.is_dir():
        matches = sorted(
            p.parent.parent
            for p in search_base.glob(
                "*/cell2_publication_validation/certified_source_atlas_states.csv"
            )
        )
        matches = [p for p in matches if _atlas_anchor(p).is_file()]
        if len(matches) == 1:
            return matches[0].resolve()
        if len(matches) > 1:
            listing = "\n".join(f"  - {p}" for p in matches)
            raise RuntimeError(
                "Multiple completed atlas projects were found. Set ATLAS_ROOT "
                "to the intended project:\n" + listing
            )

    raise FileNotFoundError(
        "Could not locate the completed atlas project. Set ATLAS_ROOT to the "
        "Drive folder containing cell2_publication_validation/."
    )


def require(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"Required saved result is missing:\n{path}")
    return path


def read_json(path: Path) -> Mapping:
    with require(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def find_result_dir(
    root: Path, patterns: Sequence[str], required_name: str
) -> Path:
    for pattern in patterns:
        for candidate in sorted(root.glob(pattern)):
            if candidate.is_dir() and (candidate / required_name).is_file():
                return candidate
    raise FileNotFoundError(
        f"Could not find a saved result directory containing {required_name} "
        f"under {root}. Run its original scientific generator once."
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_figure(fig: plt.Figure, outdir: Path, stem: str) -> dict[str, str]:
    """Save journal formats while clearing identifying image metadata."""
    outdir.mkdir(parents=True, exist_ok=True)
    paths = {
        "png": outdir / f"{stem}.png",
        "tiff": outdir / f"{stem}.tiff",
        "pdf": outdir / f"{stem}.pdf",
        "svg": outdir / f"{stem}.svg",
    }

    fig.savefig(paths["png"], dpi=PNG_DPI, bbox_inches="tight", metadata={})
    with Image.open(paths["png"]) as image:
        clean = image.copy()
    clean.save(
        paths["png"], format="PNG", dpi=(PNG_DPI, PNG_DPI), optimize=True
    )
    clean.save(
        paths["tiff"],
        format="TIFF",
        dpi=(PNG_DPI, PNG_DPI),
        compression="tiff_lzw",
        tiffinfo={},
    )
    fig.savefig(
        paths["pdf"],
        bbox_inches="tight",
        metadata={
            "Title": "",
            "Author": "",
            "Subject": "",
            "Keywords": "",
            "Creator": "",
            "Producer": "",
            "CreationDate": None,
            "ModDate": None,
        },
    )
    fig.savefig(
        paths["svg"],
        bbox_inches="tight",
        metadata={"Title": "", "Description": "", "Creator": "", "Date": None},
    )
    return {kind: str(path) for kind, path in paths.items()}


# =============================================================================
# Compact plotting helpers
# =============================================================================

def clean_axis(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.2, width=0.55, pad=1.6)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune=None))
    ax.margins(x=0.025)


def row_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        -0.16,
        1.10,
        label,
        transform=ax.transAxes,
        fontsize=10.0,
        fontweight="bold",
        ha="left",
        va="top",
        clip_on=False,
    )


def badge(
    ax: plt.Axes,
    text: str,
    *,
    color: str = BLACK,
    x: float = 0.985,
    y: float = 0.96,
    ha: str = "right",
    va: str = "top",
) -> None:
    ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        ha=ha,
        va=va,
        fontsize=5.6,
        fontweight="bold",
        color=color,
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "edgecolor": color,
            "linewidth": 0.65,
            "alpha": 0.92,
        },
        zorder=20,
    )


def legend(ax: plt.Axes, **kwargs) -> None:
    defaults = {
        "frameon": True,
        "fancybox": False,
        "framealpha": 0.90,
        "borderpad": 0.25,
        "labelspacing": 0.22,
        "handlelength": 1.6,
        "handletextpad": 0.35,
    }
    defaults.update(kwargs)
    ax.legend(**defaults)


def annotate_bar_values(
    ax: plt.Axes,
    bars,
    values: Sequence[float],
    *,
    fmt: str = ".1f",
    fontsize: float = 5.1,
) -> None:
    values = np.asarray(values, dtype=float)
    scale = max(float(np.nanmax(np.abs(values))) if len(values) else 0.0, 1.0)
    for bar, value in zip(bars, values):
        if not np.isfinite(value):
            continue
        y = float(value) + 0.025 * scale
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y,
            format(float(value), fmt),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            clip_on=False,
        )
        if abs(value) < 1.0e-12:
            ax.scatter(
                [bar.get_x() + bar.get_width() / 2],
                [0.0],
                marker="D",
                s=9,
                color=bar.get_facecolor(),
                zorder=6,
            )


def as_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes"}


def compact_signature(signature: str) -> str:
    return "·".join(
        MODEL_SHORT.get(item, item) for item in str(signature).split("|") if item
    )


def source_row(certified: pd.DataFrame, model: str, index: int) -> pd.Series:
    hit = certified[
        (certified["source_model"].astype(str) == str(model))
        & (certified["source_index"].astype(int) == int(index))
    ]
    if len(hit) != 1:
        raise RuntimeError(
            f"Expected one frozen source row for {model} #{index}; found {len(hit)}."
        )
    return hit.iloc[0]


def json_array(value) -> np.ndarray:
    return np.asarray(json.loads(str(value)), dtype=float).reshape(-1)


# =============================================================================
# Constitutive-response helpers copied from the original generators
# =============================================================================

def ogden_shifted_term(alpha: float, log_lam: np.ndarray) -> np.ndarray:
    alpha = float(alpha)
    values = np.asarray(log_lam, dtype=float)
    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            values
            + 0.5 * alpha * values**2
            + alpha**2 * values**3 / 6.0
            + alpha**3 * values**4 / 24.0
            + alpha**4 * values**5 / 120.0
        )
    return 4.0 * np.expm1(alpha * values) / alpha


def invariants(lam: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    lam2 = np.asarray(lam, dtype=float) ** 2
    return np.sum(lam2, axis=-1), (
        lam2[..., 0] * lam2[..., 1]
        + lam2[..., 1] * lam2[..., 2]
        + lam2[..., 2] * lam2[..., 0]
    )


def principal_tau_shape(
    model: str, physical: Mapping, lam: np.ndarray
) -> np.ndarray:
    lam = np.asarray(lam, dtype=float)
    i1, i2 = invariants(lam)
    x, y = i1 - 3.0, i2 - 3.0
    if model == "NH":
        return 2.0 * lam**2
    if model == "MR":
        rho = float(physical["rho"])
        return 2.0 * ((1.0 - rho) * lam**2 - rho * lam**-2.0)
    if model == "YEOH2":
        beta = float(physical["beta"])
        return 2.0 * (1.0 + 2.0 * beta * x)[..., None] * lam**2
    if model == "GENT":
        gamma = float(physical["gamma"])
        denominator = 1.0 - gamma * x
        output = np.full_like(lam, np.nan)
        valid = denominator > GENT_SINGULARITY_MARGIN
        output[valid] = 2.0 * lam[valid] ** 2 / denominator[valid, None]
        return output
    if model == "OGDEN1":
        return ogden_shifted_term(float(physical["alpha"]), np.log(lam))
    if model == "OGDEN2":
        a1 = float(physical["alpha1"])
        a2 = float(physical["alpha2"])
        weight = float(physical["weight1"])
        return weight * ogden_shifted_term(
            a1, np.log(lam)
        ) + (1.0 - weight) * ogden_shifted_term(a2, np.log(lam))
    if model == "GP2":
        rho = float(physical["rho"])
        b20, b11, b02 = (
            float(physical[key]) for key in ("beta20", "beta11", "beta02")
        )
        w1 = (1.0 - rho) + 2.0 * b20 * x + b11 * y
        w2 = rho + b11 * x + 2.0 * b02 * y
        return 2.0 * (
            w1[..., None] * lam**2 - w2[..., None] * lam**-2.0
        )
    raise KeyError(model)


def source_physical(row: pd.Series) -> dict:
    model = str(row["source_model"])
    coordinate = json_array(row["coordinate_json"])
    if model == "NH":
        return {}
    if model == "MR":
        return {"rho": float(row.get("physical_rho", coordinate[0]))}
    if model == "YEOH2":
        return {"beta": float(row.get("physical_beta", coordinate[0]))}
    if model == "GENT":
        return {"gamma": float(row.get("physical_gamma", coordinate[0]))}
    if model == "OGDEN1":
        return {"alpha": float(row.get("physical_alpha", coordinate[0]))}
    if model == "OGDEN2":
        return {
            "alpha1": float(row["physical_alpha1"]),
            "alpha2": float(row["physical_alpha2"]),
            "weight1": float(row["physical_weight1"]),
        }
    if model == "GP2":
        return {
            key: float(row[f"physical_{key}"])
            for key in ("rho", "beta20", "beta11", "beta02")
        }
    raise KeyError(model)


def simple_shear_curve(
    model: str, physical: Mapping, scale: float, gamma: np.ndarray
) -> np.ndarray:
    output = np.zeros_like(np.asarray(gamma, dtype=float))
    for i, value in enumerate(np.asarray(gamma, dtype=float)):
        deformation = np.array(
            [[1.0, value, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
        )
        left_cauchy = deformation @ deformation.T
        eigenvalues, eigenvectors = np.linalg.eigh(left_cauchy)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]
        lam = np.sqrt(np.maximum(eigenvalues, 0.0))[None, :]
        q = principal_tau_shape(model, physical, lam)[0]
        tau = eigenvectors @ np.diag(float(scale) * q) @ eigenvectors.T
        output[i] = tau[0, 1]
    return output


def standard_curves(
    model: str, physical: Mapping, scale: float
) -> dict[str, np.ndarray]:
    ut = np.linspace(0.70, 1.50, 81)
    bt = np.linspace(0.80, 1.25, 81)
    sh = np.linspace(-0.50, 0.50, 81)
    ut_lam = np.stack([ut, ut**-0.5, ut**-0.5], axis=-1)
    bt_lam = np.stack([bt, bt, bt**-2.0], axis=-1)
    ut_tau = principal_tau_shape(model, physical, ut_lam)
    bt_tau = principal_tau_shape(model, physical, bt_lam)
    return {
        "UT": float(scale) * (ut_tau[:, 0] - ut_tau[:, 2]),
        "BT": float(scale) * (bt_tau[:, 0] - bt_tau[:, 2]),
        "SH": simple_shear_curve(model, physical, scale, sh),
    }


def plot_protocol_triplet(
    ax: plt.Axes,
    truth: Mapping,
    prediction: Mapping,
    truth_label: str,
    prediction_label: str,
) -> None:
    axes = {
        "UT": np.linspace(0.0, 1.0, len(truth["UT"])),
        "BT": np.linspace(1.20, 2.20, len(truth["BT"])),
        "SH": np.linspace(2.40, 3.40, len(truth["SH"])),
    }
    for mode in ("UT", "BT", "SH"):
        ax.plot(
            axes[mode],
            truth[mode],
            color=BLUE,
            linewidth=1.45,
            label=truth_label if mode == "UT" else None,
        )
        ax.plot(
            axes[mode],
            prediction[mode],
            color=ORANGE,
            linestyle="--",
            linewidth=1.25,
            label=prediction_label if mode == "UT" else None,
        )
    ax.axvline(1.10, linewidth=0.55, alpha=0.35, color=GRAY)
    ax.axvline(2.30, linewidth=0.55, alpha=0.35, color=GRAY)
    ax.set_xticks([0.5, 1.70, 2.90], ["UT", "BT", "SH"])
    ax.set_ylabel("Stress response (kPa)")


def withheld_curves(b: float, scale: float) -> dict[str, np.ndarray]:
    ut = np.linspace(0.70, 1.50, 81)
    bt = np.linspace(0.80, 1.25, 81)
    sh = np.linspace(-0.50, 0.50, 81)
    ut_lam = np.stack([ut, ut**-0.5, ut**-0.5], axis=-1)
    bt_lam = np.stack([bt, bt, bt**-2.0], axis=-1)

    def q(lam: np.ndarray) -> np.ndarray:
        i1, _ = invariants(lam)
        return 2.0 * np.exp(float(b) * (i1 - 3.0))[..., None] * lam**2

    output = {
        "UT": float(scale)
        * np.diff(q(ut_lam)[:, [2, 0]], axis=1).reshape(-1),
        "BT": float(scale)
        * np.diff(q(bt_lam)[:, [2, 0]], axis=1).reshape(-1),
    }
    shear = np.zeros_like(sh)
    for i, value in enumerate(sh):
        deformation = np.array(
            [[1.0, value, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
        )
        left_cauchy = deformation @ deformation.T
        eigenvalues, eigenvectors = np.linalg.eigh(left_cauchy)
        order = np.argsort(eigenvalues)[::-1]
        lam = np.sqrt(np.maximum(eigenvalues[order], 0.0))[None, :]
        vectors = eigenvectors[:, order]
        tau = vectors @ np.diag(float(scale) * q(lam)[0]) @ vectors.T
        shear[i] = tau[0, 1]
    output["SH"] = shear
    return output


def ogden_difference(
    alpha: float, log_a: np.ndarray, log_b: np.ndarray
) -> np.ndarray:
    alpha = float(alpha)
    a = np.asarray(log_a, dtype=float)
    b = np.asarray(log_b, dtype=float)
    if abs(alpha) <= OGDEN_ALPHA_SERIES_THRESHOLD:
        return 4.0 * (
            (a - b)
            + 0.5 * alpha * (a**2 - b**2)
            + alpha**2 * (a**3 - b**3) / 6.0
            + alpha**3 * (a**4 - b**4) / 24.0
            + alpha**4 * (a**5 - b**5) / 120.0
        )
    return 4.0 * (np.exp(alpha * a) - np.exp(alpha * b)) / alpha


def diagonal_first_template(
    model: str, coordinate: np.ndarray, states: np.ndarray
) -> np.ndarray:
    lam = np.asarray(states, dtype=float)
    l1, l2, l3 = lam[:, 0], lam[:, 1], lam[:, 2]
    b1 = 2.0 * (l1**2 - l3**2)
    b2 = 2.0 * (l3**-2.0 - l1**-2.0)
    i1m3 = l1**2 + l2**2 + l3**2 - 3.0
    coordinate = np.asarray(coordinate, dtype=float).reshape(-1)
    if model == "NH":
        return b1
    if model == "MR":
        return (1.0 - coordinate[0]) * b1 + coordinate[0] * b2
    if model == "YEOH2":
        return b1 * (1.0 + 2.0 * coordinate[0] * i1m3)
    if model == "GENT":
        return b1 / (1.0 - coordinate[0] * i1m3)
    if model == "OGDEN1":
        return ogden_difference(coordinate[0], np.log(l1), np.log(l3))
    raise KeyError(f"External-validation renderer does not require {model}.")


def brain_template(
    model: str, coordinate: np.ndarray, *, lam=None, gamma=None
) -> np.ndarray:
    coordinate = np.asarray(coordinate, dtype=float).reshape(-1)
    if (lam is None) == (gamma is None):
        raise ValueError("Provide exactly one of lam or gamma.")
    if lam is not None:
        x = np.asarray(lam, dtype=float)
        b1 = 2.0 * (x**2 - x**-1.0)
        b2 = 2.0 * (x - x**-2.0)
        i1m3 = x**2 + 2.0 / x - 3.0

        def one_term_ogden(alpha):
            return ogden_difference(alpha, np.log(x), -0.5 * np.log(x))

    else:
        g = np.asarray(gamma, dtype=float)
        b1 = b2 = 2.0 * g
        i1m3 = g**2

        def one_term_ogden(alpha):
            root = np.sqrt(1.0 + 0.25 * g**2)
            l1, l2 = root + 0.5 * g, root - 0.5 * g
            return ogden_difference(
                alpha, np.log(l1), np.log(l2)
            ) / np.sqrt(g**2 + 4.0)

    if model == "NH":
        return b1
    if model == "MR":
        return (1.0 - coordinate[0]) * b1 + coordinate[0] * b2
    if model == "YEOH2":
        return b1 * (1.0 + 2.0 * coordinate[0] * i1m3)
    if model == "GENT":
        return b1 / (1.0 - coordinate[0] * i1m3)
    if model == "OGDEN1":
        return one_term_ogden(coordinate[0])
    raise KeyError(f"External-validation renderer does not require {model}.")


# =============================================================================
# One-page combined Figure 5
# =============================================================================

def make_combined_figure5(
    root: Path, certified: pd.DataFrame, outdir: Path
) -> tuple[dict[str, str], dict]:
    # ------------------------------------------------------------------ Row A
    example1_dir = find_result_dir(
        root,
        ["cell3_example1_certified_set_outlier*", "*example1*"],
        "example1_outlier_predictions.csv",
    )
    example1 = read_json(example1_dir / "example1_selected_source.json")
    example1_curves = pd.read_csv(
        require(example1_dir / "example1_outlier_predictions.csv")
    )
    example1_witnesses = pd.read_csv(
        require(example1_dir / "example1_certified_witnesses.csv")
    )

    # ------------------------------------------------------------------ Row B
    example2_dir = find_result_dir(
        root,
        ["cell3_example2_complexity_required*", "*example2*"],
        "example2_complete_F_error_map.csv",
    )
    example2 = read_json(example2_dir / "example2_selected_source.json")
    example2_fits = pd.read_csv(
        require(example2_dir / "example2_standard_protocol_fit_summary.csv")
    )
    example2_fstar = read_json(example2_dir / "example2_Fstar.json")
    example2_row = source_row(
        certified, str(example2["source_model"]), int(example2["source_index"])
    )

    # ------------------------------------------------------------------ Row C
    example3_dir = find_result_dir(
        root,
        ["cell3_example3_none_compatible*", "*example3*"],
        "example3_standard_fit_summary.csv",
    )
    example3_standard = pd.read_csv(
        require(example3_dir / "example3_standard_fit_summary.csv")
    )
    example3_parent = pd.read_csv(
        require(example3_dir / "example3_completeF_continuous_parent.csv")
    )
    example3_validation = pd.read_csv(
        require(example3_dir / "example3_completeF_continuous_validation.csv")
    )
    example3_summary = read_json(example3_dir / "example3_summary.json")

    # ------------------------------------------------------------------ Row D
    treloar_dir = find_result_dir(
        root,
        ["TRELOAR_VALIDATION_OUTPUT", "*TRELOAR*VALIDATION*"],
        "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json",
    )
    treloar_result = read_json(
        treloar_dir / "FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json"
    )
    treloar_fits = pd.read_csv(
        require(treloar_dir / "Treloar_uniaxial_fit_results.csv")
    )
    treloar_train = pd.read_csv(require(treloar_dir / "v68_domain_uniaxial.csv"))
    treloar_planar = pd.read_csv(
        require(treloar_dir / "v68_domain_pure_shear_planar.csv")
    )
    treloar_biaxial = pd.read_csv(
        require(treloar_dir / "v68_domain_equibiaxial.csv")
    )

    # ------------------------------------------------------------------ Row E
    brain_base = find_result_dir(
        root,
        ["SOFT_TISSUE_BRAIN_VALIDATION_OUTPUT", "*BRAIN*VALIDATION*"],
        "ALL_BRAIN_REGIONS_summary.csv",
    )
    brain_candidates = []
    for result_path in brain_base.glob(
        "*/FROZEN_ATLAS_PRIMARY_VALIDATION_RESULT.json"
    ):
        result = read_json(result_path)
        if bool(result.get("external_validation_eligible", False)):
            brain_candidates.append((result_path.parent, result))
    if len(brain_candidates) != 1:
        raise RuntimeError(
            f"Expected one eligible brain region; found {len(brain_candidates)} "
            f"under {brain_base}."
        )
    brain_dir, brain_result = brain_candidates[0]
    brain_fits = pd.read_csv(
        require(brain_dir / "compression_only_model_fits.csv")
    )
    brain_compression = pd.read_csv(
        require(brain_dir / "TRAIN_compression_only.csv")
    )
    brain_tension = pd.read_csv(require(brain_dir / "HOLDOUT_tension.csv"))

    # A true one-page figure at manuscript width: 6.8--7.0 inches wide and
    # approximately 7.8 inches high after bbox trimming.
    fig, axes = plt.subplots(
        5,
        2,
        figsize=(7.25, 8.15),
        gridspec_kw={"hspace": 0.54, "wspace": 0.25},
    )
    fig.subplots_adjust(
        left=0.095, right=0.985, bottom=0.055, top=0.945, hspace=0.68, wspace=0.27
    )
    fig.text(
        0.30,
        0.987,
        "AVAILABLE OR CERTIFIED EVIDENCE",
        ha="center",
        va="top",
        fontsize=7.2,
        fontweight="bold",
        color=GRAY,
    )
    fig.text(
        0.745,
        0.987,
        "ATLAS CONSEQUENCE OR DECISIVE TEST",
        ha="center",
        va="top",
        fontsize=7.2,
        fontweight="bold",
        color=GRAY,
    )

    # ================================================================ A: risk
    ax = axes[0, 0]
    lam = example1_curves["lambda"].to_numpy(float)
    truth = example1_curves["hidden_truth"].to_numpy(float)
    compatible_models = [
        model
        for model in str(example1["certified_compatibility_set"]).split("|")
        if model
    ]
    model_columns = {
        column.removesuffix("_certified_witness"): column
        for column in example1_curves.columns
        if column.endswith("_certified_witness")
    }
    outlier = float(example1["fixed_outlier_lambda"])
    out_index = int(np.argmin(np.abs(lam - outlier)))
    ax.axvspan(float(lam.min()), 2.0, color=BLUE, alpha=0.07)
    ax.axvspan(2.0, float(lam.max()), color=ORANGE, alpha=0.055)
    model_colors = [BLUE, ORANGE, GREEN, "#CC79A7", GOLD]
    for color, model in zip(model_colors, compatible_models):
        ax.plot(
            lam,
            example1_curves[model_columns[model]],
            color=color,
            label=MODEL_SHORT[model],
        )
    ax.plot(lam, truth, color=BLACK, linestyle="--", linewidth=1.55, label="Hidden truth")
    ax.axvline(outlier, color=GRAY, linestyle=":", linewidth=0.9)
    ax.scatter(
        [outlier],
        [truth[out_index]],
        marker="*",
        s=42,
        color=GOLD,
        edgecolor=BLACK,
        linewidth=0.45,
        zorder=8,
    )
    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel("Nominal stress (kPa)")
    ax.set_title("Controlled 1 · compatible in-domain, divergent outside")
    legend(ax, ncol=2, loc="upper left", fontsize=4.9)
    badge(
        ax,
        "set: " + compact_signature(str(example1["certified_compatibility_set"])),
        color=BLUE,
    )
    clean_axis(ax)
    row_label(ax, "A")

    ax = axes[0, 1]
    finite = example1_witnesses["finite_at_fixed_outlier"].map(as_bool).to_numpy()
    errors = example1_witnesses["error_at_fixed_outlier_percent"].to_numpy(float)
    x = np.arange(len(example1_witnesses))
    bars = ax.bar(x[finite], errors[finite], color=BLUE, width=0.68)
    miss_tolerance = float(example1["miss_tolerance_percent"])
    ax.axhline(
        miss_tolerance,
        color=ORANGE,
        linestyle="--",
        linewidth=0.9,
        label=f"{miss_tolerance:.0f}% error threshold",
    )
    ax.set_xticks(
        x,
        [
            MODEL_SHORT.get(str(model), str(model))
            for model in example1_witnesses["target_model"]
        ],
    )
    ax.set_ylabel(f"Error at λ={outlier:.2f} (%)")
    ax.set_title("Risk from forcing one compatible family")
    annotate_bar_values(ax, bars, errors[finite], fmt=".1f")
    legend(ax, loc="upper left")
    badge(
        ax,
        f"{int(example1['n_models_miss_tolerance'])}/"
        f"{int(example1['n_finite_models_at_fixed_outlier'])} exceed threshold",
        color=ORANGE,
    )
    clean_axis(ax)

    # =========================================================== B: complexity
    source_model = str(example2["source_model"])
    simple_model = str(example2["best_simple_model"])
    source_truth = standard_curves(
        source_model,
        source_physical(example2_row),
        float(example2_row["source_response_scale_kpa"]),
    )
    simple_row = example2_fits[
        example2_fits["model"].astype(str) == simple_model
    ].iloc[0]
    shape_name = str(simple_row.get("shape_parameter_name", ""))
    simple_physical = (
        {}
        if simple_model == "NH"
        else {shape_name: float(simple_row["shape_parameter"])}
    )
    simple_fit = standard_curves(
        simple_model,
        simple_physical,
        float(simple_row["response_scale_kpa"]),
    )

    ax = axes[1, 0]
    plot_protocol_triplet(
        ax,
        source_truth,
        simple_fit,
        f"{MODEL_SHORT[source_model]} hidden source",
        f"{MODEL_SHORT[simple_model]} conventional fit",
    )
    ax.set_title("Controlled 2 · standard protocols hide model complexity")
    legend(ax, loc="upper left")
    badge(
        ax,
        f"NRMSE = {format_percent(example2['standard_protocol_nrmse_percent'])}%",
        color=BLUE,
    )
    clean_axis(ax)
    row_label(ax, "B")

    ax = axes[1, 1]
    truth_values = np.asarray(
        [example2_fstar["truth_d13_kpa"], example2_fstar["truth_d23_kpa"]],
        dtype=float,
    )
    simple_values = np.asarray(
        [example2_fstar["simple_d13_kpa"], example2_fstar["simple_d23_kpa"]],
        dtype=float,
    )
    x = np.arange(2)
    width = 0.34
    truth_bars = ax.bar(
        x - width / 2,
        truth_values,
        width,
        color=BLUE,
        label=f"{MODEL_SHORT[source_model]} source",
    )
    simple_bars = ax.bar(
        x + width / 2,
        simple_values,
        width,
        color=ORANGE,
        label=f"{MODEL_SHORT[simple_model]} fit",
    )
    ax.axhline(0.0, color=GRAY, linewidth=0.55)
    ax.set_xticks(x, [r"$\tau_1-\tau_3$", r"$\tau_2-\tau_3$"])
    ax.set_ylabel("Kirchhoff stress difference (kPa)")
    ax.set_title("One admissible state exposes the missing complexity")
    ax.set_ylim(
        min(0.0, 1.08 * float(min(truth_values.min(), simple_values.min()))),
        1.34 * float(max(truth_values.max(), simple_values.max())),
    )
    annotate_bar_values(ax, truth_bars, truth_values, fmt=".1f")
    annotate_bar_values(ax, simple_bars, simple_values, fmt=".1f")
    legend(ax, loc="upper left")
    badge(
        ax,
        f"{MODEL_SHORT[simple_model]} requires "
        f"{format_percent(example2['best_simple_complete_F_required_noise_percent'])}% > 3%",
        color=ORANGE,
    )
    clean_axis(ax)

    # ========================================================= C: no compatible
    example3_winner = example3_standard[
        example3_standard["is_conventional_winner"].map(as_bool)
    ].iloc[0]
    example3_winner_model = str(example3_winner["model"])
    example3_truth = withheld_curves(
        float(example3_summary["withheld_truth"]["b"]),
        float(example3_summary["withheld_truth"]["response_scale_kpa"]),
    )
    example3_prediction = standard_curves(
        example3_winner_model,
        json.loads(str(example3_winner["physical_json"])),
        float(example3_winner["scale_kpa"]),
    )

    ax = axes[2, 0]
    plot_protocol_triplet(
        ax,
        example3_truth,
        example3_prediction,
        "Withheld exponential-I₁ response",
        f"{MODEL_SHORT[example3_winner_model]} conventional fit",
    )
    ax.set_title("Controlled 3 · an excellent relative fit can still fail")
    legend(ax, loc="upper left")
    badge(
        ax,
        f"NRMSE = {format_percent(example3_winner['standard_nrmse_percent'])}%",
        color=BLUE,
    )
    clean_axis(ax)
    row_label(ax, "C")

    ax = axes[2, 1]
    merged = pd.DataFrame({"model": MODELS}).merge(
        example3_parent[["model", "required_noise_percent"]].rename(
            columns={"required_noise_percent": "parent"}
        ),
        on="model",
    ).merge(
        example3_validation[["model", "required_noise_percent"]].rename(
            columns={"required_noise_percent": "validation"}
        ),
        on="model",
    )
    x = np.arange(len(MODELS))
    width = 0.35
    ax.bar(
        x - width / 2,
        merged["parent"],
        width,
        color=BLUE,
        label="Frozen parent grid",
    )
    ax.bar(
        x + width / 2,
        merged["validation"],
        width,
        color=ORANGE,
        label="Independent validation",
    )
    ax.axhline(3.0, color=BLACK, linestyle="--", linewidth=0.85, label="3% tolerance")
    ax.axhline(5.0, color=GREEN, linestyle=":", linewidth=0.85, label="5% margin")
    ax.set_ylim(
        0.0,
        1.62
        * float(
            max(
                merged["parent"].max(),
                merged["validation"].max(),
                5.0,
            )
        ),
    )
    ax.set_xticks(x, [MODEL_SHORT[model] for model in MODELS])
    ax.set_ylabel("Required noise (%)")
    ax.set_title("Every candidate family is rejected")
    paired_minimum = np.minimum(
        merged["parent"].to_numpy(float),
        merged["validation"].to_numpy(float),
    )
    best_index = int(np.argmin(paired_minimum))
    minimum = float(paired_minimum[best_index])
    legend(ax, loc="upper left", ncol=2, fontsize=4.8)
    badge(
        ax,
        f"best: {MODEL_SHORT[str(merged.iloc[best_index]['model'])]} = "
        f"{format_percent(minimum)}% > 3%",
        color=ORANGE,
    )
    clean_axis(ax)

    # ============================================================= D: Treloar
    treloar_pair = treloar_result["frozen_directed_pairwise_witness"]
    treloar_atlas = treloar_result["atlas_source_lookup"]
    treloar_witness = treloar_result["atlas_selected_measured_witness"]
    treloar_source = str(treloar_pair["source_model"])
    treloar_target = str(treloar_pair["target_model"])
    treloar_models = [treloar_source, treloar_target]
    treloar_lookup = {
        str(row["model"]): row for _, row in treloar_fits.iterrows()
    }

    def treloar_predict(model: str, frame: pd.DataFrame) -> np.ndarray:
        row = treloar_lookup[model]
        coordinate = json_array(row["best_coordinate_json"])
        states = frame[["lambda1", "lambda2", "lambda3"]].to_numpy(float)
        return float(row["best_response_scale_kpa"]) * diagonal_first_template(
            model, coordinate, states
        )

    ax = axes[3, 0]
    ax.scatter(
        treloar_train["stretch"],
        treloar_train["v68_tau13_kPa"],
        s=10,
        color=BLACK,
        label="Uniaxial data",
        zorder=7,
    )
    for color, model in zip((BLUE, ORANGE), treloar_models):
        row = treloar_lookup[model]
        ax.plot(
            treloar_train["stretch"],
            treloar_predict(model, treloar_train),
            color=color,
            label=(
                f"{MODEL_SHORT[model]}: "
                f"{format_percent(row['uniaxial_nrmse_percent'])}% NRMSE"
            ),
        )
    ax.set_xlabel("Uniaxial stretch, λ")
    ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title("Treloar · uniaxial calibration retains two families")
    legend(ax, loc="upper left")
    clean_axis(ax)
    row_label(ax, "D")

    witness_mode = str(treloar_witness["atlas_selected_measured_witness_mode"])
    witness_stretch = float(
        treloar_witness["atlas_selected_measured_witness_stretch"]
    )
    if witness_mode == "equibiaxial":
        treloar_holdout = treloar_biaxial
        holdout_name = "equibiaxial"
    elif witness_mode == "pure_shear_planar":
        treloar_holdout = treloar_planar
        holdout_name = "planar/pure-shear"
    else:
        raise KeyError(f"Unexpected Treloar witness mode: {witness_mode}")

    ax = axes[3, 1]
    ax.scatter(
        treloar_holdout["stretch"],
        treloar_holdout["v68_tau13_kPa"],
        s=10,
        color=BLACK,
        label=f"Held-out {holdout_name} data",
        zorder=7,
    )
    for color, model in zip((BLUE, ORANGE), treloar_models):
        ax.plot(
            treloar_holdout["stretch"],
            treloar_predict(model, treloar_holdout),
            color=color,
            label=f"{MODEL_SHORT[model]} prediction",
        )
    selected = treloar_holdout[
        np.isclose(
            treloar_holdout["stretch"].to_numpy(float),
            witness_stretch,
            rtol=0.0,
            atol=1.0e-12,
        )
    ]
    if len(selected) != 1:
        raise RuntimeError("Treloar atlas-selected witness was not found uniquely.")
    ax.scatter(
        [witness_stretch],
        [float(selected["v68_tau13_kPa"].iloc[0])],
        marker="*",
        s=42,
        color=GOLD,
        edgecolor=BLACK,
        linewidth=0.45,
        label="Atlas-selected witness",
        zorder=10,
    )
    treloar_score = float(
        treloar_atlas["certified_scores_percent"][treloar_target]
    )
    ax.set_xlabel(f"{holdout_name.capitalize()} stretch, λ")
    ax.set_ylabel(r"$\tau_{11}-\tau_{33}$ (kPa)")
    ax.set_title(f"Held-out {holdout_name} test at λ={witness_stretch:.2f}")
    legend(ax, loc="upper left", fontsize=4.9)
    badge(
        ax,
        f"{MODEL_SHORT[treloar_source]}→{MODEL_SHORT[treloar_target]} "
        f"= {format_percent(treloar_score)}% > 3%",
        color=ORANGE,
        y=0.06,
        va="bottom",
    )
    clean_axis(ax)

    # =============================================================== E: brain
    brain_source = str(brain_result["primary_source_model"])
    brain_target = str(brain_result["primary_target_model"])
    brain_models = [brain_source, brain_target]
    brain_lookup = {str(row["model"]): row for _, row in brain_fits.iterrows()}

    def brain_predict(model: str, *, lam=None, gamma=None) -> np.ndarray:
        row = brain_lookup[model]
        coordinate = json_array(row["best_coordinate_json"])
        return float(row["best_response_scale_kpa"]) * brain_template(
            model, coordinate, lam=lam, gamma=gamma
        )

    ax = axes[4, 0]
    compression_x = brain_compression["stretch"].to_numpy(float)
    ax.scatter(
        compression_x,
        brain_compression["kirchhoff_stress_kPa"],
        s=10,
        color=BLACK,
        label="Compression data",
        zorder=7,
    )
    for color, model in zip((BLUE, ORANGE), brain_models):
        row = brain_lookup[model]
        ax.plot(
            compression_x,
            brain_predict(model, lam=compression_x),
            color=color,
            label=(
                f"{MODEL_SHORT[model]}: "
                f"{format_percent(row['compression_nrmse_percent'])}% NRMSE"
            ),
        )
    ax.set_xlabel("Compressive stretch, λ")
    ax.set_ylabel("Kirchhoff stress (kPa)")
    ax.set_title("Cortex · compression calibration retains two families")
    legend(ax, loc="lower right")
    clean_axis(ax)
    row_label(ax, "E")

    ax = axes[4, 1]
    tension_x = brain_tension["stretch"].to_numpy(float)
    ax.scatter(
        tension_x,
        brain_tension["kirchhoff_stress_kPa"],
        s=10,
        color=BLACK,
        label="Held-out tension",
        zorder=7,
    )
    for color, model in zip((BLUE, ORANGE), brain_models):
        row = brain_lookup[model]
        ax.plot(
            tension_x,
            brain_predict(model, lam=tension_x),
            color=color,
            label=(
                f"{MODEL_SHORT[model]}: "
                f"{format_percent(row['heldout_tension_nrmse_percent'])}% "
                "tension-only NRMSE"
            ),
        )
    brain_witness_mode = str(brain_result["atlas_selected_measured_witness_mode"])
    brain_witness_parameter = float(
        brain_result["atlas_selected_measured_witness_deformation_parameter"]
    )
    brain_witness_index = int(
        brain_result["atlas_selected_measured_witness_local_index"]
    )
    if brain_witness_mode == "tension":
        ax.scatter(
            [tension_x[brain_witness_index]],
            [float(brain_tension["kirchhoff_stress_kPa"].iloc[brain_witness_index])],
            marker="*",
            s=42,
            color=GOLD,
            edgecolor=BLACK,
            linewidth=0.45,
            label=f"Atlas-selected λ={brain_witness_parameter:.2f}",
            zorder=10,
        )
    source_to_target = float(
        brain_result["frozen_source_to_target_critical_noise_percent"]
    )
    target_to_source = float(
        brain_result["frozen_target_to_source_critical_noise_percent"]
    )
    ax.set_xlabel("Tensile stretch, λ")
    ax.set_ylabel("Kirchhoff stress (kPa)")
    ax.set_title("Atlas-selected tensile holdout favors one family")
    legend(ax, loc="upper left", fontsize=4.9)
    badge(
        ax,
        f"{MODEL_SHORT[brain_source]}→{MODEL_SHORT[brain_target]} "
        f"{format_percent(source_to_target)}% · reverse "
        f"{format_percent(target_to_source)}%",
        color=ORANGE,
        y=0.06,
        va="bottom",
    )
    clean_axis(ax)

    paths = save_figure(fig, outdir, OUTPUT_STEM)
    source_map = {
        "A": {
            "main_source_panels": ["former Figure 5C", "former Figure 5D"],
            "supplementary_panels": ["former Figure 5A", "former Figure 5B"],
            "input_directory": str(example1_dir),
        },
        "B": {
            "main_source_panels": ["former Figure 6A", "former Figure 6D"],
            "supplementary_panels": ["former Figure 6B", "former Figure 6C"],
            "input_directory": str(example2_dir),
        },
        "C": {
            "main_source_panels": ["former Figure 7A", "former Figure 7C"],
            "supplementary_panels": ["former Figure 7B", "former Figure 7D"],
            "input_directory": str(example3_dir),
        },
        "D": {
            "main_source_panels": ["former Figure 8A", "former Figure 8C"],
            "supplementary_panels": ["former Figure 8B"],
            "input_directory": str(treloar_dir),
        },
        "E": {
            "main_source_panels": ["former Figure 9A", "former Figure 9C"],
            "supplementary_panels": ["former Figure 9B", "former Figure 9D"],
            "input_directory": str(brain_dir),
        },
    }
    return paths, source_map


# =============================================================================
# One-shot execution
# =============================================================================

ROOT = resolve_root()
CELL2 = ROOT / "cell2_publication_validation"
OUTDIR = ROOT / OUTPUT_FOLDER
CERTIFIED = pd.read_csv(require(CELL2 / "certified_source_atlas_states.csv"))

print("=" * 92)
print("STANDALONE GENERATION OF COMBINED MANUSCRIPT FIGURE 5")
print("=" * 92)
print(f"Frozen project: {ROOT}")
print(f"Output folder:  {OUTDIR}")
print("Saved scientific outputs are loaded; no fitting or certification is rerun.\n")

PATHS, SOURCE_MAP = make_combined_figure5(ROOT, CERTIFIED, OUTDIR)
plt.close("all")

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "frozen_project_root": str(ROOT),
    "source_certified_atlas": str(CELL2 / "certified_source_atlas_states.csv"),
    "scientific_recomputation": False,
    "figure": {
        "number": 5,
        "title": "Controlled and experimental outcomes of constitutive compatibility analysis",
        "layout": "five rows by two columns",
        "paths": PATHS,
        "sha256": {kind: sha256(Path(path)) for kind, path in PATHS.items()},
        "panel_source_map": SOURCE_MAP,
    },
    "presentation_changes": [
        "former manuscript Figures 5 through 9 condensed into one main Figure 5",
        "two main-message plots retained per controlled or external case",
        "secondary score, ranking, deformation-map, and shear panels assigned to Supplementary Material",
        "saved atlas scores converted to compact callouts where a full score panel was secondary",
        "shared styling and compact legends applied for one-page readability",
        "raster, PDF, and SVG identifying metadata cleared",
    ],
}

manifest_path = OUTDIR / "combined_figure_5_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, sort_keys=True)

print("DONE")
for kind, path in PATHS.items():
    print(f"  {kind.upper():4s}: {path}")
print(f"Manifest: {manifest_path}")


STANDALONE GENERATION OF COMBINED MANUSCRIPT FIGURE 5
Frozen project: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas
Output folder:  /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES
Saved scientific outputs are loaded; no fitting or certification is rerun.

DONE
  PNG : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_5_Combined_Atlas_Outcomes.png
  TIFF: /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_5_Combined_Atlas_Outcomes.tiff
  PDF : /content/drive/MyDrive/Optimal_Protocol/V68_practical_standard_protocols_complete_F_biological_compatibility_atlas/UPDATED_MANUSCRIPT_FIGURES/Figure_5_Combined_Atlas_Outcomes.pdf
  SVG : /conte